# Public release note

This is a sanitized public copy of `omniASR_CTC_1B_V4_balanced_A100_80GB(1).ipynb`. Cell outputs, execution counters,
Colab user metadata, widget state, embedded display payloads, private storage
paths, and internal run identifiers were removed before publication. The
experiment source is preserved, but public placeholders must be configured from
your own generated contracts before rerunning dependent stages. See the repository
README and `docs/REPRODUCIBILITY.md` for prerequisites, data access, execution
order, expected artifacts, and the English explanation of this experiment.

Never paste credentials into a notebook. Use Colab Secrets or environment
variables (`HF_TOKEN`, `KAGGLE_USERNAME`, and `KAGGLE_KEY`).


# AfriVoices — omniASR CTC 1B V4.1

## Correctifs Drive + audit V4.1 → mélange équilibré → A100 80 Go

Cette version conserve les correctifs rule-safe de V3 et ajoute une nouvelle phase joint
multilingue. **Aucun entraînement ne peut démarrer avant un bilan PASS des données réellement
utilisables.** Le sampler donne le même poids aux six langues, puis applique un ratio de
domaine adapté à chaque langue au lieu d'imposer arbitrairement 50/50.

Recommandation intégrée : continuer depuis le checkpoint joint exact `step_5000`, avec un
nouvel optimiseur, un plafond de **2 500 optimizer steps**, un learning rate de **3e-6**,
et évaluer les checkpoints tous les 250 pas. Le meilleur checkpoint, pas nécessairement le dernier,
devient l'unique modèle final.

Le micro-batch n'est pas supposé à partir du nom « A100 80 GB » : une calibration mesure
plusieurs configurations et conserve la plus grande restant sous 85 % de VRAM mesurée.


## 1 — Setup : Drive, dépendances, GPU, credentials Kaggle

In [ ]:
# 1.1 — Monter Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 1.2 — Dépendances : clone du dépôt PUIS installation DEPUIS le clone.
# Pourquoi pas PyPI ? La 0.2.0 publiée déclare Requires-Python <=3.12 (borne mal écrite) : le
# Python 3.12.12 de Colab la dépasse au sens strict et pip retombe silencieusement sur la 0.1.0,
# qui n'a PAS les cartes v2. Le dépôt a la borne corrigée (<3.14) et aligne code+cartes+recettes.
import subprocess, os
_torch_before, _numpy_before = None, None
try:
    import torch as _t0; _torch_before = _t0.__version__
except Exception:
    pass
try:
    import numpy as _n0; _numpy_before = _n0.__version__
except Exception:
    pass
OMNI_REPO_DIR = "/content/omnilingual-asr"
if not os.path.isdir(OMNI_REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/facebookresearch/omnilingual-asr.git", OMNI_REPO_DIR], check=True)

# PATCH idempotent du calculateur de WER du dépôt : une hypothèse CTC vide (que des blanks,
# légitime en cours d'entraînement) fait planter pad_seqs à la VALIDATION
# ("All lengths in seq_lens must be >= 1"). On substitue un unique token de padding
# (retiré au détokenisage -> chaîne vide, WER correct). À réappliquer à chaque clone frais.
import re as _re
_wc = os.path.join(OMNI_REPO_DIR, "workflows/recipes/wav2vec2/asr/wer_calculator.py")
_src = open(_wc, encoding="utf-8").read()
if "hyp_seq.new_full" in _src:
    print("PATCH wer_calculator : déjà appliqué.")
else:
    _pat = "( +)" + _re.escape("hyp_seq = hyp_seq[hyp_seq != self._blank_label]")
    _m = _re.search(_pat, _src)
    assert _m, "wer_calculator.py a changé : m'envoyer son contenu pour adapter le patch."
    _ind = _m.group(1)
    _NL = chr(10)
    _ins = (_m.group(0) + _NL + _ind + "if hyp_seq.numel() == 0:"
            + "  # patch: hypothese vide -> pad_seqs exige len >= 1" + _NL
            + _ind + "    hyp_seq = hyp_seq.new_full((1,), self._pad_idx)")
    open(_wc, "w", encoding="utf-8").write(_src.replace(_m.group(0), _ins))
    print("PATCH wer_calculator : appliqué (hypothèses vides gérées).")
!pip install -q /content/omnilingual-asr 'pyarrow>=20.0.0' soundfile jiwer pandas
import importlib.metadata as _md
# fairseq2 impose sa version de torch ; les torchaudio/torchvision préinstallés (compilés pour
# l'ancien torch) casseraient les imports du package (undefined symbol) -> alignement forcé.
_torch_base = _md.version("torch").split("+")[0]
_torch_mm = ".".join(_torch_base.split(".")[:2])
_TV_FOR_TORCH = {"2.8": "0.23.0", "2.9": "0.24.0", "2.10": "0.25.0", "2.11": "0.26.0"}
_targets = {"torchaudio": _torch_base, "torchvision": _TV_FOR_TORCH.get(_torch_mm)}
_realigned = False
for _pkg, _ver in _targets.items():
    if not _ver:
        continue
    try:
        _cur = _md.version(_pkg).split("+")[0]
    except Exception:
        continue
    if _cur != _ver:
        print(f"Alignement {_pkg} {_cur} -> {_ver} (torch {_torch_base})")
        subprocess.run(["python3", "-m", "pip", "install", "-q", f"{_pkg}=={_ver}"], check=False)
        _realigned = True
if _realigned:
    print("⚠️ torchaudio/torchvision réalignés : REDÉMARRER le runtime puis ré-exécuter depuis 1.1.")
_sha = subprocess.run(["git", "-C", OMNI_REPO_DIR, "rev-parse", "HEAD"],
                      capture_output=True, text=True).stdout.strip()
_oa_ver = _md.version("omnilingual-asr")
print("omnilingual-asr (installé depuis le clone) :", _oa_ver, "| dépôt @", _sha)
assert _oa_ver != "0.1.0", ("Toujours en 0.1.0 : l'installation depuis le clone a échoué — "
                            "lire le log pip ci-dessus.")
print("fairseq2 :", _md.version("fairseq2"))
def _base_ver(v):
    return str(v).split("+")[0]   # '2.8.0+cu128' et '2.8.0' = même version, notations différentes
for _name, _before in (("torch", _torch_before), ("numpy", _numpy_before)):
    if _before and _base_ver(_md.version(_name)) != _base_ver(_before):
        print(f"⚠️ {_name} a changé ({_before} -> {_md.version(_name)}) : "
              "REDÉMARRER le runtime puis ré-exécuter depuis 1.1.")


In [ ]:
# 1.3 — Imports communs + rapport GPU
import io, json, math, os, re, shutil, subprocess, sys, time, unicodedata
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
import torch

assert torch.cuda.is_available(), "Aucun GPU : Runtime -> Modifier le type d'exécution -> A100."
_gpu = torch.cuda.get_device_properties(0)
GPU_VRAM_GB = _gpu.total_memory / 1e9
print(f"GPU : {_gpu.name} — {GPU_VRAM_GB:.1f} GB VRAM — bf16 : {torch.cuda.is_bf16_supported()}")
if "A100" not in _gpu.name:
    print("⚠️ Pas un A100 : la calibration (section 6) ajustera, mais le débit sera plus faible.")

def now_iso():
    from datetime import datetime, timezone
    return datetime.now(timezone.utc).isoformat()

def save_json(obj, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=str)
    print("json ->", path)

_STEP_RE = re.compile(r"step[_-](\d+)")
def latest_ckpt(root):
    """Dernier RÉPERTOIRE de checkpoint step_N, par NUMÉRO de step (jamais par tri
    lexicographique : 'step_750' > 'step_5000' en tri de chaînes !). Structure fairseq2 0.6 :
    ws_*/checkpoints/step_N/{model/pp_00/tp_00/..., trainer/, optimizer/}. None si aucun."""
    dirs = [d for d in Path(root).rglob("step_*")
            if d.is_dir() and _STEP_RE.search(d.name) and (d / "model").exists()
            and not d.name.endswith(".tmp")]   # .tmp = checkpoint NON finalisé (crash en cours d'écriture)
    return str(max(dirs, key=lambda d: int(_STEP_RE.search(d.name).group(1)))) if dirs else None

def prune_checkpoints(output_dir, keep=2):
    """Ne garder que les `keep` derniers répertoires step_* (un checkpoint 1B + optimiseur
    pèse >10 GB : 20 checkpoints satureraient le Drive). Chercher aussi une option native :
    grep -rn 'keep_last' dans le dépôt (voir 6.1)."""
    dirs = [d for d in Path(output_dir).rglob("step_*") if d.is_dir() and _STEP_RE.search(d.name)]
    dirs = sorted(dirs, key=lambda d: int(_STEP_RE.search(d.name).group(1)))
    for d in dirs[:-keep] if len(dirs) > keep else []:
        shutil.rmtree(d, ignore_errors=True)
        print("checkpoint élagué :", d)


In [ ]:
# 1.4 — Credentials Kaggle (nécessaires en section 10 uniquement)
_KAGGLE_JSON_CANDIDATES = ["/content/afrivoices_project",
                           "/content/afrivoices_project/kaggle.json"]
_kg_dst = os.path.expanduser("~/.kaggle/kaggle.json")
if not os.path.exists(_kg_dst):
    for cand in _KAGGLE_JSON_CANDIDATES:
        if os.path.exists(cand):
            os.makedirs(os.path.dirname(_kg_dst), exist_ok=True)
            shutil.copyfile(cand, _kg_dst); os.chmod(_kg_dst, 0o600)
            print("kaggle.json installé depuis", cand); break
    else:
        print("⚠️ kaggle.json introuvable sur le Drive — le déposer avant la section 10 "
              "(ou uploader via l'onglet Fichiers vers ~/.kaggle/kaggle.json).")
else:
    print("kaggle.json déjà en place.")


## 2 — Configuration centrale + garde-fous

In [ ]:
# 2.1 — Configuration centrale (UNIQUE endroit à modifier) + rechargement de la calibration
DRIVE_ROOT = "/content/afrivoices_project"
FT_RUN_NAME = "experiment_run"

FT_CONFIG = {
    "run_name": FT_RUN_NAME,
    "seed": 42,
    "model_card": "omniASR_CTC_1B_v2",
    "fallback_model_card": "omniASR_CTC_300M_v2",
    "tokenizer_name": "omniASR_tokenizer_written_v2",  # tokenizer_ref de la carte omniASR_CTC_1B_v2
    "manifest_dir": os.path.join(DRIVE_ROOT, "manifests"),
    "curated_all": os.path.join(DRIVE_ROOT, "manifests", "curated_manifest_all.csv"),
    "parquet_root": "/content/experiment_run",
    # cache Drive du parquet COMPLET (~26 GB) : évite de relire 230k FLAC via FUSE à chaque
    # session (des heures). Mettre à False si le quota Drive ne le permet pas.
    "parquet_drive_cache": os.path.join(DRIVE_ROOT, "finetune_runs", FT_RUN_NAME, "parquet_cache"),
    "use_parquet_drive_cache": True,
    "experiment_run": os.path.join(DRIVE_ROOT, "finetune_runs", FT_RUN_NAME),
    "output_dir": os.path.join(DRIVE_ROOT, "finetune_runs", FT_RUN_NAME, "train_output"),
    "fast_output_dir": os.path.join(DRIVE_ROOT, "finetune_runs", FT_RUN_NAME, "fast_output"),
    "log_dir": os.path.join(DRIVE_ROOT, "finetune_runs", FT_RUN_NAME, "logs"),
    "report_dir": os.path.join(DRIVE_ROOT, "finetune_runs", FT_RUN_NAME, "reports"),
    "lr": 1e-5,
    "num_steps": 5000,
    "validate_every_n_steps": 250,
    "checkpoint_every_n_steps": 250,
    "checkpoints_keep": 2,   # JAMAIS 1 : après un SIGINT mural le dernier step peut être partiel
    "kaggle_test_filename": "anv-test-data-nt",   # vérifié via l'API (seul fichier, 20 Ko)
    "max_audio_len": 640_000,        # 40 s x 16 kHz (contrainte d'inférence CTC < 40 s)
    "max_num_elements": 5_120_000,
    "grad_accum": 8,
    "freeze_encoder_for_n_steps": 0,
    "mixed_precision": "torch.bfloat16",   # la valeur exacte est relue du YAML officiel en 6.1
    "fast_hours_per_language": 2.0,
    "fast_num_steps": 400,
    "max_wall_hours_per_session": 22.5,
}

LANG_TO_OMNI = {"sw": "swh_Latn", "kik": "kik_Latn", "kln": "kln_Latn",
                "luo": "luo_Latn", "som": "som_Latn", "mas": "mas_Latn"}
LANG_FALLBACK_OMNI = {"mas": "saq_Latn"}   # samburu : langue maa la plus proche couverte
LANG_TO_SUBMISSION = {"sw": "swa", "kik": "kik", "kln": "kln",
                      "luo": "luo", "som": "som", "mas": "mas"}
SUBMISSION_TEXT_COLUMN = "transcription"   # consigne Kaggle ambiguë (prediction/transcription)
LANGUAGES = list(LANG_TO_OMNI)

for _d in ("experiment_run", "output_dir", "fast_output_dir", "log_dir", "report_dir"):
    Path(FT_CONFIG[_d]).mkdir(parents=True, exist_ok=True)

# Les décisions de calibration (section 6) SURVIVENT aux sessions : rechargement systématique.
_calib_path = os.path.join(FT_CONFIG["report_dir"], "calibration_report.json")
if os.path.exists(_calib_path):
    _c = json.load(open(_calib_path))
    if _c.get("chosen"):
        FT_CONFIG["model_card"] = _c["model_card"]
        FT_CONFIG["grad_accum"] = _c["chosen"]["grad_accum"]
        FT_CONFIG["max_num_elements"] = _c["chosen"]["max_num_elements"]
        print(f"Calibration rechargée : {FT_CONFIG['model_card']}, accum={FT_CONFIG['grad_accum']}, "
              f"max_elem={FT_CONFIG['max_num_elements']}")
    else:
        print("⚠️ Calibration précédente NON conclusive : relancer la section 6.")
else:
    print("Pas encore de calibration : section 6 obligatoire avant 7/8.")
save_json(FT_CONFIG, os.path.join(FT_CONFIG["report_dir"], "experiment_run"))


In [ ]:
# 2.1b — V3 : révision, conformité et modes expérimentaux
# QA_PATCH: SAFE_DEFAULTS
# QA_PATCH: COMPLIANCE_SINGLE_MODEL
PIPELINE_REVISION = "afrivoices-omniasr-v3-20260709"
NORMALIZATION_VERSION = "nfc-mojibake-safe-v3"
CACHE_SCHEMA_VERSION = 3

OFFICIAL_MODEL_PARAM_LIMIT = 1_000_000_000
OMNI_CTC_1B_V2_PARAM_COUNT = 975_675_056
PARAMETER_HEADROOM = OFFICIAL_MODEL_PARAM_LIMIT - OMNI_CTC_1B_V2_PARAM_COUNT

# FINAL : un seul checkpoint acoustique. Peut pointer vers un checkpoint joint ou fusionné.
FINAL_UNIFIED_CHECKPOINT = None   # None -> dernier checkpoint joint FT_CONFIG["output_dir"]

# EXPÉRIMENTAL : ne jamais activer sans confirmation écrite explicite des organisateurs.
COMPLIANCE_MODE = True
ORGANIZER_CONFIRMED_MULTI_FULL_CHECKPOINTS = False
ALLOW_MULTI_CHECKPOINT_ROUTING = False
ALLOW_DOMAIN_ACOUSTIC_ROUTING = False
ALLOW_CACHE_DELETE = False
USE_MULTI_FULL_CHECKPOINT_ROUTING = False
EXPERIMENTAL_DOMAIN_CHECKPOINTS = {}  # ex. {("som", "unscripted"): "/.../step_2000"}

# Le routage de configurations de décodage par domaine reste activable avec un seul modèle.
USE_DOMAIN_LM_ROUTING = True
STRONG_ARTIFACT_HASH_FOR_FINAL = False  # True avant remise finale : SHA256 complet des poids

if USE_MULTI_FULL_CHECKPOINT_ROUTING and not ORGANIZER_CONFIRMED_MULTI_FULL_CHECKPOINTS:
    raise RuntimeError(
        "Routage multi-checkpoints bloqué : confirmation écrite des organisateurs requise."
    )
if OMNI_CTC_1B_V2_PARAM_COUNT >= OFFICIAL_MODEL_PARAM_LIMIT:
    raise RuntimeError("Le modèle acoustique dépasse la limite officielle.")

FT_CONFIG.update({
    "pipeline_revision": PIPELINE_REVISION,
    "normalization_version": NORMALIZATION_VERSION,
    "cache_schema_version": CACHE_SCHEMA_VERSION,
})
_compliance = {
    "pipeline_revision": PIPELINE_REVISION,
    "single_unified_checkpoint_default": not USE_MULTI_FULL_CHECKPOINT_ROUTING,
    "multi_checkpoint_confirmed": ORGANIZER_CONFIRMED_MULTI_FULL_CHECKPOINTS,
    "model_parameters": OMNI_CTC_1B_V2_PARAM_COUNT,
    "parameter_limit": OFFICIAL_MODEL_PARAM_LIMIT,
    "headroom": PARAMETER_HEADROOM,
}
save_json(_compliance, os.path.join(FT_CONFIG["report_dir"], "compliance_v3.json"))
print("V3 prête | marge théorique :", f"{PARAMETER_HEADROOM:,}", "paramètres")
print("Mode final :", "MULTI (confirmé)" if USE_MULTI_FULL_CHECKPOINT_ROUTING else "UN SEUL checkpoint")

def require_state(*names):
    missing = [name for name in names if name not in globals()]
    if missing:
        raise RuntimeError("État manquant : " + ", ".join(missing))


# V4 : pont persistant vers l'unique checkpoint sélectionné par 18.7.
_v4_final_pointer_path = os.path.join(FT_CONFIG["report_dir"], "balanced_v4",
                                      "final_unified_checkpoint_v4.json")
if os.path.exists(_v4_final_pointer_path):
    import hashlib as _v4_hashlib
    _v4_pointer = json.load(open(_v4_final_pointer_path, encoding="utf-8"))
    _v4_checkpoint = os.path.realpath(_v4_pointer["checkpoint"])
    _v4_model_root = Path(_v4_checkpoint) / "model" if os.path.isdir(_v4_checkpoint) else Path(_v4_checkpoint)
    _v4_weights = ([p for p in _v4_model_root.rglob("*") if p.is_file()]
                   if _v4_model_root.is_dir() else [_v4_model_root])
    assert len(_v4_weights) == 1 and _v4_weights[0].exists(), _v4_weights
    _v4_digest = _v4_hashlib.sha256()
    with open(_v4_weights[0], "rb") as _v4_handle:
        for _v4_block in iter(lambda: _v4_handle.read(8 << 20), b""):
            _v4_digest.update(_v4_block)
    assert _v4_digest.hexdigest() == _v4_pointer["weight_sha256"],         "Poids V4 différents du pointeur final"
    _v4_contract_path = os.path.join(FT_CONFIG["report_dir"], "balanced_v4",
                                     "run_contract_v4.json")
    _v4_contract = json.load(open(_v4_contract_path, encoding="utf-8"))
    assert _v4_contract["contract_sha256"] == _v4_pointer["run_contract_sha256"]
    FINAL_UNIFIED_CHECKPOINT = _v4_checkpoint
    print("Checkpoint V4 final rechargé :", FINAL_UNIFIED_CHECKPOINT)


In [ ]:
# 2.2 — Garde-fous V3 : aucune phase coûteuse active par défaut
RUN_CALIBRATION = False
RUN_FAST_TRAINING = False
RUN_FULL_TRAINING = False
RUN_TEST_INFERENCE = False
RUN_SPONT_BUILD = False
RUN_SPONT_TRAINING = False
RUN_MIXED_TRAINING = False
RUN_REAL_LONG_AUDIT = False
RUN_FINAL_INFERENCE = False
RUN_UNIFIED_DELTA_MERGE = False
RUN_EXPERIMENTAL_TOPUP = False
RUN_SWA_SPONT_BUILD = False
RUN_SWA_SPONT_TRAINING = False
RUN_LM_BUILD = False
RUN_LM_TUNING = False
RUN_LM_EXPERIMENTS = False

_flags = {name: value for name, value in list(globals().items())
          if name.startswith("RUN_") and isinstance(value, bool)}
_active = [name for name, value in _flags.items() if value]
print("Phases coûteuses actives :", _active or "aucune")
if len(_active) > 1:
    raise RuntimeError(f"Activer une seule phase coûteuse à la fois : {_active}")
import random
random.seed(FT_CONFIG["seed"])
np.random.seed(FT_CONFIG["seed"])
torch.manual_seed(FT_CONFIG["seed"])


In [ ]:
# 2.2b — V4 : phases équilibrées, toutes désactivées par défaut
RUN_DATA_AUDIT_V4 = False
RUN_BALANCED_BUILD_V4 = False
RUN_A100_80GB_CALIBRATION_V4 = False
RUN_BALANCED_FINETUNE_V4 = False
RUN_BALANCED_EVAL_V4 = True

# Invariants de conformité : un seul modèle acoustique complet dans le livrable final.
ALLOW_MULTI_CHECKPOINT_ROUTING = False
ALLOW_DOMAIN_ACOUSTIC_ROUTING = False
USE_MULTI_FULL_CHECKPOINT_ROUTING = False

_v4_active = [name for name, value in globals().items()
              if name.startswith("RUN_") and isinstance(value, bool) and value]
if len(_v4_active) > 1:
    raise RuntimeError(f"Une seule phase coûteuse à la fois : {_v4_active}")
print("Phases V4 actives :", _v4_active or "aucune")


In [ ]:
# 2.3 — Couverture des langues par le modèle : décision AUTOMATIQUE du repli maasai
# (mas_Latn est absent des 1672 langues supportées ; maa_Latn = mazatèque, PAS le maasai).
_supported = None
try:
    from omnilingual_asr.models.wav2vec2_llama.lang_ids import supported_langs as _supported
except Exception as exc:
    print(f"Import lang_ids différent dans cette version ({exc!r}) — repli sur le dépôt cloné :")
    try:
        _lang_py = os.path.join(OMNI_REPO_DIR, "src/omnilingual_asr/models/wav2vec2_llama/lang_ids.py")
        _ns = {}
        exec(open(_lang_py, encoding="utf-8").read(), _ns)
        _supported = _ns.get("supported_langs") or next(v for v in _ns.values()
                                                        if isinstance(v, (list, set, tuple)) and len(v) > 1000)
    except Exception as exc2:
        _h = subprocess.run(["grep", "-rn", "-e", "supported_langs", "-e", "mas_Latn",
                             os.path.join(OMNI_REPO_DIR, "src")], capture_output=True, text=True).stdout
        print(_h[:1500])
        raise SystemExit(f"Liste des langues introuvable ({exc2!r}) : adapter d'après le grep ci-dessus.")
    # NB : lang_ids.py vit sous wav2vec2_llama mais c'est l'unique liste du package ;
    # sa validité pour la carte CTC sera confirmée de facto au FAST run.
_supported = set(_supported)
print(f"{len(_supported)} langues supportées par le modèle.")
_use_mas_fallback = "mas_Latn" not in _supported
if _use_mas_fallback:
    assert LANG_FALLBACK_OMNI["mas"] in _supported, \
        f"Le repli {LANG_FALLBACK_OMNI['mas']} n'est pas supporté non plus — choisir un autre code."
    print(f"mas_Latn non supporté (attendu) -> étiquette interne '{LANG_FALLBACK_OMNI['mas']}' "
          "(samburu) pour l'entraînement ET l'inférence. Le code de SOUMISSION reste 'mas'.")
else:
    print("mas_Latn supporté (surprise !) : pas de repli.")
for _l, _code in LANG_TO_OMNI.items():
    if _l != "mas":
        assert _code in _supported, f"{_code} absent des langues supportées ?!"

def omni_lang(lang):
    if lang == "mas" and _use_mas_fallback:
        return LANG_FALLBACK_OMNI["mas"]
    return LANG_TO_OMNI[lang]

# Test d'import de la pipeline d'inférence MAINTENANT (et pas en 7.3 après des heures de GPU) :
# les torchaudio/torchvision préinstallés de Colab, compilés pour un autre torch, cassent ces
# imports (undefined symbol / operator does not exist) tant qu'ils ne sont pas réalignés (1.2).
try:
    from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline  # noqa: F401
    print("✓ Import ASRInferencePipeline OK — l'inférence sera disponible.")
except Exception as exc:
    raise SystemExit(f"Import de la pipeline d'inférence impossible : {exc!r}. "
                     "Réalignement nécessaire : ré-exécuter 1.2 (elle installe les bonnes versions "
                     "de torchaudio/torchvision), redémarrer le runtime, reprendre depuis 1.1.")

In [ ]:
# 2.4 — Fonctions d'entraînement communes (YAML, run_recipe) — DANS le chemin 1→5,
# pour que le flux session-fraîche « 1→5 puis 8.1→8.2 » soit autosuffisant.
# La valeur exacte de mixed_precision.dtype est relue du YAML OFFICIEL du dépôt cloné.
_official_yaml = os.path.join(OMNI_REPO_DIR, "workflows/recipes/wav2vec2/asr/configs/"
                                             "ctc-finetune-recommendation.yaml")
_dtype_val = FT_CONFIG["mixed_precision"]
if os.path.exists(_official_yaml):
    _m = re.search(r'mixed_precision:\s*\n\s*dtype:\s*"?([A-Za-z0-9_.]+)"?',
                   open(_official_yaml).read())
    if _m:
        _dtype_val = _m.group(1)
        print("dtype officiel relevé :", _dtype_val)
else:
    print(f"⚠️ YAML officiel introuvable ({_official_yaml}) : dtype par défaut '{_dtype_val}' — "
          "vérifier le chemin des configs dans le dépôt cloné.")
# Option native de rétention de checkpoints ? (sinon prune_checkpoints prend le relais)
_keep_hits = subprocess.run(["grep", "-rn", "--include=*.py", "--include=*.yaml", "-i",
                             "keep_last", os.path.join(OMNI_REPO_DIR, "workflows"),
                             os.path.join(OMNI_REPO_DIR, "src")],
                            capture_output=True, text=True).stdout
print("Option de rétention native :", (_keep_hits[:800] or "aucune trouvée -> prune_checkpoints"))

# Carte d'asset du DATASET : comme les modèles/tokenizers, les datasets fairseq2 se
# résolvent par carte — et c'est ELLE qui porte le chemin racine du parquet (dataset_config.data,
# cf. cards/datasets/example_dataset.yaml du dépôt). Écrite dans le dossier cards/ du package
# installé : déjà enregistré comme source d'assets, y compris pour le sous-processus recette.
import omnilingual_asr as _oa
_DATASET_CARD_PATH = os.path.join(os.path.dirname(_oa.__file__), "cards", "datasets",
                                  "afrivoices_dataset.yaml")

def write_dataset_card(data_root=None):
    data_root = data_root or os.path.join(FT_CONFIG["parquet_root"], "version=0")
    os.makedirs(os.path.dirname(_DATASET_CARD_PATH), exist_ok=True)
    with open(_DATASET_CARD_PATH, "w") as f:
        f.write(f"""name: afrivoices_dataset
dataset_family: mixture_parquet_asr_dataset
dataset_config:
  data: {data_root}
tokenizer_ref: {FT_CONFIG["tokenizer_name"]}
""")
    print("carte dataset ->", _DATASET_CARD_PATH, "| data:", data_root)

write_dataset_card()

def write_recipe_yaml(path, model_card, tsv_path, num_steps, grad_accum, max_num_elements,
                      lr=None, ckpt_every=None):
    ck = ckpt_every or FT_CONFIG["checkpoint_every_n_steps"]
    # contrainte fairseq2 : validate/checkpoint_every doivent être multiples de publish_metrics
    # example_shuffle_window: 1000 (le code interdit 0 = shuffle du dataset entier -> OOM ;
    # l'équilibre inter-langues vient déjà des poids hours^beta, la fenêtre ne fait que brasser)
    pub = 50 if ck % 50 == 0 else ck
    yaml_text = f"""model:
  name: "{model_card}"

dataset:
  name: "afrivoices_dataset"
  train_split: "train"
  valid_split: "dev"
  storage_mode: "MIXTURE_PARQUET"
  task_mode: "ASR"
  mixture_parquet_storage_config:
    dataset_summary_path: "{tsv_path}"
    beta_corpus: 0.5
    beta_language: 0.5
    fragment_loading:
      cache: True
  asr_task_config:
    max_audio_len: {FT_CONFIG["max_audio_len"]}
    max_num_elements: {max_num_elements}
    batch_shuffle_window: 1
    normalize_audio: true
    example_shuffle_window: 1000

tokenizer:
  name: "{FT_CONFIG["tokenizer_name"]}"

optimizer:
  config:
    lr: {lr if lr is not None else FT_CONFIG["lr"]}

trainer:
  freeze_encoder_for_n_steps: {FT_CONFIG["freeze_encoder_for_n_steps"]}
  mixed_precision:
    dtype: "{_dtype_val}"
  grad_accumulation:
    num_batches: {grad_accum}

regime:
  num_steps: {num_steps}
  validate_every_n_steps: {ck}
  validate_after_n_steps: {ck}
  checkpoint_every_n_steps: {ck}
  checkpoint_after_n_steps: {ck}
  publish_metrics_every_n_steps: {pub}
  publish_metrics_after_n_steps: {pub}
"""
    with open(path, "w") as f:
        f.write(yaml_text)
    print("yaml ->", path)
    return path

def load_finetuned_pipeline(ckpt_path=None, dtype=torch.bfloat16):
    """Pipeline d'inférence : carte officielle (ckpt_path=None) ou NOTRE checkpoint fine-tuné.
    Voie officielle (signature de ASRInferencePipeline) : pas de checkpoint_path, mais des
    kwargs model=/tokenizer= -> load_model(carte) + load_state_dict(checkpoint) + load_tokenizer."""
    from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline
    if ckpt_path is None:
        return ASRInferencePipeline(FT_CONFIG["model_card"])
    from fairseq2.models.hub import load_model
    from fairseq2.data.tokenizers.hub import load_tokenizer
    model = load_model(FT_CONFIG["model_card"], device=torch.device("cuda"), dtype=dtype)
    _ck = Path(ckpt_path)
    if _ck.is_dir():   # répertoire step_N : les poids sont sous model/pp_00/tp_00/
        _wfiles = sorted(q for q in (_ck / "model").rglob("*") if q.is_file())
        assert _wfiles, f"Aucun fichier de poids sous {_ck}/model"
        if len(_wfiles) > 1:
            print("Fichiers de poids multiples :", [str(q) for q in _wfiles])
            raise SystemExit("Checkpoint shardé (plusieurs fichiers) : me montrer cette liste "
                             "pour adapter le chargement.") from None
        _wfile = _wfiles[0]
    else:
        _wfile = _ck
    print("Poids :", _wfile)
    if str(_wfile).endswith(".safetensors"):
        from safetensors.torch import load_file as _st_load
        state = _st_load(str(_wfile))
    else:
        state = torch.load(str(_wfile), map_location="cpu", weights_only=False)
    sd = state.get("model", state) if isinstance(state, dict) else state
    try:
        model.load_state_dict(sd)
    except Exception:
        try:   # certains checkpoints préfixent les clés (module., model., _orig_mod.)
            _strip = lambda k: k.split("module.", 1)[-1].split("_orig_mod.", 1)[-1]
            model.load_state_dict({_strip(k): v for k, v in sd.items()})
            print("state_dict chargé après retrait des préfixes de clés.")
        except Exception as exc:
            _keys = list(sd.keys())[:12] if hasattr(sd, "keys") else type(sd)
            print("Clés du checkpoint (extrait) :", _keys)
            raise SystemExit(f"Format de state_dict inattendu ({exc!r}) : me montrer les clés "
                             "affichées ci-dessus.") from None
    model.eval()
    tok = load_tokenizer(FT_CONFIG["model_card"])
    return ASRInferencePipeline(None, model=model, tokenizer=tok)

def run_recipe(output_dir, yaml_path, log_name, wall_limit_h=None, prune_keep=None):
    """Sous-processus + stream des logs + coupure murale PROPRE (SIGINT puis drain du pipe
    jusqu'à EOF — sinon le fils bloque sur un pipe plein et survit en zombie GPU).
    Relancer avec le même output_dir reprend au dernier checkpoint PÉRIODIQUE."""
    os.makedirs(output_dir, exist_ok=True)
    log_path = os.path.join(FT_CONFIG["log_dir"], log_name)
    cmd = [sys.executable, "-m", "workflows.recipes.wav2vec2.asr", output_dir,
           "--config-file", yaml_path]
    print("CMD :", " ".join(cmd))
    t0 = time.time(); interrupted = False; _sigint_t = None
    with open(log_path, "a", encoding="utf-8") as logf:
        proc = subprocess.Popen(cmd, cwd=OMNI_REPO_DIR, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            logf.write(line)
            if any(k in line.lower() for k in ("loss", "error", "step", "oom", "memory", "cuda")):
                print(line.rstrip()[:200])
            # élagage EN COURS de run : sans lui, jusqu'à 20 checkpoints >10 GB s'accumulent
            # sur Drive avant la fin (saturation de quota en plein entraînement)
            if prune_keep and "checkpoint" in line.lower():
                try:
                    prune_checkpoints(output_dir, keep=prune_keep)
                except Exception as _pexc:
                    print("prune en cours de run impossible :", repr(_pexc)[:120])
            if wall_limit_h and not interrupted and (time.time() - t0) > wall_limit_h * 3600:
                print(f"⏱️ Limite murale {wall_limit_h}h : SIGINT (reprise au dernier checkpoint "
                      f"périodique, <= {FT_CONFIG['checkpoint_every_n_steps']} steps perdus).")
                proc.send_signal(2); _sigint_t = time.time()
                interrupted = True   # on CONTINUE de drainer le pipe jusqu'à EOF
            if interrupted and _sigint_t and (time.time() - _sigint_t) > 900:
                print("SIGINT ignoré depuis 15 min : kill.")
                proc.kill()
                _sigint_t = None
        try:
            proc.wait(timeout=900)
        except subprocess.TimeoutExpired:
            print("Le processus ne se termine pas : kill.")
            proc.kill(); proc.wait()
    print(f"Recette : code {proc.returncode} en {(time.time()-t0)/60:.1f} min ; log : {log_path}")
    return proc.returncode, log_path


## 3 — Chargement et contrôle des manifests curés

In [ ]:
# 3.1 — Manifest curé + contrôles
manifest = pd.read_csv(FT_CONFIG["curated_all"], low_memory=False)
required_cols = {"sample_id", "split", "language", "curated_audio_path", "text", "duration"}
missing = required_cols - set(manifest.columns)
assert not missing, f"Colonnes manquantes : {missing}"
manifest = manifest[manifest["language"].isin(LANGUAGES)].reset_index(drop=True)
assert set(manifest["language"].unique()) == set(LANGUAGES), \
    f"Langues du manifest != attendues : {sorted(manifest['language'].unique())} vs {LANGUAGES}"
_hours_tot = manifest["duration"].sum() / 3600
assert 300 < _hours_tot < 600, f"Volume anormal : {_hours_tot:.0f} h (attendu ~456 h)"
print(f"{len(manifest)} lignes")
print(manifest.groupby(["split", "language"])["duration"].sum().div(3600).round(2).unstack())

# Contrôle EXHAUSTIF via listing par répertoire (18 listings, rapide — jamais 230k exists FUSE).
# La validation du Prompt 1 n'échantillonnait que 600 fichiers : des écritures Drive perdues
# en fin de session ont pu passer inaperçues.
BUDGET_H = {"train": 60.0, "valid": 10.0, "local_test": 6.0}

def _drive_listdir_retry(directory, max_attempts=12, initial_delay=2.0):
    """Listing Google Drive tolérant les erreurs FUSE temporaires (Errno 5)."""
    delay, last_error = initial_delay, None
    for attempt in range(1, max_attempts + 1):
        try:
            if not os.path.isdir(directory):
                return set()
            with os.scandir(directory) as entries:
                return {entry.name for entry in entries}
        except OSError as exc:
            last_error = exc
            print(f"Drive FUSE : {directory} — essai {attempt}/{max_attempts}, "
                  f"reprise dans {delay:.1f}s")
            time.sleep(delay)
            delay = min(delay * 1.7, 30.0)
    raise RuntimeError(f"Dossier Drive illisible : {directory}") from last_error
_missing_frames = []
for (_s, _l), grp in manifest.groupby(["split", "language"]):
    _d = os.path.dirname(grp["curated_audio_path"].iloc[0])
    _have = _drive_listdir_retry(_d)
    _miss = grp[~grp["curated_audio_path"].map(os.path.basename).isin(_have)]
    if len(_miss):
        _missing_frames.append(_miss)
    _kept_h = (grp["duration"].sum() - _miss["duration"].sum()) / 3600
    print(f"  {_s}/{_l}: {len(_miss)}/{len(grp)} manquants ; restant {_kept_h:.2f}h "
          f"({_kept_h / BUDGET_H[_s]:.1%} du budget {BUDGET_H[_s]:.0f}h)")
missing_df = pd.concat(_missing_frames, ignore_index=True) if _missing_frames else manifest.iloc[0:0]
print(f"TOTAL fichiers manquants : {len(missing_df)}")
if len(missing_df):
    missing_df.to_csv(os.path.join(FT_CONFIG["report_dir"], "missing_audio_files.csv"), index=False)
    _ok_after = True
    for (_s, _l), grp in manifest.groupby(["split", "language"]):
        _m = missing_df[(missing_df["split"] == _s) & (missing_df["language"] == _l)]
        if (grp["duration"].sum() - _m["duration"].sum()) / 3600 < 0.95 * BUDGET_H[_s]:
            _ok_after = False
            print(f"⚠️ {_s}/{_l} passerait SOUS 95% du budget sans ces fichiers.")
    if not _ok_after:
        raise SystemExit("Trop de fichiers manquants : relancer la cellule 13.4 du notebook de "
                         "curation (sa reprise ne rematérialisera QUE les fichiers manquants), "
                         "puis revenir ici. Liste : reports/missing_audio_files.csv")
    manifest = manifest[~manifest["sample_id"].isin(set(missing_df["sample_id"]))].reset_index(drop=True)
    print(f"Fichiers manquants exclus du manifest (budgets ≥95% préservés) : "
          f"{len(manifest)} lignes restantes.")
else:
    print("✓ Tous les fichiers du manifest sont présents sur le Drive.")
# V3 : unicité et nouvelle preuve speaker-disjoint dans CE notebook.
assert manifest["sample_id"].notna().all(), "sample_id manquant"
assert manifest["sample_id"].is_unique, "sample_id dupliqué dans le manifest"
_speaker_col = next((c for c in ("speaker_id", "speaker", "recorder_uuid", "client_id")
                     if c in manifest.columns), None)
if _speaker_col:
    _speaker_values = manifest[_speaker_col].fillna("").astype(str)
    _known = manifest[_speaker_values.str.len().gt(0) & _speaker_values.ne("?")].copy()
    for _lang, _grp in _known.groupby("language"):
        _sets = {split: set(part[_speaker_col].astype(str))
                 for split, part in _grp.groupby("split")}
        for _a, _b in (("train", "valid"), ("train", "local_test"),
                       ("valid", "local_test")):
            if _a in _sets and _b in _sets:
                _overlap = _sets[_a] & _sets[_b]
                assert not _overlap, f"Fuite locuteur {_lang} {_a}/{_b}: {len(_overlap)}"
    print("✓ séparation speaker-disjoint réaffirmée via", _speaker_col)
else:
    print("⚠️ aucune colonne locuteur : séparation non ré-auditable dans ce fichier")


## 4 — Normalisation du texte + audit du vocabulaire CTC

In [ ]:
# 4.1 — V3 : réparation d'encodage AVANT normalisation
# QA_PATCH: KIKUYU_REPAIR_PRE_NORMALIZE
_PUNCT_RE = re.compile(r"[^\w\s'’\-]", re.UNICODE)
_WS_RE = re.compile(r"\s+")
_MOJIBAKE_MARKERS = set("ÃÂÅâðÐÑ¤¦¨©¬®¯°±²³´µ¶·¸¹º»¼½¾¿")
_KIKUYU_EXPECTED = set("ĩũĨŨ")

def _text_badness(value: str, lang: str | None = None) -> int:
    value = str(value)
    score = 50 * value.count("�")
    score += 6 * sum(ch in _MOJIBAKE_MARKERS for ch in value)
    score += 10 * sum(ord(ch) < 32 and ch not in "\n\t\r" for ch in value)
    if lang == "kik":
        score -= 2 * sum(ch in _KIKUYU_EXPECTED for ch in value)
    return score

def repair_mojibake_raw(value: str, lang: str | None = None) -> tuple[str, str]:
    """Répare uniquement si un transcodage UTF-8 réduit strictement les marqueurs suspects."""
    raw = str(value or "")
    candidates = [(raw, "none")]
    if any(ch in _MOJIBAKE_MARKERS for ch in raw) or "�" in raw:
        for source_encoding in ("latin-1", "cp1252"):
            try:
                candidate = raw.encode(source_encoding).decode("utf-8")
            except (UnicodeEncodeError, UnicodeDecodeError):
                continue
            candidates.append((candidate, f"{source_encoding}->utf8"))
    best_text, method = min(candidates, key=lambda item: _text_badness(item[0], lang))
    if _text_badness(best_text, lang) >= _text_badness(raw, lang):
        return raw, "none"
    return best_text, method

def normalize_text(value: str, lang: str | None = None) -> str:
    repaired, _ = repair_mojibake_raw(value, lang)
    repaired = unicodedata.normalize("NFC", repaired).lower()
    repaired = _PUNCT_RE.sub(" ", repaired).replace("_", " ")
    return _WS_RE.sub(" ", repaired).strip()

_raw_text = manifest["text"].fillna("").astype(str)
_repairs = [repair_mojibake_raw(text, lang)
            for text, lang in zip(_raw_text, manifest["language"].astype(str))]
manifest["text_norm"] = [normalize_text(text, lang)
                         for text, lang in zip(_raw_text, manifest["language"].astype(str))]
_repair_methods = pd.Series([method for _, method in _repairs]).value_counts().to_dict()
_empty = int(manifest["text_norm"].str.len().eq(0).sum())
manifest = manifest[manifest["text_norm"].str.len() > 0].reset_index(drop=True)
print("Réparations d'encodage :", _repair_methods, "| textes vides exclus :", _empty)

_dig = manifest["text_norm"].str.contains(r"\d")
print("Lignes avec chiffres par split :", manifest[_dig].groupby("split").size().to_dict() or 0)
_drop_ids = []
for _lang, _grp in manifest[manifest["split"] == "train"].groupby("language"):
    _dg = _grp[_grp["text_norm"].str.contains(r"\d")]
    _kept_h = (_grp["duration"].sum() - _dg["duration"].sum()) / 3600
    if len(_dg) and _kept_h >= 0.95 * BUDGET_H["train"]:
        _drop_ids.extend(_dg["sample_id"].tolist())
        print(f"  {_lang}: {len(_dg)} lignes chiffrées exclues ; reste {_kept_h:.2f} h")
    elif len(_dg):
        print(f"  ⚠️ {_lang}: {len(_dg)} lignes chiffrées conservées (plancher 95 %)")
if _drop_ids:
    manifest = manifest[~manifest["sample_id"].isin(set(_drop_ids))].reset_index(drop=True)

save_json({
    "normalization_version": NORMALIZATION_VERSION,
    "repair_methods": _repair_methods,
    "empty_removed": _empty,
    "digit_rows_removed_from_train": len(_drop_ids),
}, os.path.join(FT_CONFIG["report_dir"], "normalization_audit_v3.json"))


In [ ]:
# 4.2 — Audit : caractères du corpus vs vocabulaire CTC (via l'ENCODEUR officiel)
import glob
import omnilingual_asr
_pkg_dir = os.path.dirname(omnilingual_asr.__file__)
_cards = sorted(os.path.basename(c) for c in
                glob.glob(os.path.join(_pkg_dir, "cards", "**", "*.yaml"), recursive=True))
print("omnilingual_asr", getattr(omnilingual_asr, "__version__", "?"), "| cartes packagées :", _cards)
assert any("v2" in c for c in _cards), \
    "Aucune carte v2 dans le package installé : relancer 1.2 (omnilingual-asr==0.2.0) + redémarrer."
from fairseq2.data.tokenizers.hub import load_tokenizer
_tok = None
for _cand in (FT_CONFIG["tokenizer_name"], FT_CONFIG["model_card"]):
    try:
        _tok = load_tokenizer(_cand)
        print("Tokenizer chargé via la carte :", _cand,
              "| taille vocab :", getattr(getattr(_tok, "vocab_info", None), "size", "?"))
        break
    except Exception as exc:
        print(f"  {_cand} -> {type(exc).__name__}")
if _tok is None:
    raise SystemExit("Cartes v2 présentes mais tokenizer non résolu : l'extension fairseq2 du "
                     "package ne se charge pas — vérifier les versions fairseq2/omnilingual-asr "
                     "(1.2) et me montrer cette sortie.") from None

try:
    _enc = _tok.create_encoder()
except TypeError as exc:
    import inspect as _insp
    print("Signature :", _insp.signature(_tok.create_encoder))
    raise SystemExit(f"create_encoder exige des arguments ({exc!r}) : adapter d'après la "
                     "signature affichée, puis relancer.") from None
_unk = getattr(_tok.vocab_info, "unk_idx", None)

def _char_is_oov(c):
    try:
        ids = _enc(c)
        ids = ids.tolist() if hasattr(ids, "tolist") else list(ids)
    except Exception:
        return True
    return (not ids) or ((_unk is not None) and (_unk in ids))

_oov_report = {}
for lang, grp in manifest.groupby("language"):
    chars = sorted(set("".join(grp["text_norm"])) - {" "})
    oov = [c for c in chars if _char_is_oov(c)]
    _oov_report[lang] = oov
    print(f"  {lang}: {len(chars)} caractères distincts, {len(oov)} hors vocabulaire {oov[:15]}")
save_json(_oov_report, os.path.join(FT_CONFIG["report_dir"], "vocab_oov_report.json"))
if sum(len(v) for v in _oov_report.values()):
    print("⚠️ OOV détectés : compléter OOV_CHAR_MAP en 4.3 avant conversion.")
else:
    print("✓ Toutes les langues couvertes par le vocabulaire CTC.")

In [ ]:
# 4.3 — Mapping des caractères OOV (compléter selon vocab_oov_report.json ; no-op sinon)
OOV_CHAR_MAP = {}   # ex. {"ŋ": "ng'"}
if OOV_CHAR_MAP:
    _tr = str.maketrans(OOV_CHAR_MAP)
    manifest["text_norm"] = manifest["text_norm"].map(lambda t: t.translate(_tr))
    print("Mapping OOV appliqué :", OOV_CHAR_MAP)
else:
    print("Aucun mapping OOV appliqué.")


## 5 — Conversion au format parquet fairseq2 (MIXTURE_PARQUET)
`version=0/corpus=afrivoices/split={train,dev}/language=xxx_Latn/part-*.parquet` — colonnes
`text`, `audio_bytes` (FLAC brut), `audio_size` (échantillons exacts), `corpus`, `split`, `language`.

In [ ]:
# 5.1 — DÉCOUVERTE : format exact du TSV `dataset_summary_path` (non documenté publiquement).
# On lit le code du dépôt qui le PARSE — puis on valide en 5.2 (verrou TSV_FORMAT_CONFIRMED).
_hits = subprocess.run(["grep", "-rn", "--include=*.py", "-e", "dataset_summary_path",
                        "-e", "language_distribution",
                        os.path.join(OMNI_REPO_DIR, "src"), os.path.join(OMNI_REPO_DIR, "workflows")],
                       capture_output=True, text=True).stdout
print(_hits[:4000] if _hits else "Aucune occurrence : chercher à la main dans le dépôt.")
print("\n-> Ouvrir le(s) fichier(s) ci-dessus, noter les COLONNES attendues du TSV, ajuster")
print("   build_parquet en 5.2 si besoin, puis mettre TSV_FORMAT_CONFIRMED = True.")


In [ ]:
# 5.2 — Conversion manifests -> parquet + TSV (verrouillée tant que le format TSV n'est pas confirmé)
import pyarrow as pa
import pyarrow.parquet as pq

# Format VÉRIFIÉ dans le code source du dépôt (mixture_parquet_storage.py, l.339-351 :
# pd.read_csv(sep="\t") puis colonnes utilisées = corpus, language, hours ; et le générateur
# officiel hf_dataset_ingestion_example.py écrit exactement group_by(corpus, language)
# .agg(hours = sum(audio_size)/3600/16000)). Poids d'échantillonnage ∝ hours^beta.
TSV_FORMAT_CONFIRMED = True

PARQUET_ROOT = os.path.join(FT_CONFIG["parquet_root"], "version=0")
CORPUS = "afrivoices"
SPLIT_MAP = {"train": "train", "valid": "dev"}

def _prep_hash():
    import hashlib
    _cols = [c for c in ("sample_id", "split", "language", "curated_audio_path",
                          "duration", "text_norm") if c in manifest.columns]
    _canon = (manifest[_cols].fillna("").astype(str)
              .sort_values(_cols[:3], kind="mergesort").reset_index(drop=True))
    _row_bytes = pd.util.hash_pandas_object(_canon, index=False).values.tobytes()
    _settings = json.dumps({
        "schema": CACHE_SCHEMA_VERSION,
        "normalization": NORMALIZATION_VERSION,
        "tokenizer": FT_CONFIG["tokenizer_name"],
        "oov_map": globals().get("OOV_CHAR_MAP", {}),
    }, sort_keys=True, ensure_ascii=False).encode("utf-8")
    return hashlib.sha256(_row_bytes + _settings).hexdigest()[:20]
def dataset_meta(root=None):
    _p = os.path.join(os.path.dirname(root or PARQUET_ROOT + "/"), "dataset_meta.json")
    _p = os.path.join(FT_CONFIG["parquet_root"], "dataset_meta.json") if root is None else _p
    return json.load(open(_p)) if os.path.exists(_p) else None

def build_parquet(df, max_rows_per_file=2000, subset_hours_per_lang=None, root=PARQUET_ROOT):
    if not TSV_FORMAT_CONFIRMED:
        raise SystemExit("Format du TSV non confirmé : faire 5.1, ajuster si besoin, "
                         "puis TSV_FORMAT_CONFIRMED = True (même verrou que les autres découvertes).")
    t0 = time.time(); n_files = 0; summary_rows = []
    for (split, lang), grp in df[df["split"].isin(SPLIT_MAP)].groupby(["split", "language"]):
        if subset_hours_per_lang:
            grp = grp.sample(frac=1.0, random_state=FT_CONFIG["seed"])
            keep, acc = [], 0.0
            for _, r in grp.iterrows():
                if acc >= subset_hours_per_lang * 3600: break
                keep.append(r); acc += float(r["duration"])
            grp = pd.DataFrame(keep)
        out_dir = os.path.join(root, f"corpus={CORPUS}", f"split={SPLIT_MAP[split]}",
                               f"language={omni_lang(lang)}")
        os.makedirs(out_dir, exist_ok=True)
        from concurrent.futures import ThreadPoolExecutor
        def _read_bytes(pth):
            with open(pth, "rb") as fh:
                return fh.read()
        with ThreadPoolExecutor(max_workers=32) as _pool:   # FUSE : latence/fichier -> paralléliser
            _blobs = list(_pool.map(_read_bytes, grp["curated_audio_path"].tolist()))
        rows, part, total_sz = [], 0, 0
        for (_, r), blob in zip(grp.iterrows(), _blobs):
            frames = int(sf.info(io.BytesIO(blob)).frames)   # exact, zéro I/O supplémentaire
            total_sz += frames
            # corpus/split/language UNIQUEMENT dans les chemins hive (le lecteur officiel les
            # infère du partitionnement ; en colonne aussi -> ArrowTypeError: Unable to merge)
            rows.append({"text": r["text_norm"], "audio_bytes": blob, "audio_size": frames})
            if len(rows) >= max_rows_per_file:
                pq.write_table(pa.Table.from_pylist(rows),
                               os.path.join(out_dir, f"part-{part:04d}.parquet"), row_group_size=100)
                part += 1; n_files += 1; rows = []
        if rows:
            pq.write_table(pa.Table.from_pylist(rows),
                           os.path.join(out_dir, f"part-{part:04d}.parquet"), row_group_size=100)
            n_files += 1
        summary_rows.append({"corpus": CORPUS, "split": SPLIT_MAP[split],
                             "language": omni_lang(lang), "num_examples": int(len(grp)),
                             "total_audio_size": int(total_sz)})
        print(f"  {split}/{lang} -> {omni_lang(lang)} : {len(grp)} clips")
    # TSV officiel : EXACTEMENT 3 colonnes (corpus, language, hours), agrégées SANS split
    _summary = pd.DataFrame(summary_rows)
    _tsv_df = (_summary.groupby(["corpus", "language"], as_index=False)["total_audio_size"].sum()
               .assign(hours=lambda d: d["total_audio_size"] / 16000 / 3600)
               [["corpus", "language", "hours"]])
    tsv_path = os.path.join(os.path.dirname(root), "language_distribution_0.tsv")
    _tsv_df.to_csv(tsv_path, sep="\t", index=False)
    print(_tsv_df.round(2).to_string(index=False))
    # méta : QUEL dataset est présent (empêche d'entraîner le run complet sur un sous-ensemble
    # laissé par la calibration/le FAST — même chemin de TSV) + hash de préparation (invalidation)
    save_json({"subset_hours_per_lang": subset_hours_per_lang, "prep_hash": _prep_hash(),
               "rows": int(sum(r["num_examples"] for r in summary_rows)), "timestamp": now_iso()},
              os.path.join(os.path.dirname(root), "dataset_meta.json"))
    print(f"{n_files} parquet en {time.time()-t0:.0f}s ; résumé -> {tsv_path}")
    return tsv_path

def restore_or_build_full_parquet():
    """Parquet COMPLET : restauré depuis le cache Drive si présent (rapide), sinon construit
    depuis les FLAC (lent : ~230k petits fichiers via FUSE) puis mis en cache."""
    cache = FT_CONFIG["parquet_drive_cache"]
    local = FT_CONFIG["parquet_root"]
    tsv_local = os.path.join(local, "language_distribution_0.tsv")
    _marker = os.path.join(cache, "_COMPLETE")          # écrit EN DERNIER : cache partiel = invalide
    _meta_c = os.path.join(cache, "dataset_meta.json")
    if FT_CONFIG["use_parquet_drive_cache"] and os.path.exists(_marker) and os.path.exists(_meta_c):
        _m = json.load(open(_meta_c))
        if _m.get("prep_hash") == _prep_hash() and _m.get("subset_hours_per_lang") is None:
            print("Cache parquet Drive valide : copie locale (1 flux, >> plus rapide que 230k FLAC)…")
            shutil.rmtree(local, ignore_errors=True)
            subprocess.run(["cp", "-r", cache, local], check=True)
            return tsv_local
        print(f"Cache présent mais périmé (hash {_m.get('prep_hash')} != {_prep_hash()} "
              "ou sous-ensemble) : reconstruction.")
    shutil.rmtree(local, ignore_errors=True)
    tsv = build_parquet(manifest)
    if FT_CONFIG["use_parquet_drive_cache"]:
        print("Mise en cache du parquet sur Drive (~26 GB — vérifier le quota)…")
        shutil.rmtree(cache, ignore_errors=True)
        subprocess.run(["cp", "-r", local, cache], check=True)
        open(_marker, "w").write(now_iso())            # marqueur de complétude, en tout dernier
    return tsv

print("Conversion prête (exécution réelle dans les sections 6/7/8).")


## 6 — Calibration VRAM/débit (obligatoire avant tout entraînement)

In [ ]:
# 6.1 — Calibration : mini-dataset (0.25 h/langue), 30 steps, échelle grad-accum
if not RUN_CALIBRATION:
    raise SystemExit("Calibration bloquée. Mettre RUN_CALIBRATION=True en 2.2, relancer 2.2, puis ici.")
_meta = dataset_meta()
if _meta and _meta.get("subset_hours_per_lang") == 0.25 and _meta.get("prep_hash") == _prep_hash():
    _tsv = os.path.join(FT_CONFIG["parquet_root"], "language_distribution_0.tsv")
    print("Mini-parquet de calibration déjà présent et valide : réutilisé.")
else:
    shutil.rmtree(FT_CONFIG["parquet_root"], ignore_errors=True)
    _tsv = build_parquet(manifest, subset_hours_per_lang=0.25)

_ladder = [
    {"grad_accum": 8,  "max_num_elements": FT_CONFIG["max_num_elements"]},
    {"grad_accum": 16, "max_num_elements": FT_CONFIG["max_num_elements"] // 2},
    {"grad_accum": 32, "max_num_elements": FT_CONFIG["max_num_elements"] // 4},
]
calibration = {"timestamp": now_iso(), "gpu": _gpu.name, "attempts": [], "chosen": None,
               "model_card": FT_CONFIG["model_card"]}
for att in _ladder:
    _out = f"/content/calib_{att['grad_accum']}"
    shutil.rmtree(_out, ignore_errors=True)
    _yaml = write_recipe_yaml(f"/content/calib_{att['grad_accum']}.yaml", FT_CONFIG["model_card"],
                              _tsv, num_steps=30, ckpt_every=30, **att)
    rc, log_path = run_recipe(_out, _yaml, f"calib_accum{att['grad_accum']}.log")
    _tail = open(log_path, encoding="utf-8").read()[-5000:]
    _oom = ("CUDA out of memory" in _tail) or ("OutOfMemoryError" in _tail)
    calibration["attempts"].append({**att, "returncode": rc, "oom": bool(_oom)})
    print("Essai :", calibration["attempts"][-1])
    if rc == 0 and not _oom:
        calibration["chosen"] = att
        break
    if rc != 0 and not _oom:
        # erreur de config/recette : descendre l'échelle d'accumulation n'y changera RIEN
        save_json(calibration, os.path.join(FT_CONFIG["report_dir"], "calibration_report.json"))
        raise SystemExit(f"Échec de RECETTE (pas un OOM) à grad_accum={att['grad_accum']} : "
                         f"lire {log_path} (dernières lignes ci-dessus) et me les montrer. "
                         "Ne PAS conclure sur la VRAM.") from None
save_json(calibration, os.path.join(FT_CONFIG["report_dir"], "calibration_report.json"))
if calibration["chosen"] is None:
    # Aucun réglage 1B ne tient : repli officiel = checkpoint plus petit. chosen reste None
    # tant que le 300M n'a pas été RÉELLEMENT calibré (pas de fausse validation).
    raise SystemExit(f"Le {FT_CONFIG['model_card']} ne tient pas en VRAM. "
                     f"Mettre FT_CONFIG['model_card']='{FT_CONFIG['fallback_model_card']}' en 2.1 "
                     "(défaut) et RELANCER 2.1 puis cette cellule pour calibrer le 300M.")
FT_CONFIG["grad_accum"] = calibration["chosen"]["grad_accum"]
FT_CONFIG["max_num_elements"] = calibration["chosen"]["max_num_elements"]
save_json(FT_CONFIG, os.path.join(FT_CONFIG["report_dir"], "experiment_run"))
print("CHOIX :", FT_CONFIG["model_card"], calibration["chosen"])


## 7 — FAST run : valider TOUTE la boucle (train → checkpoint → reprise → inférence → CER)

In [ ]:
# 7.1 — FAST training (2 h/langue, ~400 steps)
if not RUN_FAST_TRAINING:
    raise SystemExit("FAST bloqué. RUN_FAST_TRAINING=True en 2.2 puis relancer 2.2 et ici.")
shutil.rmtree(FT_CONFIG["parquet_root"], ignore_errors=True)
_fast_tsv = build_parquet(manifest, subset_hours_per_lang=FT_CONFIG["fast_hours_per_language"])
_fast_yaml = write_recipe_yaml(os.path.join(FT_CONFIG["experiment_run"], "fast_recipe.yaml"),
                               FT_CONFIG["model_card"], _fast_tsv,
                               num_steps=FT_CONFIG["fast_num_steps"], ckpt_every=100,
                               grad_accum=FT_CONFIG["grad_accum"],
                               max_num_elements=FT_CONFIG["max_num_elements"])
rc, _log = run_recipe(FT_CONFIG["fast_output_dir"], _fast_yaml, "fast_training.log",
                      prune_keep=FT_CONFIG["checkpoints_keep"])
assert rc == 0, f"FAST training en échec (code {rc}) — lire {_log} AVANT toute suite."
prune_checkpoints(FT_CONFIG["fast_output_dir"], keep=FT_CONFIG["checkpoints_keep"])
print("Dernier checkpoint FAST :", latest_ckpt(FT_CONFIG["fast_output_dir"]))


In [ ]:
# 7.2 — Test de REPRISE : relancer la même commande doit repartir du checkpoint (pas de 0)
if not RUN_FAST_TRAINING:
    raise SystemExit("FAST bloqué.")
_fast_tsv2 = os.path.join(FT_CONFIG["parquet_root"], "language_distribution_0.tsv")
_meta = dataset_meta()
assert os.path.exists(_fast_tsv2) and _meta, "Parquet absent : exécuter 7.1 d'abord dans cette session."
assert _meta["subset_hours_per_lang"] == FT_CONFIG["fast_hours_per_language"], \
    f"Le parquet présent n'est pas le sous-ensemble FAST ({_meta}) : relancer 7.1."
_fast_yaml = write_recipe_yaml(os.path.join(FT_CONFIG["experiment_run"], "fast_recipe.yaml"),
                               FT_CONFIG["model_card"], _fast_tsv2,
                               num_steps=FT_CONFIG["fast_num_steps"], ckpt_every=100,
                               grad_accum=FT_CONFIG["grad_accum"],
                               max_num_elements=FT_CONFIG["max_num_elements"])   # idempotent
rc2, _log2 = run_recipe(FT_CONFIG["fast_output_dir"], _fast_yaml, "fast_training_resume.log")
print("Code retour reprise :", rc2)
print("⚠️ VÉRIFIER dans le log que la reprise part du step sauvegardé "
      "(chercher 'restor'/'resum'/numéro de step) — garantie anti-24h du run complet.")


In [ ]:
# 7.3 — Inférence du checkpoint FAST + CER par langue (mini-échantillon valid)
if not RUN_FAST_TRAINING:
    raise SystemExit("FAST bloqué.")
import jiwer

_experiment_run = latest_ckpt(FT_CONFIG["fast_output_dir"])
assert _experiment_run is not None, ("Aucun checkpoint 'model*step_*.pt' sous fast_output_dir — "
                              "inspecter la structure réelle (ls -R) et adapter latest_ckpt.")
print("Checkpoint :", _experiment_run)
pipe_ft = load_finetuned_pipeline(_experiment_run)   # voie officielle model=/tokenizer= (cf. 2.4)

_sample = (manifest[manifest["split"] == "valid"]
           .sample(frac=1.0, random_state=FT_CONFIG["seed"])
           .groupby("language").head(20))              # pas de groupby.apply (déprécié pandas>=2.2)
_cer = {}
for lang, grp in _sample.groupby("language"):
    files, refs = grp["curated_audio_path"].tolist(), grp["text_norm"].tolist()
    hyps = [normalize_text(h) for h in
            pipe_ft.transcribe(files, lang=[omni_lang(lang)] * len(files), batch_size=8)]
    _cer[lang] = round(jiwer.cer(refs, hyps), 4)
    save_json({"timestamp": now_iso(), "cer_fast": _cer},   # sauvegarde INCRÉMENTALE par langue
              os.path.join(FT_CONFIG["report_dir"], "fast_eval_report.json"))
    print(f"  {lang}: CER={_cer[lang]} | réf «{refs[0][:60]}» / hyp «{hyps[0][:60]}»")
print("GO section 8 si : reprise OK (7.2) + CER plausibles (pas de charabia).")


## 8 — Entraînement complet (multi-sessions ; relancer 8.1→8.2 à chaque session)

In [ ]:
# 8.1 — Restaurer (cache Drive) ou construire le parquet complet
if not RUN_FULL_TRAINING:
    raise SystemExit("Run complet bloqué. RUN_FULL_TRAINING=True en 2.2 puis relancer 2.2 et ici.")
_full_tsv = restore_or_build_full_parquet()


In [ ]:
# 8.2 — Lancer / reprendre (autonome : régénère son YAML, indépendant de 8.1 pour les variables)
if not RUN_FULL_TRAINING:
    raise SystemExit("Run complet bloqué.")
_full_tsv = os.path.join(FT_CONFIG["parquet_root"], "language_distribution_0.tsv")
_meta = dataset_meta()
assert os.path.exists(_full_tsv) and _meta, "Parquet absent : exécuter 8.1 d'abord dans cette session."
assert _meta["subset_hours_per_lang"] is None, \
    (f"Le parquet présent est un SOUS-ENSEMBLE ({_meta['subset_hours_per_lang']} h/langue, laissé "
     "par la calibration ou le FAST) : exécuter 8.1 pour restaurer/construire le dataset complet.")
_full_yaml = write_recipe_yaml(os.path.join(FT_CONFIG["experiment_run"], "full_recipe.yaml"),
                               FT_CONFIG["model_card"], _full_tsv,
                               num_steps=FT_CONFIG["num_steps"],
                               grad_accum=FT_CONFIG["grad_accum"],
                               max_num_elements=FT_CONFIG["max_num_elements"])
rc, _log = run_recipe(FT_CONFIG["output_dir"], _full_yaml, "full_training.log",
                      wall_limit_h=FT_CONFIG["max_wall_hours_per_session"],
                      prune_keep=FT_CONFIG["checkpoints_keep"])
prune_checkpoints(FT_CONFIG["output_dir"], keep=FT_CONFIG["checkpoints_keep"])
print("Code :", rc, "| dernier checkpoint :", latest_ckpt(FT_CONFIG["output_dir"]))
print("Si coupé par la limite murale : nouvelle session -> 1→5 -> RUN_FULL_TRAINING=True -> 8.1 -> 8.2.")


## 9 — Évaluation finale : zéro-shot vs fine-tuné sur local_test

In [ ]:
# 9.1 — CER/WER par langue, zéro-shot vs fine-tuné — REPRENABLE (sessions Colab fragiles) :
# les couples (modèle, langue) déjà présents dans final_eval_report.json sont sautés, et un
# modèle dont la passe est complète n'est même pas chargé.
import jiwer

EVAL_CLIPS_PER_LANG = 250   # None = tout local_test (plus long)
_eval = (manifest[manifest["split"] == "local_test"]
         .sample(frac=1.0, random_state=FT_CONFIG["seed"])
         .groupby("language").head(EVAL_CLIPS_PER_LANG or 10**9))

_experiment_run = latest_ckpt(FT_CONFIG["output_dir"])
assert _experiment_run is not None, "Aucun checkpoint du run complet : section 8 d'abord."
print("Checkpoint évalué :", _experiment_run)

_report_path = os.path.join(FT_CONFIG["report_dir"], "final_eval_report.json")
results = {}
if os.path.exists(_report_path):
    try:
        results = json.load(open(_report_path)).get("results", {})
        print("Reprise éval — déjà calculé :", {k: sorted(v) for k, v in results.items()})
    except Exception as _exc:
        print("Rapport précédent illisible :", repr(_exc))

_pipe_factories = {"zeroshot": lambda: load_finetuned_pipeline(None),
                   "finetuned": lambda: load_finetuned_pipeline(_experiment_run)}
for name, factory in _pipe_factories.items():
    results.setdefault(name, {})
    _todo = [l for l in LANGUAGES if l not in results[name]]
    if not _todo:
        print(f"[{name}] déjà complet, modèle non chargé.")
        continue
    pipe = factory()
    for lang, grp in _eval.groupby("language"):
        if lang not in _todo:
            continue
        files, refs = grp["curated_audio_path"].tolist(), grp["text_norm"].tolist()
        hyps = [normalize_text(h) for h in
                pipe.transcribe(files, lang=[omni_lang(lang)] * len(files), batch_size=16)]
        results[name][lang] = {"cer": round(jiwer.cer(refs, hyps), 4),
                               "wer": round(jiwer.wer(refs, hyps), 4), "n": len(refs)}
        save_json({"timestamp": now_iso(), "results": results}, _report_path)  # incrémental
        print(f"[{name}] {lang}: CER={results[name][lang]['cer']} WER={results[name][lang]['wer']}")
    del pipe
    torch.cuda.empty_cache()

_tbl = pd.DataFrame({(n, m): {l: results[n][l][m] for l in results[n]}
                     for n in results for m in ("cer", "wer")})
print(_tbl)
_reg = [l for l in results.get("finetuned", {})
        if results["finetuned"][l]["cer"] > results["zeroshot"][l]["cer"]]
print(("⚠️ Sans amélioration : " + str(_reg) + " -> envisager le top-up 9.2 (surtout mas).")
      if _reg else "✓ Amélioration sur toutes les langues : GO soumission.")
# QA_PATCH: WORD_WEIGHTED_EVAL
# V3 : métrique Kaggle exacte = moyenne non pondérée des corpus-WER linguistiques.
def language_averaged_corpus_wer(per_language: dict) -> float:
    values = [float(per_language[lang]["wer"]) for lang in LANGUAGES]
    if len(values) != 6:
        raise ValueError(f"Six langues requises, reçu {len(values)}")
    return float(np.mean(values))

for _model_name in ("zeroshot", "finetuned"):
    if all(lang in results.get(_model_name, {}) for lang in LANGUAGES):
        _macro = language_averaged_corpus_wer(results[_model_name])
        print(f"{_model_name} | language_averaged_corpus_WER = {_macro:.5f}")


## 10 — Test Kaggle → `submission.csv`

In [ ]:
# 10.2 — Inventaire du dataset test + CACHE DRIVE (structure {lang}/{Scripted,Unscripted}/*.parquet)
# Dossiers = codes de soumission (swa, kik, kln, luo, som, mas). Le cache Drive évite de
# re-télécharger les 37,5 GB à chaque nouvelle session (kagglehub met en cache sur la VM,
# effacée au redémarrage).
if not RUN_TEST_INFERENCE:
    raise SystemExit("RUN_TEST_INFERENCE=True requis (2.2).")
import glob, subprocess
import pyarrow.parquet as pq

USE_TEST_DRIVE_CACHE = True
TEST_DRIVE = os.path.join(DRIVE_ROOT, "kaggle_test_full")
_marker = os.path.join(TEST_DRIVE, "_COMPLETE")

if USE_TEST_DRIVE_CACHE and os.path.exists(_marker):
    TEST_ROOT = TEST_DRIVE
    print("Dataset test repris du cache Drive (aucun téléchargement) :", TEST_ROOT)
else:
    subprocess.run(["pip", "install", "-q", "kagglehub"], check=False)
    import kagglehub
    _dl = kagglehub.dataset_download("digitalumuganda/anv-test-data-nt")
    print("Téléchargé via kagglehub :", _dl)
    if USE_TEST_DRIVE_CACHE:
        print("Copie structurée sur Drive (~37,5 GB, une seule fois — patiente)…")
        _tmp = TEST_DRIVE + ".tmp"
        subprocess.run(["rm", "-rf", _tmp], check=False)
        os.makedirs(_tmp, exist_ok=True)
        subprocess.run(["cp", "-r"] + glob.glob(os.path.join(_dl, "*")) + [_tmp], check=True)
        subprocess.run(["rm", "-rf", TEST_DRIVE], check=False)
        subprocess.run(["mv", _tmp, TEST_DRIVE], check=True)
        open(_marker, "w").write(now_iso())      # marqueur écrit EN DERNIER : cache partiel = invalide
        TEST_ROOT = TEST_DRIVE
        print("Cache Drive prêt :", TEST_ROOT)
    else:
        TEST_ROOT = _dl

EXPECTED_TEST_LANGS = {"swa", "kik", "kln", "luo", "som", "mas"}
# code de dossier (= code de soumission) -> code interne omnilingual pour l'inférence
TEST_DIR_TO_OMNI = {"swa": "swh_Latn", "kik": "kik_Latn", "kln": "kln_Latn",
                    "luo": "luo_Latn", "som": "som_Latn",
                    "mas": (LANG_FALLBACK_OMNI["mas"] if _use_mas_fallback else "mas_Latn")}
_found = {d for d in os.listdir(TEST_ROOT) if os.path.isdir(os.path.join(TEST_ROOT, d))}
assert EXPECTED_TEST_LANGS <= _found, f"Langues manquantes : {EXPECTED_TEST_LANGS - _found} (trouvé {_found})"

test_shards, TEST_TOTAL_ROWS = [], 0
for _lang in sorted(EXPECTED_TEST_LANGS):
    _paths = sorted(glob.glob(os.path.join(TEST_ROOT, _lang, "**", "*.parquet"), recursive=True))
    _rows = 0
    for _p in _paths:
        _n = pq.ParquetFile(_p).metadata.num_rows
        test_shards.append((_lang, _p, _n))
        _rows += _n
    TEST_TOTAL_ROWS += _rows
    print(f"  {_lang}: {len(_paths)} shard(s), {_rows} clips")
print(f"TOTAL : {len(test_shards)} shards, {TEST_TOTAL_ROWS} clips à transcrire")
save_json({"timestamp": now_iso(), "root": TEST_ROOT, "total_rows": TEST_TOTAL_ROWS,
           "shards": [{"lang": l, "path": p, "rows": n} for l, p, n in test_shards]},
          os.path.join(FT_CONFIG["report_dir"], "test_inventory.json"))

In [ ]:
# 10.3 — Transcription du test (REPRENABLE par shard) + submission.csv
if not RUN_TEST_INFERENCE:
    raise SystemExit("Inférence test bloquée.")
assert "test_shards" in globals(), "Inventaire absent : exécuter 10.2 d'abord."
import pyarrow.parquet as pq

_experiment_run = latest_ckpt(FT_CONFIG["output_dir"])
assert _experiment_run is not None, "Aucun checkpoint du run complet."
pipe = load_finetuned_pipeline(_experiment_run)

_pred_dir = os.path.join(FT_CONFIG["report_dir"], "test_predictions")
os.makedirs(_pred_dir, exist_ok=True)
_MAX_S = 38.0          # limite d'inférence CTC : < 40 s -> découpage des clips plus longs
_BATCH = 16

_B64_CHARS = set(b"ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789+/=\n\r")
def _ffmpeg_decode(raw, sr_out=16000):
    import subprocess
    try:
        proc = subprocess.run(["ffmpeg", "-v", "quiet", "-nostdin", "-i", "pipe:0",
                               "-f", "f32le", "-ac", "1", "-ar", str(sr_out), "pipe:1"],
                              input=raw, capture_output=True, check=True)
        wav = np.frombuffer(proc.stdout, dtype=np.float32).copy()
        return (wav, sr_out) if len(wav) else (None, None)
    except Exception:
        return None, None

def _decode_audio(cell):
    """Décodage TOLÉRANT : dict{bytes,path} / bytes / numpy int8-uint8 / memoryview ;
    soundfile puis repli ffmpeg ; (None, None) si le clip est réellement indécodable
    (le test contient probablement ~1% de clips corrompus, comme le train)."""
    raw = cell.get("bytes") if isinstance(cell, dict) else cell
    if raw is None:
        return None, None
    if hasattr(raw, "tobytes"):
        raw = raw.tobytes()
    elif isinstance(raw, (memoryview, bytearray)):
        raw = bytes(raw)
    # Certains clips stockent l'audio en BASE64 (texte) au lieu d'octets bruts
    # (ex: som/test_scripted_000 : 257 clips, header 'UklGR...' = base64 de 'RIFF').
    if raw[:5] == b"UklGR" or (len(raw) > 32 and set(raw[:80]) <= _B64_CHARS):
        try:
            import base64 as _b64
            _dec = _b64.b64decode(raw, validate=False)
            if _dec[:4] in (b"RIFF", b"fLaC", b"OggS") or _dec[:3] == b"ID3":
                raw = _dec
        except Exception:
            pass
    try:
        wav, sr = sf.read(io.BytesIO(raw), dtype="float32", always_2d=False)
    except Exception:
        return _ffmpeg_decode(raw)
    if getattr(wav, "ndim", 1) > 1:
        wav = wav.mean(axis=1)
    return wav, sr

_printed_type = False

for _lang, _shard, _nrows in test_shards:
    _out_csv = os.path.join(_pred_dir, f"{_lang}__{os.path.basename(os.path.dirname(_shard))}__"
                                       f"{os.path.basename(_shard)}.csv")
    if os.path.exists(_out_csv):
        continue   # reprise : shard déjà transcrit
    _t0 = time.time()
    _tbl = pq.read_table(_shard, columns=["id", "audio"]).to_pandas()
    # découpage éventuel des clips >= 40 s en segments <= 38 s (rejoints par espace)
    if not _printed_type:
        _c0 = _tbl.iloc[0]["audio"]
        print("type de la colonne audio :", type(_c0).__name__,
              "| clés :" if isinstance(_c0, dict) else "",
              list(_c0.keys()) if isinstance(_c0, dict) else "")
        _printed_type = True
    _items, _owner, _bad_rows = [], [], []
    for _ri, _row in _tbl.iterrows():
        wav, sr = _decode_audio(_row["audio"])
        if wav is None:
            _bad_rows.append(_ri)          # indécodable -> placeholder à l'assemblage
            continue
        if len(wav) < int(0.2 * sr):        # clip quasi vide : segment minimal (évite un rejet)
            wav = np.pad(wav, (0, int(0.2 * sr) - len(wav)))
        _limit = int(39.5 * sr)
        if len(wav) <= _limit:
            _items.append({"waveform": wav, "sample_rate": sr}); _owner.append(_ri)
        else:
            _step = int(_MAX_S * sr)
            for _off in range(0, len(wav), _step):
                _seg = wav[_off:_off + _step]
                if len(_seg) < int(0.2 * sr):
                    continue
                _items.append({"waveform": _seg, "sample_rate": sr}); _owner.append(_ri)
    if len(_bad_rows) > 0.5 * len(_tbl):
        _c0 = _tbl.iloc[_bad_rows[0]]["audio"]
        raise SystemExit(f"{len(_bad_rows)}/{len(_tbl)} clips indécodables dans ce shard : "
                         f"format inattendu (type={type(_c0).__name__}) — me montrer cette sortie.")
    _hyps = pipe.transcribe(_items, lang=[TEST_DIR_TO_OMNI[_lang]] * len(_items),
                            batch_size=_BATCH)
    _per_row = {}
    for _o, _h in zip(_owner, _hyps):
        _per_row.setdefault(_o, []).append(str(_h))
    _out = pd.DataFrame({"id": _tbl["id"],
                         "language": _lang,
                         "text": [normalize_text(" ".join(_per_row.get(_ri, [""])))
                                  for _ri in _tbl.index]})
    _out.to_csv(_out_csv, index=False)
    _done = len(glob.glob(os.path.join(_pred_dir, "*.csv")))
    print(f"[{_done}/{len(test_shards)}] {_lang}/{os.path.basename(_shard)} : {len(_tbl)} clips "
          f"({len(_bad_rows)} indécodables) en {(time.time()-_t0)/60:.1f} min")

# ---- Assemblage du submission.csv ----
_parts = [pd.read_csv(f, dtype=str) for f in sorted(glob.glob(os.path.join(_pred_dir, "*.csv")))]
submission = pd.concat(_parts, ignore_index=True)
submission["text"] = submission["text"].fillna("")
_empty = submission["text"].str.strip().eq("")
if int(_empty.sum()):
    print(f"⚠️ {int(_empty.sum())} prédiction(s) vide(s) remplacée(s) par un mot de remplissage.")
    submission.loc[_empty, "text"] = "a"
submission = submission.rename(columns={"text": SUBMISSION_TEXT_COLUMN})
submission = submission[["id", "language", SUBMISSION_TEXT_COLUMN]]
assert submission["language"].isin(set(LANG_TO_SUBMISSION.values())).all()
assert submission["id"].is_unique, "IDs dupliqués entre shards ?!"
assert len(submission) == TEST_TOTAL_ROWS, f"{len(submission)} lignes != {TEST_TOTAL_ROWS} attendues"
sub_path = os.path.join(FT_CONFIG["report_dir"], "submission.csv")
submission.to_csv(sub_path, index=False)
print(f"submission.csv : {len(submission)} lignes -> {sub_path}")
print(f"Submission assembled: {len(submission)} rows; preview omitted.")
print(f"Rappel : en-tête texte = '{SUBMISSION_TEXT_COLUMN}' (changer en 2.1 si Kaggle rejette) ; "
      "si les IDs sont refusés, variante sans extension : submission['id'].str.replace('.wav','')")

## 11 — Résumé final du run

In [ ]:
# 11.1 — Synthèse
_summary = {
    "timestamp": now_iso(),
    "run": FT_RUN_NAME,
    "model": FT_CONFIG["model_card"],
    "grad_accum": FT_CONFIG["grad_accum"],
    "max_num_elements": FT_CONFIG["max_num_elements"],
    "mas_fallback": bool(_use_mas_fallback),
    "reports": sorted(os.listdir(FT_CONFIG["report_dir"])),
    "last_checkpoint": latest_ckpt(FT_CONFIG["output_dir"]),
}
save_json(_summary, os.path.join(FT_CONFIG["report_dir"], "run_summary.json"))
for k, v in _summary.items():
    print(f"{k}: {v}")


In [ ]:
# 12.1 — Construire un KenLM par langue (sur les transcriptions du TRAIN)
if not RUN_LM_BUILD:
    raise SystemExit("RUN_LM_BUILD=False")
import subprocess, glob

KENLM_DIR = "/content/kenlm"
LM_DRIVE = os.path.join(FT_CONFIG["experiment_run"], "kenlm_models")   # persistant sur Drive
os.makedirs(LM_DRIVE, exist_ok=True)
KENLM_ORDER = 5

subprocess.run(["pip", "install", "-q", "pyctcdecode"], check=False)

# Outils KenLM (lmplz, build_binary) : compilés une seule fois sur la VM
_lmplz = os.path.join(KENLM_DIR, "build", "bin", "lmplz")
_bbin = os.path.join(KENLM_DIR, "build", "bin", "build_binary")
if not os.path.exists(_lmplz):
    subprocess.run("apt-get install -y libeigen3-dev libboost-all-dev",
                   shell=True, capture_output=True)
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/kpu/kenlm.git", KENLM_DIR], check=True)
    _r = subprocess.run(f"cd {KENLM_DIR} && mkdir -p build && cd build && cmake .. && make -j4",
                        shell=True, capture_output=True, text=True)
    if not os.path.exists(_lmplz):
        print(_r.stdout[-1500:], _r.stderr[-1500:])
        raise SystemExit("Compilation KenLM échouée (lmplz introuvable) — voir logs ci-dessus.")
print("KenLM prêt :", _lmplz)

# Un LM par langue depuis le TRAIN (déjà normalisé : text_norm)
LM_PATHS = {}
for lang in LANGUAGES:
    _bin = os.path.join(LM_DRIVE, f"{lang}.binary")
    if os.path.exists(_bin):
        LM_PATHS[lang] = _bin
        print(f"  {lang}: LM déjà présent ({os.path.getsize(_bin)//1024} Ko)")
        continue
    _sents = manifest[(manifest.split == "train") & (manifest.language == lang)]["text_norm"].tolist()
    _txt = f"/content/lm_{lang}.txt"
    with open(_txt, "w", encoding="utf-8") as f:
        f.write("\n".join(_sents))
    _arpa = f"/content/lm_{lang}.arpa"
    _order = KENLM_ORDER
    _r = subprocess.run(f"{_lmplz} -o {_order} --discount_fallback < {_txt} > {_arpa}",
                        shell=True, capture_output=True, text=True)
    if _r.returncode != 0:            # repli ordre 3 si le corpus est trop petit
        _order = 3
        _r = subprocess.run(f"{_lmplz} -o {_order} --discount_fallback < {_txt} > {_arpa}",
                            shell=True, capture_output=True, text=True)
    assert _r.returncode == 0, f"lmplz {lang} échec : {_r.stderr[-600:]}"
    subprocess.run(f"{_bbin} {_arpa} {_bin}", shell=True, check=True)
    LM_PATHS[lang] = _bin
    print(f"  {lang}: {len(_sents)} phrases -> {_order}-gramme -> {os.path.getsize(_bin)//1024} Ko")

# Vocabulaire par langue (unigrammes) : aide pyctcdecode à mieux borner les mots
LM_UNIGRAMS = {}
for lang in LANGUAGES:
    _words = set()
    for _t in manifest[(manifest.split == "train") & (manifest.language == lang)]["text_norm"]:
        _words.update(str(_t).split())
    LM_UNIGRAMS[lang] = sorted(_words)
    print(f"  {lang}: {len(LM_UNIGRAMS[lang])} mots-vocabulaire")

save_json({"lm_paths": LM_PATHS, "order": KENLM_ORDER,
           "unigram_counts": {l: len(v) for l, v in LM_UNIGRAMS.items()}},
          os.path.join(FT_CONFIG["report_dir"], "kenlm_build_report.json"))
print("LMs prêts :", {k: os.path.basename(v) for k, v in LM_PATHS.items()})


In [ ]:
# 12.1b — KenLM v3 : corpus enrichi = train + valid par langue, local_test EXCLU car c'est
if not RUN_LM_BUILD:
    raise SystemExit("RUN_LM_BUILD=False")
# l'échantillon de réglage de 12.2c — l'inclure ferait mémoriser au LM les phrases de
# référence du tuning (fuite constatée avec les binaires v2 : WER fictifs, alpha au plafond).
# Binaires sous kenlm_models_v3/ ; les v1 (train seul) restent — 12.2c choisit par langue.
import subprocess, glob

KENLM_DIR = "/content/kenlm"
LM_DRIVE_V3 = os.path.join(FT_CONFIG["experiment_run"], "kenlm_models_v3")
os.makedirs(LM_DRIVE_V3, exist_ok=True)
_lmplz = os.path.join(KENLM_DIR, "build", "bin", "lmplz")
_bbin = os.path.join(KENLM_DIR, "build", "bin", "build_binary")
if not os.path.exists(_lmplz):
    subprocess.run("apt-get install -y libeigen3-dev libboost-all-dev",
                   shell=True, capture_output=True)
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/kpu/kenlm.git", KENLM_DIR], check=True)
    _r = subprocess.run(f"cd {KENLM_DIR} && mkdir -p build && cd build && cmake .. && make -j4",
                        shell=True, capture_output=True, text=True)
    if not os.path.exists(_lmplz):
        print(_r.stdout[-1500:], _r.stderr[-1500:])
        raise SystemExit("Compilation KenLM échouée (lmplz introuvable) — voir logs ci-dessus.")
print("KenLM prêt :", _lmplz)

LM_PATHS_V3 = {}
for lang in LANGUAGES:
    _bin = os.path.join(LM_DRIVE_V3, f"{lang}.binary")
    if os.path.exists(_bin):
        LM_PATHS_V3[lang] = _bin
        print(f"  {lang}: LM v3 déjà présent ({os.path.getsize(_bin)//1024} Ko)")
        continue
    _sents = manifest[manifest.split.isin(["train", "valid"])
                  & (manifest.language == lang)]["text_norm"].tolist()
    _txt, _arpa = f"/content/lm2_{lang}.txt", f"/content/lm2_{lang}.arpa"
    with open(_txt, "w", encoding="utf-8") as f:
        f.write("\n".join(str(s) for s in _sents))
    def _lmplz_run(order):
        with open(_txt, "rb") as fi, open(_arpa, "wb") as fo:
            return subprocess.run([_lmplz, "-o", str(order), "--discount_fallback"],
                                  stdin=fi, stdout=fo, stderr=subprocess.PIPE, text=False)
    _order = 5
    _r = _lmplz_run(_order)
    if _r.returncode != 0:            # repli ordre 3 si le corpus est trop petit
        _order = 3
        _r = _lmplz_run(_order)
    assert _r.returncode == 0, f"lmplz {lang} échec : {_r.stderr[-600:].decode(errors='replace')}"
    subprocess.run([_bbin, _arpa, _bin], check=True)
    LM_PATHS_V3[lang] = _bin
    print(f"  {lang}: {len(_sents)} phrases -> {_order}-gramme v3 -> {os.path.getsize(_bin)//1024} Ko")
print("LMs v3 prêts -> 12.2c")


In [ ]:
# 12.2 — Décodeurs pyctcdecode + réglage alpha/beta sur local_test (valid)
if not RUN_LM_TUNING:
    raise SystemExit("RUN_LM_TUNING=False")
import numpy as np, jiwer
from pyctcdecode import build_ctcdecoder

LM_DRIVE = os.path.join(FT_CONFIG["experiment_run"], "kenlm_models")
if "LM_PATHS" not in globals():
    LM_PATHS = {l: os.path.join(LM_DRIVE, f"{l}.binary") for l in LANGUAGES}
assert all(os.path.exists(p) for p in LM_PATHS.values()), "LMs manquants : exécuter 12.1."
if "LM_UNIGRAMS" not in globals():
    LM_UNIGRAMS = {}
    for l in LANGUAGES:
        _w = set()
        for t in manifest[(manifest.split == "train") & (manifest.language == l)]["text_norm"]:
            _w.update(str(t).split())
        LM_UNIGRAMS[l] = sorted(_w)

# Pipeline + labels pour pyctcdecode : pièces SentencePiece (UNIQUES par construction ;
# create_decoder produisait des doublons "" pour bos/pad/eos/unk). blank = index 0 -> "".
_pipe_lm = load_finetuned_pipeline(latest_ckpt(FT_CONFIG["output_dir"]))
_tkm = getattr(_pipe_lm.tokenizer, "_model", None)
LABELS = None
for _meth in ("index_to_token", "id_to_piece", "IdToPiece"):
    if _tkm is not None and hasattr(_tkm, _meth):
        try:
            LABELS = [str(getattr(_tkm, _meth)(i))
                      for i in range(_pipe_lm.tokenizer.vocab_info.size)]
            print("vocab via _model.%s" % _meth)
            break
        except Exception:
            LABELS = None
assert LABELS is not None, "Vocab SentencePiece inaccessible (voir _pipe_lm.tokenizer._model)."
_dups = len(LABELS) - len(set(LABELS))
assert _dups == 0, f"{_dups} pièces SP dupliquées — me montrer un échantillon."
LABELS[0] = ""   # blank CTC (index 0, dominant en argmax)

# Capture des log-probs CTC (spy sur le forward ; ordre garanti par batch_size=1)
def capture_logprobs(inputs, omni_code):
    caps = []
    _orig = _pipe_lm.model.forward
    def _spy(src, bl, *a, **kw):
        out = _orig(src, bl, *a, **kw)
        lg = out[0] if isinstance(out, tuple) else out
        caps.append(torch.log_softmax(lg[0].detach().float(), -1).cpu().numpy())
        return out
    _pipe_lm.model.forward = _spy
    try:
        _pipe_lm.transcribe(inputs, lang=[omni_code] * len(inputs), batch_size=1)
    finally:
        _pipe_lm.model.forward = _orig
    return caps

# Test de cohérence : pyctcdecode SANS LM doit reproduire le greedy du pipeline (sinon
# labels/blank à revoir) — quelques secondes, évite un réglage/inférence sur une base fausse.
_pr = manifest[manifest.split == "valid"].iloc[0]
_lp0 = capture_logprobs([_pr["curated_audio_path"]], omni_lang(_pr["language"]))[0]
_g_ctc = normalize_text(build_ctcdecoder(LABELS).decode(_lp0))
_g_pipe = normalize_text(_pipe_lm.transcribe([_pr["curated_audio_path"]],
                         lang=[omni_lang(_pr["language"])], batch_size=1)[0])
print("sanity ctc :", _g_ctc[:70])
print("sanity pipe:", _g_pipe[:70])
assert jiwer.wer([_g_pipe], [_g_ctc]) < 0.15, \
    "pyctcdecode ne reproduit pas le greedy pipeline -> labels/blank/▁ à revoir (me montrer ces 2 lignes)."

# Grille alpha (poids LM) / beta (insertion) sur ~120 clips valid par langue
_val = (manifest[manifest.split == "valid"].sample(frac=1.0, random_state=FT_CONFIG["seed"])
        .groupby("language").head(120))
ALPHAS = [0.0, 0.3, 0.5, 0.8]
BETAS = [0.0, 1.0]
LM_BEST = {}
for lang in LANGUAGES:
    sub = _val[_val.language == lang]
    lp = capture_logprobs(sub["curated_audio_path"].tolist(), omni_lang(lang))
    refs = sub["text_norm"].tolist()
    dec0 = build_ctcdecoder(LABELS, unigrams=LM_UNIGRAMS[lang])
    wer_greedy = jiwer.wer(refs, [normalize_text(dec0.decode(x)) for x in lp])
    best = (wer_greedy, 0.0, 0.0)
    for a in ALPHAS:
        for b in BETAS:
            if a == 0.0 and b == 0.0:
                continue
            dec = build_ctcdecoder(LABELS, LM_PATHS[lang], unigrams=LM_UNIGRAMS[lang], alpha=a, beta=b)
            w = jiwer.wer(refs, [normalize_text(dec.decode(x)) for x in lp])
            if w < best[0]:
                best = (w, a, b)
    LM_BEST[lang] = {"alpha": best[1], "beta": best[2],
                     "wer_greedy": round(wer_greedy, 4), "wer_best": round(best[0], 4)}
    print(f"{lang}: greedy {wer_greedy:.4f} -> LM {best[0]:.4f}  (alpha={best[1]}, beta={best[2]})")
save_json(LM_BEST, os.path.join(FT_CONFIG["report_dir"], "kenlm_tuning.json"))
_gain = float(np.mean([LM_BEST[l]["wer_greedy"] - LM_BEST[l]["wer_best"] for l in LANGUAGES]))
print(f"\nGain WER moyen (valid) : {_gain:+.4f}  -> "
      f"{'LM UTILE, lancer 12.3' if _gain > 0.002 else 'gain faible, me montrer les chiffres'}")


In [ ]:
# 12.2b — Raffinage alpha/beta autour de l'optimum 12.2 + largeur de beam (rejouable)
if not RUN_LM_TUNING:
    raise SystemExit("RUN_LM_TUNING=False")
# La grille 12.2 était grossière (alpha max 0.8, atteint par certaines langues = optimum au bord).
# Ici : grille LOCALE par langue centrée sur kenlm_tuning.json, échantillon valid élargi
# 120->200 clips (même seed : les 120 premiers sont identiques + 80 nouveaux, réglage plus stable),
# puis essai beam_width=250 au meilleur couple (décodage ~2,5x plus lent : gardé si gain réel).
# À la fin : sauvegarde kenlm_tuning.json (backup v1) et INVALIDE les CSV test des langues dont
# les paramètres ont changé -> relancer 12.3 (la reprise ne refera que ces langues-là).
import glob, json, shutil, subprocess
try:      # kenlm (module python) requis par pyctcdecode pour charger les .binary — il n'est
    import kenlm, pyctcdecode          # installé par AUCUNE cellule (venait d'une sonde) -> auto
except ImportError:
    subprocess.run(["pip", "install", "-q", "pyctcdecode", "kenlm"], check=False)
    import kenlm, pyctcdecode
import numpy as np, jiwer
from pyctcdecode import build_ctcdecoder

_tune_path = os.path.join(FT_CONFIG["report_dir"], "kenlm_tuning.json")
assert os.path.exists(_tune_path), "kenlm_tuning.json absent : exécuter 12.2 d'abord."
_prev = json.load(open(_tune_path))

LM_DRIVE = os.path.join(FT_CONFIG["experiment_run"], "kenlm_models")
if "LM_PATHS" not in globals():
    LM_PATHS = {l: os.path.join(LM_DRIVE, f"{l}.binary") for l in LANGUAGES}
assert all(os.path.exists(p) for p in LM_PATHS.values()), "LMs manquants : exécuter 12.1."
if "LM_UNIGRAMS" not in globals():
    LM_UNIGRAMS = {}
    for l in LANGUAGES:
        _w = set()
        for t in manifest[(manifest.split == "train") & (manifest.language == l)]["text_norm"]:
            _w.update(str(t).split())
        LM_UNIGRAMS[l] = sorted(_w)

if "capture_logprobs" not in globals() or "LABELS" not in globals():
    # session neuve : pipeline + labels + capture reconstruits (repris de 12.2, mêmes garde-fous)
    _pipe_lm = load_finetuned_pipeline(latest_ckpt(FT_CONFIG["output_dir"]))
    _tkm = getattr(_pipe_lm.tokenizer, "_model", None)
    LABELS = None
    for _meth in ("index_to_token", "id_to_piece", "IdToPiece"):
        if _tkm is not None and hasattr(_tkm, _meth):
            try:
                LABELS = [str(getattr(_tkm, _meth)(i))
                          for i in range(_pipe_lm.tokenizer.vocab_info.size)]
                break
            except Exception:
                LABELS = None
    assert LABELS is not None and len(LABELS) == len(set(LABELS)), "Vocab SP inaccessible/dupliqué."
    LABELS[0] = ""   # blank CTC
    def capture_logprobs(inputs, omni_code):
        caps = []
        _orig = _pipe_lm.model.forward
        def _spy(src, bl, *a, **kw):
            out = _orig(src, bl, *a, **kw)
            lg = out[0] if isinstance(out, tuple) else out
            caps.append(torch.log_softmax(lg[0].detach().float(), -1).cpu().numpy())
            return out
        _pipe_lm.model.forward = _spy
        try:
            _pipe_lm.transcribe(inputs, lang=[omni_code] * len(inputs), batch_size=1)
        finally:
            _pipe_lm.model.forward = _orig
        return caps

_val = (manifest[manifest.split == "valid"].sample(frac=1.0, random_state=FT_CONFIG["seed"])
        .groupby("language").head(200))
LM_BEST = {}
for lang in LANGUAGES:
    _t0 = time.time()
    a0, b0 = float(_prev[lang]["alpha"]), float(_prev[lang]["beta"])
    _as = sorted({round(max(0.0, a0 + d), 2) for d in (-0.15, 0.0, 0.15, 0.3, 0.5)})
    _bs = sorted({round(max(0.0, b0 + d), 2) for d in (-1.0, 0.0, 1.0, 2.0)})
    sub = _val[_val.language == lang]
    refs = sub["text_norm"].tolist()
    lp = capture_logprobs(sub["curated_audio_path"].tolist(), omni_lang(lang))
    best = None                       # (a0,b0) est dans la grille -> baseline comparable incluse
    for a in _as:
        for b in _bs:
            dec = build_ctcdecoder(LABELS, LM_PATHS[lang], unigrams=LM_UNIGRAMS[lang],
                                   alpha=a, beta=b)
            w = jiwer.wer(refs, [normalize_text(dec.decode(x)) for x in lp])
            if best is None or w < best[0]:
                best = (w, a, b)
    w100, a1, b1 = best
    dec = build_ctcdecoder(LABELS, LM_PATHS[lang], unigrams=LM_UNIGRAMS[lang], alpha=a1, beta=b1)
    w250 = jiwer.wer(refs, [normalize_text(dec.decode(x, beam_width=250)) for x in lp])
    beam = 250 if w250 < w100 - 0.002 else 100
    LM_BEST[lang] = {"alpha": a1, "beta": b1, "beam": beam,
                     "wer_best": round(w250 if beam == 250 else w100, 4),
                     "wer_12_2": _prev[lang].get("wer_best"),
                     "wer_greedy": _prev[lang].get("wer_greedy")}
    print(f"{lang}: (a={a0}, b={b0}) wer12.2={_prev[lang].get('wer_best')} -> "
          f"(a={a1}, b={b1}, beam={beam}) wer={LM_BEST[lang]['wer_best']}"
          f"  [{(time.time()-_t0)/60:.1f} min]")
    del lp

_v1 = os.path.join(FT_CONFIG["report_dir"], "kenlm_tuning_v1.json")
if not os.path.exists(_v1):
    shutil.copy(_tune_path, _v1)      # garder la version 12.2 (celle de la soumission 0.38888)
save_json(LM_BEST, _tune_path)

# Invalidation des prédictions test des langues dont les paramètres ont CHANGÉ (critère =
# paramètres, pas WER : les WER 12.2/12.2b ne sont pas sur le même nombre de clips).
_MANIFEST_TO_DIR = {"sw": "swa"}      # seuls les CSV utilisent les codes des dossiers test
_lm_dir = os.path.join(FT_CONFIG["report_dir"], "test_predictions_lm")
_stale, _changed = 0, []
for lang in LANGUAGES:
    _same = (LM_BEST[lang]["alpha"] == float(_prev[lang]["alpha"])
             and LM_BEST[lang]["beta"] == float(_prev[lang]["beta"])
             and LM_BEST[lang]["beam"] == 100)
    if not _same:
        _changed.append(lang)
        for f in glob.glob(os.path.join(_lm_dir, f"{_MANIFEST_TO_DIR.get(lang, lang)}__*.csv")):
            os.remove(f); _stale += 1
_gain = float(np.mean([float(_prev[l]["wer_best"]) - LM_BEST[l]["wer_best"] for l in LANGUAGES]))
print(f"\nGain WER moyen supplémentaire (valid) : {_gain:+.4f}")
if _changed:
    print(f"Langues modifiées : {_changed} -> {_stale} CSV de shards invalidés.")
    print("RELANCER 12.3 : la reprise saute les langues inchangées et refait celles-ci.")
else:
    print("Aucun paramètre modifié : l'optimum 12.2 était déjà le bon, ne pas relancer 12.3.")


In [ ]:
# 12.2c — POLISH sans fuite : re-réglage alpha/beta/beam par langue sur les logits ACTUELS
if not RUN_LM_TUNING:
    raise SystemExit("RUN_LM_TUNING=False")
# (routage top-ups) + choix v1 (train) vs v3 (train+valid) par langue. Réglage sur LOCAL_TEST,
# jamais vu par les LM ni par aucune décision antérieure — régler sur valid avec un LM qui
# contient valid mémorise les références (fuite v2 : WER fictifs 0.04-0.23, alpha au plafond).
# ~1h30. Invalide les CSV des langues modifiées -> relancer 12.3 puis SOUMETTRE.
import subprocess, json, glob, shutil
try:
    import kenlm, pyctcdecode
except ImportError:
    subprocess.run(["pip", "install", "-q", "pyctcdecode", "kenlm"], check=False)
    import kenlm, pyctcdecode
import numpy as np, jiwer
from pyctcdecode import build_ctcdecoder

assert "LM_PATHS_V3" in globals(), "Exécuter 12.1b d'abord."
LM_PATHS_V1 = {l: os.path.join(FT_CONFIG["experiment_run"], "kenlm_models", f"{l}.binary")
               for l in LANGUAGES}

# Routage joint/top-ups : TOUJOURS resynchronisé depuis les verdicts 13.3 — le nettoyage GPU
# de 13.2 vide TOPUP_PIPES, un dict partiel ou un capture_logprobs non routé (12.2b) peuvent
# subsister dans la session -> tout est redéfini ici, sans condition.
if "TOPUP_PIPES" not in globals():
    TOPUP_PIPES = {}
if "_pipe_joint" not in globals():
    _pipe_joint = load_finetuned_pipeline(latest_ckpt(FT_CONFIG["output_dir"]))
for _f in glob.glob(os.path.join(FT_CONFIG["report_dir"], "topup_choice_*.json")):
    _c = json.load(open(_f))
    if _c["chosen"] != "joint" and omni_lang(_c["language"]) not in TOPUP_PIPES:
        TOPUP_PIPES[omni_lang(_c["language"])] = load_finetuned_pipeline(_c["ckpt"])
def _labels_from(pipe):
    _tkm = getattr(pipe.tokenizer, "_model", None)
    for _m in ("index_to_token", "id_to_piece", "IdToPiece"):
        if _tkm is not None and hasattr(_tkm, _m):
            try:
                L = [str(getattr(_tkm, _m)(i)) for i in range(pipe.tokenizer.vocab_info.size)]
                assert len(L) == len(set(L))
                L[0] = ""
                return L
            except Exception:
                pass
    raise SystemExit("Vocab SentencePiece inaccessible.")
def _capture(pipe, inputs, omni_code):
    caps = []
    _orig = pipe.model.forward
    def _spy(src, bl, *a, **kw):
        out = _orig(src, bl, *a, **kw)
        lg = out[0] if isinstance(out, tuple) else out
        caps.append(torch.log_softmax(lg[0].detach().float(), -1).cpu().numpy())
        return out
    pipe.model.forward = _spy
    try:
        hyps = pipe.transcribe(inputs, lang=[omni_code] * len(inputs), batch_size=1)
    finally:
        pipe.model.forward = _orig
    return caps, hyps
LABELS = _labels_from(_pipe_joint)
def capture_logprobs(inputs, omni_code):
    return _capture(TOPUP_PIPES.get(omni_code, _pipe_joint), inputs, omni_code)[0]
print("routage :", (", ".join(TOPUP_PIPES) + " -> top-up, ") if TOPUP_PIPES else "", "autres -> joint")

if "LM_UNIGRAMS" not in globals():                       # unigrammes v1 : train seul
    LM_UNIGRAMS = {}
    for l in LANGUAGES:
        _w = set()
        for t in manifest[(manifest.split == "train") & (manifest.language == l)]["text_norm"]:
            _w.update(str(t).split())
        LM_UNIGRAMS[l] = sorted(_w)
_uni3 = {}                                               # unigrammes v3 : train + valid
for l in LANGUAGES:
    _w = set()
    for t in manifest[manifest.split.isin(["train", "valid"])
                      & (manifest.language == l)]["text_norm"]:
        _w.update(str(t).split())
    _uni3[l] = sorted(_w)

_prev_p = os.path.join(FT_CONFIG["report_dir"], "kenlm_tuning_pre_polish.json")
# base de comparaison = derniers réglages SAINS (le backup pré-polish si la 12.2c fuyarde a tourné)
_prev = json.load(open(_prev_p if os.path.exists(_prev_p)
                       else os.path.join(FT_CONFIG["report_dir"], "kenlm_tuning.json")))
ALPHAS = [0.3, 0.5, 0.65, 0.8, 1.0]
_val = (manifest[manifest.split == "local_test"]
        .sample(frac=1.0, random_state=FT_CONFIG["seed"]).groupby("language").head(200))
assert len(_val), "local_test absent du manifest ?"
LM_BEST = {}
for lang in LANGUAGES:
    _t0 = time.time()
    sub = _val[_val.language == lang]
    refs = sub["text_norm"].tolist()
    lp = capture_logprobs(sub["curated_audio_path"].tolist(), omni_lang(lang))
    _b0 = float(_prev[lang]["beta"])
    BETAS = sorted({max(0.0, _b0 - 1.0), _b0, _b0 + 1.0})
    best = None
    for _tag, _bin, _uni in (("v1", LM_PATHS_V1[lang], LM_UNIGRAMS[lang]),
                             ("v3", LM_PATHS_V3[lang], _uni3[lang])):
        for a in ALPHAS:
            for b in BETAS:
                dec = build_ctcdecoder(LABELS, _bin, unigrams=_uni, alpha=a, beta=b)
                w = jiwer.wer(refs, [normalize_text(dec.decode(x)) for x in lp])
                if best is None or w < best[0]:
                    best = (w, _tag, _bin, a, b)
    w1, _tag, _bin, a1, b1 = best
    dec = build_ctcdecoder(LABELS, _bin, unigrams=(_uni3[lang] if _tag == "v3" else LM_UNIGRAMS[lang]),
                           alpha=a1, beta=b1)
    w250 = jiwer.wer(refs, [normalize_text(dec.decode(x, beam_width=250)) for x in lp])
    beam = 250 if w250 < w1 - 0.002 else 100
    LM_BEST[lang] = {"alpha": a1, "beta": b1, "beam": beam, "lm": _tag, "lm_bin": _bin,
                     "wer_best": round(min(w1, w250) if beam == 250 else w1, 4),
                     "wer_avant_polish": _prev[lang].get("wer_best")}
    print(f"{lang}: LM {_tag} (a={a1}, b={b1}, beam={beam})  WER {LM_BEST[lang]['wer_best']}"
          f"  (avant polish {_prev[lang].get('wer_best')})  [{(time.time()-_t0)/60:.1f} min]")
    del lp

if not os.path.exists(_prev_p):        # ne jamais écraser le backup sain d'origine
    shutil.copy(os.path.join(FT_CONFIG["report_dir"], "kenlm_tuning.json"), _prev_p)
save_json(LM_BEST, os.path.join(FT_CONFIG["report_dir"], "kenlm_tuning.json"))
LM_PATHS = {l: LM_BEST[l]["lm_bin"] for l in LANGUAGES}          # utilisés par 12.3 (_DECODERS)
LM_UNIGRAMS = {l: (_uni3[l] if LM_BEST[l]["lm"] == "v3" else LM_UNIGRAMS[l]) for l in LANGUAGES}

# invalider les CSV test des langues dont les réglages ont changé
_lm_dir = os.path.join(FT_CONFIG["report_dir"], "test_predictions_lm")
_MAN2DIR = {"sw": "swa"}
_n, _changed = 0, []
for lang in LANGUAGES:
    _o = _prev[lang]
    if (LM_BEST[lang]["alpha"], LM_BEST[lang]["beta"], LM_BEST[lang]["beam"], LM_BEST[lang]["lm"]) \
       != (float(_o["alpha"]), float(_o["beta"]), int(_o.get("beam", 100)), _o.get("lm", "v1")):
        _changed.append(lang)
        for f in glob.glob(os.path.join(_lm_dir, f"{_MAN2DIR.get(lang, lang)}__*.csv")):
            os.remove(f)
            _n += 1
print(f"\nLangues modifiées : {_changed} -> {_n} CSV invalidés. RELANCER 12.3 puis soumettre.")


In [ ]:
# 12.2d — V3 : restauration rule-safe avec UN checkpoint acoustique
import gc, glob, hashlib, json, subprocess
from pathlib import Path
try:
    import kenlm, pyctcdecode
except ImportError:
    subprocess.run(["pip", "install", "-q", "pyctcdecode", "kenlm"], check=False)
    import kenlm, pyctcdecode
from pyctcdecode import build_ctcdecoder

require_state("FT_CONFIG", "load_finetuned_pipeline", "latest_ckpt", "manifest")
_joint_ckpt = latest_ckpt(FT_CONFIG["output_dir"])
PRIMARY_ACOUSTIC_CKPT = os.path.realpath(FINAL_UNIFIED_CHECKPOINT or _joint_ckpt)
assert os.path.exists(PRIMARY_ACOUSTIC_CKPT), PRIMARY_ACOUSTIC_CKPT

# Ancien état potentiellement dangereux : toujours le vider avant la restauration finale.
for _name in ("_pipe_joint", "_pipe_lm", "_pipe"):
    if _name in globals():
        del globals()[_name]
if "TOPUP_PIPES" in globals():
    TOPUP_PIPES.clear()
gc.collect()
torch.cuda.empty_cache()

def _labels_from(pipe):
    _tkm = getattr(pipe.tokenizer, "_model", None)
    for _method in ("index_to_token", "id_to_piece", "IdToPiece"):
        if _tkm is not None and hasattr(_tkm, _method):
            try:
                labels = [str(getattr(_tkm, _method)(i))
                          for i in range(pipe.tokenizer.vocab_info.size)]
                assert len(labels) == len(set(labels))
                labels[0] = ""
                return labels
            except Exception:
                pass
    raise RuntimeError("Vocabulaire du tokenizer inaccessible")

def _capture(pipe, inputs, omni_code):
    caps = []
    original = pipe.model.forward
    def spy(src, bl, *args, **kwargs):
        out = original(src, bl, *args, **kwargs)
        logits = out[0] if isinstance(out, tuple) else out
        caps.append(torch.log_softmax(logits[0].detach().float(), -1).cpu().numpy())
        return out
    pipe.model.forward = spy
    try:
        pipe.transcribe(inputs, lang=[omni_code] * len(inputs), batch_size=1)
    finally:
        pipe.model.forward = original
    return caps

_pipe_joint = load_finetuned_pipeline(PRIMARY_ACOUSTIC_CKPT)
_actual_params = sum(parameter.numel() for parameter in _pipe_joint.model.parameters())
assert _actual_params < OFFICIAL_MODEL_PARAM_LIMIT, (
    f"{_actual_params:,} paramètres >= {OFFICIAL_MODEL_PARAM_LIMIT:,}"
)
LABELS = _labels_from(_pipe_joint)

DOMAINS = ("scripted", "unscripted")
TEST_DIR_TO_MANIFEST = {"swa": "sw", "kik": "kik", "kln": "kln",
                        "luo": "luo", "som": "som", "mas": "mas"}

def canonical_domain(value):
    text = str(value).strip().lower().replace("_", "-")
    if "unscript" in text or "spont" in text or text in {"false", "0"}:
        return "unscripted"
    if "script" in text or text in {"true", "1"}:
        return "scripted"
    raise ValueError(f"Domaine inconnu : {value!r}")

def shard_domain(path):
    return canonical_domain(Path(path).parent.name)

_default_routes = {(lang, domain): PRIMARY_ACOUSTIC_CKPT
                   for lang in LANGUAGES for domain in DOMAINS}
ACOUSTIC_ROUTES = dict(_default_routes)
if USE_MULTI_FULL_CHECKPOINT_ROUTING:
    assert ALLOW_MULTI_CHECKPOINT_ROUTING and ALLOW_DOMAIN_ACOUSTIC_ROUTING
    assert ORGANIZER_CONFIRMED_MULTI_FULL_CHECKPOINTS
    for key, checkpoint in EXPERIMENTAL_DOMAIN_CHECKPOINTS.items():
        lang, domain = key
        assert lang in LANGUAGES and canonical_domain(domain) == domain
        ACOUSTIC_ROUTES[(lang, domain)] = os.path.realpath(checkpoint)

_distinct_ckpts = sorted(set(ACOUSTIC_ROUTES.values()))
if COMPLIANCE_MODE:
    assert len(_distinct_ckpts) == 1, (
        "Mode conformité : un seul checkpoint acoustique autorisé dans le package final."
    )
assert all(os.path.exists(path) for path in _distinct_ckpts)

_PIPE_CACHE = {PRIMARY_ACOUSTIC_CKPT: _pipe_joint}
_labels_digest = hashlib.sha256("\0".join(LABELS).encode()).hexdigest()

def _get_pipe(checkpoint):
    checkpoint = os.path.realpath(checkpoint)
    if checkpoint not in _PIPE_CACHE:
        pipe = load_finetuned_pipeline(checkpoint)
        labels = _labels_from(pipe)
        assert hashlib.sha256("\0".join(labels).encode()).hexdigest() == _labels_digest
        _PIPE_CACHE[checkpoint] = pipe
    return _PIPE_CACHE[checkpoint]

def capture_logprobs(inputs, omni_code, manifest_lang=None, domains=None):
    manifest_lang = manifest_lang or next(
        (lang for lang in LANGUAGES if omni_lang(lang) == omni_code), None)
    assert manifest_lang in LANGUAGES
    if isinstance(domains, str):
        domains = [canonical_domain(domains)] * len(inputs)
    if domains is None:
        _paths = {ACOUSTIC_ROUTES[(manifest_lang, d)] for d in DOMAINS}
        assert len(_paths) == 1, "Domaine obligatoire avec plusieurs routes acoustiques"
        domains = ["scripted"] * len(inputs)
    assert len(inputs) == len(domains)
    result = [None] * len(inputs)
    grouped = {}
    for index, domain in enumerate(domains):
        checkpoint = ACOUSTIC_ROUTES[(manifest_lang, canonical_domain(domain))]
        grouped.setdefault(checkpoint, []).append(index)
    for checkpoint, indices in grouped.items():
        values = _capture(_get_pipe(checkpoint), [inputs[i] for i in indices], omni_code)
        for index, value in zip(indices, values):
            result[index] = value
    assert all(value is not None for value in result)
    return result

LM_BEST = json.load(open(os.path.join(FT_CONFIG["report_dir"], "kenlm_tuning.json")))
LM_DRIVE = os.path.join(FT_CONFIG["experiment_run"], "kenlm_models")
LM_PATHS = {lang: LM_BEST.get(lang, {}).get("lm_bin")
            or os.path.join(LM_DRIVE, f"{lang}.binary") for lang in LANGUAGES}
assert all(os.path.exists(path) for path in LM_PATHS.values()), "Binaire LM manquant"
LM_UNIGRAMS = {}
for lang in LANGUAGES:
    config = LM_BEST.get(lang, {})
    unigram_file = config.get("uni_file")
    if unigram_file and os.path.exists(unigram_file):
        LM_UNIGRAMS[lang] = open(unigram_file, encoding="utf-8").read().splitlines()
    else:
        rows = manifest[manifest.split == "train"]
        words = {word for text in rows[rows.language == lang]["text_norm"]
                 for word in str(text).split()}
        LM_UNIGRAMS[lang] = sorted(words)

MODEL_COMPLIANCE = {
    "pipeline_revision": PIPELINE_REVISION,
    "checkpoint_count": len(_distinct_ckpts),
    "checkpoints": _distinct_ckpts,
    "actual_acoustic_parameters": _actual_params,
    "parameter_limit": OFFICIAL_MODEL_PARAM_LIMIT,
    "single_checkpoint": len(_distinct_ckpts) == 1,
    "edge_peak_rss_still_required": True,
}
save_json(MODEL_COMPLIANCE,
          os.path.join(FT_CONFIG["report_dir"], "model_compliance_v3.json"))
print("Restauration V3 :", MODEL_COMPLIANCE)


## 12.3 — Inférence historique désactivée dans V3

L'ancienne cellule mélangeait état global, suppressions manuelles et caches sans empreinte.
Elle est remplacée par la **section 17**, qui applique un routage LM par domaine, un seul
checkpoint acoustique par défaut et un répertoire de cache dérivé de toute la provenance.


In [ ]:
# 13.1 — SONDE : carte d'asset "omniASR_CTC_1B_v2_ft5000" pointant sur NOTRE checkpoint step_5000.
# Le top-up doit PARTIR du modèle joint fine-tuné (pas du modèle Meta) ; la voie fairseq2 propre
# est une carte héritant de omniASR_CTC_1B_v2 avec un checkpoint local. La vérification tourne en
# SOUS-PROCESSUS (scan des cartes garanti frais, le processus courant peut avoir mis en cache la
# liste) et compare des tenseurs carte-chargée vs fichier checkpoint. ~2-4 min, CPU seulement.
import subprocess, sys, glob
import omnilingual_asr as _oa

_ck = latest_ckpt(FT_CONFIG["output_dir"])
assert _ck, "Aucun checkpoint du run joint : section 8 d'abord."
TOPUP_SEED_CKPT = _ck
if os.path.isdir(_ck):
    _w = [q for q in glob.glob(os.path.join(_ck, "model", "**", "*"), recursive=True)
          if os.path.isfile(q)]
    assert len(_w) == 1, f"Poids multiples/absents sous {_ck}/model : {_w}"
    TOPUP_SEED_CKPT = _w[0]
TOPUP_SEED_CARD = "omniASR_CTC_1B_v2_ft5000"
_card_path = os.path.join(os.path.dirname(_oa.__file__), "cards", "models",
                          "afrivoices_topup.yaml")
os.makedirs(os.path.dirname(_card_path), exist_ok=True)

_PROBE = "/content/probe_topup_card.py"
open(_PROBE, "w").write('''
import sys, torch
ckpt_file, card = sys.argv[1], sys.argv[2]
import omnilingual_asr                                  # enregistre les cartes du package
from fairseq2.models.hub import load_model
try:                                                    # notre propre artefact ; True d'abord
    st = torch.load(ckpt_file, map_location="cpu", weights_only=True)
except Exception:
    st = torch.load(ckpt_file, map_location="cpu", weights_only=False)
sd = st.get("model", st) if isinstance(st, dict) else st
m = load_model(card, device=torch.device("cpu"), dtype=torch.float32)
msd = m.state_dict()
common = [k for k in sd.keys() if k in msd]
assert common, f"Aucune cle commune. ckpt: {list(sd)[:5]} / modele: {list(msd)[:5]}"
bad = 0
for k in (common[0], common[len(common) // 2], common[-1]):
    d = (msd[k].float() - sd[k].float()).abs().max().item()
    print(f"  {k}: ecart max {d:.2e}")
    bad += d > 1e-3
sys.exit(1 if bad else 0)
''')

_ok = False
for _uri in (f"file://{TOPUP_SEED_CKPT}", TOPUP_SEED_CKPT):   # deux formes de chemin local
    with open(_card_path, "w") as f:
        f.write(f'name: {TOPUP_SEED_CARD}\nbase: {FT_CONFIG["model_card"]}\n'
                f'checkpoint: "{_uri}"\n')
    _r = subprocess.run([sys.executable, _PROBE, TOPUP_SEED_CKPT, TOPUP_SEED_CARD],
                        capture_output=True, text=True)
    print(f"— essai checkpoint: {_uri}\n{_r.stdout}")
    if _r.returncode == 0:
        _ok = True
        break
    print(_r.stderr[-1200:])
assert _ok, ("La carte ne charge pas nos poids : me montrer TOUTE la sortie ci-dessus "
             "(plan B possible : copie du checkpoint dans le nouveau output_dir).")
print(f"✅ Carte '{TOPUP_SEED_CARD}' opérationnelle -> {_card_path}")
print(f"   poids = {TOPUP_SEED_CKPT}")

In [ ]:
# 13.2 — TOP-UP mono-langue : parquet {lang} (train+valid), carte dataset repointée, recette
if not RUN_EXPERIMENTAL_TOPUP:
    raise SystemExit("RUN_EXPERIMENTAL_TOPUP=False")
# depuis la carte ft5000 (13.1), sortie Drive topup_{lang}/. kln et mas = pires langues (WER
# valid LM 0.554/0.521) : un top-up SPÉCIALISE le modèle ; chaque langue utilisera son propre
# checkpoint à l'inférence (13.4), donc l'oubli des autres langues est sans importance.
# ~30-40 min de build (FLAC via FUSE) + ~2h30 de train (1000 steps). REPRENABLE : relancer la
# même cellule reprend au dernier checkpoint (et saute le build si le parquet local est là).
TOPUP_LANG = "kln"          # "kln" puis "mas" — vide = cellule inerte (protège « Tout exécuter »)
TOPUP_STEPS = 2000       # checkpoints/validation à 500 et 1000 -> choix du meilleur en 13.3
if not TOPUP_LANG:
    raise SystemExit("TOPUP_LANG vide : mettre 'kln' ou 'mas' puis relancer.")
assert TOPUP_LANG in LANGUAGES, f"Langue inconnue : {TOPUP_LANG}"
assert "TOPUP_SEED_CARD" in globals(), "Exécuter 13.1 d'abord (carte + sonde)."
assert "build_parquet" in globals(), "Exécuter 5.2 d'abord (définitions build_parquet/TSV)."

# LIBÉRER LE GPU DU NOYAU : les pipelines d'inférence (13.3/13.4/12.3) laissent ~8-11 GB sur
# cuda:0, or la recette tourne dans un SOUS-PROCESSUS qui a besoin des 40 GB entiers (OOM sinon).
import gc
for _v in ("_pipe_joint", "_pipe_lm", "_pipe"):
    if _v in globals():
        del globals()[_v]
if "TOPUP_PIPES" in globals():
    TOPUP_PIPES.clear()
gc.collect()
torch.cuda.empty_cache()
print(f"GPU noyau libéré : {torch.cuda.memory_allocated() / 2**30:.1f} GB restants alloués")

_t_root = f"/content/topup_{TOPUP_LANG}"
_t_parq = os.path.join(_t_root, "version=0")
_t_tsv = os.path.join(_t_root, "language_distribution_0.tsv")
if os.path.exists(_t_tsv) and (dataset_meta(_t_parq) or {}).get("rows"):
    print("Parquet top-up déjà présent :", _t_parq)
else:
    shutil.rmtree(_t_root, ignore_errors=True)
    _tsv = build_parquet(manifest[manifest.language == TOPUP_LANG].reset_index(drop=True),
                         root=_t_parq)
    assert _tsv == _t_tsv, (_tsv, _t_tsv)
write_dataset_card(data_root=_t_parq)   # la carte GLOBALE pointe maintenant sur le top-up ;
                                        # 2.4 la remet sur le dataset complet à chaque session

_t_out = os.path.join(FT_CONFIG["experiment_run"], f"topup_{TOPUP_LANG}")
_t_yaml = write_recipe_yaml(os.path.join(FT_CONFIG["experiment_run"], f"topup_{TOPUP_LANG}_recipe.yaml"),
                            TOPUP_SEED_CARD, _t_tsv, num_steps=TOPUP_STEPS,
                            grad_accum=FT_CONFIG["grad_accum"],
                            max_num_elements=FT_CONFIG["max_num_elements"], ckpt_every=500)
rc, _log = run_recipe(_t_out, _t_yaml, f"topup_{TOPUP_LANG}.log",
                      wall_limit_h=FT_CONFIG["max_wall_hours_per_session"])
print("Code :", rc, "| dernier checkpoint :", latest_ckpt(_t_out))
print("(~12-15 GB par checkpoint sur Drive : vider la corbeille Drive si tu en supprimes.)")
print("Suite : 13.3 (comparer joint vs step_500 vs step_1000 sur valid).")


In [ ]:
# 13.3 — Choisir le checkpoint : joint step_5000 vs top-up step_500 vs step_1000, WER greedy et
if not RUN_EXPERIMENTAL_TOPUP:
    raise SystemExit("RUN_EXPERIMENTAL_TOPUP=False")
# WER+LM (alpha/beta/beam de kenlm_tuning.json) sur 200 clips valid de la langue. ~15-20 min.
# Verdict sauvé dans reports/topup_choice_{lang}.json (lu par 13.4).
import subprocess, json
try:
    import kenlm, pyctcdecode
except ImportError:
    subprocess.run(["pip", "install", "-q", "pyctcdecode", "kenlm"], check=False)
    import kenlm, pyctcdecode
import jiwer
from pathlib import Path
from pyctcdecode import build_ctcdecoder

TOPUP_LANG_EVAL = ""     # vide = reprendre TOPUP_LANG de 13.2
_tl = TOPUP_LANG_EVAL or globals().get("TOPUP_LANG", "")
assert _tl in LANGUAGES, "Préciser TOPUP_LANG_EVAL (ex. 'kln') ou exécuter 13.2 d'abord."

def _labels_from(pipe):
    _tkm = getattr(pipe.tokenizer, "_model", None)
    for _m in ("index_to_token", "id_to_piece", "IdToPiece"):
        if _tkm is not None and hasattr(_tkm, _m):
            try:
                L = [str(getattr(_tkm, _m)(i)) for i in range(pipe.tokenizer.vocab_info.size)]
                assert len(L) == len(set(L))
                L[0] = ""            # blank CTC
                return L
            except Exception:
                pass
    raise SystemExit("Vocab SentencePiece inaccessible (voir pipe.tokenizer._model).")

def _capture(pipe, inputs, omni_code):
    """(logprobs, hypothèses greedy) — spy sur le forward, batch_size=1 = ordre garanti."""
    caps = []
    _orig = pipe.model.forward
    def _spy(src, bl, *a, **kw):
        out = _orig(src, bl, *a, **kw)
        lg = out[0] if isinstance(out, tuple) else out
        caps.append(torch.log_softmax(lg[0].detach().float(), -1).cpu().numpy())
        return out
    pipe.model.forward = _spy
    try:
        hyps = pipe.transcribe(inputs, lang=[omni_code] * len(inputs), batch_size=1)
    finally:
        pipe.model.forward = _orig
    return caps, hyps

_tune = json.load(open(os.path.join(FT_CONFIG["report_dir"], "kenlm_tuning.json")))
_a, _b = float(_tune[_tl]["alpha"]), float(_tune[_tl]["beta"])
_bw = int(_tune[_tl].get("beam", 100))
_lm_bin = os.path.join(FT_CONFIG["experiment_run"], "kenlm_models", f"{_tl}.binary")
assert os.path.exists(_lm_bin), f"LM manquant : {_lm_bin}"
_uni = set()
for t in manifest[(manifest.split == "train") & (manifest.language == _tl)]["text_norm"]:
    _uni.update(str(t).split())
_uni = sorted(_uni)

_t_out = os.path.join(FT_CONFIG["experiment_run"], f"topup_{_tl}")
_steps = sorted((d for d in Path(_t_out).rglob("step_*")
                 if d.is_dir() and not d.name.endswith(".tmp") and (d / "model").exists()),
                key=lambda d: int(d.name.split("_")[1]))
assert _steps, f"Aucun checkpoint top-up sous {_t_out} : exécuter 13.2 d'abord."
# un changement de num_steps crée un NOUVEAU workspace ws_* (hash de config) : des step_N
# homonymes peuvent coexister entre l'ancien et le nouveau -> désambiguïser par le hash
_names = [d.name for d in _steps]
def _lbl(d):
    return d.name if _names.count(d.name) == 1 \
        else f"{d.name}@{d.parent.parent.name.split('.')[-1]}"
_cands = [("joint", latest_ckpt(FT_CONFIG["output_dir"]))] + [(_lbl(d), str(d)) for d in _steps]

_val = (manifest[(manifest.split == "valid") & (manifest.language == _tl)]
        .sample(frac=1.0, random_state=FT_CONFIG["seed"]).head(200))
refs = _val["text_norm"].tolist()
_res = {}
for _name, _ckpt in _cands:
    _pipe = load_finetuned_pipeline(_ckpt)
    L = _labels_from(_pipe)
    caps, hyps = _capture(_pipe, _val["curated_audio_path"].tolist(), omni_lang(_tl))
    _dec = build_ctcdecoder(L, _lm_bin, unigrams=_uni, alpha=_a, beta=_b)
    wg = jiwer.wer(refs, [normalize_text(h) for h in hyps])
    wl = jiwer.wer(refs, [normalize_text(_dec.decode(x, beam_width=_bw)) for x in caps])
    _res[_name] = {"ckpt": _ckpt, "wer_greedy": round(wg, 4), "wer_lm": round(wl, 4)}
    print(f"{_tl}/{_name}: greedy {wg:.4f} | +LM {wl:.4f}")
    del _pipe, caps
    torch.cuda.empty_cache()

_best = min(_res, key=lambda n: _res[n]["wer_lm"])
save_json({"language": _tl, "chosen": _best, **_res[_best], "all": _res},
          os.path.join(FT_CONFIG["report_dir"], f"topup_choice_{_tl}.json"))
if _best == "joint":
    print(f"⚠️ Le top-up ne bat PAS le joint sur valid ({_tl}) : NE PAS faire 13.4. "
          "Me montrer le tableau ci-dessus.")
else:
    _d = _res["joint"]["wer_lm"] - _res[_best]["wer_lm"]
    print(f"✅ {_best} retenu pour {_tl} : WER LM {_res[_best]['wer_lm']} "
          f"(joint {_res['joint']['wer_lm']}, gain {_d:+.4f}) -> exécuter 13.4.")


In [ ]:
# 13.4 — V3 : enregistrer un candidat sans modifier la soumission finale
require_state("FT_CONFIG", "LANGUAGES")
_tl = globals().get("TOPUP_LANG_EVAL", "") or globals().get("TOPUP_LANG", "")
assert _tl in LANGUAGES, "Définir la langue évaluée"
_choice_path = os.path.join(FT_CONFIG["report_dir"], f"topup_choice_{_tl}.json")
assert os.path.exists(_choice_path), _choice_path
_choice = json.load(open(_choice_path, encoding="utf-8"))
assert _choice.get("chosen") != "joint" and os.path.exists(_choice["ckpt"])

_candidate_registry = os.path.join(FT_CONFIG["report_dir"],
                                   "experimental_checkpoint_candidates_v3.json")
_registry = json.load(open(_candidate_registry)) if os.path.exists(_candidate_registry) else {}
_registry[_tl] = {"registered_at": now_iso(), **_choice}
save_json(_registry, _candidate_registry)

if COMPLIANCE_MODE or not USE_MULTI_FULL_CHECKPOINT_ROUTING:
    print("Candidat enregistré, mais NON déployé : le checkpoint final unique reste inchangé.")
    print("Pour un modèle final unique, utiliser 16.10 (fusion de deltas) puis le valider.")
else:
    assert ALLOW_MULTI_CHECKPOINT_ROUTING and ALLOW_DOMAIN_ACOUSTIC_ROUTING
    assert ORGANIZER_CONFIRMED_MULTI_FULL_CHECKPOINTS
    print("Mode expérimental confirmé : renseigner EXPERIMENTAL_DOMAIN_CHECKPOINTS en 2.1b.")


In [ ]:
# 14.1 — A/B swa : le modèle DE BASE (Meta, non fine-tuné) bat-il notre fine-tuné sur le test ?
if not RUN_LM_EXPERIMENTS:
    raise SystemExit("RUN_LM_EXPERIMENTS=False")
# Notre swa d'entraînement = 100 % agriculture SCRIPTÉE (Afrivoice) ; le test swa = 100 %
# SPONTANÉ. Le fine-tuning a pu dégrader le swahili spontané que le pré-entraîné (tier H,
# données massives) maîtrisait. Aucune donnée interne ne peut le mesurer -> A/B par SOUMISSION :
# on régénère SEULEMENT swa avec le modèle de base (mêmes LM/alpha/beta), on soumet, on compare.
import shutil, glob
assert "TOPUP_PIPES" in globals() and "capture_logprobs" in globals(), "Exécuter 12.2c d'abord."

_lm_dir = os.path.join(FT_CONFIG["report_dir"], "test_predictions_lm")
_bak = os.path.join(FT_CONFIG["report_dir"], "test_predictions_lm_swa_ft")
os.makedirs(_bak, exist_ok=True)
_moved = 0
for f in glob.glob(os.path.join(_lm_dir, "swa__*.csv")):
    shutil.move(f, os.path.join(_bak, os.path.basename(f)))   # prédictions FT mises de côté
    _moved += 1
assert _moved or glob.glob(os.path.join(_bak, "swa__*.csv")), \
    "Aucun CSV swa à mettre de côté : exécuter 12.3 (v7) d'abord."
print(f"{_moved} CSV swa (fine-tuné) sauvegardés -> {_bak}")

_pipe_base = load_finetuned_pipeline(None)     # carte officielle = poids Meta d'origine
TOPUP_PIPES["swh_Latn"] = _pipe_base           # swa -> BASE ; les autres langues inchangées
print("swa routé vers le modèle DE BASE (les LM/réglages ne changent pas).")
print("RELANCER 12.3 (26 shards swa, ~50 min) -> soumettre -> comparer au score v7 :")
print("  si MEILLEUR : l'oubli de domaine est confirmé, on garde et on creuse ;")
print("  si PIRE : exécuter 14.2 pour restaurer les prédictions fine-tunées.")


In [ ]:
# 14.2 — (SEULEMENT si l'A/B 14.1 est perdant) restaurer les prédictions swa fine-tunées
if not RUN_LM_EXPERIMENTS:
    raise SystemExit("RUN_LM_EXPERIMENTS=False")
import shutil, glob
_lm_dir = os.path.join(FT_CONFIG["report_dir"], "test_predictions_lm")
_bak = os.path.join(FT_CONFIG["report_dir"], "test_predictions_lm_swa_ft")
for f in glob.glob(os.path.join(_lm_dir, "swa__*.csv")):
    os.remove(f)
_n = 0
for f in glob.glob(os.path.join(_bak, "swa__*.csv")):
    shutil.copy(f, os.path.join(_lm_dir, os.path.basename(f)))
    _n += 1
assert _n, f"Aucune sauvegarde sous {_bak} ?"
if "TOPUP_PIPES" in globals():
    TOPUP_PIPES.pop("swh_Latn", None)          # swa re-routé vers le joint
print(f"{_n} CSV swa fine-tunés restaurés ; relancer 12.3 (tout est présent : "
      "elle ne fera que réassembler submission_lm.csv).")


In [ ]:
# 14.3 — LM swa « web » : Wikipedia swahili (donnée externe PUBLIQUE et gratuite, autorisée par
if not RUN_LM_EXPERIMENTS:
    raise SystemExit("RUN_LM_EXPERIMENTS=False")
# les règles « External Tools & Data ») + notre texte train+valid. Notre LM swa ne connaît que
# l'agriculture scriptée ; le test swa est 100 % spontané et généraliste -> un LM à large
# couverture guide mieux le beam là où l'acoustique hésite. ~15-25 min.
import subprocess
import re as _re
subprocess.run(["pip", "install", "-q", "datasets"], check=False)
from datasets import load_dataset

_lmplz = "/content/kenlm/build/bin/lmplz"
_bbin = "/content/kenlm/build/bin/build_binary"
assert os.path.exists(_lmplz), "KenLM non compilé : exécuter 12.1b d'abord."
LM_WEB_DIR = os.path.join(FT_CONFIG["experiment_run"], "kenlm_models_web")
os.makedirs(LM_WEB_DIR, exist_ok=True)
LM_WEB_SW = os.path.join(LM_WEB_DIR, "sw.binary")
UNI_WEB_SW = os.path.join(LM_WEB_DIR, "sw_unigrams.txt")

if os.path.exists(LM_WEB_SW) and os.path.exists(UNI_WEB_SW):
    print("LM web swa déjà présent :", LM_WEB_SW)
else:
    print("Téléchargement Wikipedia swahili…")
    _wiki = load_dataset("wikimedia/wikipedia", "20231101.sw", split="train")
    _sents = [str(t) for t in manifest[manifest.split.isin(["train", "valid"])
                                       & (manifest.language == "sw")]["text_norm"]]
    _seen = set()
    for _a in _wiki["text"]:
        for _s in _re.split(r"[.!?\n]+", _a):
            _s = normalize_text(_s)
            if len(_s.split()) >= 3 and _s not in _seen:
                _seen.add(_s)
                _sents.append(_s)
    print(f"{len(_sents)} phrases (dont {len(_seen)} de Wikipedia)")
    _txt, _arpa = "/content/lm_web_sw.txt", "/content/lm_web_sw.arpa"
    with open(_txt, "w", encoding="utf-8") as f:
        f.write("\n".join(_sents))
    with open(_txt, "rb") as fi, open(_arpa, "wb") as fo:
        _r = subprocess.run([_lmplz, "-o", "5", "--discount_fallback", "-S", "30%"],
                            stdin=fi, stdout=fo, stderr=subprocess.PIPE)
    assert _r.returncode == 0, _r.stderr[-600:].decode(errors="replace")
    subprocess.run([_bbin, _arpa, LM_WEB_SW], check=True)
    from collections import Counter
    _cnt = Counter(w for s in _sents for w in s.split())
    with open(UNI_WEB_SW, "w", encoding="utf-8") as f:      # 200k mots les plus fréquents
        f.write("\n".join(w for w, _ in _cnt.most_common(200_000)))
    print(f"LM web swa -> {LM_WEB_SW} ({os.path.getsize(LM_WEB_SW)//2**20} Mo), "
          f"vocab {min(len(_cnt), 200_000)} mots")
print("Suite : 14.4 (contrôle + bascule des réglages swa).")


In [ ]:
# 14.4 — Brancher le LM web sur swa : contrôle sur local_test (PROXY imparfait : il est scripté/
if not RUN_LM_EXPERIMENTS:
    raise SystemExit("RUN_LM_EXPERIMENTS=False")
# agricole, le vrai juge du spontané est le LEADERBOARD), puis bascule des réglages swa.
# Critère : si le LM web tient à <1 pt du LM actuel sur le proxy, il mérite l'A/B en soumission
# (son avantage attendu est précisément hors domaine, invisible sur le proxy).
import json, glob, jiwer
from pyctcdecode import build_ctcdecoder
assert "LM_WEB_SW" in globals() and os.path.exists(LM_WEB_SW), "Exécuter 14.3 d'abord."
assert "capture_logprobs" in globals() and "LABELS" in globals(), "Exécuter 12.2c d'abord."

_uni_web = open(UNI_WEB_SW, encoding="utf-8").read().splitlines()
_val = (manifest[(manifest.split == "local_test") & (manifest.language == "sw")]
        .sample(frac=1.0, random_state=FT_CONFIG["seed"]).head(200))
refs = _val["text_norm"].tolist()
lp = capture_logprobs(_val["curated_audio_path"].tolist(), omni_lang("sw"))

_tun_p = os.path.join(FT_CONFIG["report_dir"], "kenlm_tuning.json")
_tun = json.load(open(_tun_p))
_cur = _tun["sw"]
_dec0 = build_ctcdecoder(LABELS, _cur["lm_bin"], unigrams=LM_UNIGRAMS["sw"],
                         alpha=float(_cur["alpha"]), beta=float(_cur["beta"]))
w_cur = jiwer.wer(refs, [normalize_text(_dec0.decode(x)) for x in lp])
print(f"swa ACTUEL ({_cur['lm']}, a={_cur['alpha']}, b={_cur['beta']}) : "
      f"WER local_test {w_cur:.4f}")
best = None
for a in (0.3, 0.5, 0.65):
    for b in (0.0, 1.0):
        _d = build_ctcdecoder(LABELS, LM_WEB_SW, unigrams=_uni_web, alpha=a, beta=b)
        w = jiwer.wer(refs, [normalize_text(_d.decode(x)) for x in lp])
        if best is None or w < best[0]:
            best = (w, a, b)
print(f"swa LM WEB : meilleur {best[0]:.4f} (a={best[1]}, b={best[2]})")

if best[0] <= w_cur + 0.01:
    _tun["sw"] = {"alpha": best[1], "beta": best[2], "beam": 100, "lm": "web",
                  "lm_bin": LM_WEB_SW, "wer_best": round(best[0], 4),
                  "wer_avant_polish": _cur.get("wer_best")}
    save_json(_tun, _tun_p)
    LM_BEST["sw"] = _tun["sw"]
    LM_PATHS["sw"] = LM_WEB_SW
    LM_UNIGRAMS["sw"] = _uni_web
    _lm_dir = os.path.join(FT_CONFIG["report_dir"], "test_predictions_lm")
    _n = 0
    for f in glob.glob(os.path.join(_lm_dir, "swa__*.csv")):
        os.remove(f)
        _n += 1
    print(f"{_n} CSV swa invalidés -> RELANCER 12.3 puis SOUMETTRE (A/B leaderboard).")
    print("(Si le score se dégrade : remettre _tun['sw'] depuis kenlm_tuning_pre_polish/12.2c "
          "n'est PAS automatique — me prévenir, je te donne la restauration.)")
else:
    print("LM web nettement pire même sur le proxy scripté : ne pas déployer, me montrer ces chiffres.")


In [ ]:
# 14.5 — Corpus externes PUBLICS + LMs v4 pour sw/kik/kln/luo/som (deep-research 2026-07-06).
if not RUN_LM_EXPERIMENTS:
    raise SystemExit("RUN_LM_EXPERIMENTS=False")
# Sources (autorisées : publiques/gratuites) : thinkKenya kln+luo (~35k/29k phrases, CC-BY-4.0),
# Bibles eBible kik/luo/som (~700k mots), Leipzig som (newscrawl 100K), Wikipedia sw (14.3).
# Recette : in-domain TRAIN dupliqué x4 + externe dédupliqué ; AUCUN texte de valid -> le
# réglage 14.6 se fait sur VALID (le proxy prouvé), sans fuite. mas : pas de source fiable.
import subprocess, zipfile, tarfile, json, glob
import io as _io
import re as _re
subprocess.run(["pip", "install", "-q", "datasets", "requests"], check=False)
import requests
from datasets import load_dataset

# Authentification HF (thinkKenya = dataset gated ; conditions à accepter UNE fois sur sa page
# web) : token lu depuis hf_token.json sur le Drive ({"name":"HF_TOKEN","key":"hf_..."}).
try:
    _tk = ["/content/afrivoices_project", os.path.join(DRIVE_ROOT, "hf_token.json")]
    _tp = next((p for p in _tk if os.path.exists(p)), None)
    if _tp is None:
        _hits = glob.glob("/content/persistent_storage/*/hf_token.json")
        _tp = _hits[0] if _hits else None
    if _tp:
        from huggingface_hub import login
        login(token=json.load(open(_tp))["key"])
        print("HF authentifié via", _tp)
    else:
        print("⚠️ hf_token.json introuvable sur le Drive : thinkKenya (gated) échouera.")
except Exception as _e:
    print(f"⚠️ login HF : {_e!r} — thinkKenya échouera probablement.")

_lmplz = "/content/kenlm/build/bin/lmplz"
_bbin = "/content/kenlm/build/bin/build_binary"
assert os.path.exists(_lmplz), "KenLM non compilé : exécuter 12.1b d'abord."
LM_V4_DIR = os.path.join(FT_CONFIG["experiment_run"], "kenlm_models_v4")
os.makedirs(LM_V4_DIR, exist_ok=True)
V4_LANGS = ["sw", "kik", "kln", "luo", "som"]

def _clean_ext(s):
    s = _re.sub(r"\d+", " ", str(s))          # les chiffres n'existent pas dans nos réfs ASR
    s = normalize_text(s)
    return s if len(s.split()) >= 3 else ""

_ext = {l: [] for l in V4_LANGS}

# 1) thinkKenya — paires parallèles kalenjin/dholuo <-> kiswahili
def _pick_text(row, cols, keys):
    for k in keys:
        if k in cols and isinstance(row.get(k), str):
            return row[k]
    for c in cols:
        v = row.get(c)
        if isinstance(v, dict):
            for k in keys:
                if isinstance(v.get(k), str):
                    return v[k]
    return None

for _lang, _cfg, _keys in (("kln", "kln_swa", ("kln", "kalenjin", "kal")),
                           ("luo", "luo_swa", ("luo", "dholuo"))):
    try:
        _ds = load_dataset("thinkKenya/kenyan-low-resource-language-data", _cfg)
        _n0 = len(_ext[_lang])
        for _split in _ds:
            _cols = _ds[_split].column_names
            for _row in _ds[_split]:
                _t = _pick_text(_row, _cols, _keys)
                if _t is None:
                    raise SystemExit(f"thinkKenya {_cfg}: colonne {_lang} introuvable dans "
                                     f"{_cols} — me montrer ce message et une ligne d'exemple.")
                _s = _clean_ext(_t)
                if _s:
                    _ext[_lang].append(_s)
        print(f"  thinkKenya {_lang}: {len(_ext[_lang]) - _n0} phrases")
    except SystemExit:
        raise
    except Exception as e:
        print(f"⚠️ thinkKenya {_cfg} : {e!r} (non bloquant, me le signaler)")

# 2) eBible — Bibles complètes redistributables (kik/luo/som)
for _lang, _tid in (("kik", "kik"), ("luo", "luo"), ("som", "som")):
    try:
        _r = requests.get(f"https://ebible.org/Scriptures/{_tid}_readaloud.zip", timeout=180)
        _r.raise_for_status()
        _z = zipfile.ZipFile(_io.BytesIO(_r.content))
        _n0 = len(_ext[_lang])
        for _nm in _z.namelist():
            if not _nm.lower().endswith(".txt"):
                continue
            for _line in _z.read(_nm).decode("utf-8", errors="replace").splitlines():
                _s = _clean_ext(_line)
                if _s:
                    _ext[_lang].append(_s)
        print(f"  eBible {_lang}: {len(_ext[_lang]) - _n0} lignes")
    except Exception as e:
        print(f"⚠️ eBible {_lang} : {e!r} (non bloquant, me le signaler)")

# 3) Leipzig — somali newscrawl 100K
try:
    _r = requests.get("https://downloads.wortschatz-leipzig.de/corpora/"
                      "som_newscrawl_2011_100K.tar.gz", timeout=300)
    _r.raise_for_status()
    _t = tarfile.open(fileobj=_io.BytesIO(_r.content), mode="r:gz")
    _n0 = len(_ext["som"])
    for _m in _t.getmembers():
        if _m.name.endswith("-sentences.txt"):
            for _line in _t.extractfile(_m).read().decode("utf-8", errors="replace").splitlines():
                _s = _clean_ext(_line.split("\t", 1)[-1])
                if _s:
                    _ext["som"].append(_s)
    print(f"  Leipzig som: {len(_ext['som']) - _n0} phrases")
except Exception as e:
    print(f"⚠️ Leipzig som : {e!r} (non bloquant)")

# 4) Wikipedia sw (SANS texte valid, contrairement au LM « web » de 14.3)
try:
    _wiki = load_dataset("wikimedia/wikipedia", "20231101.sw", split="train")
    for _a in _wiki["text"]:
        for _s0 in _re.split(r"[.!?\n]+", _a):
            _s = _clean_ext(_s0)
            if _s:
                _ext["sw"].append(_s)
    print(f"  Wikipedia sw: {len(_ext['sw'])} phrases")
except Exception as e:
    print(f"⚠️ Wikipedia sw : {e!r} (non bloquant)")

# 5) Construction des LMs v4 : train x4 + externe dédupliqué
from collections import Counter
LM_PATHS_V4, UNI_V4 = {}, {}
for lang in V4_LANGS:
    _extl = sorted(set(_ext[lang]))
    if not _extl:
        print(f"⚠️ {lang}: aucun texte externe -> v4 sautée pour cette langue")
        continue
    _bin = os.path.join(LM_V4_DIR, f"{lang}.binary")
    _unif = os.path.join(LM_V4_DIR, f"{lang}_unigrams.txt")
    _ind = [str(t) for t in manifest[(manifest.split == "train")
                                     & (manifest.language == lang)]["text_norm"]]
    # reconstruction si le binaire manque OU si le corpus externe a changé (ex. thinkKenya
    # débloqué après un premier passage sans authentification)
    _metaf = os.path.join(LM_V4_DIR, f"{lang}_meta.json")
    _old_n = json.load(open(_metaf)).get("externes", -1) if os.path.exists(_metaf) else -1
    if not (os.path.exists(_bin) and os.path.exists(_unif)) or _old_n != len(_extl):
        if _old_n >= 0 and _old_n != len(_extl):
            print(f"  {lang}: corpus externe {_old_n} -> {len(_extl)} phrases : reconstruction")
        _sents = _ind * 4 + _extl
        _txt, _arpa = f"/content/lm4_{lang}.txt", f"/content/lm4_{lang}.arpa"
        with open(_txt, "w", encoding="utf-8") as f:
            f.write("\n".join(_sents))
        def _run(order):
            with open(_txt, "rb") as fi, open(_arpa, "wb") as fo:
                return subprocess.run([_lmplz, "-o", str(order), "--discount_fallback",
                                       "-S", "30%"], stdin=fi, stdout=fo,
                                      stderr=subprocess.PIPE)
        _r = _run(5)
        if _r.returncode != 0:
            _r = _run(3)
        assert _r.returncode == 0, f"lmplz {lang}: {_r.stderr[-600:].decode(errors='replace')}"
        subprocess.run([_bbin, _arpa, _bin], check=True)
        _cnt = Counter(w for s in (_ind + _extl) for w in s.split())
        with open(_unif, "w", encoding="utf-8") as f:
            f.write("\n".join(w for w, _ in _cnt.most_common(200_000)))
        with open(_metaf, "w") as f:
            json.dump({"externes": len(_extl)}, f)
    LM_PATHS_V4[lang] = _bin
    UNI_V4[lang] = _unif
    print(f"  {lang}: LM v4 = {len(_ind)} in-domain x4 + {len(_extl)} externes "
          f"-> {os.path.getsize(_bin) // 2**20} Mo")
print("LMs v4 prêts -> 14.6 (arbitrage sur valid + seuils pyctcdecode).")


In [ ]:
# 14.6 — ARBITRE v4 : par langue (sw/kik/kln/luo/som), candidat A = config PRÉ-POLISH prouvée
if not RUN_LM_EXPERIMENTS:
    raise SystemExit("RUN_LM_EXPERIMENTS=False")
# (v1, celle de v6=0.38201) vs candidat B = LM v4 (grille alpha/beta) ; puis micro-grille des
# SEUILS pyctcdecode (token_min_logp/beam_prune_logp/beam — jamais réglés, défauts
# calibrés pour l'anglais) sur le vainqueur. Réglage sur VALID (aucun corpus v4 n'en contient).
# Écrit kenlm_tuning.json + invalide les CSV -> RELANCER 12.3 (version mise à jour !) puis
# SOUMETTRE. mas reste inchangée (config identique depuis v6).
import subprocess, json, glob
try:
    import kenlm, pyctcdecode
except ImportError:
    subprocess.run(["pip", "install", "-q", "pyctcdecode", "kenlm"], check=False)
    import kenlm, pyctcdecode
import jiwer
from pyctcdecode import build_ctcdecoder

assert "LM_PATHS_V4" in globals() and LM_PATHS_V4, "Exécuter 14.5 d'abord."
assert "capture_logprobs" in globals() and "LABELS" in globals(), "Exécuter 12.2d d'abord."

_prev_p = os.path.join(FT_CONFIG["report_dir"], "kenlm_tuning_pre_polish.json")
_base = json.load(open(_prev_p if os.path.exists(_prev_p)
                       else os.path.join(FT_CONFIG["report_dir"], "kenlm_tuning.json")))
_tun_p = os.path.join(FT_CONFIG["report_dir"], "kenlm_tuning.json")
_tun = json.load(open(_tun_p))
_V1_BIN = {l: os.path.join(FT_CONFIG["experiment_run"], "kenlm_models", f"{l}.binary")
           for l in LANGUAGES}
_uniA = {}                                     # unigrammes v1 : train seul
for l in LANGUAGES:
    _w = set()
    for t in manifest[(manifest.split == "train") & (manifest.language == l)]["text_norm"]:
        _w.update(str(t).split())
    _uniA[l] = sorted(_w)

ALPHAS = [0.3, 0.5, 0.65, 0.8]
BETAS = [0.0, 1.0, 2.0]
KNOBS = ((-8.0, -10.0, 100), (-12.0, -15.0, 100), (-8.0, -10.0, 250))
_val = (manifest[manifest.split == "valid"].sample(frac=1.0, random_state=FT_CONFIG["seed"])
        .groupby("language").head(200))
_changed = []
for lang in [l for l in ("sw", "kik", "kln", "luo", "som") if l in LM_PATHS_V4]:
    _t0 = time.time()
    sub = _val[_val.language == lang]
    refs = sub["text_norm"].tolist()
    lp = capture_logprobs(sub["curated_audio_path"].tolist(), omni_lang(lang))
    _uniB = open(UNI_V4[lang], encoding="utf-8").read().splitlines()

    # A — config pré-polish (v1)
    _pp = _base[lang]
    _decA = build_ctcdecoder(LABELS, _V1_BIN[lang], unigrams=_uniA[lang],
                             alpha=float(_pp["alpha"]), beta=float(_pp["beta"]))
    wA = jiwer.wer(refs, [normalize_text(_decA.decode(
        x, beam_width=int(_pp.get("beam", 100)))) for x in lp])

    # B — LM v4, grille alpha/beta
    bestB = None
    for a in ALPHAS:
        for b in BETAS:
            _d = build_ctcdecoder(LABELS, LM_PATHS_V4[lang], unigrams=_uniB, alpha=a, beta=b)
            w = jiwer.wer(refs, [normalize_text(_d.decode(x)) for x in lp])
            if bestB is None or w < bestB[0]:
                bestB = (w, a, b)

    if bestB[0] < wA - 0.002:
        _win = {"alpha": bestB[1], "beta": bestB[2], "beam": 100, "lm": "v4",
                "lm_bin": LM_PATHS_V4[lang], "uni_file": UNI_V4[lang],
                "wer_best": round(bestB[0], 4)}
        _dec = build_ctcdecoder(LABELS, LM_PATHS_V4[lang], unigrams=_uniB,
                                alpha=bestB[1], beta=bestB[2])
        _wwin = bestB[0]
    else:
        _win = {"alpha": float(_pp["alpha"]), "beta": float(_pp["beta"]),
                "beam": int(_pp.get("beam", 100)), "lm": "v1", "lm_bin": _V1_BIN[lang],
                "wer_best": round(wA, 4)}
        _dec = _decA
        _wwin = wA

    # seuils pyctcdecode sur le vainqueur
    for tml, bpl, bw in KNOBS:
        w = jiwer.wer(refs, [normalize_text(_dec.decode(
            x, beam_width=max(bw, _win["beam"]), token_min_logp=tml,
            beam_prune_logp=bpl)) for x in lp])
        if w < _wwin - 0.002:
            _wwin = w
            _win.update({"tml": tml, "bpl": bpl, "beam": max(bw, _win["beam"]),
                         "wer_best": round(w, 4)})

    _tun[lang] = _win
    _changed.append(lang)
    print(f"{lang}: pré-polish(v1) {wA:.4f} | v4 {bestB[0]:.4f} (a={bestB[1]},b={bestB[2]}) "
          f"-> retenu {_win['lm']} WER {_win['wer_best']}"
          f"{' + seuils ' + str((_win.get('tml'), _win.get('bpl'), _win['beam'])) if 'tml' in _win else ''}"
          f"  [{(time.time() - _t0) / 60:.1f} min]")
    del lp

save_json(_tun, _tun_p)
LM_BEST = _tun
LM_PATHS = {l: LM_BEST[l].get("lm_bin") or _V1_BIN[l] for l in LANGUAGES}
LM_UNIGRAMS = {}
for l in LANGUAGES:
    _uf = LM_BEST.get(l, {}).get("uni_file")
    if _uf and os.path.exists(_uf):
        LM_UNIGRAMS[l] = open(_uf, encoding="utf-8").read().splitlines()
    elif LM_BEST.get(l, {}).get("lm") == "v3":
        _w = set()
        for t in manifest[manifest.split.isin(["train", "valid"])
                          & (manifest.language == l)]["text_norm"]:
            _w.update(str(t).split())
        LM_UNIGRAMS[l] = sorted(_w)
    else:
        LM_UNIGRAMS[l] = _uniA[l]

_lm_dir = os.path.join(FT_CONFIG["report_dir"], "test_predictions_lm")
_MAN2DIR = {"sw": "swa"}
_n = 0
for l in _changed:
    for f in glob.glob(os.path.join(_lm_dir, f"{_MAN2DIR.get(l, l)}__*.csv")):
        os.remove(f)
        _n += 1
print(f"\n{_n} CSV invalidés ({_changed}). RELANCER 12.3 (VERSION MISE À JOUR — elle lit les "
      "seuils) puis SOUMETTRE. mas inchangée.")


In [ ]:
# 16.1 — swa SPONTANÉ : construire un dataset fairseq2 de ~120 h depuis Afrivoice_Swahili
if not RUN_SWA_SPONT_BUILD:
    raise SystemExit("RUN_SWA_SPONT_BUILD=False")
# (domaines health/education/financial/government — l'agriculture est déjà dans le joint).
# Par archive : manifest_N.jsonl <-> audio_N.tar.xz (~25-30 h). Sélection par durée depuis les
# manifests SEULS, puis téléchargement ciblé, décodage webm->FLAC 16k mono (ffmpeg, 12 threads),
# parquet fairseq2 direct (train + dev ~3 h disjoint par locuteur). Vérifie la DISJONCTION avec
# les IDs du test Kaggle (règle 4b). REPRENABLE par archive ; cache Drive final. ~45-60 min.
import subprocess, tarfile, json, glob, shutil, hashlib
import io as _io
import re as _re
import pyarrow as pa
import pyarrow.parquet as _pq
from concurrent.futures import ThreadPoolExecutor
from huggingface_hub import hf_hub_download, login

SWA_TARGET_HOURS = 120
SWA_DEV_HOURS = 3
_DOMAINS = ["health", "education", "financial", "government"]
_REPO = "DigitalUmuganda/Afrivoice_Swahili"

_tp = next((p for p in ("/content/afrivoices_project",
                        os.path.join(DRIVE_ROOT, "hf_token.json")) if os.path.exists(p)), None)
assert _tp, "hf_token.json introuvable"
login(token=json.load(open(_tp))["key"])

SWA_SPONT_VERSION = "v4_swa_speaker_audit_20260709"
SWA_SPONT_SPEC = {"version": SWA_SPONT_VERSION, "language": "sw",
                  "target_hours": SWA_TARGET_HOURS, "dev_hours": SWA_DEV_HOURS,
                  "seed": int(FT_CONFIG["seed"]), "domains": _DOMAINS}
SWA_SPONT_BUILD_ID = SWA_SPONT_VERSION + "_" + hashlib.sha256(
    json.dumps(SWA_SPONT_SPEC, sort_keys=True).encode()).hexdigest()[:12]
SPONT_ROOT = f"/content/topup_sw_spont_{SWA_SPONT_BUILD_ID}"
SPONT_PARQ = os.path.join(SPONT_ROOT, "version=0")
SPONT_AUDIT = os.path.join(SPONT_ROOT, "audit")
SPONT_DRIVE = os.path.join(DRIVE_ROOT, "dataset_caches",
                           f"topup_sw_spont_{SWA_SPONT_BUILD_ID}")
_marker = os.path.join(SPONT_DRIVE, "_COMPLETE.json")
if os.path.exists(_marker) and not os.path.exists(os.path.join(SPONT_ROOT, "_COMPLETE.json")):
    print("Cache Drive trouvé : copie locale…")
    shutil.rmtree(SPONT_ROOT, ignore_errors=True)
    subprocess.run(["cp", "-r", SPONT_DRIVE, SPONT_ROOT], check=True)

if not os.path.exists(os.path.join(SPONT_ROOT, "_COMPLETE.json")):
    # IDs du test Kaggle (préuve de disjonction, règle 4b)
    _test_ids = set()
    for _l, _sh, _n in test_shards:
        if _l == "swa":
            for _i in _pq.read_table(_sh, columns=["id"])["id"].to_pylist():
                _test_ids.add(str(_i).replace(".wav", "").replace(".webm", ""))
    print(f"{len(_test_ids)} IDs test swa chargés (contrôle de disjonction)")

    # Aucun locuteur du corpus curé ne doit réapparaître dans le nouveau cache.
    _curated_sw_speakers = set(manifest.loc[
        manifest.language.eq("sw"), "speaker_id"
    ].dropna().astype(str))
    print("Locuteurs swahili curés exclus :", len(_curated_sw_speakers))

    # Phase A — sélection par manifests (round-robin sur les domaines)
    _sel, _hours = [], 0.0
    for _chunk in range(0, 20):
        for _dom in _DOMAINS:
            if _hours >= SWA_TARGET_HOURS + SWA_DEV_HOURS:
                break
            _dtrain = f"{_dom}_swahili_train"
            try:
                _mf = hf_hub_download(_REPO, f"{_dtrain}/manifest_{_chunk}.jsonl",
                                      repo_type="dataset")
            except Exception:
                continue
            _rows = []
            for _line in open(_mf, encoding="utf-8"):
                try:
                    r = json.loads(_line)
                except Exception:
                    continue
                _txt = _re.sub(r"\[cs\]", " ", str(r.get("normalized_transcription")
                                                   or r.get("transcription") or ""))
                _txt = normalize_text(_txt)
                _dur = float(r.get("duration") or 0)
                if _txt and len(_txt.split()) >= 2 and 0.5 <= _dur <= 38.0:
                    _rows.append({"key": r["key"], "file": r["audio_filepath"],
                                  "spk": r.get("voice_creator_id") or r.get("uid") or "?",
                                  "text": _txt, "dur": _dur})
            _rows = [row for row in _rows
                     if str(row["spk"]) not in _curated_sw_speakers]
            if not _rows:
                continue
            assert not (_test_ids & {r["key"] for r in _rows}), \
                f"CHEVAUCHEMENT avec le test détecté dans {_dtrain}/chunk{_chunk} — STOP (règle 4b)"
            _h = sum(r["dur"] for r in _rows) / 3600
            _sel.append((_dtrain, _chunk, _rows, _h))
            _hours += _h
            print(f"  + {_dtrain}/chunk{_chunk}: {len(_rows)} clips, {_h:.1f} h "
                  f"(cumul {_hours:.1f} h) — disjonction test OK")
        if _hours >= SWA_TARGET_HOURS + SWA_DEV_HOURS:
            break

    # dev disjoint par locuteur (~SWA_DEV_HOURS)
    _spk_h = {}
    for _, _, _rows, _ in _sel:
        for r in _rows:
            _spk_h[r["spk"]] = _spk_h.get(r["spk"], 0.0) + r["dur"] / 3600
    _dev_spk, _acc = set(), 0.0
    for _s, _h in sorted(_spk_h.items(), key=lambda kv: kv[1]):
        if _acc >= SWA_DEV_HOURS:
            break
        _dev_spk.add(_s)
        _acc += _h
    print(f"dev spontané : {len(_dev_spk)} locuteurs, {_acc:.1f} h")

    # Phase B — téléchargement + conversion, REPRENABLE par archive
    def _to_flac(args):
        _path, _text, _spk, _sample_id = args
        try:
            _p = subprocess.run(["ffmpeg", "-v", "quiet", "-nostdin", "-i", _path, "-f", "f32le",
                                 "-ac", "1", "-ar", "16000", "pipe:1"],
                                capture_output=True, check=True)
            wav = np.frombuffer(_p.stdout, dtype=np.float32)
            if len(wav) < 3200:
                return None
            _b = _io.BytesIO()
            sf.write(_b, wav, 16000, format="FLAC")
            return {"text": _text, "audio_bytes": _b.getvalue(), "audio_size": len(wav),
                    "_dev": _spk in _dev_spk, "_sample_id": str(_sample_id),
                    "_speaker_id": str(_spk), "_duration_s": len(wav) / 16000.0}
        except Exception:
            return None

    _done_f = os.path.join(SPONT_ROOT, "chunks_done.json")
    _done = set(json.load(open(_done_f))) if os.path.exists(_done_f) else set()
    for _dtrain, _chunk, _rows, _h in _sel:
        _tag = f"{_dtrain}__c{_chunk}"
        if _tag in _done:
            print(f"  {_tag}: déjà converti")
            continue
        _t0 = time.time()
        _tarp = hf_hub_download(_REPO, f"{_dtrain}/audio/audio_{_chunk}.tar.xz",
                                repo_type="dataset")
        _ex = f"/content/_spont_extract_{_chunk}"
        shutil.rmtree(_ex, ignore_errors=True)
        with tarfile.open(_tarp, "r:xz") as _t:
            _t.extractall(_ex)
        _byname = {}
        for _root, _, _fs in os.walk(_ex):
            for _f in _fs:
                _byname[_f] = os.path.join(_root, _f)
        _jobs = [(_byname[r["file"]], r["text"], r["spk"], r["key"])
                 for r in _rows if r["file"] in _byname]
        with ThreadPoolExecutor(max_workers=12) as _pool:
            _out = [x for x in _pool.map(_to_flac, _jobs) if x]
        for _split in ("train", "dev"):
            _chosen = [x for x in _out if x["_dev"] == (_split == "dev")]
            _sub = [{"text": x["text"], "audio_bytes": x["audio_bytes"],
                     "audio_size": x["audio_size"]} for x in _chosen]
            _audit = [{"sample_id": x["_sample_id"], "speaker_id": x["_speaker_id"],
                       "split": _split, "duration_s": x["_duration_s"]} for x in _chosen]
            if not _sub:
                continue
            _dir = os.path.join(SPONT_PARQ, "corpus=afrivoices", f"split={_split}",
                                "language=swh_Latn")
            os.makedirs(_dir, exist_ok=True)
            os.makedirs(SPONT_AUDIT, exist_ok=True)
            _pq.write_table(pa.Table.from_pylist(_sub),
                            os.path.join(_dir, f"part-{_tag}.parquet"), row_group_size=100)
            # Un seul audit par archive contient train puis dev; 18.2 filtre par split.
            _audit_path = os.path.join(SPONT_AUDIT, f"part-{_tag}.parquet")
            if os.path.exists(_audit_path):
                _previous = _pq.read_table(_audit_path).to_pylist()
            else:
                _previous = []
            _pq.write_table(pa.Table.from_pylist(_previous + _audit), _audit_path,
                            row_group_size=100)
        _done.add(_tag)
        save_json(sorted(_done), _done_f)
        shutil.rmtree(_ex, ignore_errors=True)
        os.remove(_tarp)
        print(f"  {_tag}: {len(_out)}/{len(_rows)} clips convertis en "
              f"{(time.time()-_t0)/60:.1f} min")

    # TSV fairseq2 + texte pour LM + marqueur + cache Drive
    _tot = 0
    for _f in glob.glob(os.path.join(SPONT_PARQ, "corpus=afrivoices", "split=train", "*", "*.parquet")):
        _tot += sum(_pq.read_table(_f, columns=["audio_size"])["audio_size"].to_pylist())
    with open(os.path.join(SPONT_ROOT, "language_distribution_0.tsv"), "w") as f:
        f.write("corpus\tlanguage\thours\n")
        f.write(f"afrivoices\tswh_Latn\t{_tot / 16000 / 3600:.4f}\n")
    _txts = []
    for _, _, _rows, _ in _sel:
        _txts += [r["text"] for r in _rows if r["spk"] not in _dev_spk]
    with open(os.path.join(SPONT_ROOT, "swa_spont_text.txt"), "w", encoding="utf-8") as f:
        f.write("\n".join(_txts))
    _dev_tot = 0
    for _f in glob.glob(os.path.join(SPONT_PARQ, "corpus=afrivoices", "split=dev",
                                     "*", "*.parquet")):
        _dev_tot += sum(_pq.read_table(_f, columns=["audio_size"])["audio_size"].to_pylist())
    _complete = {"spec": SWA_SPONT_SPEC, "build_id": SWA_SPONT_BUILD_ID,
                 "stats": {"train_seconds": _tot / 16000,
                           "dev_seconds": _dev_tot / 16000}, "finished_at": now_iso()}
    save_json(_complete, os.path.join(SPONT_ROOT, "_COMPLETE.json"))
    print(f"train spontané : {_tot / 16000 / 3600:.1f} h — mise en cache Drive…")
    shutil.rmtree(SPONT_DRIVE, ignore_errors=True)
    subprocess.run(["cp", "-r", SPONT_ROOT, SPONT_DRIVE], check=True)

print("Dataset swa spontané V4 auditable prêt :", SPONT_PARQ, SWA_SPONT_BUILD_ID)
print("Suite : 16.2 (top-up spontané swa).")


In [ ]:
# Récupération de 16.1 : copie locale -> Drive, reprenable et vérifiée.
import json
import os
import shutil
import time
from pathlib import Path

import pyarrow.parquet as pq

assert "SPONT_ROOT" in globals(), "État 16.1 absent : SPONT_ROOT introuvable"
assert "SPONT_DRIVE" in globals(), "État 16.1 absent : SPONT_DRIVE introuvable"

_src_root = Path(SPONT_ROOT)
_dst_root = Path(SPONT_DRIVE)
_src_marker = _src_root / "_COMPLETE.json"
_dst_marker = _dst_root / "_COMPLETE.json"

assert _src_root.is_dir(), f"Cache local absent : {_src_root}"
assert _src_marker.is_file(), (
    "Le cache local n'est pas terminé : _COMPLETE.json absent. "
    "Il faudra alors relancer 16.1."
)

def _retry(operation, label, attempts=12):
    delay = 2.0
    last_error = None

    for attempt in range(1, attempts + 1):
        try:
            return operation()
        except OSError as exc:
            last_error = exc
            print(
                f"{label} — essai {attempt}/{attempts} échoué : {exc!r}; "
                f"reprise dans {delay:.1f}s"
            )
            time.sleep(delay)
            delay = min(delay * 1.7, 30.0)

    raise RuntimeError(f"Échec Drive persistant : {label}") from last_error


_retry(
    lambda: _dst_root.mkdir(parents=True, exist_ok=True),
    f"création {_dst_root}",
)

# Un cache incomplet ne doit jamais contenir le marqueur final.
if _dst_marker.exists():
    _retry(lambda: _dst_marker.unlink(), "suppression ancien marqueur _COMPLETE")

_source_files = sorted(
    path for path in _src_root.rglob("*")
    if path.is_file() and path.name != "_COMPLETE.json"
)

assert _source_files, "Aucun fichier à transférer"

def _destination(source):
    return _dst_root / source.relative_to(_src_root)

def _same_size(source, destination):
    try:
        return (
            destination.is_file()
            and destination.stat().st_size == source.stat().st_size
        )
    except OSError:
        return False

_remaining_bytes = sum(
    source.stat().st_size
    for source in _source_files
    if not _same_size(source, _destination(source))
)

try:
    _drive_free = shutil.disk_usage(DRIVE_ROOT).free
except OSError:
    _drive_free = 0

print(f"Source locale : {_src_root}")
print(f"Destination   : {_dst_root}")
print(f"Fichiers      : {len(_source_files)}")
print(f"Reste à copier: {_remaining_bytes / 2**30:.2f} Gio")

if _drive_free:
    print(f"Libre Drive   : {_drive_free / 2**30:.2f} Gio")
    assert _drive_free >= _remaining_bytes + 5 * 2**30, (
        "Espace Drive insuffisant : libérer au moins "
        f"{(_remaining_bytes + 5 * 2**30) / 2**30:.1f} Gio"
    )

def _copy_resumable(source, destination, attempts=12):
    source_size = source.stat().st_size

    if _same_size(source, destination):
        return "déjà présent"

    temporary = Path(str(destination) + ".part")
    delay = 2.0
    last_error = None

    for attempt in range(1, attempts + 1):
        try:
            destination.parent.mkdir(parents=True, exist_ok=True)

            if _same_size(source, destination):
                return "déjà présent"

            offset = temporary.stat().st_size if temporary.exists() else 0

            if offset > source_size:
                temporary.unlink()
                offset = 0

            with source.open("rb") as input_file:
                input_file.seek(offset)

                with temporary.open("ab") as output_file:
                    while True:
                        block = input_file.read(32 * 1024**2)
                        if not block:
                            break
                        output_file.write(block)
                    output_file.flush()

            copied_size = temporary.stat().st_size
            if copied_size != source_size:
                raise IOError(
                    f"copie partielle : {copied_size}/{source_size} octets"
                )

            os.replace(temporary, destination)

            if not _same_size(source, destination):
                raise IOError("taille différente après publication")

            return "copié"

        except (OSError, IOError) as exc:
            last_error = exc
            print(
                f"  reprise {source.name} — essai {attempt}/{attempts}: "
                f"{exc!r}"
            )
            time.sleep(delay)
            delay = min(delay * 1.7, 30.0)

    raise RuntimeError(f"Impossible de copier {source}") from last_error

for index, source in enumerate(_source_files, start=1):
    destination = _destination(source)
    status = _copy_resumable(source, destination)
    print(
        f"[{index:03d}/{len(_source_files):03d}] "
        f"{status:12s} {source.relative_to(_src_root)}"
    )

# Vérification des tailles et des métadonnées parquet.
_validation_errors = []

for source in _source_files:
    destination = _destination(source)

    if not _same_size(source, destination):
        _validation_errors.append(
            f"taille incorrecte : {destination.relative_to(_dst_root)}"
        )
        continue

    if source.suffix == ".parquet":
        try:
            source_rows = pq.ParquetFile(source).metadata.num_rows
            destination_rows = _retry(
                lambda path=destination: pq.ParquetFile(path).metadata.num_rows,
                f"lecture parquet {destination.name}",
            )

            if source_rows != destination_rows:
                _validation_errors.append(
                    f"nombre de lignes différent : "
                    f"{destination.relative_to(_dst_root)}"
                )
        except Exception as exc:
            _validation_errors.append(
                f"parquet illisible : {destination} — {exc!r}"
            )

assert not _validation_errors, "\n".join(_validation_errors[:20])

# Le marqueur est copié en dernier : il certifie que le cache Drive est complet.
_copy_resumable(_src_marker, _dst_marker)

_local_complete = json.load(open(_src_marker, encoding="utf-8"))
_drive_complete = _retry(
    lambda: json.load(open(_dst_marker, encoding="utf-8")),
    "lecture du marqueur Drive",
)

assert _local_complete == _drive_complete, "Marqueur Drive différent du local"

print()
print("✅ Cache Swahili copié et vérifié sur Drive")
print("SPONT_DRIVE =", SPONT_DRIVE)
print("Train =", round(
    _drive_complete["stats"]["train_seconds"] / 3600, 2
), "heures")
print("Dev =", round(
    _drive_complete["stats"]["dev_seconds"] / 3600, 2
), "heures")

In [ ]:
# 16.2 — TOP-UP SPONTANÉ swa : 2000 steps depuis la carte ft5000 sur les 145 h spontanées
if not RUN_SWA_SPONT_TRAINING:
    raise SystemExit("RUN_SWA_SPONT_TRAINING=False")
# (16.1), validation pendant l'entraînement sur le DEV SPONTANÉ (split dev du parquet).
# Checkpoints à 1000/1500/2000 -> 16.3 arbitre. ~4h20 (A100). REPRENABLE (même cellule).
SPONT_STEPS = 2000
assert "TOPUP_SEED_CARD" in globals(), "Exécuter 13.1 d'abord (carte ft5000 + sonde, ~3 min)."
SPONT_ROOT = "/content/topup_swa_spont"
SPONT_PARQ = os.path.join(SPONT_ROOT, "version=0")
assert os.path.exists(os.path.join(SPONT_ROOT, "_COMPLETE")), \
    "Dataset spontané absent : exécuter 16.1 (restaure depuis le cache Drive si déjà construit)."

# libérer le GPU du noyau (cf. 13.2 : la recette a besoin des 40 GB entiers)
import gc
for _v in ("_pipe_joint", "_pipe_lm", "_pipe"):
    if _v in globals():
        del globals()[_v]
if "TOPUP_PIPES" in globals():
    TOPUP_PIPES.clear()
gc.collect()
torch.cuda.empty_cache()
print(f"GPU noyau libéré : {torch.cuda.memory_allocated() / 2**30:.1f} GB restants alloués")

write_dataset_card(data_root=SPONT_PARQ)   # 2.4 remettra la carte sur le dataset complet
_s_tsv = os.path.join(SPONT_ROOT, "language_distribution_0.tsv")
_s_out = os.path.join(FT_CONFIG["experiment_run"], "topup_swa_spont")
_s_yaml = write_recipe_yaml(os.path.join(FT_CONFIG["experiment_run"], "topup_swa_spont_recipe.yaml"),
                            TOPUP_SEED_CARD, _s_tsv, num_steps=SPONT_STEPS,
                            grad_accum=FT_CONFIG["grad_accum"],
                            max_num_elements=FT_CONFIG["max_num_elements"], ckpt_every=500)
rc, _log = run_recipe(_s_out, _s_yaml, "topup_swa_spont.log",
                      wall_limit_h=FT_CONFIG["max_wall_hours_per_session"])
print("Code :", rc, "| dernier checkpoint :", latest_ckpt(_s_out))
print("Suite : 16.3 (double thermomètre : dev spontané + valid agricole).")


In [ ]:
# 16.3 — DOUBLE THERMOMÈTRE swa : joint vs checkpoints spontanés (1000/1500/2000), mesurés
if not RUN_SWA_SPONT_TRAINING:
    raise SystemExit("RUN_SWA_SPONT_TRAINING=False")
# avec le LM/réglages sw ACTUELS sur (a) 200 clips du DEV SPONTANÉ (16.1) et (b) 200 clips du
# valid agricole. Adoption si le spontané gagne SANS régression >1 pt sur l'agricole (le test
# swa est 100 % spontané, mais prudence). Écrit topup_choice_sw.json -> 13.4 -> 12.3. ~30 min.
import subprocess, json, glob
try:
    import kenlm, pyctcdecode
except ImportError:
    subprocess.run(["pip", "install", "-q", "pyctcdecode", "kenlm"], check=False)
    import kenlm, pyctcdecode
import jiwer
import pyarrow.parquet as _pq
from pathlib import Path
from pyctcdecode import build_ctcdecoder

SPONT_ROOT = "/content/topup_swa_spont"
SPONT_PARQ = os.path.join(SPONT_ROOT, "version=0")
assert os.path.exists(os.path.join(SPONT_ROOT, "_COMPLETE")), "Exécuter 16.1 d'abord."

def _labels_from(pipe):
    _tkm = getattr(pipe.tokenizer, "_model", None)
    for _m in ("index_to_token", "id_to_piece", "IdToPiece"):
        if _tkm is not None and hasattr(_tkm, _m):
            try:
                L = [str(getattr(_tkm, _m)(i)) for i in range(pipe.tokenizer.vocab_info.size)]
                assert len(L) == len(set(L))
                L[0] = ""
                return L
            except Exception:
                pass
    raise SystemExit("Vocab SentencePiece inaccessible.")
def _capture(pipe, inputs, omni_code):
    caps = []
    _orig = pipe.model.forward
    def _spy(src, bl, *a, **kw):
        out = _orig(src, bl, *a, **kw)
        lg = out[0] if isinstance(out, tuple) else out
        caps.append(torch.log_softmax(lg[0].detach().float(), -1).cpu().numpy())
        return out
    pipe.model.forward = _spy
    try:
        pipe.transcribe(inputs, lang=[omni_code] * len(inputs), batch_size=1)
    finally:
        pipe.model.forward = _orig
    return caps

# jeu (a) : dev spontané — 200 clips depuis le parquet
_dev_rows = []
for _f in sorted(glob.glob(os.path.join(SPONT_PARQ, "corpus=afrivoices", "split=dev",
                                        "*", "*.parquet"))):
    _dev_rows += _pq.read_table(_f).to_pylist()
import random as _rnd
_rnd.Random(FT_CONFIG["seed"]).shuffle(_dev_rows)
_dev_rows = _dev_rows[:200]
_dev_in = [{"waveform": sf.read(io.BytesIO(r["audio_bytes"]), dtype="float32")[0],
            "sample_rate": 16000} for r in _dev_rows]
_dev_refs = [r["text"] for r in _dev_rows]
# jeu (b) : valid agricole (curation)
_agr = (manifest[(manifest.split == "valid") & (manifest.language == "sw")]
        .sample(frac=1.0, random_state=FT_CONFIG["seed"]).head(200))
_agr_refs = _agr["text_norm"].tolist()

# LM/réglages sw actuels (v4 Wikipedia, arbitrés en 14.6)
_tun = json.load(open(os.path.join(FT_CONFIG["report_dir"], "kenlm_tuning.json")))
_cfg = _tun["sw"]
_uf = _cfg.get("uni_file")
_uni = (open(_uf, encoding="utf-8").read().splitlines() if _uf and os.path.exists(_uf)
        else sorted({w for t in manifest[(manifest.split == "train")
                                         & (manifest.language == "sw")]["text_norm"]
                     for w in str(t).split()}))
_dec = None                                  # construit au 1er tour (LABELS garanti)
_bw = int(_cfg.get("beam", 100))

_s_out = os.path.join(FT_CONFIG["experiment_run"], "topup_swa_spont")
_steps = sorted((d for d in Path(_s_out).rglob("step_*")
                 if d.is_dir() and not d.name.endswith(".tmp") and (d / "model").exists()),
                key=lambda d: int(d.name.split("_")[1]))
assert _steps, f"Aucun checkpoint sous {_s_out} : exécuter 16.2 d'abord."
_cands = [("joint", latest_ckpt(FT_CONFIG["output_dir"]))] + [(d.name, str(d)) for d in _steps]

_res = {}
for _name, _ck in _cands:
    _pipe = load_finetuned_pipeline(_ck)
    if "LABELS" not in globals():
        LABELS = _labels_from(_pipe)
    if _dec is None:
        _dec = build_ctcdecoder(LABELS, _cfg["lm_bin"], unigrams=_uni,
                                alpha=float(_cfg["alpha"]), beta=float(_cfg["beta"]))
    _lp_dev = _capture(_pipe, _dev_in, omni_lang("sw"))
    w_dev = jiwer.wer(_dev_refs, [normalize_text(_dec.decode(x, beam_width=_bw))
                                  for x in _lp_dev])
    _lp_agr = _capture(_pipe, _agr["curated_audio_path"].tolist(), omni_lang("sw"))
    w_agr = jiwer.wer(_agr_refs, [normalize_text(_dec.decode(x, beam_width=_bw))
                                  for x in _lp_agr])
    _res[_name] = {"ckpt": _ck, "wer_spont": round(w_dev, 4), "wer_agri": round(w_agr, 4)}
    print(f"sw/{_name}: SPONTANÉ {w_dev:.4f} | agricole {w_agr:.4f}")
    del _pipe, _lp_dev, _lp_agr
    torch.cuda.empty_cache()

_best = min(_res, key=lambda n: _res[n]["wer_spont"])
_dspont = _res["joint"]["wer_spont"] - _res[_best]["wer_spont"]
_dagri = _res[_best]["wer_agri"] - _res["joint"]["wer_agri"]
if _best != "joint" and _dspont > 0.005 and _dagri < 0.01:
    save_json({"language": "sw", "chosen": _best, "ckpt": _res[_best]["ckpt"],
               "wer_lm": _res[_best]["wer_spont"], "all": _res},
              os.path.join(FT_CONFIG["report_dir"], "topup_choice_sw.json"))
    print(f"\n✅ {_best} adopté : spontané {_dspont:+.4f}, agricole {_dagri:+.4f} "
          f"-> exécuter 13.4 (TOPUP_LANG_EVAL='sw') puis 12.3, puis SOUMETTRE.")
else:
    print(f"\n⚠️ Pas d'adoption (best={_best}, gain spontané {_dspont:+.4f}, "
          f"régression agricole {_dagri:+.4f}) — me montrer le tableau complet.")


In [ ]:
# 16.3b — V3 : invariants texte, cache et split spontané
# QA_PATCH: CURATED_OVERLAP_GUARD
import hashlib, unicodedata

SPONT_DATA_VERSION = "v3_textfix_longdev_20260709"
SPONT_DEV_HASH_FRACTION = 0.10
SPONT_LONG_DEV_TARGET_HOURS = 3.0
SPONT_LONG_MIN_S, SPONT_LONG_MAX_S = 39.5, 600.0
SPONT_LONG_MIN_WORDS, SPONT_LONG_MIN_WPS = 5, 0.10

_KIK_MOJ_MAP = {}
for _char in "ĩũĨŨ":
    _bad = _char.encode("utf-8").decode("latin-1")
    _KIK_MOJ_MAP[_bad] = _char
    _KIK_MOJ_MAP[_bad.lower()] = _char.lower()
_KIK_BAD_MARKERS = ("Ã", "Â", "Å", "Ä", "å©", "ä©", "â€", "ðŸ", "�")

def _moji_badness(value):
    value = str(value)
    return (sum(4 * value.count(marker) for marker in _KIK_BAD_MARKERS)
            + sum(unicodedata.category(ch) == "Cc" and ch not in "\n\t\r"
                  for ch in value))

def repair_kikuyu_source(raw):
    value = str(raw or "")
    for _ in range(3):
        before = value
        for bad, good in _KIK_MOJ_MAP.items():
            value = value.replace(bad, good)
        candidates = [value]
        for encoding in ("latin-1", "cp1252"):
            try:
                candidates.append(value.encode(encoding).decode("utf-8"))
            except (UnicodeEncodeError, UnicodeDecodeError):
                pass
        value = min(candidates, key=lambda text: (_moji_badness(text), len(text)))
        if value == before:
            break
    return unicodedata.normalize("NFC", value)

assert repair_kikuyu_source("kÅ©handa") == "kũhanda"
assert repair_kikuyu_source("mÄ©aka") == "mĩaka"
assert repair_kikuyu_source("mũndũ") == "mũndũ"

def _speaker_is_dev(lang, speaker, seed=42):
    if not speaker or speaker == "__UNKNOWN__":
        return False
    number = int(hashlib.sha256(f"{seed}:{lang}:{speaker}".encode()).hexdigest()[:16], 16)
    return number / 2**64 < SPONT_DEV_HASH_FRACTION

def _spont_layout(lang, target_h, dev_h):
    spec = {
        "schema": CACHE_SCHEMA_VERSION,
        "version": SPONT_DATA_VERSION,
        "language": lang,
        "target_hours": float(target_h),
        "dev_hours": float(dev_h),
        "dev_fraction": SPONT_DEV_HASH_FRACTION,
        "long_target_hours": SPONT_LONG_DEV_TARGET_HOURS,
        "long_min_s": SPONT_LONG_MIN_S,
        "long_max_s": SPONT_LONG_MAX_S,
        "seed": int(FT_CONFIG["seed"]),
        "normalization_version": NORMALIZATION_VERSION,
    }
    fingerprint = hashlib.sha256(json.dumps(spec, sort_keys=True).encode()).hexdigest()[:12]
    build_id = f"{SPONT_DATA_VERSION}_{fingerprint}"
    root = f"/content/topup_{lang}_spont_{build_id}"
    drive = os.path.join(DRIVE_ROOT, "dataset_caches", f"topup_{lang}_spont_{build_id}")
    return spec, build_id, root, drive

def _atomic_parquet(table, path, row_group_size=100):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    temporary = path + f".tmp-{os.getpid()}"
    _pq.write_table(table, temporary, row_group_size=row_group_size)
    os.replace(temporary, path)

print("Helpers spontanés V3 prêts :", SPONT_DATA_VERSION)


In [ ]:
# Correctif robuste du mojibake kikuyu — à exécuter après 16.3b
import re
import unicodedata

# Construction automatique des variantes UTF-8 mal décodées en Latin-1/CP1252.
_KIK_ROBUST_MAP = {}

_codepoints = (
    list(range(0x00A0, 0x0250)) +   # Latin-1 + Latin étendu
    list(range(0x1E00, 0x1F00)) +   # Latin étendu additionnel
    list(range(0x2000, 0x2070))      # apostrophes, guillemets, tirets…
)

for codepoint in _codepoints:
    correct = chr(codepoint)
    encoded = correct.encode("utf-8")

    for wrong_encoding in ("latin-1", "cp1252"):
        try:
            broken = encoded.decode(wrong_encoding)
        except (UnicodeDecodeError, UnicodeEncodeError):
            continue

        if broken != correct:
            _KIK_ROBUST_MAP[broken] = correct

            # Certaines sources ont ensuite été mises en minuscules.
            _KIK_ROBUST_MAP.setdefault(
                broken.lower(),
                correct.lower(),
            )

# Cas observés explicitement dans Afrivoice.
_KIK_ROBUST_MAP.update({
    "Ä©": "ĩ",
    "ä©": "ĩ",
    "Å©": "ũ",
    "å©": "ũ",
    "Ä¨": "Ĩ",
    "ä¨": "ĩ",
    "Å¨": "Ũ",
    "å¨": "ũ",
    "â€™": "’",
    "â€˜": "‘",
    "â€œ": "“",
    "â€": "”",
    "â€“": "–",
    "â€”": "—",
    "â€¦": "…",
    "Â ": " ",
})

_KIK_ROBUST_PATTERN = re.compile(
    "|".join(
        re.escape(key)
        for key in sorted(
            _KIK_ROBUST_MAP,
            key=len,
            reverse=True,
        )
    )
)

_KIK_RESIDUAL_MARKERS = (
    "Ã",
    "Â",
    "Ä",
    "Å",
    "â€",
    "ðŸ",
    "�",
)

_KIK_RESIDUAL_PATTERN = re.compile(
    "|".join(re.escape(marker) for marker in _KIK_RESIDUAL_MARKERS)
)

def repair_kikuyu_source(raw):
    value = str(raw or "")

    # Plusieurs passages gèrent aussi les doubles encodages.
    for _ in range(5):
        previous = value
        value = _KIK_ROBUST_PATTERN.sub(
            lambda match: _KIK_ROBUST_MAP[match.group(0)],
            value,
        )

        if value == previous:
            break

    # Les rares fragments réellement irrécupérables sont remplacés par un espace.
    value = _KIK_RESIDUAL_PATTERN.sub(" ", value)

    # Suppression des contrôles provenant d'un mauvais décodage CP1252.
    value = "".join(
        " " if unicodedata.category(character) == "Cc"
        and character not in "\n\t\r"
        else character
        for character in value
    )

    value = unicodedata.normalize("NFC", value)
    value = re.sub(r"[ \t]+", " ", value)

    return value.strip()

# Tests minimaux
assert repair_kikuyu_source("kÅ©handa") == "kũhanda"
assert repair_kikuyu_source("mÄ©aka") == "mĩaka"
assert repair_kikuyu_source("HÄ©ndÄ© Ä©rÄ©a") == "Hĩndĩ ĩrĩa"
assert repair_kikuyu_source("NÄ©Å©rakora atÄ©") == "Nĩũrakora atĩ"

_test = repair_kikuyu_source(
    "Ici nÄ© tuta iria ihÅ©thÄ©kaga kÅ©handÄ©ra mÅ©no"
)
assert _moji_badness(_test) == 0, _test

print("✅ Réparateur kikuyu robuste installé")
print(_test)

In [ ]:
# 16.4 — V3 : données spontanées propres, textes longs et vrai dev long
# QA_PATCH: LONG_TEXT_RETENTION
# QA_PATCH: REAL_LONG_DEV
require_state("SPONT_DATA_VERSION", "test_shards", "normalize_text",
              "repair_kikuyu_source", "omni_lang", "FT_CONFIG")
SPONT_LANG = "mas"                 # kln / mas / som / kik / luo
SPONT_TARGET_HOURS = 80.0
SPONT_DEV_HOURS = 2.5
SPONT_HARVEST_ALL_TEXT_SHARDS = True
if not RUN_SPONT_BUILD:
    raise SystemExit("RUN_SPONT_BUILD=False : activer seulement pour reconstruire une langue.")
assert SPONT_LANG in ("kln", "mas", "som", "kik", "luo")

import glob, io as _io, shutil, subprocess
import pyarrow as pa
import pyarrow.parquet as _pq
from concurrent.futures import ThreadPoolExecutor
from huggingface_hub import HfApi, hf_hub_download, login

os.environ["HF_HUB_DISABLE_XET"] = "1"
_repo = "MCAA1-MSU/anv_data_ke"
_token_file = next((path for path in ("/content/afrivoices_project",
                     os.path.join(DRIVE_ROOT, "hf_token.json")) if os.path.exists(path)), None)
assert _token_file, "hf_token.json introuvable"
login(token=json.load(open(_token_file))["key"])

SPONT_SPEC, SPONT_BUILD_ID, S_ROOT, S_DRIVE = _spont_layout(
    SPONT_LANG, SPONT_TARGET_HOURS, SPONT_DEV_HOURS)
S_PARQ = os.path.join(S_ROOT, "version=0")
S_TEXT = os.path.join(S_ROOT, "lm_text")
S_LONG = os.path.join(S_ROOT, "eval_long")
S_AUDIT = os.path.join(S_ROOT, "audit")
S_COMPLETE = os.path.join(S_ROOT, "_COMPLETE.json")
S_PARTIAL = S_DRIVE + ".partial"

def _complete_matches(path):
    try:
        meta = json.load(open(path, encoding="utf-8"))
        return meta.get("spec") == SPONT_SPEC and meta.get("build_id") == SPONT_BUILD_ID
    except Exception:
        return False

if not os.path.exists(S_ROOT):
    source_cache = S_DRIVE if _complete_matches(os.path.join(S_DRIVE, "_COMPLETE.json")) \
        else S_PARTIAL if os.path.exists(os.path.join(S_PARTIAL, "state.json")) else None
    if source_cache:
        print("Restauration cache :", source_cache)
        shutil.copytree(source_cache, S_ROOT)
for directory in (S_ROOT, S_PARQ, S_TEXT, S_LONG, S_AUDIT):
    os.makedirs(directory, exist_ok=True)

if not _complete_matches(S_COMPLETE):
    _test_ids = set()
    for _lang, _shard, _rows in test_shards:
        if _lang == SPONT_LANG:
            _test_ids.update(os.path.basename(str(value)) for value in
                             _pq.read_table(_shard, columns=["id"])["id"].to_pylist())

    # Anti-fuite corpus curé : les IDs source sont obligatoires pour toute langue concernée.
    _source_id_col = next((column for column in
                           ("source_filename", "filename", "source_id", "original_id")
                           if column in manifest.columns), None)
    _curated_ids = set()
    if _source_id_col:
        _curated_ids = {os.path.basename(str(value)) for value in
                        manifest[_source_id_col].dropna().astype(str)}

    _speaker_col = next((column for column in
                         ("speaker_id", "speaker", "recorder_uuid", "client_id")
                         if column in manifest.columns), None)
    assert _speaker_col is not None, "speaker_id requis pour un cache speaker-disjoint"
    _lang_curated = manifest[manifest.language.eq(SPONT_LANG)]
    _curated_all_speakers = set(_lang_curated[_speaker_col].dropna().astype(str))
    _curated_dev_speakers = set(_lang_curated[
        _lang_curated.split.isin(["valid", "local_test"])][_speaker_col]
        .dropna().astype(str))
    print(SPONT_LANG, "| locuteurs curés exclus :", len(_curated_all_speakers))

    _tree = f"{SPONT_LANG}/train/unscripted/audios"
    _shards = sorted(item.path for item in
                     HfApi().list_repo_tree(_repo, _tree, repo_type="dataset")
                     if item.path.endswith(".parquet"))
    assert _shards, f"Aucun shard : {_tree}"

    _state_path = os.path.join(S_ROOT, "state.json")
    _state = json.load(open(_state_path)) if os.path.exists(_state_path) else {
        "done": [], "train_seconds": 0.0, "dev_seconds": 0.0,
        "long_dev_seconds": 0.0, "long_dev_clips": 0,
        "text_rows": 0, "long_text_rows": 0, "repaired_rows": 0,
        "missing_speaker_rows": 0, "curated_overlap_rows": 0,
        "dev_short_speakers": [],
    }

    def _duration_from_raw(raw):
        try:
            return float(sf.info(_io.BytesIO(raw)).duration)
        except Exception:
            return None

    def _decode_to_flac(row):
        try:
            proc = subprocess.run(["ffmpeg", "-v", "quiet", "-nostdin", "-i", "pipe:0",
                                   "-f", "f32le", "-ac", "1", "-ar", "16000", "pipe:1"],
                                  input=row["raw"], capture_output=True, check=True)
            wav = np.frombuffer(proc.stdout, dtype=np.float32)
            if len(wav) < 8000:
                return None
            buffer = _io.BytesIO()
            sf.write(buffer, wav, 16000, format="FLAC")
            return {**{key: row[key] for key in
                       ("sample_id", "speaker_id", "text", "is_dev")},
                    "audio_bytes": buffer.getvalue(), "audio_size": len(wav),
                    "duration_s": len(wav) / 16000.0}
        except Exception:
            return None

    def _mirror(paths):
        for path in paths:
            relative = os.path.relpath(path, S_ROOT)
            destination = os.path.join(S_PARTIAL, relative)
            os.makedirs(os.path.dirname(destination), exist_ok=True)
            shutil.copy2(path, destination)
        shutil.copy2(_state_path, os.path.join(S_PARTIAL, "state.json"))

    os.makedirs(S_PARTIAL, exist_ok=True)
    for _shard_path in _shards:
        _tag = os.path.basename(_shard_path).replace(".parquet", "")
        if _tag in _state["done"]:
            continue
        _short_target_met = _state["train_seconds"] >= 3600 * SPONT_TARGET_HOURS
        _dev_target_met = _state["dev_seconds"] >= 3600 * SPONT_DEV_HOURS
        _long_target_met = _state["long_dev_seconds"] >= 3600 * SPONT_LONG_DEV_TARGET_HOURS
        if _short_target_met and _dev_target_met and _long_target_met \
                and not SPONT_HARVEST_ALL_TEXT_SHARDS:
            break

        _local = hf_hub_download(_repo, _shard_path, repo_type="dataset")
        _rows = _pq.read_table(_local).to_pylist()
        _text_rows, _audio_jobs, _unresolved = [], [], []
        for _index, row in enumerate(_rows):
            _sid = os.path.basename(str(row.get("filename") or row.get("id")
                                        or f"{_tag}:{_index}"))
            assert _sid not in _test_ids, f"Chevauchement test : {_sid}"
            if _sid in _curated_ids:
                _state["curated_overlap_rows"] += 1
                continue
            _raw_text = str(row.get("transcription") or "")
            _fixed = (repair_kikuyu_source(_raw_text) if SPONT_LANG == "kik"
                      else unicodedata.normalize("NFC", _raw_text))
            if SPONT_LANG == "kik" and _moji_badness(_fixed):
                _unresolved.append((_sid, _raw_text[:120], _fixed[:120]))
            _text = normalize_text(_fixed, SPONT_LANG)
            if not _text or len(_text.split()) < 2:
                continue
            _speaker = str(row.get("recorder_uuid") or "").strip()
            if not _speaker or _speaker.lower() in {"none", "nan", "?"}:
                _state["missing_speaker_rows"] += 1
                continue
            if _speaker in _curated_all_speakers:
                _state["curated_overlap_rows"] += 1
                continue
            if _speaker in _curated_dev_speakers:
                continue
            _is_dev = _speaker_is_dev(SPONT_LANG, _speaker, FT_CONFIG["seed"])
            _audio = row.get("audio")
            _raw = _audio.get("bytes") if isinstance(_audio, dict) else _audio
            _raw = bytes(_raw) if _raw else None
            _duration = float(row.get("duration") or 0) or (
                _duration_from_raw(_raw) if _raw else None)
            _audio_class = ("long" if _duration and _duration > 38.0
                            else "short" if _duration else "unknown")
            _record = {"sample_id": _sid, "speaker_id": _speaker, "text": _text,
                       "audio_class": _audio_class, "duration_s": _duration,
                       "repaired": bool(_fixed != _raw_text), "is_dev": _is_dev}
            if not _is_dev:
                _text_rows.append(_record)
            if _raw:
                _audio_jobs.append({**_record, "raw": _raw})
        assert not _unresolved, f"Mojibake Kikuyu non résolu : {_unresolved[:5]}"

        _created = []
        _text_path = os.path.join(S_TEXT, f"part-{_tag}.parquet")
        if _text_rows:
            _atomic_parquet(pa.Table.from_pylist(_text_rows), _text_path)
            _created.append(_text_path)
            _state["text_rows"] += len(_text_rows)
            _state["long_text_rows"] += sum(r["audio_class"] == "long" for r in _text_rows)
            _state["repaired_rows"] += sum(r["repaired"] for r in _text_rows)

        # Convertir les courts tant que les budgets ne sont pas atteints.
        _short_jobs = [row for row in _audio_jobs if row["audio_class"] != "long"
                       and ((not row["is_dev"] and not _short_target_met)
                            or (row["is_dev"] and not _dev_target_met))]
        with ThreadPoolExecutor(max_workers=12) as pool:
            _short_out = [value for value in pool.map(_decode_to_flac, _short_jobs) if value]
        _audit_rows = []
        for _split in ("train", "dev"):
            _subset = [{"text": row["text"], "audio_bytes": row["audio_bytes"],
                        "audio_size": row["audio_size"]} for row in _short_out
                       if row["is_dev"] == (_split == "dev") and row["duration_s"] <= 38.0]
            if _subset:
                _directory = os.path.join(S_PARQ, "corpus=afrivoices", f"split={_split}",
                                          f"language={omni_lang(SPONT_LANG)}")
                _path = os.path.join(_directory, f"part-{_tag}.parquet")
                _atomic_parquet(pa.Table.from_pylist(_subset), _path)
                _created.append(_path)
            for row in _short_out:
                if row["is_dev"] == (_split == "dev") and row["duration_s"] <= 38.0:
                    _audit_rows.append({"sample_id": row["sample_id"],
                                        "speaker_id": row["speaker_id"], "split": _split,
                                        "duration_s": row["duration_s"]})
                    _state[f"{_split}_seconds"] += row["duration_s"]
                    if _split == "dev" and row["speaker_id"] not in _state["dev_short_speakers"]:
                        _state["dev_short_speakers"].append(row["speaker_id"])

        # Vrai dev long : mêmes locuteurs dev, jamais inclus dans le LM.
        _long_rows = []
        if not _long_target_met:
            for row in _audio_jobs:
                duration = row.get("duration_s") or 0
                if (not row["is_dev"]
                        or row["speaker_id"] not in set(_state["dev_short_speakers"])
                        or not (SPONT_LONG_MIN_S <= duration <= SPONT_LONG_MAX_S)):
                    continue
                if len(row["text"].split()) < SPONT_LONG_MIN_WORDS \
                        or len(row["text"].split()) / duration < SPONT_LONG_MIN_WPS:
                    continue
                decoded = _decode_to_flac(row)
                if decoded is None:
                    continue
                _long_rows.append({"sample_id": decoded["sample_id"],
                                   "speaker_id": decoded["speaker_id"],
                                   "text": decoded["text"],
                                   "audio_bytes": decoded["audio_bytes"],
                                   "audio_size": decoded["audio_size"],
                                   "duration_s": decoded["duration_s"]})
                _audit_rows.append({"sample_id": decoded["sample_id"],
                                    "speaker_id": decoded["speaker_id"], "split": "long_dev",
                                    "duration_s": decoded["duration_s"]})
                _state["long_dev_seconds"] += decoded["duration_s"]
                _state["long_dev_clips"] += 1
                if _state["long_dev_seconds"] >= 3600 * SPONT_LONG_DEV_TARGET_HOURS:
                    break
        if _long_rows:
            _path = os.path.join(S_LONG, f"part-{_tag}.parquet")
            _atomic_parquet(pa.Table.from_pylist(_long_rows), _path)
            _created.append(_path)
        if _audit_rows:
            _path = os.path.join(S_AUDIT, f"part-{_tag}.parquet")
            _atomic_parquet(pa.Table.from_pylist(_audit_rows), _path)
            _created.append(_path)

        _state["done"].append(_tag)
        save_json(_state, _state_path)
        _mirror(_created)
        shutil.rmtree(os.path.expanduser(
            "~/.cache/huggingface/hub/datasets--MCAA1-MSU--anv_data_ke"),
            ignore_errors=True)
        print(_tag, "| train", round(_state["train_seconds"] / 3600, 1),
              "h | dev", round(_state["dev_seconds"] / 3600, 1),
              "h | longs", round(_state["long_dev_seconds"] / 3600, 1), "h")

    _train_parts = glob.glob(os.path.join(S_PARQ, "corpus=afrivoices", "split=train",
                                          "*", "*.parquet"))
    _total_samples = sum(sum(_pq.read_table(path, columns=["audio_size"])["audio_size"].to_pylist())
                         for path in _train_parts)
    with open(os.path.join(S_ROOT, "language_distribution_0.tsv"), "w") as handle:
        handle.write("corpus\tlanguage\thours\n")
        handle.write(f"afrivoices\t{omni_lang(SPONT_LANG)}\t"
                     f"{_total_samples / 16000 / 3600:.4f}\n")

    _audit_files = sorted(glob.glob(os.path.join(S_AUDIT, "*.parquet")))
    _text_files = sorted(glob.glob(os.path.join(S_TEXT, "*.parquet")))
    assert _audit_files and _text_files
    _audit = pd.concat([_pq.read_table(path).to_pandas() for path in _audit_files])
    _texts = pd.concat([_pq.read_table(path).to_pandas() for path in _text_files])
    _train_speakers = set(_audit.loc[_audit.split.eq("train"), "speaker_id"])
    _short_dev_speakers = set(_audit.loc[_audit.split.eq("dev"), "speaker_id"])
    _long_speakers = set(_audit.loc[_audit.split.eq("long_dev"), "speaker_id"])
    assert _train_speakers.isdisjoint(_short_dev_speakers | _long_speakers)
    assert _long_speakers <= _short_dev_speakers
    assert set(_texts.speaker_id).isdisjoint(_short_dev_speakers | _long_speakers)
    assert _state["dev_seconds"] >= 1800, "Dev court inférieur à 30 min"
    assert _state["long_dev_seconds"] >= 900 and _state["long_dev_clips"] >= 8
    assert (_texts.audio_class == "long").any(), "Aucun texte long récolté"
    if SPONT_LANG == "kik":
        assert not any(_moji_badness(text) for text in _texts.text.astype(str))

    _complete = {"spec": SPONT_SPEC, "build_id": SPONT_BUILD_ID,
                 "stats": _state, "finished_at": now_iso()}
    save_json(_complete, S_COMPLETE)
    _stage = S_DRIVE + ".staging"
    shutil.rmtree(_stage, ignore_errors=True)
    shutil.copytree(S_ROOT, _stage)
    _backup = S_DRIVE + ".backup"
    shutil.rmtree(_backup, ignore_errors=True)
    if os.path.exists(S_DRIVE):
        os.replace(S_DRIVE, _backup)
    os.replace(_stage, S_DRIVE)
    print("Cache V3 validé et publié :", S_DRIVE)

_meta = json.load(open(S_COMPLETE))
assert _meta["spec"] == SPONT_SPEC and _meta["build_id"] == SPONT_BUILD_ID
print("Dataset spontané V3 prêt :", SPONT_BUILD_ID, _meta["stats"])


In [ ]:
# 16.5 — V3 : top-up spontané versionné (candidat, jamais déployé implicitement)
require_state("SPONT_LANG", "SPONT_BUILD_ID", "S_ROOT", "TOPUP_SEED_CARD")
if not RUN_SPONT_TRAINING:
    raise SystemExit("RUN_SPONT_TRAINING=False")
_complete_path = os.path.join(S_ROOT, "_COMPLETE.json")
assert os.path.exists(_complete_path), "Construire/restaurer 16.4 V3"
_source_meta = json.load(open(_complete_path))
assert _source_meta["build_id"] == SPONT_BUILD_ID

SPONT_TRAIN_STEPS = 1500
SPONT_TRAIN_LR = 5e-6
import gc
for _name in ("_pipe_joint", "_pipe_lm", "_pipe"):
    if _name in globals():
        del globals()[_name]
if "TOPUP_PIPES" in globals():
    TOPUP_PIPES.clear()
gc.collect(); torch.cuda.empty_cache()

write_dataset_card(data_root=os.path.join(S_ROOT, "version=0"))
_output = os.path.join(FT_CONFIG["experiment_run"],
                       f"topup_{SPONT_LANG}_spont_{SPONT_BUILD_ID}")
_yaml = write_recipe_yaml(
    os.path.join(FT_CONFIG["experiment_run"],
                 f"topup_{SPONT_LANG}_spont_{SPONT_BUILD_ID}_recipe.yaml"),
    TOPUP_SEED_CARD, os.path.join(S_ROOT, "language_distribution_0.tsv"),
    num_steps=SPONT_TRAIN_STEPS, grad_accum=FT_CONFIG["grad_accum"],
    max_num_elements=FT_CONFIG["max_num_elements"], lr=SPONT_TRAIN_LR,
    ckpt_every=250)
_rc, _log = run_recipe(_output, _yaml,
                       f"topup_{SPONT_LANG}_spont_{SPONT_BUILD_ID}.log",
                       wall_limit_h=FT_CONFIG["max_wall_hours_per_session"])
print("Code :", _rc, "| dernier checkpoint :", latest_ckpt(_output))
print("Suite : 16.6 V3 comparera joint, spont et mixed sans déployer automatiquement.")


In [ ]:
# 16.6 — V3 : joint vs spont vs mixed, court + scripté + vrais longs
# QA_PATCH: MIXED_CHECKPOINT_DISCOVERY
require_state("SPONT_LANG", "manifest", "load_finetuned_pipeline", "latest_ckpt",
              "normalize_text", "omni_lang", "S_ROOT")
import glob, hashlib, io, random as _rnd, re
import pyarrow.parquet as _pq
from pathlib import Path
from pyctcdecode import build_ctcdecoder

_step_re = re.compile(r"step_(\d+)$")
_candidates, _seen = [], set()
def _add_candidate(name, family, checkpoint):
    if not checkpoint:
        return
    checkpoint = os.path.normpath(str(checkpoint))
    model_dir = Path(checkpoint) / "model"
    if checkpoint in _seen or not model_dir.exists():
        return
    _seen.add(checkpoint)
    _candidates.append({"name": name, "family": family, "ckpt": checkpoint})

_add_candidate("joint", "baseline", latest_ckpt(FT_CONFIG["output_dir"]))
_old_choice_path = os.path.join(FT_CONFIG["report_dir"], f"topup_choice_{SPONT_LANG}.json")
if os.path.exists(_old_choice_path):
    _old_choice = json.load(open(_old_choice_path))
    if _old_choice.get("chosen") != "joint":
        _add_candidate(f"legacy:{_old_choice['chosen']}", "baseline", _old_choice.get("ckpt"))

for _family in ("spont", "mix"):
    for _root in sorted(Path(FT_CONFIG["experiment_run"]).glob(
            f"topup_{SPONT_LANG}_{_family}*")):
        if not _root.is_dir():
            continue
        _steps = [path for path in _root.rglob("step_*")
                  if path.is_dir() and not path.name.endswith(".tmp")
                  and (path / "model").exists() and _step_re.search(path.name)]
        for _step in sorted(_steps, key=lambda path: int(_step_re.search(path.name).group(1))):
            _suffix = hashlib.sha1(str(_step).encode()).hexdigest()[:6]
            _add_candidate(f"{_family}:{_root.name}:{_step.name}:{_suffix}",
                           _family, str(_step))
assert any(item["family"] in {"spont", "mix"} for item in _candidates), (
    "Aucun checkpoint spont/mix finalisé. Un step 1800 sans dossier model n'est pas évaluable."
)

def _labels_for(pipe):
    model = getattr(pipe.tokenizer, "_model", None)
    for method in ("index_to_token", "id_to_piece", "IdToPiece"):
        if model is not None and hasattr(model, method):
            labels = [str(getattr(model, method)(i))
                      for i in range(pipe.tokenizer.vocab_info.size)]
            if len(labels) == len(set(labels)):
                labels[0] = ""; return labels
    raise RuntimeError("Vocab inaccessible")

def _capture_for(pipe, inputs, code):
    values, original = [], pipe.model.forward
    def spy(src, bl, *args, **kwargs):
        out = original(src, bl, *args, **kwargs)
        logits = out[0] if isinstance(out, tuple) else out
        values.append(torch.log_softmax(logits[0].detach().float(), -1).cpu().numpy())
        return out
    pipe.model.forward = spy
    try:
        pipe.transcribe(inputs, lang=[code] * len(inputs), batch_size=1)
    finally:
        pipe.model.forward = original
    return values

_dev_rows = []
for _path in sorted(glob.glob(os.path.join(S_ROOT, "version=0", "corpus=afrivoices",
                                            "split=dev", "*", "*.parquet"))):
    _dev_rows += _pq.read_table(_path).to_pylist()
_rnd.Random(FT_CONFIG["seed"]).shuffle(_dev_rows)
_dev_rows = _dev_rows[:200]
_dev_inputs = [{"waveform": sf.read(io.BytesIO(row["audio_bytes"]),
                                      dtype="float32")[0], "sample_rate": 16000}
               for row in _dev_rows]
_dev_refs = [row["text"] for row in _dev_rows]
_script = (manifest[(manifest.split == "valid") & (manifest.language == SPONT_LANG)]
           .sample(frac=1.0, random_state=FT_CONFIG["seed"]).head(200))
_script_refs = _script["text_norm"].tolist()

_long_rows = []
for _path in sorted(glob.glob(os.path.join(S_ROOT, "eval_long", "*.parquet"))):
    _long_rows += _pq.read_table(_path).to_pylist()
_long_rows = sorted(_long_rows, key=lambda row: hashlib.sha256(
    f"{FT_CONFIG['seed']}:{row['sample_id']}".encode()).hexdigest())[:60]

_lm_cfg = json.load(open(os.path.join(FT_CONFIG["report_dir"], "kenlm_tuning.json")))[SPONT_LANG]
_lm_path = _lm_cfg.get("lm_bin") or os.path.join(
    FT_CONFIG["experiment_run"], "kenlm_models", f"{SPONT_LANG}.binary")
_uni_file = _lm_cfg.get("uni_file")
_unigrams = (open(_uni_file, encoding="utf-8").read().splitlines()
             if _uni_file and os.path.exists(_uni_file)
             else sorted({word for text in manifest[(manifest.split == "train")
                                                      & (manifest.language == SPONT_LANG)]["text_norm"]
                          for word in str(text).split()}))
_beam = int(_lm_cfg.get("beam", 100))

def _long_logprobs(pipe, row):
    wav, sr = sf.read(io.BytesIO(row["audio_bytes"]), dtype="float32", always_2d=False)
    if getattr(wav, "ndim", 1) > 1:
        wav = wav.mean(axis=1)
    assert sr == 16000
    win, hop, trim = 38.0, 32.0, 3.0
    segments, offset, n = [], 0, len(wav)
    while offset < n:
        segment = wav[offset:offset + int(win * sr)]
        last = offset + int(win * sr) >= n
        if len(segment) >= int(0.2 * sr):
            segments.append((segment, offset > 0, not last))
        if last: break
        offset += int(hop * sr)
    logits = _capture_for(pipe, [{"waveform": segment, "sample_rate": sr}
                                 for segment, _, _ in segments], omni_lang(SPONT_LANG))
    parts = []
    for (segment, trim_left, trim_right), values in zip(segments, logits):
        fps = len(values) / (len(segment) / sr)
        left = int(round(trim * fps)) if trim_left else 0
        right = len(values) - (int(round(trim * fps)) if trim_right else 0)
        parts.append(values[left:right])
    return parts[0] if len(parts) == 1 else np.concatenate(parts, axis=0)

_results = {}
for _candidate in _candidates:
    _pipe = load_finetuned_pipeline(_candidate["ckpt"])
    _labels = _labels_for(_pipe)
    _decoder = build_ctcdecoder(_labels, _lm_path, unigrams=_unigrams,
                                alpha=float(_lm_cfg["alpha"]),
                                beta=float(_lm_cfg["beta"]))
    _dev_hyps = [normalize_text(_decoder.decode(values, beam_width=_beam), SPONT_LANG)
                 for values in _capture_for(_pipe, _dev_inputs, omni_lang(SPONT_LANG))]
    _script_hyps = [normalize_text(_decoder.decode(values, beam_width=_beam), SPONT_LANG)
                    for values in _capture_for(_pipe, _script["curated_audio_path"].tolist(),
                                               omni_lang(SPONT_LANG))]
    _long_hyps = [normalize_text(_decoder.decode(_long_logprobs(_pipe, row),
                                                  beam_width=_beam), SPONT_LANG)
                  for row in _long_rows]
    _results[_candidate["name"]] = {
        "family": _candidate["family"], "ckpt": _candidate["ckpt"],
        "wer_spont_short": round(jiwer.wer(_dev_refs, _dev_hyps), 5),
        "wer_spont_long": round(jiwer.wer([row["text"] for row in _long_rows],
                                           _long_hyps), 5) if _long_rows else None,
        "wer_scripted": round(jiwer.wer(_script_refs, _script_hyps), 5),
        "n_short": len(_dev_refs), "n_long": len(_long_rows),
        "n_scripted": len(_script_refs),
    }
    print(_candidate["name"], _results[_candidate["name"]])
    del _pipe; torch.cuda.empty_cache()

_report = {"language": SPONT_LANG, "dataset_build_id": globals().get("SPONT_BUILD_ID"),
           "created_at": now_iso(), "results": _results,
           "note": "Corpus-WER séparés; aucune pondération par durée."}
_report_path = os.path.join(FT_CONFIG["report_dir"], f"candidate_eval_{SPONT_LANG}_v3.json")
save_json(_report, _report_path)
print("Rapport candidat :", _report_path)
print("Aucune route finale n'a été modifiée. Utiliser 16.10 pour créer un modèle unique.")


In [ ]:
# 16.7 — V3 : LM v6 avec TOUS les textes train, y compris les clips longs
require_state("SPONT_LANG", "S_ROOT", "SPONT_BUILD_ID", "manifest", "normalize_text")
import glob, hashlib, subprocess
import pyarrow.parquet as _pq
from collections import Counter

_text_files = sorted(glob.glob(os.path.join(S_ROOT, "lm_text", "*.parquet")))
_audit_files = sorted(glob.glob(os.path.join(S_ROOT, "audit", "*.parquet")))
assert _text_files and _audit_files, "Reconstruire 16.4 V3"
_text_frame = pd.concat([_pq.read_table(path).to_pandas() for path in _text_files],
                        ignore_index=True)
_audit = pd.concat([_pq.read_table(path).to_pandas() for path in _audit_files],
                   ignore_index=True)
_dev_speakers = set(_audit.loc[_audit.split.isin(["dev", "long_dev"]), "speaker_id"])
assert set(_text_frame.speaker_id).isdisjoint(_dev_speakers)
_spontaneous_text = [str(value) for value in _text_frame.text
                     if isinstance(value, str) and len(value.split()) >= 2]
assert _spontaneous_text and (_text_frame.audio_class == "long").any()
if SPONT_LANG == "kik":
    assert not any(_moji_badness(text) for text in _spontaneous_text)

_scripted_text = [str(value) for value in manifest[(manifest.split == "train")
                                                   & (manifest.language == SPONT_LANG)]["text_norm"]]
_sentences = _scripted_text * 2 + sorted(set(_spontaneous_text)) * 2
_hash = hashlib.sha256()
for sentence in _sentences:
    _hash.update(sentence.encode("utf-8")); _hash.update(b"\n")
_lm_id = _hash.hexdigest()[:16]
_lm_dir = os.path.join(FT_CONFIG["experiment_run"], "kenlm_models_v6_textfix_longtext")
os.makedirs(_lm_dir, exist_ok=True)
_binary = os.path.join(_lm_dir, f"{SPONT_LANG}_{_lm_id}.binary")
_unigram_file = os.path.join(_lm_dir, f"{SPONT_LANG}_{_lm_id}_unigrams.txt")
_meta_file = os.path.join(_lm_dir, f"{SPONT_LANG}_{_lm_id}.json")
_lmplz, _build_binary = "/content/kenlm/build/bin/lmplz", "/content/kenlm/build/bin/build_binary"
assert os.path.exists(_lmplz), "Exécuter 12.1 pour compiler KenLM"

if not all(os.path.exists(path) for path in (_binary, _unigram_file, _meta_file)):
    _text_path, _arpa = f"/content/lm6_{SPONT_LANG}.txt", f"/content/lm6_{SPONT_LANG}.arpa"
    with open(_text_path, "w", encoding="utf-8") as handle:
        handle.write("\n".join(_sentences))
    with open(_text_path, "rb") as source, open(_arpa, "wb") as output:
        _run = subprocess.run([_lmplz, "-o", "5", "--discount_fallback", "-S", "30%"],
                              stdin=source, stdout=output, stderr=subprocess.PIPE)
    assert _run.returncode == 0, _run.stderr[-800:].decode(errors="replace")
    subprocess.run([_build_binary, _arpa, _binary], check=True)
    counts = Counter(word for sentence in _sentences for word in sentence.split())
    with open(_unigram_file, "w", encoding="utf-8") as handle:
        handle.write("\n".join(word for word, _ in counts.most_common(200_000)))
    save_json({"language": SPONT_LANG, "lm_id": _lm_id,
               "dataset_build_id": SPONT_BUILD_ID,
               "sentence_count": len(_sentences),
               "unique_spontaneous": len(set(_spontaneous_text)),
               "long_text_count": int((_text_frame.audio_class == "long").sum()),
               "created_at": now_iso()}, _meta_file)
assert json.load(open(_meta_file))["lm_id"] == _lm_id

# Enregistrer un candidat par domaine sans écraser la configuration globale historique.
_domain_path = os.path.join(FT_CONFIG["report_dir"], "kenlm_tuning_by_domain_v3.json")
_domain = json.load(open(_domain_path)) if os.path.exists(_domain_path) else {}
_baseline = json.load(open(os.path.join(FT_CONFIG["report_dir"], "kenlm_tuning.json")))[SPONT_LANG]
_domain[f"{SPONT_LANG}|scripted"] = dict(_baseline)
_domain[f"{SPONT_LANG}|unscripted"] = {
    "alpha": float(_baseline["alpha"]), "beta": float(_baseline["beta"]),
    "beam": int(_baseline.get("beam", 100)), "lm": "v6_textfix_longtext",
    "lm_bin": _binary, "uni_file": _unigram_file,
    "corpus_id": _lm_id, "dataset_build_id": SPONT_BUILD_ID,
    "needs_domain_tuning": True,
}
save_json(_domain, _domain_path)
print(f"LM v6 {SPONT_LANG}: {len(_spontaneous_text)} textes spontanés, "
      f"{int((_text_frame.audio_class == 'long').sum())} longs")
print("Candidat enregistré :", _domain_path, "| régler alpha/beta avant la finale.")


In [ ]:
# 16.7b — V3 : réglage LM Unscripted sur courts + vrais longs, pondéré par mots
require_state("SPONT_LANG", "S_ROOT", "capture_logprobs", "LABELS",
              "PRIMARY_ACOUSTIC_CKPT", "build_ctcdecoder")
import glob, hashlib, io
import pyarrow.parquet as _pq

_domain_path = os.path.join(FT_CONFIG["report_dir"], "kenlm_tuning_by_domain_v3.json")
_domain = json.load(open(_domain_path))
_candidate = _domain[f"{SPONT_LANG}|unscripted"]
_baseline = json.load(open(os.path.join(FT_CONFIG["report_dir"],
                                        "kenlm_tuning.json")))[SPONT_LANG]

def _resolve_unigrams(config):
    path = config.get("uni_file")
    if path and os.path.exists(path):
        return open(path, encoding="utf-8").read().splitlines()
    return sorted({word for text in manifest[(manifest.split == "train")
                                              & (manifest.language == SPONT_LANG)]["text_norm"]
                   for word in str(text).split()})

_short_rows = []
for _path in sorted(glob.glob(os.path.join(S_ROOT, "version=0", "corpus=afrivoices",
                                            "split=dev", "*", "*.parquet"))):
    _short_rows += _pq.read_table(_path).to_pylist()
_short_rows = sorted(_short_rows, key=lambda row: hashlib.sha256(
    f"{FT_CONFIG['seed']}:{row['text']}".encode()).hexdigest())[:120]
_short_inputs = [{"waveform": sf.read(io.BytesIO(row["audio_bytes"]),
                                        dtype="float32")[0], "sample_rate": 16000}
                 for row in _short_rows]
_short_refs = [row["text"] for row in _short_rows]
_short_lp = capture_logprobs(_short_inputs, omni_lang(SPONT_LANG),
                             manifest_lang=SPONT_LANG, domains="unscripted")

_long_rows = []
for _path in sorted(glob.glob(os.path.join(S_ROOT, "eval_long", "*.parquet"))):
    _long_rows += _pq.read_table(_path).to_pylist()
_long_rows = sorted(_long_rows, key=lambda row: hashlib.sha256(
    f"{FT_CONFIG['seed']}:{row['sample_id']}".encode()).hexdigest())[:40]

def _long_lp(row):
    wav, sr = sf.read(io.BytesIO(row["audio_bytes"]), dtype="float32", always_2d=False)
    if getattr(wav, "ndim", 1) > 1: wav = wav.mean(axis=1)
    segments, offset, total = [], 0, len(wav)
    while offset < total:
        segment = wav[offset:offset + int(38 * sr)]
        last = offset + int(38 * sr) >= total
        if len(segment) >= int(0.2 * sr):
            segments.append((segment, offset > 0, not last))
        if last: break
        offset += int(32 * sr)
    values = capture_logprobs([{"waveform": segment, "sample_rate": sr}
                               for segment, _, _ in segments], omni_lang(SPONT_LANG),
                              manifest_lang=SPONT_LANG, domains="unscripted")
    parts = []
    for (segment, leexperiment_run, right_trim), logits in zip(segments, values):
        fps = len(logits) / (len(segment) / sr)
        left = int(round(3 * fps)) if leexperiment_run else 0
        right = len(logits) - (int(round(3 * fps)) if right_trim else 0)
        parts.append(logits[left:right])
    return parts[0] if len(parts) == 1 else np.concatenate(parts, axis=0)

_long_values = [_long_lp(row) for row in _long_rows]
_refs = _short_refs + [row["text"] for row in _long_rows]
_logprobs = _short_lp + _long_values
assert _refs and len(_refs) == len(_logprobs)

_baseline_path = _baseline.get("lm_bin") or os.path.join(
    FT_CONFIG["experiment_run"], "kenlm_models", f"{SPONT_LANG}.binary")
_candidates = [
    ("baseline", _baseline_path, _resolve_unigrams(_baseline)),
    ("v6", _candidate["lm_bin"], _resolve_unigrams(_candidate)),
]
_alphas = sorted({0.3, 0.5, 0.65, 0.8, 1.0, float(_baseline["alpha"])})
_betas = sorted({0.0, 1.0, 2.0, float(_baseline["beta"])})
_beam = int(_baseline.get("beam", 100))
_best, _grid = None, []
for _tag, _binary, _unigrams in _candidates:
    for _alpha in _alphas:
        for _beta in _betas:
            decoder = build_ctcdecoder(LABELS, _binary, unigrams=_unigrams,
                                       alpha=_alpha, beta=_beta)
            hypotheses = [normalize_text(decoder.decode(values, beam_width=_beam),
                                         SPONT_LANG) for values in _logprobs]
            score = jiwer.wer(_refs, hypotheses)  # somme erreurs / somme mots
            row = {"lm": _tag, "alpha": _alpha, "beta": _beta, "wer": score}
            _grid.append(row)
            if _best is None or score < _best[0]:
                _best = (score, _tag, _binary, _unigrams, _alpha, _beta)
_score, _tag, _binary, _unigrams, _alpha, _beta = _best
_domain[f"{SPONT_LANG}|unscripted"] = {
    "alpha": _alpha, "beta": _beta, "beam": _beam,
    "lm": "v6_textfix_longtext" if _tag == "v6" else _baseline.get("lm", "v1"),
    "lm_bin": _binary,
    **({"uni_file": _candidate["uni_file"]} if _tag == "v6"
       else ({"uni_file": _baseline["uni_file"]} if _baseline.get("uni_file") else {})),
    "wer_dev_word_weighted": round(_score, 5), "needs_domain_tuning": False,
    "n_short": len(_short_rows), "n_long": len(_long_rows),
    "dataset_build_id": SPONT_BUILD_ID,
}
save_json(_domain, _domain_path)
save_json({"language": SPONT_LANG, "created_at": now_iso(), "grid": _grid,
           "best": _domain[f"{SPONT_LANG}|unscripted"]},
          os.path.join(FT_CONFIG["report_dir"],
                       f"domain_lm_tuning_{SPONT_LANG}_{SPONT_BUILD_ID}.json"))
print("Meilleur LM Unscripted :", _domain[f"{SPONT_LANG}|unscripted"])


## 16.8 — Le thermomètre synthétique est retiré

Les vrais audios longs speaker-disjoint sont maintenant matérialisés par 16.4 V3 dans
`eval_long/` et participent directement au réglage word-weighted de 16.7b. La concaténation
artificielle de trois clips courts n'est plus utilisée pour adopter un réglage final.


In [ ]:
# 16.9 — V3 : mixed-batch versionné, découvert automatiquement par 16.6
require_state("SPONT_LANG", "SPONT_BUILD_ID", "S_ROOT", "TOPUP_SEED_CARD",
              "manifest", "write_recipe_yaml", "run_recipe")
if not RUN_MIXED_TRAINING:
    raise SystemExit("RUN_MIXED_TRAINING=False")
import glob, hashlib, io, shutil
import pyarrow as pa
import pyarrow.parquet as _pq

_source_meta_path = os.path.join(S_ROOT, "_COMPLETE.json")
assert os.path.exists(_source_meta_path)
_source_meta = json.load(open(_source_meta_path))
assert _source_meta["build_id"] == SPONT_BUILD_ID

MIX_SCRIPT_RATIO = 0.30
MIX_LR = 2e-6
MIX_STEPS = 1000
_mix_spec = {"schema": CACHE_SCHEMA_VERSION, "source_build_id": SPONT_BUILD_ID,
             "script_ratio": MIX_SCRIPT_RATIO, "lr": MIX_LR,
             "steps": MIX_STEPS, "seed": int(FT_CONFIG["seed"])}
_mix_hash = hashlib.sha256(json.dumps(_mix_spec, sort_keys=True).encode()).hexdigest()[:12]
MIX_BUILD_ID = f"mixv3_{_mix_hash}"
M_ROOT = f"/content/topup_{SPONT_LANG}_mix_{MIX_BUILD_ID}"
M_PARQ = os.path.join(M_ROOT, "version=0")
M_COMPLETE = os.path.join(M_ROOT, "_COMPLETE.json")

if not os.path.exists(M_COMPLETE):
    shutil.rmtree(M_ROOT, ignore_errors=True)
    shutil.copytree(os.path.join(S_ROOT, "version=0"), M_PARQ)
    _spont_samples = sum(sum(_pq.read_table(path, columns=["audio_size"])["audio_size"].to_pylist())
                         for path in glob.glob(os.path.join(M_PARQ, "corpus=afrivoices",
                                                            "split=train", "*", "*.parquet")))
    _spont_seconds = _spont_samples / 16000
    _needed_seconds = _spont_seconds * MIX_SCRIPT_RATIO / (1 - MIX_SCRIPT_RATIO)
    _source = (manifest[(manifest.split == "train") & (manifest.language == SPONT_LANG)]
               .sample(frac=1.0, random_state=FT_CONFIG["seed"]))
    _selected, _seconds = [], 0.0
    for _, row in _source.iterrows():
        if _seconds >= _needed_seconds: break
        _selected.append(row); _seconds += float(row["duration"])
    assert _selected
    _directory = os.path.join(M_PARQ, "corpus=afrivoices", "split=train",
                              f"language={omni_lang(SPONT_LANG)}")
    os.makedirs(_directory, exist_ok=True)
    rows, part = [], 0
    for row in _selected:
        with open(row["curated_audio_path"], "rb") as handle:
            blob = handle.read()
        frames = int(sf.info(io.BytesIO(blob)).frames)
        rows.append({"text": row["text_norm"], "audio_bytes": blob, "audio_size": frames})
        if len(rows) >= 2000:
            _atomic_parquet(pa.Table.from_pylist(rows),
                            os.path.join(_directory, f"part-scripted-{part:03d}.parquet"))
            rows, part = [], part + 1
    if rows:
        _atomic_parquet(pa.Table.from_pylist(rows),
                        os.path.join(_directory, f"part-scripted-{part:03d}.parquet"))
    _ratio = _seconds / (_seconds + _spont_seconds)
    assert 0.27 <= _ratio <= 0.33, _ratio
    _total_seconds = _seconds + _spont_seconds
    with open(os.path.join(M_ROOT, "language_distribution_0.tsv"), "w") as handle:
        handle.write("corpus\tlanguage\thours\n")
        handle.write(f"afrivoices\t{omni_lang(SPONT_LANG)}\t{_total_seconds / 3600:.4f}\n")
    save_json({"spec": _mix_spec, "build_id": MIX_BUILD_ID,
               "spontaneous_hours": _spont_seconds / 3600,
               "scripted_hours": _seconds / 3600, "created_at": now_iso()}, M_COMPLETE)

write_dataset_card(data_root=M_PARQ)
_output = os.path.join(FT_CONFIG["experiment_run"], f"topup_{SPONT_LANG}_mix_{MIX_BUILD_ID}")
_yaml = write_recipe_yaml(
    os.path.join(FT_CONFIG["experiment_run"],
                 f"topup_{SPONT_LANG}_mix_{MIX_BUILD_ID}_recipe.yaml"),
    TOPUP_SEED_CARD, os.path.join(M_ROOT, "language_distribution_0.tsv"),
    num_steps=MIX_STEPS, grad_accum=FT_CONFIG["grad_accum"],
    max_num_elements=FT_CONFIG["max_num_elements"], lr=MIX_LR, ckpt_every=250)
_rc, _log = run_recipe(_output, _yaml,
                       f"topup_{SPONT_LANG}_mix_{MIX_BUILD_ID}.log",
                       wall_limit_h=FT_CONFIG["max_wall_hours_per_session"])
print("Code :", _rc, "| dernier checkpoint :", latest_ckpt(_output))
print("16.6 V3 découvrira automatiquement ce run et le run Somali legacy.")


In [ ]:
# 16.10 — V3 : fusion de deltas en UN checkpoint acoustique (optionnel)
# Format : [{"name":"som_spont", "ckpt":"/.../step_1500", "weight":0.25}, ...]
MERGE_COMPONENTS = []
if not RUN_UNIFIED_DELTA_MERGE:
    raise SystemExit("RUN_UNIFIED_DELTA_MERGE=False")
assert MERGE_COMPONENTS, "Renseigner au moins un composant validé par 16.6"
import gc, hashlib

def _weight_file(checkpoint):
    checkpoint = Path(checkpoint)
    if checkpoint.is_file(): return checkpoint
    files = [path for path in (checkpoint / "model").rglob("*") if path.is_file()]
    assert len(files) == 1, files
    return files[0]

_base_checkpoint = latest_ckpt(FT_CONFIG["output_dir"])
_base_file = _weight_file(_base_checkpoint)
_base_raw = torch.load(str(_base_file), map_location="cpu", weights_only=False)
_base = _base_raw.get("model", _base_raw)
_merged = {key: value.clone() if torch.is_tensor(value) else value
           for key, value in _base.items()}
_spec = []
for component in MERGE_COMPONENTS:
    weight = float(component["weight"])
    assert -1.0 <= weight <= 1.0
    candidate_file = _weight_file(component["ckpt"])
    raw = torch.load(str(candidate_file), map_location="cpu", weights_only=False)
    state = raw.get("model", raw)
    assert state.keys() == _base.keys()
    for key, base_value in _base.items():
        if torch.is_tensor(base_value) and torch.is_floating_point(base_value):
            _merged[key].add_(state[key].to(_merged[key].dtype) - base_value, alpha=weight)
    _spec.append({"name": component["name"], "ckpt": str(component["ckpt"]),
                  "weight": weight})
    del raw, state; gc.collect()

_merge_id = hashlib.sha256(json.dumps(_spec, sort_keys=True).encode()).hexdigest()[:12]
_out = Path(FT_CONFIG["experiment_run"]) / "unified_delta_merges" / _merge_id / "step_0"
_weight_out = _out / "model" / "pp_00" / "tp_00" / "sdp_00.pt"
_weight_out.parent.mkdir(parents=True, exist_ok=True)
torch.save({"model": _merged}, _weight_out)
save_json({"base": str(_base_checkpoint), "components": _spec,
           "created_at": now_iso(), "parameter_count_unchanged": True},
          str(_out / "merge_manifest.json"))
print("Checkpoint unifié :", _out)
print("Définir FINAL_UNIFIED_CHECKPOINT sur ce chemin, puis réévaluer toutes les langues.")


## 17 — Soumission finale V3 : domaine, conformité et provenance

Cette section est la seule voie de génération de la soumission finale. Elle recharge un
checkpoint acoustique unique, choisit le LM selon `(langue, Scripted/Unscripted)`, calcule
une empreinte de toute la configuration et refuse tout CSV ancien ou altéré.


In [ ]:
# 17.1 — Runtime final : LM par domaine, modèle unique et empreintes
# QA_PATCH: DOMAIN_AWARE_ROUTING
# QA_PATCH: CACHE_PROVENANCE
require_state("MODEL_COMPLIANCE", "PRIMARY_ACOUSTIC_CKPT", "capture_logprobs",
              "LABELS", "LM_BEST", "LM_PATHS", "LM_UNIGRAMS",
              "canonical_domain", "shard_domain")
import hashlib, json, os
from pathlib import Path
from pyctcdecode import build_ctcdecoder

def _canonical_hash(value):
    payload = json.dumps(value, sort_keys=True, ensure_ascii=False,
                         separators=(",", ":")).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

def _sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024**2), b""):
            digest.update(block)
    return digest.hexdigest()

_ARTIFACT_DIGEST_CACHE = {}
def artifact_digest(path, strong=None):
    strong = STRONG_ARTIFACT_HASH_FOR_FINAL if strong is None else bool(strong)
    path = os.path.realpath(path)
    key = (path, strong)
    if key in _ARTIFACT_DIGEST_CACHE:
        return _ARTIFACT_DIGEST_CACHE[key]
    root = Path(path)
    files = ([root] if root.is_file() else sorted(
        item for item in ((root / "model") if (root / "model").exists() else root).rglob("*")
        if item.is_file()))
    digest = hashlib.sha256()
    base = root.parent if root.is_file() else root
    for item in files:
        stat = item.stat()
        relative = str(item.relative_to(base)).replace(os.sep, "/").encode()
        digest.update(relative); digest.update(str(stat.st_size).encode())
        digest.update(str(stat.st_mtime_ns).encode())
        with open(item, "rb") as handle:
            if strong or stat.st_size <= 2 * 1024**2:
                for block in iter(lambda: handle.read(8 * 1024**2), b""):
                    digest.update(block)
            else:
                digest.update(handle.read(1024**2))
                handle.seek(max(0, stat.st_size - 1024**2))
                digest.update(handle.read(1024**2))
    _ARTIFACT_DIGEST_CACHE[key] = digest.hexdigest()
    return digest.hexdigest()

def sequence_digest(values):
    digest = hashlib.sha256()
    for value in values:
        encoded = str(value).encode("utf-8")
        digest.update(len(encoded).to_bytes(8, "big")); digest.update(encoded)
    return digest.hexdigest()

def _atomic_json(value, path):
    temporary = f"{path}.tmp-{os.getpid()}"
    with open(temporary, "w", encoding="utf-8") as handle:
        json.dump(value, handle, ensure_ascii=False, indent=2)
    os.replace(temporary, path)

def _atomic_csv(frame, path):
    temporary = f"{path}.tmp-{os.getpid()}"
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)

_domain_config_path = os.path.join(FT_CONFIG["report_dir"],
                                   "kenlm_tuning_by_domain_v3.json")
_saved_domain = (json.load(open(_domain_config_path))
                 if os.path.exists(_domain_config_path) else {})
DOMAIN_LM_CONFIGS, DOMAIN_LM_PATHS, DOMAIN_UNIGRAMS = {}, {}, {}
for _lang in LANGUAGES:
    for _domain in DOMAINS:
        _key = f"{_lang}|{_domain}"
        _config = dict(_saved_domain.get(_key, LM_BEST[_lang]))
        if _config.get("needs_domain_tuning"):
            raise RuntimeError(f"{_key} : exécuter 16.7b avant la soumission finale")
        _lm_path = _config.get("lm_bin") or LM_PATHS[_lang]
        assert os.path.exists(_lm_path), _lm_path
        _unigram_file = _config.get("uni_file")
        _unigrams = (open(_unigram_file, encoding="utf-8").read().splitlines()
                     if _unigram_file and os.path.exists(_unigram_file)
                     else LM_UNIGRAMS[_lang])
        DOMAIN_LM_CONFIGS[(_lang, _domain)] = _config
        DOMAIN_LM_PATHS[(_lang, _domain)] = _lm_path
        DOMAIN_UNIGRAMS[(_lang, _domain)] = _unigrams

_decoder_cache, DOMAIN_DECODERS = {}, {}
for _key, _config in DOMAIN_LM_CONFIGS.items():
    _lm_path = DOMAIN_LM_PATHS[_key]
    _unigrams = DOMAIN_UNIGRAMS[_key]
    _decoder_key = _canonical_hash({"lm": artifact_digest(_lm_path),
                                    "unigrams": sequence_digest(_unigrams),
                                    "alpha": float(_config["alpha"]),
                                    "beta": float(_config["beta"])})
    if _decoder_key not in _decoder_cache:
        _decoder_cache[_decoder_key] = build_ctcdecoder(
            LABELS, _lm_path, unigrams=_unigrams,
            alpha=float(_config["alpha"]), beta=float(_config["beta"]))
    DOMAIN_DECODERS[_key] = _decoder_cache[_decoder_key]

_static_model_bytes = sum(parameter.numel() * parameter.element_size()
                          for parameter in _pipe_joint.model.parameters())
_static_lm_bytes = sum(os.path.getsize(path) for path in set(DOMAIN_LM_PATHS.values()))
MODEL_COMPLIANCE.update({"minimum_static_bytes": _static_model_bytes + _static_lm_bytes,
                         "edge_ram_limit_bytes": 8 * 1024**3,
                         "edge_peak_rss_still_required": True})
assert MODEL_COMPLIANCE["minimum_static_bytes"] < MODEL_COMPLIANCE["edge_ram_limit_bytes"]

FINAL_CHUNKING = {"window_s": 38.0, "overlap_s": 6.0, "trim_s": 3.0,
                  "short_threshold_s": 39.5, "decode_once": True}
_run_spec = {
    "pipeline_revision": PIPELINE_REVISION,
    "normalization_version": NORMALIZATION_VERSION,
    "acoustic_checkpoint": artifact_digest(PRIMARY_ACOUSTIC_CKPT),
    "tokenizer": sequence_digest(LABELS),
    "model_compliance": MODEL_COMPLIANCE,
    "chunking": FINAL_CHUNKING,
    "domain_lms": {f"{lang}|{domain}": {
        "config": DOMAIN_LM_CONFIGS[(lang, domain)],
        "lm_digest": artifact_digest(DOMAIN_LM_PATHS[(lang, domain)]),
        "unigram_digest": sequence_digest(DOMAIN_UNIGRAMS[(lang, domain)]),
    } for lang in LANGUAGES for domain in DOMAINS},
}
FINAL_RUN_ID = _canonical_hash(_run_spec)[:16]
FINAL_RUN_SPEC = _run_spec
print("Runtime final V3 :", FINAL_RUN_ID,
      "| checkpoint(s) :", MODEL_COMPLIANCE["checkpoint_count"],
      "| décodeurs uniques :", len(_decoder_cache))


In [ ]:
# 17.1b — Runtime mémoire : un seul décodeur KenLM actif à la fois
import ctypes
import gc
import os

import psutil
import torch
from pyctcdecode import build_ctcdecoder
from pyctcdecode.decoder import BeamSearchDecoderCTC

require_state(
    "DOMAIN_LM_CONFIGS", "DOMAIN_LM_PATHS", "DOMAIN_UNIGRAMS",
    "FINAL_RUN_SPEC", "FINAL_RUN_ID", "MODEL_COMPLIANCE",
    "test_shards", "TEST_DIR_TO_MANIFEST", "shard_domain",
)

_active_domain_keys = {
    (TEST_DIR_TO_MANIFEST[submission_language], shard_domain(shard))
    for submission_language, shard, _ in test_shards
}
assert len(_active_domain_keys) == 11, sorted(_active_domain_keys)

# Libérer les 12 décodeurs eager construits par 17.1.
if "_decoder" in globals():
    del _decoder
if "DOMAIN_DECODERS" in globals():
    try:
        DOMAIN_DECODERS.clear()
    except Exception:
        pass
if "_decoder_cache" in globals():
    try:
        _decoder_cache.clear()
    except Exception:
        pass
BeamSearchDecoderCTC.clear_class_models()

# Objets volumineux des phases d'audit, inutiles pour l'inférence test.
for _temporary_name in (
    "_full_index", "_lm_eval_frame", "_dev_lm", "_inventory_frame",
    "_scripted_eval", "_unscripted_eval", "_local_test", "_group_report",
    "eval_frame", "selected_dev", "eligible_dev", "selected_train",
    "_tune_frame", "_holdout_frame", "_combined", "_sample",
    "_logits", "_all_logits", "_tune_logits", "_holdout_logits",
):
    if _temporary_name in globals():
        del globals()[_temporary_name]

gc.collect()
torch.cuda.empty_cache()
try:
    ctypes.CDLL("libc.so.6").malloc_trim(0)
except Exception:
    pass


class _LazyDomainDecoderStore:
    """Mapping minimal compatible avec DOMAIN_DECODERS[(lang, domain)]."""

    def __init__(self, active_keys):
        self._active_keys = frozenset(active_keys)
        self._current_key = None
        self._decoder = None
        self.peak_rss_bytes = psutil.Process(os.getpid()).memory_info().rss
        self.build_count = 0

    def __len__(self):
        return len(self._active_keys)

    def __iter__(self):
        return iter(self._active_keys)

    def keys(self):
        return self._active_keys

    def clear(self):
        if self._decoder is not None:
            try:
                self._decoder.cleanup()
            except Exception:
                pass
        self._decoder = None
        self._current_key = None
        gc.collect()
        try:
            ctypes.CDLL("libc.so.6").malloc_trim(0)
        except Exception:
            pass

    def __getitem__(self, key):
        if key not in self._active_keys:
            raise KeyError(f"Domaine test inactif : {key}")
        if key == self._current_key and self._decoder is not None:
            return self._decoder

        if self._decoder is not None:
            try:
                self._decoder.cleanup()
            except Exception:
                pass
        self._decoder = None
        self._current_key = None
        gc.collect()
        try:
            ctypes.CDLL("libc.so.6").malloc_trim(0)
        except Exception:
            pass

        config = DOMAIN_LM_CONFIGS[key]
        self._decoder = build_ctcdecoder(
            LABELS,
            DOMAIN_LM_PATHS[key],
            unigrams=DOMAIN_UNIGRAMS[key],
            alpha=float(config["alpha"]),
            beta=float(config["beta"]),
        )
        self._current_key = key
        self.build_count += 1
        rss = psutil.Process(os.getpid()).memory_info().rss
        self.peak_rss_bytes = max(self.peak_rss_bytes, rss)
        MODEL_COMPLIANCE["lazy_decoder_peak_rss_bytes"] = self.peak_rss_bytes
        print(
            "[décodeur lazy]", f"{key[0]}|{key[1]}",
            "| RSS", round(rss / 2**30, 3), "GiB",
        )
        return self._decoder


DOMAIN_DECODERS = _LazyDomainDecoderStore(_active_domain_keys)
MODEL_COMPLIANCE.update({
    "decoder_runtime": "lazy_single_active",
    "decoder_active_domain_count": len(_active_domain_keys),
    "decoder_resident_limit": 1,
    "eager_decoder_count": 0,
})

# Le mode mémoire fait partie du contrat du run, même si les textes produits sont identiques.
FINAL_RUN_SPEC = dict(FINAL_RUN_SPEC)
FINAL_RUN_SPEC["model_compliance"] = dict(MODEL_COMPLIANCE)
FINAL_RUN_SPEC["decoder_runtime"] = {
    "mode": "lazy_single_active",
    "resident_limit": 1,
    "active_domains": sorted(f"{lang}|{domain}" for lang, domain in _active_domain_keys),
}
FINAL_RUN_ID = _canonical_hash(FINAL_RUN_SPEC)[:16]

_rss_after_cleanup = psutil.Process(os.getpid()).memory_info().rss
MODEL_COMPLIANCE["rss_after_eager_decoder_cleanup_bytes"] = _rss_after_cleanup

# Sonde du plus gros LM actif : vérifie le pic attendu sans lancer l'inférence complète.
_largest_active_key = max(
    _active_domain_keys,
    key=lambda key: os.path.getsize(DOMAIN_LM_PATHS[key]),
)
DOMAIN_DECODERS[_largest_active_key]
_largest_decoder_probe_rss = psutil.Process(os.getpid()).memory_info().rss
MODEL_COMPLIANCE["largest_decoder_probe_key"] = (
    f"{_largest_active_key[0]}|{_largest_active_key[1]}"
)
MODEL_COMPLIANCE["largest_decoder_probe_rss_bytes"] = _largest_decoder_probe_rss
DOMAIN_DECODERS.clear()
gc.collect()
try:
    ctypes.CDLL("libc.so.6").malloc_trim(0)
except Exception:
    pass

print("✅ Runtime KenLM paresseux activé")
print("FINAL_RUN_ID             :", FINAL_RUN_ID)
print("Domaines actifs          :", len(DOMAIN_DECODERS))
print("Décodeurs résidents max  : 1")
print("RSS après nettoyage      :", round(_rss_after_cleanup / 2**30, 3), "GiB")
print(
    "Pic sonde plus gros LM    :",
    round(_largest_decoder_probe_rss / 2**30, 3), "GiB",
    "|", MODEL_COMPLIANCE["largest_decoder_probe_key"],
)
print(
    "Minimum statique        :",
    round(MODEL_COMPLIANCE["minimum_static_bytes"] / 2**30, 3), "GiB",
)

In [ ]:
import os
import psutil

_process = psutil.Process(os.getpid())
_rss_gib = _process.memory_info().rss / 2**30
_static_gib = MODEL_COMPLIANCE["minimum_static_bytes"] / 2**30

_active_keys = {
    (
        TEST_DIR_TO_MANIFEST[submission_language],
        shard_domain(shard),
    )
    for submission_language, shard, _ in test_shards
}

_unused_keys = sorted(set(DOMAIN_DECODERS) - _active_keys)

print("RSS actuel            :", round(_rss_gib, 3), "GiB")
print("Minimum statique      :", round(_static_gib, 3), "GiB")
print("Décodeurs construits  :", len(DOMAIN_DECODERS))
print("Décodeurs utilisés    :", len(_active_keys))
print("Décodeurs non utilisés:", _unused_keys)

In [ ]:
# 17.1c — Nettoyage des objets d’audit avant inférence
import ctypes
import gc
import os

import pandas as pd
import psutil
import pyarrow as pa
import torch

_removed = []
_freed_bytes = 0

for _name, _value in list(globals().items()):
    if isinstance(_value, pd.DataFrame):
        try:
            _size = int(_value.memory_usage(index=True, deep=True).sum())
        except Exception:
            _size = 0

        _removed.append(_name)
        _freed_bytes += _size
        del globals()[_name]

    elif isinstance(_value, pa.Table):
        try:
            _size = int(_value.nbytes)
        except Exception:
            _size = 0

        _removed.append(_name)
        _freed_bytes += _size
        del globals()[_name]

for _name in (
    "all_results",
    "detail_rows",
    "eval_groups",
    "_grid",
    "_summary",
    "_language_rows",
    "_audit",
    "_texts",
):
    if _name in globals():
        del globals()[_name]

DOMAIN_DECODERS.clear()

gc.collect()
torch.cuda.empty_cache()

try:
    ctypes.CDLL("libc.so.6").malloc_trim(0)
except Exception:
    pass

_process = psutil.Process(os.getpid())
_rss_clean = _process.memory_info().rss

# Nouvelle sonde du plus gros LM après nettoyage.
DOMAIN_DECODERS.peak_rss_bytes = _rss_clean
DOMAIN_DECODERS[_largest_active_key]

_rss_probe = _process.memory_info().rss

MODEL_COMPLIANCE["rss_clean_before_inference_bytes"] = _rss_clean
MODEL_COMPLIANCE["largest_decoder_clean_probe_rss_bytes"] = _rss_probe

DOMAIN_DECODERS.clear()
gc.collect()

try:
    ctypes.CDLL("libc.so.6").malloc_trim(0)
except Exception:
    pass

print("Objets supprimés       :", len(_removed))
print("Taille logique libérée :", round(_freed_bytes / 2**30, 3), "GiB")
print("RSS nettoyé            :", round(_rss_clean / 2**30, 3), "GiB")
print("Pic avec plus gros LM  :", round(_rss_probe / 2**30, 3), "GiB")
print("FINAL_RUN_ID           :", FINAL_RUN_ID)

In [ ]:
assert FINAL_RUN_ID != "REPLACE_WITH_LOCAL_RUN_ID"
print("Nouveau FINAL_RUN_ID :", FINAL_RUN_ID)
print("TEST_ROOT :", TEST_ROOT)

In [ ]:
# 17.1d — Stager le test Kaggle sur le disque local rapide avant 17.2
import json
import os
import shutil
import subprocess

require_state(
    "TEST_ROOT", "test_shards", "TEST_TOTAL_ROWS", "_canonical_hash",
)

_test_source_root = os.path.realpath(TEST_ROOT)
_source_inventory = [
    {
        "language": language,
        "relative_path": os.path.relpath(path, _test_source_root),
        "bytes": int(os.path.getsize(path)),
        "rows": int(rows),
    }
    for language, path, rows in test_shards
]
_source_bytes = sum(item["bytes"] for item in _source_inventory)
_inventory_sha256 = _canonical_hash(_source_inventory)

_candidate_bases = ["/mnt/disks/local-scratch", "/content"]
_local_base = None
for _candidate in _candidate_bases:
    if not os.path.isdir(_candidate) or not os.access(_candidate, os.W_OK):
        continue
    if shutil.disk_usage(_candidate).free >= _source_bytes + 8 * 2**30:
        _local_base = _candidate
        break
assert _local_base is not None, (
    f"Il faut au moins {(_source_bytes + 8 * 2**30) / 2**30:.1f} Gio "
    "libres sur /mnt/disks/local-scratch ou /content"
)

LOCAL_TEST_ROOT = os.path.join(_local_base, "kaggle_test_full_fast")
os.makedirs(LOCAL_TEST_ROOT, exist_ok=True)
_local_marker = os.path.join(LOCAL_TEST_ROOT, "_LOCAL_COMPLETE.json")


def _local_copy_complete():
    try:
        marker = json.load(open(_local_marker, encoding="utf-8"))
    except Exception:
        return False
    if marker.get("inventory_sha256") != _inventory_sha256:
        return False
    for item in _source_inventory:
        target = os.path.join(LOCAL_TEST_ROOT, item["relative_path"])
        if not os.path.isfile(target) or os.path.getsize(target) != item["bytes"]:
            return False
    return True


if _local_copy_complete():
    print("[cache local valide]", LOCAL_TEST_ROOT)
else:
    assert shutil.which("rsync"), "rsync absent du runtime"
    print(
        "Copie Drive → disque local :",
        round(_source_bytes / 2**30, 2), "Gio — reprise automatique si relancée",
    )
    subprocess.run(
        [
            "rsync", "-a", "--partial", "--info=progress2",
            _test_source_root.rstrip(os.sep) + os.sep,
            LOCAL_TEST_ROOT.rstrip(os.sep) + os.sep,
        ],
        check=True,
    )
    for _item in _source_inventory:
        _target = os.path.join(LOCAL_TEST_ROOT, _item["relative_path"])
        assert os.path.isfile(_target), _target
        assert os.path.getsize(_target) == _item["bytes"], _target
    _temporary_marker = _local_marker + ".tmp"
    with open(_temporary_marker, "w", encoding="utf-8") as _handle:
        json.dump(
            {
                "source_root": _test_source_root,
                "inventory_sha256": _inventory_sha256,
                "source_bytes": _source_bytes,
                "shards": len(_source_inventory),
                "rows": int(TEST_TOTAL_ROWS),
            },
            _handle,
            ensure_ascii=False,
            indent=2,
        )
    os.replace(_temporary_marker, _local_marker)

LOCAL_TEST_SOURCE_ROOT = _test_source_root
test_shards = [
    (
        item["language"],
        os.path.join(LOCAL_TEST_ROOT, item["relative_path"]),
        item["rows"],
    )
    for item in _source_inventory
]
assert sum(rows for _, _, rows in test_shards) == TEST_TOTAL_ROWS
assert all(os.path.exists(path) for _, path, _ in test_shards)
TEST_ROOT = LOCAL_TEST_ROOT

print("✅ Test prêt sur disque local :", TEST_ROOT)
print("Shards :", len(test_shards), "| clips :", TEST_TOTAL_ROWS)

In [ ]:
# 17.1e-cleanup — restaurer proprement le batch acoustique 1
import ctypes
import gc
import torch

assert "_CAPTURE_BATCH1_REFERENCE" in globals()
_capture = _CAPTURE_BATCH1_REFERENCE

for _name in (
    "_CAPTURE_BATCH1_REFERENCE",
    "_probe_decoder", "_probe_inputs", "_probe_durations", "_probe_meta",
    "_probe_omni", "_probe_pipe", "_reference_logits", "_reference_texts",
    "_reference_ctc_paths", "_candidate_logits", "_candidate_texts",
    "_selected_logits", "_verified_candidates", "_winner",
):
    globals().pop(_name, None)

DOMAIN_DECODERS.clear()
gc.collect()
torch.cuda.empty_cache()

try:
    ctypes.CDLL("libc.so.6").malloc_trim(0)
except Exception:
    pass

print("✅ Batch acoustique 1 restauré")
print("TEST_ROOT    :", TEST_ROOT)
print("FINAL_RUN_ID :", FINAL_RUN_ID)

In [ ]:
RUN_TEST_INFERENCE = False
RUN_FINAL_INFERENCE = True

assert torch.cuda.is_available()
assert os.path.basename(
    os.path.realpath(PRIMARY_ACOUSTIC_CKPT)
) == "step_1250"

assert FINAL_RUN_ID != "REPLACE_WITH_LOCAL_RUN_ID"

print("FINAL_RUN_ID :", FINAL_RUN_ID)
print("TEST_ROOT    :", TEST_ROOT)
print("Checkpoint   :", PRIMARY_ACOUSTIC_CKPT)

In [ ]:
# 17.2 — Inférence finale reprenable avec cache signé
require_state("RUN_FINAL_INFERENCE", "test_shards", "TEST_TOTAL_ROWS",
              "TEST_DIR_TO_OMNI", "DOMAIN_DECODERS", "FINAL_RUN_ID",
              "FINAL_RUN_SPEC", "capture_logprobs", "normalize_text")
if not RUN_FINAL_INFERENCE:
    raise SystemExit("RUN_FINAL_INFERENCE=False")
import base64, glob, io, subprocess
import pyarrow.parquet as pq

_B64_CHARS = set(b"ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789+/=\n\r")
def _ffmpeg_decode(raw, sample_rate=16000):
    try:
        process = subprocess.run(["ffmpeg", "-v", "quiet", "-nostdin", "-i", "pipe:0",
                                  "-f", "f32le", "-ac", "1", "-ar", str(sample_rate),
                                  "pipe:1"], input=raw, capture_output=True, check=True)
        wav = np.frombuffer(process.stdout, dtype=np.float32).copy()
        return (wav, sample_rate) if len(wav) else (None, None)
    except Exception:
        return None, None

def _decode_audio(cell):
    raw = cell.get("bytes") if isinstance(cell, dict) else cell
    if raw is None: return None, None
    if hasattr(raw, "tobytes"): raw = raw.tobytes()
    elif isinstance(raw, (memoryview, bytearray)): raw = bytes(raw)
    if raw[:5] == b"UklGR" or (len(raw) > 32 and set(raw[:80]) <= _B64_CHARS):
        try:
            decoded = base64.b64decode(raw, validate=False)
            if decoded[:4] in (b"RIFF", b"fLaC", b"OggS") or decoded[:3] == b"ID3":
                raw = decoded
        except Exception:
            pass
    try:
        wav, sample_rate = sf.read(io.BytesIO(raw), dtype="float32", always_2d=False)
    except Exception:
        return _ffmpeg_decode(raw)
    if getattr(wav, "ndim", 1) > 1: wav = wav.mean(axis=1)
    return wav, sample_rate

def _windows(wav, sample_rate):
    total, offset, output = len(wav), 0, []
    threshold = int(FINAL_CHUNKING["short_threshold_s"] * sample_rate)
    if total <= threshold:
        return [(wav, False, False)]
    window = int(FINAL_CHUNKING["window_s"] * sample_rate)
    hop = int((FINAL_CHUNKING["window_s"] - FINAL_CHUNKING["overlap_s"]) * sample_rate)
    while offset < total:
        segment = wav[offset:offset + window]
        last = offset + window >= total
        if len(segment) >= int(0.2 * sample_rate):
            output.append((segment, offset > 0, not last))
        if last: break
        offset += hop
    return output

def _decode_long(segments, omni_code, manifest_lang, domain, decoder, kwargs, sample_rate):
    values = capture_logprobs(
        [{"waveform": segment, "sample_rate": sample_rate} for segment, _, _ in segments],
        omni_code, manifest_lang=manifest_lang, domains=domain)
    parts = []
    for (segment, trim_left, trim_right), logits in zip(segments, values):
        fps = len(logits) / (len(segment) / float(sample_rate))
        trim = int(round(FINAL_CHUNKING["trim_s"] * fps))
        left = trim if trim_left else 0
        right = len(logits) - (trim if trim_right else 0)
        assert right > left
        parts.append(logits[left:right])
    full = parts[0] if len(parts) == 1 else np.concatenate(parts, axis=0)
    return decoder.decode(full, **kwargs)

_cache_dir = os.path.join(FT_CONFIG["report_dir"], "test_predictions_v3", FINAL_RUN_ID)
os.makedirs(_cache_dir, exist_ok=True)
_expected = []
for _submission_lang, _shard, _expected_rows in test_shards:
    _domain = shard_domain(_shard)
    _manifest_lang = TEST_DIR_TO_MANIFEST[_submission_lang]
    _config = DOMAIN_LM_CONFIGS[(_manifest_lang, _domain)]
    _decoder = DOMAIN_DECODERS[(_manifest_lang, _domain)]
    _omni = TEST_DIR_TO_OMNI[_submission_lang]
    _id_table = pq.read_table(_shard, columns=["id"]).to_pandas()
    _ids = _id_table["id"].astype(str).tolist()
    assert len(_ids) == _expected_rows and len(_ids) == len(set(_ids))
    _shard_spec = {
        "run_id": FINAL_RUN_ID, "language": _manifest_lang, "domain": _domain,
        "source_relative": os.path.relpath(_shard, TEST_ROOT),
        "source_bytes": os.path.getsize(_shard), "rows": _expected_rows,
        "ids_digest": sequence_digest(_ids),
    }
    _spec_hash = _canonical_hash(_shard_spec)
    _output = os.path.join(_cache_dir,
                           f"{_submission_lang}__{Path(_shard).parent.name}__"
                           f"{Path(_shard).name}.csv")
    _meta_path = _output + ".meta.json"
    _valid = False
    if os.path.exists(_output) and os.path.exists(_meta_path):
        try:
            _meta = json.load(open(_meta_path))
            _cached = pd.read_csv(_output, dtype=str)
            _valid = (_meta.get("spec_sha256") == _spec_hash
                      and _meta.get("csv_sha256") == _sha256_file(_output)
                      and list(_cached.columns) == ["id", "language", "text"]
                      and _cached["id"].astype(str).tolist() == _ids)
        except Exception:
            _valid = False
    _expected.append((_output, _meta_path, _spec_hash, _ids, _shard_spec))
    if _valid:
        print("[cache valide]", _submission_lang, _domain, Path(_shard).name)
        continue

    table = pq.read_table(_shard, columns=["id", "audio"]).to_pandas()
    assert table["id"].astype(str).tolist() == _ids
    kwargs = {"beam_width": int(_config.get("beam", 100))}
    if _config.get("tml") is not None:
        kwargs["token_min_logp"] = float(_config["tml"])
    if _config.get("bpl") is not None:
        kwargs["beam_prune_logp"] = float(_config["bpl"])
    short_inputs, short_indices, texts = [], [], {}
    for index, row in table.iterrows():
        wav, sample_rate = _decode_audio(row["audio"])
        if wav is None: continue
        if len(wav) < int(0.2 * sample_rate):
            wav = np.pad(wav, (0, int(0.2 * sample_rate) - len(wav)))
        segments = _windows(wav, sample_rate)
        if len(segments) == 1:
            short_inputs.append({"waveform": segments[0][0], "sample_rate": sample_rate})
            short_indices.append(index)
        else:
            texts[index] = _decode_long(segments, _omni, _manifest_lang, _domain,
                                         _decoder, kwargs, sample_rate)
    for start in range(0, len(short_inputs), 24):
        values = capture_logprobs(short_inputs[start:start + 24], _omni,
                                  manifest_lang=_manifest_lang, domains=_domain)
        for index, logits in zip(short_indices[start:start + 24], values):
            texts[index] = _decoder.decode(logits, **kwargs)
    predictions = [normalize_text(texts.get(index, ""), _manifest_lang)
                   for index in table.index]
    frame = pd.DataFrame({"id": _ids, "language": _submission_lang,
                          "text": predictions})
    _atomic_csv(frame, _output)
    _atomic_json({"created_at": now_iso(), "spec_sha256": _spec_hash,
                  "csv_sha256": _sha256_file(_output), "spec": _shard_spec}, _meta_path)
    print("[calculé]", _submission_lang, _domain, Path(_shard).name)

parts, inputs_meta = [], []
for output, meta_path, spec_hash, ids, shard_spec in _expected:
    assert os.path.exists(output) and os.path.exists(meta_path)
    meta = json.load(open(meta_path))
    assert meta["spec_sha256"] == spec_hash
    assert meta["csv_sha256"] == _sha256_file(output)
    frame = pd.read_csv(output, dtype=str)
    assert frame["id"].astype(str).tolist() == ids
    parts.append(frame)
    inputs_meta.append({"file": os.path.basename(output),
                        "spec_sha256": spec_hash, "csv_sha256": meta["csv_sha256"]})
submission = pd.concat(parts, ignore_index=True)
submission["text"] = submission["text"].fillna("")
empty = submission["text"].str.strip().eq("")
submission.loc[empty, "text"] = "a"
submission = submission.rename(columns={"text": SUBMISSION_TEXT_COLUMN})[
    ["id", "language", SUBMISSION_TEXT_COLUMN]]
assert submission["id"].is_unique and len(submission) == TEST_TOTAL_ROWS
assert set(submission["language"]) == set(LANG_TO_SUBMISSION.values())
_submission_path = os.path.join(FT_CONFIG["report_dir"],
                                f"submission_v3_{FINAL_RUN_ID}.csv")
_atomic_csv(submission, _submission_path)
_atomic_json({"created_at": now_iso(), "run_id": FINAL_RUN_ID,
              "run_spec": FINAL_RUN_SPEC, "model_compliance": MODEL_COMPLIANCE,
              "inputs": inputs_meta, "empty_predictions_replaced": int(empty.sum()),
              "submission_sha256": _sha256_file(_submission_path)},
             _submission_path + ".meta.json")
print("Soumission finale :", _submission_path)
print("SHA256 :", _sha256_file(_submission_path))


In [ ]:
# 17.3 — Contrôle final avant soumission Kaggle
import hashlib
import json
import os

import pandas as pd
import pyarrow.parquet as pq

SUBMISSION_PATH = (
    "/content/afrivoices_project/"
    "finetune_runs/experiment_run/reports/"
    "submission_v3_REPLACE_WITH_LOCAL_RUN_ID.csv"
)
META_PATH = SUBMISSION_PATH + ".meta.json"
EXPECTED_SHA256 = "REPLACE_WITH_LOCAL_SHA256"
EXPECTED_RUN_ID = "REPLACE_WITH_LOCAL_RUN_ID"

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024**2), b""):
            digest.update(block)
    return digest.hexdigest()

assert os.path.isfile(SUBMISSION_PATH), SUBMISSION_PATH
assert os.path.isfile(META_PATH), META_PATH

submission = pd.read_csv(
    SUBMISSION_PATH,
    dtype=str,
    keep_default_na=False,
)

assert list(submission.columns) == [
    "id", "language", "transcription"
], submission.columns.tolist()

assert len(submission) == 41733, len(submission)
assert submission["id"].is_unique
assert set(submission["language"]) == {
    "swa", "kik", "kln", "luo", "som", "mas"
}

texts = submission["transcription"].astype(str)
assert not texts.str.strip().eq("").any()
assert not texts.str.lower().isin({"nan", "none", "null"}).any()
assert not texts.str.contains(r"[\r\n]", regex=True).any()

# Vérification exacte des IDs, de leur ordre et de leur langue.
expected_ids = []
expected_languages = []

for language, shard, expected_rows in test_shards:
    ids = [
        str(value)
        for value in pq.read_table(shard, columns=["id"])["id"].to_pylist()
    ]
    assert len(ids) == expected_rows
    expected_ids.extend(ids)
    expected_languages.extend([language] * len(ids))

assert submission["id"].tolist() == expected_ids
assert submission["language"].tolist() == expected_languages

actual_sha256 = sha256_file(SUBMISSION_PATH)
assert actual_sha256 == EXPECTED_SHA256

metadata = json.load(open(META_PATH, encoding="utf-8"))
assert metadata["run_id"] == EXPECTED_RUN_ID
assert metadata["submission_sha256"] == actual_sha256
assert len(metadata["inputs"]) == 94

word_counts = texts.str.split().str.len()
summary = (
    submission.assign(words=word_counts)
    .groupby("language")
    .agg(
        rows=("id", "size"),
        words_median=("words", "median"),
        words_p95=("words", lambda values: values.quantile(0.95)),
        words_max=("words", "max"),
    )
)

print(summary)
print()
print("Prédictions vides remplacées :", metadata["empty_predictions_replaced"])
print("SHA256 :", actual_sha256)
print("✅ QA FINAL PASS — fichier prêt pour Kaggle")
print(SUBMISSION_PATH)

## 18 — Fine-tuning joint V4 après bilan des données

Ordre obligatoire :

1. `18.1` détecte le matériel et choisit le disque de travail.
2. `18.2` construit l'index et le bilan réel des données.
3. `18.3` produit un plan de sampling et signe le PASS/FAIL.
4. `18.4` matérialise le mélange uniquement si le rapport est encore valide.
5. `18.5` calibre le micro-batch sur l'A100 80 Go.
6. `18.6` poursuit le checkpoint joint pendant au maximum 2 500 steps.
7. `18.7` compare tous les checkpoints sur un dev mixte avant adoption.


### Ordre d'exécution V4 (sessions séparées acceptées)

1. Exécuter le setup, la configuration, le manifest et la normalisation (`1` à `4`).
2. Mettre uniquement `RUN_DATA_AUDIT_V4=True`, exécuter `18.1` puis `18.2`.
3. Lire les CSV/JSON. Si le statut est `FAIL`, corriger les causes — notamment les caches
   spontanés non versionnés — puis refaire l'audit. Pour le premier cache swahili auditable,
   exécuter l'inventaire test `10.2`, puis le builder `16.1` modifié par V4 ; il écrit désormais
   les IDs et locuteurs dans `dataset_caches/`. Ne pas signer un `FAIL`.
4. Quand le statut est `PASS`, renseigner `V4_AUDIT_APPROVAL` dans `18.3`.
5. Activer successivement et séparément : build `18.4`, calibration `18.5`, training `18.6`,
   puis évaluation `18.7`. Une seule phase coûteuse doit être vraie à la fois.

Le plafond de 2 500 pas correspond à environ 1 778 heures audio échantillonnées, soit
296 heures d'exposition nominale par langue. Avec un corpus retenu proche de 100–120 h par
langue, cela représente environ 2,5–3 passages. Le notebook évalue chaque checkpoint ;
l'optimum attendu est plutôt entre 1 250 et 2 250 pas que forcément à 2 500.
Si l'audit trouve moins de données, il réduit automatiquement le nombre de pas pour viser
au plus trois passages sur la langue la plus petite. Après remplacement de VM, les index et
signatures restent sur Drive ; rematérialiser seulement le dataset audio local en `18.4`.


In [ ]:
# 18.1 — Matériel et espace de travail A100 80 Go
import hashlib, platform, shutil
from pathlib import Path

V4_REVISION = "balanced-joint-a10080-v4.1-20260710"
V4_REPORT_DIR = os.path.join(FT_CONFIG["report_dir"], "balanced_v4")
os.makedirs(V4_REPORT_DIR, exist_ok=True)

def _free_bytes(path):
    return shutil.disk_usage(path).free if os.path.exists(path) else 0

_scratch_candidates = ["/content/local-scratch", "/content", "/tmp"]
_scratch_base = ("/content/local-scratch"
                 if _free_bytes("/content/local-scratch") >= 150 * 2**30
                 else max(_scratch_candidates, key=_free_bytes))
V4_SCRATCH_ROOT = os.path.join(_scratch_base, "afrivoices_balanced_v4")
# Index/sélections compacts sur Drive : ils survivent au remplacement de la VM.
V4_INDEX_PATH = os.path.join(V4_REPORT_DIR, "data_index_v4.parquet")
V4_DATASET_ROOT = os.path.join(V4_SCRATCH_ROOT, "dataset")
V4_DATA_VERSION_ROOT = os.path.join(V4_DATASET_ROOT, "version=0")
V4_AUDIT_PATH = os.path.join(V4_REPORT_DIR, "data_audit_v4.json")
V4_AUDIT_CSV = os.path.join(V4_REPORT_DIR, "data_inventory_v4.csv")
V4_PLAN_PATH = os.path.join(V4_REPORT_DIR, "sampling_plan_v4.json")
V4_SELECTION_PATH = os.path.join(V4_REPORT_DIR, "selected_records_v4.parquet")
V4_DEV_SELECTION_PATH = os.path.join(V4_REPORT_DIR, "selected_dev_v4.parquet")
V4_BUILD_META = os.path.join(V4_DATASET_ROOT, "_COMPLETE.json")
V4_CALIBRATION_PATH = os.path.join(V4_REPORT_DIR, "calibration_v4.json")
V4_TRAIN_OUTPUT = os.path.join(FT_CONFIG["experiment_run"], "balanced_joint_v4", "train_output")
V4_CANDIDATES_ROOT = os.path.join(FT_CONFIG["experiment_run"], "balanced_joint_v4", "candidates")
V4_CANDIDATES_DIR = V4_CANDIDATES_ROOT
V4_RUN_CONTRACT = os.path.join(V4_REPORT_DIR, "run_contract_v4.json")
os.makedirs(V4_SCRATCH_ROOT, exist_ok=True)

_gpu = torch.cuda.get_device_properties(0)
GPU_TOTAL_BYTES = int(_gpu.total_memory)
gpu_fingerprint = {
    "name": _gpu.name,
    "total_memory": GPU_TOTAL_BYTES,
    "total_gib": round(GPU_TOTAL_BYTES / 2**30, 2),
    "compute_capability": f"{_gpu.major}.{_gpu.minor}",
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
}
assert torch.cuda.is_bf16_supported(), "BF16 requis pour ce profil A100"
assert GPU_TOTAL_BYTES >= 75 * 2**30, (
    f"Ce chemin est calibré pour une A100 80 Go; GPU détecté : {gpu_fingerprint}"
)
_hardware = {
    "revision": V4_REVISION,
    "gpu_fingerprint": gpu_fingerprint,
    "system_ram_gib": round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30, 2),
    "scratch_root": V4_SCRATCH_ROOT,
    "scratch_free_gib": round(_free_bytes(_scratch_base) / 2**30, 2),
    "drive_free_gib": round(_free_bytes(DRIVE_ROOT) / 2**30, 2),
    "platform": platform.platform(),
}
assert _hardware["system_ram_gib"] >= 120, "Au moins 120 Gio de RAM système requis"
assert _free_bytes(_scratch_base) >= 150 * 2**30, "Au moins 150 Gio libres requis en scratch"
save_json(_hardware, os.path.join(V4_REPORT_DIR, "hardware_v4.json"))
print(json.dumps(_hardware, indent=2, ensure_ascii=False))


In [ ]:
import json
import os
import shutil
import time
from pathlib import Path

from tqdm.auto import tqdm

BACKUP = Path(
    "/content/afrivoices_project/"
    "dataset_caches/afrivoices_balanced_v4_dataset"
)

DESTINATION = Path(V4_DATASET_ROOT)
STAGING = Path(str(DESTINATION) + ".restore-staging")

SOURCE_MARKER = BACKUP / "_COMPLETE.json"
assert SOURCE_MARKER.is_file(), "Sauvegarde Drive incomplète"

metadata = json.load(open(SOURCE_MARKER, encoding="utf-8"))

source_files = sorted(
    path
    for path in BACKUP.rglob("*")
    if path.is_file()
    and path.name != "_COMPLETE.json"
    and not path.name.endswith(".part")
)

total_bytes = sum(path.stat().st_size for path in source_files)
free_bytes = shutil.disk_usage(DESTINATION.parent).free

print("À restaurer :", round(total_bytes / 2**30, 2), "Gio")
print("Scratch libre :", round(free_bytes / 2**30, 2), "Gio")

assert free_bytes >= total_bytes + 20 * 2**30, (
    "Scratch insuffisant pour restaurer le dataset avec 20 Gio de marge"
)

shutil.rmtree(STAGING, ignore_errors=True)
STAGING.mkdir(parents=True, exist_ok=True)

def copy_with_retry(source, destination, attempts=12):
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = Path(str(destination) + ".part")
    delay = 2.0

    for attempt in range(1, attempts + 1):
        try:
            source_size = source.stat().st_size

            if (
                destination.is_file()
                and destination.stat().st_size == source_size
            ):
                return

            try:
                offset = temporary.stat().st_size
            except OSError:
                offset = 0

            if offset > source_size:
                temporary.unlink(missing_ok=True)
                offset = 0

            with source.open("rb") as source_file:
                source_file.seek(offset)

                with temporary.open("ab") as destination_file:
                    while True:
                        block = source_file.read(32 * 1024**2)

                        if not block:
                            break

                        destination_file.write(block)

            assert temporary.stat().st_size == source_size
            os.replace(temporary, destination)
            assert destination.stat().st_size == source_size
            return

        except (OSError, AssertionError) as exc:
            print(
                f"\nReprise {source.name} "
                f"{attempt}/{attempts} : {exc!r}"
            )
            time.sleep(delay)
            delay = min(delay * 1.7, 30.0)

    raise RuntimeError(f"Échec de restauration : {source}")

for source in tqdm(
    source_files,
    desc="Restauration dataset V4",
    unit="fichier",
):
    relative = source.relative_to(BACKUP)
    copy_with_retry(source, STAGING / relative)

# Vérification des fichiers déclarés dans _COMPLETE.json.
for item in metadata["files"]:
    restored = STAGING / item["path"]

    assert restored.is_file(), f"Fichier restauré absent : {item['path']}"
    assert restored.stat().st_size == item["size"], (
        f"Taille incorrecte : {item['path']}"
    )

copy_with_retry(
    SOURCE_MARKER,
    STAGING / "_COMPLETE.json",
)

old_destination = Path(str(DESTINATION) + ".old")
shutil.rmtree(old_destination, ignore_errors=True)

if DESTINATION.exists():
    os.replace(DESTINATION, old_destination)

os.replace(STAGING, DESTINATION)
shutil.rmtree(old_destination, ignore_errors=True)

V4_TSV_PATH = str(DESTINATION / "language_distribution_0.tsv")
V4_DATA_VERSION_ROOT = str(DESTINATION / "version=0")
V4_BUILD_META = str(DESTINATION / "_COMPLETE.json")

write_dataset_card(data_root=V4_DATA_VERSION_ROOT)

print("✅ Dataset V4 restauré :", DESTINATION)
print("Build SHA256 :", metadata["build_sha256"])

In [ ]:
# 18.2 — QA_V4: DATA_AUDIT — bilan réel avant tout fine-tuning
import glob, hashlib, json, os, re, time
from collections import Counter, defaultdict
import pyarrow.parquet as pq

V4_AUDIT_SCHEMA = 3
V4_SEED = 42
V4_MAX_HOURS_PER_LANGUAGE = 160.0
V4_MIN_HOURS_PER_LANGUAGE = 30.0
V4_RATIO_TOLERANCE = 0.03

# Cible = part des NOUVELLES données proches du test, et non une assertion aveugle
# « scripted/unscripted ». L'audit conserve speech_method séparément.
V4_TARGET_NEW_IN_DOMAIN = {
    "sw": 0.90, "kik": 0.67, "luo": 0.67,
    "som": 0.82, "kln": 0.82, "mas": 0.83,
}
V4_OFFICIAL_REFERENCE_HOURS = {
    "sw": {"read": 0, "spontaneous": 2979},
    "kik": {"read": 183, "spontaneous": 571},
    "luo": {"read": 195, "spontaneous": 528},
    "som": {"read": 118, "spontaneous": 884},
    "kln": {"read": 122, "spontaneous": 399},
    "mas": {"read": 51, "spontaneous": 454},
}
SCRIPTED_TERMS = {"scripted", "read", "read method", "read_method", "prompted read"}
UNSCRIPTED_TERMS = {"unscripted", "spontaneous", "spontaneous method",
                    "spontaneous_method", "conversation", "conversational", "extempore"}

# Provenance composite déterministe du manifest curé.
def _provenance_value(value):
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass
    text = str(value).strip()
    return "" if text.lower() in {"", "nan", "none", "null", "?", "<na>"} else text

def _make_composite_source_id(row):
    fields = ("source_dataset", "source_split", "original_row_index",
              "audio_ref", "local_source_path", "sample_id")
    payload = "|".join(_provenance_value(row.get(field)) for field in fields)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:32]

manifest["source_id"] = [
    _make_composite_source_id(row) for row in manifest.to_dict("records")
]
assert manifest["source_id"].notna().all() and manifest["source_id"].is_unique
V4_SOURCE_ID_MODE = "composite_manifest_provenance"

def _sha256_file(path, chunk=8 << 20):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(chunk), b""):
            digest.update(block)
    return digest.hexdigest()

def _json_digest(value):
    payload = json.dumps(value, sort_keys=True, ensure_ascii=False,
                         separators=(",", ":"), default=str).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

def _frame_sha256(frame, sort_key="sample_key"):
    ordered = frame.copy()
    columns = sorted(ordered.columns)
    ordered = ordered[columns]
    if sort_key in ordered.columns:
        ordered = ordered.sort_values(sort_key, kind="mergesort")
    ordered = ordered.reset_index(drop=True)
    values = pd.util.hash_pandas_object(ordered.fillna("").astype(str),
                                        index=False).values.tobytes()
    schema = json.dumps(columns, ensure_ascii=False).encode("utf-8")
    return hashlib.sha256(schema + values).hexdigest()

def _clean_label(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return ""
    text = str(value).strip()
    return "" if text.lower() in {"", "nan", "none", "null", "?", "unknown"} else text

def _canonical_id(value):
    text = _clean_label(value)
    if not text:
        return ""
    text = os.path.basename(text.replace("\\", "/")).strip().lower()
    return re.sub(r"\.(wav|wave|flac|mp3|ogg|opus|webm|m4a)$", "", text)

def _manifest_sha256():
    cols = [c for c in ("sample_id", "split", "language", "curated_audio_path",
                         "duration", "text", "text_norm", "is_scripted", "type",
                         "domain", "speech_method", "method", "read_method",
                         "speaker_id", "speaker", "recorder_uuid", "client_id",
                         "source_filename", "filename", "source_id", "original_id")
            if c in manifest.columns]
    frame = manifest[cols].fillna("").astype(str).sort_values(
        [c for c in ("language", "split", "sample_id") if c in cols],
        kind="mergesort").reset_index(drop=True)
    hashed = pd.util.hash_pandas_object(frame, index=False).values.tobytes()
    return hashlib.sha256(hashed).hexdigest()

def _boolish(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    text = str(value).strip().lower()
    if text in {"true", "1", "yes", "y", "oui"}: return True
    if text in {"false", "0", "no", "n", "non"}: return False
    return None

def _classify_method(row):
    signals = []
    flag = _boolish(row.get("is_scripted"))
    if flag is not None:
        signals.append("scripted" if flag else "unscripted")
    for column in ("speech_method", "method", "type", "read_method", "domain"):
        value = str(row.get(column, "")).strip().lower().replace("-", " ")
        if value in SCRIPTED_TERMS: signals.append("scripted")
        if value in UNSCRIPTED_TERMS: signals.append("unscripted")
    path = str(row.get("curated_audio_path", "")).replace("\\", "/").lower()
    if re.search(r"(?:^|/)unscripted(?:/|$)", path): signals.append("unscripted")
    elif re.search(r"(?:^|/)scripted(?:/|$)", path): signals.append("scripted")
    unique = sorted(set(signals))
    return (unique[0] if len(unique) == 1 else "unknown", len(unique) > 1)

def _quality_flags(language, split, seconds, text, method, conflict, exists=True):
    flags = []
    if not exists: flags.append("MISSING_AUDIO")
    if not np.isfinite(seconds) or seconds < 0.5: flags.append("BAD_DURATION")
    if split == "train" and seconds > 38.0: flags.append("TOO_LONG_FOR_CTC")
    words = len(str(text).split())
    if not str(text).strip() or words < 2: flags.append("EMPTY_OR_TOO_SHORT_TEXT")
    if seconds > 0 and words / seconds < 0.20: flags.append("VERY_LOW_WPS")
    if seconds > 0 and words / seconds > 6.00: flags.append("VERY_HIGH_WPS")
    badness = (_moji_badness(text) if "_moji_badness" in globals()
               else _text_badness(text, language) if "_text_badness" in globals() else 0)
    if "�" in str(text) or badness > 0: flags.append("ENCODING_SUSPECT")
    if method == "unknown": flags.append("UNKNOWN_METHOD")
    if conflict: flags.append("METHOD_CONFLICT")
    return flags

def _cache_candidates():
    inventory, valid = [], []
    pattern = os.path.join(DRIVE_ROOT, "dataset_caches", "topup_*_spont_*")
    for root in sorted(glob.glob(pattern)):
        if not os.path.isdir(root):
            continue
        item = {"root": root, "valid": False, "issues": []}
        meta_path = os.path.join(root, "_COMPLETE.json")
        try:
            meta = json.load(open(meta_path, encoding="utf-8"))
            spec = meta.get("spec", {})
            language = str(spec.get("language", ""))
            language = "sw" if language == "swa" else language
            item.update(language=language, build_id=meta.get("build_id"),
                        finished_at=meta.get("finished_at"), meta_path=meta_path)
            if language not in LANGUAGES: item["issues"].append("BAD_LANGUAGE")
            if not meta.get("build_id"): item["issues"].append("MISSING_BUILD_ID")
            elif not os.path.basename(root).endswith(str(meta["build_id"])):
                item["issues"].append("BUILD_ID_PATH_MISMATCH")
            if glob.glob(os.path.join(root, "**", "*.tmp-*"), recursive=True):
                item["issues"].append("TMP_FILES_PRESENT")
            audio_parts = sorted(glob.glob(os.path.join(
                root, "version=0", "corpus=*", "split=*", "language=*", "*.parquet")))
            audit_parts = sorted(glob.glob(os.path.join(root, "audit", "*.parquet")))
            if not audio_parts: item["issues"].append("NO_AUDIO_PARQUET")
            if not audit_parts: item["issues"].append("NO_AUDIT_PARQUET")
            splits = {re.search(r"split=([^/\\]+)", p).group(1) for p in audio_parts
                      if re.search(r"split=([^/\\]+)", p)}
            if not {"train", "dev"} <= splits: item["issues"].append("TRAIN_OR_DEV_MISSING")
            expected_omni = omni_lang(language)
            if expected_omni and any(f"language={expected_omni}" not in p for p in audio_parts):
                item["issues"].append("PARTITION_LANGUAGE_MISMATCH")
            if audio_parts:
                train_samples = sum(sum(pq.read_table(p, columns=["audio_size"])
                                        ["audio_size"].to_pylist())
                                    for p in audio_parts if "split=train" in p)
                declared = float(meta.get("stats", {}).get("train_seconds") or 0)
                measured = train_samples / 16000.0
                if declared <= 0 or abs(measured-declared) > max(60.0, .02*measured):
                    item["issues"].append("DECLARED_TRAIN_HOURS_MISMATCH")
            item["audio_parts"] = len(audio_parts)
            item["audit_parts"] = len(audit_parts)
            item["meta"] = meta
            item["valid"] = not item["issues"]
            if item["valid"]: valid.append(item)
        except Exception as exc:
            item["issues"].append(f"INVALID_COMPLETE_JSON:{type(exc).__name__}")
        inventory.append({k: v for k, v in item.items()
                          if k not in {"meta"}})
    legacy = os.path.join(DRIVE_ROOT, "topup_swa_spont_cache")
    if os.path.exists(legacy):
        inventory.append({"root": legacy, "language": "sw", "valid": False,
                          "issues": ["LEGACY_WITHOUT_ROW_LEVEL_SPEAKER_AUDIT"]})
    chosen = {}
    for language in LANGUAGES:
        options = [x for x in valid if x.get("language") == language]
        if options:
            chosen[language] = max(options, key=lambda x: (
                str(x.get("finished_at") or ""), str(x.get("build_id") or "")))
    return inventory, chosen

def _quick_source_fingerprint():
    entries = []
    for language, item in sorted(V4_CHOSEN_CACHES.items()):
        root = item["root"]
        for path in [item["meta_path"]] + sorted(glob.glob(
                os.path.join(root, "**", "*.parquet"), recursive=True)):
            stat = os.stat(path)
            entry = {"language": language, "path": os.path.relpath(path, DRIVE_ROOT),
                     "size": int(stat.st_size)}
            if path.endswith(".json"):
                entry["sha256"] = _sha256_file(path)
            else:
                metadata = pq.ParquetFile(path).metadata
                entry.update(rows=metadata.num_rows, row_groups=metadata.num_row_groups,
                             schema=str(pq.ParquetFile(path).schema_arrow))
            entries.append(entry)
    curated_entries = []
    if os.path.exists(V4_SELECTION_PATH):
        selection = pd.read_parquet(V4_SELECTION_PATH,
                                    columns=["source_kind", "audio_ref"])
        for path in sorted(set(selection.loc[
                selection.source_kind.eq("file"), "audio_ref"].astype(str))):
            stat = os.stat(path)
            curated_entries.append({"path": os.path.relpath(path, DRIVE_ROOT),
                                    "size": int(stat.st_size),
                                    "mtime_ns": int(stat.st_mtime_ns)})
    payload = {"manifest_sha256": _manifest_sha256(), "cache_files": entries,
               "curated_selected_files": curated_entries}
    return {"payload": payload, "sha256": _json_digest(payload)}

if not RUN_DATA_AUDIT_V4:
    raise SystemExit("Helpers 18.2 chargés. Pour recalculer le bilan : RUN_DATA_AUDIT_V4=True.")
CACHE_INVENTORY_V4, V4_CHOSEN_CACHES = _cache_candidates()

# Listing Drive robuste contre les erreurs FUSE temporaires.
def _drive_dir_names(directory, attempts=12):
    delay, last_error = 2.0, None
    for attempt in range(1, attempts + 1):
        try:
            if not os.path.isdir(directory):
                return set()
            with os.scandir(directory) as entries:
                return {entry.name for entry in entries}
        except OSError as exc:
            last_error = exc
            print(f"Drive FUSE : {directory} — essai {attempt}/{attempts}, "
                  f"reprise dans {delay:.1f}s")
            time.sleep(delay)
            delay = min(30.0, delay*1.7)
    raise RuntimeError(f"Dossier Drive illisible : {directory}") from last_error

_dir_files = {}
for directory in manifest["curated_audio_path"].astype(str).map(os.path.dirname).unique():
    _dir_files[directory] = _drive_dir_names(directory)

rows, issues, warnings = [], [], []
speaker_col = next((c for c in ("speaker_id", "speaker", "recorder_uuid", "client_id")
                    if c in manifest.columns), None)
source_id_col = "source_id"
warnings.append({
    "code": "COMPOSITE_SOURCE_ID_USED",
    "detail": ("Le manifest ne contient pas l'ID source brut; une provenance composite "
               "déterministe est utilisée et les caches doivent rester speaker-disjoint."),
})
for raw in manifest.to_dict("records"):
    language = str(raw["language"])
    split = {"valid": "dev"}.get(str(raw["split"]), str(raw["split"]))
    path = str(raw["curated_audio_path"])
    exists = os.path.basename(path) in _dir_files.get(os.path.dirname(path), set())
    text_norm_value = _clean_label(raw.get("text_norm"))
    text = text_norm_value or normalize_text(raw.get("text", ""), language)
    method, conflict = _classify_method(raw)
    seconds = float(raw.get("duration") or 0)
    sample_id = _clean_label(raw.get("sample_id"))
    source_id = _canonical_id(raw.get(source_id_col) if source_id_col else sample_id)
    flags = _quality_flags(language, split, seconds, text, method, conflict, exists)
    # UNKNOWN_METHOD reste visible mais n'est pas fatal : le plan équilibre
    # adaptation_role et ne prétend pas résoudre une taxonomie incertaine.
    fatal = {"MISSING_AUDIO", "BAD_DURATION", "TOO_LONG_FOR_CTC",
             "EMPTY_OR_TOO_SHORT_TEXT", "ENCODING_SUSPECT", "METHOD_CONFLICT"}
    rows.append({
        "sample_key": f"curated:{sample_id}", "source_sample_id": source_id,
        "language": language, "split": split, "speech_method": method,
        "adaptation_role": "legacy_replay", "topic": str(raw.get("domain") or "unknown"),
        "source_dataset": "curated_manifest", "cache_build_id": "",
        "seconds": seconds, "words": len(text.split()),
        "wps": len(text.split()) / seconds if seconds > 0 else np.nan,
        "speaker_id": _clean_label(raw.get(speaker_col)) if speaker_col else "",
        "source_kind": "file", "audio_ref": path, "parquet_path": "", "row_index": -1,
        "text": text, "taxonomy_conflict": bool(conflict),
        "quality_flags": "|".join(flags), "usable": not bool(fatal & set(flags)),
    })

# Ajout des caches spontanés versionnés : métadonnées audio uniquement, jamais les octets.
for language, item in sorted(V4_CHOSEN_CACHES.items()):
    root, build_id = item["root"], str(item["build_id"])
    audio_parts = sorted(glob.glob(os.path.join(
        root, "version=0", "corpus=*", "split=*", "language=*", "*.parquet")))
    for audio_path in audio_parts:
        match = re.search(r"split=([^/\\]+)", audio_path)
        split = match.group(1) if match else "unknown"
        if split not in {"train", "dev"}:
            issues.append({"code": "CACHE_UNKNOWN_SPLIT", "language": language,
                           "detail": audio_path})
            continue
        if f"language={omni_lang(language)}" not in audio_path:
            issues.append({"code": "CACHE_PARTITION_LANGUAGE_MISMATCH",
                           "language": language, "detail": audio_path})
            continue
        audit_path = os.path.join(root, "audit", os.path.basename(audio_path))
        if not os.path.exists(audit_path):
            issues.append({"code": "CACHE_AUDIT_PART_MISSING", "language": language,
                           "detail": audit_path})
            continue
        audio = pq.read_table(audio_path, columns=["text", "audio_size"]).to_pandas()
        audit = pq.read_table(audit_path).to_pandas()
        audit = audit[audit["split"].astype(str).eq(split)].reset_index(drop=True)
        if len(audio) != len(audit):
            issues.append({"code": "CACHE_POSITIONAL_JOIN_MISMATCH", "language": language,
                           "detail": f"{audio_path}: audio={len(audio)} audit={len(audit)}"})
            continue
        audio_seconds = audio["audio_size"].astype(float).to_numpy() / 16000.0
        audit_seconds = pd.to_numeric(audit["duration_s"], errors="coerce").to_numpy()
        tolerances = np.maximum(.25, .05 * audio_seconds)
        duration_bad = (~np.isfinite(audit_seconds)) | \
            (np.abs(audio_seconds-audit_seconds) > tolerances)
        if duration_bad.any():
            bad_indices = np.flatnonzero(duration_bad)[:10].tolist()
            issues.append({"code": "CACHE_POSITIONAL_DURATION_MISMATCH",
                           "language": language, "detail": audio_path,
                           "indices": bad_indices, "count": int(duration_bad.sum())})
            continue
        for index, (a, m) in enumerate(zip(audio.to_dict("records"),
                                            audit.to_dict("records"))):
            text = str(a.get("text") or "")
            seconds = float(a.get("audio_size") or 0) / 16000.0
            sample_id = _clean_label(m.get("sample_id")) or \
                f"{os.path.basename(audio_path)}:{index}"
            flags = _quality_flags(language, split, seconds, text,
                                   "unscripted", False, True)
            fatal = {"BAD_DURATION", "TOO_LONG_FOR_CTC", "EMPTY_OR_TOO_SHORT_TEXT",
                     "ENCODING_SUSPECT"}
            rows.append({
                "sample_key": f"cache:{build_id}:{sample_id}",
                "source_sample_id": _canonical_id(sample_id), "language": language,
                "split": split,
                "speech_method": "unscripted", "adaptation_role": "new_in_domain",
                "topic": "spontaneous_cache", "source_dataset": "versioned_spont_cache",
                "cache_build_id": build_id, "seconds": seconds,
                "words": len(text.split()), "wps": len(text.split()) / seconds,
                "speaker_id": _clean_label(m.get("speaker_id")), "source_kind": "parquet",
                "audio_ref": "", "parquet_path": audio_path, "row_index": int(index),
                "text": text, "taxonomy_conflict": False,
                "quality_flags": "|".join(flags), "usable": not bool(fatal & set(flags)),
            })

index = pd.DataFrame(rows)
assert len(index), "Index V4 vide"

_unknown_method = index[index.speech_method.eq("unknown")]
if len(_unknown_method):
    warnings.append({
        "code": "UNKNOWN_SPEECH_METHOD_RETAINED_AS_LEGACY_REPLAY",
        "detail": (_unknown_method.groupby(["language", "split"])
                   .agg(clips=("sample_key", "size"),
                        hours=("seconds", lambda values: values.sum()/3600))
                   .reset_index().round(5).to_dict("records")),
    })
os.makedirs(os.path.dirname(V4_INDEX_PATH), exist_ok=True)
index.to_parquet(V4_INDEX_PATH, index=False)

# Contrôles croisés : IDs, locuteurs, encodage, disponibilité des caches.
duplicate_keys = index[index.duplicated("sample_key", keep=False)]
if len(duplicate_keys):
    issues.append({"code": "DUPLICATE_SAMPLE_KEY", "count": len(duplicate_keys)})
known_ids = index[index.source_sample_id.astype(str).str.len().gt(0)]
source_overlap = (known_ids[known_ids.split.eq("train")].groupby(
    ["language", "source_sample_id"])["adaptation_role"].nunique())
source_overlap = source_overlap[source_overlap > 1]
if len(source_overlap):
    issues.append({"code": "CURATED_CACHE_SOURCE_ID_OVERLAP", "count": len(source_overlap)})
split_id_leaks = []
for language, grp in known_ids.groupby("language"):
    train_ids = set(grp.loc[grp.split.eq("train"), "source_sample_id"])
    held_ids = set(grp.loc[grp.split.isin(["dev", "local_test"]), "source_sample_id"])
    overlap = sorted(train_ids & held_ids)
    if overlap:
        split_id_leaks.append({"language": language, "count": len(overlap),
                               "examples": overlap[:10]})
if split_id_leaks:
    issues.append({"code": "SOURCE_ID_TRAIN_HELDOUT_LEAKAGE", "details": split_id_leaks})

# Recontrôle indépendant du test lorsque son inventaire Drive existe.
test_inventory_path = os.path.join(FT_CONFIG["report_dir"], "test_inventory.json")
if os.path.exists(test_inventory_path):
    test_inventory = json.load(open(test_inventory_path, encoding="utf-8"))
    test_ids = defaultdict(set)
    missing_test_shards = []
    for shard in test_inventory.get("shards", []):
        path = shard.get("path")
        language = "sw" if shard.get("lang") == "swa" else shard.get("lang")
        if path and os.path.exists(path):
            for value in pq.read_table(path, columns=["id"])["id"].to_pylist():
                canonical = _canonical_id(value)
                if canonical: test_ids[language].add(canonical)
        else:
            missing_test_shards.append(path)
    if missing_test_shards:
        warnings.append({"code": "TEST_SHARDS_NOT_MOUNTED_FOR_RECHECK",
                         "detail": {"count": len(missing_test_shards),
                                    "examples": missing_test_shards[:5]}})
    for language in LANGUAGES:
        overlap = set(known_ids[(known_ids.language.eq(language)) &
                                known_ids.split.eq("train")].source_sample_id) & \
                  test_ids.get(language, set())
        if overlap:
            issues.append({"code": "TRAIN_TEST_ID_LEAKAGE", "language": language,
                           "count": len(overlap), "examples": sorted(overlap)[:10]})
else:
    warnings.append({"code": "TEST_ID_RECHECK_NOT_AVAILABLE",
                     "detail": "test_inventory.json absent; les builders source gardent leur contrôle anti-test."})
speaker_leaks = []
for language, grp in index[index.speaker_id.astype(str).str.len().gt(0)].groupby("language"):
    train_speakers = set(grp.loc[grp.split.eq("train"), "speaker_id"])
    held_speakers = set(grp.loc[grp.split.isin(["dev", "local_test"]), "speaker_id"])
    overlap = sorted(train_speakers & held_speakers)
    if overlap: speaker_leaks.append({"language": language, "count": len(overlap),
                                      "examples": overlap[:10]})
if speaker_leaks:
    issues.append({"code": "SPEAKER_LEAKAGE", "details": speaker_leaks})

_cross_role_speakers = []
for language, grp in index[
        index.split.eq("train") & index.speaker_id.astype(str).str.len().gt(0)
        ].groupby("language"):
    counts = grp.groupby("speaker_id")["adaptation_role"].nunique()
    overlap = counts[counts > 1].index.astype(str).tolist()
    if overlap:
        _cross_role_speakers.append({"language": language, "count": len(overlap),
                                     "examples": overlap[:10]})
if _cross_role_speakers:
    issues.append({"code": "SPEAKER_OVERLAP_BETWEEN_ADAPTATION_ROLES",
                   "details": _cross_role_speakers})
for (language, role), grp in index[index.split.eq("train")].groupby(
        ["language", "adaptation_role"]):
    missing_fraction = grp.speaker_id.astype(str).str.len().eq(0).mean()
    if missing_fraction > .05:
        issues.append({"code": "SPEAKER_PROVENANCE_MISSING", "language": language,
                       "role": role, "fraction": round(float(missing_fraction), 5)})
    elif missing_fraction > .01:
        warnings.append({"code": f"SPEAKER_MISSING_{language.upper()}_{role.upper()}",
                         "detail": round(float(missing_fraction), 5)})
text_split_overlap = []
for language, grp in index[index.text.astype(str).str.len().gt(0)].groupby("language"):
    train_text = set(grp.loc[grp.split.eq("train"), "text"])
    held_text = set(grp.loc[grp.split.isin(["dev", "local_test"]), "text"])
    overlap = train_text & held_text
    if overlap: text_split_overlap.append({"language": language, "count": len(overlap)})
if text_split_overlap:
    warnings.append({"code": "NORMALIZED_TEXT_CROSS_SPLIT_DUPLICATES",
                     "detail": text_split_overlap})
for language in LANGUAGES:
    if language not in V4_CHOSEN_CACHES:
        issues.append({"code": "VERSIONED_SPONT_CACHE_MISSING", "language": language,
                       "detail": "Construire un cache V3 versionné avec audit locuteur."})
    valid_options = [item for item in CACHE_INVENTORY_V4
                     if item.get("language") == language and item.get("valid")]
    if len(valid_options) > 1:
        warnings.append({"code": f"MULTIPLE_COMPLETE_CACHES_{language.upper()}",
                         "detail": [item.get("root") for item in valid_options]})
    for role in ("legacy_replay", "new_in_domain"):
        dev_h = index[(index.language.eq(language)) & index.split.eq("dev") &
                      index.adaptation_role.eq(role) & index.usable].seconds.sum()/3600
        if dev_h < .25:
            issues.append({"code": "INSUFFICIENT_DEV_ROLE_HOURS", "language": language,
                           "role": role, "detail": f"{dev_h:.3f} h < 0.25 h"})

sw_scripted_h = index[(index.language.eq("sw")) & index.speech_method.eq("scripted")][
    "seconds"].sum() / 3600
if sw_scripted_h > 0:
    warnings.append({"code": "SW_METHOD_LABEL_DIFFERS_FROM_OFFICIAL_REFERENCE",
                     "detail": f"{sw_scripted_h:.2f} h locales marquées scripted; table de référence=0 h read. "
                               "Le plan utilise adaptation_role et ne rebaptise pas ces lignes."})

# Inventaires persistants avant sélection.
inv = (index.groupby(["language", "split", "speech_method", "adaptation_role"],
                     dropna=False).agg(clips=("sample_key", "size"),
                     hours=("seconds", lambda x: x.sum() / 3600),
                     words=("words", "sum"), speakers=("speaker_id", "nunique"),
                     duration_p50=("seconds", "median"),
                     duration_p95=("seconds", lambda x: x.quantile(.95)),
                     duration_p99=("seconds", lambda x: x.quantile(.99)),
                     rejected=("usable", lambda x: int((~x).sum()))).reset_index())
inv.to_csv(V4_AUDIT_CSV, index=False)
bucket_labels = ["[0,.5)", "[.5,2)", "[2,10)", "[10,20)", "[20,30)", "[30,38]", ">38"]
bucket_codes = pd.cut(index.seconds, [-np.inf, .5, 2, 10, 20, 30, 38, np.inf],
                      labels=bucket_labels, right=True)
duration_inv = (index.assign(duration_bucket=bucket_codes).groupby(
    ["language", "split", "duration_bucket"], observed=True).agg(
    clips=("sample_key", "size"), hours=("seconds", lambda x: x.sum()/3600)).reset_index())
duration_inv.to_csv(os.path.join(V4_REPORT_DIR, "duration_buckets_v4.csv"), index=False)
index[index.quality_flags.astype(str).str.len().gt(0)][[
    "sample_key", "language", "split", "quality_flags"]].to_csv(
    os.path.join(V4_REPORT_DIR, "quality_flags_v4.csv"), index=False)
index[index.taxonomy_conflict][[
    "sample_key", "language", "speech_method", "topic", "audio_ref"]].to_csv(
    os.path.join(V4_REPORT_DIR, "taxonomy_conflicts_v4.csv"), index=False)
pd.DataFrame(CACHE_INVENTORY_V4).drop(columns=["audio_parts", "audit_parts"],
    errors="ignore").to_csv(os.path.join(V4_REPORT_DIR, "cache_inventory_v4.csv"), index=False)

def _deterministic_take(pool, target_seconds, salt):
    pool = pool.copy()
    pool["_speaker"] = pool.speaker_id.astype(str)
    pool.loc[pool._speaker.str.len().eq(0), "_speaker"] = pool.loc[
        pool._speaker.str.len().eq(0), "sample_key"]
    by_speaker = {}
    for speaker, grp in pool.groupby("_speaker", sort=False):
        by_speaker[speaker] = sorted(grp.index, key=lambda i: hashlib.sha256(
            f"{V4_SEED}:{salt}:{pool.at[i, 'sample_key']}".encode()).hexdigest())
    speaker_order = sorted(by_speaker, key=lambda s: hashlib.sha256(
        f"{V4_SEED}:{salt}:speaker:{s}".encode()).hexdigest())
    order, depth = [], 0
    while True:
        added = False
        for speaker in speaker_order:
            if depth < len(by_speaker[speaker]):
                order.append(by_speaker[speaker][depth]); added = True
        if not added: break
        depth += 1
    selected, total = [], 0.0
    for idx in order:
        selected.append(idx); total += float(pool.at[idx, "seconds"])
        if total >= target_seconds: break
    return pool.loc[selected].drop(columns=["_speaker"])

selected_frames, plan_rows = [], []
eligible = index[index.split.eq("train") & index.usable].copy()
for language in LANGUAGES:
    lang = eligible[eligible.language.eq(language)]
    new_pool = lang[lang.adaptation_role.eq("new_in_domain")]
    replay_pool = lang[lang.adaptation_role.eq("legacy_replay")]
    new_h, replay_h = new_pool.seconds.sum()/3600, replay_pool.seconds.sum()/3600
    ratio = V4_TARGET_NEW_IN_DOMAIN[language]
    total_h = min(new_h / ratio if ratio else np.inf,
                  replay_h / (1-ratio) if ratio < 1 else np.inf,
                  V4_MAX_HOURS_PER_LANGUAGE)
    if not np.isfinite(total_h): total_h = min(new_h + replay_h, V4_MAX_HOURS_PER_LANGUAGE)
    target_new_h, target_replay_h = total_h * ratio, total_h * (1-ratio)
    new_sel = _deterministic_take(new_pool, target_new_h*3600, f"{language}:new") \
        if target_new_h > 0 and len(new_pool) else new_pool.iloc[0:0]
    replay_sel = _deterministic_take(replay_pool, target_replay_h*3600, f"{language}:replay") \
        if target_replay_h > 0 and len(replay_pool) else replay_pool.iloc[0:0]
    chosen = pd.concat([new_sel, replay_sel], ignore_index=False)
    selected_frames.append(chosen)
    actual_h = chosen.seconds.sum()/3600
    actual_new_h = chosen[chosen.adaptation_role.eq("new_in_domain")].seconds.sum()/3600
    actual_ratio = actual_new_h/actual_h if actual_h else 0.0
    method_hours = chosen.groupby("speech_method").seconds.sum().div(3600).to_dict()
    plan_rows.append({
        "language": language, "available_new_hours": round(new_h, 4),
        "available_replay_hours": round(replay_h, 4), "target_new_ratio": ratio,
        "planned_hours": round(actual_h, 4), "planned_new_hours": round(actual_new_h, 4),
        "achieved_new_ratio": round(actual_ratio, 5), "exposure_factor": None,
        "scripted_hours": round(float(method_hours.get("scripted", 0)), 4),
        "unscripted_hours": round(float(method_hours.get("unscripted", 0)), 4),
        "achieved_unscripted_ratio": round(
            float(method_hours.get("unscripted", 0))/actual_h, 5) if actual_h else None,
        "selected_clips": int(len(chosen)),
    })
    if actual_h < V4_MIN_HOURS_PER_LANGUAGE:
        issues.append({"code": "TOO_FEW_PLANNED_HOURS", "language": language,
                       "detail": f"{actual_h:.2f} h < {V4_MIN_HOURS_PER_LANGUAGE} h"})
    if abs(actual_ratio-ratio) > V4_RATIO_TOLERANCE:
        issues.append({"code": "DOMAIN_ROLE_RATIO_UNMET", "language": language,
                       "detail": f"target={ratio:.3f}, actual={actual_ratio:.3f}"})

# Viser au plus trois passages sur la langue la plus petite, avec plafond absolu 2 500.
hours_per_step_per_language = 40_960_000 / 16000 / 3600 / 6
min_planned_h = min((row["planned_hours"] for row in plan_rows), default=0)
steps_for_three_passes = int((min_planned_h*3 / hours_per_step_per_language)//250*250)
recommended_max_steps = max(500, min(2500, steps_for_three_passes))
exposure_hours = recommended_max_steps * hours_per_step_per_language
for row in plan_rows:
    row["exposure_factor"] = round(exposure_hours/row["planned_hours"], 3) \
        if row["planned_hours"] else None
    if row["exposure_factor"] and row["exposure_factor"] > 4:
        warnings.append({"code": f"HIGH_EXPOSURE_{row['language'].upper()}",
                         "detail": f"{row['exposure_factor']:.2f} passages nominaux"})
    if row["exposure_factor"] and row["exposure_factor"] > 8:
        issues.append({"code": "EXCESSIVE_EXPOSURE", "language": row["language"],
                       "detail": row["exposure_factor"]})

selected = pd.concat(selected_frames, ignore_index=True) if selected_frames else index.iloc[0:0]
selected.to_parquet(V4_SELECTION_PATH, index=False)
selection_sha256 = _frame_sha256(selected)
index_sha256 = _frame_sha256(index)
source_fingerprint = _quick_source_fingerprint()
language_weights = {language: 1/6 for language in LANGUAGES}
assert abs(sum(language_weights.values()) - 1.0) < 1e-12

plan_payload = {"schema": V4_AUDIT_SCHEMA, "axis": "adaptation_role",
        "axis_note": "new_in_domain vs legacy_replay; ce n'est pas un renommage scripted/unscripted",
        "target_new_in_domain": V4_TARGET_NEW_IN_DOMAIN, "languages": plan_rows,
        "language_weights": language_weights, "beta_language": 0.0,
        "absolute_step_cap": 2500, "recommended_max_steps": recommended_max_steps,
        "learning_rate": 3e-6,
        "effective_elements_per_update": 40_960_000,
        "selection_path": V4_SELECTION_PATH, "selection_sha256": selection_sha256}
plan_sha256 = _json_digest(plan_payload)
plan = {**plan_payload, "plan_sha256": plan_sha256}
save_json(plan, V4_PLAN_PATH)

payload = {
    "schema": V4_AUDIT_SCHEMA, "source_id_mode": V4_SOURCE_ID_MODE, "status": "PASS" if not issues else "FAIL",
    "revision": V4_REVISION, "manifest_sha256": _manifest_sha256(),
    "data_index_path": V4_INDEX_PATH, "data_index_sha256": index_sha256,
    "selection_path": V4_SELECTION_PATH, "plan_path": V4_PLAN_PATH,
    "plan_sha256": plan_sha256, "selection_sha256": selection_sha256,
    "source_fingerprint": source_fingerprint,
    "chosen_caches": {k: {"root": v["root"], "build_id": v["build_id"]}
                      for k, v in V4_CHOSEN_CACHES.items()},
    "inventory_rows": int(len(index)), "eligible_train_rows": int(len(eligible)),
    "selected_rows": int(len(selected)), "plan": plan_rows,
    "official_reference_hours": V4_OFFICIAL_REFERENCE_HOURS,
    "critical_issues": issues, "warnings": warnings,
}
audit_sha256 = _json_digest(payload)
save_json({"created_at": now_iso(), "audit": payload,
           "audit_sha256": audit_sha256}, V4_AUDIT_PATH)
print("\nSTATUT AUDIT :", payload["status"])
print("audit_sha256 =", audit_sha256)
print("plan_sha256  =", plan_sha256)
display(inv.round(4))
display(pd.DataFrame(plan_rows))
if issues:
    print("\nBLOQUANTS (corriger puis relancer l'audit) :")
    for item in issues: print(" -", item)
if warnings:
    print("\nWARNINGS à lire et acquitter dans 18.3 :")
    for item in warnings: print(" -", item)


In [ ]:
# 18.3 — QA_V4: AUDIT_GATE — signature humaine + vérification anti-dérive
V4_APPROVAL_PATH = os.path.join(V4_REPORT_DIR, "audit_approval_v4.json")

# Après lecture de data_inventory_v4.csv, sampling_plan_v4.json et des warnings,
# recopier les deux empreintes affichées par 18.2, renseigner le nom et l'heure UTC.
V4_AUDIT_APPROVAL = {
    "decision": "GO",                 # exactement "GO"
    "approved_by": "notebook_owner",              # non vide
    "approved_at_utc": now_iso(),          # ISO-8601
    "audit_sha256": "REPLACE_WITH_LOCAL_SHA256",
    "plan_sha256": "REPLACE_WITH_LOCAL_SHA256",
    "acknowledged_warnings": [
        "COMPOSITE_SOURCE_ID_USED",
        "UNKNOWN_SPEECH_METHOD_RETAINED_AS_LEGACY_REPLAY",
        "NORMALIZED_TEXT_CROSS_SPLIT_DUPLICATES"
    ],     # codes exacts du rapport
}
if os.path.exists(V4_APPROVAL_PATH) and V4_AUDIT_APPROVAL["decision"] != "GO":
    V4_AUDIT_APPROVAL = json.load(open(V4_APPROVAL_PATH, encoding="utf-8"))

def require_audit_pass():
    assert os.path.exists(V4_AUDIT_PATH), "Exécuter 18.2 : data_audit_v4.json absent"
    report = json.load(open(V4_AUDIT_PATH, encoding="utf-8"))
    payload = report["audit"]
    digest = _json_digest(payload)
    assert digest == report["audit_sha256"], "Rapport d'audit modifié : relancer 18.2"
    assert payload["status"] == "PASS", payload["critical_issues"]
    assert payload["manifest_sha256"] == _manifest_sha256(), \
        "Manifest modifié depuis l'audit"
    assert os.path.exists(V4_INDEX_PATH) and _frame_sha256(
        pd.read_parquet(V4_INDEX_PATH)) == \
        payload["data_index_sha256"], "Index V4 absent ou modifié"
    assert os.path.exists(V4_SELECTION_PATH) and _frame_sha256(
        pd.read_parquet(V4_SELECTION_PATH)) == \
        payload["selection_sha256"], "Sélection V4 absente ou modifiée"
    plan = json.load(open(V4_PLAN_PATH, encoding="utf-8"))
    declared_plan_hash = plan.pop("plan_sha256")
    assert _json_digest(plan) == declared_plan_hash == payload["plan_sha256"], \
        "Plan JSON modifié depuis l'approbation"
    assert plan["selection_sha256"] == payload["selection_sha256"]

    # Redécouverte : un cache plus récent ou supprimé invalide la décision.
    _, current_caches = _cache_candidates()
    current_ids = {k: v["build_id"] for k, v in current_caches.items()}
    audited_ids = {k: v["build_id"] for k, v in payload["chosen_caches"].items()}
    assert current_ids == audited_ids, \
        f"Caches modifiés depuis l'audit : audit={audited_ids}, actuel={current_ids}"
    old_chosen = globals().get("V4_CHOSEN_CACHES")
    globals()["V4_CHOSEN_CACHES"] = current_caches
    current_fp = _quick_source_fingerprint()
    if old_chosen is not None:
        globals()["V4_CHOSEN_CACHES"] = old_chosen
    assert current_fp["sha256"] == payload["source_fingerprint"]["sha256"], \
        "Sources modifiées depuis l'audit"

    approval = V4_AUDIT_APPROVAL
    assert approval["decision"] == "GO", "Lire le bilan puis signer V4_AUDIT_APPROVAL"
    assert str(approval["approved_by"]).strip(), "approved_by vide"
    assert str(approval["approved_at_utc"]).strip(), "approved_at_utc vide"
    assert approval["audit_sha256"] == digest, "Mauvais audit_sha256 dans l'approbation"
    assert approval["plan_sha256"] == payload["plan_sha256"], \
        "Mauvais plan_sha256 dans l'approbation"
    expected = sorted(w["code"] for w in payload.get("warnings", []))
    assert sorted(approval.get("acknowledged_warnings", [])) == expected, \
        f"Warnings à acquitter exactement : {expected}"
    save_json(approval, V4_APPROVAL_PATH)
    return payload

if any((RUN_BALANCED_BUILD_V4, RUN_A100_80GB_CALIBRATION_V4,
        RUN_BALANCED_FINETUNE_V4, RUN_BALANCED_EVAL_V4)):
    _audit_ok = require_audit_pass()
    print("✅ Audit signé et intact :", _audit_ok["plan_sha256"][:16])
else:
    print("Gate prêt. Exécuter d'abord 18.2, lire le bilan, puis remplir V4_AUDIT_APPROVAL.")


In [ ]:
# 18.4 — Matérialisation du mélange signé (pas de nouvelle sélection ici)
if not RUN_BALANCED_BUILD_V4:
    raise SystemExit("Build V4 bloqué : mettre RUN_BALANCED_BUILD_V4=True après PASS + signature.")
audit = require_audit_pass()   # avant toute suppression/écriture lourde

import io
import pyarrow as pa
import pyarrow.parquet as pq

selected_train = pd.read_parquet(V4_SELECTION_PATH)
full_index = pd.read_parquet(V4_INDEX_PATH)
assert set(selected_train.sample_key).issubset(set(full_index.sample_key))
assert selected_train.split.eq("train").all() and selected_train.usable.all()

def _take_dev_v4(pool, target_seconds, salt):
    pool = pool.copy()
    pool["_rank"] = pool.sample_key.map(lambda key: hashlib.sha256(
        f"{V4_SEED}:{salt}:{key}".encode()).hexdigest())
    pool = pool.sort_values("_rank")
    keep, total = [], 0.0
    for idx, row in pool.iterrows():
        keep.append(idx); total += float(row.seconds)
        if total >= target_seconds: break
    return pool.loc[keep].drop(columns="_rank")

# Dev de recette de 2 h/langue, réparti selon le même ratio de rôles que le plan.
dev_parts = []
eligible_dev = full_index[full_index.split.eq("dev") & full_index.usable].copy()
for language in LANGUAGES:
    ratio = V4_TARGET_NEW_IN_DOMAIN[language]
    role_targets = {"legacy_replay": 2*3600*(1-ratio),
                    "new_in_domain": 2*3600*ratio}
    for role, target_seconds in role_targets.items():
        pool = eligible_dev[(eligible_dev.language.eq(language)) &
                            eligible_dev.adaptation_role.eq(role)]
        if len(pool):
            dev_parts.append(_take_dev_v4(pool, target_seconds,
                                          f"dev:{language}:{role}"))
selected_dev = pd.concat(dev_parts, ignore_index=True) if dev_parts else eligible_dev.iloc[0:0]
assert set(selected_train.sample_key).isdisjoint(set(selected_dev.sample_key))
selected_dev.to_parquet(V4_DEV_SELECTION_PATH, index=False)

stage = V4_DATASET_ROOT + ".staging"
shutil.rmtree(stage, ignore_errors=True)
stage_version = os.path.join(stage, "version=0")
os.makedirs(stage_version, exist_ok=True)

def _atomic_parquet_v4(records, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    temporary = path + f".tmp-{os.getpid()}"
    pq.write_table(pa.Table.from_pylist(records), temporary, row_group_size=100,
                   compression="zstd", compression_level=3)
    os.replace(temporary, path)

_cached_path, _cached_rows = None, None
def _materialize_row(row):
    global _cached_path, _cached_rows
    if row["source_kind"] == "file":
        path = row["audio_ref"]
        info = sf.info(path)
        assert info.samplerate == 16000 and info.channels == 1, \
            f"Audio curé non 16k mono : {path}"
        with open(path, "rb") as handle:
            audio_bytes = handle.read()
        return {"text": row["text"], "audio_bytes": audio_bytes,
                "audio_size": int(info.frames)}
    path = row["parquet_path"]
    if path != _cached_path:
        _cached_rows = pq.read_table(path).to_pylist()
        _cached_path = path
    source = _cached_rows[int(row["row_index"])]
    return {"text": str(row["text"]), "audio_bytes": bytes(source["audio_bytes"]),
            "audio_size": int(source["audio_size"])}

materialized_stats, written = [], []
for split, frame in (("train", selected_train), ("dev", selected_dev)):
    for language in LANGUAGES:
        subset = frame[frame.language.eq(language)].copy()
        subset["_sort"] = subset.apply(lambda r: hashlib.sha256(
            f"{V4_SEED}:materialize:{r.sample_key}".encode()).hexdigest(), axis=1)
        subset = subset.sort_values(["source_kind", "parquet_path", "_sort"])
        out_dir = os.path.join(stage_version, "corpus=afrivoices_balanced_v4",
                               f"split={split}", f"language={omni_lang(language)}")
        records, part, samples = [], 0, 0
        role_samples, role_clips = defaultdict(int), Counter()
        for row in subset.to_dict("records"):
            materialized = _materialize_row(row)
            records.append(materialized); samples += materialized["audio_size"]
            role_samples[row["adaptation_role"]] += materialized["audio_size"]
            role_clips[row["adaptation_role"]] += 1
            if len(records) >= 500:
                path = os.path.join(out_dir, f"part-{part:05d}.parquet")
                _atomic_parquet_v4(records, path); written.append(path)
                records, part = [], part + 1
        if records:
            path = os.path.join(out_dir, f"part-{part:05d}.parquet")
            _atomic_parquet_v4(records, path); written.append(path)
        for role in ("legacy_replay", "new_in_domain"):
            materialized_stats.append({"split": split, "language": language, "role": role,
                                       "clips": int(role_clips[role]),
                                       "hours": round(role_samples[role]/16000/3600, 6)})
        print(split, language, "hours=", round(samples/16000/3600, 4),
              dict(role_clips))

# Heures RÉELLES dans le TSV. beta_language=0.0 imposera ensuite exactement 1/6.
train_stats = pd.DataFrame(materialized_stats)
train_stats = train_stats[train_stats.split.eq("train")]
assert set(train_stats.language) == set(LANGUAGES)
for language in LANGUAGES:
    group = train_stats[train_stats.language.eq(language)]
    total_h = group.hours.sum()
    new_h = group.loc[group.role.eq("new_in_domain"), "hours"].sum()
    actual_ratio = new_h/total_h if total_h else 0
    assert total_h >= V4_MIN_HOURS_PER_LANGUAGE
    assert abs(actual_ratio-V4_TARGET_NEW_IN_DOMAIN[language]) <= V4_RATIO_TOLERANCE, \
        f"Ratio physique divergent {language}: {actual_ratio:.4f}"
tsv_stage = os.path.join(stage, "language_distribution_0.tsv")
with open(tsv_stage, "w", encoding="utf-8") as handle:
    handle.write("corpus\tlanguage\thours\n")
    for language in LANGUAGES:
        hours = float(train_stats.loc[train_stats.language.eq(language), "hours"].sum())
        handle.write(f"afrivoices_balanced_v4\t{omni_lang(language)}\t{hours:.6f}\n")

file_inventory = []
for path in sorted(written):
    stat = os.stat(path)
    parquet = pq.ParquetFile(path)
    sample_sum = sum(pq.read_table(path, columns=["audio_size"])["audio_size"].to_pylist())
    file_inventory.append({"path": os.path.relpath(path, stage), "size": int(stat.st_size),
                           "rows": parquet.metadata.num_rows, "audio_samples": int(sample_sum),
                           "sha256": _sha256_file(path)})
content_payload = {
    "schema": 2, "revision": V4_REVISION,
    "audit_sha256": json.load(open(V4_AUDIT_PATH))["audit_sha256"],
    "manifest_sha256": audit["manifest_sha256"], "plan_sha256": audit["plan_sha256"],
    "selection_sha256": audit["selection_sha256"],
    "dev_selection_sha256": _frame_sha256(selected_dev), "stats": materialized_stats,
    "files": file_inventory, "beta_language": 0.0, "language_probability": 1/6,
    "tsv_sha256": _sha256_file(tsv_stage),
}
meta_payload = {"created_at": now_iso(), **content_payload,
                "build_sha256": _json_digest(content_payload)}
save_json(meta_payload, os.path.join(stage, "_COMPLETE.json"))
backup = V4_DATASET_ROOT + ".backup"
shutil.rmtree(backup, ignore_errors=True)
if os.path.exists(V4_DATASET_ROOT): os.replace(V4_DATASET_ROOT, backup)
try:
    os.replace(stage, V4_DATASET_ROOT)
except Exception:
    if os.path.exists(backup): os.replace(backup, V4_DATASET_ROOT)
    raise
shutil.rmtree(backup, ignore_errors=True)
V4_TSV_PATH = os.path.join(V4_DATASET_ROOT, "language_distribution_0.tsv")
write_dataset_card(data_root=V4_DATA_VERSION_ROOT)
print("✅ Dataset V4 matérialisé :", V4_DATASET_ROOT)
display(pd.DataFrame(materialized_stats))


In [ ]:
# Sauvegarde reprenable du dataset V4 local vers Google Drive
import hashlib
import json
import os
import shutil
import time
from pathlib import Path

import pyarrow.parquet as pq
from tqdm.auto import tqdm

SOURCE_DATASET = Path("/content/afrivoices_balanced_v4/dataset")

DRIVE_BASE = Path(
    globals().get(
        "DRIVE_ROOT",
        "/content/afrivoices_project",
    )
)

DRIVE_BACKUP = (
    DRIVE_BASE
    / "dataset_caches"
    / "afrivoices_balanced_v4_dataset"
)

SOURCE_MARKER = SOURCE_DATASET / "_COMPLETE.json"
DRIVE_MARKER = DRIVE_BACKUP / "_COMPLETE.json"

assert SOURCE_DATASET.is_dir(), (
    f"Dataset local introuvable : {SOURCE_DATASET}"
)
assert SOURCE_MARKER.is_file(), (
    "Le dataset local n'est pas finalisé : _COMPLETE.json absent"
)

source_metadata = json.load(open(SOURCE_MARKER, encoding="utf-8"))
assert source_metadata.get("build_sha256"), (
    "build_sha256 absent du marqueur local"
)

def retry_drive(operation, label, attempts=12):
    delay = 2.0
    last_error = None

    for attempt in range(1, attempts + 1):
        try:
            return operation()
        except OSError as exc:
            last_error = exc
            print(
                f"{label} — essai {attempt}/{attempts} : {exc!r}; "
                f"reprise dans {delay:.1f}s"
            )
            time.sleep(delay)
            delay = min(delay * 1.7, 30.0)

    raise RuntimeError(
        f"Google Drive reste inaccessible : {label}"
    ) from last_error

retry_drive(
    lambda: DRIVE_BACKUP.mkdir(parents=True, exist_ok=True),
    "création du dossier de sauvegarde",
)

# Le marqueur final doit rester absent pendant une copie incomplète.
if DRIVE_MARKER.exists():
    retry_drive(
        lambda: DRIVE_MARKER.unlink(),
        "suppression temporaire de l'ancien marqueur",
    )

source_files = sorted(
    path
    for path in SOURCE_DATASET.rglob("*")
    if path.is_file()
    and path.name != "_COMPLETE.json"
    and ".tmp-" not in path.name
    and not path.name.endswith(".part")
)

assert source_files, "Aucun fichier trouvé dans le dataset"

def destination_for(source):
    return DRIVE_BACKUP / source.relative_to(SOURCE_DATASET)

def same_size(source, destination):
    try:
        return (
            destination.is_file()
            and destination.stat().st_size == source.stat().st_size
        )
    except OSError:
        return False

def remaining_bytes(source, destination):
    if same_size(source, destination):
        return 0

    temporary = Path(str(destination) + ".part")

    try:
        partial_size = temporary.stat().st_size
    except OSError:
        partial_size = 0

    if partial_size > source.stat().st_size:
        partial_size = 0

    return source.stat().st_size - partial_size

total_size = sum(path.stat().st_size for path in source_files)
remaining_size = sum(
    remaining_bytes(source, destination_for(source))
    for source in source_files
)

try:
    drive_free = shutil.disk_usage(DRIVE_BASE).free
except OSError:
    drive_free = 0

print("Source           :", SOURCE_DATASET)
print("Destination      :", DRIVE_BACKUP)
print("Nombre de fichiers :", len(source_files))
print("Taille totale    :", round(total_size / 2**30, 2), "Gio")
print("Reste à transférer :", round(remaining_size / 2**30, 2), "Gio")

if drive_free:
    print("Espace Drive libre :", round(drive_free / 2**30, 2), "Gio")
    assert drive_free >= remaining_size + 5 * 2**30, (
        "Espace Drive insuffisant. Il faut conserver au moins "
        "5 Gio de marge après la copie."
    )

def copy_resumable(source, destination, attempts=12):
    source_size = source.stat().st_size

    if same_size(source, destination):
        return "déjà présent"

    temporary = Path(str(destination) + ".part")
    delay = 2.0
    last_error = None

    for attempt in range(1, attempts + 1):
        try:
            destination.parent.mkdir(parents=True, exist_ok=True)

            if same_size(source, destination):
                return "déjà présent"

            try:
                offset = temporary.stat().st_size
            except OSError:
                offset = 0

            if offset > source_size:
                temporary.unlink(missing_ok=True)
                offset = 0

            with source.open("rb") as input_file:
                input_file.seek(offset)

                with temporary.open("ab") as output_file:
                    while True:
                        block = input_file.read(32 * 1024**2)

                        if not block:
                            break

                        output_file.write(block)

                    output_file.flush()

            copied_size = temporary.stat().st_size

            if copied_size != source_size:
                raise IOError(
                    f"copie partielle : "
                    f"{copied_size}/{source_size} octets"
                )

            os.replace(temporary, destination)

            if not same_size(source, destination):
                raise IOError(
                    "taille incorrecte après publication sur Drive"
                )

            return "copié"

        except (OSError, IOError) as exc:
            last_error = exc
            print(
                f"\nReprise {source.name}, "
                f"essai {attempt}/{attempts} : {exc!r}"
            )
            time.sleep(delay)
            delay = min(delay * 1.7, 30.0)

    raise RuntimeError(
        f"Impossible de copier {source}"
    ) from last_error

copied_count = 0
skipped_count = 0

for source in tqdm(
    source_files,
    desc="Sauvegarde dataset V4",
    unit="fichier",
):
    destination = destination_for(source)
    status = copy_resumable(source, destination)

    if status == "copié":
        copied_count += 1
    else:
        skipped_count += 1

print("Copiés       :", copied_count)
print("Déjà présents :", skipped_count)

# Vérification des tailles et des métadonnées Parquet.
def parquet_signature(path):
    parquet_file = pq.ParquetFile(path)

    return (
        parquet_file.metadata.num_rows,
        str(parquet_file.schema_arrow),
    )

validation_errors = []

for source in tqdm(
    source_files,
    desc="Vérification Drive",
    unit="fichier",
):
    destination = destination_for(source)

    if not same_size(source, destination):
        validation_errors.append(
            f"Taille incorrecte : "
            f"{destination.relative_to(DRIVE_BACKUP)}"
        )
        continue

    if source.suffix == ".parquet":
        try:
            source_signature = parquet_signature(source)
            drive_signature = retry_drive(
                lambda path=destination: parquet_signature(path),
                f"lecture de {destination.name}",
            )

            if source_signature != drive_signature:
                validation_errors.append(
                    f"Parquet différent : "
                    f"{destination.relative_to(DRIVE_BACKUP)}"
                )

        except Exception as exc:
            validation_errors.append(
                f"Parquet illisible : {destination} — {exc!r}"
            )

assert not validation_errors, "\n".join(validation_errors[:20])

# Publication du marqueur uniquement après validation complète.
copy_resumable(SOURCE_MARKER, DRIVE_MARKER)

drive_metadata = retry_drive(
    lambda: json.load(open(DRIVE_MARKER, encoding="utf-8")),
    "lecture du marqueur final",
)

assert drive_metadata == source_metadata, (
    "Le marqueur Drive diffère du marqueur local"
)

assert (
    drive_metadata["build_sha256"]
    == source_metadata["build_sha256"]
)

print()
print("✅ Dataset V4 sauvegardé et vérifié sur Drive")
print("Destination :", DRIVE_BACKUP)
print("Build SHA256 :", drive_metadata["build_sha256"])

In [ ]:
import json
import os
from pathlib import Path

BACKUP = Path(
    "/content/afrivoices_project/"
    "dataset_caches/afrivoices_balanced_v4_dataset"
)

MARKER = BACKUP / "_COMPLETE.json"

assert MARKER.is_file(), (
    "Sauvegarde incomplète : _COMPLETE.json absent. "
    "Ne supprime pas l'environnement."
)

metadata = json.load(open(MARKER, encoding="utf-8"))

missing = []
bad_sizes = []

for item in metadata["files"]:
    path = BACKUP / item["path"]

    if not path.is_file():
        missing.append(item["path"])
    elif path.stat().st_size != item["size"]:
        bad_sizes.append(item["path"])

tsv = BACKUP / "language_distribution_0.tsv"

assert not missing, f"Fichiers absents : {missing[:10]}"
assert not bad_sizes, f"Tailles incorrectes : {bad_sizes[:10]}"
assert tsv.is_file(), "language_distribution_0.tsv absent"
assert metadata.get("build_sha256"), "build_sha256 absent"

print("✅ SAUVEGARDE DRIVE COMPLÈTE")
print("Fichiers Parquet :", len(metadata["files"]))
print("Build SHA256     :", metadata["build_sha256"])
print("Emplacement      :", BACKUP)
print("Tu peux supprimer l'environnement.")

In [ ]:
# 18.5 — QA_V4: A100_CALIBRATION — mesure réelle, objectif >=15 % de marge VRAM
import gc, queue, threading, signal

V4_TSV_PATH = os.path.join(V4_DATASET_ROOT, "language_distribution_0.tsv")
_v4_plan_runtime = json.load(open(V4_PLAN_PATH, encoding="utf-8")) \
    if os.path.exists(V4_PLAN_PATH) else {}
V4_MAX_NEW_STEPS = int(_v4_plan_runtime.get("recommended_max_steps", 2500))
assert 500 <= V4_MAX_NEW_STEPS <= 2500 and V4_MAX_NEW_STEPS % 250 == 0
V4_LR = 3e-6
V4_CKPT_EVERY = 250
V4_TARGET_EFFECTIVE_ELEMENTS = 40_960_000
V4_A100_LADDER = [
    {"max_num_elements": 10_240_000, "grad_accum": 4},
    {"max_num_elements": 8_192_000, "grad_accum": 5},
    {"max_num_elements": 6_826_667, "grad_accum": 6},
    {"max_num_elements": 5_120_000, "grad_accum": 8},
    {"max_num_elements": 4_096_000, "grad_accum": 10},
]
for item in V4_A100_LADDER:
    item["effective_elements"] = item["max_num_elements"] * item["grad_accum"]
    assert abs(item["effective_elements"] - V4_TARGET_EFFECTIVE_ELEMENTS) / \
        V4_TARGET_EFFECTIVE_ELEMENTS < 1e-5

def require_v4_build(audit=None):
    audit = audit or require_audit_pass()
    assert os.path.exists(V4_BUILD_META), "Exécuter 18.4 : dataset V4 absent"
    meta = json.load(open(V4_BUILD_META, encoding="utf-8"))
    digest = meta.get("build_sha256")
    content = dict(meta); content.pop("build_sha256", None); content.pop("created_at", None)
    assert digest == _json_digest(content), "_COMPLETE.json V4 modifié"
    assert meta["manifest_sha256"] == audit["manifest_sha256"]
    assert meta["plan_sha256"] == audit["plan_sha256"]
    assert meta["selection_sha256"] == _frame_sha256(pd.read_parquet(V4_SELECTION_PATH))
    assert meta["dev_selection_sha256"] == _frame_sha256(
        pd.read_parquet(V4_DEV_SELECTION_PATH))
    for item in meta["files"]:
        path = os.path.join(V4_DATASET_ROOT, item["path"])
        assert os.path.exists(path) and os.path.getsize(path) == item["size"], \
            f"Parquet V4 absent/tronqué : {path}"
        assert _sha256_file(path) == item["sha256"], f"SHA256 parquet divergent : {path}"
        parquet = pq.ParquetFile(path)
        assert parquet.metadata.num_rows == item["rows"]
        samples = sum(pq.read_table(path, columns=["audio_size"])["audio_size"].to_pylist())
        assert int(samples) == item["audio_samples"]
    assert os.path.exists(V4_TSV_PATH) and _sha256_file(V4_TSV_PATH) == meta["tsv_sha256"]
    return meta

def _seed_weight_file():
    roots = [p for p in Path(FT_CONFIG["output_dir"]).rglob("step_5000")
             if p.is_dir() and (p / "model").is_dir() and not p.name.endswith(".tmp")]
    complete = [p for p in roots if (p / "optimizer").exists() and (p / "trainer").exists()]
    assert len(complete) == 1, \
        f"Il faut exactement un checkpoint joint COMPLET step_5000, trouvé : {complete}"
    weights = [p for p in (complete[0] / "model").rglob("*") if p.is_file()]
    assert len(weights) == 1, f"Un seul fichier de poids attendu : {weights}"
    return str(complete[0]), str(weights[0])

def _cached_seed_hash(weight):
    path = os.path.join(V4_REPORT_DIR, "seed_weight_hash_v4.json")
    stat = os.stat(weight)
    if os.path.exists(path):
        value = json.load(open(path, encoding="utf-8"))
        if value.get("path") == weight and value.get("size") == stat.st_size \
                and value.get("mtime_ns") == stat.st_mtime_ns:
            return value["sha256"]
    value = {"path": weight, "size": stat.st_size, "mtime_ns": stat.st_mtime_ns,
             "sha256": _sha256_file(weight), "computed_at": now_iso()}
    save_json(value, path)
    return value["sha256"]

def prepare_v4_seed_card():
    step_dir, weight = _seed_weight_file()
    assert int(re.search(r"step_(\d+)$", step_dir).group(1)) == 5000
    seed_hash = _cached_seed_hash(weight)
    import omnilingual_asr as oa
    card_name = "omniASR_CTC_1B_v2_ft5000_balanced_seed_v4"
    card_path = os.path.join(os.path.dirname(oa.__file__), "cards", "models",
                             "afrivoices_balanced_seed_v4.yaml")
    os.makedirs(os.path.dirname(card_path), exist_ok=True)
    probe_path = os.path.join(V4_SCRATCH_ROOT, "probe_seed_v4.py")
    probe_lines = [
        "import json, sys, torch",
        "weight, card, out = sys.argv[1:]",
        "import omnilingual_asr",
        "from fairseq2.models.hub import load_model",
        'model = load_model(card, device=torch.device("cpu"), dtype=torch.float32)',
        "params = sum(p.numel() for p in model.parameters())",
        "assert params == 975_675_056 and params < 1_000_000_000, params",
        "if weight.endswith('.safetensors'):",
        "    from safetensors.torch import load_file",
        "    raw = load_file(weight, device='cpu')",
        "else:",
        "    try: raw = torch.load(weight, map_location='cpu', weights_only=True)",
        "    except Exception: raw = torch.load(weight, map_location='cpu', weights_only=False)",
        "state = raw.get('model', raw) if isinstance(raw, dict) else raw",
        "current = model.state_dict()",
        "common = [k for k in state if k in current and state[k].shape == current[k].shape]",
        "assert len(common) > 100, (len(common), list(state)[:5], list(current)[:5])",
        "for key in (common[0], common[len(common)//2], common[-1]):",
        "    assert torch.equal(current[key].cpu(), state[key].cpu()), key",
        'json.dump({"params": params, "checked_tensors": 3}, open(out, "w"))',
    ]
    with open(probe_path, "w", encoding="utf-8") as handle:
        handle.write("\n".join(probe_lines) + "\n")
    probe_out = os.path.join(V4_REPORT_DIR, "seed_probe_v4.json")
    compile(open(probe_path, encoding="utf-8").read(), probe_path, "exec")
    result = None
    for uri in (f"file://{weight}", weight):
        with open(card_path, "w", encoding="utf-8") as handle:
            handle.write(f'name: {card_name}\nbase: {FT_CONFIG["model_card"]}\n'
                         f'checkpoint: "{uri}"\n')
        result = subprocess.run([sys.executable, probe_path, weight, card_name, probe_out],
                                capture_output=True, text=True)
        if result.returncode == 0: break
    assert result and result.returncode == 0, result.stdout + "\n" + result.stderr[-3000:]
    probe = json.load(open(probe_out))
    return {"card": card_name, "card_path": card_path, "step_dir": step_dir,
            "weight": weight, "weight_sha256": seed_hash, **probe}

def write_recipe_yaml_v4(path, model_card, num_steps, max_num_elements, grad_accum,
                         ckpt_every=250, lr=V4_LR):
    # QA_V4: JOINT_SAMPLER est déclaré après la calibration dans l'ordre logique QA.
    # Heures réelles dans le TSV + beta_language=0.0 => masse exacte 1/6 par langue.
    pub = 50 if ckpt_every % 50 == 0 else ckpt_every
    lr_text = f"{float(lr):.8e}"
    lines = [
        "model:", f'  name: "{model_card}"', "",
        "dataset:", '  name: "afrivoices_dataset"', '  train_split: "train"',
        '  valid_split: "dev"', '  storage_mode: "MIXTURE_PARQUET"',
        '  task_mode: "ASR"', "  mixture_parquet_storage_config:",
        f'    dataset_summary_path: "{V4_TSV_PATH}"', "    beta_corpus: 1.0",
        "    beta_language: 0.0", "    fragment_loading:", "      cache: False",
        "  asr_task_config:", "    max_audio_len: 640000",
        f"    max_num_elements: {max_num_elements}", "    batch_shuffle_window: 1",
        "    normalize_audio: true", "    example_shuffle_window: 1000", "",
        "tokenizer:", f'  name: "{FT_CONFIG["tokenizer_name"]}"', "",
        "optimizer:", "  config:", f"    lr: {lr_text}", "",
        "trainer:", "  freeze_encoder_for_n_steps: 0", "  mixed_precision:",
        f'    dtype: "{_dtype_val}"', "  grad_accumulation:",
        f"    num_batches: {grad_accum}", "",
        "regime:", f"  num_steps: {num_steps}",
        f"  validate_every_n_steps: {ckpt_every}",
        f"  validate_after_n_steps: {ckpt_every}",
        f"  checkpoint_every_n_steps: {ckpt_every}",
        f"  checkpoint_after_n_steps: {ckpt_every}",
        f"  publish_metrics_every_n_steps: {pub}",
        f"  publish_metrics_after_n_steps: {pub}",
    ]
    text = "\n".join(lines) + "\n"
    import yaml
    parsed = yaml.safe_load(text)
    assert parsed["dataset"]["mixture_parquet_storage_config"]["beta_language"] == 0.0
    assert parsed["dataset"]["mixture_parquet_storage_config"]["fragment_loading"]["cache"] is False
    assert parsed["dataset"]["asr_task_config"]["max_num_elements"] == max_num_elements
    assert float(parsed["optimizer"]["config"]["lr"]) == float(lr)
    assert parsed["trainer"]["grad_accumulation"]["num_batches"] == grad_accum
    assert ckpt_every % pub == 0
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as handle: handle.write(text)
    return path

def _gpu_mem_mib():
    output = subprocess.check_output([
        "nvidia-smi", "--query-gpu=memory.used,memory.total",
        "--format=csv,noheader,nounits"], text=True)
    used, total = map(int, output.strip().splitlines()[0].split(","))
    return used, total

def _host_ram_fraction():
    values = {}
    for line in open("/proc/meminfo"):
        key, value = line.split(":", 1)
        values[key] = int(value.strip().split()[0])
    return 1.0 - values["MemAvailable"] / values["MemTotal"]

def _complete_step_dirs(output_dir):
    result = []
    for path in Path(output_dir).rglob("step_*"):
        match = re.fullmatch(r"step_(\d+)", path.name)
        model_files = [p for p in (path/"model").rglob("*") if p.is_file()] \
            if (path/"model").is_dir() else []
        optimizer_files = [p for p in (path/"optimizer").rglob("*") if p.is_file()] \
            if (path/"optimizer").is_dir() else []
        trainer_files = [p for p in (path/"trainer").rglob("*") if p.is_file()] \
            if (path/"trainer").is_dir() else []
        files = model_files + optimizer_files + trainer_files
        if (path.is_dir() and match and not path.name.endswith(".tmp")
                and len(model_files) == 1 and optimizer_files and trainer_files
                and all(p.stat().st_size > 0 for p in files)
                and all(time.time()-p.stat().st_mtime > 10 for p in files)
                and not list(path.rglob("*.tmp*"))):
            result.append((int(match.group(1)), path))
    return sorted(result)

def _archive_candidates_and_prune(output_dir, keep_full=2):
    os.makedirs(V4_CANDIDATES_DIR, exist_ok=True)
    complete = _complete_step_dirs(output_dir)
    contract = json.load(open(V4_RUN_CONTRACT, encoding="utf-8"))
    for step, path in complete:
        if step < 250 or step > V4_MAX_NEW_STEPS or step % V4_CKPT_EVERY:
            continue
        destination = Path(V4_CANDIDATES_DIR) / f"step_{step}"
        if not (destination / "model").is_dir():
            temporary = Path(str(destination) + ".tmp")
            shutil.rmtree(temporary, ignore_errors=True)
            temporary.mkdir(parents=True, exist_ok=True)
            shutil.copytree(path / "model", temporary / "model")
            source_weights = [p for p in (path/"model").rglob("*") if p.is_file()]
            copied_weights = [p for p in (temporary/"model").rglob("*") if p.is_file()]
            assert len(source_weights) == len(copied_weights) == 1
            source_hash = _sha256_file(source_weights[0])
            copied_hash = _sha256_file(copied_weights[0])
            assert source_hash == copied_hash
            manifest = {"step": step, "contract_sha256": contract["contract_sha256"],
                        "weight_sha256": copied_hash, "weight_size": copied_weights[0].stat().st_size,
                        "source_checkpoint": str(path), "archived_at": now_iso()}
            save_json(manifest, str(temporary/"candidate_manifest.json"))
            os.replace(temporary, destination)
            print("Candidat modèle archivé :", destination)
    # Deux checkpoints complets sont conservés pour une reprise sûre.
    for step, path in complete[:-keep_full]:
        sidecar = Path(V4_CANDIDATES_DIR)/f"step_{step}"/"candidate_manifest.json"
        if sidecar.exists(): shutil.rmtree(path, ignore_errors=True)

def run_recipe_v4(output_dir, yaml_path, log_name, wall_limit_h=None,
                  archive_candidates=False):
    os.makedirs(output_dir, exist_ok=True)
    log_path = os.path.join(FT_CONFIG["log_dir"], log_name)
    command = [sys.executable, "-m", "workflows.recipes.wav2vec2.asr", output_dir,
               "--config-file", yaml_path]
    process = subprocess.Popen(command, cwd=OMNI_REPO_DIR, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1,
                               start_new_session=True)
    messages, done = queue.Queue(), threading.Event()
    def reader():
        for line in process.stdout: messages.put(line)
        done.set()
    threading.Thread(target=reader, daemon=True).start()
    start, last_housekeeping, last_gpu_sample = time.time(), 0.0, 0.0
    samples, interrupt_reason, interrupted_at = [], None, None
    try:
        with open(log_path, "a", encoding="utf-8") as log:
            while process.poll() is None or not (done.is_set() and messages.empty()):
                try:
                    line = messages.get(timeout=1)
                    log.write(line); log.flush()
                    if any(k in line.lower() for k in ("loss", "error", "step", "oom", "nan")):
                        print(line.rstrip()[:220])
                except queue.Empty:
                    pass
                now = time.time()
                if now-last_gpu_sample >= 1:
                    try: samples.append(_gpu_mem_mib())
                    except Exception: pass
                    last_gpu_sample = now
                reason = None
                if archive_candidates and now-last_housekeeping > 30:
                    try:
                        _archive_candidates_and_prune(output_dir)
                    except Exception as exc:
                        reason = f"CANDIDATE_ARCHIVE_FAILED:{type(exc).__name__}:{exc}"
                    last_housekeeping = now
                if _host_ram_fraction() > .92: reason = "HOST_RAM_ABOVE_92_PERCENT"
                elif shutil.disk_usage(V4_SCRATCH_ROOT).free < 80 * 2**30:
                    reason = "SCRATCH_FREE_BELOW_80_GIB"
                elif shutil.disk_usage(DRIVE_ROOT).free < 40 * 2**30:
                    reason = "DRIVE_FREE_BELOW_40_GIB"
                elif wall_limit_h and now-start > wall_limit_h*3600:
                    reason = "WALL_LIMIT"
                if reason and not interrupt_reason and process.poll() is None:
                    interrupt_reason, interrupted_at = reason, now
                    print("Arrêt propre demandé :", reason)
                    os.killpg(process.pid, signal.SIGINT)
                if interrupted_at and now-interrupted_at > 900 and process.poll() is None:
                    os.killpg(process.pid, signal.SIGKILL)
            process.wait()
    except BaseException:
        if process.poll() is None:
            try: os.killpg(process.pid, signal.SIGINT); process.wait(timeout=30)
            except Exception:
                try: os.killpg(process.pid, signal.SIGKILL)
                except Exception: pass
                process.wait()
        raise
    finally:
        if process.poll() is None:
            try: os.killpg(process.pid, signal.SIGKILL)
            except Exception: pass
            process.wait()
    if archive_candidates:
        time.sleep(11)
        _archive_candidates_and_prune(output_dir)
    peak = max((u for u, _ in samples), default=None)
    total = max((t for _, t in samples), default=None)
    return {"returncode": process.returncode, "log_path": log_path,
            "peak_nvidia_smi_mib": peak, "total_nvidia_smi_mib": total,
            "peak_fraction": peak/total if peak is not None and total else None,
            "interrupt_reason": interrupt_reason,
            "elapsed_minutes": round((time.time()-start)/60, 2)}

if not RUN_A100_80GB_CALIBRATION_V4:
    raise SystemExit("Calibration V4 bloquée : activer RUN_A100_80GB_CALIBRATION_V4 après 18.4.")
audit = require_audit_pass()
build = require_v4_build(audit)
seed = prepare_v4_seed_card()
write_dataset_card(data_root=V4_DATA_VERSION_ROOT)
for name in ("_pipe", "_pipe_joint", "_pipe_lm", "pipe"):
    if name in globals(): del globals()[name]
gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
assert torch.cuda.memory_allocated() < 1 * 2**30, "Libérer les pipelines CUDA avant calibration"

calibration = {"schema": 2, "created_at": now_iso(), "gpu_fingerprint": gpu_fingerprint,
               "audit_sha256": json.load(open(V4_AUDIT_PATH))["audit_sha256"],
               "plan_sha256": audit["plan_sha256"], "build_sha256": build["build_sha256"],
               "seed": seed, "target_effective_elements": V4_TARGET_EFFECTIVE_ELEMENTS,
               "headroom_rule": "peak nvidia-smi <= 85%", "attempts": [], "chosen": None}
for attempt in V4_A100_LADDER:
    out = os.path.join(V4_SCRATCH_ROOT, f"calibration_{attempt['max_num_elements']}")
    shutil.rmtree(out, ignore_errors=True)
    yaml_path = write_recipe_yaml_v4(
        os.path.join(V4_SCRATCH_ROOT, f"calibration_{attempt['max_num_elements']}.yaml"),
        seed["card"], 40, attempt["max_num_elements"], attempt["grad_accum"],
        ckpt_every=40)
    kernel_before = int(torch.cuda.max_memory_allocated())
    result = run_recipe_v4(out, yaml_path,
                           f"balanced_v4_calib_{attempt['max_num_elements']}.log",
                           wall_limit_h=1.0)
    kernel_after = int(torch.cuda.max_memory_allocated())
    tail = open(result["log_path"], encoding="utf-8", errors="ignore").read()[-20000:].lower()
    bad = any(token in tail for token in ("out of memory", "outofmemory", "nan", "inf loss"))
    accepted = (result["returncode"] == 0 and result["interrupt_reason"] is None and not bad and
                result["peak_fraction"] is not None and result["peak_fraction"] <= .85)
    record = {**attempt, **result, "log_has_oom_or_nonfinite": bad,
              "kernel_max_memory_allocated_before": kernel_before,
              "kernel_max_memory_allocated_after": kernel_after,
              "note": "kernel max_memory_allocated n'observe pas le sous-processus",
              "accepted": accepted}
    calibration["attempts"].append(record)
    shutil.rmtree(out, ignore_errors=True)
    print(record)
    if accepted:
        calibration["chosen"] = attempt; break
save_json(calibration, V4_CALIBRATION_PATH)
assert calibration["chosen"], "Aucun micro-batch sûr : inspecter les logs de calibration"
print("✅ Configuration A100 80 Go choisie :", calibration["chosen"])


In [ ]:
import json

calibration = json.load(
    open(V4_CALIBRATION_PATH, encoding="utf-8")
)

candidate = next(
    attempt
    for attempt in calibration["attempts"]
    if int(attempt["max_num_elements"]) == 4_096_000
    and int(attempt["grad_accum"]) == 10
)

assert candidate["returncode"] == 0
assert candidate["interrupt_reason"] is None
assert not candidate["log_has_oom_or_nonfinite"]
assert candidate["peak_fraction"] < 0.88

candidate["accepted"] = True
candidate["manual_override"] = True
candidate["manual_override_reason"] = (
    "Configuration stable pendant 40 steps. "
    "Pic VRAM accepté manuellement malgré la cible conservatrice de 85 %."
)

calibration["chosen"] = {
    "max_num_elements": 4_096_000,
    "grad_accum": 10,
    "effective_elements": 40_960_000,
}

calibration["headroom_rule"] = (
    "Manual override: measured peak <= 88%; "
    "measured value = "
    f"{candidate['peak_fraction']:.6f}"
)

calibration["manual_override"] = {
    "created_at": now_iso(),
    "measured_peak_fraction": candidate["peak_fraction"],
    "measured_peak_percent": round(
        candidate["peak_fraction"] * 100,
        4,
    ),
    "remaining_vram_mib": (
        candidate["total_nvidia_smi_mib"]
        - candidate["peak_nvidia_smi_mib"]
    ),
    "configuration": calibration["chosen"],
}

save_json(calibration, V4_CALIBRATION_PATH)

print("✅ Configuration choisie manuellement")
print(json.dumps(calibration["chosen"], indent=2))
print(
    "Pic VRAM :",
    round(candidate["peak_fraction"] * 100, 2),
    "%",
)
print(
    "VRAM restante :",
    candidate["total_nvidia_smi_mib"]
    - candidate["peak_nvidia_smi_mib"],
    "MiB",
)

In [ ]:
# 18.6 — QA_V4: UNIFIED_TRAINING — un seul run joint depuis step_5000
if not RUN_BALANCED_FINETUNE_V4:
    raise SystemExit("Training V4 bloqué : activer RUN_BALANCED_FINETUNE_V4 après calibration.")
audit = require_audit_pass()
build = require_v4_build(audit)
assert os.path.exists(V4_CALIBRATION_PATH), "Exécuter 18.5 d'abord"
calibration = json.load(open(V4_CALIBRATION_PATH, encoding="utf-8"))
assert calibration["chosen"], "Calibration sans configuration retenue"
assert calibration["gpu_fingerprint"] == gpu_fingerprint, \
    "GPU/runtime différent de la calibration : refaire 18.5"
assert calibration["plan_sha256"] == audit["plan_sha256"]
assert calibration["build_sha256"] == build["build_sha256"]
seed = prepare_v4_seed_card()
assert calibration["seed"]["weight_sha256"] == seed["weight_sha256"]
chosen = calibration["chosen"]

# Démonstration statique du sampler : heures TSV réelles, poids logiques uniformes.
tsv_frame = pd.read_csv(V4_TSV_PATH, sep="\t")
assert len(tsv_frame) == 6 and set(tsv_frame.language) == \
    {omni_lang(x) for x in LANGUAGES}
raw_language_weights = np.power(tsv_frame.hours.to_numpy()/tsv_frame.hours.sum(), 0.0)
sampler_weights = raw_language_weights/raw_language_weights.sum()
assert np.allclose(sampler_weights, np.repeat(1/6, 6), atol=1e-12)
print("Poids cible des langues (beta_language=0.0) :", sampler_weights.tolist())
plan = json.load(open(V4_PLAN_PATH, encoding="utf-8"))
display(pd.DataFrame(plan["languages"])[[
    "language", "planned_hours", "achieved_new_ratio", "exposure_factor"]])

write_dataset_card(data_root=V4_DATA_VERSION_ROOT)
recipe_path = os.path.join(V4_REPORT_DIR, "balanced_joint_v4_recipe.yaml")
write_recipe_yaml_v4(recipe_path, seed["card"], V4_MAX_NEW_STEPS,
                     int(chosen["max_num_elements"]), int(chosen["grad_accum"]),
                     ckpt_every=V4_CKPT_EVERY, lr=V4_LR)
contract_payload = {
    "schema": 2, "revision": V4_REVISION, "seed_step": 5000,
    "seed_weight_sha256": seed["weight_sha256"],
    "audit_sha256": json.load(open(V4_AUDIT_PATH))["audit_sha256"],
    "manifest_sha256": audit["manifest_sha256"], "plan_sha256": audit["plan_sha256"],
    "build_sha256": build["build_sha256"], "recipe_sha256": _sha256_file(recipe_path),
    "calibration_sha256": _sha256_file(V4_CALIBRATION_PATH),
    "learning_rate": V4_LR, "max_new_optimizer_steps": V4_MAX_NEW_STEPS,
    "checkpoint_every": V4_CKPT_EVERY, "max_num_elements": chosen["max_num_elements"],
    "grad_accum": chosen["grad_accum"], "effective_elements": chosen["effective_elements"],
    "beta_language": 0.0, "language_probability": 1/6,
    "mixed_precision": "bfloat16", "freeze_encoder_for_n_steps": 0,
    "single_unified_checkpoint": True,
}
contract = {"contract": contract_payload, "contract_sha256": _json_digest(contract_payload)}
if os.path.exists(V4_RUN_CONTRACT):
    existing = json.load(open(V4_RUN_CONTRACT, encoding="utf-8"))
    assert existing == contract, \
        "Contrat du run existant différent : nouveau dossier requis, ne pas reprendre ce run"
else:
    assert not os.path.exists(V4_TRAIN_OUTPUT) or not any(Path(V4_TRAIN_OUTPUT).iterdir()), \
        "Output V4 non vide sans contrat : inspection manuelle requise"
    save_json(contract, V4_RUN_CONTRACT)
V4_CANDIDATES_DIR = os.path.join(V4_CANDIDATES_ROOT,
                                 contract["contract_sha256"][:16])
os.makedirs(V4_CANDIDATES_DIR, exist_ok=True)

for name in ("_pipe", "_pipe_joint", "_pipe_lm", "pipe", "TOPUP_PIPES"):
    if name in globals():
        try: globals()[name].clear()
        except Exception: pass
        del globals()[name]
gc.collect(); torch.cuda.empty_cache()
assert torch.cuda.memory_allocated() < 1 * 2**30, "Libérer le GPU avant le sous-processus"
result = run_recipe_v4(
    V4_TRAIN_OUTPUT, recipe_path, "balanced_joint_v4_training.log",
    wall_limit_h=FT_CONFIG["max_wall_hours_per_session"], archive_candidates=True)
status = {"created_at": now_iso(), "contract_sha256": contract["contract_sha256"],
          "result": result,
          "candidate_steps": [step for step, _ in _complete_step_dirs(V4_TRAIN_OUTPUT)] +
              sorted(int(p.name.split("_")[-1]) for p in Path(V4_CANDIDATES_DIR).glob("step_*")
                     if (p/"model").is_dir())}
save_json(status, os.path.join(V4_REPORT_DIR, "training_status_v4.json"))
if result["returncode"] != 0 and not result["interrupt_reason"]:
    raise RuntimeError(f"Training échoué : {result}")
print("Training terminé/interrompu proprement :", result)
print("Relancer la même cellule et le même output reprend les deux derniers checkpoints complets.")


In [ ]:
# 18.6b CORRIGÉ — nouveau run explicitement initialisé depuis step_750
# Optimiseur neuf, LR réduit, 750 steps locaux = total nominal 1500.

import gc
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

EXTENSION_BASE_TOTAL_STEP = 750
EXTENSION_LOCAL_STEPS = 750
EXTENSION_FINAL_TOTAL_STEP = 1500
EXTENSION_LR = 1.5e-6

assert RUN_BALANCED_FINETUNE_V4
assert not RUN_BALANCED_EVAL_V4

# ------------------------------------------------------------------
# 1. Retrouver le candidat step_750 validé par 18.7
# ------------------------------------------------------------------

evaluation_path = os.path.join(
    V4_REPORT_DIR,
    "balanced_checkpoint_eval_v4.json",
)

evaluation = json.load(
    open(evaluation_path, encoding="utf-8")
)

step_750_info = evaluation["candidates"]["step_750"]
step_750_checkpoint = Path(step_750_info["checkpoint"])
step_750_model_dir = step_750_checkpoint / "model"

assert step_750_model_dir.is_dir(), step_750_model_dir

step_750_weights = [
    path
    for path in step_750_model_dir.rglob("*")
    if path.is_file()
]

assert len(step_750_weights) == 1, step_750_weights

step_750_weight = step_750_weights[0]

sidecar_path = (
    step_750_checkpoint
    / "candidate_manifest.json"
)

assert sidecar_path.is_file()

step_750_sidecar = json.load(
    open(sidecar_path, encoding="utf-8")
)

assert (
    step_750_sidecar["weight_sha256"]
    == step_750_info["weight_sha256"]
)
assert (
    step_750_sidecar["weight_size"]
    == step_750_weight.stat().st_size
)

print("Seed acoustique explicite :", step_750_checkpoint)
print("Poids seed :", step_750_weight)
print("SHA256 :", step_750_info["weight_sha256"])

# ------------------------------------------------------------------
# 2. Créer une carte modèle pointant explicitement sur step_750
# ------------------------------------------------------------------

import omnilingual_asr as oa

extension_card_name = (
    "omniASR_CTC_1B_v2_v4_step750_extension_seed"
)

extension_card_path = os.path.join(
    os.path.dirname(oa.__file__),
    "cards",
    "models",
    "afrivoices_v4_step750_extension_seed.yaml",
)

os.makedirs(
    os.path.dirname(extension_card_path),
    exist_ok=True,
)

with open(
    extension_card_path,
    "w",
    encoding="utf-8",
) as card_file:
    card_file.write(
        f"name: {extension_card_name}\n"
        f"base: {FT_CONFIG['model_card']}\n"
        f'checkpoint: "file://{step_750_weight}"\n'
    )

print("Carte extension :", extension_card_path)

# Probe CPU : vérifie que la carte charge réellement step_750.
probe_path = os.path.join(
    V4_SCRATCH_ROOT,
    "probe_extension_step750.py",
)

probe_output = os.path.join(
    V4_REPORT_DIR,
    "probe_extension_step750.json",
)

probe_code = f"""
import json
import torch
from fairseq2.models.hub import load_model

model = load_model(
    "{extension_card_name}",
    device=torch.device("cpu"),
    dtype=torch.float32,
)

params = sum(parameter.numel() for parameter in model.parameters())

assert 950_000_000 <= params < 1_000_000_000, params

json.dump(
    {{
        "params": params,
        "seed_checkpoint": {json.dumps(str(step_750_checkpoint))},
        "seed_weight_sha256": {json.dumps(step_750_info["weight_sha256"])},
    }},
    open({json.dumps(probe_output)}, "w"),
)
"""

with open(
    probe_path,
    "w",
    encoding="utf-8",
) as probe_file:
    probe_file.write(probe_code)

probe_result = subprocess.run(
    [sys.executable, probe_path],
    capture_output=True,
    text=True,
)

assert probe_result.returncode == 0, (
    probe_result.stdout
    + "\n"
    + probe_result.stderr[-4000:]
)

probe = json.load(
    open(probe_output, encoding="utf-8")
)

print("✅ Carte step_750 vérifiée :", probe)

# ------------------------------------------------------------------
# 3. Nouveau dossier de sortie : aucun mélange avec le run précédent
# ------------------------------------------------------------------

extension_root = os.path.join(
    FT_CONFIG["experiment_run"],
    "balanced_joint_v4_extension_from_step750",
)

extension_output = os.path.join(
    extension_root,
    "train_output",
)

os.makedirs(extension_root, exist_ok=True)

extension_recipe = os.path.join(
    V4_REPORT_DIR,
    "balanced_joint_v4_from750_to1500.yaml",
)

calibration = json.load(
    open(V4_CALIBRATION_PATH, encoding="utf-8")
)

chosen = calibration["chosen"]

assert int(chosen["max_num_elements"]) == 4_096_000
assert int(chosen["grad_accum"]) == 10

write_recipe_yaml_v4(
    extension_recipe,
    extension_card_name,
    EXTENSION_LOCAL_STEPS,
    int(chosen["max_num_elements"]),
    int(chosen["grad_accum"]),
    ckpt_every=250,
    lr=EXTENSION_LR,
)

# Le writer utilise normalement "after 250", ce qui avait sauté step_250.
# Ici, on autorise explicitement validation/checkpoint dès le premier multiple.
recipe_text = open(
    extension_recipe,
    encoding="utf-8",
).read()

replacements = {
    "validate_after_n_steps: 250":
        "validate_after_n_steps: 0",
    "checkpoint_after_n_steps: 250":
        "checkpoint_after_n_steps: 0",
    "publish_metrics_after_n_steps: 50":
        "publish_metrics_after_n_steps: 0",
}

for old, new in replacements.items():
    assert old in recipe_text, (
        f"Champ YAML introuvable : {old}"
    )
    recipe_text = recipe_text.replace(old, new)

with open(
    extension_recipe,
    "w",
    encoding="utf-8",
) as recipe_file:
    recipe_file.write(recipe_text)

print("Recette extension :", extension_recipe)

# ------------------------------------------------------------------
# 4. Contrat corrigé
# ------------------------------------------------------------------

initial_contract_path = os.path.join(
    V4_REPORT_DIR,
    "run_contract_v4_initial_750.json",
)

initial_contract = json.load(
    open(initial_contract_path, encoding="utf-8")
)

current_contract = json.load(
    open(V4_RUN_CONTRACT, encoding="utf-8")
)

faulty_contract_backup = os.path.join(
    V4_REPORT_DIR,
    "run_contract_v4_faulty_workspace_resume.json",
)

if (
    current_contract["contract_sha256"]
    != initial_contract["contract_sha256"]
    and not os.path.exists(faulty_contract_backup)
):
    save_json(
        current_contract,
        faulty_contract_backup,
    )

extension_payload = dict(initial_contract["contract"])

extension_payload.update({
    "schema": 4,
    "revision": (
        f"{V4_REVISION}-new-optimizer-"
        "step750-to1500"
    ),
    "recipe_sha256": _sha256_file(
        extension_recipe
    ),
    "calibration_sha256": _sha256_file(
        V4_CALIBRATION_PATH
    ),
    "learning_rate": EXTENSION_LR,
    "max_new_optimizer_steps": (
        EXTENSION_FINAL_TOTAL_STEP
    ),
    "checkpoint_every": 250,
    "extension_mode": (
        "new_optimizer_from_step750_weights"
    ),
    "extension_optimizer_reset": True,
    "extension_local_num_steps": (
        EXTENSION_LOCAL_STEPS
    ),
    "extension_seed_total_step": (
        EXTENSION_BASE_TOTAL_STEP
    ),
    "extension_seed_checkpoint": str(
        step_750_checkpoint
    ),
    "extension_seed_weight_sha256": (
        step_750_info["weight_sha256"]
    ),
    "extension_parent_contract_sha256": (
        initial_contract["contract_sha256"]
    ),
    "extension_total_checkpoint_steps": [
        1000,
        1250,
        1500,
    ],
})

corrected_contract = {
    "contract": extension_payload,
    "contract_sha256": _json_digest(
        extension_payload
    ),
}

save_json(
    corrected_contract,
    V4_RUN_CONTRACT,
)

V4_MAX_NEW_STEPS = EXTENSION_FINAL_TOTAL_STEP
V4_CKPT_EVERY = 250

V4_CANDIDATES_DIR = os.path.join(
    V4_CANDIDATES_ROOT,
    corrected_contract["contract_sha256"][:16],
)

os.makedirs(
    V4_CANDIDATES_DIR,
    exist_ok=True,
)

print(
    "Contrat corrigé :",
    corrected_contract["contract_sha256"],
)
print(
    "Candidats :",
    V4_CANDIDATES_DIR,
)

# Pointeur temporaire cohérent avec le contrat corrigé.
final_pointer_path = os.path.join(
    V4_REPORT_DIR,
    "final_unified_checkpoint_v4.json",
)

if os.path.exists(final_pointer_path):
    temporary_pointer = json.load(
        open(final_pointer_path, encoding="utf-8")
    )

    temporary_pointer.update({
        "selected_at": now_iso(),
        "run_contract_sha256": (
            corrected_contract["contract_sha256"]
        ),
        "selection_rule": (
            "Pointeur temporaire pendant l’extension "
            "step750→step1500 avec optimiseur neuf."
        ),
        "extension_in_progress": True,
    })

    save_json(
        temporary_pointer,
        final_pointer_path,
    )

# ------------------------------------------------------------------
# 5. Fonction d’archivage avec conversion step local → step total
# ------------------------------------------------------------------

def archive_extension_candidate(
    source_checkpoint,
    total_step,
    local_step,
):
    source_checkpoint = Path(source_checkpoint)
    destination = (
        Path(V4_CANDIDATES_DIR)
        / f"step_{total_step}"
    )

    if (destination / "model").is_dir():
        existing_sidecar = json.load(
            open(
                destination / "candidate_manifest.json",
                encoding="utf-8",
            )
        )

        assert (
            existing_sidecar["contract_sha256"]
            == corrected_contract["contract_sha256"]
        )

        return destination

    temporary = Path(str(destination) + ".tmp")
    shutil.rmtree(temporary, ignore_errors=True)
    temporary.mkdir(parents=True, exist_ok=True)

    shutil.copytree(
        source_checkpoint / "model",
        temporary / "model",
    )

    source_weights = [
        path
        for path in (
            source_checkpoint / "model"
        ).rglob("*")
        if path.is_file()
    ]

    copied_weights = [
        path
        for path in (
            temporary / "model"
        ).rglob("*")
        if path.is_file()
    ]

    assert len(source_weights) == 1
    assert len(copied_weights) == 1

    source_hash = _sha256_file(
        source_weights[0]
    )
    copied_hash = _sha256_file(
        copied_weights[0]
    )

    assert source_hash == copied_hash

    candidate_manifest = {
        "step": total_step,
        "extension_local_step": local_step,
        "contract_sha256": (
            corrected_contract["contract_sha256"]
        ),
        "weight_sha256": copied_hash,
        "weight_size": copied_weights[0].stat().st_size,
        "source_checkpoint": str(
            source_checkpoint
        ),
        "archived_at": now_iso(),
        "seed_total_step": (
            EXTENSION_BASE_TOTAL_STEP
        ),
        "optimizer_reset_at_seed": True,
    }

    save_json(
        candidate_manifest,
        str(
            temporary
            / "candidate_manifest.json"
        ),
    )

    os.replace(temporary, destination)

    print(
        f"Candidat total step_{total_step} "
        f"archivé depuis extension step_{local_step}"
    )

    return destination

# Conserver step_750 dans le nouveau contrat pour la comparaison.
archive_extension_candidate(
    step_750_checkpoint,
    total_step=750,
    local_step=0,
)

# ------------------------------------------------------------------
# 6. Lancer le nouveau run
# ------------------------------------------------------------------

for variable_name in (
    "_pipe",
    "_pipe_joint",
    "_pipe_lm",
    "pipe",
    "TOPUP_PIPES",
    "FINAL_PIPELINES",
):
    if variable_name in globals():
        try:
            globals()[variable_name].clear()
        except Exception:
            pass
        del globals()[variable_name]

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

assert torch.cuda.memory_allocated() < 1 * 2**30

print()
print("▶ Extension acoustique depuis les poids step_750")
print("Compteur local : 0 → 750")
print("Total nominal  : 750 → 1500")
print("LR             :", EXTENSION_LR)
print(
    "Micro-batch    :",
    chosen["max_num_elements"],
    "×",
    chosen["grad_accum"],
)

extension_result = run_recipe_v4(
    extension_output,
    extension_recipe,
    "balanced_joint_v4_corrected_from750_to1500.log",
    wall_limit_h=FT_CONFIG["max_wall_hours_per_session"],
    archive_candidates=False,
)

# ------------------------------------------------------------------
# 7. Archiver les checkpoints disponibles
# ------------------------------------------------------------------

time.sleep(11)

complete_extension_steps = dict(
    _complete_step_dirs(extension_output)
)

step_mapping = {
    250: 1000,
    500: 1250,
    750: 1500,
}

for local_step, total_step in step_mapping.items():
    if local_step in complete_extension_steps:
        archive_extension_candidate(
            complete_extension_steps[local_step],
            total_step=total_step,
            local_step=local_step,
        )

archived_steps = sorted(
    int(path.name.split("_")[-1])
    for path in Path(
        V4_CANDIDATES_DIR
    ).glob("step_*")
    if path.is_dir()
    and (path / "model").is_dir()
)

status = {
    "created_at": now_iso(),
    "contract_sha256": (
        corrected_contract["contract_sha256"]
    ),
    "seed_total_step": 750,
    "local_target_steps": 750,
    "total_target_step": 1500,
    "result": extension_result,
    "complete_local_steps": sorted(
        complete_extension_steps
    ),
    "archived_total_steps": archived_steps,
}

save_json(
    status,
    os.path.join(
        V4_REPORT_DIR,
        "training_status_v4_corrected_extension_to1500.json",
    ),
)

if (
    extension_result["returncode"] != 0
    and not extension_result["interrupt_reason"]
):
    raise RuntimeError(
        f"Extension échouée : {extension_result}"
    )

if extension_result["returncode"] == 0:
    expected = {750, 1000, 1250, 1500}
    assert expected <= set(archived_steps), (
        f"Candidats manquants : "
        f"{expected - set(archived_steps)}"
    )

print()
print(
    "Extension terminée/interrompue proprement :",
    extension_result,
)
print(
    "Candidats totaux disponibles :",
    archived_steps,
)

if 1500 in archived_steps:
    print(
        "✅ Total nominal step_1500 atteint. "
        "Relancer ensuite 18.7."
    )
else:
    print(
        "Relancer cette même cellule pour reprendre "
        "le dernier checkpoint local complet."
    )

In [ ]:
# 18.7 — Sélection du meilleur checkpoint sur un dev mixte réellement audité
if not RUN_BALANCED_EVAL_V4:
    raise SystemExit("Évaluation V4 bloquée : activer RUN_BALANCED_EVAL_V4 après le training.")
audit = require_audit_pass()
build = require_v4_build(audit)
seed = prepare_v4_seed_card()
import io, jiwer
import pyarrow.parquet as pq

assert os.path.exists(V4_RUN_CONTRACT), "Contrat training V4 absent"
run_contract = json.load(open(V4_RUN_CONTRACT, encoding="utf-8"))
assert _json_digest(run_contract["contract"]) == run_contract["contract_sha256"], \
    "Contrat V4 modifié"
current_audit_sha = json.load(open(V4_AUDIT_PATH, encoding="utf-8"))["audit_sha256"]
contract_payload = run_contract["contract"]
assert contract_payload["seed_weight_sha256"] == seed["weight_sha256"]
assert contract_payload["audit_sha256"] == current_audit_sha
assert contract_payload["plan_sha256"] == audit["plan_sha256"]
assert contract_payload["build_sha256"] == build["build_sha256"]
assert contract_payload["calibration_sha256"] == _sha256_file(V4_CALIBRATION_PATH)
V4_CANDIDATES_DIR = os.path.join(V4_CANDIDATES_ROOT,
                                 run_contract["contract_sha256"][:16])

candidates = [{"name": "joint_step_5000", "kind": "baseline", "step": 0,
               "checkpoint": seed["step_dir"],
               "weight_sha256": seed["weight_sha256"]}]
for path in sorted(Path(V4_CANDIDATES_DIR).glob("step_*"),
                   key=lambda p: int(p.name.split("_")[-1])):
    step = int(path.name.split("_")[-1])
    if (path/"model").is_dir():
        sidecar_path = path/"candidate_manifest.json"
        assert sidecar_path.exists(), f"Sidecar candidat absent : {path}"
        sidecar = json.load(open(sidecar_path, encoding="utf-8"))
        assert sidecar["contract_sha256"] == run_contract["contract_sha256"]
        weight_files = [p for p in (path/"model").rglob("*") if p.is_file()]
        assert len(weight_files) == 1 and _sha256_file(weight_files[0]) == \
            sidecar["weight_sha256"]
        candidates.append({"name": path.name, "kind": "v4", "step": step,
                           "checkpoint": str(path),
                           "weight_sha256": sidecar["weight_sha256"]})
assert len(candidates) > 1, "Aucun candidat V4 archivé (minimum attendu : step_250)"

# 120 clips maximum par langue × rôle. Ordre hashé, aucune donnée local_test/Kaggle.
dev = pd.read_parquet(V4_DEV_SELECTION_PATH)
dev = dev[dev.split.eq("dev") & dev.usable & dev.seconds.le(38.0)].copy()
eval_groups = []
for language in LANGUAGES:
    for role in ("legacy_replay", "new_in_domain"):
        pool = dev[(dev.language.eq(language)) & dev.adaptation_role.eq(role)].copy()
        if len(pool):
            pool["_rank"] = pool.sample_key.map(lambda key: hashlib.sha256(
                f"{V4_SEED}:eval:{key}".encode()).hexdigest())
            eval_groups.append(pool.sort_values("_rank").head(120).drop(columns="_rank"))
eval_frame = pd.concat(eval_groups, ignore_index=True)
assert set(eval_frame.language) == set(LANGUAGES)
assert eval_frame.split.eq("dev").all()
eval_sample_sha256 = _frame_sha256(eval_frame)

_eval_cache_path, _eval_cache_rows = None, None
def _eval_input(row):
    global _eval_cache_path, _eval_cache_rows
    if row["source_kind"] == "file": return row["audio_ref"]
    if row["parquet_path"] != _eval_cache_path:
        _eval_cache_rows = pq.read_table(row["parquet_path"],
                                         columns=["audio_bytes", "audio_size"]).to_pylist()
        _eval_cache_path = row["parquet_path"]
    source = _eval_cache_rows[int(row["row_index"])]
    waveform, sample_rate = sf.read(io.BytesIO(source["audio_bytes"]), dtype="float32")
    if getattr(waveform, "ndim", 1) > 1: waveform = waveform.mean(axis=1)
    assert sample_rate == 16000
    return {"waveform": waveform, "sample_rate": 16000}

def _transcribe_group(pipe, frame, language):
    outputs = []
    records = frame.sort_values(["source_kind", "parquet_path", "row_index"]).to_dict("records")
    for start in range(0, len(records), 16):
        batch = records[start:start+16]
        inputs = [_eval_input(row) for row in batch]
        text = pipe.transcribe(inputs, lang=[omni_lang(language)]*len(inputs), batch_size=8)
        outputs.extend(normalize_text(value, language) for value in text)
    refs = [normalize_text(row["text"], language) for row in records]
    return refs, outputs

all_results, detail_rows = {}, []
for candidate in candidates:
    pipe = load_finetuned_pipeline(candidate["checkpoint"])
    language_scores = {}
    role_outputs = {}
    for language in LANGUAGES:
        role_scores = {}
        for role in ("legacy_replay", "new_in_domain"):
            group = eval_frame[(eval_frame.language.eq(language)) &
                               eval_frame.adaptation_role.eq(role)]
            if not len(group): continue
            refs, hyps = _transcribe_group(pipe, group, language)
            score = float(jiwer.wer(refs, hyps))
            words = sum(len(value.split()) for value in refs)
            role_outputs[(language, role)] = score
            role_scores[role] = score
            detail_rows.append({"candidate": candidate["name"], "language": language,
                                "role": role, "wer": score, "clips": len(refs),
                                "reference_words": words})
        assert set(role_scores) == {"legacy_replay", "new_in_domain"}, \
            f"Les deux rôles dev sont requis pour {language}: {role_scores}"
        target_new = V4_TARGET_NEW_IN_DOMAIN[language]
        language_scores[language] = float(
            target_new*role_scores["new_in_domain"] +
            (1-target_new)*role_scores["legacy_replay"])
    macro = float(np.mean([language_scores[x] for x in LANGUAGES]))
    all_results[candidate["name"]] = {**candidate, "macro_wer": macro,
                                       "language_wer": language_scores,
                                       "role_wer": {f"{k[0]}:{k[1]}": v
                                                    for k, v in role_outputs.items()}}
    print(candidate["name"], "macro-WER=", round(macro, 5), language_scores)
    del pipe; gc.collect(); torch.cuda.empty_cache()

baseline = all_results["joint_step_5000"]
eligible = []
for name, result in all_results.items():
    if result["kind"] != "v4": continue
    regressions = {}
    for language in LANGUAGES:
        key = f"{language}:legacy_replay"
        if key in result["role_wer"] and key in baseline["role_wer"]:
            delta = result["role_wer"][key] - baseline["role_wer"][key]
            if delta > .015: regressions[language] = delta
    result["legacy_regressions_over_1_5pt"] = regressions
    if not regressions: eligible.append(result)
# La baseline reste éligible : ne jamais remplacer step_5000 par un modèle globalement pire.
best = min([baseline] + eligible, key=lambda value: value["macro_wer"])
FINAL_UNIFIED_CHECKPOINT = best["checkpoint"]
final_pointer = {
    "schema": 2, "selected_at": now_iso(), "checkpoint": FINAL_UNIFIED_CHECKPOINT,
    "step": best["step"], "macro_wer": best["macro_wer"],
    "baseline_macro_wer": baseline["macro_wer"], "single_acoustic_model": True,
    "weight_sha256": best["weight_sha256"],
    "run_contract_sha256": run_contract["contract_sha256"],
    "seed_weight_sha256": seed["weight_sha256"],
    "calibration_sha256": _sha256_file(V4_CALIBRATION_PATH),
    "eval_sample_sha256": eval_sample_sha256,
    "audit_sha256": json.load(open(V4_AUDIT_PATH))["audit_sha256"],
    "plan_sha256": audit["plan_sha256"], "build_sha256": build["build_sha256"],
    "selection_rule": "minimum macro-WER 6 langues; aucune régression legacy > 0.015/langue",
}
report = {"created_at": now_iso(), "candidates": all_results,
          "detail": detail_rows, "chosen": final_pointer}
save_json(report, os.path.join(V4_REPORT_DIR, "balanced_checkpoint_eval_v4.json"))
save_json(final_pointer, os.path.join(V4_REPORT_DIR, "final_unified_checkpoint_v4.json"))
PRIMARY_ACOUSTIC_CKPT = FINAL_UNIFIED_CHECKPOINT
FINAL_ACOUSTIC_CKPT = FINAL_UNIFIED_CHECKPOINT
for name in ("_pipe", "_pipe_joint", "_pipe_lm", "FINAL_PIPELINES"):
    if name in globals():
        try: globals()[name].clear()
        except Exception: pass
        del globals()[name]
print("✅ FINAL_UNIFIED_CHECKPOINT =", FINAL_UNIFIED_CHECKPOINT)
print("Le dernier step n'est pas favorisé. Rejouer 12.2d/16.7b puis 17.1–17.2 pour")
print("reconstruire le décodage KenLM et les caches avec cet unique checkpoint.")


In [ ]:
import json
import os
import pandas as pd

REPORT_PATH = os.path.join(
    FT_CONFIG["report_dir"],
    "balanced_v4",
    "balanced_checkpoint_eval_v4.json",
)

report = json.load(open(REPORT_PATH, encoding="utf-8"))

summary_rows = []

for candidate_name, candidate in report["candidates"].items():
    summary_rows.append({
        "candidate": candidate_name,
        "step": candidate["step"],
        "macro_wer": candidate["macro_wer"],
        "legacy_regressions": candidate.get(
            "legacy_regressions_over_1_5pt",
            {},
        ),
        **{
            f"wer_{language}": candidate["language_wer"][language]
            for language in LANGUAGES
        },
    })

summary = pd.DataFrame(summary_rows).sort_values("macro_wer")

print("=== COMPARAISON GLOBALE ===")
display(summary)

detail = pd.DataFrame(report["detail"])

print("=== WER PAR LANGUE ET PAR RÔLE ===")
display(
    detail.pivot_table(
        index=["language", "role"],
        columns="candidate",
        values="wer",
    )
)

print("=== CHECKPOINT RETENU ===")
print(json.dumps(report["chosen"], indent=2, ensure_ascii=False))

In [ ]:
# 18.7b — Retenir step_1250 pour l’objectif macro-WER Kaggle
import os
import gc
import json
import hashlib
from pathlib import Path

_TARGET_NAME = "step_1250"
_TARGET_STEP = 1250

_eval_path = os.path.join(
    V4_REPORT_DIR,
    "balanced_checkpoint_eval_v4.json",
)
_pointer_path = os.path.join(
    V4_REPORT_DIR,
    "final_unified_checkpoint_v4.json",
)
_contract_path = os.path.join(
    V4_REPORT_DIR,
    "run_contract_v4.json",
)

assert os.path.exists(_eval_path), _eval_path
assert os.path.exists(_pointer_path), _pointer_path
assert os.path.exists(_contract_path), _contract_path

_eval_report = json.load(open(_eval_path, encoding="utf-8"))
assert _TARGET_NAME in _eval_report["candidates"], (
    f"{_TARGET_NAME} absent du rapport"
)

_candidate = _eval_report["candidates"][_TARGET_NAME]
_target_checkpoint = os.path.realpath(_candidate["checkpoint"])
_sidecar_path = os.path.join(
    _target_checkpoint,
    "candidate_manifest.json",
)

assert os.path.isdir(_target_checkpoint), _target_checkpoint
assert os.path.exists(_sidecar_path), _sidecar_path

_sidecar = json.load(open(_sidecar_path, encoding="utf-8"))
_run_contract = json.load(open(_contract_path, encoding="utf-8"))

assert _candidate["step"] == _TARGET_STEP
assert _sidecar["contract_sha256"] == _run_contract["contract_sha256"]

_weight_files = [
    path for path in Path(_target_checkpoint, "model").rglob("*")
    if path.is_file()
]
assert len(_weight_files) == 1, _weight_files

def _full_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(8 << 20), b""):
            digest.update(block)
    return digest.hexdigest()

_weight_sha256 = _full_sha256(_weight_files[0])
assert _weight_sha256 == _sidecar["weight_sha256"]
assert _weight_sha256 == _candidate["weight_sha256"]

_pointer = json.load(open(_pointer_path, encoding="utf-8"))
_pointer.update({
    "schema": 3,
    "selected_at": now_iso(),
    "checkpoint": _target_checkpoint,
    "step": _TARGET_STEP,
    "macro_wer": float(_candidate["macro_wer"]),
    "weight_sha256": _weight_sha256,
    "run_contract_sha256": _run_contract["contract_sha256"],
    "single_acoustic_model": True,
    "selection_rule": (
        "minimum macro-WER équilibré sur 6 langues ; "
        "régression legacy signalée mais non bloquante pour l’objectif Kaggle"
    ),
    "manual_competition_override": {
        "enabled": True,
        "previous_automatic_checkpoint": _eval_report["chosen"]["checkpoint"],
        "reason": (
            "step_1250 obtient le meilleur macro-WER audité : "
            f"{float(_candidate['macro_wer']):.6f}"
        ),
    },
})

save_json(_pointer, _pointer_path)

FINAL_UNIFIED_CHECKPOINT = _target_checkpoint
PRIMARY_ACOUSTIC_CKPT = _target_checkpoint
FINAL_ACOUSTIC_CKPT = _target_checkpoint

# Supprimer tout modèle éventuellement encore chargé.
for _name in (
    "_pipe",
    "_pipe_joint",
    "_pipe_lm",
    "_PIPE_CACHE",
    "FINAL_PIPELINES",
):
    if _name in globals():
        try:
            globals()[_name].clear()
        except Exception:
            pass
        del globals()[_name]

gc.collect()
torch.cuda.empty_cache()

print("✅ Checkpoint Kaggle verrouillé")
print("Checkpoint :", FINAL_UNIFIED_CHECKPOINT)
print("Step       :", _pointer["step"])
print("Macro-WER  :", _pointer["macro_wer"])
print("SHA256     :", _pointer["weight_sha256"])

In [ ]:
# 18.8a — Préflight du réglage KenLM V4
import os
import glob
import json
import pandas as pd

assert PRIMARY_ACOUSTIC_CKPT.endswith("/step_1250"), PRIMARY_ACOUSTIC_CKPT
assert os.path.exists(V4_DEV_SELECTION_PATH), V4_DEV_SELECTION_PATH

_dev_lm = pd.read_parquet(V4_DEV_SELECTION_PATH).copy()

def _decoding_domain(row):
    language = str(row["language"])
    method = str(row.get("speech_method", "")).strip().lower()
    role = str(row.get("adaptation_role", "")).strip().lower()

    # Le swahili de cette compétition est essentiellement spontané.
    if language == "sw":
        return "unscripted"

    if "unscript" in method or "spont" in method or method in {"false", "0"}:
        return "unscripted"

    if "script" in method or "read" in method or method in {"true", "1"}:
        return "scripted"

    return "unscripted" if role == "new_in_domain" else "scripted"

_dev_lm["decode_domain"] = _dev_lm.apply(_decoding_domain, axis=1)

_group_report = (
    _dev_lm.groupby(
        ["language", "decode_domain", "adaptation_role"],
        dropna=False,
    )
    .agg(
        clips=("sample_key", "size"),
        hours=("seconds", lambda values: float(values.sum()) / 3600),
    )
    .reset_index()
)

print("=== DONNÉES DISPONIBLES POUR LE RÉGLAGE ===")
display(_group_report)

_domain_path = os.path.join(
    FT_CONFIG["report_dir"],
    "kenlm_tuning_by_domain_v3.json",
)
_saved_domain = (
    json.load(open(_domain_path, encoding="utf-8"))
    if os.path.exists(_domain_path)
    else {}
)

_inventory = []

def _add_candidate(language, domain, tag, path, unigram_file=None):
    if not path:
        return

    path = os.path.realpath(path)
    key = (language, domain, path)

    if any(
        (row["language"], row["domain"], row["path"]) == key
        for row in _inventory
    ):
        return

    _inventory.append({
        "language": language,
        "domain": domain,
        "candidate": tag,
        "path": path,
        "exists": os.path.exists(path),
        "size_mib": (
            round(os.path.getsize(path) / 2**20, 2)
            if os.path.exists(path)
            else None
        ),
        "unigram_file": unigram_file,
        "unigrams_exist": (
            os.path.exists(unigram_file)
            if unigram_file
            else None
        ),
    })

for _language in LANGUAGES:
    for _domain in ("scripted", "unscripted"):
        # LM actuellement configuré.
        _base = dict(LM_BEST[_language])
        _add_candidate(
            _language,
            _domain,
            f"current_{_base.get('lm', 'base')}",
            _base.get("lm_bin") or LM_PATHS[_language],
            _base.get("uni_file"),
        )

        # Configuration historique spécifique au domaine.
        _domain_config = _saved_domain.get(
            f"{_language}|{_domain}",
            {},
        )
        if _domain_config:
            _add_candidate(
                _language,
                _domain,
                f"domain_{_domain_config.get('lm', 'saved')}",
                _domain_config.get("lm_bin"),
                _domain_config.get("uni_file"),
            )

        # KenLM train seul.
        _add_candidate(
            _language,
            _domain,
            "v1_train",
            os.path.join(
                FT_CONFIG["experiment_run"],
                "kenlm_models",
                f"{_language}.binary",
            ),
        )

        # KenLM train + valid.
        _add_candidate(
            _language,
            _domain,
            "v3_train_valid",
            os.path.join(
                FT_CONFIG["experiment_run"],
                "kenlm_models_v3",
                f"{_language}.binary",
            ),
        )

        # LM V6 enrichi avec les textes spontanés.
        if _domain == "unscripted":
            for _binary in sorted(glob.glob(os.path.join(
                FT_CONFIG["experiment_run"],
                "kenlm_models_v6_textfix_longtext",
                f"{_language}_*.binary",
            ))):
                _stem = _binary[:-len(".binary")]
                _add_candidate(
                    _language,
                    _domain,
                    "v6_spont_longtext",
                    _binary,
                    _stem + "_unigrams.txt",
                )

        # LM Wikipedia swahili.
        if _language == "sw" and _domain == "unscripted":
            _add_candidate(
                _language,
                _domain,
                "web_sw",
                os.path.join(
                    FT_CONFIG["experiment_run"],
                    "kenlm_models_web",
                    "sw.binary",
                ),
                os.path.join(
                    FT_CONFIG["experiment_run"],
                    "kenlm_models_web",
                    "sw_unigrams.txt",
                ),
            )

_inventory_frame = pd.DataFrame(_inventory)

print("\n=== INVENTAIRE DES CANDIDATS KENLM ===")
display(
    _inventory_frame.sort_values(
        ["language", "domain", "candidate"],
    ).reset_index(drop=True)
)

_missing_required = _inventory_frame[
    _inventory_frame["candidate"].str.startswith("current_")
    & ~_inventory_frame["exists"]
]

assert _missing_required.empty, (
    "Des LM actuellement configurés sont absents :\n"
    + _missing_required.to_string(index=False)
)

_available_counts = (
    _inventory_frame[_inventory_frame.exists]
    .groupby(["language", "domain"])
    .size()
    .rename("available_lms")
    .reset_index()
)

print("\n=== NOMBRE DE LM DISPONIBLES ===")
display(_available_counts)

print("✅ Préflight KenLM terminé")
print("Checkpoint :", PRIMARY_ACOUSTIC_CKPT)
print("Groupes de tuning :", len(_group_report))
print("Candidats LM existants :", int(_inventory_frame.exists.sum()))

In [ ]:
# Correctif avant 18.8b — initialiser le routage KenLM
import os

_domain_path = os.path.join(
    FT_CONFIG["report_dir"],
    "kenlm_tuning_by_domain_v3.json",
)

if not os.path.exists(_domain_path):
    _initial_domain_configs = {}

    for _language in LANGUAGES:
        for _decode_domain in ("scripted", "unscripted"):
            _config = dict(LM_BEST[_language])
            _config["lm_bin"] = (
                _config.get("lm_bin")
                or LM_PATHS[_language]
            )
            _config["needs_domain_tuning"] = False

            _initial_domain_configs[
                f"{_language}|{_decode_domain}"
            ] = _config

    save_json(_initial_domain_configs, _domain_path)

print("✅ Routage KenLM initialisé :", _domain_path)
print("Nombre de configurations :", len(
    json.load(open(_domain_path, encoding="utf-8"))
))

In [ ]:
# 18.8b — Réglage KenLM V4 reprenable pour step_1250
import gc
import glob
import hashlib
import io
import json
import os
import re
import shutil
import time
from collections import Counter

import jiwer
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import torch
from pyctcdecode import build_ctcdecoder

assert PRIMARY_ACOUSTIC_CKPT.endswith("/step_1250"), PRIMARY_ACOUSTIC_CKPT
assert "_dev_lm" in globals() and "_inventory_frame" in globals(), \
    "Exécuter 18.8a avant 18.8b"
assert "capture_logprobs" in globals() and "LABELS" in globals(), \
    "Exécuter 12.2d avant 18.8b"

# Les langues faibles passent en premier. Une relance saute les groupes déjà terminés.
LM_V4_LANGUAGES = ["som", "kln", "mas", "luo", "kik", "sw"]
LM_V4_MAX_GROUP_CLIPS = 72
LM_V4_MAX_HOLDOUT_CLIPS = 24
LM_V4_COARSE_ALPHAS = [0.0, 0.4, 0.8, 1.2]
LM_V4_COARSE_BETAS = [0.0, 1.0, 2.0]
LM_V4_BEAMS = [100, 250]
LM_V4_HOLDOUT_GUARD = 0.003

_domain_path = os.path.join(
    FT_CONFIG["report_dir"], "kenlm_tuning_by_domain_v3.json"
)
_domain_backup_path = os.path.join(
    FT_CONFIG["report_dir"], "kenlm_tuning_by_domain_v3_pre_step1250.json"
)
_report_path = os.path.join(
    V4_REPORT_DIR, "kenlm_tuning_v4_step1250_leakfree.json"
)
_unigram_dir = os.path.join(
    FT_CONFIG["experiment_run"], "kenlm_models_v4_step1250_unigrams"
)
os.makedirs(_unigram_dir, exist_ok=True)

# Une nouvelle exécution V4 peut ne pas encore avoir de fichier par domaine.
# Dans ce cas, partir proprement du meilleur LM global de chaque langue.
if not os.path.exists(_domain_path):
    _initial_domain_configs = {}
    for _language in LANGUAGES:
        for _decode_domain in ("scripted", "unscripted"):
            _config = dict(LM_BEST[_language])
            _config["lm_bin"] = _config.get("lm_bin") or LM_PATHS[_language]
            _config["needs_domain_tuning"] = False
            _initial_domain_configs[f"{_language}|{_decode_domain}"] = _config
    save_json(_initial_domain_configs, _domain_path)
    print("Fichier de routage KenLM initialisé :", _domain_path)

if not os.path.exists(_domain_backup_path):
    shutil.copy2(_domain_path, _domain_backup_path)

_baseline_domain = json.load(open(_domain_backup_path, encoding="utf-8"))
_pointer = json.load(open(
    os.path.join(V4_REPORT_DIR, "final_unified_checkpoint_v4.json"),
    encoding="utf-8",
))
assert _pointer["step"] == 1250


def _manifest_domain(row):
    value = row.get("is_scripted")
    if pd.notna(value):
        text = str(value).strip().lower()
        if text in {"true", "1"}:
            return "scripted"
        if text in {"false", "0"}:
            return "unscripted"
    text = str(row.get("type", "")).strip().lower()
    if "unscript" in text or "spont" in text:
        return "unscripted"
    if "script" in text or "read" in text:
        return "scripted"
    return "unknown"


# Protocole sans fuite :
# - scripted : local_test, exclu de tous les corpus LM train/train+valid ;
# - unscripted : uniquement new_in_domain/dev, dont les locuteurs et textes sont exclus
#   des corpus spontanés servant à construire les LM V4/V5/V6.
_local_test = manifest[manifest["split"].eq("local_test")].copy()
_local_test["decode_domain"] = _local_test.apply(_manifest_domain, axis=1)
_local_test = _local_test[_local_test["decode_domain"].eq("scripted")].copy()
_scripted_eval = pd.DataFrame({
    "sample_key": "local_test:" + _local_test["sample_id"].astype(str),
    "language": _local_test["language"].astype(str),
    "decode_domain": "scripted",
    "adaptation_role": "legacy_replay",
    "source_kind": "file",
    "audio_ref": _local_test["curated_audio_path"].astype(str),
    "parquet_path": "",
    "row_index": -1,
    "text": _local_test["text_norm"].astype(str),
    "seconds": pd.to_numeric(_local_test["duration"], errors="coerce"),
})
_scripted_eval = _scripted_eval[
    _scripted_eval["audio_ref"].map(os.path.exists)
    & _scripted_eval["text"].str.strip().ne("")
    & _scripted_eval["seconds"].gt(0)
].copy()

_unscripted_eval = _dev_lm[
    _dev_lm["decode_domain"].eq("unscripted")
    & _dev_lm["adaptation_role"].eq("new_in_domain")
].copy()

_lm_eval_frame = pd.concat(
    [_scripted_eval, _unscripted_eval], ignore_index=True, sort=False
)
assert set(_scripted_eval["language"]) == set(LANGUAGES) - {"sw"}
assert set(_unscripted_eval["language"]) == set(LANGUAGES)
print("=== PROTOCOLE KENLM SANS FUITE ===")
display(_lm_eval_frame.groupby(["language", "decode_domain"]).agg(
    clips=("sample_key", "size"),
    hours=("seconds", lambda values: float(values.sum()) / 3600),
).reset_index())


def _json_sha(value):
    raw = json.dumps(
        value, sort_keys=True, ensure_ascii=False, separators=(",", ":")
    ).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()


_selection_keys = sorted(_lm_eval_frame["sample_key"].astype(str).tolist())
_tuning_signature = _json_sha({
    "schema": 2,
    "checkpoint_sha256": _pointer["weight_sha256"],
    "selection_sha256": _json_sha(_selection_keys),
    "scripted_source": "manifest/local_test_only",
    "unscripted_source": "v4_dev/new_in_domain_only",
    "max_group_clips": LM_V4_MAX_GROUP_CLIPS,
    "max_holdout_clips": LM_V4_MAX_HOLDOUT_CLIPS,
    "coarse_alphas": LM_V4_COARSE_ALPHAS,
    "coarse_betas": LM_V4_COARSE_BETAS,
    "beams": LM_V4_BEAMS,
    "holdout_guard": LM_V4_HOLDOUT_GUARD,
})

if os.path.exists(_report_path):
    _report = json.load(open(_report_path, encoding="utf-8"))
    assert _report["tuning_signature"] == _tuning_signature, \
        "Le contrat du tuning existant diffère : changer le nom du rapport."
    _domain_configs = json.load(open(_domain_path, encoding="utf-8"))
else:
    # Effacer tout réglage éventuellement produit par l'ancienne cellule contaminée.
    _domain_configs = {key: dict(value) for key, value in _baseline_domain.items()}
    save_json(_domain_configs, _domain_path)
    _report = {
        "schema": 2,
        "created_at": now_iso(),
        "checkpoint": PRIMARY_ACOUSTIC_CKPT,
        "checkpoint_sha256": _pointer["weight_sha256"],
        "tuning_signature": _tuning_signature,
        "leak_free_protocol": {
            "scripted": "local_test_only",
            "unscripted": "new_in_domain_dev_only",
            "invalidated_report": os.path.join(
                V4_REPORT_DIR, "kenlm_tuning_v4_step1250.json"
            ),
        },
        "groups": {},
    }


def _norm(value, language):
    try:
        return normalize_text(value, language)
    except TypeError:
        return normalize_text(value)


def _safe_tag(value):
    return re.sub(r"[^a-zA-Z0-9_.-]+", "_", str(value)).strip("_")


def _write_unigrams(language, tag, splits):
    path = os.path.join(
        _unigram_dir, f"{language}_{_safe_tag(tag)}_unigrams.txt"
    )
    if os.path.exists(path):
        return path
    rows = manifest[
        manifest["language"].eq(language)
        & manifest["split"].isin(list(splits))
    ]
    counts = Counter()
    for text in rows["text_norm"].astype(str):
        counts.update(text.split())
    assert counts, (language, tag, splits)
    temporary = path + f".tmp-{os.getpid()}"
    with open(temporary, "w", encoding="utf-8") as handle:
        handle.write("\n".join(word for word, _ in counts.most_common(200_000)))
    os.replace(temporary, path)
    return path


def _candidate_configs(language, domain):
    rows = _inventory_frame[
        _inventory_frame["language"].eq(language)
        & _inventory_frame["domain"].eq(domain)
        & _inventory_frame["exists"].eq(True)
    ].copy()
    output, seen = [], set()
    for row in rows.to_dict("records"):
        path = os.path.realpath(row["path"])
        if path in seen:
            continue
        seen.add(path)
        tag = str(row["candidate"])
        unigram_file = row.get("unigram_file")
        if pd.isna(unigram_file):
            unigram_file = None
        if not unigram_file or not os.path.exists(unigram_file):
            splits = ("train",) if tag == "v1_train" else ("train", "valid")
            unigram_file = _write_unigrams(language, tag, splits)
        output.append({
            "tag": tag,
            "path": path,
            "uni_file": os.path.realpath(unigram_file),
        })
    assert output, (language, domain)
    return output


_parquet_cache_path = None
_parquet_cache_rows = None


def _audio_input(row):
    global _parquet_cache_path, _parquet_cache_rows
    if row["source_kind"] == "file":
        return row["audio_ref"]
    path = row["parquet_path"]
    if path != _parquet_cache_path:
        _parquet_cache_rows = pq.read_table(
            path, columns=["audio_bytes", "audio_size"]
        ).to_pylist()
        _parquet_cache_path = path
    source = _parquet_cache_rows[int(row["row_index"])]
    waveform, sample_rate = sf.read(
        io.BytesIO(source["audio_bytes"]), dtype="float32", always_2d=False
    )
    if getattr(waveform, "ndim", 1) > 1:
        waveform = waveform.mean(axis=1)
    assert sample_rate == 16000, sample_rate
    return {"waveform": waveform, "sample_rate": 16000}


def _split_group(frame, language, domain):
    frame = frame.copy()
    frame["_rank"] = frame["sample_key"].map(lambda key: hashlib.sha256(
        f"{V4_SEED}:kenlm-v4:{language}:{domain}:{key}".encode()
    ).hexdigest())
    frame = frame.sort_values("_rank").head(LM_V4_MAX_GROUP_CLIPS)
    assert len(frame) >= 36, f"Groupe trop petit : {language}/{domain} = {len(frame)}"
    holdout_n = min(LM_V4_MAX_HOLDOUT_CLIPS, max(12, len(frame) // 3))
    tune = frame.iloc[:-holdout_n].drop(columns="_rank")
    holdout = frame.iloc[-holdout_n:].drop(columns="_rank")
    assert set(tune.sample_key).isdisjoint(set(holdout.sample_key))
    return tune, holdout


def _decode_score(decoder, alpha, beta, beam, logits, references, language):
    assert hasattr(decoder, "reset_params"), \
        "Version pyctcdecode incompatible : reset_params absent"
    decoder.reset_params(alpha=float(alpha), beta=float(beta))
    hypotheses = [
        _norm(decoder.decode(value, beam_width=int(beam)), language)
        for value in logits
    ]
    return float(jiwer.wer(references, hypotheses))


def _make_decoder(candidate, alpha=0.0, beta=0.0):
    unigrams = open(candidate["uni_file"], encoding="utf-8").read().splitlines()
    return build_ctcdecoder(
        LABELS,
        candidate["path"],
        unigrams=unigrams,
        alpha=float(alpha),
        beta=float(beta),
    )


def _candidate_for_path(candidates, path):
    path = os.path.realpath(path)
    return next((item for item in candidates if item["path"] == path), None)


for _language in LM_V4_LANGUAGES:
    _domains = ["unscripted"] if _language == "sw" else ["scripted", "unscripted"]
    for _decode_domain in _domains:
        _group_key = f"{_language}|{_decode_domain}"

        completed = _report["groups"].get(_group_key)
        if completed and completed.get("tuning_signature") == _tuning_signature:
            _domain_configs[_group_key] = completed["deployed_config"]
            print("[déjà terminé]", _group_key,
                  "holdout=", completed["deployed_config"].get("wer_holdout"))
            continue

        _group = _lm_eval_frame[
            _lm_eval_frame["language"].eq(_language)
            & _lm_eval_frame["decode_domain"].eq(_decode_domain)
        ].copy()
        _tune_frame, _holdout_frame = _split_group(
            _group, _language, _decode_domain
        )
        _combined = pd.concat([_tune_frame, _holdout_frame], ignore_index=True)

        print("\n" + "=" * 72)
        print(
            _group_key,
            "| tune", len(_tune_frame),
            "| holdout", len(_holdout_frame),
            "| extraction acoustique...",
        )
        _started = time.time()
        _inputs = [_audio_input(row) for row in _combined.to_dict("records")]
        _all_logits = capture_logprobs(
            _inputs,
            omni_lang(_language),
            manifest_lang=_language,
            domains=_decode_domain,
        )
        _all_refs = [_norm(value, _language) for value in _combined["text"]]
        _tune_n = len(_tune_frame)
        _tune_logits, _holdout_logits = _all_logits[:_tune_n], _all_logits[_tune_n:]
        _tune_refs, _holdout_refs = _all_refs[:_tune_n], _all_refs[_tune_n:]
        del _inputs, _all_logits, _all_refs
        gc.collect()
        torch.cuda.empty_cache()
        print(f"Logits prêts en {(time.time() - _started) / 60:.1f} min")

        _candidates = _candidate_configs(_language, _decode_domain)
        _old_config = dict(
            _baseline_domain.get(_group_key, LM_BEST[_language])
        )
        _old_path = os.path.realpath(
            _old_config.get("lm_bin") or LM_PATHS[_language]
        )
        _old_candidate = _candidate_for_path(_candidates, _old_path)
        if _old_candidate is None:
            _old_candidate = {
                "tag": f"historical_{_old_config.get('lm', 'lm')}",
                "path": _old_path,
                "uni_file": _old_config.get("uni_file") or _write_unigrams(
                    _language, "historical", ("train", "valid")
                ),
            }
            _candidates.append(_old_candidate)

        _grid = []
        _best = None
        for _candidate_index, _candidate in enumerate(_candidates, 1):
            print(
                f"[{_group_key}] LM {_candidate_index}/{len(_candidates)}:",
                _candidate["tag"],
            )
            _decoder = _make_decoder(_candidate)
            _pairs = [
                (alpha, beta)
                for alpha in LM_V4_COARSE_ALPHAS
                for beta in LM_V4_COARSE_BETAS
            ]
            if _candidate["path"] == _old_path:
                _pairs.append((
                    float(_old_config.get("alpha", 0.5)),
                    float(_old_config.get("beta", 1.0)),
                ))
            _pairs = list(dict.fromkeys(_pairs))
            for _index, (_alpha, _beta) in enumerate(_pairs, 1):
                _score = _decode_score(
                    _decoder, _alpha, _beta, 100,
                    _tune_logits, _tune_refs, _language,
                )
                _row = {
                    "stage": "coarse",
                    "lm": _candidate["tag"],
                    "lm_bin": _candidate["path"],
                    "alpha": float(_alpha),
                    "beta": float(_beta),
                    "beam": 100,
                    "wer_tune": _score,
                }
                _grid.append(_row)
                if _best is None or _score < _best[0]:
                    _best = (_score, _candidate, float(_alpha), float(_beta), 100)
                if _index % 4 == 0 or _index == len(_pairs):
                    print(
                        f"  grille {_index}/{len(_pairs)} | meilleur global",
                        f"{_best[0]:.4f}",
                    )
            del _decoder
            gc.collect()

        # Raffinement autour du meilleur couple alpha/beta, uniquement pour le meilleur LM.
        _, _best_candidate, _best_alpha, _best_beta, _ = _best
        _fine_alphas = sorted({
            round(max(0.0, _best_alpha + delta), 4)
            for delta in (-0.2, 0.0, 0.2)
        })
        _fine_betas = sorted({
            round(_best_beta + delta, 4)
            for delta in (-0.5, 0.0, 0.5)
        })
        _decoder = _make_decoder(_best_candidate)
        for _alpha in _fine_alphas:
            for _beta in _fine_betas:
                _score = _decode_score(
                    _decoder, _alpha, _beta, 100,
                    _tune_logits, _tune_refs, _language,
                )
                _grid.append({
                    "stage": "fine",
                    "lm": _best_candidate["tag"],
                    "lm_bin": _best_candidate["path"],
                    "alpha": float(_alpha),
                    "beta": float(_beta),
                    "beam": 100,
                    "wer_tune": _score,
                })
                if _score < _best[0]:
                    _best = (
                        _score, _best_candidate,
                        float(_alpha), float(_beta), 100,
                    )

        # Comparaison beam 100/250 au meilleur alpha/beta.
        _, _best_candidate, _best_alpha, _best_beta, _ = _best
        for _beam in LM_V4_BEAMS:
            _score = _decode_score(
                _decoder, _best_alpha, _best_beta, _beam,
                _tune_logits, _tune_refs, _language,
            )
            _grid.append({
                "stage": "beam",
                "lm": _best_candidate["tag"],
                "lm_bin": _best_candidate["path"],
                "alpha": float(_best_alpha),
                "beta": float(_best_beta),
                "beam": int(_beam),
                "wer_tune": _score,
            })
            if _score < _best[0]:
                _best = (
                    _score, _best_candidate,
                    float(_best_alpha), float(_best_beta), int(_beam),
                )
        del _decoder

        _best_tune, _best_candidate, _best_alpha, _best_beta, _best_beam = _best
        _best_decoder = _make_decoder(_best_candidate)
        _best_holdout = _decode_score(
            _best_decoder, _best_alpha, _best_beta, _best_beam,
            _holdout_logits, _holdout_refs, _language,
        )
        del _best_decoder

        _old_decoder = _make_decoder(_old_candidate)
        _old_holdout = _decode_score(
            _old_decoder,
            float(_old_config.get("alpha", 0.5)),
            float(_old_config.get("beta", 1.0)),
            int(_old_config.get("beam", 100)),
            _holdout_logits,
            _holdout_refs,
            _language,
        )
        del _old_decoder

        _deploy_tuned = _best_holdout <= _old_holdout + LM_V4_HOLDOUT_GUARD
        if _deploy_tuned:
            _deployed = {
                "alpha": float(_best_alpha),
                "beta": float(_best_beta),
                "beam": int(_best_beam),
                "lm": _best_candidate["tag"],
                "lm_bin": _best_candidate["path"],
                "uni_file": _best_candidate["uni_file"],
            }
            _decision = "tuned"
        else:
            _deployed = dict(_old_config)
            _deployed["lm_bin"] = _old_path
            _deployed["uni_file"] = _old_candidate["uni_file"]
            _decision = "historical_guardrail"

        _deployed.update({
            "needs_domain_tuning": False,
            "wer_tune": round(float(_best_tune), 6),
            "wer_holdout": round(float(
                _best_holdout if _deploy_tuned else _old_holdout
            ), 6),
            "tuned_holdout_wer": round(float(_best_holdout), 6),
            "historical_holdout_wer": round(float(_old_holdout), 6),
            "checkpoint_step": 1250,
            "checkpoint_sha256": _pointer["weight_sha256"],
            "tuning_signature": _tuning_signature,
            "selection_decision": _decision,
            "n_tune": len(_tune_frame),
            "n_holdout": len(_holdout_frame),
        })

        _group_result = {
            "tuning_signature": _tuning_signature,
            "language": _language,
            "domain": _decode_domain,
            "completed_at": now_iso(),
            "best_tune_wer": float(_best_tune),
            "tuned_holdout_wer": float(_best_holdout),
            "historical_holdout_wer": float(_old_holdout),
            "decision": _decision,
            "deployed_config": _deployed,
            "grid": sorted(_grid, key=lambda row: row["wer_tune"]),
        }
        _report["groups"][_group_key] = _group_result
        _domain_configs[_group_key] = _deployed
        save_json(_report, _report_path)
        save_json(_domain_configs, _domain_path)

        print(
            f"✅ {_group_key} | tune={_best_tune:.4f}",
            f"| holdout tuned={_best_holdout:.4f}",
            f"| ancien={_old_holdout:.4f}",
            f"| décision={_decision}",
        )
        print(
            "   ", _deployed["lm"],
            f"alpha={_deployed['alpha']}",
            f"beta={_deployed['beta']}",
            f"beam={_deployed.get('beam', 100)}",
        )
        del _tune_logits, _holdout_logits
        gc.collect()
        torch.cuda.empty_cache()

# Aucun dev scripted sw n'existe dans cette compétition : conserver sa configuration historique.
for _language in LANGUAGES:
    for _decode_domain in ("scripted", "unscripted"):
        _key = f"{_language}|{_decode_domain}"
        if _key not in _domain_configs:
            _domain_configs[_key] = dict(LM_BEST[_language])
        _domain_configs[_key]["needs_domain_tuning"] = False

save_json(_domain_configs, _domain_path)
_report["finished_at"] = now_iso()
_report["complete_groups"] = sorted(_report["groups"])
save_json(_report, _report_path)

_summary = []
for _key, _result in _report["groups"].items():
    _config = _result["deployed_config"]
    _summary.append({
        "group": _key,
        "decision": _result["decision"],
        "lm": _config["lm"],
        "alpha": _config["alpha"],
        "beta": _config["beta"],
        "beam": _config.get("beam", 100),
        "tuned_holdout_wer": _result["tuned_holdout_wer"],
        "historical_holdout_wer": _result["historical_holdout_wer"],
        "deployed_holdout_wer": _config["wer_holdout"],
    })

print("\n✅ Réglage KenLM V4 terminé")
print("Rapport :", _report_path)
print("Configurations :", _domain_path)
display(pd.DataFrame(_summary).sort_values("group").reset_index(drop=True))

In [ ]:
# 18.8c — Audit KenLM sans doublons inter-splits (aucune mutation des réglages)
import gc
import hashlib
import json
import os
from collections import Counter

import jiwer
import pandas as pd
import torch

assert PRIMARY_ACOUSTIC_CKPT.endswith("/step_1250"), PRIMARY_ACOUSTIC_CKPT
assert "_lm_eval_frame" in globals(), "Exécuter la version leak-free de 18.8b"
assert "_audio_input" in globals() and "_make_decoder" in globals()
assert "_decode_score" in globals() and "_write_unigrams" in globals()

AUDIT_MAX_CLIPS_PER_GROUP = 120
AUDIT_MIN_CLIPS_PER_GROUP = 24
AUDIT_REGRESSION_TOLERANCE = 0.005

_current_path = os.path.join(
    FT_CONFIG["report_dir"], "kenlm_tuning_by_domain_v3.json"
)
_historical_path = os.path.join(
    FT_CONFIG["report_dir"], "kenlm_tuning_by_domain_v3_pre_step1250.json"
)
_output_path = os.path.join(
    V4_REPORT_DIR, "kenlm_audit_v4_step1250_nodup.json"
)

assert os.path.exists(_current_path), _current_path
assert os.path.exists(_historical_path), _historical_path
_current = json.load(open(_current_path, encoding="utf-8"))
_historical = json.load(open(_historical_path, encoding="utf-8"))


def _audit_sha(value):
    payload = json.dumps(
        value, sort_keys=True, ensure_ascii=False, separators=(",", ":")
    ).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


def _audit_norm(value, language):
    try:
        return normalize_text(value, language)
    except TypeError:
        return normalize_text(value)


def _config_candidate(config, language, tag):
    path = os.path.realpath(config.get("lm_bin") or LM_PATHS[language])
    unigram_file = config.get("uni_file")
    if not unigram_file or not os.path.exists(unigram_file):
        lm_name = str(config.get("lm", "")).lower()
        splits = ("train",) if lm_name in {"v1", "v1_train"} else ("train", "valid")
        unigram_file = _write_unigrams(language, tag, splits)
    return {
        "tag": tag,
        "path": path,
        "uni_file": os.path.realpath(unigram_file),
    }


# Union conservatrice de tous les textes acoustiques/LM internes connus.
_full_index = pd.read_parquet(V4_INDEX_PATH, columns=["language", "split", "text"])
_forbidden = {}
for _language in LANGUAGES:
    _values = set(
        manifest[
            manifest["language"].eq(_language)
            & manifest["split"].isin(["train", "valid"])
        ]["text_norm"].dropna().astype(str)
    )
    _values.update(
        _audit_norm(text, _language)
        for text in _full_index[
            _full_index["language"].eq(_language)
            & _full_index["split"].eq("train")
        ]["text"].dropna().astype(str)
    )
    _forbidden[_language] = {text for text in _values if text}

_contract = {
    "schema": 1,
    "checkpoint": PRIMARY_ACOUSTIC_CKPT,
    "max_clips_per_group": AUDIT_MAX_CLIPS_PER_GROUP,
    "min_clips_per_group": AUDIT_MIN_CLIPS_PER_GROUP,
    "regression_tolerance": AUDIT_REGRESSION_TOLERANCE,
    "policy": "unique normalized references absent from manifest train/valid and V4 train",
    "current_config_sha256": _audit_sha(_current),
    "historical_config_sha256": _audit_sha(_historical),
}
_contract_sha = _audit_sha(_contract)

if os.path.exists(_output_path):
    _report = json.load(open(_output_path, encoding="utf-8"))
    assert _report["contract_sha256"] == _contract_sha, \
        "Contrat 18.8c différent : renommer l'ancien rapport"
else:
    _report = {
        "schema": 1,
        "created_at": now_iso(),
        "contract": _contract,
        "contract_sha256": _contract_sha,
        "groups": {},
    }


def _select_nodup(language, domain):
    pool = _lm_eval_frame[
        _lm_eval_frame["language"].eq(language)
        & _lm_eval_frame["decode_domain"].eq(domain)
    ].copy()
    before = len(pool)
    pool["_norm"] = [
        _audit_norm(value, language) for value in pool["text"].astype(str)
    ]
    pool = pool[
        pool["_norm"].str.strip().ne("")
        & ~pool["_norm"].isin(_forbidden[language])
    ].copy()
    after_overlap_filter = len(pool)
    pool = pool.drop_duplicates("_norm", keep="first")
    after_internal_dedup = len(pool)
    pool["_rank"] = pool["sample_key"].map(lambda key: hashlib.sha256(
        f"{V4_SEED}:kenlm-nodup:{language}:{domain}:{key}".encode()
    ).hexdigest())
    pool = pool.sort_values("_rank").head(AUDIT_MAX_CLIPS_PER_GROUP)
    assert len(pool) >= AUDIT_MIN_CLIPS_PER_GROUP, (
        language, domain, before, after_overlap_filter, after_internal_dedup
    )
    return pool.drop(columns="_rank"), {
        "before": before,
        "after_overlap_filter": after_overlap_filter,
        "after_internal_dedup": after_internal_dedup,
        "audited": len(pool),
    }


_groups = [
    (language, domain)
    for language in LANGUAGES
    for domain in (("unscripted",) if language == "sw" else ("scripted", "unscripted"))
]

for _language, _domain in _groups:
    _key = f"{_language}|{_domain}"
    if _key in _report["groups"]:
        print("[déjà audité]", _key)
        continue

    _sample, _counts = _select_nodup(_language, _domain)
    print("\n", "=" * 68, sep="")
    print(_key, "|", _counts, "| extraction acoustique...")
    _inputs = [_audio_input(row) for row in _sample.to_dict("records")]
    _logits = capture_logprobs(
        _inputs,
        omni_lang(_language),
        manifest_lang=_language,
        domains=_domain,
    )
    _references = [_audit_norm(value, _language) for value in _sample["text"]]
    del _inputs
    gc.collect()
    torch.cuda.empty_cache()

    _new_config = dict(_current[_key])
    _old_config = dict(_historical.get(_key, LM_BEST[_language]))
    _new_candidate = _config_candidate(_new_config, _language, "deployed")
    _old_candidate = _config_candidate(_old_config, _language, "historical")

    _new_decoder = _make_decoder(_new_candidate)
    _new_wer = _decode_score(
        _new_decoder,
        float(_new_config.get("alpha", 0.5)),
        float(_new_config.get("beta", 1.0)),
        int(_new_config.get("beam", 100)),
        _logits,
        _references,
        _language,
    )
    del _new_decoder

    _old_decoder = _make_decoder(_old_candidate)
    _old_wer = _decode_score(
        _old_decoder,
        float(_old_config.get("alpha", 0.5)),
        float(_old_config.get("beta", 1.0)),
        int(_old_config.get("beam", 100)),
        _logits,
        _references,
        _language,
    )
    del _old_decoder, _logits

    _delta = float(_new_wer - _old_wer)
    _approved = _delta <= AUDIT_REGRESSION_TOLERANCE
    _report["groups"][_key] = {
        "language": _language,
        "domain": _domain,
        "counts": _counts,
        "sample_sha256": _audit_sha(sorted(_sample["sample_key"].astype(str))),
        "reference_words": int(sum(len(text.split()) for text in _references)),
        "deployed_wer": float(_new_wer),
        "historical_wer": float(_old_wer),
        "delta": _delta,
        "approved": bool(_approved),
        "deployed_config": _new_config,
        "historical_config": _old_config,
    }
    save_json(_report, _output_path)
    print(
        f"{_key}: nouveau={_new_wer:.4f} | ancien={_old_wer:.4f}",
        f"| delta={_delta:+.4f} | {'PASS' if _approved else 'REVIEW'}",
    )
    gc.collect()
    torch.cuda.empty_cache()


# Mélange des domaines d'après les nombres de lignes réels des shards test.
if "test_shards" not in globals():
    _inventory_path = os.path.join(FT_CONFIG["report_dir"], "test_inventory.json")
    assert os.path.exists(_inventory_path), (
        "test_inventory.json absent : exécuter 10.2 une fois, puis relancer 18.8c"
    )
    _inventory = json.load(open(_inventory_path, encoding="utf-8"))
    test_shards = [
        (row["lang"], row["path"], int(row["rows"]))
        for row in _inventory["shards"]
    ]
    TEST_TOTAL_ROWS = int(_inventory["total_rows"])
    print(
        "Inventaire test restauré :",
        len(test_shards), "shards |", TEST_TOTAL_ROWS, "clips",
    )

_test_counts = Counter()
for _submission_language, _shard, _rows in test_shards:
    _language = TEST_DIR_TO_MANIFEST[_submission_language]
    _test_counts[(_language, shard_domain(_shard))] += int(_rows)

_summary, _language_rows = [], []
for _key, _result in sorted(_report["groups"].items()):
    _summary.append({
        "group": _key,
        "audited_clips": _result["counts"]["audited"],
        "new_wer": _result["deployed_wer"],
        "old_wer": _result["historical_wer"],
        "delta": _result["delta"],
        "approved": _result["approved"],
    })

for _language in LANGUAGES:
    _domains = ("unscripted",) if _language == "sw" else ("scripted", "unscripted")
    _total = sum(_test_counts[(_language, domain)] for domain in _domains)
    if not _total:
        continue
    _new = sum(
        _test_counts[(_language, domain)]
        * _report["groups"][f"{_language}|{domain}"]["deployed_wer"]
        for domain in _domains
    ) / _total
    _old = sum(
        _test_counts[(_language, domain)]
        * _report["groups"][f"{_language}|{domain}"]["historical_wer"]
        for domain in _domains
    ) / _total
    _language_rows.append({
        "language": _language,
        "test_rows": _total,
        "proxy_new_wer": _new,
        "proxy_old_wer": _old,
        "delta": _new - _old,
    })

_macro_new = float(pd.DataFrame(_language_rows)["proxy_new_wer"].mean())
_macro_old = float(pd.DataFrame(_language_rows)["proxy_old_wer"].mean())
_review = [row["group"] for row in _summary if not row["approved"]]
_report.update({
    "finished_at": now_iso(),
    "test_domain_counts": {f"{key[0]}|{key[1]}": value for key, value in _test_counts.items()},
    "language_proxy": _language_rows,
    "proxy_macro_new": _macro_new,
    "proxy_macro_old": _macro_old,
    "proxy_macro_delta": _macro_new - _macro_old,
    "review_groups": _review,
    "status": "PASS" if not _review else "REVIEW",
})
save_json(_report, _output_path)

print("\n=== AUDIT 18.8c PAR GROUPE ===")
display(pd.DataFrame(_summary))
print("\n=== PROXY PAR LANGUE, PONDÉRÉ PAR LES SHARDS TEST ===")
display(pd.DataFrame(_language_rows))
print(
    f"Macro proxy : ancien={_macro_old:.5f} | nouveau={_macro_new:.5f}",
    f"| delta={_macro_new - _macro_old:+.5f}",
)
print("STATUT 18.8c :", _report["status"], "| groupes à revoir :", _review)
print("Rapport :", _output_path)

In [ ]:
# Restaurer test_shards depuis l’inventaire sauvegardé
_inventory_path = os.path.join(
    FT_CONFIG["report_dir"],
    "test_inventory.json",
)

assert os.path.exists(_inventory_path), _inventory_path

_inventory = json.load(open(_inventory_path, encoding="utf-8"))

test_shards = [
    (row["lang"], row["path"], int(row["rows"]))
    for row in _inventory["shards"]
]

TEST_TOTAL_ROWS = int(_inventory["total_rows"])

print("✅ Inventaire restauré")
print("Shards :", len(test_shards))
print("Clips  :", TEST_TOTAL_ROWS)

In [ ]:
# 18.8d — Sélection finale stricte des LM après audit sans doublons
import hashlib
import json
import os
import shutil

import pandas as pd

AUDIT_PATH = os.path.join(
    V4_REPORT_DIR, "kenlm_audit_v4_step1250_nodup.json"
)
DOMAIN_CONFIG_PATH = os.path.join(
    FT_CONFIG["report_dir"], "kenlm_tuning_by_domain_v3.json"
)
DOMAIN_PRE_GUARD_PATH = os.path.join(
    FT_CONFIG["report_dir"], "kenlm_tuning_by_domain_v3_pre_nodup_guard.json"
)
FINAL_SELECTION_PATH = os.path.join(
    V4_REPORT_DIR, "kenlm_selection_v4_step1250_final.json"
)

assert os.path.exists(AUDIT_PATH), AUDIT_PATH
assert os.path.exists(DOMAIN_CONFIG_PATH), DOMAIN_CONFIG_PATH

audit = json.load(open(AUDIT_PATH, encoding="utf-8"))
assert len(audit["groups"]) == 11
assert "test_domain_counts" in audit, "Relancer la fin de 18.8c"

if not os.path.exists(DOMAIN_PRE_GUARD_PATH):
    shutil.copy2(DOMAIN_CONFIG_PATH, DOMAIN_PRE_GUARD_PATH)

configs = json.load(open(DOMAIN_CONFIG_PATH, encoding="utf-8"))
decisions = {}

# Règle prudente : une égalité ne justifie pas un LM/beam supplémentaire.
for key, result in sorted(audit["groups"].items()):
    delta = float(result["delta"])
    if delta < 0.0:
        chosen = dict(result["deployed_config"])
        decision = "keep_tuned"
        proxy_wer = float(result["deployed_wer"])
    else:
        chosen = dict(result["historical_config"])
        decision = "restore_historical"
        proxy_wer = float(result["historical_wer"])

    chosen["needs_domain_tuning"] = False
    chosen["nodup_guard_decision"] = decision
    chosen["nodup_audit_delta"] = delta
    chosen["checkpoint_step"] = 1250
    configs[key] = chosen
    decisions[key] = {
        "decision": decision,
        "audit_delta": delta,
        "proxy_wer": proxy_wer,
        "selected_lm": chosen.get("lm"),
        "alpha": float(chosen.get("alpha", 0.0)),
        "beta": float(chosen.get("beta", 0.0)),
        "beam": int(chosen.get("beam", 100)),
    }

save_json(configs, DOMAIN_CONFIG_PATH)

# Recalcul du proxy avec les configurations réellement déployées.
language_proxy = []
for language in LANGUAGES:
    domains = ("unscripted",) if language == "sw" else ("scripted", "unscripted")
    counts = {
        domain: int(audit["test_domain_counts"].get(f"{language}|{domain}", 0))
        for domain in domains
    }
    total = sum(counts.values())
    assert total > 0, (language, counts)
    final_wer = sum(
        counts[domain] * decisions[f"{language}|{domain}"]["proxy_wer"]
        for domain in domains
    ) / total
    old_wer = sum(
        counts[domain]
        * float(audit["groups"][f"{language}|{domain}"]["historical_wer"])
        for domain in domains
    ) / total
    language_proxy.append({
        "language": language,
        "test_rows": total,
        "proxy_final_wer": final_wer,
        "proxy_historical_wer": old_wer,
        "delta": final_wer - old_wer,
    })

macro_final = sum(row["proxy_final_wer"] for row in language_proxy) / len(language_proxy)
macro_old = sum(row["proxy_historical_wer"] for row in language_proxy) / len(language_proxy)

payload = {
    "schema": 1,
    "created_at": now_iso(),
    "checkpoint": PRIMARY_ACOUSTIC_CKPT,
    "checkpoint_step": 1250,
    "rule": "keep tuned only when no-duplicate audit delta < 0; ties restore historical",
    "audit_path": AUDIT_PATH,
    "audit_contract_sha256": audit["contract_sha256"],
    "decisions": decisions,
    "language_proxy": language_proxy,
    "proxy_macro_final": macro_final,
    "proxy_macro_historical": macro_old,
    "proxy_macro_delta": macro_final - macro_old,
    "domain_config_path": DOMAIN_CONFIG_PATH,
}
payload["selection_sha256"] = hashlib.sha256(json.dumps(
    payload, sort_keys=True, ensure_ascii=False, separators=(",", ":")
).encode("utf-8")).hexdigest()
save_json(payload, FINAL_SELECTION_PATH)

for name in ("DOMAIN_DECODERS", "_decoder_cache", "FINAL_PIPELINES"):
    if name in globals():
        try:
            globals()[name].clear()
        except Exception:
            pass

print("=== DÉCISIONS FINALES KENLM ===")
display(pd.DataFrame([
    {"group": key, **value} for key, value in decisions.items()
]))
print("\n=== PROXY FINAL PAR LANGUE ===")
display(pd.DataFrame(language_proxy))
print(
    f"Macro proxy historique={macro_old:.5f}",
    f"| final={macro_final:.5f}",
    f"| gain={macro_old - macro_final:.5f}",
)
print("✅ Configurations finales sauvegardées :", DOMAIN_CONFIG_PATH)
print("Rapport de sélection :", FINAL_SELECTION_PATH)

In [ ]:
# Préparation du runtime final
RUN_FINAL_INFERENCE = True
STRONG_ARTIFACT_HASH_FOR_FINAL = True

_inventory_path = os.path.join(
    FT_CONFIG["report_dir"],
    "test_inventory.json",
)
assert os.path.exists(_inventory_path), _inventory_path

_inventory = json.load(open(_inventory_path, encoding="utf-8"))

TEST_ROOT = _inventory["root"]
TEST_TOTAL_ROWS = int(_inventory["total_rows"])

test_shards = [
    (row["lang"], row["path"], int(row["rows"]))
    for row in _inventory["shards"]
]

TEST_DIR_TO_MANIFEST = {
    "swa": "sw",
    "kik": "kik",
    "kln": "kln",
    "luo": "luo",
    "som": "som",
    "mas": "mas",
}

TEST_DIR_TO_OMNI = {
    submission_language: omni_lang(internal_language)
    for submission_language, internal_language
    in TEST_DIR_TO_MANIFEST.items()
}

LANG_TO_SUBMISSION = {
    internal_language: submission_language
    for submission_language, internal_language
    in TEST_DIR_TO_MANIFEST.items()
}

assert all(os.path.exists(path) for _, path, _ in test_shards)
assert PRIMARY_ACOUSTIC_CKPT.endswith("/step_1250")
assert len(test_shards) > 0 and TEST_TOTAL_ROWS > 0

print("✅ Runtime final préparé")
print("Checkpoint :", PRIMARY_ACOUSTIC_CKPT)
print("Shards     :", len(test_shards))
print("Clips      :", TEST_TOTAL_ROWS)
print("Test root  :", TEST_ROOT)

In [ ]:
# 19.1 — Geler le résultat 0.36880 et vérifier les KenLM historiques
import hashlib
import json
import os
import shutil

REPORT_DIR = FT_CONFIG["report_dir"]

ACTIVE_CONFIG = os.path.join(
    REPORT_DIR, "kenlm_tuning_by_domain_v3.json"
)
V4_CONFIG_BACKUP = os.path.join(
    REPORT_DIR, "kenlm_tuning_by_domain_v3_score_036880.json"
)
HISTORICAL_CONFIG = os.path.join(
    REPORT_DIR, "kenlm_tuning_by_domain_v3_pre_step1250.json"
)

SUBMISSION_PATH = os.path.join(
    REPORT_DIR, "submission_v3_REPLACE_WITH_LOCAL_RUN_ID.csv"
)

def json_sha256(path):
    value = json.load(open(path, encoding="utf-8"))
    payload = json.dumps(
        value,
        sort_keys=True,
        ensure_ascii=False,
        separators=(",", ":"),
    ).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

assert os.path.isfile(ACTIVE_CONFIG), ACTIVE_CONFIG
assert os.path.isfile(HISTORICAL_CONFIG), HISTORICAL_CONFIG
assert os.path.isfile(SUBMISSION_PATH), SUBMISSION_PATH

# Backup immuable de la configuration ayant obtenu 0.36880.
if not os.path.exists(V4_CONFIG_BACKUP):
    shutil.copy2(ACTIVE_CONFIG, V4_CONFIG_BACKUP)

active = json.load(open(ACTIVE_CONFIG, encoding="utf-8"))
historical = json.load(open(HISTORICAL_CONFIG, encoding="utf-8"))

expected_groups = {
    f"{language}|{domain}"
    for language in ("sw", "kik", "kln", "luo", "som", "mas")
    for domain in ("scripted", "unscripted")
}

active_groups = {key for key in active if "|" in key}
historical_groups = {key for key in historical if "|" in key}

assert expected_groups <= active_groups, (
    "Groupes actifs manquants :",
    sorted(expected_groups - active_groups),
)
assert expected_groups <= historical_groups, (
    "Groupes historiques manquants :",
    sorted(expected_groups - historical_groups),
)

leaderboard_record = {
    "submission": SUBMISSION_PATH,
    "run_id": "REPLACE_WITH_LOCAL_RUN_ID",
    "submission_sha256":
        "REPLACE_WITH_LOCAL_SHA256",
    "public_macro_wer": 0.36880,
    "previous_best_macro_wer": 0.36529,
    "delta": 0.36880 - 0.36529,
    "active_config_sha256": json_sha256(ACTIVE_CONFIG),
    "historical_config_sha256": json_sha256(HISTORICAL_CONFIG),
}

save_json(
    leaderboard_record,
    os.path.join(REPORT_DIR, "leaderboard_result_v4_036880.json"),
)

print("✅ Résultat 0.36880 gelé")
print("Config V4       :", V4_CONFIG_BACKUP)
print("SHA config V4   :", json_sha256(V4_CONFIG_BACKUP))
print("Config historique:", HISTORICAL_CONFIG)
print("SHA historique  :", json_sha256(HISTORICAL_CONFIG))
print("Groupes actifs  :", len(active_groups))
print("Groupes historiques :", len(historical_groups))

In [ ]:
# 19.2 — A/B : step_1250 + KenLM historiques
import ctypes
import gc
import hashlib
import json
import os
import shutil

from pyctcdecode.decoder import BeamSearchDecoderCTC

REPORT_DIR = FT_CONFIG["report_dir"]

ACTIVE_CONFIG = os.path.join(
    REPORT_DIR, "kenlm_tuning_by_domain_v3.json"
)
V4_CONFIG_BACKUP = os.path.join(
    REPORT_DIR, "kenlm_tuning_by_domain_v3_score_036880.json"
)
HISTORICAL_CONFIG = os.path.join(
    REPORT_DIR, "kenlm_tuning_by_domain_v3_pre_step1250.json"
)

EXPECTED_V4_SHA = (
    "REPLACE_WITH_LOCAL_SHA256"
)
EXPECTED_HISTORICAL_SHA = (
    "REPLACE_WITH_LOCAL_SHA256"
)

def json_sha256(path):
    value = json.load(open(path, encoding="utf-8"))
    payload = json.dumps(
        value,
        sort_keys=True,
        ensure_ascii=False,
        separators=(",", ":"),
    ).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

assert os.path.isfile(ACTIVE_CONFIG)
assert os.path.isfile(V4_CONFIG_BACKUP)
assert os.path.isfile(HISTORICAL_CONFIG)

assert json_sha256(ACTIVE_CONFIG) == EXPECTED_V4_SHA
assert json_sha256(V4_CONFIG_BACKUP) == EXPECTED_V4_SHA
assert json_sha256(HISTORICAL_CONFIG) == EXPECTED_HISTORICAL_SHA

# L'acoustique doit rester strictement step_1250.
assert os.path.basename(
    os.path.realpath(PRIMARY_ACOUSTIC_CKPT)
) == "step_1250", PRIMARY_ACOUSTIC_CKPT

# Libération complète des anciens KenLM.
if "_decoder" in globals():
    del _decoder

if "DOMAIN_DECODERS" in globals():
    try:
        DOMAIN_DECODERS.clear()
    except Exception:
        pass

if "_decoder_cache" in globals():
    try:
        _decoder_cache.clear()
    except Exception:
        pass

BeamSearchDecoderCTC.clear_class_models()
gc.collect()
torch.cuda.empty_cache()

try:
    ctypes.CDLL("libc.so.6").malloc_trim(0)
except Exception:
    pass

# Remplacement atomique de la configuration active.
temporary = ACTIVE_CONFIG + ".historical.tmp"
shutil.copy2(HISTORICAL_CONFIG, temporary)
os.replace(temporary, ACTIVE_CONFIG)

assert json_sha256(ACTIVE_CONFIG) == EXPECTED_HISTORICAL_SHA

ab_contract = {
    "experiment": "step1250_historical_kenlm",
    "acoustic_checkpoint": os.path.realpath(PRIMARY_ACOUSTIC_CKPT),
    "historical_config": HISTORICAL_CONFIG,
    "historical_config_sha256": EXPECTED_HISTORICAL_SHA,
    "previous_v4_config_backup": V4_CONFIG_BACKUP,
    "previous_v4_config_sha256": EXPECTED_V4_SHA,
    "reference_submission_run_id": "REPLACE_WITH_LOCAL_RUN_ID",
    "reference_public_wer": 0.36880,
}

save_json(
    ab_contract,
    os.path.join(
        REPORT_DIR,
        "ab_contract_step1250_historical_kenlm.json",
    ),
)

print("✅ KenLM historiques activés")
print("Checkpoint :", PRIMARY_ACOUSTIC_CKPT)
print("Config SHA :", json_sha256(ACTIVE_CONFIG))
print("Test local :", TEST_ROOT)

if not os.path.isdir("/content/kaggle_test_full_fast"):
    print("⚠️ Cache test local absent : il faudra rejouer 17.1d")

In [ ]:
# 19.2R — Restauration session fraîche :
# step_1250 + KenLM historiques

import hashlib
import json
import os
import shutil

assert torch.cuda.is_available(), (
    "17.2 exige un GPU. Repasser le runtime en A100."
)

REPORT_DIR = FT_CONFIG["report_dir"]

STEP1250_CHECKPOINT = (
    "/content/afrivoices_project/"
    "finetune_runs/experiment_run/"
    "balanced_joint_v4/candidates/REPLACE_WITH_LOCAL_RUN_ID/step_1250"
)

ACTIVE_CONFIG = os.path.join(
    REPORT_DIR, "kenlm_tuning_by_domain_v3.json"
)
HISTORICAL_CONFIG = os.path.join(
    REPORT_DIR, "kenlm_tuning_by_domain_v3_pre_step1250.json"
)
V4_CONFIG_BACKUP = os.path.join(
    REPORT_DIR, "kenlm_tuning_by_domain_v3_score_036880.json"
)

EXPECTED_HISTORICAL_SHA = (
    "REPLACE_WITH_LOCAL_SHA256"
)
EXPECTED_V4_SHA = (
    "REPLACE_WITH_LOCAL_SHA256"
)

def json_sha256(path):
    value = json.load(open(path, encoding="utf-8"))
    payload = json.dumps(
        value,
        sort_keys=True,
        ensure_ascii=False,
        separators=(",", ":"),
    ).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

assert os.path.isdir(STEP1250_CHECKPOINT), STEP1250_CHECKPOINT
assert os.path.isfile(HISTORICAL_CONFIG), HISTORICAL_CONFIG
assert os.path.isfile(V4_CONFIG_BACKUP), V4_CONFIG_BACKUP
assert json_sha256(HISTORICAL_CONFIG) == EXPECTED_HISTORICAL_SHA
assert json_sha256(V4_CONFIG_BACKUP) == EXPECTED_V4_SHA

# Restaurer atomiquement les réglages historiques.
temporary = ACTIVE_CONFIG + ".historical.tmp"
shutil.copy2(HISTORICAL_CONFIG, temporary)
os.replace(temporary, ACTIVE_CONFIG)
assert json_sha256(ACTIVE_CONFIG) == EXPECTED_HISTORICAL_SHA

# 2.1b peut avoir restauré le step_5000 rejeté :
# on impose explicitement le checkpoint de cette A/B.
FINAL_UNIFIED_CHECKPOINT = os.path.realpath(STEP1250_CHECKPOINT)

RUN_TEST_INFERENCE = True
RUN_FINAL_INFERENCE = False

save_json(
    {
        "experiment": "step1250_historical_kenlm_fresh_resume",
        "checkpoint": FINAL_UNIFIED_CHECKPOINT,
        "kenlm_config": HISTORICAL_CONFIG,
        "kenlm_config_sha256": EXPECTED_HISTORICAL_SHA,
        "gpu": torch.cuda.get_device_name(0),
    },
    os.path.join(
        REPORT_DIR,
        "ab_contract_step1250_historical_kenlm.json",
    ),
)

print("✅ Restauration A/B prête")
print("GPU        :", torch.cuda.get_device_name(0))
print("Checkpoint :", FINAL_UNIFIED_CHECKPOINT)
print("Config SHA :", json_sha256(ACTIVE_CONFIG))

In [ ]:
# 19.3 — Geler définitivement la pile ayant produit 0.36529
import hashlib
import json
import os
import shutil

REPORT_DIR = FT_CONFIG["report_dir"]
FROZEN_DIR = os.path.join(REPORT_DIR, "frozen_best_036529")
os.makedirs(FROZEN_DIR, exist_ok=True)

sources = [
    os.path.join(REPORT_DIR, "submission_lm.csv"),
    os.path.join(
        REPORT_DIR,
        "kenlm_tuning_by_domain_v3_pre_step1250.json",
    ),
    os.path.join(REPORT_DIR, "topup_choice_kik.json"),
    os.path.join(REPORT_DIR, "topup_choice_kln.json"),
    os.path.join(REPORT_DIR, "topup_choice_mas.json"),
]

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024**2), b""):
            digest.update(block)
    return digest.hexdigest()

inventory = []

for source in sources:
    assert os.path.isfile(source), source
    destination = os.path.join(
        FROZEN_DIR,
        os.path.basename(source),
    )

    if not os.path.exists(destination):
        shutil.copy2(source, destination)

    assert sha256_file(destination) == sha256_file(source)

    inventory.append({
        "file": os.path.basename(destination),
        "sha256": sha256_file(destination),
        "bytes": os.path.getsize(destination),
    })

manifest = {
    "public_macro_wer": 0.36529,
    "submission": "submission_lm.csv",
    "acoustic_routes": {
        "default": (
            "train_output/workspace_selected/checkpoints/step_5000"
        ),
        "kik": (
            "topup_kik/workspace_selected/checkpoints/step_1000"
        ),
        "kln": (
            "topup_kln/workspace_selected/checkpoints/step_2000"
        ),
        "mas": (
            "topup_mas_spont/workspace_selected/checkpoints/step_2000"
        ),
    },
    "kenlm": {
        "kik": "v4",
        "sw": "v4",
        "kln": "v5",
        "luo": "v5",
        "mas": "v5",
        "som": "v5",
    },
    "files": inventory,
}

manifest_path = os.path.join(
    FROZEN_DIR,
    "winning_stack_036529.json",
)

with open(manifest_path, "w", encoding="utf-8") as handle:
    json.dump(
        manifest,
        handle,
        ensure_ascii=False,
        indent=2,
    )

print("✅ Pile 0.36529 gelée :", FROZEN_DIR)
print(json.dumps(inventory, indent=2))

In [ ]:
# 19.4 — Construire M1/M2 : un unique checkpoint 975M par fusion de deltas
# Aucun entraînement. Exécutable sur CPU haute RAM ou GPU.
import copy
import gc
import hashlib
import json
import os
import shutil
import subprocess
from datetime import datetime, timezone
from pathlib import Path

import psutil
import torch


FT_ROOT = Path(
    "/content/afrivoices_project/"
    "finetune_runs/experiment_run"
)
REPORT_DIR = FT_ROOT / "reports"

SOURCE_CHECKPOINTS = {
    "base_step5000": FT_ROOT / "train_output/workspace_selected/checkpoints/step_5000",
    "kln_step2000": FT_ROOT / "topup_kln/workspace_selected/checkpoints/step_2000",
    "mas_spont_step2000": (
        FT_ROOT / "topup_mas_spont/workspace_selected/checkpoints/step_2000"
    ),
}

# Kikuyu est volontairement exclu du premier tour : son gain historique était faible.
MERGE_CANDIDATES = {
    "M1": {"kln_step2000": 0.25, "mas_spont_step2000": 0.25},
    "M2": {"kln_step2000": 0.50, "mas_spont_step2000": 0.50},
}

EXPECTED_ACOUSTIC_PARAMETERS = 975_675_056
LOCAL_SOURCE_DIR = Path("/content/delta_merge_sources_v2")
LOCAL_BUILD_DIR = Path("/content/delta_merge_builds_v2")
DRIVE_OUTPUT_ROOT = FT_ROOT / "unified_delta_merges_v2"
LOCAL_SOURCE_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_BUILD_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)


def now_iso():
    return datetime.now(timezone.utc).isoformat()


def atomic_json(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".tmp-{os.getpid()}")
    with open(temporary, "w", encoding="utf-8") as handle:
        json.dump(value, handle, ensure_ascii=False, indent=2)
    os.replace(temporary, path)


def canonical_hash(value):
    payload = json.dumps(
        value, sort_keys=True, ensure_ascii=False, separators=(",", ":")
    ).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


def sha256_file(path):
    path = Path(path)
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(16 * 1024**2), b""):
            digest.update(block)
    return digest.hexdigest()


def weight_file(checkpoint):
    checkpoint = Path(checkpoint)
    assert checkpoint.exists(), checkpoint
    if checkpoint.is_file():
        return checkpoint
    model_root = checkpoint / "model"
    files = sorted(path for path in model_root.rglob("*") if path.is_file())
    assert len(files) == 1, (checkpoint, files)
    return files[0]


def rsync_file(source, destination):
    source, destination = Path(source), Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    assert shutil.which("rsync"), "rsync absent du runtime"
    subprocess.run(
        [
            "rsync", "-a", "--partial", "--append-verify", "--info=progress2",
            str(source), str(destination),
        ],
        check=True,
    )
    assert destination.is_file()
    assert destination.stat().st_size == source.stat().st_size


available_ram = psutil.virtual_memory().available
free_disk = shutil.disk_usage("/content").free
assert available_ram >= 32 * 2**30, (
    f"Au moins 32 Gio de RAM disponible requis, trouvé {available_ram / 2**30:.1f} Gio"
)
assert free_disk >= 28 * 2**30, (
    f"Au moins 28 Gio libres dans /content requis, trouvé {free_disk / 2**30:.1f} Gio"
)

print("RAM disponible :", round(available_ram / 2**30, 1), "Gio")
print("Disque libre    :", round(free_disk / 2**30, 1), "Gio")

# Copier une seule fois les trois gros poids depuis Drive vers le disque local.
source_files = {}
source_meta = {}
for source_name, checkpoint in SOURCE_CHECKPOINTS.items():
    drive_weight = weight_file(checkpoint)
    local_weight = LOCAL_SOURCE_DIR / f"{source_name}{drive_weight.suffix or '.pt'}"
    if not local_weight.exists() or local_weight.stat().st_size != drive_weight.stat().st_size:
        print(f"\nCopie locale de {source_name}...")
        rsync_file(drive_weight, local_weight)
    print(f"Hash de {source_name}...")
    digest = sha256_file(local_weight)
    source_files[source_name] = local_weight
    source_meta[source_name] = {
        "checkpoint": str(checkpoint),
        "drive_weight": str(drive_weight),
        "local_weight": str(local_weight),
        "bytes": int(local_weight.stat().st_size),
        "sha256": digest,
    }
    print(source_name, "->", digest[:16], round(local_weight.stat().st_size / 2**30, 2), "Gio")


def load_state(path):
    raw = torch.load(str(path), map_location="cpu", weights_only=False)
    state = raw.get("model", raw) if isinstance(raw, dict) else raw
    assert isinstance(state, dict) and state
    return raw, state


print("\nChargement des états en RAM...")
raw_objects = {}
states = {}
for source_name, local_weight in source_files.items():
    raw, state = load_state(local_weight)
    raw_objects[source_name] = raw
    states[source_name] = state
    print(source_name, ":", len(state), "tenseurs/entrées")

base_state = states["base_step5000"]
base_keys = list(base_state.keys())
for source_name, state in states.items():
    assert list(state.keys()) == base_keys, f"Clés divergentes : {source_name}"
    for key, base_value in base_state.items():
        value = state[key]
        if torch.is_tensor(base_value):
            assert torch.is_tensor(value)
            assert value.shape == base_value.shape, (source_name, key)
            assert value.dtype == base_value.dtype, (source_name, key, value.dtype, base_value.dtype)
            if not torch.is_floating_point(base_value):
                assert torch.equal(value, base_value), (
                    "Buffer entier/booléen divergent", source_name, key
                )

state_tensor_numel = sum(
    int(value.numel()) for value in base_state.values() if torch.is_tensor(value)
)
print("Éléments du state_dict (paramètres + buffers) :", f"{state_tensor_numel:,}")
print("Paramètres architecturaux attendus             :", f"{EXPECTED_ACOUSTIC_PARAMETERS:,}")

published = []
for candidate_name, component_weights in MERGE_CANDIDATES.items():
    spec = {
        "schema": 2,
        "algorithm": "float32_task_arithmetic_then_restore_source_dtype",
        "base": source_meta["base_step5000"],
        "components": [
            {
                "name": name,
                "weight": float(weight),
                "source": source_meta[name],
            }
            for name, weight in sorted(component_weights.items())
        ],
        "expected_acoustic_parameters": EXPECTED_ACOUSTIC_PARAMETERS,
    }
    spec_sha256 = canonical_hash(spec)
    candidate_id = f"{candidate_name.lower()}_{spec_sha256[:16]}"
    final_root = DRIVE_OUTPUT_ROOT / candidate_id
    final_step = final_root / "step_0"
    final_weight = final_step / "model/pp_00/tp_00/sdp_00.pt"
    final_manifest = final_step / "merge_manifest.json"
    final_complete = final_root / "_COMPLETE.json"

    if final_weight.exists() and final_manifest.exists() and final_complete.exists():
        existing = json.load(open(final_manifest, encoding="utf-8"))
        complete = json.load(open(final_complete, encoding="utf-8"))
        assert existing["spec_sha256"] == spec_sha256
        assert complete["spec_sha256"] == spec_sha256
        assert complete["output_weight_sha256"] == existing["output_weight_sha256"]
        assert existing["output_bytes"] == final_weight.stat().st_size
        print(f"\n[Déjà publié] {candidate_name} -> {final_step}")
        published.append(existing)
        continue

    assert not final_root.exists(), (
        f"Dossier final incomplet détecté : {final_root}. "
        "Ne pas l'écraser ; inspecter ou renommer ce dossier."
    )

    print(f"\nFusion {candidate_name} : {component_weights}")
    merged = {}
    with torch.no_grad():
        for index, key in enumerate(base_keys, start=1):
            base_value = base_state[key]
            if torch.is_tensor(base_value):
                if torch.is_floating_point(base_value):
                    base_float = base_value.float()
                    accumulator = base_float.clone()
                    for component_name, weight in component_weights.items():
                        component_float = states[component_name][key].float()
                        accumulator.add_(component_float - base_float, alpha=float(weight))
                        del component_float
                    merged[key] = accumulator.to(dtype=base_value.dtype)
                    del accumulator, base_float
                else:
                    merged[key] = base_value.clone()
            else:
                merged[key] = copy.deepcopy(base_value)
            if index % 100 == 0 or index == len(base_keys):
                print(f"  {index}/{len(base_keys)} entrées fusionnées")

    local_candidate = LOCAL_BUILD_DIR / f"{candidate_id}.pt"
    local_temporary = local_candidate.with_suffix(".pt.tmp")
    if local_temporary.exists():
        local_temporary.unlink()
    if isinstance(raw_objects["base_step5000"], dict):
        output_wrapper = copy.copy(raw_objects["base_step5000"])
        output_wrapper["model"] = merged
    else:
        output_wrapper = merged
    torch.save(output_wrapper, local_temporary)
    os.replace(local_temporary, local_candidate)
    del merged, output_wrapper
    gc.collect()

    # Relecture structurelle locale avant toute publication Drive.
    validation_raw, validation_state = load_state(local_candidate)
    assert list(validation_state.keys()) == base_keys
    for key, base_value in base_state.items():
        value = validation_state[key]
        if torch.is_tensor(base_value):
            assert torch.is_tensor(value)
            assert value.shape == base_value.shape
            assert value.dtype == base_value.dtype
    del validation_raw, validation_state
    gc.collect()

    print("Hash du checkpoint fusionné local...")
    output_sha256 = sha256_file(local_candidate)
    output_bytes = int(local_candidate.stat().st_size)

    staging_root = DRIVE_OUTPUT_ROOT / f".{candidate_id}.staging"
    staging_weight = staging_root / "step_0/model/pp_00/tp_00/sdp_00.pt"
    staging_manifest = staging_root / "step_0/merge_manifest.json"
    staging_complete = staging_root / "_COMPLETE.json"
    staging_weight.parent.mkdir(parents=True, exist_ok=True)
    print("Publication Drive de", candidate_name, "...")
    rsync_file(local_candidate, staging_weight)
    print("Vérification SHA sur Drive...")
    assert sha256_file(staging_weight) == output_sha256

    manifest = {
        "created_at": now_iso(),
        "candidate": candidate_name,
        "candidate_id": candidate_id,
        "checkpoint": str(final_step),
        "spec": spec,
        "spec_sha256": spec_sha256,
        "output_weight_sha256": output_sha256,
        "output_bytes": output_bytes,
        "state_dict_tensor_numel": state_tensor_numel,
        "acoustic_parameter_count": EXPECTED_ACOUSTIC_PARAMETERS,
        "single_acoustic_checkpoint": True,
        "evaluation_status": "NOT_EVALUATED",
    }
    atomic_json(manifest, staging_manifest)
    atomic_json(
        {
            "schema": 2,
            "candidate": candidate_name,
            "spec_sha256": spec_sha256,
            "output_weight_sha256": output_sha256,
            "output_bytes": output_bytes,
            "completed_at": now_iso(),
        },
        staging_complete,
    )
    os.replace(staging_root, final_root)
    assert final_weight.exists() and final_manifest.exists() and final_complete.exists()
    print("✅", candidate_name, "publié :", final_step)
    published.append(manifest)

    # Le poids est maintenant vérifié sur Drive ; libérer le disque VM.
    local_candidate.unlink()
    gc.collect()

summary = {
    "created_at": now_iso(),
    "candidates": published,
    "next_step": "Evaluate M1/M2 end-to-end on long-aware word-weighted dev; do not submit yet.",
}
atomic_json(summary, REPORT_DIR / "delta_merge_build_v2.json")

print("\n✅ Construction M1/M2 terminée")
for item in published:
    print(item["candidate"], "->", item["checkpoint"], item["output_weight_sha256"][:16])
print("Rapport ->", REPORT_DIR / "delta_merge_build_v2.json")
print("NE PAS lancer 17.2 : ces candidats doivent d'abord être évalués localement.")

In [ ]:
# 19.4b — Compacter M1/M2 sans recalculer les fusions
#
# La premiere publication 19.4 a conserve le state_dict plat du checkpoint source ET a ajoute
# le state_dict fusionne sous la cle "model". Les poids fusionnes sont corrects, mais le fichier
# contient donc deux jeux de tenseurs (~7.27 Gio). Cette cellule extrait uniquement le state_dict
# fusionne, verifie l'egalite tenseur par tenseur, puis publie un checkpoint compact (~3.63 Gio).

import gc
import hashlib
import json
import os
import shutil
import subprocess
from datetime import datetime, timezone
from pathlib import Path

import psutil
import torch


FT_ROOT = Path(
    "/content/afrivoices_project/"
    "finetune_runs/experiment_run"
)
SOURCE_ROOT = FT_ROOT / "unified_delta_merges_v2"
OUTPUT_ROOT = FT_ROOT / "unified_delta_merges_v3_compact"
REPORT_PATH = FT_ROOT / "reports/delta_merge_compaction_v3.json"
LOCAL_BUILD_ROOT = Path("/content/delta_merge_compact_builds_v3")
LOCAL_EVAL_ROOT = Path("/content/delta_merge_eval_checkpoints_v2")
EXPECTED_STATE_ELEMENTS = 975_675_056

SOURCES = {
    "M1": SOURCE_ROOT / "m1_REPLACE_WITH_LOCAL_RUN_ID/step_0",
    "M2": SOURCE_ROOT / "m2_REPLACE_WITH_LOCAL_RUN_ID/step_0",
}

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_BUILD_ROOT.mkdir(parents=True, exist_ok=True)
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

assert psutil.virtual_memory().available >= 20 * 2**30, (
    f"Au moins 20 Gio de RAM disponible requis ; trouve "
    f"{psutil.virtual_memory().available / 2**30:.1f} Gio"
)
assert shutil.disk_usage("/content").free >= 18 * 2**30, (
    f"Au moins 18 Gio libres dans /content requis ; trouve "
    f"{shutil.disk_usage('/content').free / 2**30:.1f} Gio"
)


def now_iso():
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path, block_size=16 << 20):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()


def canonical_sha256(value):
    raw = json.dumps(
        value, sort_keys=True, ensure_ascii=False, separators=(",", ":"), default=str
    ).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()


def atomic_json(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".tmp-{os.getpid()}")
    with open(temporary, "w", encoding="utf-8") as handle:
        json.dump(value, handle, ensure_ascii=False, indent=2, default=str)
    os.replace(temporary, path)


def weight_file(checkpoint):
    checkpoint = Path(checkpoint)
    files = sorted(path for path in (checkpoint / "model").rglob("*") if path.is_file())
    assert len(files) == 1, (checkpoint, files)
    return files[0]


def rsync_file(source, destination):
    source, destination = Path(source), Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and destination.stat().st_size > source.stat().st_size:
        quarantine = destination.with_name(
            destination.name + f".oversize-{int(datetime.now().timestamp())}"
        )
        os.replace(destination, quarantine)
    subprocess.run([
        "rsync", "-a", "--partial", "--append-verify", "--info=progress2",
        str(source), str(destination),
    ], check=True)
    assert destination.stat().st_size == source.stat().st_size


def source_manifest(checkpoint):
    path = Path(checkpoint) / "merge_manifest.json"
    assert path.exists(), path
    payload = json.load(open(path, encoding="utf-8"))
    return path, payload


def compact_destination(name, source_sha256, source_spec_sha256):
    identity = {
        "schema": 1,
        "algorithm": "extract_nested_merged_state_without_recomputing_weights",
        "candidate": name,
        "source_weight_sha256": source_sha256,
        "source_spec_sha256": source_spec_sha256,
        "output_format": "flat_state_dict",
    }
    candidate_id = f"{name.lower()}_compact_{canonical_sha256(identity)[:16]}"
    root = OUTPUT_ROOT / candidate_id
    return candidate_id, root, root / "step_0"


def validate_completed(name, source_sha256, final_root, final_step):
    manifest_path = final_step / "merge_manifest.json"
    complete_path = final_root / "_COMPLETE.json"
    final_weight = weight_file(final_step)
    manifest = json.load(open(manifest_path, encoding="utf-8"))
    complete = json.load(open(complete_path, encoding="utf-8"))
    assert manifest["candidate_id"] == final_root.name
    assert manifest["checkpoint"] == str(final_step)
    assert manifest["candidate"] == name
    assert manifest["source_duplicated_weight_sha256"] == source_sha256
    assert manifest["compact_flat_state_dict"] is True
    assert manifest["single_acoustic_checkpoint"] is True
    assert manifest["acoustic_parameter_count"] == EXPECTED_STATE_ELEMENTS
    assert manifest["spec"]["tensor_equality_verified"] is True
    assert complete["candidate"] == name
    assert complete["spec_sha256"] == manifest["spec_sha256"]
    assert complete["output_weight_sha256"] == manifest["output_weight_sha256"]
    assert complete["output_bytes"] == manifest["output_bytes"]
    assert final_weight.stat().st_size == manifest["output_bytes"]
    assert sha256_file(final_weight) == manifest["output_weight_sha256"]
    return manifest


published = []
for name, source_step in SOURCES.items():
    print("\n" + "=" * 76)
    print("COMPACTION", name)
    source_manifest_path, old_manifest = source_manifest(source_step)
    assert old_manifest["candidate"] == name, (name, old_manifest["candidate"])
    expected_source_sha = old_manifest["output_weight_sha256"]
    source_drive_weight = weight_file(source_step)
    assert source_drive_weight.stat().st_size == int(old_manifest["output_bytes"])
    source_complete_path = source_step.parent / "_COMPLETE.json"
    assert source_complete_path.exists(), source_complete_path
    source_complete = json.load(open(source_complete_path, encoding="utf-8"))
    assert source_complete["candidate"] == name
    assert source_complete["spec_sha256"] == old_manifest["spec_sha256"]
    assert source_complete["output_weight_sha256"] == expected_source_sha
    assert int(source_complete["output_bytes"]) == int(old_manifest["output_bytes"])

    candidate_id, final_root, final_step = compact_destination(
        name, expected_source_sha, old_manifest["spec_sha256"]
    )
    if (
        (final_step / "merge_manifest.json").exists()
        and (final_root / "_COMPLETE.json").exists()
        and (final_step / "model").is_dir()
    ):
        manifest = validate_completed(
            name, expected_source_sha, final_root, final_step
        )
        print("[deja compact]", final_step)
        published.append(manifest)
        continue

    assert not final_root.exists(), (
        f"Publication compacte incomplete : {final_root}. Renommer ce dossier avant reprise."
    )

    # 19.5 a deja copie les gros fichiers en local. Les reutiliser si leur SHA correspond ;
    # sinon lire la version Drive.
    local_eval_weight = (
        LOCAL_EVAL_ROOT / name / "step_0/model/pp_00/tp_00/sdp_00.pt"
    )
    if not local_eval_weight.exists() or \
            local_eval_weight.stat().st_size != source_drive_weight.stat().st_size:
        print("Copie locale du checkpoint 7.27 Gio depuis Drive...")
        rsync_file(source_drive_weight, local_eval_weight)
    print("Verification de la copie locale 19.5...")
    if sha256_file(local_eval_weight) != expected_source_sha:
        local_eval_weight.unlink()
        rsync_file(source_drive_weight, local_eval_weight)
        assert sha256_file(local_eval_weight) == expected_source_sha
    source_weight = local_eval_weight
    print("Copie locale 19.5 reutilisee.")

    print("Verification du checkpoint duplique...")
    source_sha = sha256_file(source_weight)
    assert source_sha == expected_source_sha, (source_sha, expected_source_sha)
    source_bytes = int(source_weight.stat().st_size)
    assert source_bytes > 6 * 2**30, (
        f"{name} n'est pas le packaging duplique attendu : {source_bytes / 2**30:.2f} Gio"
    )

    print("Chargement de", round(source_bytes / 2**30, 2), "Gio...")
    try:
        raw = torch.load(
            str(source_weight), map_location="cpu", weights_only=False, mmap=True
        )
    except (TypeError, RuntimeError):
        raw = torch.load(str(source_weight), map_location="cpu", weights_only=False)
    assert isinstance(raw, dict) and isinstance(raw.get("model"), dict), (
        "Le state_dict fusionne imbrique est absent", type(raw)
    )
    merged_state = raw["model"]
    merged_keys = list(merged_state.keys())
    assert len(merged_keys) == 807, len(merged_keys)
    state_elements = sum(
        int(value.numel()) for value in merged_state.values() if torch.is_tensor(value)
    )
    assert state_elements == EXPECTED_STATE_ELEMENTS, state_elements

    # Preuve du doublon : le niveau externe contient aussi l'ancien state_dict plat.
    outer_keys = [key for key in raw.keys() if key != "model"]
    assert outer_keys == merged_keys, (
        "Les 807 cles externes ne correspondent pas exactement au state fusionne",
        len(outer_keys), len(merged_keys),
    )
    duplicated_tensor_keys = [
        key for key in outer_keys
        if torch.is_tensor(raw[key]) and torch.is_tensor(merged_state[key])
    ]
    assert len(raw) == 808 and len(duplicated_tensor_keys) == 807, (
        len(raw), len(duplicated_tensor_keys)
    )
    for key in outer_keys:
        before, merged = raw[key], merged_state[key]
        assert type(before) is type(merged), (key, type(before), type(merged))
        if torch.is_tensor(before):
            assert before.shape == merged.shape, key
            assert before.dtype == merged.dtype, key
            assert before.layout == merged.layout, key
    print("Tenseurs externes dupliques detectes :", len(duplicated_tensor_keys))

    local_compact = LOCAL_BUILD_ROOT / f"{candidate_id}.pt"
    temporary = local_compact.with_name(local_compact.name + f".tmp-{os.getpid()}")
    if temporary.exists():
        temporary.unlink()
    print("Ecriture du state_dict fusionne seul...")
    torch.save(merged_state, temporary)
    os.replace(temporary, local_compact)
    compact_bytes = int(local_compact.stat().st_size)
    assert 3 * 2**30 < compact_bytes < 4.5 * 2**30, (
        compact_bytes / 2**30
    )
    assert compact_bytes < 0.65 * source_bytes, (compact_bytes, source_bytes)

    # Relecture et equivalence exacte, cle/taille/dtype/valeur.
    print("Relecture et comparaison tenseur par tenseur...")
    compact_raw = torch.load(str(local_compact), map_location="cpu", weights_only=False)
    compact_state = compact_raw.get("model", compact_raw) \
        if isinstance(compact_raw, dict) else compact_raw
    assert list(compact_state.keys()) == merged_keys
    with torch.no_grad():
        for index, key in enumerate(merged_keys, 1):
            before, after = merged_state[key], compact_state[key]
            if torch.is_tensor(before):
                assert torch.is_tensor(after)
                assert before.shape == after.shape and before.dtype == after.dtype
                assert before.layout == after.layout
                assert torch.equal(before, after), key
            else:
                assert type(before) is type(after), (key, type(before), type(after))
                assert before == after, key
            if index % 100 == 0 or index == len(merged_keys):
                print(f"  {index}/{len(merged_keys)} entrees identiques")

    del compact_raw, compact_state
    gc.collect()
    print("Hash du checkpoint compact...")
    compact_sha = sha256_file(local_compact)

    spec = {
        "schema": 1,
        "algorithm": "extract_nested_merged_state_without_recomputing_weights",
        "candidate": name,
        "source_checkpoint": str(source_step),
        "source_manifest": str(source_manifest_path),
        "source_manifest_sha256": sha256_file(source_manifest_path),
        "source_duplicated_weight_sha256": source_sha,
        "source_bytes": source_bytes,
        "output_format": "flat_state_dict",
        "state_entries": len(merged_keys),
        "state_elements": state_elements,
        "tensor_equality_verified": True,
    }
    spec_sha = canonical_sha256(spec)
    staging_root = OUTPUT_ROOT / f".{candidate_id}.staging"
    staging_step = staging_root / "step_0"
    staging_weight = staging_step / "model/pp_00/tp_00/sdp_00.pt"
    staging_manifest_path = staging_step / "merge_manifest.json"
    if staging_manifest_path.exists():
        staged_manifest = json.load(open(staging_manifest_path, encoding="utf-8"))
        if staged_manifest.get("source_duplicated_weight_sha256") != source_sha:
            quarantine = OUTPUT_ROOT / (
                f".{candidate_id}.quarantine-{int(datetime.now().timestamp())}"
            )
            os.replace(staging_root, quarantine)
    print("Publication Drive compacte...")
    rsync_file(local_compact, staging_weight)
    assert sha256_file(staging_weight) == compact_sha

    manifest = {
        "created_at": now_iso(),
        "candidate": name,
        "candidate_id": candidate_id,
        "checkpoint": str(final_step),
        "spec": spec,
        "spec_sha256": spec_sha,
        "source_duplicated_weight_sha256": source_sha,
        "output_weight_sha256": compact_sha,
        "output_bytes": compact_bytes,
        "acoustic_parameter_count": EXPECTED_STATE_ELEMENTS,
        "single_acoustic_checkpoint": True,
        "compact_flat_state_dict": True,
        "evaluation_status": "NOT_EVALUATED",
    }
    atomic_json(manifest, staging_step / "merge_manifest.json")
    atomic_json({
        "schema": 1,
        "candidate": name,
        "spec_sha256": spec_sha,
        "source_duplicated_weight_sha256": source_sha,
        "output_weight_sha256": compact_sha,
        "output_bytes": compact_bytes,
        "completed_at": now_iso(),
    }, staging_root / "_COMPLETE.json")
    os.replace(staging_root, final_root)
    manifest = validate_completed(name, source_sha, final_root, final_step)
    print(
        "✅", name, "compact :", final_step,
        "|", round(compact_bytes / 2**30, 2), "Gio",
        "|", compact_sha[:16],
    )
    published.append(manifest)

    # Remplacer uniquement la copie ephemere 19.5 par la version compacte verifiee.
    # Les publications Drive V2 de 7.27 Gio restent intactes comme provenance.
    del merged_state, raw
    gc.collect()
    local_eval_weight.parent.mkdir(parents=True, exist_ok=True)
    local_eval_tmp = local_eval_weight.with_name(
        local_eval_weight.name + f".compact-tmp-{os.getpid()}"
    )
    shutil.copy2(local_compact, local_eval_tmp)
    os.replace(local_eval_tmp, local_eval_weight)
    assert sha256_file(local_eval_weight) == compact_sha
    local_compact.unlink()
    gc.collect()


report = {
    "schema": 1,
    "created_at": now_iso(),
    "candidates": published,
    "old_v2_publications_preserved": True,
    "next_step": "Run corrected 19.5 against unified_delta_merges_v3_compact.",
}
atomic_json(report, REPORT_PATH)
print("\n✅ Compaction M1/M2 terminee")
for item in published:
    print(
        item["candidate"], "->", item["checkpoint"],
        f"{item['output_bytes'] / 2**30:.2f} Gio",
        item["output_weight_sha256"][:16],
    )
print("Rapport ->", REPORT_PATH)

In [ ]:
# 19.5 V3 COMPACT+SEMANTIC — Comparaison rule-safe de joint / step1250 / M1 / M2
#
# Objectif : mesurer les quatre checkpoints sur le MEME dev V4 fige, avec :
#   - un seul checkpoint acoustique charge a la fois ;
#   - batch_size=1 (la sonde 17.1e a montre que les batches >1 divergent) ;
#   - le routage KenLM HISTORIQUE V4/V5 du score 0.36529, sans retuning ;
#   - le WER corpus exact dans chaque langue, puis la macro-moyenne des 6 langues ;
#   - des caches atomiques par blocs de 32 clips sur Drive.
#
# Cette cellule NE modifie PAS FINAL_UNIFIED_CHECKPOINT ni les fichiers de routage actifs.

import gc
import hashlib
import importlib.metadata
import io
import json
import os
import re
import shutil
import subprocess
import sys
import time
import unicodedata
from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import soundfile as sf
import torch
from IPython.display import display


# -----------------------------------------------------------------------------
# 0. Prerequis et constantes explicites
# -----------------------------------------------------------------------------

assert torch.cuda.is_available(), (
    "19.5 requiert un GPU. Un L4 suffit : changer le type d'execution, "
    "puis relancer les cellules d'installation/FT_CONFIG avant 19.5."
)
assert "FT_CONFIG" in globals(), (
    "FT_CONFIG absent : relancer les cellules 1.x -> 2.4 du notebook avant 19.5."
)
assert "omni_lang" in globals(), (
    "omni_lang absent : relancer la cellule 2.3. Elle applique notamment le code "
    "de repli correct pour le maasai."
)
assert callable(omni_lang)

try:
    import jiwer
    import kenlm  # noqa: F401
    import pyctcdecode
except ImportError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "jiwer", "pyctcdecode", "kenlm"],
        check=True,
    )
    import jiwer
    import kenlm  # noqa: F401
    import pyctcdecode

from pyctcdecode import build_ctcdecoder


FT_ROOT = Path(
    "/content/afrivoices_project/"
    "finetune_runs/experiment_run"
)
REPORT_DIR = FT_ROOT / "reports"
V4_REPORT_DIR = REPORT_DIR / "balanced_v4"
DEV_SELECTION_PATH = V4_REPORT_DIR / "selected_dev_v4.parquet"
TRAIN_SELECTION_PATH = V4_REPORT_DIR / "selected_records_v4.parquet"
HISTORICAL_LM_CONFIG = REPORT_DIR / "kenlm_tuning_by_domain_v3_pre_step1250.json"
PREVIOUSLY_RECORDED_HISTORICAL_CONFIG_SHA256 = (
    "REPLACE_WITH_LOCAL_SHA256"
)
COMPACTION_REPORT = REPORT_DIR / "delta_merge_compaction_v3.json"

EVAL_ROOT = REPORT_DIR / "delta_merge_eval_v2"
LOCAL_CHECKPOINT_ROOT = Path("/content/delta_merge_eval_checkpoints_v2")
LOCAL_AUDIO_ROOT = Path("/content/delta_merge_eval_audio_v2")
PART_SIZE = 32
EXPECTED_ACOUSTIC_PARAMETERS = 975_675_056
LANGUAGES = ["sw", "kik", "kln", "luo", "som", "mas"]
OMNI_CODES = {language: omni_lang(language) for language in LANGUAGES}
NORMALIZATION_VERSION = "nfc-mojibake-safe-v3"
BOOTSTRAP_REPETITIONS = 2_000
BOOTSTRAP_SEED = 19_500_042

# Assets exacts de la soumission historique 0.36529. On ne reconstruit jamais ici un
# vocabulaire depuis le manifest actuel et on ne devine pas un LM a partir du fichier actif.
HISTORICAL_ASSET_LAYOUT = {
    "kik": {
        "lm_bin": FT_ROOT / "kenlm_models_v4/kik.binary",
        "uni_file": FT_ROOT / "kenlm_models_v4/kik_unigrams.txt",
    },
    "kln": {
        "lm_bin": FT_ROOT / "kenlm_models_v5/kln.binary",
        "uni_file": FT_ROOT / "kenlm_models_v5/kln_unigrams.txt",
    },
    "luo": {
        "lm_bin": FT_ROOT / "kenlm_models_v5/luo.binary",
        "uni_file": FT_ROOT / "kenlm_models_v5/luo_unigrams.txt",
    },
    "mas": {
        "lm_bin": FT_ROOT / "kenlm_models_v5/mas.binary",
        "uni_file": FT_ROOT / "kenlm_models_v5/mas_unigrams.txt",
    },
    "som": {
        "lm_bin": FT_ROOT / "kenlm_models_v5/som.binary",
        "uni_file": FT_ROOT / "kenlm_models_v5/som_unigrams.txt",
    },
    "sw": {
        "lm_bin": FT_ROOT / "kenlm_models_v4/sw.binary",
        "uni_file": FT_ROOT / "kenlm_models_v4/sw_unigrams.txt",
    },
}

for required in (
    DEV_SELECTION_PATH, TRAIN_SELECTION_PATH, HISTORICAL_LM_CONFIG, COMPACTION_REPORT
):
    assert required.exists(), required
EVAL_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_AUDIO_ROOT.mkdir(parents=True, exist_ok=True)
assert shutil.disk_usage("/content").free >= 32 * 2**30, (
    "19.5 requiert au moins 32 Gio libres dans /content pour les quatre poids "
    "et le dev audio local."
)


_compaction = json.load(open(COMPACTION_REPORT, encoding="utf-8"))
assert _compaction.get("schema") == 1, _compaction.get("schema")
assert len(_compaction.get("candidates", [])) == 2
_compact_by_name = {item["candidate"]: item for item in _compaction["candidates"]}
assert set(_compact_by_name) == {"M1", "M2"}, set(_compact_by_name)
_source_prefixes = {"M1": "REPLACE_WITH_LOCAL_RUN_ID", "M2": "REPLACE_WITH_LOCAL_RUN_ID"}
_compact_root_real = os.path.realpath(FT_ROOT / "unified_delta_merges_v3_compact")
for _name, _item in _compact_by_name.items():
    assert _item["source_duplicated_weight_sha256"].startswith(
        _source_prefixes[_name]
    ), (_name, _item["source_duplicated_weight_sha256"])
    assert _item.get("compact_flat_state_dict") is True
    assert int(_item.get("acoustic_parameter_count", -1)) == EXPECTED_ACOUSTIC_PARAMETERS
    assert _item.get("spec", {}).get("tensor_equality_verified") is True
    _step = Path(_item["checkpoint"])
    assert os.path.commonpath([
        _compact_root_real, os.path.realpath(_step)
    ]) == _compact_root_real, _step
    _manifest_path = _step / "merge_manifest.json"
    _complete_path = _step.parent / "_COMPLETE.json"
    assert _manifest_path.exists() and _complete_path.exists(), _step
    _manifest = json.load(open(_manifest_path, encoding="utf-8"))
    _complete = json.load(open(_complete_path, encoding="utf-8"))
    for _field in (
        "candidate", "checkpoint", "source_duplicated_weight_sha256",
        "output_weight_sha256", "output_bytes", "spec_sha256",
    ):
        assert _manifest[_field] == _item[_field], (_name, _field)
    assert _complete["candidate"] == _name
    assert _complete["spec_sha256"] == _manifest["spec_sha256"]
    assert _complete["output_weight_sha256"] == _manifest["output_weight_sha256"]
    assert int(_complete["output_bytes"]) == int(_manifest["output_bytes"])
    _weight_files = [path for path in (_step / "model").rglob("*") if path.is_file()]
    assert len(_weight_files) == 1, (_step, _weight_files)
    assert _weight_files[0].stat().st_size == int(_manifest["output_bytes"])

CANDIDATES = OrderedDict([
    ("joint_step5000", {
        "checkpoint": FT_ROOT / "train_output/workspace_selected/checkpoints/step_5000",
        "expected_sha256": (
            "REPLACE_WITH_LOCAL_SHA256"
        ),
        "kind": "reference",
    }),
    ("balanced_step1250", {
        "checkpoint": (
            FT_ROOT / "balanced_joint_v4/candidates/REPLACE_WITH_LOCAL_RUN_ID/step_1250"
        ),
        "kind": "reference",
    }),
    ("M1", {
        "checkpoint": Path(_compact_by_name["M1"]["checkpoint"]),
        "expected_sha256": _compact_by_name["M1"]["output_weight_sha256"],
        "kind": "merge",
    }),
    ("M2", {
        "checkpoint": Path(_compact_by_name["M2"]["checkpoint"]),
        "expected_sha256": _compact_by_name["M2"]["output_weight_sha256"],
        "kind": "merge",
    }),
])


# -----------------------------------------------------------------------------
# 1. Helpers generaux, normalisation et chargement du modele
# -----------------------------------------------------------------------------

def now_iso():
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path, block_size=16 << 20):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()


def canonical_sha256(value):
    raw = json.dumps(
        value, sort_keys=True, ensure_ascii=False, separators=(",", ":"), default=str
    ).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()


def atomic_json(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".tmp-{os.getpid()}")
    with open(temporary, "w", encoding="utf-8") as handle:
        json.dump(value, handle, ensure_ascii=False, indent=2, default=str)
    os.replace(temporary, path)


def retry_io(action, label, attempts=8):
    delay = 2.0
    last_error = None
    for attempt in range(1, attempts + 1):
        try:
            return action()
        except (OSError, IOError) as exc:
            last_error = exc
            if attempt == attempts:
                break
            print(
                f"Drive I/O temporaire ({label}) : essai {attempt}/{attempts}, "
                f"reprise dans {delay:.1f}s"
            )
            time.sleep(delay)
            delay = min(30.0, delay * 1.7)
    raise RuntimeError(f"Drive illisible apres {attempts} essais : {label}") from last_error


def frame_sha256(frame, sort_key="sample_key"):
    ordered = frame.copy()
    columns = sorted(ordered.columns)
    ordered = ordered[columns]
    if sort_key in ordered.columns:
        ordered = ordered.sort_values(sort_key, kind="mergesort")
    ordered = ordered.reset_index(drop=True)
    values = pd.util.hash_pandas_object(
        ordered.fillna("").astype(str), index=False
    ).values.tobytes()
    schema = json.dumps(columns, ensure_ascii=False).encode("utf-8")
    return hashlib.sha256(schema + values).hexdigest()


def weight_file(checkpoint):
    checkpoint = Path(checkpoint)
    assert checkpoint.exists(), checkpoint
    if checkpoint.is_file():
        return checkpoint
    files = sorted(path for path in (checkpoint / "model").rglob("*") if path.is_file())
    assert len(files) == 1, (checkpoint, files)
    return files[0]


def declared_candidate_hash(name, config):
    if config.get("expected_sha256"):
        return config["expected_sha256"]
    checkpoint = Path(config["checkpoint"])
    standard = checkpoint / "candidate_manifest.json"
    merge = checkpoint / "merge_manifest.json"
    if standard.exists():
        payload = json.load(open(standard, encoding="utf-8"))
        return payload["weight_sha256"]
    if merge.exists():
        payload = json.load(open(merge, encoding="utf-8"))
        return payload["output_weight_sha256"]
    raise AssertionError(f"Manifest de poids absent pour {name}: {checkpoint}")


def candidate_manifest_path(config):
    checkpoint = Path(config["checkpoint"])
    for filename in ("candidate_manifest.json", "merge_manifest.json"):
        path = checkpoint / filename
        if path.exists():
            return path
    return None


def stage_checkpoint(name, checkpoint, expected_sha256):
    source = weight_file(checkpoint)
    local_step = LOCAL_CHECKPOINT_ROOT / name / "step_0"
    destination = local_step / "model/pp_00/tp_00/sdp_00.pt"
    destination.parent.mkdir(parents=True, exist_ok=True)
    if not destination.exists() or destination.stat().st_size != source.stat().st_size:
        if destination.exists():
            destination.unlink()
        print(f"Copie locale du checkpoint {name} ({source.stat().st_size / 2**30:.2f} Gio)...")
        subprocess.run([
            "rsync", "-a", "--partial", "--append-verify", "--info=progress2",
            str(source), str(destination),
        ], check=True)
    print("Verification SHA256 de", name, "...")
    actual = sha256_file(destination)
    if actual != expected_sha256:
        # Une ancienne copie locale de meme taille peut provenir d'un essai different.
        print("Copie locale invalide : recopie integrale de", name)
        destination.unlink()
        subprocess.run([
            "rsync", "-a", "--partial", "--append-verify", "--info=progress2",
            str(source), str(destination),
        ], check=True)
        actual = sha256_file(destination)
    assert actual == expected_sha256, (name, actual, expected_sha256)
    return local_step, destination, actual


_PUNCT_RE = re.compile(r"[^\w\s'’\-]", re.UNICODE)
_WS_RE = re.compile(r"\s+")
_MOJIBAKE_MARKERS = set("ÃÂÅâðÐÑ¤¦¨©¬®¯°±²³´µ¶·¸¹º»¼½¾¿")
_KIKUYU_EXPECTED = set("ĩũĨŨ")


def _text_badness(value, language=None):
    value = str(value)
    score = 50 * value.count("�")
    score += 6 * sum(char in _MOJIBAKE_MARKERS for char in value)
    score += 10 * sum(ord(char) < 32 and char not in "\n\t\r" for char in value)
    if language == "kik":
        score -= 2 * sum(char in _KIKUYU_EXPECTED for char in value)
    return score


def _repair_mojibake(value, language=None):
    raw = str(value or "")
    candidates = [(raw, "none")]
    if any(char in _MOJIBAKE_MARKERS for char in raw) or "�" in raw:
        for source_encoding in ("latin-1", "cp1252"):
            try:
                candidate = raw.encode(source_encoding).decode("utf-8")
            except (UnicodeEncodeError, UnicodeDecodeError):
                continue
            candidates.append((candidate, f"{source_encoding}->utf8"))
    best, method = min(candidates, key=lambda item: _text_badness(item[0], language))
    if _text_badness(best, language) >= _text_badness(raw, language):
        return raw, "none"
    return best, method


def normalize_eval_text(value, language=None):
    repaired, _ = _repair_mojibake(value, language)
    repaired = unicodedata.normalize("NFC", repaired).lower()
    repaired = _PUNCT_RE.sub(" ", repaired).replace("_", " ")
    return _WS_RE.sub(" ", repaired).strip()


def load_eval_pipeline(checkpoint, dtype=torch.bfloat16):
    from fairseq2.data.tokenizers.hub import load_tokenizer
    from fairseq2.models.hub import load_model
    from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline

    model_card = FT_CONFIG["model_card"]
    model = load_model(model_card, device=torch.device("cuda"), dtype=dtype)
    source = weight_file(checkpoint)
    raw = torch.load(str(source), map_location="cpu", weights_only=False)
    state = raw.get("model", raw) if isinstance(raw, dict) else raw
    assert isinstance(state, dict) and state
    model.load_state_dict(state)
    del raw, state
    model.eval()
    tokenizer = load_tokenizer(model_card)
    return ASRInferencePipeline(None, model=model, tokenizer=tokenizer)


def labels_from_pipeline(pipe):
    tokenizer_model = getattr(pipe.tokenizer, "_model", None)
    for method in ("index_to_token", "id_to_piece", "IdToPiece"):
        if tokenizer_model is not None and hasattr(tokenizer_model, method):
            try:
                labels = [
                    str(getattr(tokenizer_model, method)(index))
                    for index in range(pipe.tokenizer.vocab_info.size)
                ]
                assert len(labels) == len(set(labels))
                labels[0] = ""
                return labels
            except Exception:
                pass
    raise RuntimeError("Vocabulaire du tokenizer inaccessible")


def capture_one(pipe, audio_input, omni_code, language):
    captured = []
    original_forward = pipe.model.forward

    def spy(source_seqs, source_seq_lens, *args, **kwargs):
        output = original_forward(source_seqs, source_seq_lens, *args, **kwargs)
        assert (
            isinstance(output, tuple)
            and len(output) > 1
            and hasattr(output[1], "seq_lens")
        ), "La sortie CTC ne fournit pas output_layout.seq_lens"
        logits, output_layout = output[0], output[1]
        assert torch.is_tensor(logits), type(logits)
        assert logits.shape[0] == 1, logits.shape
        assert len(output_layout.seq_lens) == 1
        length = output_layout.seq_lens[0]
        length = int(length.item() if hasattr(length, "item") else length)
        assert 0 < length <= logits.shape[1], (length, logits.shape)
        log_probs = torch.log_softmax(
            logits[0, :length].detach().float(), dim=-1
        ).cpu()
        assert log_probs.shape[1] == len(LABELS), (log_probs.shape, len(LABELS))
        assert bool(torch.isfinite(log_probs).all()), "Log-probabilites CTC non finies"
        captured.append(log_probs.numpy())
        return output

    pipe.model.forward = spy
    try:
        output = pipe.transcribe([audio_input], lang=[omni_code], batch_size=1)
    finally:
        pipe.model.forward = original_forward
    assert len(output) == 1 and len(captured) == 1, (len(output), len(captured))
    return normalize_eval_text(output[0], language), captured[0]


def edit_counts(reference, hypothesis):
    result = jiwer.process_words(reference, hypothesis)
    return {
        "hits": int(result.hits),
        "substitutions": int(result.substitutions),
        "deletions": int(result.deletions),
        "insertions": int(result.insertions),
    }


# -----------------------------------------------------------------------------
# 2. Gel et verification des quatre checkpoints
# -----------------------------------------------------------------------------

candidate_meta = {}
print("=== CHECKPOINTS ===")
for candidate_name, candidate_config in CANDIDATES.items():
    checkpoint = Path(candidate_config["checkpoint"])
    expected = declared_candidate_hash(candidate_name, candidate_config)
    local_step, local_weight, actual = stage_checkpoint(
        candidate_name, checkpoint, expected
    )
    manifest_path = candidate_manifest_path(candidate_config)
    candidate_meta[candidate_name] = {
        "name": candidate_name,
        "kind": candidate_config["kind"],
        "source_checkpoint": str(checkpoint),
        "source_weight": str(weight_file(checkpoint)),
        "local_checkpoint": str(local_step),
        "local_weight": str(local_weight),
        "weight_sha256": actual,
        "weight_bytes": int(local_weight.stat().st_size),
        "manifest_path": str(manifest_path) if manifest_path else None,
        "manifest_sha256": sha256_file(manifest_path) if manifest_path else None,
    }
    print(candidate_name, "->", actual[:16])

assert len({item["weight_sha256"] for item in candidate_meta.values()}) == 4, (
    "Deux candidats ont exactement les memes poids"
)


# -----------------------------------------------------------------------------
# 3. Dev V4 fige et domaines de decodage
# -----------------------------------------------------------------------------

dev = pd.read_parquet(DEV_SELECTION_PATH)
required_columns = {
    "sample_key", "language", "split", "speech_method", "adaptation_role",
    "source_kind", "audio_ref", "parquet_path", "row_index", "text", "seconds",
}
missing_columns = required_columns - set(dev.columns)
assert not missing_columns, missing_columns
# Hash de l'artefact exactement tel qu'il est stocke, avant toute colonne temporaire/cast.
dev_selection_sha256 = frame_sha256(dev)
# Ne jamais modifier silencieusement la cohorte signee : toute ligne invalide bloque.
assert dev["split"].eq("dev").all(), dev["split"].value_counts().to_dict()
assert dev["language"].isin(LANGUAGES).all()
assert set(dev["language"]) == set(LANGUAGES)
if "usable" in dev.columns:
    assert dev["usable"].astype(bool).all(), "Le dev fige contient une ligne non utilisable"
seconds_numeric = pd.to_numeric(dev["seconds"], errors="coerce")
assert seconds_numeric.between(0.5, 38.0).all(), (
    "Le dev fige contient une duree hors [0.5, 38] secondes"
)
dev = dev.copy()
dev["seconds"] = seconds_numeric
assert dev["sample_key"].is_unique

train_keys = set(pd.read_parquet(
    TRAIN_SELECTION_PATH, columns=["sample_key"]
)["sample_key"].astype(str))
assert set(dev["sample_key"].astype(str)).isdisjoint(train_keys)

dev["reference"] = [
    normalize_eval_text(text, language)
    for text, language in zip(dev["text"].astype(str), dev["language"].astype(str))
]
assert dev["reference"].str.len().gt(0).all()


def canonical_domain(row):
    method = str(row["speech_method"]).strip().lower().replace("_", "-")
    if "unscript" in method or "spont" in method:
        domain = "unscripted"
    elif "script" in method or "read" in method:
        domain = "scripted"
    elif str(row["language"]) == "sw":
        # Le manifest sw historique portait 'unknown', alors que la source officielle
        # et tout le test sw sont spontanes/unscripted.
        domain = "unscripted"
    else:
        raise ValueError(
            "Domaine non resolu : "
            f"{row['sample_key']} / {row['language']} / {row['speech_method']} / "
            f"{row['adaptation_role']}"
        )
    if str(row["adaptation_role"]).strip().lower() == "new_in_domain":
        assert domain == "unscripted", (
            "Invariant V4 rompu : new_in_domain n'est pas unscripted", row["sample_key"]
        )
    return domain


dev["decode_domain"] = [canonical_domain(row) for row in dev.to_dict("records")]

# Si le _COMPLETE du build a ete restaure/copie, son hash signe doit correspondre.
build_meta_candidates = [
    Path("/content/afrivoices_balanced_v4/dataset/_COMPLETE.json"),
    Path("/content/afrivoices_project/"
         "afrivoices_balanced_v4/dataset/_COMPLETE.json"),
    Path("/content/afrivoices_project/"
         "afrivoices_balanced_v4/_COMPLETE.json"),
]
build_meta_path = next((path for path in build_meta_candidates if path.exists()), None)
if build_meta_path is not None:
    build_meta = json.load(open(build_meta_path, encoding="utf-8"))
    assert build_meta["dev_selection_sha256"] == dev_selection_sha256, (
        build_meta["dev_selection_sha256"], dev_selection_sha256
    )
    print("Hash du dev confirme par :", build_meta_path)
else:
    print("Note : _COMPLETE du dataset non restaure ; le contrat 19.5 fige le SHA du dev.")

hours = dev.groupby("language")["seconds"].sum().div(3600)
assert hours.between(1.90, 2.10).all(), hours.to_dict()
print("\n=== DEV FIGE ===")
print("clips :", len(dev), "| SHA :", dev_selection_sha256)
display(dev.groupby(["language", "decode_domain"]).agg(
    clips=("sample_key", "size"),
    hours=("seconds", lambda values: float(values.sum()) / 3600),
    reference_words=("reference", lambda values: sum(len(value.split()) for value in values)),
).reset_index())


# Les memes 2 795 clips sont lus quatre fois. Les materialiser une seule fois sur le disque
# local evite les lectures Drive FUSE repetees pendant plusieurs heures d'inference.
def _audio_destination(sample_key, language):
    digest = hashlib.sha256(str(sample_key).encode("utf-8")).hexdigest()[:24]
    return LOCAL_AUDIO_ROOT / language / f"{digest}.flac"


def _publish_audio_bytes(payload, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and destination.stat().st_size > 0:
        return
    temporary = destination.with_name(destination.name + f".tmp-{os.getpid()}")
    with open(temporary, "wb") as handle:
        handle.write(payload)
    os.replace(temporary, destination)


print("\nMaterialisation unique du dev sur disque local...")
dev["eval_audio_path"] = ""
staged = 0

# Fichiers cures : lecture/copie une fois seulement.
file_indices = dev.index[dev["source_kind"].eq("file")].tolist()
for index in file_indices:
    row = dev.loc[index]
    source = Path(str(row["audio_ref"]))
    retry_io(lambda: source.stat(), str(source))
    destination = _audio_destination(row["sample_key"], row["language"])
    if not destination.exists() or destination.stat().st_size <= 0:
        temporary = destination.with_name(destination.name + f".tmp-{os.getpid()}")
        destination.parent.mkdir(parents=True, exist_ok=True)
        retry_io(lambda: shutil.copyfile(source, temporary), str(source))
        os.replace(temporary, destination)
    dev.at[index, "eval_audio_path"] = str(destination)
    staged += 1
    if staged % 250 == 0:
        print(f"  {staged}/{len(dev)} clips locaux")

# Caches spontanes : chaque parquet source n'est ouvert qu'une fois.
parquet_rows = dev[dev["source_kind"].ne("file")]
for parquet_path, group in parquet_rows.groupby("parquet_path", sort=True):
    parquet_path = str(parquet_path)
    retry_io(lambda: os.stat(parquet_path), parquet_path)
    source_rows = retry_io(
        lambda: pq.read_table(
            parquet_path, columns=["audio_bytes", "audio_size"]
        ).to_pylist(),
        parquet_path,
    )
    for index, row in group.iterrows():
        source = source_rows[int(row["row_index"])]
        destination = _audio_destination(row["sample_key"], row["language"])
        _publish_audio_bytes(bytes(source["audio_bytes"]), destination)
        dev.at[index, "eval_audio_path"] = str(destination)
        staged += 1
        if staged % 250 == 0:
            print(f"  {staged}/{len(dev)} clips locaux")

assert staged == len(dev)
assert dev["eval_audio_path"].map(os.path.exists).all()
local_audio_bytes = int(sum(
    Path(path).stat().st_size for path in dev["eval_audio_path"].astype(str).unique()
))
for path in dev["eval_audio_path"].iloc[::max(1, len(dev)//50)]:
    info = sf.info(path)
    assert info.samplerate == 16000 and info.channels == 1, (path, info)
print(
    "Dev local pret :", len(dev), "clips |",
    round(local_audio_bytes / 2**30, 2), "Gio"
)
print("Empreinte SHA256 des audios locaux...")
audio_fingerprint = hashlib.sha256()
for row in dev.sort_values("sample_key", kind="mergesort").itertuples():
    path = Path(row.eval_audio_path)
    digest = sha256_file(path)
    audio_fingerprint.update(str(row.sample_key).encode("utf-8"))
    audio_fingerprint.update(b"\0")
    audio_fingerprint.update(digest.encode("ascii"))
    audio_fingerprint.update(b"\0")
    audio_fingerprint.update(str(path.stat().st_size).encode("ascii"))
    audio_fingerprint.update(b"\n")
audio_fingerprint_sha256 = audio_fingerprint.hexdigest()
print("Empreinte audio :", audio_fingerprint_sha256)


# -----------------------------------------------------------------------------
# 4. Configuration KenLM historique gelee et contrat de cache
# -----------------------------------------------------------------------------

historical_config_sha256 = sha256_file(HISTORICAL_LM_CONFIG)
historical_configs = json.load(open(HISTORICAL_LM_CONFIG, encoding="utf-8"))

# Le SHA brut precedemment note (139be...) a change : cela peut venir d'une re-ecriture JSON,
# mais on ne l'accepte pas aveuglement. On exige ici les 12 decisions semantiques exactes de
# la configuration qui avait produit 0.36529. Les champs additionnels, non consommes, sont ignores.
EXPECTED_LM_PROFILE = {
    "kik": {"lm": "v4", "alpha": 0.50, "beta": 0.0, "beam": 100},
    "kln": {"lm": "v5", "alpha": 0.50, "beta": 0.0, "beam": 250},
    "luo": {"lm": "v5", "alpha": 0.65, "beta": 1.0, "beam": 100},
    "mas": {"lm": "v5", "alpha": 0.65, "beta": 2.0, "beam": 100},
    "som": {"lm": "v5", "alpha": 0.65, "beta": 0.0, "beam": 100},
    "sw":  {"lm": "v4", "alpha": 0.30, "beta": 0.0, "beam": 100},
}
expected_config_groups = {
    f"{language}|{domain}"
    for language in LANGUAGES for domain in ("scripted", "unscripted")
}
assert set(historical_configs) == expected_config_groups, (
    "Groupes KenLM historiques modifies",
    sorted(set(historical_configs) - expected_config_groups),
    sorted(expected_config_groups - set(historical_configs)),
)


def _strict_number(value, key, field):
    assert not isinstance(value, (bool, np.bool_)), (key, field, value)
    number = float(value)
    assert np.isfinite(number), (key, field, value)
    return number


historical_semantics = {}
for language in LANGUAGES:
    expected = EXPECTED_LM_PROFILE[language]
    frozen_lm_path = Path(HISTORICAL_ASSET_LAYOUT[language]["lm_bin"])
    for domain in ("scripted", "unscripted"):
        key = f"{language}|{domain}"
        config = historical_configs[key]
        required_fields = {"lm", "lm_bin", "alpha", "beta", "beam"}
        assert required_fields <= set(config), (key, required_fields - set(config))
        lm_name = str(config["lm"]).strip().lower()
        alpha = _strict_number(config["alpha"], key, "alpha")
        beta = _strict_number(config["beta"], key, "beta")
        beam_number = _strict_number(config["beam"], key, "beam")
        assert beam_number.is_integer(), (key, "beam", beam_number)
        beam = int(beam_number)
        configured_path = Path(str(config["lm_bin"]))
        assert os.path.realpath(configured_path) == os.path.realpath(frozen_lm_path), (
            key, configured_path, frozen_lm_path
        )
        assert lm_name == expected["lm"], (key, lm_name, expected["lm"])
        assert alpha == expected["alpha"], (key, alpha, expected["alpha"])
        assert beta == expected["beta"], (key, beta, expected["beta"])
        assert beam == expected["beam"], (key, beam, expected["beam"])
        historical_semantics[key] = {
            "lm": lm_name,
            "lm_bin": str(frozen_lm_path),
            "alpha": alpha,
            "beta": beta,
            "beam": beam,
        }

historical_semantic_sha256 = canonical_sha256(historical_semantics)
print(
    "Configuration KenLM semantiquement historique confirmee | SHA brut :",
    historical_config_sha256,
)
if historical_config_sha256 != PREVIOUSLY_RECORDED_HISTORICAL_CONFIG_SHA256:
    print(
        "Note : serialisation JSON differente de l'ancien SHA",
        PREVIOUSLY_RECORDED_HISTORICAL_CONFIG_SHA256,
        "| SHA semantique :", historical_semantic_sha256,
    )


def resolve_lm_assets(language, domain):
    key = f"{language}|{domain}"
    assert key in historical_configs, key
    semantic = historical_semantics[key]
    frozen = HISTORICAL_ASSET_LAYOUT[language]
    lm_bin = Path(frozen["lm_bin"])
    unigram_file = Path(frozen["uni_file"])
    configured_lm = Path(semantic["lm_bin"])
    assert os.path.realpath(configured_lm) == os.path.realpath(lm_bin), (
        key, configured_lm, lm_bin
    )
    assert unigram_file.exists(), (key, unigram_file)
    assert lm_bin.exists(), (key, lm_bin)
    return {
        "group": key,
        "lm_name": semantic["lm"],
        "lm_bin": str(lm_bin.resolve()),
        "lm_sha256": sha256_file(lm_bin),
        "lm_bytes": int(lm_bin.stat().st_size),
        "uni_file": str(unigram_file.resolve()),
        "uni_sha256": sha256_file(unigram_file),
        "uni_bytes": int(unigram_file.stat().st_size),
        "alpha": semantic["alpha"],
        "beta": semantic["beta"],
        "beam": semantic["beam"],
    }


active_groups = sorted({
    f"{row.language}|{row.decode_domain}"
    for row in dev[["language", "decode_domain"]].drop_duplicates().itertuples()
})
lm_assets = {}
print("\nHash des LM historiques...")
for group in active_groups:
    language, domain = group.split("|", 1)
    lm_assets[group] = resolve_lm_assets(language, domain)
    print(group, "->", lm_assets[group]["lm_sha256"][:16],
          f"beam={lm_assets[group]['beam']}")


def package_version(name):
    try:
        return importlib.metadata.version(name)
    except Exception:
        return "unknown"


# Les labels du tokenizer ne dependent pas du checkpoint ; on les extrait sans conserver
# de pipeline supplementaire en VRAM.
print("\nExtraction du vocabulaire de reference...")
_probe_name = next(iter(candidate_meta))
_probe_pipe = load_eval_pipeline(candidate_meta[_probe_name]["local_checkpoint"])
LABELS = labels_from_pipeline(_probe_pipe)
labels_sha256 = hashlib.sha256("\0".join(LABELS).encode("utf-8")).hexdigest()
probe_params = sum(parameter.numel() for parameter in _probe_pipe.model.parameters())
assert probe_params == EXPECTED_ACOUSTIC_PARAMETERS, probe_params
del _probe_pipe
gc.collect()
torch.cuda.empty_cache()

contract = {
    "schema": 2,
    "purpose": "single-checkpoint delta-merge comparison",
    "created_from": "cell_19_5_evaluate_unified_candidates.py",
    "candidates": candidate_meta,
    "dev_selection_path": str(DEV_SELECTION_PATH),
    "dev_selection_sha256": dev_selection_sha256,
    "dev_clips": int(len(dev)),
    "dev_audio_fingerprint_sha256": audio_fingerprint_sha256,
    "dev_audio_bytes": local_audio_bytes,
    "historical_lm_config": str(HISTORICAL_LM_CONFIG),
    "historical_lm_config_raw_sha256": historical_config_sha256,
    "historical_lm_config_previously_recorded_raw_sha256": (
        PREVIOUSLY_RECORDED_HISTORICAL_CONFIG_SHA256
    ),
    "historical_lm_config_semantic_sha256": historical_semantic_sha256,
    "historical_lm_semantics": historical_semantics,
    "lm_assets": lm_assets,
    "labels_sha256": labels_sha256,
    "omni_language_codes": OMNI_CODES,
    "normalization_version": NORMALIZATION_VERSION,
    "dtype": "torch.bfloat16",
    "batch_size": 1,
    "part_size": PART_SIZE,
    "metric": "per-language corpus WER then unweighted mean of six languages",
    "versions": {
        "python": sys.version,
        "torch": torch.__version__,
        "jiwer": package_version("jiwer"),
        "pyctcdecode": package_version("pyctcdecode"),
        "kenlm": package_version("kenlm"),
    },
}
contract_sha256 = canonical_sha256(contract)
run_root = EVAL_ROOT / contract_sha256[:16]
cache_root = run_root / "cache"
cache_root.mkdir(parents=True, exist_ok=True)
contract_path = run_root / "contract.json"
if contract_path.exists():
    existing_contract = json.load(open(contract_path, encoding="utf-8"))
    assert canonical_sha256(existing_contract) == contract_sha256
else:
    atomic_json(contract, contract_path)
print("Contrat 19.5 :", contract_sha256)


# -----------------------------------------------------------------------------
# 5. Entrees audio, decodeur paresseux et cache atomique
# -----------------------------------------------------------------------------

def audio_input(row):
    path = str(row["eval_audio_path"])
    assert os.path.exists(path), path
    return path


def make_decoder(group):
    asset = lm_assets[group]
    unigrams = open(asset["uni_file"], encoding="utf-8").read().splitlines()
    decoder = build_ctcdecoder(
        LABELS,
        asset["lm_bin"],
        unigrams=unigrams,
        alpha=asset["alpha"],
        beta=asset["beta"],
    )
    assert hasattr(decoder, "reset_params")
    decoder.reset_params(alpha=asset["alpha"], beta=asset["beta"])
    return decoder


def atomic_parquet(records, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".tmp-{os.getpid()}")
    table = pa.Table.from_pylist(records)
    pq.write_table(table, temporary, compression="zstd", compression_level=3)
    os.replace(temporary, path)


def existing_group_rows(group_dir, expected_frame, candidate_name, candidate_sha256,
                        language, domain):
    expected_keys = set(expected_frame["sample_key"].astype(str))
    expected_references = dict(zip(
        expected_frame["sample_key"].astype(str),
        expected_frame["reference"].astype(str),
    ))
    frames = []
    for path in sorted(group_dir.glob("part-*.parquet")):
        frame = pd.read_parquet(path)
        required = {
            "contract_sha256", "candidate", "candidate_sha256", "sample_key",
            "language", "decode_domain", "reference", "greedy_hypothesis",
            "lm_hypothesis", "lm_hits", "lm_substitutions", "lm_deletions",
            "lm_insertions",
        }
        assert required <= set(frame.columns), (path, required - set(frame.columns))
        assert frame["contract_sha256"].eq(contract_sha256).all(), path
        assert frame["candidate"].eq(candidate_name).all(), path
        assert frame["candidate_sha256"].eq(candidate_sha256).all(), path
        assert frame["language"].eq(language).all(), path
        assert frame["decode_domain"].eq(domain).all(), path
        frames.append(frame)
    if not frames:
        return pd.DataFrame(), set(), 0
    combined = pd.concat(frames, ignore_index=True)
    assert combined["sample_key"].is_unique
    done = set(combined["sample_key"].astype(str))
    assert done.issubset(expected_keys)
    for row in combined[["sample_key", "reference"]].itertuples(index=False):
        assert str(row.reference) == expected_references[str(row.sample_key)]
    next_part = 1 + max(
        int(path.stem.split("-")[-1]) for path in group_dir.glob("part-*.parquet")
    )
    return combined, done, next_part


def flush_buffer(buffer, group_dir, part_number):
    if not buffer:
        return part_number
    destination = group_dir / f"part-{part_number:05d}.parquet"
    assert not destination.exists(), destination
    atomic_parquet(buffer, destination)
    print("    cache ->", destination.name, "|", len(buffer), "clips")
    buffer.clear()
    return part_number + 1


# -----------------------------------------------------------------------------
# 6. Evaluation reprenable des quatre candidats
# -----------------------------------------------------------------------------

started_at = time.time()
for candidate_index, (candidate_name, metadata) in enumerate(candidate_meta.items(), 1):
    print("\n" + "=" * 78)
    print(f"CANDIDAT {candidate_index}/{len(candidate_meta)} : {candidate_name}")
    pipe = load_eval_pipeline(metadata["local_checkpoint"])
    actual_parameters = sum(parameter.numel() for parameter in pipe.model.parameters())
    assert actual_parameters == EXPECTED_ACOUSTIC_PARAMETERS, (
        candidate_name, actual_parameters
    )
    candidate_labels = labels_from_pipeline(pipe)
    candidate_labels_sha = hashlib.sha256(
        "\0".join(candidate_labels).encode("utf-8")
    ).hexdigest()
    assert candidate_labels_sha == labels_sha256
    del candidate_labels

    for group in active_groups:
        language, domain = group.split("|", 1)
        group_frame = dev[
            dev["language"].eq(language) & dev["decode_domain"].eq(domain)
        ].copy()
        group_frame = group_frame.sort_values(
            ["source_kind", "parquet_path", "row_index", "sample_key"],
            kind="mergesort",
        )
        expected_keys = set(group_frame["sample_key"].astype(str))
        group_dir = cache_root / candidate_name / f"{language}__{domain}"
        group_dir.mkdir(parents=True, exist_ok=True)
        _, done_keys, part_number = existing_group_rows(
            group_dir, group_frame, candidate_name, metadata["weight_sha256"],
            language, domain,
        )
        missing = group_frame[
            ~group_frame["sample_key"].astype(str).isin(done_keys)
        ]
        if missing.empty:
            print(f"[cache complet] {candidate_name} | {group} | {len(done_keys)} clips")
            continue

        print(f"[{candidate_name}] {group} | {len(done_keys)}/{len(group_frame)} deja faits")
        decoder = make_decoder(group)
        asset = lm_assets[group]
        buffer = []
        group_start = time.time()
        for clip_index, row in enumerate(missing.to_dict("records"), 1):
            reference = str(row["reference"])
            greedy_hypothesis, log_probs = capture_one(
                pipe, audio_input(row), OMNI_CODES[language], language
            )
            lm_hypothesis = normalize_eval_text(
                decoder.decode(log_probs, beam_width=asset["beam"]), language
            )
            greedy_counts = edit_counts(reference, greedy_hypothesis)
            lm_counts = edit_counts(reference, lm_hypothesis)
            buffer.append({
                "contract_sha256": contract_sha256,
                "candidate": candidate_name,
                "candidate_sha256": metadata["weight_sha256"],
                "sample_key": str(row["sample_key"]),
                "language": language,
                "decode_domain": domain,
                "adaptation_role": str(row["adaptation_role"]),
                "seconds": float(row["seconds"]),
                "reference": reference,
                "greedy_hypothesis": greedy_hypothesis,
                "lm_hypothesis": lm_hypothesis,
                "greedy_hits": greedy_counts["hits"],
                "greedy_substitutions": greedy_counts["substitutions"],
                "greedy_deletions": greedy_counts["deletions"],
                "greedy_insertions": greedy_counts["insertions"],
                "lm_hits": lm_counts["hits"],
                "lm_substitutions": lm_counts["substitutions"],
                "lm_deletions": lm_counts["deletions"],
                "lm_insertions": lm_counts["insertions"],
            })
            del log_probs
            if len(buffer) >= PART_SIZE:
                part_number = flush_buffer(buffer, group_dir, part_number)
            if clip_index % 25 == 0 or clip_index == len(missing):
                elapsed = max(time.time() - group_start, 1e-9)
                rate = clip_index / elapsed
                print(
                    f"    {clip_index}/{len(missing)} nouveaux clips | "
                    f"{rate:.2f} clip/s"
                )
        part_number = flush_buffer(buffer, group_dir, part_number)
        del decoder
        gc.collect()
        torch.cuda.empty_cache()

    del pipe
    gc.collect()
    torch.cuda.empty_cache()
    print(
        f"Candidat {candidate_name} termine | temps total cellule : "
        f"{(time.time() - started_at) / 60:.1f} min"
    )


# -----------------------------------------------------------------------------
# 7. Agregation exacte, bootstrap apparie et recommandation (sans mutation)
# -----------------------------------------------------------------------------

all_frames = []
expected_all_keys = set(dev["sample_key"].astype(str))
for candidate_name, metadata in candidate_meta.items():
    parts = sorted((cache_root / candidate_name).glob("**/part-*.parquet"))
    assert parts, candidate_name
    frame = pd.concat([pd.read_parquet(path) for path in parts], ignore_index=True)
    assert frame["sample_key"].is_unique, candidate_name
    assert set(frame["sample_key"].astype(str)) == expected_all_keys, (
        candidate_name,
        len(frame),
        len(expected_all_keys),
    )
    assert frame["contract_sha256"].eq(contract_sha256).all()
    assert frame["candidate_sha256"].eq(metadata["weight_sha256"]).all()
    all_frames.append(frame)
results = pd.concat(all_frames, ignore_index=True)


def aggregate_wer(frame, prefix):
    substitutions = int(frame[f"{prefix}_substitutions"].sum())
    deletions = int(frame[f"{prefix}_deletions"].sum())
    insertions = int(frame[f"{prefix}_insertions"].sum())
    hits = int(frame[f"{prefix}_hits"].sum())
    reference_words = hits + substitutions + deletions
    assert reference_words > 0
    return {
        "wer": (substitutions + deletions + insertions) / reference_words,
        "hits": hits,
        "substitutions": substitutions,
        "deletions": deletions,
        "insertions": insertions,
        "reference_words": reference_words,
    }


language_rows = []
domain_rows = []
candidate_summary = {}
for candidate_name in candidate_meta:
    candidate_frame = results[results["candidate"].eq(candidate_name)]
    greedy_language, lm_language = {}, {}
    for language in LANGUAGES:
        language_frame = candidate_frame[candidate_frame["language"].eq(language)]
        greedy = aggregate_wer(language_frame, "greedy")
        lm = aggregate_wer(language_frame, "lm")
        greedy_language[language] = greedy["wer"]
        lm_language[language] = lm["wer"]
        language_rows.append({
            "candidate": candidate_name,
            "language": language,
            "clips": int(len(language_frame)),
            "hours": float(language_frame["seconds"].sum()) / 3600,
            "greedy_wer": greedy["wer"],
            "lm_wer": lm["wer"],
            "lm_reference_words": lm["reference_words"],
            "lm_substitutions": lm["substitutions"],
            "lm_deletions": lm["deletions"],
            "lm_insertions": lm["insertions"],
        })
        for domain in sorted(language_frame["decode_domain"].unique()):
            domain_frame = language_frame[
                language_frame["decode_domain"].eq(domain)
            ]
            domain_rows.append({
                "candidate": candidate_name,
                "language": language,
                "domain": domain,
                "clips": int(len(domain_frame)),
                "greedy_wer": aggregate_wer(domain_frame, "greedy")["wer"],
                "lm_wer": aggregate_wer(domain_frame, "lm")["wer"],
            })
    candidate_summary[candidate_name] = {
        **candidate_meta[candidate_name],
        "actual_acoustic_parameters": EXPECTED_ACOUSTIC_PARAMETERS,
        "greedy_language_wer": greedy_language,
        "lm_language_wer": lm_language,
        "greedy_macro_wer": float(np.mean([greedy_language[x] for x in LANGUAGES])),
        "lm_macro_wer": float(np.mean([lm_language[x] for x in LANGUAGES])),
    }

reference_name = min(
    ("joint_step5000", "balanced_step1250"),
    key=lambda name: candidate_summary[name]["lm_macro_wer"],
)


def paired_bootstrap(candidate_name, reference_candidate, repetitions, seed):
    rng = np.random.default_rng(seed)
    deltas = np.empty(repetitions, dtype=np.float64)
    candidate_frame = results[results["candidate"].eq(candidate_name)].set_index("sample_key")
    reference_frame = results[results["candidate"].eq(reference_candidate)].set_index("sample_key")
    assert set(candidate_frame.index) == set(reference_frame.index)

    language_arrays = {}
    for language in LANGUAGES:
        keys = sorted(
            candidate_frame[candidate_frame["language"].eq(language)].index.astype(str)
        )
        candidate_part = candidate_frame.loc[keys]
        reference_part = reference_frame.loc[keys]
        candidate_errors = (
            candidate_part["lm_substitutions"].to_numpy(np.int64)
            + candidate_part["lm_deletions"].to_numpy(np.int64)
            + candidate_part["lm_insertions"].to_numpy(np.int64)
        )
        reference_errors = (
            reference_part["lm_substitutions"].to_numpy(np.int64)
            + reference_part["lm_deletions"].to_numpy(np.int64)
            + reference_part["lm_insertions"].to_numpy(np.int64)
        )
        candidate_words = (
            candidate_part["lm_hits"].to_numpy(np.int64)
            + candidate_part["lm_substitutions"].to_numpy(np.int64)
            + candidate_part["lm_deletions"].to_numpy(np.int64)
        )
        reference_words = (
            reference_part["lm_hits"].to_numpy(np.int64)
            + reference_part["lm_substitutions"].to_numpy(np.int64)
            + reference_part["lm_deletions"].to_numpy(np.int64)
        )
        assert np.array_equal(candidate_words, reference_words)
        language_arrays[language] = (
            candidate_errors, reference_errors, candidate_words
        )

    for repetition in range(repetitions):
        language_deltas = []
        for language in LANGUAGES:
            candidate_errors, reference_errors, words = language_arrays[language]
            indices = rng.integers(0, len(words), size=len(words))
            denominator = words[indices].sum()
            language_deltas.append(
                candidate_errors[indices].sum() / denominator
                - reference_errors[indices].sum() / denominator
            )
        deltas[repetition] = float(np.mean(language_deltas))
    return {
        "reference": reference_candidate,
        "delta_candidate_minus_reference": float(
            candidate_summary[candidate_name]["lm_macro_wer"]
            - candidate_summary[reference_candidate]["lm_macro_wer"]
        ),
        "ci95_low": float(np.quantile(deltas, 0.025)),
        "ci95_high": float(np.quantile(deltas, 0.975)),
        "probability_candidate_better": float(np.mean(deltas < 0.0)),
        "repetitions": int(repetitions),
    }


bootstrap = {}
for offset, candidate_name in enumerate(candidate_meta):
    if candidate_name == reference_name:
        continue
    bootstrap[candidate_name] = paired_bootstrap(
        candidate_name,
        reference_name,
        BOOTSTRAP_REPETITIONS,
        BOOTSTRAP_SEED + offset,
    )

eligible_merges = []
for candidate_name in ("M1", "M2"):
    summary = candidate_summary[candidate_name]
    reference = candidate_summary[reference_name]
    language_regressions = {
        language: summary["lm_language_wer"][language]
                  - reference["lm_language_wer"][language]
        for language in LANGUAGES
        if summary["lm_language_wer"][language]
           - reference["lm_language_wer"][language] > 0.015
    }
    summary["language_regressions_over_0_015"] = language_regressions
    summary["bootstrap_vs_reference"] = bootstrap[candidate_name]
    if (
        summary["lm_macro_wer"] <= reference["lm_macro_wer"] - 0.002
        and bootstrap[candidate_name]["probability_candidate_better"] >= 0.90
        and not language_regressions
    ):
        eligible_merges.append(candidate_name)

if not eligible_merges:
    recommendation = reference_name
    recommendation_reason = "Aucun merge ne franchit les garde-fous preregistres."
else:
    eligible_merges.sort(key=lambda name: candidate_summary[name]["lm_macro_wer"])
    recommendation = eligible_merges[0]
    if len(eligible_merges) == 2:
        difference = abs(
            candidate_summary["M1"]["lm_macro_wer"]
            - candidate_summary["M2"]["lm_macro_wer"]
        )
        if difference < 0.002:
            recommendation = "M1"
            recommendation_reason = (
                "M1 et M2 sont a moins de 0.002 ; M1 est la fusion la plus prudente."
            )
        else:
            recommendation_reason = "Meilleur merge admissible selon la macro-WER LM."
    else:
        recommendation_reason = "Seul merge admissible selon les garde-fous."

report = {
    "schema": 2,
    "created_at": now_iso(),
    "contract_sha256": contract_sha256,
    "contract_path": str(contract_path),
    "metric_note": (
        "WER corpus exact par langue; macro-moyenne non ponderee des six langues. "
        "Le dev reste un proxy local, pas une estimation garantie du leaderboard."
    ),
    "candidates": candidate_summary,
    "language_results": language_rows,
    "domain_diagnostics": domain_rows,
    "reference_candidate": reference_name,
    "bootstrap": bootstrap,
    "eligible_merges": eligible_merges,
    "recommendation": recommendation,
    "recommendation_reason": recommendation_reason,
    "final_pointer_mutated": False,
    "elapsed_minutes": round((time.time() - started_at) / 60, 2),
}
report_path = run_root / "delta_merge_eval_v2.json"
atomic_json(report, report_path)
atomic_json(
    {
        "contract_sha256": contract_sha256,
        "report": str(report_path),
        "recommendation": recommendation,
        "recommendation_reason": recommendation_reason,
        "created_at": report["created_at"],
    },
    EVAL_ROOT / "LATEST.json",
)

global_table = pd.DataFrame([
    {
        "candidate": name,
        "greedy_macro_wer": value["greedy_macro_wer"],
        "lm_macro_wer": value["lm_macro_wer"],
        **{f"wer_{language}": value["lm_language_wer"][language]
           for language in LANGUAGES},
    }
    for name, value in candidate_summary.items()
]).sort_values("lm_macro_wer")

print("\n=== COMPARAISON GLOBALE 19.5 ===")
display(global_table)
print("\n=== WER LM PAR LANGUE ===")
display(pd.DataFrame(language_rows).pivot(
    index="language", columns="candidate", values="lm_wer"
))
print("\n=== BOOTSTRAP APPARIE VS", reference_name, "===")
display(pd.DataFrame([
    {"candidate": name, **value} for name, value in bootstrap.items()
]))
print("\nRECOMMANDATION LOCALE :", recommendation)
print(recommendation_reason)
print("Rapport ->", report_path)
print("Aucun pointeur final n'a ete modifie. Envoyer ces trois tableaux avant la suite.")

In [ ]:
# 19.6a — Audit de faisabilite des adaptateurs bas-rang Kalenjin/Maasai
#
# AUCUN poids final n'est cree ou modifie ici. Cette cellule :
#   1) compare les specialistes Kalenjin/Maasai au modele partage step1250 ;
#   2) mesure l'energie de chaque delta de tenseur ;
#   3) estime les 16 premieres valeurs singulieres des matrices 2D ;
#   4) alloue au plus 10 M parametres par adaptateur ;
#   5) prouve que base + adaptateurs reste sous 1 milliard de parametres.
#
# Les resultats intermediaires sont sauvegardes par blocs sur Drive. Une relance reprend.

import gc
import hashlib
import importlib.metadata
import json
import math
import os
import re
import shutil
import subprocess
import sys
import time
from collections import OrderedDict, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import psutil
import torch
from IPython.display import display


# -----------------------------------------------------------------------------
# 0. Contrat de l'audit
# -----------------------------------------------------------------------------

assert torch.cuda.is_available(), "19.6a requiert un GPU ; un L4 suffit."
assert torch.cuda.get_device_properties(0).total_memory >= 20 * 2**30, (
    "Au moins 20 Gio de VRAM requis pour l'analyse matricielle."
)

FT_ROOT = Path(
    "/content/afrivoices_project/"
    "finetune_runs/experiment_run"
)
REPORT_ROOT = FT_ROOT / "reports/low_rank_adapter_audit_v1"
LOCAL_SOURCE_ROOT = Path("/content/low_rank_adapter_sources_v1")

BASE_PARAMETER_COUNT = 975_675_056
OFFICIAL_PARAMETER_LIMIT = 1_000_000_000
AVAILABLE_HEADROOM = OFFICIAL_PARAMETER_LIMIT - BASE_PARAMETER_COUNT
PER_ADAPTER_BUDGET = 10_000_000
TOTAL_ADAPTER_BUDGET = 2 * PER_ADAPTER_BUDGET
SAFETY_HEADROOM = AVAILABLE_HEADROOM - TOTAL_ADAPTER_BUDGET
MAX_ANALYZED_RANK = 16
RANDOMIZED_OVERSAMPLING = 4
RANDOMIZED_NITER = 2
CACHE_PART_SIZE = 32

assert AVAILABLE_HEADROOM == 24_324_944
assert TOTAL_ADAPTER_BUDGET <= AVAILABLE_HEADROOM
assert SAFETY_HEADROOM == 4_324_944

SOURCES = OrderedDict([
    ("base_step1250", {
        "checkpoint": (
            FT_ROOT / "balanced_joint_v4/candidates/REPLACE_WITH_LOCAL_RUN_ID/step_1250"
        ),
        "role": "shared_base",
    }),
    ("kln_specialist", {
        "checkpoint": FT_ROOT / "topup_kln/workspace_selected/checkpoints/step_2000",
        "role": "adapter_target",
        "language": "kln",
    }),
    ("mas_specialist", {
        "checkpoint": (
            FT_ROOT / "topup_mas_spont/workspace_selected/checkpoints/step_2000"
        ),
        "role": "adapter_target",
        "language": "mas",
    }),
])

REPORT_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_SOURCE_ROOT.mkdir(parents=True, exist_ok=True)

assert psutil.virtual_memory().available >= 18 * 2**30, (
    f"Au moins 18 Gio de RAM disponible requis ; trouve "
    f"{psutil.virtual_memory().available / 2**30:.1f} Gio"
)
assert shutil.disk_usage("/content").free >= 18 * 2**30, (
    f"Au moins 18 Gio libres dans /content requis ; trouve "
    f"{shutil.disk_usage('/content').free / 2**30:.1f} Gio"
)


# -----------------------------------------------------------------------------
# 1. Helpers de provenance et copie locale
# -----------------------------------------------------------------------------

def now_iso():
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path, block_size=16 << 20):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()


def canonical_sha256(value):
    raw = json.dumps(
        value, sort_keys=True, ensure_ascii=False, separators=(",", ":"), default=str
    ).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()


def atomic_json(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".tmp-{os.getpid()}")
    with open(temporary, "w", encoding="utf-8") as handle:
        json.dump(value, handle, ensure_ascii=False, indent=2, default=str)
    os.replace(temporary, path)


def atomic_parquet(records, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".tmp-{os.getpid()}")
    pq.write_table(
        pa.Table.from_pylist(records), temporary,
        compression="zstd", compression_level=3,
    )
    os.replace(temporary, path)


def weight_file(checkpoint):
    checkpoint = Path(checkpoint)
    assert checkpoint.exists(), checkpoint
    if checkpoint.is_file():
        return checkpoint
    files = sorted(path for path in (checkpoint / "model").rglob("*") if path.is_file())
    assert len(files) == 1, (checkpoint, files)
    return files[0]


def manifest_hash_for_source(source_name, checkpoint):
    checkpoint = Path(checkpoint)
    for filename in ("candidate_manifest.json", "merge_manifest.json"):
        path = checkpoint / filename
        if path.exists():
            payload = json.load(open(path, encoding="utf-8"))
            expected = payload.get("weight_sha256") or payload.get("output_weight_sha256")
            if expected:
                return expected, str(path), sha256_file(path)
    # Les anciens checkpoints specialises ont leur choix signe dans reports/.
    if source_name == "kln_specialist":
        path = FT_ROOT / "reports/topup_choice_kln.json"
    elif source_name == "mas_specialist":
        path = FT_ROOT / "reports/topup_choice_mas.json"
    else:
        path = None
    return None, str(path) if path and path.exists() else None, \
        sha256_file(path) if path and path.exists() else None


def rsync_file(source, destination):
    source, destination = Path(source), Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and destination.stat().st_size > source.stat().st_size:
        destination.unlink()
    subprocess.run([
        "rsync", "-a", "--partial", "--append-verify", "--info=progress2",
        str(source), str(destination),
    ], check=True)
    assert destination.stat().st_size == source.stat().st_size


source_meta = {}
print("=== SOURCES DE L'AUDIT ===")
for source_name, config in SOURCES.items():
    checkpoint = Path(config["checkpoint"])
    drive_weight = weight_file(checkpoint)
    declared_hash, manifest_path, manifest_sha = manifest_hash_for_source(
        source_name, checkpoint
    )
    local_weight = LOCAL_SOURCE_ROOT / f"{source_name}.pt"
    if not local_weight.exists() or local_weight.stat().st_size != drive_weight.stat().st_size:
        print(f"Copie locale de {source_name} ({drive_weight.stat().st_size / 2**30:.2f} Gio)...")
        rsync_file(drive_weight, local_weight)
    print("Hash de", source_name, "...")
    actual_hash = sha256_file(local_weight)
    if declared_hash is not None:
        assert actual_hash == declared_hash, (source_name, actual_hash, declared_hash)
    source_meta[source_name] = {
        **{key: value for key, value in config.items() if key != "checkpoint"},
        "checkpoint": str(checkpoint),
        "drive_weight": str(drive_weight),
        "local_weight": str(local_weight),
        "weight_sha256": actual_hash,
        "weight_bytes": int(local_weight.stat().st_size),
        "manifest_path": manifest_path,
        "manifest_sha256": manifest_sha,
    }
    print(source_name, "->", actual_hash[:16])


def package_version(name):
    try:
        return importlib.metadata.version(name)
    except Exception:
        return "unknown"


contract = {
    "schema": 1,
    "purpose": "low-rank delta compressibility audit; no final adapter weights",
    "sources": source_meta,
    "base_parameter_count": BASE_PARAMETER_COUNT,
    "official_parameter_limit": OFFICIAL_PARAMETER_LIMIT,
    "available_headroom": AVAILABLE_HEADROOM,
    "per_adapter_budget": PER_ADAPTER_BUDGET,
    "total_adapter_budget": TOTAL_ADAPTER_BUDGET,
    "safety_headroom": SAFETY_HEADROOM,
    "eligible_tensor_rule": "floating 2D matrices with both dimensions >= 2",
    "max_analyzed_rank": MAX_ANALYZED_RANK,
    "randomized_oversampling": RANDOMIZED_OVERSAMPLING,
    "randomized_niter": RANDOMIZED_NITER,
    "dtype_for_analysis": "float32",
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "python": sys.version,
}
contract_sha256 = canonical_sha256(contract)
RUN_ROOT = REPORT_ROOT / contract_sha256[:16]
CACHE_ROOT = RUN_ROOT / "cache"
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
contract_path = RUN_ROOT / "contract.json"
if contract_path.exists():
    existing = json.load(open(contract_path, encoding="utf-8"))
    assert canonical_sha256(existing) == contract_sha256
else:
    atomic_json(contract, contract_path)
print("Contrat :", contract_sha256)


# -----------------------------------------------------------------------------
# 2. Chargement structurel et analyse SVD reprenable
# -----------------------------------------------------------------------------

def load_state_mmap(path):
    try:
        raw = torch.load(str(path), map_location="cpu", weights_only=False, mmap=True)
    except (TypeError, RuntimeError):
        raw = torch.load(str(path), map_location="cpu", weights_only=False)
    state = raw.get("model", raw) if isinstance(raw, dict) else raw
    assert isinstance(state, dict) and state
    return raw, state


print("Chargement mmap de la base step1250...")
base_raw, base_state = load_state_mmap(source_meta["base_step1250"]["local_weight"])
base_keys = list(base_state.keys())
assert len(base_keys) == 807, len(base_keys)
base_state_elements = sum(
    int(value.numel()) for value in base_state.values() if torch.is_tensor(value)
)
assert base_state_elements == BASE_PARAMETER_COUNT, base_state_elements


def stable_seed(adapter_name, tensor_key):
    raw = hashlib.sha256(f"{adapter_name}\0{tensor_key}".encode()).digest()[:8]
    return int.from_bytes(raw, "little") % (2**31 - 1)


def top_singular_values(delta_cpu, max_rank, seed):
    assert delta_cpu.ndim == 2
    rows, columns = delta_cpu.shape
    rank = min(int(max_rank), int(rows), int(columns))
    if rank <= 0:
        return []
    device = torch.device("cuda")
    torch.cuda.empty_cache()
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    matrix = delta_cpu.to(device=device, dtype=torch.float32)
    try:
        if min(rows, columns) <= 64 or matrix.numel() <= 1_500_000:
            singular = torch.linalg.svdvals(matrix)[:rank]
        else:
            q = min(rank + RANDOMIZED_OVERSAMPLING, min(rows, columns))
            _, singular, _ = torch.pca_lowrank(
                matrix, q=q, center=False, niter=RANDOMIZED_NITER
            )
            singular = torch.sort(singular, descending=True).values[:rank]
        result = singular.detach().double().cpu().tolist()
    except torch.cuda.OutOfMemoryError:
        del matrix
        torch.cuda.empty_cache()
        print("    OOM GPU pour SVD ; repli CPU :", tuple(delta_cpu.shape))
        q = min(rank + RANDOMIZED_OVERSAMPLING, min(rows, columns))
        torch.manual_seed(seed)
        _, singular, _ = torch.pca_lowrank(
            delta_cpu.float(), q=q, center=False, niter=RANDOMIZED_NITER
        )
        result = torch.sort(singular, descending=True).values[:rank].double().tolist()
    finally:
        if "matrix" in locals():
            del matrix
        torch.cuda.empty_cache()
    assert len(result) == rank and all(math.isfinite(value) and value >= 0 for value in result)
    return result


def existing_rows(adapter_name):
    directory = CACHE_ROOT / adapter_name
    frames = []
    for path in sorted(directory.glob("part-*.parquet")):
        frame = pd.read_parquet(path)
        assert frame["contract_sha256"].eq(contract_sha256).all(), path
        assert frame["adapter"].eq(adapter_name).all(), path
        frames.append(frame)
    if not frames:
        return pd.DataFrame(), set(), 0
    frame = pd.concat(frames, ignore_index=True)
    assert frame["tensor_key"].is_unique
    next_part = 1 + max(
        int(path.stem.split("-")[-1]) for path in directory.glob("part-*.parquet")
    )
    return frame, set(frame["tensor_key"].astype(str)), next_part


def flush_rows(buffer, adapter_name, part_number):
    if not buffer:
        return part_number
    directory = CACHE_ROOT / adapter_name
    destination = directory / f"part-{part_number:05d}.parquet"
    assert not destination.exists(), destination
    atomic_parquet(buffer, destination)
    print("  cache ->", destination.name, "|", len(buffer), "tenseurs")
    buffer.clear()
    return part_number + 1


adapter_sources = OrderedDict([
    ("kln", "kln_specialist"),
    ("mas", "mas_specialist"),
])

started_at = time.time()
for adapter_name, source_name in adapter_sources.items():
    print("\n" + "=" * 76)
    print("ANALYSE ADAPTATEUR", adapter_name)
    specialist_raw, specialist_state = load_state_mmap(source_meta[source_name]["local_weight"])
    assert list(specialist_state.keys()) == base_keys

    _, done_keys, part_number = existing_rows(adapter_name)
    buffer = []
    for index, key in enumerate(base_keys, 1):
        if key in done_keys:
            continue
        base_value = base_state[key]
        specialist_value = specialist_state[key]
        row = {
            "contract_sha256": contract_sha256,
            "adapter": adapter_name,
            "source": source_name,
            "tensor_key": key,
            "entry_index": index - 1,
            "is_tensor": bool(torch.is_tensor(base_value)),
            "eligible_2d": False,
            "rank1_parameter_cost": 0,
            "delta_energy": 0.0,
            "top_singular_values_json": "[]",
            "analyzed_rank": 0,
            "exclusion_reason": "",
        }
        if not torch.is_tensor(base_value):
            assert type(base_value) is type(specialist_value)
            assert base_value == specialist_value
            row.update(shape_json="[]", dtype=type(base_value).__name__, ndim=-1,
                       numel=0, exclusion_reason="non_tensor_equal")
        else:
            assert torch.is_tensor(specialist_value)
            assert base_value.shape == specialist_value.shape, key
            assert base_value.dtype == specialist_value.dtype, key
            assert base_value.layout == specialist_value.layout, key
            row.update(
                shape_json=json.dumps(list(base_value.shape)),
                dtype=str(base_value.dtype),
                ndim=int(base_value.ndim),
                numel=int(base_value.numel()),
            )
            if not torch.is_floating_point(base_value):
                assert torch.equal(base_value, specialist_value), key
                row["exclusion_reason"] = "non_floating_equal"
            else:
                delta = specialist_value.detach().float() - base_value.detach().float()
                # Somme en float64 sans materialiser deux copies float64 du gros tenseur.
                energy = float(torch.sum(delta.square(), dtype=torch.float64).item())
                assert math.isfinite(energy) and energy >= 0
                row["delta_energy"] = energy
                if energy == 0.0:
                    row["exclusion_reason"] = "zero_delta"
                elif delta.ndim != 2:
                    row["exclusion_reason"] = f"ndim_{delta.ndim}_not_supported_v1"
                elif min(delta.shape) < 2:
                    row["exclusion_reason"] = "matrix_dimension_below_2"
                else:
                    rows, columns = map(int, delta.shape)
                    singular = top_singular_values(
                        delta, MAX_ANALYZED_RANK, stable_seed(adapter_name, key)
                    )
                    row.update(
                        eligible_2d=True,
                        rank1_parameter_cost=rows + columns,
                        top_singular_values_json=json.dumps(singular),
                        analyzed_rank=len(singular),
                        exclusion_reason="",
                    )
                del delta
        buffer.append(row)
        if len(buffer) >= CACHE_PART_SIZE:
            part_number = flush_rows(buffer, adapter_name, part_number)
        if index % 25 == 0 or index == len(base_keys):
            print(
                f"  {index}/{len(base_keys)} entrees | "
                f"{(time.time() - started_at) / 60:.1f} min cumulees"
            )
    part_number = flush_rows(buffer, adapter_name, part_number)
    del specialist_state, specialist_raw
    gc.collect()
    torch.cuda.empty_cache()


# -----------------------------------------------------------------------------
# 3. Allocation des rangs sous budget strict
# -----------------------------------------------------------------------------

all_rows = []
for adapter_name in adapter_sources:
    parts = sorted((CACHE_ROOT / adapter_name).glob("part-*.parquet"))
    assert parts, adapter_name
    frame = pd.concat([pd.read_parquet(path) for path in parts], ignore_index=True)
    assert len(frame) == len(base_keys), (adapter_name, len(frame), len(base_keys))
    assert frame["tensor_key"].is_unique
    all_rows.append(frame)
audit_frame = pd.concat(all_rows, ignore_index=True)


def allocate_ranks(frame, budget):
    eligible = frame[frame["eligible_2d"].astype(bool)].copy()
    singular_by_key = {
        row.tensor_key: [float(value) for value in json.loads(row.top_singular_values_json)]
        for row in eligible.itertuples()
    }
    cost_by_key = {
        row.tensor_key: int(row.rank1_parameter_cost)
        for row in eligible.itertuples()
    }
    ranks = {key: 0 for key in singular_by_key}
    used = 0
    captured = 0.0
    # Greedy marginal avec contrainte de precedence : seul le prochain rang de chaque matrice
    # est candidat a chaque iteration.
    while True:
        best = None
        for key, values in singular_by_key.items():
            rank = ranks[key]
            if rank >= len(values):
                continue
            cost = cost_by_key[key]
            if used + cost > budget:
                continue
            gain = values[rank] ** 2
            score = gain / cost if cost else -1.0
            candidate = (score, gain, key, cost)
            if best is None or candidate > best:
                best = candidate
        if best is None:
            break
        _, gain, key, cost = best
        ranks[key] += 1
        used += cost
        captured += gain

    selected = {key: rank for key, rank in ranks.items() if rank > 0}
    cap_saturated = any(
        rank == len(singular_by_key[key]) == MAX_ANALYZED_RANK
        for key, rank in selected.items()
    )
    return selected, used, captured, cap_saturated


adapter_plans = {}
plan_rows = []
for adapter_name in adapter_sources:
    frame = audit_frame[audit_frame["adapter"].eq(adapter_name)].copy()
    selected, used, captured, cap_saturated = allocate_ranks(
        frame, PER_ADAPTER_BUDGET
    )
    total_energy = float(frame["delta_energy"].sum())
    eligible_energy = float(frame.loc[frame["eligible_2d"], "delta_energy"].sum())
    selected_rows = []
    for key, rank in sorted(selected.items()):
        source = frame[frame["tensor_key"].eq(key)].iloc[0]
        singular = [float(value) for value in json.loads(source.top_singular_values_json)]
        captured_tensor = float(sum(value * value for value in singular[:rank]))
        record = {
            "adapter": adapter_name,
            "tensor_key": key,
            "shape": json.loads(source.shape_json),
            "rank": int(rank),
            "parameters": int(rank * int(source.rank1_parameter_cost)),
            "delta_energy": float(source.delta_energy),
            "captured_energy": captured_tensor,
            "tensor_capture_ratio": (
                captured_tensor / float(source.delta_energy)
                if float(source.delta_energy) > 0 else 0.0
            ),
        }
        selected_rows.append(record)
        plan_rows.append(record)
    adapter_plans[adapter_name] = {
        "language": adapter_name,
        "source_specialist": source_meta[adapter_sources[adapter_name]],
        "parameter_budget": PER_ADAPTER_BUDGET,
        "planned_parameters": int(used),
        "unused_budget": int(PER_ADAPTER_BUDGET - used),
        "active_matrices": len(selected_rows),
        "maximum_rank": max((row["rank"] for row in selected_rows), default=0),
        "total_delta_energy": total_energy,
        "eligible_2d_delta_energy": eligible_energy,
        "captured_delta_energy_estimate": float(captured),
        "capture_ratio_total_delta": captured / total_energy if total_energy else 0.0,
        "capture_ratio_eligible_2d": captured / eligible_energy if eligible_energy else 0.0,
        "rank_cap_saturated": bool(cap_saturated),
        "selected_matrices": selected_rows,
    }

planned_adapter_parameters = sum(
    plan["planned_parameters"] for plan in adapter_plans.values()
)
planned_total_parameters = BASE_PARAMETER_COUNT + planned_adapter_parameters
assert planned_adapter_parameters <= TOTAL_ADAPTER_BUDGET
assert planned_total_parameters < OFFICIAL_PARAMETER_LIMIT

status = "PASS"
issues = []
for adapter_name, plan in adapter_plans.items():
    if plan["capture_ratio_eligible_2d"] < 0.50:
        issues.append({
            "adapter": adapter_name,
            "code": "LOW_ELIGIBLE_ENERGY_CAPTURE",
            "value": plan["capture_ratio_eligible_2d"],
        })
    if plan["rank_cap_saturated"]:
        issues.append({
            "adapter": adapter_name,
            "code": "RANK_CAP_SATURATED",
            "detail": "Relancer l'audit avec MAX_ANALYZED_RANK plus eleve avant construction.",
        })
if issues:
    status = "REVIEW"

report = {
    "schema": 1,
    "created_at": now_iso(),
    "status": status,
    "contract_sha256": contract_sha256,
    "contract_path": str(contract_path),
    "base_parameter_count": BASE_PARAMETER_COUNT,
    "adapter_parameter_budget": TOTAL_ADAPTER_BUDGET,
    "planned_adapter_parameters": planned_adapter_parameters,
    "planned_total_neural_parameters": planned_total_parameters,
    "official_parameter_limit": OFFICIAL_PARAMETER_LIMIT,
    "remaining_parameter_headroom": OFFICIAL_PARAMETER_LIMIT - planned_total_parameters,
    "adapters": adapter_plans,
    "issues": issues,
    "limitations": [
        "Energy capture is an SVD proxy; WER must be measured after adapter construction.",
        "Only 2D floating tensors are eligible in adapter format v1.",
        "KenLM parameter-count interpretation remains separately documented.",
        "No final model pointer or checkpoint was modified by this audit.",
    ],
    "elapsed_minutes": round((time.time() - started_at) / 60, 2),
}
report_path = RUN_ROOT / "low_rank_adapter_audit_v1.json"
plan_path = RUN_ROOT / "low_rank_adapter_plan_v1.parquet"
atomic_json(report, report_path)
atomic_parquet(plan_rows, plan_path)
atomic_json({
    "contract_sha256": contract_sha256,
    "report": str(report_path),
    "plan": str(plan_path),
    "status": status,
    "created_at": report["created_at"],
}, REPORT_ROOT / "LATEST.json")

summary_rows = []
for adapter_name, plan in adapter_plans.items():
    summary_rows.append({
        "adapter": adapter_name,
        "planned_parameters": plan["planned_parameters"],
        "active_matrices": plan["active_matrices"],
        "maximum_rank": plan["maximum_rank"],
        "capture_total": plan["capture_ratio_total_delta"],
        "capture_eligible_2d": plan["capture_ratio_eligible_2d"],
        "rank_cap_saturated": plan["rank_cap_saturated"],
    })

print("\n=== PLAN D'ADAPTATEURS BAS-RANG ===")
display(pd.DataFrame(summary_rows))
print("\nParametres base       :", f"{BASE_PARAMETER_COUNT:,}")
print("Parametres adaptateurs:", f"{planned_adapter_parameters:,}")
print("Total neural          :", f"{planned_total_parameters:,}")
print("Marge sous 1 milliard :", f"{OFFICIAL_PARAMETER_LIMIT - planned_total_parameters:,}")
print("STATUT 19.6a          :", status)
if issues:
    print("Points a examiner :")
    for issue in issues:
        print(" -", issue)
print("Rapport ->", report_path)
print("Plan    ->", plan_path)
print("Aucun checkpoint final n'a ete cree ou modifie.")

In [ ]:
# 19.6a — Audit de faisabilite des adaptateurs bas-rang Kalenjin/Maasai
#
# AUCUN poids final n'est cree ou modifie ici. Cette cellule :
#   1) compare les specialistes Kalenjin/Maasai au modele partage step1250 ;
#   2) mesure l'energie de chaque delta de tenseur ;
#   3) estime les 16 premieres valeurs singulieres des matrices 2D ;
#   4) alloue au plus 10 M parametres par adaptateur ;
#   5) prouve que base + adaptateurs reste sous 1 milliard de parametres.
#
# Les resultats intermediaires sont sauvegardes par blocs sur Drive. Une relance reprend.

import gc
import hashlib
import importlib.metadata
import json
import math
import os
import re
import shutil
import subprocess
import sys
import time
from collections import OrderedDict, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import psutil
import torch
from IPython.display import display


# -----------------------------------------------------------------------------
# 0. Contrat de l'audit
# -----------------------------------------------------------------------------

assert torch.cuda.is_available(), "19.6a requiert un GPU ; un L4 suffit."
assert torch.cuda.get_device_properties(0).total_memory >= 20 * 2**30, (
    "Au moins 20 Gio de VRAM requis pour l'analyse matricielle."
)

FT_ROOT = Path(
    "/content/afrivoices_project/"
    "finetune_runs/experiment_run"
)
REPORT_ROOT = FT_ROOT / "reports/low_rank_adapter_audit_v1"
LOCAL_SOURCE_ROOT = Path("/content/low_rank_adapter_sources_v1")

BASE_PARAMETER_COUNT = 975_675_056
OFFICIAL_PARAMETER_LIMIT = 1_000_000_000
AVAILABLE_HEADROOM = OFFICIAL_PARAMETER_LIMIT - BASE_PARAMETER_COUNT
PER_ADAPTER_BUDGET = 10_000_000
TOTAL_ADAPTER_BUDGET = 2 * PER_ADAPTER_BUDGET
SAFETY_HEADROOM = AVAILABLE_HEADROOM - TOTAL_ADAPTER_BUDGET
MAX_ANALYZED_RANK = 64
RANDOMIZED_OVERSAMPLING = 4
RANDOMIZED_NITER = 2
CACHE_PART_SIZE = 32

assert AVAILABLE_HEADROOM == 24_324_944
assert TOTAL_ADAPTER_BUDGET <= AVAILABLE_HEADROOM
assert SAFETY_HEADROOM == 4_324_944

SOURCES = OrderedDict([
    ("base_step1250", {
        "checkpoint": (
            FT_ROOT / "balanced_joint_v4/candidates/REPLACE_WITH_LOCAL_RUN_ID/step_1250"
        ),
        "role": "shared_base",
    }),
    ("kln_specialist", {
        "checkpoint": FT_ROOT / "topup_kln/workspace_selected/checkpoints/step_2000",
        "role": "adapter_target",
        "language": "kln",
    }),
    ("mas_specialist", {
        "checkpoint": (
            FT_ROOT / "topup_mas_spont/workspace_selected/checkpoints/step_2000"
        ),
        "role": "adapter_target",
        "language": "mas",
    }),
])

REPORT_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_SOURCE_ROOT.mkdir(parents=True, exist_ok=True)

assert psutil.virtual_memory().available >= 18 * 2**30, (
    f"Au moins 18 Gio de RAM disponible requis ; trouve "
    f"{psutil.virtual_memory().available / 2**30:.1f} Gio"
)
assert shutil.disk_usage("/content").free >= 18 * 2**30, (
    f"Au moins 18 Gio libres dans /content requis ; trouve "
    f"{shutil.disk_usage('/content').free / 2**30:.1f} Gio"
)


# -----------------------------------------------------------------------------
# 1. Helpers de provenance et copie locale
# -----------------------------------------------------------------------------

def now_iso():
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path, block_size=16 << 20):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()


def canonical_sha256(value):
    raw = json.dumps(
        value, sort_keys=True, ensure_ascii=False, separators=(",", ":"), default=str
    ).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()


def atomic_json(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".tmp-{os.getpid()}")
    with open(temporary, "w", encoding="utf-8") as handle:
        json.dump(value, handle, ensure_ascii=False, indent=2, default=str)
    os.replace(temporary, path)


def atomic_parquet(records, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".tmp-{os.getpid()}")
    pq.write_table(
        pa.Table.from_pylist(records), temporary,
        compression="zstd", compression_level=3,
    )
    os.replace(temporary, path)


def weight_file(checkpoint):
    checkpoint = Path(checkpoint)
    assert checkpoint.exists(), checkpoint
    if checkpoint.is_file():
        return checkpoint
    files = sorted(path for path in (checkpoint / "model").rglob("*") if path.is_file())
    assert len(files) == 1, (checkpoint, files)
    return files[0]


def manifest_hash_for_source(source_name, checkpoint):
    checkpoint = Path(checkpoint)
    for filename in ("candidate_manifest.json", "merge_manifest.json"):
        path = checkpoint / filename
        if path.exists():
            payload = json.load(open(path, encoding="utf-8"))
            expected = payload.get("weight_sha256") or payload.get("output_weight_sha256")
            if expected:
                return expected, str(path), sha256_file(path)
    # Les anciens checkpoints specialises ont leur choix signe dans reports/.
    if source_name == "kln_specialist":
        path = FT_ROOT / "reports/topup_choice_kln.json"
    elif source_name == "mas_specialist":
        path = FT_ROOT / "reports/topup_choice_mas.json"
    else:
        path = None
    return None, str(path) if path and path.exists() else None, \
        sha256_file(path) if path and path.exists() else None


def rsync_file(source, destination):
    source, destination = Path(source), Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and destination.stat().st_size > source.stat().st_size:
        destination.unlink()
    subprocess.run([
        "rsync", "-a", "--partial", "--append-verify", "--info=progress2",
        str(source), str(destination),
    ], check=True)
    assert destination.stat().st_size == source.stat().st_size


source_meta = {}
print("=== SOURCES DE L'AUDIT ===")
for source_name, config in SOURCES.items():
    checkpoint = Path(config["checkpoint"])
    drive_weight = weight_file(checkpoint)
    declared_hash, manifest_path, manifest_sha = manifest_hash_for_source(
        source_name, checkpoint
    )
    local_weight = LOCAL_SOURCE_ROOT / f"{source_name}.pt"
    if not local_weight.exists() or local_weight.stat().st_size != drive_weight.stat().st_size:
        print(f"Copie locale de {source_name} ({drive_weight.stat().st_size / 2**30:.2f} Gio)...")
        rsync_file(drive_weight, local_weight)
    print("Hash de", source_name, "...")
    actual_hash = sha256_file(local_weight)
    if declared_hash is not None:
        assert actual_hash == declared_hash, (source_name, actual_hash, declared_hash)
    source_meta[source_name] = {
        **{key: value for key, value in config.items() if key != "checkpoint"},
        "checkpoint": str(checkpoint),
        "drive_weight": str(drive_weight),
        "local_weight": str(local_weight),
        "weight_sha256": actual_hash,
        "weight_bytes": int(local_weight.stat().st_size),
        "manifest_path": manifest_path,
        "manifest_sha256": manifest_sha,
    }
    print(source_name, "->", actual_hash[:16])


def package_version(name):
    try:
        return importlib.metadata.version(name)
    except Exception:
        return "unknown"


contract = {
    "schema": 1,
    "purpose": "low-rank delta compressibility audit; no final adapter weights",
    "sources": source_meta,
    "base_parameter_count": BASE_PARAMETER_COUNT,
    "official_parameter_limit": OFFICIAL_PARAMETER_LIMIT,
    "available_headroom": AVAILABLE_HEADROOM,
    "per_adapter_budget": PER_ADAPTER_BUDGET,
    "total_adapter_budget": TOTAL_ADAPTER_BUDGET,
    "safety_headroom": SAFETY_HEADROOM,
    "eligible_tensor_rule": "floating 2D matrices with both dimensions >= 2",
    "max_analyzed_rank": MAX_ANALYZED_RANK,
    "randomized_oversampling": RANDOMIZED_OVERSAMPLING,
    "randomized_niter": RANDOMIZED_NITER,
    "dtype_for_analysis": "float32",
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "python": sys.version,
}
contract_sha256 = canonical_sha256(contract)
RUN_ROOT = REPORT_ROOT / contract_sha256[:16]
CACHE_ROOT = RUN_ROOT / "cache"
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
contract_path = RUN_ROOT / "contract.json"
if contract_path.exists():
    existing = json.load(open(contract_path, encoding="utf-8"))
    assert canonical_sha256(existing) == contract_sha256
else:
    atomic_json(contract, contract_path)
print("Contrat :", contract_sha256)


# -----------------------------------------------------------------------------
# 2. Chargement structurel et analyse SVD reprenable
# -----------------------------------------------------------------------------

def load_state_mmap(path):
    try:
        raw = torch.load(str(path), map_location="cpu", weights_only=False, mmap=True)
    except (TypeError, RuntimeError):
        raw = torch.load(str(path), map_location="cpu", weights_only=False)
    state = raw.get("model", raw) if isinstance(raw, dict) else raw
    assert isinstance(state, dict) and state
    return raw, state


print("Chargement mmap de la base step1250...")
base_raw, base_state = load_state_mmap(source_meta["base_step1250"]["local_weight"])
base_keys = list(base_state.keys())
assert len(base_keys) == 807, len(base_keys)
base_state_elements = sum(
    int(value.numel()) for value in base_state.values() if torch.is_tensor(value)
)
assert base_state_elements == BASE_PARAMETER_COUNT, base_state_elements


def stable_seed(adapter_name, tensor_key):
    raw = hashlib.sha256(f"{adapter_name}\0{tensor_key}".encode()).digest()[:8]
    return int.from_bytes(raw, "little") % (2**31 - 1)


def top_singular_values(delta_cpu, max_rank, seed):
    assert delta_cpu.ndim == 2
    rows, columns = delta_cpu.shape
    rank = min(int(max_rank), int(rows), int(columns))
    if rank <= 0:
        return []
    device = torch.device("cuda")
    torch.cuda.empty_cache()
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    matrix = delta_cpu.to(device=device, dtype=torch.float32)
    try:
        if min(rows, columns) <= 64 or matrix.numel() <= 1_500_000:
            singular = torch.linalg.svdvals(matrix)[:rank]
        else:
            q = min(rank + RANDOMIZED_OVERSAMPLING, min(rows, columns))
            _, singular, _ = torch.pca_lowrank(
                matrix, q=q, center=False, niter=RANDOMIZED_NITER
            )
            singular = torch.sort(singular, descending=True).values[:rank]
        result = singular.detach().double().cpu().tolist()
    except torch.cuda.OutOfMemoryError:
        del matrix
        torch.cuda.empty_cache()
        print("    OOM GPU pour SVD ; repli CPU :", tuple(delta_cpu.shape))
        q = min(rank + RANDOMIZED_OVERSAMPLING, min(rows, columns))
        torch.manual_seed(seed)
        _, singular, _ = torch.pca_lowrank(
            delta_cpu.float(), q=q, center=False, niter=RANDOMIZED_NITER
        )
        result = torch.sort(singular, descending=True).values[:rank].double().tolist()
    finally:
        if "matrix" in locals():
            del matrix
        torch.cuda.empty_cache()
    assert len(result) == rank and all(math.isfinite(value) and value >= 0 for value in result)
    return result


def existing_rows(adapter_name):
    directory = CACHE_ROOT / adapter_name
    frames = []
    for path in sorted(directory.glob("part-*.parquet")):
        frame = pd.read_parquet(path)
        assert frame["contract_sha256"].eq(contract_sha256).all(), path
        assert frame["adapter"].eq(adapter_name).all(), path
        frames.append(frame)
    if not frames:
        return pd.DataFrame(), set(), 0
    frame = pd.concat(frames, ignore_index=True)
    assert frame["tensor_key"].is_unique
    next_part = 1 + max(
        int(path.stem.split("-")[-1]) for path in directory.glob("part-*.parquet")
    )
    return frame, set(frame["tensor_key"].astype(str)), next_part


def flush_rows(buffer, adapter_name, part_number):
    if not buffer:
        return part_number
    directory = CACHE_ROOT / adapter_name
    destination = directory / f"part-{part_number:05d}.parquet"
    assert not destination.exists(), destination
    atomic_parquet(buffer, destination)
    print("  cache ->", destination.name, "|", len(buffer), "tenseurs")
    buffer.clear()
    return part_number + 1


adapter_sources = OrderedDict([
    ("kln", "kln_specialist"),
    ("mas", "mas_specialist"),
])

started_at = time.time()
for adapter_name, source_name in adapter_sources.items():
    print("\n" + "=" * 76)
    print("ANALYSE ADAPTATEUR", adapter_name)
    specialist_raw, specialist_state = load_state_mmap(source_meta[source_name]["local_weight"])
    assert list(specialist_state.keys()) == base_keys

    _, done_keys, part_number = existing_rows(adapter_name)
    buffer = []
    for index, key in enumerate(base_keys, 1):
        if key in done_keys:
            continue
        base_value = base_state[key]
        specialist_value = specialist_state[key]
        row = {
            "contract_sha256": contract_sha256,
            "adapter": adapter_name,
            "source": source_name,
            "tensor_key": key,
            "entry_index": index - 1,
            "is_tensor": bool(torch.is_tensor(base_value)),
            "eligible_2d": False,
            "rank1_parameter_cost": 0,
            "delta_energy": 0.0,
            "top_singular_values_json": "[]",
            "analyzed_rank": 0,
            "exclusion_reason": "",
        }
        if not torch.is_tensor(base_value):
            assert type(base_value) is type(specialist_value)
            assert base_value == specialist_value
            row.update(shape_json="[]", dtype=type(base_value).__name__, ndim=-1,
                       numel=0, exclusion_reason="non_tensor_equal")
        else:
            assert torch.is_tensor(specialist_value)
            assert base_value.shape == specialist_value.shape, key
            assert base_value.dtype == specialist_value.dtype, key
            assert base_value.layout == specialist_value.layout, key
            row.update(
                shape_json=json.dumps(list(base_value.shape)),
                dtype=str(base_value.dtype),
                ndim=int(base_value.ndim),
                numel=int(base_value.numel()),
            )
            if not torch.is_floating_point(base_value):
                assert torch.equal(base_value, specialist_value), key
                row["exclusion_reason"] = "non_floating_equal"
            else:
                delta = specialist_value.detach().float() - base_value.detach().float()
                # Somme en float64 sans materialiser deux copies float64 du gros tenseur.
                energy = float(torch.sum(delta.square(), dtype=torch.float64).item())
                assert math.isfinite(energy) and energy >= 0
                row["delta_energy"] = energy
                if energy == 0.0:
                    row["exclusion_reason"] = "zero_delta"
                elif delta.ndim != 2:
                    row["exclusion_reason"] = f"ndim_{delta.ndim}_not_supported_v1"
                elif min(delta.shape) < 2:
                    row["exclusion_reason"] = "matrix_dimension_below_2"
                else:
                    rows, columns = map(int, delta.shape)
                    singular = top_singular_values(
                        delta, MAX_ANALYZED_RANK, stable_seed(adapter_name, key)
                    )
                    row.update(
                        eligible_2d=True,
                        rank1_parameter_cost=rows + columns,
                        top_singular_values_json=json.dumps(singular),
                        analyzed_rank=len(singular),
                        exclusion_reason="",
                    )
                del delta
        buffer.append(row)
        if len(buffer) >= CACHE_PART_SIZE:
            part_number = flush_rows(buffer, adapter_name, part_number)
        if index % 25 == 0 or index == len(base_keys):
            print(
                f"  {index}/{len(base_keys)} entrees | "
                f"{(time.time() - started_at) / 60:.1f} min cumulees"
            )
    part_number = flush_rows(buffer, adapter_name, part_number)
    del specialist_state, specialist_raw
    gc.collect()
    torch.cuda.empty_cache()


# -----------------------------------------------------------------------------
# 3. Allocation des rangs sous budget strict
# -----------------------------------------------------------------------------

all_rows = []
for adapter_name in adapter_sources:
    parts = sorted((CACHE_ROOT / adapter_name).glob("part-*.parquet"))
    assert parts, adapter_name
    frame = pd.concat([pd.read_parquet(path) for path in parts], ignore_index=True)
    assert len(frame) == len(base_keys), (adapter_name, len(frame), len(base_keys))
    assert frame["tensor_key"].is_unique
    all_rows.append(frame)
audit_frame = pd.concat(all_rows, ignore_index=True)


def allocate_ranks(frame, budget):
    eligible = frame[frame["eligible_2d"].astype(bool)].copy()
    singular_by_key = {
        row.tensor_key: [float(value) for value in json.loads(row.top_singular_values_json)]
        for row in eligible.itertuples()
    }
    cost_by_key = {
        row.tensor_key: int(row.rank1_parameter_cost)
        for row in eligible.itertuples()
    }
    ranks = {key: 0 for key in singular_by_key}
    used = 0
    captured = 0.0
    # Greedy marginal avec contrainte de precedence : seul le prochain rang de chaque matrice
    # est candidat a chaque iteration.
    while True:
        best = None
        for key, values in singular_by_key.items():
            rank = ranks[key]
            if rank >= len(values):
                continue
            cost = cost_by_key[key]
            if used + cost > budget:
                continue
            gain = values[rank] ** 2
            score = gain / cost if cost else -1.0
            candidate = (score, gain, key, cost)
            if best is None or candidate > best:
                best = candidate
        if best is None:
            break
        _, gain, key, cost = best
        ranks[key] += 1
        used += cost
        captured += gain

    selected = {key: rank for key, rank in ranks.items() if rank > 0}
    cap_saturated = any(
        rank == len(singular_by_key[key]) == MAX_ANALYZED_RANK
        for key, rank in selected.items()
    )
    return selected, used, captured, cap_saturated


adapter_plans = {}
plan_rows = []
for adapter_name in adapter_sources:
    frame = audit_frame[audit_frame["adapter"].eq(adapter_name)].copy()
    selected, used, captured, cap_saturated = allocate_ranks(
        frame, PER_ADAPTER_BUDGET
    )
    total_energy = float(frame["delta_energy"].sum())
    eligible_energy = float(frame.loc[frame["eligible_2d"], "delta_energy"].sum())
    selected_rows = []
    for key, rank in sorted(selected.items()):
        source = frame[frame["tensor_key"].eq(key)].iloc[0]
        singular = [float(value) for value in json.loads(source.top_singular_values_json)]
        captured_tensor = float(sum(value * value for value in singular[:rank]))
        record = {
            "adapter": adapter_name,
            "tensor_key": key,
            "shape": json.loads(source.shape_json),
            "rank": int(rank),
            "parameters": int(rank * int(source.rank1_parameter_cost)),
            "delta_energy": float(source.delta_energy),
            "captured_energy": captured_tensor,
            "tensor_capture_ratio": (
                captured_tensor / float(source.delta_energy)
                if float(source.delta_energy) > 0 else 0.0
            ),
        }
        selected_rows.append(record)
        plan_rows.append(record)
    adapter_plans[adapter_name] = {
        "language": adapter_name,
        "source_specialist": source_meta[adapter_sources[adapter_name]],
        "parameter_budget": PER_ADAPTER_BUDGET,
        "planned_parameters": int(used),
        "unused_budget": int(PER_ADAPTER_BUDGET - used),
        "active_matrices": len(selected_rows),
        "maximum_rank": max((row["rank"] for row in selected_rows), default=0),
        "total_delta_energy": total_energy,
        "eligible_2d_delta_energy": eligible_energy,
        "captured_delta_energy_estimate": float(captured),
        "capture_ratio_total_delta": captured / total_energy if total_energy else 0.0,
        "capture_ratio_eligible_2d": captured / eligible_energy if eligible_energy else 0.0,
        "rank_cap_saturated": bool(cap_saturated),
        "selected_matrices": selected_rows,
    }

planned_adapter_parameters = sum(
    plan["planned_parameters"] for plan in adapter_plans.values()
)
planned_total_parameters = BASE_PARAMETER_COUNT + planned_adapter_parameters
assert planned_adapter_parameters <= TOTAL_ADAPTER_BUDGET
assert planned_total_parameters < OFFICIAL_PARAMETER_LIMIT

status = "PASS"
issues = []
for adapter_name, plan in adapter_plans.items():
    if plan["capture_ratio_eligible_2d"] < 0.50:
        issues.append({
            "adapter": adapter_name,
            "code": "LOW_ELIGIBLE_ENERGY_CAPTURE",
            "value": plan["capture_ratio_eligible_2d"],
        })
    if plan["rank_cap_saturated"]:
        issues.append({
            "adapter": adapter_name,
            "code": "RANK_CAP_SATURATED",
            "detail": "Relancer l'audit avec MAX_ANALYZED_RANK plus eleve avant construction.",
        })
if issues:
    status = "REVIEW"

report = {
    "schema": 1,
    "created_at": now_iso(),
    "status": status,
    "contract_sha256": contract_sha256,
    "contract_path": str(contract_path),
    "base_parameter_count": BASE_PARAMETER_COUNT,
    "adapter_parameter_budget": TOTAL_ADAPTER_BUDGET,
    "planned_adapter_parameters": planned_adapter_parameters,
    "planned_total_neural_parameters": planned_total_parameters,
    "official_parameter_limit": OFFICIAL_PARAMETER_LIMIT,
    "remaining_parameter_headroom": OFFICIAL_PARAMETER_LIMIT - planned_total_parameters,
    "adapters": adapter_plans,
    "issues": issues,
    "limitations": [
        "Energy capture is an SVD proxy; WER must be measured after adapter construction.",
        "Only 2D floating tensors are eligible in adapter format v1.",
        "KenLM parameter-count interpretation remains separately documented.",
        "No final model pointer or checkpoint was modified by this audit.",
    ],
    "elapsed_minutes": round((time.time() - started_at) / 60, 2),
}
report_path = RUN_ROOT / "low_rank_adapter_audit_v1.json"
plan_path = RUN_ROOT / "low_rank_adapter_plan_v1.parquet"
atomic_json(report, report_path)
atomic_parquet(plan_rows, plan_path)
atomic_json({
    "contract_sha256": contract_sha256,
    "report": str(report_path),
    "plan": str(plan_path),
    "status": status,
    "created_at": report["created_at"],
}, REPORT_ROOT / "LATEST.json")

summary_rows = []
for adapter_name, plan in adapter_plans.items():
    summary_rows.append({
        "adapter": adapter_name,
        "planned_parameters": plan["planned_parameters"],
        "active_matrices": plan["active_matrices"],
        "maximum_rank": plan["maximum_rank"],
        "capture_total": plan["capture_ratio_total_delta"],
        "capture_eligible_2d": plan["capture_ratio_eligible_2d"],
        "rank_cap_saturated": plan["rank_cap_saturated"],
    })

print("\n=== PLAN D'ADAPTATEURS BAS-RANG ===")
display(pd.DataFrame(summary_rows))
print("\nParametres base       :", f"{BASE_PARAMETER_COUNT:,}")
print("Parametres adaptateurs:", f"{planned_adapter_parameters:,}")
print("Total neural          :", f"{planned_total_parameters:,}")
print("Marge sous 1 milliard :", f"{OFFICIAL_PARAMETER_LIMIT - planned_total_parameters:,}")
print("STATUT 19.6a          :", status)
if issues:
    print("Points a examiner :")
    for issue in issues:
        print(" -", issue)
print("Rapport ->", report_path)
print("Plan    ->", plan_path)
print("Aucun checkpoint final n'a ete cree ou modifie.")

In [ ]:
# 19.7a — Audit de compatibilite LoRA route Kalenjin/Maasai
#
# Cette cellule NE FAIT AUCUN ENTRAINEMENT et NE CREE AUCUN CHECKPOINT FINAL.
# Elle verifie, avant de depenser du GPU, que :
#   1) le checkpoint partage step_1250 contient exactement 975 675 056 parametres ;
#   2) deux adaptateurs LoRA (kln et mas) tiennent ensemble sous 1 milliard ;
#   3) seules les matrices lineaires internes eligibles sont adaptees ;
#   4) un LoRA initialise a zero reproduit exactement les logits et textes step_1250 ;
#   5) le routage kln/mas et le gel des poids de base fonctionnent ;
#   6) l'injection est entierement reversible.
#
# Prerequis : executer les cellules d'installation/configuration 1.x -> 2.4 du
# notebook, puis cette cellule sur un GPU. Un L4 suffit.

import gc
import hashlib
import importlib.metadata
import json
import math
import os
import re
import shutil
import sys
from collections import Counter, OrderedDict
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display


# -----------------------------------------------------------------------------
# 0. Contrat strict
# -----------------------------------------------------------------------------

assert torch.cuda.is_available(), "19.7a requiert un GPU ; un L4 suffit."
assert "FT_CONFIG" in globals(), (
    "FT_CONFIG absent : relancer les cellules 1.x -> 2.4 avant 19.7a."
)
assert "omni_lang" in globals() and callable(omni_lang), (
    "omni_lang absent : relancer la cellule 2.3 avant 19.7a."
)

FT_ROOT = Path(
    "/content/afrivoices_project/"
    "finetune_runs/experiment_run"
)
BASE_CHECKPOINT = (
    FT_ROOT / "balanced_joint_v4/candidates/REPLACE_WITH_LOCAL_RUN_ID/step_1250"
)
BASE_WEIGHT_SHA256 = (
    "REPLACE_WITH_LOCAL_RUN_ID"  # prefixe fige ; le hash complet est relu du manifest
)
LOCAL_BASE_WEIGHT = Path("/content/low_rank_adapter_sources_v1/base_step1250.pt")
LOCAL_AUDIO_ROOT = Path("/content/delta_merge_eval_audio_v2")
REPORT_ROOT = FT_ROOT / "reports/lora_compatibility_audit_v1"

BASE_PARAMETER_COUNT = 975_675_056
OFFICIAL_PARAMETER_LIMIT = 1_000_000_000
PER_ADAPTER_BUDGET = 9_900_000
ADAPTER_NAMES = ("kln", "mas")
MIN_LORA_RANK = 4
MAX_LORA_RANK = 32
LORA_SEED = 19_700_001
PROBES_PER_LANGUAGE = 2

# On conserve volontairement plus de 4,5 M de marge pour les petits parametres
# auxiliaires eventuels du packaging final.
assert BASE_PARAMETER_COUNT + len(ADAPTER_NAMES) * PER_ADAPTER_BUDGET < OFFICIAL_PARAMETER_LIMIT


def now_iso():
    return datetime.now(timezone.utc).isoformat()


def canonical_sha256(value):
    raw = json.dumps(
        value, sort_keys=True, ensure_ascii=False, separators=(",", ":"), default=str
    ).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()


def sha256_file(path, block_size=16 << 20):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()


def atomic_json(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".tmp-{os.getpid()}")
    with open(temporary, "w", encoding="utf-8") as handle:
        json.dump(value, handle, ensure_ascii=False, indent=2, default=str)
    os.replace(temporary, path)
    print("json ->", path)


def weight_file(checkpoint):
    checkpoint = Path(checkpoint)
    assert checkpoint.exists(), checkpoint
    if checkpoint.is_file():
        return checkpoint
    files = sorted(path for path in (checkpoint / "model").rglob("*") if path.is_file())
    assert len(files) == 1, (checkpoint, files)
    return files[0]


def package_version(name):
    try:
        return importlib.metadata.version(name)
    except Exception:
        return "unknown"


def base_declared_hash():
    manifest = BASE_CHECKPOINT / "candidate_manifest.json"
    assert manifest.exists(), manifest
    payload = json.load(open(manifest, encoding="utf-8"))
    expected = payload.get("weight_sha256")
    assert expected and expected.startswith(BASE_WEIGHT_SHA256), expected
    return expected, manifest, sha256_file(manifest)


expected_weight_sha256, base_manifest, base_manifest_sha256 = base_declared_hash()
drive_weight = weight_file(BASE_CHECKPOINT)

if not LOCAL_BASE_WEIGHT.exists() or LOCAL_BASE_WEIGHT.stat().st_size != drive_weight.stat().st_size:
    raise AssertionError(
        "Copie locale step_1250 absente. Relancer 19.6a, qui cree et verifie : "
        f"{LOCAL_BASE_WEIGHT}"
    )

print("Hash du checkpoint step_1250...")
actual_weight_sha256 = sha256_file(LOCAL_BASE_WEIGHT)
assert actual_weight_sha256 == expected_weight_sha256, (
    actual_weight_sha256, expected_weight_sha256
)
print("step_1250 ->", actual_weight_sha256[:16])


# -----------------------------------------------------------------------------
# 1. Chargement du modele partage et inventaire des couches lineaires
# -----------------------------------------------------------------------------

from fairseq2.data.tokenizers.hub import load_tokenizer
from fairseq2.models.hub import load_model
from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline

torch.manual_seed(LORA_SEED)
torch.cuda.manual_seed_all(LORA_SEED)
device = torch.device("cuda")
model_card = FT_CONFIG["model_card"]
model_dtype = torch.bfloat16

print("Chargement du modele step_1250...")
model = load_model(model_card, device=device, dtype=model_dtype)
raw_state = torch.load(
    str(LOCAL_BASE_WEIGHT), map_location="cpu", weights_only=False, mmap=True
)
state = raw_state.get("model", raw_state) if isinstance(raw_state, dict) else raw_state
assert isinstance(state, dict) and state
model.load_state_dict(state)
del raw_state, state
gc.collect()
model.eval()

base_parameter_count_runtime = sum(parameter.numel() for parameter in model.parameters())
assert base_parameter_count_runtime == BASE_PARAMETER_COUNT, (
    base_parameter_count_runtime, BASE_PARAMETER_COUNT
)

# L'identite des objets Parameter permet de prouver ensuite que l'injection puis
# le retrait des wrappers ne remplace et ne modifie aucun poids de base.
base_parameter_identity = {
    id(parameter): {
        "numel": int(parameter.numel()),
        "data_ptr": int(parameter.data_ptr()),
        "dtype": str(parameter.dtype),
    }
    for parameter in model.parameters()
}
assert sum(item["numel"] for item in base_parameter_identity.values()) == BASE_PARAMETER_COUNT


def projection_dimensions(module):
    """Retourne (entree, sortie) pour nn.Linear OU une projection Fairseq2."""
    weight = getattr(module, "weight", None)
    if not isinstance(weight, torch.Tensor) or weight.ndim != 2:
        return None
    class_name = type(module).__name__.lower()
    class_module = type(module).__module__.lower()
    # Embedding/conv/norm peuvent aussi exposer un poids 2D, mais ne realisent
    # pas la projection y = x W^T attendue par LoRA.
    forbidden = ("embedding", "embed", "convolution", "conv", "norm")
    if any(token in class_name for token in forbidden):
        return None
    if isinstance(module, (nn.Embedding, nn.Conv1d, nn.Conv2d, nn.Conv3d)):
        return None
    # Une projection doit etre une feuille. Cela evite de prendre un bloc entier
    # qui possederait par hasard un attribut weight en plus de sous-modules.
    if any(True for _ in module.children()):
        return None

    input_dim = getattr(module, "in_features", None)
    output_dim = getattr(module, "out_features", None)
    if input_dim is None:
        input_dim = getattr(module, "input_dim", None)
    if output_dim is None:
        output_dim = getattr(module, "output_dim", None)
    input_dim = int(input_dim) if input_dim is not None else int(weight.shape[1])
    output_dim = int(output_dim) if output_dim is not None else int(weight.shape[0])

    # Fairseq2 Linear utilise la meme orientation que F.linear : [sortie, entree].
    if tuple(weight.shape) != (output_dim, input_dim):
        return None
    is_known_projection = (
        isinstance(module, nn.Linear)
        or "projection" in class_name
        or "linear" in class_name
        or "projection" in class_module
    )
    if not is_known_projection:
        return None
    return input_dim, output_dim


def is_lora_eligible(name, module):
    dimensions = projection_dimensions(module)
    if dimensions is None:
        return False, "not_supported_projection", None
    input_dim, output_dim = dimensions
    if input_dim < 128 or output_dim < 128:
        return False, "dimension_lt_128", dimensions
    # La tete CTC reste partagee. On adapte les projections internes de
    # l'encodeur, pas la couche qui definit le vocabulaire de sortie.
    lowered = name.lower()
    if re.search(r"(^|\.)(final_proj|output_projection|ctc_proj|classifier|lm_head)$", lowered):
        return False, "shared_output_head", dimensions
    return True, "eligible_internal_projection", dimensions


linear_inventory = []
eligible = []
for module_name, module in model.named_modules():
    dimensions = projection_dimensions(module)
    if dimensions is None:
        continue
    accepted, reason, dimensions = is_lora_eligible(module_name, module)
    input_dim, output_dim = dimensions
    record = {
        "module": module_name,
        "module_class": f"{type(module).__module__}.{type(module).__name__}",
        "in_features": int(input_dim),
        "out_features": int(output_dim),
        "weight_parameters": int(module.weight.numel()),
        "rank_unit_cost": int(input_dim + output_dim),
        "eligible": bool(accepted),
        "reason": reason,
    }
    linear_inventory.append(record)
    if accepted:
        eligible.append(record)

assert linear_inventory, (
    "Aucune projection lineaire compatible detectee (torch ou Fairseq2)."
)
assert eligible, "Aucune projection lineaire interne eligible pour LoRA."
assert len({record["module"] for record in eligible}) == len(eligible)


# -----------------------------------------------------------------------------
# 2. Allocation deterministe des rangs sous 9,9 M par adaptateur
# -----------------------------------------------------------------------------

minimum_cost = sum(record["rank_unit_cost"] * MIN_LORA_RANK for record in eligible)
assert minimum_cost <= PER_ADAPTER_BUDGET, (
    "Le rang minimal depasse le budget", minimum_cost, PER_ADAPTER_BUDGET
)

ranks = {record["module"]: MIN_LORA_RANK for record in eligible}
used = minimum_cost

# On augmente d'abord les couches au rang le plus faible. A rang egal, les
# matrices offrant le plus de poids de base par parametre LoRA sont prioritaires.
while True:
    candidates = [
        record for record in eligible
        if ranks[record["module"]] < MAX_LORA_RANK
        and used + record["rank_unit_cost"] <= PER_ADAPTER_BUDGET
    ]
    if not candidates:
        break
    chosen = min(
        candidates,
        key=lambda record: (
            ranks[record["module"]],
            -record["weight_parameters"] / record["rank_unit_cost"],
            record["module"],
        ),
    )
    ranks[chosen["module"]] += 1
    used += chosen["rank_unit_cost"]

for record in eligible:
    record["rank"] = int(ranks[record["module"]])
    record["adapter_parameters"] = int(record["rank"] * record["rank_unit_cost"])

adapter_parameters = sum(record["adapter_parameters"] for record in eligible)
assert adapter_parameters == used <= PER_ADAPTER_BUDGET
two_adapter_parameters = len(ADAPTER_NAMES) * adapter_parameters
total_neural_parameters = BASE_PARAMETER_COUNT + two_adapter_parameters
parameter_margin = OFFICIAL_PARAMETER_LIMIT - total_neural_parameters
assert total_neural_parameters < OFFICIAL_PARAMETER_LIMIT
assert parameter_margin >= 4_324_944, parameter_margin

plan_rows = [
    {
        "module": record["module"],
        "in_features": record["in_features"],
        "out_features": record["out_features"],
        "rank": record["rank"],
        "parameters_per_adapter": record["adapter_parameters"],
    }
    for record in eligible
]

contract = {
    "schema": 1,
    "purpose": "zero-init routed LoRA compatibility audit; no training",
    "base_checkpoint": str(BASE_CHECKPOINT),
    "base_weight_sha256": actual_weight_sha256,
    "base_manifest": str(base_manifest),
    "base_manifest_sha256": base_manifest_sha256,
    "model_card": model_card,
    "base_parameters": BASE_PARAMETER_COUNT,
    "official_parameter_limit": OFFICIAL_PARAMETER_LIMIT,
    "adapter_names": list(ADAPTER_NAMES),
    "per_adapter_budget": PER_ADAPTER_BUDGET,
    "minimum_rank": MIN_LORA_RANK,
    "maximum_rank": MAX_LORA_RANK,
    "eligibility": (
        "torch/Fairseq2 leaf projection with 2D [out,in] weight; in/out >=128; "
        "exclude embedding/conv/norm and exact final/output/ctc/classifier heads"
    ),
    "eligible_modules": len(eligible),
    "parameters_per_adapter": adapter_parameters,
    "two_adapter_parameters": two_adapter_parameters,
    "total_neural_parameters": total_neural_parameters,
    "parameter_margin": parameter_margin,
    "seed": LORA_SEED,
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "omnilingual_asr": package_version("omnilingual-asr"),
    "fairseq2": package_version("fairseq2"),
    "python": sys.version,
}
contract_sha256 = canonical_sha256(contract)
RUN_ROOT = REPORT_ROOT / contract_sha256[:16]
RUN_ROOT.mkdir(parents=True, exist_ok=True)
atomic_json(contract, RUN_ROOT / "contract.json")
pd.DataFrame(linear_inventory).to_parquet(
    RUN_ROOT / "linear_inventory.parquet", index=False
)
pd.DataFrame(plan_rows).to_parquet(RUN_ROOT / "lora_plan.parquet", index=False)


# -----------------------------------------------------------------------------
# 3. Wrapper LoRA route et injection reversible
# -----------------------------------------------------------------------------

class RoutedLoRAProjection(nn.Module):
    def __init__(self, base, rank, adapter_names, alpha=None):
        super().__init__()
        dimensions = projection_dimensions(base)
        assert dimensions is not None, type(base)
        input_dim, output_dim = dimensions
        self.base = base
        self.input_dim = int(input_dim)
        self.output_dim = int(output_dim)
        self.rank = int(rank)
        self.alpha = float(alpha if alpha is not None else rank)
        self.scaling = self.alpha / self.rank
        self.active_adapter = None
        self.lora_A = nn.ParameterDict()
        self.lora_B = nn.ParameterDict()
        for adapter_name in adapter_names:
            a = nn.Parameter(torch.empty(
                self.rank, self.input_dim,
                device=base.weight.device, dtype=base.weight.dtype,
            ))
            b = nn.Parameter(torch.zeros(
                self.output_dim, self.rank,
                device=base.weight.device, dtype=base.weight.dtype,
            ))
            nn.init.kaiming_uniform_(a, a=math.sqrt(5))
            self.lora_A[adapter_name] = a
            self.lora_B[adapter_name] = b

    def set_active_adapter(self, adapter_name):
        assert adapter_name is None or adapter_name in self.lora_A, adapter_name
        self.active_adapter = adapter_name

    def forward(self, inputs, *args, **kwargs):
        output = self.base(inputs, *args, **kwargs)
        if self.active_adapter is None:
            return output
        adapter_name = self.active_adapter
        update = F.linear(
            F.linear(inputs, self.lora_A[adapter_name]),
            self.lora_B[adapter_name],
        )
        return output + update * self.scaling


def parent_and_child(root, dotted_name):
    parts = dotted_name.split(".")
    parent = root
    for part in parts[:-1]:
        parent = getattr(parent, part)
    return parent, parts[-1]


def inject_lora(root, plan, adapter_names):
    wrappers = OrderedDict()
    for row in sorted(plan, key=lambda item: item["module"]):
        name = row["module"]
        parent, child = parent_and_child(root, name)
        base = getattr(parent, child)
        dimensions = projection_dimensions(base)
        assert dimensions == (row["in_features"], row["out_features"]), (
            name, type(base), dimensions, row
        )
        wrapper = RoutedLoRAProjection(base, row["rank"], adapter_names)
        setattr(parent, child, wrapper)
        wrappers[name] = wrapper
    return wrappers


def set_active_adapter(wrappers, adapter_name):
    for wrapper in wrappers.values():
        wrapper.set_active_adapter(adapter_name)


def set_trainable_adapter(root, wrappers, adapter_name):
    assert adapter_name in ADAPTER_NAMES
    for parameter in root.parameters():
        parameter.requires_grad_(False)
    for wrapper in wrappers.values():
        wrapper.lora_A[adapter_name].requires_grad_(True)
        wrapper.lora_B[adapter_name].requires_grad_(True)
    set_active_adapter(wrappers, adapter_name)
    trainable = sum(parameter.numel() for parameter in root.parameters() if parameter.requires_grad)
    assert trainable == adapter_parameters, (adapter_name, trainable, adapter_parameters)
    return trainable


def unwrap_lora(root, wrappers):
    for name in sorted(wrappers, key=lambda value: value.count("."), reverse=True):
        parent, child = parent_and_child(root, name)
        wrapper = getattr(parent, child)
        assert wrapper is wrappers[name] and isinstance(wrapper, RoutedLoRAProjection)
        setattr(parent, child, wrapper.base)


# -----------------------------------------------------------------------------
# 4. Probes acoustiques exactes avant/apres injection
# -----------------------------------------------------------------------------

assert LOCAL_AUDIO_ROOT.exists(), (
    f"Dev local absent : {LOCAL_AUDIO_ROOT}. Relancer d'abord 19.5 ; "
    "19.7a reutilise seulement quelques clips deja audites."
)


def shortest_probe_files(language, count):
    root = LOCAL_AUDIO_ROOT / language
    assert root.exists(), root
    candidates = []
    for path in sorted(root.glob("*.flac")):
        try:
            info = sf.info(path)
            if info.samplerate == 16000 and info.channels == 1 and info.frames > 0:
                candidates.append((info.frames / info.samplerate, path))
        except Exception:
            continue
    assert len(candidates) >= count, (language, len(candidates), count)
    return [path for _, path in sorted(candidates)[:count]]


probe_rows = []
for language in ADAPTER_NAMES:
    for path in shortest_probe_files(language, PROBES_PER_LANGUAGE):
        probe_rows.append({
            "language": language,
            "omni_code": omni_lang(language),
            "audio_path": str(path),
            "duration": float(sf.info(path).duration),
        })

tokenizer = load_tokenizer(model_card)
pipe = ASRInferencePipeline(None, model=model, tokenizer=tokenizer)


def capture_transcription_and_logits(pipe, audio_path, omni_code):
    captured = []
    original_forward = pipe.model.forward

    def spy(source_seqs, source_seq_lens, *args, **kwargs):
        output = original_forward(source_seqs, source_seq_lens, *args, **kwargs)
        assert isinstance(output, tuple) and len(output) >= 2
        logits, layout = output[0], output[1]
        assert logits.shape[0] == 1 and hasattr(layout, "seq_lens")
        length = layout.seq_lens[0]
        length = int(length.item() if hasattr(length, "item") else length)
        captured.append(logits[0, :length].detach().cpu().clone())
        return output

    pipe.model.forward = spy
    try:
        with torch.inference_mode():
            texts = pipe.transcribe([str(audio_path)], lang=[omni_code], batch_size=1)
    finally:
        pipe.model.forward = original_forward
    assert len(texts) == 1 and len(captured) == 1
    return str(texts[0]), captured[0]


print("Probes de reference step_1250...")
baseline = {}
for row in probe_rows:
    key = (row["language"], row["audio_path"])
    baseline[key] = capture_transcription_and_logits(
        pipe, row["audio_path"], row["omni_code"]
    )

for parameter in model.parameters():
    parameter.requires_grad_(False)
wrappers = inject_lora(model, plan_rows, ADAPTER_NAMES)

actual_adapter_parameter_count = sum(
    parameter.numel()
    for wrapper in wrappers.values()
    for group in (wrapper.lora_A, wrapper.lora_B)
    for parameter in group.values()
)
assert actual_adapter_parameter_count == two_adapter_parameters

# Tous les objets Parameter de la base doivent etre les memes, a la meme adresse.
injected_base_identity = {
    id(parameter): {
        "numel": int(parameter.numel()),
        "data_ptr": int(parameter.data_ptr()),
        "dtype": str(parameter.dtype),
    }
    for wrapper in wrappers.values()
    for parameter in wrapper.base.parameters(recurse=False)
}
# Certaines lineaires ont seulement weight, d'autres weight+bias. Les autres poids
# de base sont verifies apres retrait via l'ensemble complet des identites.
for parameter_id, metadata in injected_base_identity.items():
    assert base_parameter_identity[parameter_id] == metadata

trainable_counts = {
    adapter_name: set_trainable_adapter(model, wrappers, adapter_name)
    for adapter_name in ADAPTER_NAMES
}

comparisons = []
for adapter_name in ADAPTER_NAMES:
    set_active_adapter(wrappers, adapter_name)
    for row in probe_rows:
        # On teste chaque route sur les deux langues : a zero, aucune route ne doit
        # modifier le modele partage, meme si un mauvais label lui etait passe.
        key = (row["language"], row["audio_path"])
        reference_text, reference_logits = baseline[key]
        routed_text, routed_logits = capture_transcription_and_logits(
            pipe, row["audio_path"], row["omni_code"]
        )
        same_shape = tuple(reference_logits.shape) == tuple(routed_logits.shape)
        tensor_equal = bool(same_shape and torch.equal(reference_logits, routed_logits))
        max_abs_difference = (
            float((reference_logits - routed_logits).abs().max().item())
            if same_shape and reference_logits.numel() else float("inf")
        )
        comparisons.append({
            "active_adapter": adapter_name,
            "probe_language": row["language"],
            "audio_path": row["audio_path"],
            "duration": row["duration"],
            "same_shape": same_shape,
            "tensor_equal": tensor_equal,
            "max_abs_difference": max_abs_difference,
            "text_equal": reference_text == routed_text,
            "reference_text": reference_text,
            "routed_text": routed_text,
        })

assert all(row["tensor_equal"] for row in comparisons), comparisons
assert all(row["text_equal"] for row in comparisons), comparisons
assert all(row["max_abs_difference"] == 0.0 for row in comparisons), comparisons

# Retrait integral des wrappers et verification des identites de TOUS les poids.
set_active_adapter(wrappers, None)
unwrap_lora(model, wrappers)
restored_parameter_identity = {
    id(parameter): {
        "numel": int(parameter.numel()),
        "data_ptr": int(parameter.data_ptr()),
        "dtype": str(parameter.dtype),
    }
    for parameter in model.parameters()
}
assert restored_parameter_identity == base_parameter_identity
assert sum(parameter.numel() for parameter in model.parameters()) == BASE_PARAMETER_COUNT

# Une derniere sonde apres retrait couvre aussi la reversibilite fonctionnelle.
first = probe_rows[0]
restored_text, restored_logits = capture_transcription_and_logits(
    pipe, first["audio_path"], first["omni_code"]
)
reference_text, reference_logits = baseline[(first["language"], first["audio_path"])]
unwrap_functionally_exact = bool(
    restored_text == reference_text and torch.equal(restored_logits, reference_logits)
)
assert unwrap_functionally_exact


# -----------------------------------------------------------------------------
# 5. Rapport final — aucun poids LoRA n'est publie a ce stade
# -----------------------------------------------------------------------------

rank_histogram = dict(sorted(Counter(row["rank"] for row in plan_rows).items()))
report = {
    "schema": 1,
    "created_at": now_iso(),
    "status": "PASS",
    "meaning": (
        "architecture compatible; zero-init exact; parameter budget valid; "
        "training not performed"
    ),
    "contract_sha256": contract_sha256,
    "base_checkpoint": str(BASE_CHECKPOINT),
    "base_weight_sha256": actual_weight_sha256,
    "base_parameters": BASE_PARAMETER_COUNT,
    "eligible_linear_modules": len(eligible),
    "excluded_linear_modules": len(linear_inventory) - len(eligible),
    "rank_histogram": {str(key): value for key, value in rank_histogram.items()},
    "minimum_rank_used": min(ranks.values()),
    "maximum_rank_used": max(ranks.values()),
    "parameters_per_adapter": adapter_parameters,
    "adapter_names": list(ADAPTER_NAMES),
    "two_adapter_parameters": two_adapter_parameters,
    "total_neural_parameters": total_neural_parameters,
    "official_parameter_limit": OFFICIAL_PARAMETER_LIMIT,
    "parameter_margin": parameter_margin,
    "runtime_adapter_parameters": actual_adapter_parameter_count,
    "trainable_parameters_by_route": trainable_counts,
    "zero_init_comparisons": comparisons,
    "zero_init_exact": True,
    "base_parameter_objects_preserved": True,
    "unwrap_functionally_exact": unwrap_functionally_exact,
    "final_checkpoint_created": False,
    "active_routing_modified": False,
    "next_step": (
        "19.7b: train kln and mas LoRA separately from step_1250 with the base frozen; "
        "evaluate before any test inference"
    ),
}
report_path = RUN_ROOT / "lora_compatibility_audit_v1.json"
atomic_json(report, report_path)

summary = pd.DataFrame([{
    "base_parameters": BASE_PARAMETER_COUNT,
    "eligible_linears": len(eligible),
    "rank_min": min(ranks.values()),
    "rank_max": max(ranks.values()),
    "parameters_kln": adapter_parameters,
    "parameters_mas": adapter_parameters,
    "total_neural": total_neural_parameters,
    "margin_under_1b": parameter_margin,
    "zero_init_exact": True,
    "reversible": unwrap_functionally_exact,
}])
print("\n=== AUDIT LORA 19.7a ===")
display(summary)
print("Rangs :", rank_histogram)
print("STATUT 19.7a : PASS")
print("Rapport :", report_path)
print("Plan    :", RUN_ROOT / "lora_plan.parquet")
print("Aucun checkpoint final n'a ete cree ou modifie.")
print("Etape suivante : 19.7b, entrainement LoRA kln/mas avec la base gelee.")

del pipe, tokenizer, wrappers, baseline, restored_logits, reference_logits, model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# 19.7b — Smoke test d'entrainement LoRA Kalenjin/Maasai (20 pas)
#
# Cette cellule valide l'integration LoRA dans la recette officielle OmniASR.
# Elle NE lance PAS encore le vrai run et NE modifie aucun checkpoint final.
#
# Invariants verifies pour kln puis mas :
#   - checkpoint seed = step_1250 audite par 19.7a ;
#   - exactement 9 898 496 parametres LoRA entrainables ;
#   - 975 675 056 parametres de base geles ;
#   - les poids B LoRA quittent zero apres 20 pas ;
#   - chaque tenseur de base reste exactement identique au seed ;
#   - total final theorique des deux adaptateurs = 995 472 048 < 1 milliard.
#
# Un L4 suffit. Duree typique : 15 a 35 minutes pour les deux langues.

import gc
import hashlib
import json
import os
import re
import shutil
import signal
import subprocess
import sys
import textwrap
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import torch


# -----------------------------------------------------------------------------
# 0. Contrat issu de 19.7a
# -----------------------------------------------------------------------------

assert torch.cuda.is_available(), "19.7b requiert un GPU ; un L4 suffit."
assert "FT_CONFIG" in globals(), "Relancer les cellules 1.x -> 2.4 avant 19.7b."
assert "OMNI_REPO_DIR" in globals() and Path(OMNI_REPO_DIR).is_dir(), OMNI_REPO_DIR
assert "write_dataset_card" in globals(), "Relancer la cellule 2.4."

FT_ROOT = Path(
    "/content/afrivoices_project/"
    "finetune_runs/experiment_run"
)
AUDIT_ROOT = FT_ROOT / "reports/lora_compatibility_audit_v1/REPLACE_WITH_LOCAL_RUN_ID"
AUDIT_PATH = AUDIT_ROOT / "lora_compatibility_audit_v1.json"
PLAN_PATH = AUDIT_ROOT / "lora_plan.parquet"
CONTRACT_PATH = AUDIT_ROOT / "contract.json"
BASE_CHECKPOINT = (
    FT_ROOT / "balanced_joint_v4/candidates/REPLACE_WITH_LOCAL_RUN_ID/step_1250"
)
LOCAL_BASE_WEIGHT = Path("/content/low_rank_adapter_sources_v1/base_step1250.pt")

BASE_PARAMETERS = 975_675_056
PARAMETERS_PER_ADAPTER = 9_898_496
TWO_ADAPTER_PARAMETERS = 19_796_992
TOTAL_NEURAL_PARAMETERS = 995_472_048
PARAMETER_LIMIT = 1_000_000_000
ADAPTER_LANGUAGES = ("kln", "mas")

SMOKE_STEPS = 20
SMOKE_LR = 5e-5
SMOKE_MAX_NUM_ELEMENTS = 1_024_000
SMOKE_GRAD_ACCUM = 4
SMOKE_ROOT = Path("/content/lora_training_smoke_v1")
REPORT_PATH = FT_ROOT / "reports/lora_training_smoke_v1.json"

for required in (AUDIT_PATH, PLAN_PATH, CONTRACT_PATH, LOCAL_BASE_WEIGHT):
    assert Path(required).exists(), required

audit = json.load(open(AUDIT_PATH, encoding="utf-8"))
contract_19_7a = json.load(open(CONTRACT_PATH, encoding="utf-8"))
assert audit["status"] == "PASS" and audit["zero_init_exact"] is True
assert audit["base_parameters"] == BASE_PARAMETERS
assert audit["parameters_per_adapter"] == PARAMETERS_PER_ADAPTER
assert audit["two_adapter_parameters"] == TWO_ADAPTER_PARAMETERS
assert audit["total_neural_parameters"] == TOTAL_NEURAL_PARAMETERS
assert TOTAL_NEURAL_PARAMETERS < PARAMETER_LIMIT
assert audit["contract_sha256"].startswith("REPLACE_WITH_LOCAL_RUN_ID")


def now_iso():
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path, block_size=16 << 20):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()


def atomic_json(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".tmp-{os.getpid()}")
    with open(temporary, "w", encoding="utf-8") as handle:
        json.dump(value, handle, ensure_ascii=False, indent=2, default=str)
    os.replace(temporary, path)
    print("json ->", path)


base_weight_sha256 = sha256_file(LOCAL_BASE_WEIGHT)
assert base_weight_sha256 == audit["base_weight_sha256"]
plan = pd.read_parquet(PLAN_PATH)
assert len(plan) == audit["eligible_linear_modules"] == 289
assert int(plan.parameters_per_adapter.sum()) == PARAMETERS_PER_ADAPTER
assert set(plan["rank"].unique()) == {8, 9}


# -----------------------------------------------------------------------------
# 1. Retrouver/restaurer le dataset V4 et creer un TSV mono-langue
# -----------------------------------------------------------------------------

local_dataset_root_candidates = [
    Path("/content/afrivoices_balanced_v4/dataset"),
    Path("/content/local-scratch/afrivoices_balanced_v4/dataset"),
]
SOURCE_DATASET_ROOT = next(
    (path for path in local_dataset_root_candidates
     if (path / "version=0").is_dir()
     and (path / "language_distribution_0.tsv").exists()),
    None,
)

if SOURCE_DATASET_ROOT is None:
    drive_dataset_candidates = [
        Path("/content/afrivoices_project/"
             "dataset_caches/afrivoices_balanced_v4_dataset"),
        Path("/content/afrivoices_project/"
             "afrivoices_balanced_v4/dataset"),
        Path("/content/afrivoices_project/dataset"),
        FT_ROOT / "afrivoices_balanced_v4/dataset",
        FT_ROOT / "balanced_joint_v4/afrivoices_balanced_v4/dataset",
    ]
    SOURCE_DATASET_ROOT = next(
        (path for path in drive_dataset_candidates
         if (path / "version=0").is_dir()
         and (path / "language_distribution_0.tsv").exists()),
        None,
    )

if SOURCE_DATASET_ROOT is None:
    # Dernier recours borne : recherche des dossiers nommes exactement
    # afrivoices_balanced_v4/dataset, sans parcourir indefiniment Drive.
    search = subprocess.run(
        ["find", "/content/persistent_storage", "-maxdepth", "7", "-type", "d",
         "-path", "*/afrivoices_balanced_v4/dataset"],
        capture_output=True, text=True, timeout=300,
    )
    discovered = [Path(line.strip()) for line in search.stdout.splitlines() if line.strip()]
    SOURCE_DATASET_ROOT = next(
        (path for path in discovered
         if (path / "version=0").is_dir()
         and (path / "language_distribution_0.tsv").exists()),
        None,
    )

assert SOURCE_DATASET_ROOT is not None, (
    "Sauvegarde Drive du dataset V4 introuvable. Afficher les dossiers contenant "
    "afrivoices_balanced_v4 afin d'ajouter son chemin aux candidats."
)
print("Dataset V4 source :", SOURCE_DATASET_ROOT)

V4_TSV_PATH = SOURCE_DATASET_ROOT / "language_distribution_0.tsv"
summary = pd.read_csv(V4_TSV_PATH, sep="\t")
assert "language" in summary.columns, summary.columns.tolist()

SMOKE_ROOT.mkdir(parents=True, exist_ok=True)
language_tsv = {}
for language in ADAPTER_LANGUAGES:
    omni_code = omni_lang(language)
    filtered = summary[summary.language.astype(str).eq(omni_code)].copy()
    assert len(filtered) == 1, (language, omni_code, filtered)
    path = SMOKE_ROOT / f"language_distribution_{language}.tsv"
    filtered.to_csv(path, sep="\t", index=False)
    language_tsv[language] = str(path)
    print(language, "->", filtered.to_dict("records"))

# Si la source est sur Drive, ne copier que kln/mas (train+dev). Le dataset
# complet fait ~46,6 Gio et n'est pas necessaire pour ce smoke test.
if str(SOURCE_DATASET_ROOT).startswith("/content/drive/"):
    V4_DATASET_ROOT = Path("/content/lora_training_dataset_v1")
    V4_DATA_VERSION_ROOT = V4_DATASET_ROOT / "version=0"
    source_version = SOURCE_DATASET_ROOT / "version=0"
    corpus_dirs = sorted(path for path in source_version.glob("corpus=*") if path.is_dir())
    assert len(corpus_dirs) == 1, corpus_dirs
    source_corpus = corpus_dirs[0]
    destination_corpus = V4_DATA_VERSION_ROOT / source_corpus.name

    def copy_partition(split, language):
        code = omni_lang(language)
        source = source_corpus / f"split={split}" / f"language={code}"
        destination = destination_corpus / f"split={split}" / f"language={code}"
        assert source.is_dir(), source
        destination.mkdir(parents=True, exist_ok=True)
        print(f"Restauration Drive -> local : {language}/{split}")
        subprocess.run([
            "rsync", "-a", "--partial", "--append-verify", "--info=progress2",
            str(source) + "/", str(destination) + "/",
        ], check=True)
        source_files = sorted(path for path in source.rglob("*") if path.is_file())
        destination_files = sorted(path for path in destination.rglob("*") if path.is_file())
        assert len(source_files) == len(destination_files) > 0, (
            source, len(source_files), destination, len(destination_files)
        )
        source_sizes = {
            str(path.relative_to(source)): int(path.stat().st_size) for path in source_files
        }
        destination_sizes = {
            str(path.relative_to(destination)): int(path.stat().st_size)
            for path in destination_files
        }
        assert source_sizes == destination_sizes, f"Copie incomplete : {language}/{split}"
        return sum(source_sizes.values())

    copied_bytes = 0
    for language in ADAPTER_LANGUAGES:
        for split in ("train", "dev"):
            copied_bytes += copy_partition(split, language)
    print(
        "Dataset LoRA local pret :", V4_DATA_VERSION_ROOT,
        "|", round(copied_bytes / 2**30, 2), "Gio",
    )
else:
    V4_DATASET_ROOT = SOURCE_DATASET_ROOT
    V4_DATA_VERSION_ROOT = V4_DATASET_ROOT / "version=0"

# La carte globale ne change que le chemin des donnees, pas les poids finaux.
write_dataset_card(data_root=str(V4_DATA_VERSION_ROOT))


# -----------------------------------------------------------------------------
# 2. Carte seed step_1250
# -----------------------------------------------------------------------------

import omnilingual_asr as oa

CARD_NAME = "omniASR_CTC_1B_v2_step1250_lora_seed_v1"
CARD_PATH = (
    Path(oa.__file__).parent / "cards/models/afrivoices_step1250_lora_seed_v1.yaml"
)
CARD_PATH.parent.mkdir(parents=True, exist_ok=True)
CARD_TEXT = (
    f'name: {CARD_NAME}\n'
    f'base: {FT_CONFIG["model_card"]}\n'
    f'checkpoint: "file://{LOCAL_BASE_WEIGHT}"\n'
)
CARD_PATH.write_text(CARD_TEXT, encoding="utf-8")
print("Carte seed LoRA :", CARD_PATH)


# -----------------------------------------------------------------------------
# 3. Recette personnalisee : injection avant creation de l'optimiseur
# -----------------------------------------------------------------------------

CUSTOM_PACKAGE = Path(OMNI_REPO_DIR) / "workflows/recipes/wav2vec2/asr_lora"
CUSTOM_PACKAGE.mkdir(parents=True, exist_ok=True)
(CUSTOM_PACKAGE / "__init__.py").write_text("", encoding="utf-8")

RECIPE_SOURCE = r'''
from __future__ import annotations

import math
import os
import re
from collections import OrderedDict

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from fairseq2.logging import log
from fairseq2.recipe.base import RecipeContext
from fairseq2.recipe.model import RecipeModel

from ..asr.recipe import Wav2Vec2AsrRecipe


def projection_dimensions(module):
    weight = getattr(module, "weight", None)
    if not isinstance(weight, torch.Tensor) or weight.ndim != 2:
        return None
    class_name = type(module).__name__.lower()
    class_module = type(module).__module__.lower()
    forbidden = ("embedding", "embed", "convolution", "conv", "norm")
    if any(token in class_name for token in forbidden):
        return None
    if isinstance(module, (nn.Embedding, nn.Conv1d, nn.Conv2d, nn.Conv3d)):
        return None
    if any(True for _ in module.children()):
        return None
    input_dim = getattr(module, "in_features", None)
    output_dim = getattr(module, "out_features", None)
    if input_dim is None:
        input_dim = getattr(module, "input_dim", None)
    if output_dim is None:
        output_dim = getattr(module, "output_dim", None)
    input_dim = int(input_dim) if input_dim is not None else int(weight.shape[1])
    output_dim = int(output_dim) if output_dim is not None else int(weight.shape[0])
    if tuple(weight.shape) != (output_dim, input_dim):
        return None
    if not (
        isinstance(module, nn.Linear)
        or "projection" in class_name
        or "linear" in class_name
        or "projection" in class_module
    ):
        return None
    return input_dim, output_dim


class RoutedLoRAProjection(nn.Module):
    def __init__(self, base, rank, adapter_name, alpha=None):
        super().__init__()
        dimensions = projection_dimensions(base)
        assert dimensions is not None, type(base)
        input_dim, output_dim = dimensions
        self.base = base
        self.rank = int(rank)
        self.alpha = float(alpha if alpha is not None else rank)
        self.scaling = self.alpha / self.rank
        self.adapter_name = str(adapter_name)
        self.lora_A = nn.Parameter(torch.empty(
            self.rank, input_dim, device=base.weight.device, dtype=base.weight.dtype
        ))
        self.lora_B = nn.Parameter(torch.zeros(
            output_dim, self.rank, device=base.weight.device, dtype=base.weight.dtype
        ))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

    def forward(self, inputs, *args, **kwargs):
        output = self.base(inputs, *args, **kwargs)
        update = F.linear(F.linear(inputs, self.lora_A), self.lora_B)
        return output + update * self.scaling


def parent_and_child(root, dotted_name):
    parts = dotted_name.split(".")
    parent = root
    for part in parts[:-1]:
        parent = getattr(parent, part)
    return parent, parts[-1]


def inject_lora(root, plan, adapter_name):
    wrappers = OrderedDict()
    for row in sorted(plan, key=lambda item: item["module"]):
        name = row["module"]
        parent, child = parent_and_child(root, name)
        base = getattr(parent, child)
        dimensions = projection_dimensions(base)
        expected = (int(row["in_features"]), int(row["out_features"]))
        assert dimensions == expected, (name, type(base), dimensions, expected)
        wrapper = RoutedLoRAProjection(base, int(row["rank"]), adapter_name)
        setattr(parent, child, wrapper)
        wrappers[name] = wrapper
    return wrappers


class LoRAWav2Vec2AsrRecipe(Wav2Vec2AsrRecipe):
    def prepare_model(self, context: RecipeContext, model: RecipeModel) -> RecipeModel:
        model = super().prepare_model(context, model)
        root = model.module
        plan_path = os.environ["AFRIVOICES_LORA_PLAN"]
        adapter_name = os.environ["AFRIVOICES_LORA_ADAPTER"]
        expected_base = int(os.environ["AFRIVOICES_LORA_BASE_PARAMETERS"])
        expected_adapter = int(os.environ["AFRIVOICES_LORA_PARAMETERS"])
        plan = pd.read_parquet(plan_path).to_dict("records")

        base_count = sum(parameter.numel() for parameter in root.parameters())
        assert base_count == expected_base, (base_count, expected_base)
        for parameter in root.parameters():
            parameter.requires_grad_(False)

        wrappers = inject_lora(root, plan, adapter_name)
        trainable = sum(
            parameter.numel() for parameter in root.parameters()
            if parameter.requires_grad
        )
        total = sum(parameter.numel() for parameter in root.parameters())
        assert len(wrappers) == len(plan)
        assert trainable == expected_adapter, (trainable, expected_adapter)
        assert total == expected_base + expected_adapter, (
            total, expected_base + expected_adapter
        )
        log.info(
            "AFRIVOICES_LORA_READY adapter={} wrappers={} base={} trainable={} total={}",
            adapter_name, len(wrappers), base_count, trainable, total,
        )
        return model
'''

MAIN_SOURCE = r'''
from fairseq2.recipe.cli import train_main
from .recipe import LoRAWav2Vec2AsrRecipe

recipe = LoRAWav2Vec2AsrRecipe()
train_main(recipe)
'''

(CUSTOM_PACKAGE / "recipe.py").write_text(
    textwrap.dedent(RECIPE_SOURCE).lstrip(), encoding="utf-8"
)
(CUSTOM_PACKAGE / "__main__.py").write_text(
    textwrap.dedent(MAIN_SOURCE).lstrip(), encoding="utf-8"
)
compile((CUSTOM_PACKAGE / "recipe.py").read_text(encoding="utf-8"),
        str(CUSTOM_PACKAGE / "recipe.py"), "exec")
compile((CUSTOM_PACKAGE / "__main__.py").read_text(encoding="utf-8"),
        str(CUSTOM_PACKAGE / "__main__.py"), "exec")
print("Recette LoRA :", CUSTOM_PACKAGE)


# -----------------------------------------------------------------------------
# 4. YAML de smoke test et execution sequentielle
# -----------------------------------------------------------------------------

def write_smoke_yaml(language):
    path = SMOKE_ROOT / f"recipe_{language}.yaml"
    dtype_value = str(FT_CONFIG.get("mixed_precision", "torch.bfloat16"))
    if dtype_value.startswith("torch."):
        dtype_value = dtype_value.split(".", 1)[1]
    lines = [
        "model:", f'  name: "{CARD_NAME}"', "",
        "dataset:", '  name: "afrivoices_dataset"',
        '  train_split: "train"', '  valid_split: "dev"',
        '  storage_mode: "MIXTURE_PARQUET"', '  task_mode: "ASR"',
        "  mixture_parquet_storage_config:",
        f'    dataset_summary_path: "{language_tsv[language]}"',
        "    beta_corpus: 1.0", "    beta_language: 0.0",
        "    fragment_loading:", "      cache: False",
        "  asr_task_config:", "    max_audio_len: 640000",
        f"    max_num_elements: {SMOKE_MAX_NUM_ELEMENTS}",
        "    batch_shuffle_window: 1", "    normalize_audio: true",
        "    example_shuffle_window: 1000", "",
        "tokenizer:", f'  name: "{FT_CONFIG["tokenizer_name"]}"', "",
        "optimizer:", "  config:", f"    lr: {SMOKE_LR:.8e}", "",
        "trainer:", "  freeze_encoder_for_n_steps: 0", "  mixed_precision:",
        f'    dtype: "{dtype_value}"', "  grad_accumulation:",
        f"    num_batches: {SMOKE_GRAD_ACCUM}", "",
        "regime:", f"  num_steps: {SMOKE_STEPS}",
        f"  validate_every_n_steps: {SMOKE_STEPS}",
        f"  validate_after_n_steps: {SMOKE_STEPS}",
        f"  checkpoint_every_n_steps: {SMOKE_STEPS}",
        f"  checkpoint_after_n_steps: {SMOKE_STEPS}",
        "  publish_metrics_every_n_steps: 5",
        "  publish_metrics_after_n_steps: 5",
    ]
    path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return path


def run_streaming(command, cwd, env, log_path):
    process = subprocess.Popen(
        command, cwd=str(cwd), env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, start_new_session=True,
    )
    started = time.time()
    try:
        with open(log_path, "w", encoding="utf-8") as log:
            for line in process.stdout:
                log.write(line)
                log.flush()
                lowered = line.lower()
                if any(token in lowered for token in (
                    "afrivoices_lora", "step", "loss", "wer", "error", "oom", "checkpoint"
                )):
                    print(line.rstrip()[:260])
        returncode = process.wait()
    except BaseException:
        if process.poll() is None:
            try:
                os.killpg(process.pid, signal.SIGINT)
                process.wait(timeout=30)
            except Exception:
                try:
                    os.killpg(process.pid, signal.SIGKILL)
                except Exception:
                    pass
        raise
    return {
        "returncode": returncode,
        "elapsed_minutes": round((time.time() - started) / 60, 2),
        "log_path": str(log_path),
    }


def latest_complete_checkpoint(root):
    candidates = []
    for path in Path(root).rglob("step_*"):
        match = re.fullmatch(r"step_(\d+)", path.name)
        model_files = list((path / "model").rglob("*")) if (path / "model").is_dir() else []
        model_files = [item for item in model_files if item.is_file()]
        if match and len(model_files) == 1 and (path / "trainer").is_dir():
            candidates.append((int(match.group(1)), path, model_files[0]))
    assert candidates, root
    return max(candidates, key=lambda item: item[0])


def load_flat_state(path):
    raw = torch.load(str(path), map_location="cpu", weights_only=False, mmap=True)
    state = raw.get("model", raw) if isinstance(raw, dict) else raw
    assert isinstance(state, dict) and state
    return raw, state


def canonical_base_key(checkpoint_key):
    # Chaque projection adaptee devient <module>.base.weight/bias dans le wrapper.
    return checkpoint_key.replace(".base.weight", ".weight").replace(
        ".base.bias", ".bias"
    )


def object_owns_cuda(value):
    """Detecte les principaux objets globaux qui retiennent un modele/tensor CUDA."""
    try:
        if isinstance(value, torch.Tensor):
            return bool(value.is_cuda)
        if isinstance(value, torch.nn.Module):
            return any(parameter.is_cuda for parameter in value.parameters()) or any(
                buffer.is_cuda for buffer in value.buffers()
            )
        if isinstance(value, dict):
            return any(object_owns_cuda(item) for item in list(value.values())[:100])
        if isinstance(value, (list, tuple, set)):
            return any(object_owns_cuda(item) for item in list(value)[:100])
        for attribute in ("model", "module", "_model", "pipeline", "pipe"):
            child = getattr(value, attribute, None)
            if child is not None and child is not value and object_owns_cuda(child):
                return True
    except Exception:
        return False
    return False


protected_globals = {
    "torch", "FT_CONFIG", "OMNI_REPO_DIR", "write_dataset_card", "omni_lang",
    "object_owns_cuda", "protected_globals",
}
released_cuda_globals = []
for variable, value in list(globals().items()):
    if variable.startswith("__") or variable in protected_globals:
        continue
    if object_owns_cuda(value):
        released_cuda_globals.append(variable)
        globals().pop(variable, None)

# Caches historiques connus qui peuvent retenir indirectement un pipeline.
for variable in (
    "TOPUP_PIPES", "DECODER_CACHE", "_DECODER_CACHE", "FINAL_DECODERS",
    "_probe_pipe", "_probe_model", "_pipe_ft", "_pipe_lm", "_pipe_joint",
):
    value = globals().pop(variable, None)
    try:
        value.clear()
    except Exception:
        pass

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
kernel_cuda_allocated = int(torch.cuda.memory_allocated())
kernel_cuda_reserved = int(torch.cuda.memory_reserved())
print("Objets CUDA globaux supprimes :", released_cuda_globals)
print(
    "CUDA noyau apres nettoyage :",
    round(kernel_cuda_allocated / 2**30, 3), "Gio alloues |",
    round(kernel_cuda_reserved / 2**30, 3), "Gio reserves",
)

# Le smoke test LoRA utilise un micro-batch tres conservateur. Jusqu'a 3 Gio
# residuels dans le noyau restent acceptables sur un L4 de 24 Gio.
assert kernel_cuda_allocated < 3 * 2**30, (
    "Plus de 3 Gio restent reellement alloues dans le noyau. Executer la petite "
    "cellule de nettoyage fournie, puis relancer 19.7b."
)

results = []
base_raw, base_state = load_flat_state(LOCAL_BASE_WEIGHT)

for language in ADAPTER_LANGUAGES:
    print("\n" + "=" * 76)
    print("SMOKE LoRA", language)
    output_dir = SMOKE_ROOT / f"output_{language}"
    yaml_path = write_smoke_yaml(language)
    log_path = SMOKE_ROOT / f"smoke_{language}.log"
    env = os.environ.copy()
    env.update({
        "AFRIVOICES_LORA_PLAN": str(PLAN_PATH),
        "AFRIVOICES_LORA_ADAPTER": language,
        "AFRIVOICES_LORA_BASE_PARAMETERS": str(BASE_PARAMETERS),
        "AFRIVOICES_LORA_PARAMETERS": str(PARAMETERS_PER_ADAPTER),
        "PYTHONUNBUFFERED": "1",
    })
    command = [
        sys.executable, "-m", "workflows.recipes.wav2vec2.asr_lora",
        str(output_dir), "--config-file", str(yaml_path),
    ]
    reusable = None
    if output_dir.exists() and log_path.exists():
        try:
            previous_step, previous_checkpoint, previous_weight = \
                latest_complete_checkpoint(output_dir)
            previous_log = log_path.read_text(encoding="utf-8", errors="ignore")
            if (
                previous_step == SMOKE_STEPS
                and "AFRIVOICES_LORA_READY" in previous_log
                and "Task finished" in previous_log
                and "out of memory" not in previous_log.lower()
                and re.search(r"\bnan\b", previous_log.lower()) is None
            ):
                reusable = (previous_step, previous_checkpoint, previous_weight)
        except Exception:
            reusable = None

    if reusable is not None:
        print("[reutilise] smoke", language, "deja termine :", reusable[1])
        run = {
            "returncode": 0,
            "elapsed_minutes": 0.0,
            "log_path": str(log_path),
            "reused_completed_smoke": True,
        }
    else:
        shutil.rmtree(output_dir, ignore_errors=True)
        run = run_streaming(command, OMNI_REPO_DIR, env, log_path)

    full_log = log_path.read_text(encoding="utf-8", errors="ignore")
    tail = full_log[-30000:]
    assert run["returncode"] == 0, tail
    assert "AFRIVOICES_LORA_READY" in full_log, tail[-5000:]
    assert re.search(r"\bnan\b", full_log.lower()) is None, tail[-5000:]
    assert "out of memory" not in full_log.lower(), tail[-5000:]

    step, checkpoint, checkpoint_weight = latest_complete_checkpoint(output_dir)
    assert step == SMOKE_STEPS, (step, checkpoint)
    trained_raw, trained_state = load_flat_state(checkpoint_weight)

    lora_a_keys = sorted(key for key in trained_state if key.endswith(".lora_A"))
    lora_b_keys = sorted(key for key in trained_state if key.endswith(".lora_B"))
    assert len(lora_a_keys) == len(lora_b_keys) == len(plan) == 289
    stored_lora_parameters = sum(
        trained_state[key].numel() for key in lora_a_keys + lora_b_keys
    )
    assert stored_lora_parameters == PARAMETERS_PER_ADAPTER
    changed_b = sum(
        int(torch.count_nonzero(trained_state[key]).item()) for key in lora_b_keys
    )
    assert changed_b > 0, "Les matrices LoRA B sont encore toutes nulles apres 20 pas."

    # Verification exhaustive : aucun poids de base n'a change.
    checked_base_tensors = 0
    checked_base_parameters = 0
    for key, value in trained_state.items():
        if key in lora_a_keys or key in lora_b_keys:
            continue
        canonical = canonical_base_key(key)
        assert canonical in base_state, (key, canonical)
        reference = base_state[canonical]
        assert value.shape == reference.shape, (key, value.shape, reference.shape)
        assert torch.equal(value.cpu(), reference.cpu()), f"Poids de base modifie : {key}"
        checked_base_tensors += 1
        checked_base_parameters += int(value.numel())
    assert checked_base_parameters == BASE_PARAMETERS, (
        checked_base_parameters, BASE_PARAMETERS
    )

    record = {
        "language": language,
        "status": "PASS",
        **run,
        "checkpoint": str(checkpoint),
        "checkpoint_weight": str(checkpoint_weight),
        "step": step,
        "lora_A_tensors": len(lora_a_keys),
        "lora_B_tensors": len(lora_b_keys),
        "lora_parameters": stored_lora_parameters,
        "nonzero_lora_B_elements": changed_b,
        "base_tensors_checked": checked_base_tensors,
        "base_parameters_checked": checked_base_parameters,
        "base_bitwise_unchanged": True,
    }
    results.append(record)
    print("PASS", language, "|", record)
    del trained_raw, trained_state
    gc.collect()

del base_raw, base_state
gc.collect()

final_report = {
    "schema": 1,
    "created_at": now_iso(),
    "status": "PASS",
    "purpose": "20-step LoRA integration smoke test; not a final model",
    "audit_19_7a": str(AUDIT_PATH),
    "audit_19_7a_contract_sha256": audit["contract_sha256"],
    "base_checkpoint": str(BASE_CHECKPOINT),
    "base_weight_sha256": base_weight_sha256,
    "parameters_per_adapter": PARAMETERS_PER_ADAPTER,
    "two_adapter_parameters": TWO_ADAPTER_PARAMETERS,
    "total_neural_parameters": TOTAL_NEURAL_PARAMETERS,
    "parameter_limit": PARAMETER_LIMIT,
    "smoke_steps": SMOKE_STEPS,
    "learning_rate": SMOKE_LR,
    "max_num_elements": SMOKE_MAX_NUM_ELEMENTS,
    "grad_accumulation": SMOKE_GRAD_ACCUM,
    "results": results,
    "final_checkpoint_created": False,
    "active_routing_modified": False,
    "next_step": "19.7c full LoRA training with checkpoints every 250 steps",
}
atomic_json(final_report, REPORT_PATH)

print("\n=== RESULTAT 19.7b ===")
print(pd.DataFrame(results)[[
    "language", "status", "step", "elapsed_minutes", "lora_parameters",
    "nonzero_lora_B_elements", "base_parameters_checked", "base_bitwise_unchanged",
]])
print("STATUT 19.7b : PASS")
print("Rapport :", REPORT_PATH)
print("Les checkpoints sous /content sont seulement des preuves de smoke test.")
print("Ne pas lancer 19.7c avant lecture de ce rapport.")

In [ ]:
# 19.7c — Entrainement LoRA complet Kalenjin + Maasai depuis step_1250
#
# Deux runs independants, un seul adaptateur actif par run :
#   - 1 500 pas par langue ;
#   - LR 5e-5 ; micro-batch 1 024 000 ; accumulation 4 ;
#   - checkpoint/validation tous les 250 pas a partir de step_500 ;
#   - extraction immediate de l'adaptateur seul (~20 Mio) sur Drive ;
#   - conservation de deux checkpoints complets pour une reprise sure ;
#   - base step_1250 gelee et controlee pendant l'archivage.
#
# Cette cellule ne selectionne et ne deploie encore aucun adaptateur.
# 19.7d comparera les candidats 250..1500 sur le dev fige avec KenLM.

import gc
import hashlib
import json
import os
import re
import shutil
import signal
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display


# -----------------------------------------------------------------------------
# 0. Contrat signe par 19.7a et 19.7b
# -----------------------------------------------------------------------------

assert torch.cuda.is_available(), "19.7c requiert un GPU ; un L4 suffit."
assert "FT_CONFIG" in globals() and "OMNI_REPO_DIR" in globals()
assert "write_dataset_card" in globals() and "omni_lang" in globals()

FT_ROOT = Path(
    "/content/afrivoices_project/"
    "finetune_runs/experiment_run"
)
AUDIT_ROOT = FT_ROOT / "reports/lora_compatibility_audit_v1/REPLACE_WITH_LOCAL_RUN_ID"
AUDIT_PATH = AUDIT_ROOT / "lora_compatibility_audit_v1.json"
PLAN_PATH = AUDIT_ROOT / "lora_plan.parquet"
SMOKE_REPORT_PATH = FT_ROOT / "reports/lora_training_smoke_v1.json"
LOCAL_BASE_WEIGHT = Path("/content/low_rank_adapter_sources_v1/base_step1250.pt")
DRIVE_DATASET_ROOT = (
    Path("/content/afrivoices_project/")
    / "dataset_caches/afrivoices_balanced_v4_dataset"
)
LOCAL_DATASET_ROOT = Path("/content/lora_training_dataset_v1")
LOCAL_DATA_VERSION = LOCAL_DATASET_ROOT / "version=0"
CUSTOM_PACKAGE = Path(OMNI_REPO_DIR) / "workflows/recipes/wav2vec2/asr_lora"
RECIPE_BACKUP = FT_ROOT / "reports/lora_recipe_v1"
RUNS_ROOT = FT_ROOT / "lora_runs_v1"
STATUS_PATH = FT_ROOT / "reports/lora_training_full_v1.json"

BASE_PARAMETERS = 975_675_056
PARAMETERS_PER_ADAPTER = 9_898_496
TOTAL_NEURAL_PARAMETERS = 995_472_048
PARAMETER_LIMIT = 1_000_000_000
LANGUAGES = ("kln", "mas")
NUM_STEPS = 1_500
CHECKPOINT_EVERY = 250
LEARNING_RATE = 5e-5
MAX_NUM_ELEMENTS = 1_024_000
GRAD_ACCUMULATION = 4
KEEP_FULL_CHECKPOINTS = 2

for path in (AUDIT_PATH, PLAN_PATH, SMOKE_REPORT_PATH, LOCAL_BASE_WEIGHT):
    assert Path(path).exists(), path
assert DRIVE_DATASET_ROOT.is_dir(), DRIVE_DATASET_ROOT
assert NUM_STEPS % CHECKPOINT_EVERY == 0
assert TOTAL_NEURAL_PARAMETERS < PARAMETER_LIMIT

audit = json.load(open(AUDIT_PATH, encoding="utf-8"))
smoke = json.load(open(SMOKE_REPORT_PATH, encoding="utf-8"))
assert audit["status"] == smoke["status"] == "PASS"
assert audit["zero_init_exact"] is True
assert all(item["base_bitwise_unchanged"] for item in smoke["results"])
assert {item["language"] for item in smoke["results"]} == set(LANGUAGES)
assert audit["parameters_per_adapter"] == PARAMETERS_PER_ADAPTER
assert audit["total_neural_parameters"] == TOTAL_NEURAL_PARAMETERS


def now_iso():
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path, block_size=16 << 20):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()


def canonical_sha256(value):
    raw = json.dumps(
        value, sort_keys=True, ensure_ascii=False, separators=(",", ":"), default=str
    ).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()


def atomic_json(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".tmp-{os.getpid()}")
    with open(temporary, "w", encoding="utf-8") as handle:
        json.dump(value, handle, ensure_ascii=False, indent=2, default=str)
    os.replace(temporary, path)
    print("json ->", path)


base_weight_sha256 = sha256_file(LOCAL_BASE_WEIGHT)
assert base_weight_sha256 == audit["base_weight_sha256"]
plan_sha256 = sha256_file(PLAN_PATH)
plan = pd.read_parquet(PLAN_PATH)
assert len(plan) == 289
assert int(plan.parameters_per_adapter.sum()) == PARAMETERS_PER_ADAPTER


# -----------------------------------------------------------------------------
# 1. Dataset local specialise : reprise rsync, sans recopier les autres langues
# -----------------------------------------------------------------------------

source_version = DRIVE_DATASET_ROOT / "version=0"
source_corpora = sorted(path for path in source_version.glob("corpus=*") if path.is_dir())
assert len(source_corpora) == 1, source_corpora
source_corpus = source_corpora[0]
destination_corpus = LOCAL_DATA_VERSION / source_corpus.name


def sync_partition(language, split):
    code = omni_lang(language)
    source = source_corpus / f"split={split}" / f"language={code}"
    destination = destination_corpus / f"split={split}" / f"language={code}"
    assert source.is_dir(), source
    destination.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "rsync", "-a", "--partial", "--append-verify", "--info=progress2",
        str(source) + "/", str(destination) + "/",
    ], check=True)
    source_files = sorted(path for path in source.rglob("*") if path.is_file())
    destination_files = sorted(path for path in destination.rglob("*") if path.is_file())
    assert len(source_files) == len(destination_files) > 0
    source_sizes = {str(path.relative_to(source)): path.stat().st_size for path in source_files}
    destination_sizes = {
        str(path.relative_to(destination)): path.stat().st_size for path in destination_files
    }
    assert source_sizes == destination_sizes, (language, split)
    return int(sum(source_sizes.values()))


synced_bytes = 0
for language in LANGUAGES:
    for split in ("train", "dev"):
        print("Verification dataset :", language, split)
        synced_bytes += sync_partition(language, split)
print("Dataset local :", LOCAL_DATA_VERSION, "|", round(synced_bytes / 2**30, 2), "Gio")
write_dataset_card(data_root=str(LOCAL_DATA_VERSION))

source_summary = pd.read_csv(
    DRIVE_DATASET_ROOT / "language_distribution_0.tsv", sep="\t"
)
assert "language" in source_summary.columns


# -----------------------------------------------------------------------------
# 2. Restaurer et figer la recette LoRA validee pendant le smoke test
# -----------------------------------------------------------------------------

recipe_files = ("__init__.py", "__main__.py", "recipe.py")
RECIPE_BACKUP.mkdir(parents=True, exist_ok=True)
if all((CUSTOM_PACKAGE / name).exists() for name in recipe_files):
    for name in recipe_files:
        source = CUSTOM_PACKAGE / name
        destination = RECIPE_BACKUP / name
        if not destination.exists():
            shutil.copy2(source, destination)
        else:
            assert sha256_file(source) == sha256_file(destination), (
                "La recette locale differe de la recette validee", source, destination
            )
else:
    assert all((RECIPE_BACKUP / name).exists() for name in recipe_files), (
        "Recette LoRA absente : relancer 19.7b une fois dans ce runtime."
    )
    CUSTOM_PACKAGE.mkdir(parents=True, exist_ok=True)
    for name in recipe_files:
        shutil.copy2(RECIPE_BACKUP / name, CUSTOM_PACKAGE / name)

recipe_source_hashes = {
    name: sha256_file(CUSTOM_PACKAGE / name) for name in recipe_files
}
compile((CUSTOM_PACKAGE / "recipe.py").read_text(encoding="utf-8"),
        str(CUSTOM_PACKAGE / "recipe.py"), "exec")


# Carte du seed acoustique step_1250.
import omnilingual_asr as oa

CARD_NAME = "omniASR_CTC_1B_v2_step1250_lora_seed_v1"
CARD_PATH = Path(oa.__file__).parent / "cards/models/afrivoices_step1250_lora_seed_v1.yaml"
CARD_PATH.parent.mkdir(parents=True, exist_ok=True)
CARD_PATH.write_text(
    f'name: {CARD_NAME}\nbase: {FT_CONFIG["model_card"]}\n'
    f'checkpoint: "file://{LOCAL_BASE_WEIGHT}"\n',
    encoding="utf-8",
)


# -----------------------------------------------------------------------------
# 3. Contrats, YAML et repertoires persistants par langue
# -----------------------------------------------------------------------------

def write_language_tsv(language, run_root):
    code = omni_lang(language)
    filtered = source_summary[source_summary.language.astype(str).eq(code)].copy()
    assert len(filtered) == 1, (language, code, filtered)
    path = run_root / f"language_distribution_{language}.tsv"
    filtered.to_csv(path, sep="\t", index=False)
    return path, filtered.iloc[0].to_dict()


def write_recipe_yaml(language, run_root, tsv_path):
    dtype_value = str(FT_CONFIG.get("mixed_precision", "torch.bfloat16"))
    if dtype_value.startswith("torch."):
        dtype_value = dtype_value.split(".", 1)[1]
    path = run_root / f"lora_{language}_1500.yaml"
    lines = [
        "model:", f'  name: "{CARD_NAME}"', "",
        "dataset:", '  name: "afrivoices_dataset"',
        '  train_split: "train"', '  valid_split: "dev"',
        '  storage_mode: "MIXTURE_PARQUET"', '  task_mode: "ASR"',
        "  mixture_parquet_storage_config:",
        f'    dataset_summary_path: "{tsv_path}"',
        "    beta_corpus: 1.0", "    beta_language: 0.0",
        "    fragment_loading:", "      cache: False",
        "  asr_task_config:", "    max_audio_len: 640000",
        f"    max_num_elements: {MAX_NUM_ELEMENTS}",
        "    batch_shuffle_window: 1", "    normalize_audio: true",
        "    example_shuffle_window: 1000", "",
        "tokenizer:", f'  name: "{FT_CONFIG["tokenizer_name"]}"', "",
        "optimizer:", "  config:", f"    lr: {LEARNING_RATE:.8e}", "",
        "trainer:", "  freeze_encoder_for_n_steps: 0", "  mixed_precision:",
        f'    dtype: "{dtype_value}"', "  grad_accumulation:",
        f"    num_batches: {GRAD_ACCUMULATION}", "",
        "regime:", f"  num_steps: {NUM_STEPS}",
        f"  validate_every_n_steps: {CHECKPOINT_EVERY}",
        f"  validate_after_n_steps: {CHECKPOINT_EVERY}",
        f"  checkpoint_every_n_steps: {CHECKPOINT_EVERY}",
        f"  checkpoint_after_n_steps: {CHECKPOINT_EVERY}",
        "  publish_metrics_every_n_steps: 50",
        "  publish_metrics_after_n_steps: 50",
    ]
    path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return path


language_runtime = {}
for language in LANGUAGES:
    run_root = RUNS_ROOT / language
    run_root.mkdir(parents=True, exist_ok=True)
    tsv_path, summary_row = write_language_tsv(language, run_root)
    yaml_path = write_recipe_yaml(language, run_root, tsv_path)
    contract_payload = {
        "schema": 1,
        "purpose": "train one routed LoRA adapter from frozen balanced step_1250",
        "language": language,
        "omni_code": omni_lang(language),
        "base_weight_sha256": base_weight_sha256,
        "audit_19_7a_contract_sha256": audit["contract_sha256"],
        "smoke_report_sha256": sha256_file(SMOKE_REPORT_PATH),
        "plan_sha256": plan_sha256,
        "recipe_source_hashes": recipe_source_hashes,
        "dataset_summary": summary_row,
        "dataset_tsv_sha256": sha256_file(tsv_path),
        "num_steps": NUM_STEPS,
        "checkpoint_every": CHECKPOINT_EVERY,
        "learning_rate": LEARNING_RATE,
        "max_num_elements": MAX_NUM_ELEMENTS,
        "grad_accumulation": GRAD_ACCUMULATION,
        "base_parameters": BASE_PARAMETERS,
        "adapter_parameters": PARAMETERS_PER_ADAPTER,
        "combined_two_adapter_total": TOTAL_NEURAL_PARAMETERS,
    }
    contract = {
        "contract": contract_payload,
        "contract_sha256": canonical_sha256(contract_payload),
    }
    contract_path = run_root / "run_contract.json"
    if contract_path.exists():
        assert json.load(open(contract_path, encoding="utf-8")) == contract, (
            "Contrat existant different : ne pas reprendre ce run", contract_path
        )
    else:
        atomic_json(contract, contract_path)
    language_runtime[language] = {
        "run_root": run_root,
        "output_dir": run_root / "train_output",
        "candidates_dir": run_root / "adapter_candidates" / contract["contract_sha256"][:16],
        "yaml_path": yaml_path,
        "contract": contract,
        "log_path": run_root / "training.log",
    }


# -----------------------------------------------------------------------------
# 4. Archivage adaptateur-seul et verification des checkpoints
# -----------------------------------------------------------------------------

def complete_checkpoints(output_dir):
    result = []
    for path in Path(output_dir).rglob("step_*"):
        match = re.fullmatch(r"step_(\d+)", path.name)
        if not match or path.name.endswith(".tmp") or not (path / "model").is_dir():
            continue
        model_files = [item for item in (path / "model").rglob("*") if item.is_file()]
        trainer_files = [item for item in (path / "trainer").rglob("*") if item.is_file()] \
            if (path / "trainer").is_dir() else []
        if len(model_files) == 1 and trainer_files and all(item.stat().st_size > 0 for item in model_files + trainer_files):
            result.append((int(match.group(1)), path, model_files[0]))
    return sorted(result)


def load_flat_state(path):
    raw = torch.load(str(path), map_location="cpu", weights_only=False, mmap=True)
    state = raw.get("model", raw) if isinstance(raw, dict) else raw
    assert isinstance(state, dict) and state
    return raw, state


def canonical_base_key(key):
    return key.replace(".base.weight", ".weight").replace(".base.bias", ".bias")


base_raw, base_state = load_flat_state(LOCAL_BASE_WEIGHT)
base_keys = sorted(base_state)
probe_indices = sorted(set(
    [0, len(base_keys) - 1] +
    [round((len(base_keys) - 1) * index / 8) for index in range(1, 8)]
))
base_probe_keys = [base_keys[index] for index in probe_indices]


def archive_adapter(language, step, checkpoint, checkpoint_weight, exhaustive=False):
    runtime = language_runtime[language]
    destination = runtime["candidates_dir"] / f"step_{step}"
    manifest_path = destination / "adapter_manifest.json"
    if manifest_path.exists() and (destination / "adapter.pt").exists():
        manifest = json.load(open(manifest_path, encoding="utf-8"))
        assert manifest["step"] == step
        assert manifest["contract_sha256"] == runtime["contract"]["contract_sha256"]
        assert sha256_file(destination / "adapter.pt") == manifest["adapter_sha256"]
        return manifest

    raw, state = load_flat_state(checkpoint_weight)
    lora_keys = sorted(
        key for key in state if key.endswith(".lora_A") or key.endswith(".lora_B")
    )
    assert len(lora_keys) == 578, (step, len(lora_keys))
    lora_parameters = sum(int(state[key].numel()) for key in lora_keys)
    assert lora_parameters == PARAMETERS_PER_ADAPTER
    b_nonzero = sum(
        int(torch.count_nonzero(state[key]).item())
        for key in lora_keys if key.endswith(".lora_B")
    )
    assert b_nonzero > 0

    if exhaustive:
        keys_to_check = [key for key in state if key not in set(lora_keys)]
    else:
        # Retrouver dans le checkpoint wrapper les neuf cles probes du seed.
        by_canonical = {
            canonical_base_key(key): key for key in state if key not in set(lora_keys)
        }
        keys_to_check = [by_canonical[key] for key in base_probe_keys]
    checked_parameters = 0
    for key in keys_to_check:
        canonical = canonical_base_key(key)
        assert canonical in base_state, (key, canonical)
        assert torch.equal(state[key].cpu(), base_state[canonical].cpu()), (
            "Poids de base modifie", language, step, key
        )
        checked_parameters += int(state[key].numel())
    if exhaustive:
        assert checked_parameters == BASE_PARAMETERS, checked_parameters

    adapter_payload = {
        "schema": 1,
        "language": language,
        "step": step,
        "base_weight_sha256": base_weight_sha256,
        "plan_sha256": plan_sha256,
        "state_dict": {key: state[key].detach().cpu().clone() for key in lora_keys},
    }
    temporary = destination.parent / f"{destination.name}.tmp-{os.getpid()}"
    shutil.rmtree(temporary, ignore_errors=True)
    temporary.mkdir(parents=True, exist_ok=True)
    adapter_path = temporary / "adapter.pt"
    torch.save(adapter_payload, adapter_path)
    adapter_sha256 = sha256_file(adapter_path)
    manifest = {
        "schema": 1,
        "created_at": now_iso(),
        "language": language,
        "step": step,
        "contract_sha256": runtime["contract"]["contract_sha256"],
        "base_weight_sha256": base_weight_sha256,
        "plan_sha256": plan_sha256,
        "source_checkpoint": str(checkpoint),
        "source_checkpoint_weight": str(checkpoint_weight),
        "adapter_file": "adapter.pt",
        "adapter_sha256": adapter_sha256,
        "adapter_bytes": int(adapter_path.stat().st_size),
        "adapter_tensors": len(lora_keys),
        "adapter_parameters": lora_parameters,
        "nonzero_lora_B_elements": b_nonzero,
        "base_check": "exhaustive" if exhaustive else "nine_tensor_probe",
        "base_parameters_checked": checked_parameters,
        "base_unchanged": True,
    }
    atomic_json(manifest, temporary / "adapter_manifest.json")
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        shutil.rmtree(destination)
    os.replace(temporary, destination)
    print("Adaptateur archive :", language, f"step_{step}", "->", destination)
    del raw, state, adapter_payload
    gc.collect()
    return manifest


def archive_available_and_prune(language):
    runtime = language_runtime[language]
    checkpoints = complete_checkpoints(runtime["output_dir"])
    for step, checkpoint, weight in checkpoints:
        if step % CHECKPOINT_EVERY == 0 and step <= NUM_STEPS:
            archive_adapter(
                language, step, checkpoint, weight,
                exhaustive=(step == NUM_STEPS),
            )
    # Un checkpoint n'est supprime qu'apres publication de son adaptateur signe.
    for step, checkpoint, _ in checkpoints[:-KEEP_FULL_CHECKPOINTS]:
        manifest = runtime["candidates_dir"] / f"step_{step}/adapter_manifest.json"
        if manifest.exists():
            shutil.rmtree(checkpoint, ignore_errors=True)
            print("Checkpoint complet elague :", checkpoint)


# -----------------------------------------------------------------------------
# 5. Nettoyage CUDA, execution live et reprise automatique
# -----------------------------------------------------------------------------

def owns_cuda(value):
    try:
        if isinstance(value, torch.Tensor):
            return bool(value.is_cuda)
        if isinstance(value, torch.nn.Module):
            return any(parameter.is_cuda for parameter in value.parameters())
        if isinstance(value, dict):
            return any(owns_cuda(item) for item in list(value.values())[:100])
        if isinstance(value, (list, tuple, set)):
            return any(owns_cuda(item) for item in list(value)[:100])
        for attribute in ("model", "module", "_model", "pipeline", "pipe"):
            child = getattr(value, attribute, None)
            if child is not None and child is not value and owns_cuda(child):
                return True
    except Exception:
        return False
    return False


protected = {
    "torch", "FT_CONFIG", "OMNI_REPO_DIR", "write_dataset_card", "omni_lang",
    "owns_cuda", "protected", "base_raw", "base_state",
}
released = []
for name, value in list(globals().items()):
    if name.startswith("__") or name in protected:
        continue
    if owns_cuda(value):
        released.append(name)
        globals().pop(name, None)
gc.collect()
torch.cuda.empty_cache()
assert torch.cuda.memory_allocated() < 3 * 2**30
print("CUDA libere :", released, "|", round(torch.cuda.memory_allocated()/2**30, 3), "Gio")


def run_language(language):
    runtime = language_runtime[language]
    runtime["output_dir"].mkdir(parents=True, exist_ok=True)
    runtime["candidates_dir"].mkdir(parents=True, exist_ok=True)
    archive_available_and_prune(language)

    existing_steps = [step for step, _, _ in complete_checkpoints(runtime["output_dir"])]
    archived_steps = sorted(
        int(path.name.split("_")[-1]) for path in runtime["candidates_dir"].glob("step_*")
        if (path / "adapter_manifest.json").exists()
    )
    if NUM_STEPS in archived_steps:
        print(language, ": step_1500 deja archive, run saute.")
        return {
            "language": language, "returncode": 0, "elapsed_minutes": 0.0,
            "reused_completed_run": True, "archived_steps": archived_steps,
        }

    env = os.environ.copy()
    env.update({
        "AFRIVOICES_LORA_PLAN": str(PLAN_PATH),
        "AFRIVOICES_LORA_ADAPTER": language,
        "AFRIVOICES_LORA_BASE_PARAMETERS": str(BASE_PARAMETERS),
        "AFRIVOICES_LORA_PARAMETERS": str(PARAMETERS_PER_ADAPTER),
        "PYTHONUNBUFFERED": "1",
    })
    command = [
        sys.executable, "-m", "workflows.recipes.wav2vec2.asr_lora",
        str(runtime["output_dir"]), "--config-file", str(runtime["yaml_path"]),
    ]
    process = subprocess.Popen(
        command, cwd=str(OMNI_REPO_DIR), env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, start_new_session=True,
    )
    started = time.time()
    returncode = None
    try:
        with open(runtime["log_path"], "a", encoding="utf-8") as log:
            for line in process.stdout:
                log.write(line)
                log.flush()
                lowered = line.lower()
                if any(token in lowered for token in (
                    "afrivoices_lora", "train metrics", "validation metrics",
                    "checkpoint at step", "end of training", "error", "oom", "nan",
                )):
                    print(line.rstrip()[:280])
                saved = re.search(r"Checkpoint at step (\d+) saved", line)
                if saved:
                    time.sleep(3)
                    archive_available_and_prune(language)
        returncode = process.wait()
    except BaseException:
        if process.poll() is None:
            try:
                os.killpg(process.pid, signal.SIGINT)
                process.wait(timeout=60)
            except Exception:
                try:
                    os.killpg(process.pid, signal.SIGKILL)
                except Exception:
                    pass
        raise
    finally:
        if process.poll() is None:
            os.killpg(process.pid, signal.SIGKILL)
            process.wait()

    time.sleep(5)
    archive_available_and_prune(language)
    log_text = runtime["log_path"].read_text(encoding="utf-8", errors="ignore")
    assert returncode == 0, log_text[-10000:]
    assert "out of memory" not in log_text.lower(), log_text[-5000:]
    assert re.search(r"\bnan\b", log_text.lower()) is None, log_text[-5000:]
    archived_steps = sorted(
        int(path.name.split("_")[-1]) for path in runtime["candidates_dir"].glob("step_*")
        if (path / "adapter_manifest.json").exists()
    )
    # Fairseq2 applique `checkpoint_after_n_steps=250` strictement : le premier
    # checkpoint periodique est donc 500, puis 750, ..., 1500.
    expected_archived_steps = list(
        range(2 * CHECKPOINT_EVERY, NUM_STEPS + 1, CHECKPOINT_EVERY)
    )
    assert archived_steps == expected_archived_steps, (
        language, archived_steps
    )
    return {
        "language": language,
        "returncode": returncode,
        "elapsed_minutes": round((time.time() - started) / 60, 2),
        "reused_completed_run": False,
        "existing_steps_before": existing_steps,
        "archived_steps": archived_steps,
        "log_path": str(runtime["log_path"]),
        "output_dir": str(runtime["output_dir"]),
        "candidates_dir": str(runtime["candidates_dir"]),
    }


results = []
for language in LANGUAGES:
    print("\n" + "=" * 80)
    print("ENTRAINEMENT LORA", language, f"0/reprise -> {NUM_STEPS}")
    result = run_language(language)
    results.append(result)
    atomic_json({
        "schema": 1,
        "updated_at": now_iso(),
        "status": "IN_PROGRESS" if language != LANGUAGES[-1] else "PASS",
        "base_weight_sha256": base_weight_sha256,
        "total_neural_parameters": TOTAL_NEURAL_PARAMETERS,
        "results": results,
    }, STATUS_PATH)

del base_raw, base_state
gc.collect()

print("\n=== RESULTAT 19.7c ===")
display(pd.DataFrame(results))
print("STATUT 19.7c : PASS")
print("Adaptateurs candidats :")
for language in LANGUAGES:
    print(" ", language, "->", language_runtime[language]["candidates_dir"])
print("Ne pas lancer l'inference test : etape suivante 19.7d, evaluation locale 250..1500.")

In [ ]:
# 19.7d — Evaluation locale des adaptateurs LoRA kln/mas avec KenLM historique
#
# Compare, pour chaque langue :
#   base step_1250, LoRA step_500, 750, 1000, 1250 et 1500.
# Protocole : dev V4 fige (~2 h/langue), batch=1, WER corpus exact, KenLM V5
# historique du score 0.36529, diagnostics scripted/unscripted et bootstrap apparie.
#
# Aucun adaptateur n'est deploye ici. Les caches par blocs de 32 clips sont repris
# automatiquement apres une interruption.

import gc
import hashlib
import importlib.metadata
import json
import math
import os
import re
import shutil
import subprocess
import sys
import time
import unicodedata
from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display


# -----------------------------------------------------------------------------
# 0. Prerequis et actifs signes
# -----------------------------------------------------------------------------

assert torch.cuda.is_available(), "19.7d requiert un GPU ; un L4 suffit."
assert "FT_CONFIG" in globals() and "omni_lang" in globals()

try:
    import jiwer
    import kenlm  # noqa: F401
    from pyctcdecode import build_ctcdecoder
except ImportError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "jiwer", "pyctcdecode", "kenlm"],
        check=True,
    )
    import jiwer
    import kenlm  # noqa: F401
    from pyctcdecode import build_ctcdecoder

FT_ROOT = Path(
    "/content/afrivoices_project/"
    "finetune_runs/experiment_run"
)
REPORT_DIR = FT_ROOT / "reports"
V4_REPORT_DIR = REPORT_DIR / "balanced_v4"
DEV_SELECTION_PATH = V4_REPORT_DIR / "selected_dev_v4.parquet"
TRAIN_SELECTION_PATH = V4_REPORT_DIR / "selected_records_v4.parquet"
PLAN_PATH = (
    REPORT_DIR / "lora_compatibility_audit_v1/REPLACE_WITH_LOCAL_RUN_ID/lora_plan.parquet"
)
AUDIT_PATH = (
    REPORT_DIR / "lora_compatibility_audit_v1/REPLACE_WITH_LOCAL_RUN_ID/"
    "lora_compatibility_audit_v1.json"
)
TRAINING_STATUS_PATH = REPORT_DIR / "lora_training_full_v1.json"
HISTORICAL_CONFIG_PATH = REPORT_DIR / "kenlm_tuning_by_domain_v3_pre_step1250.json"
BASE_CHECKPOINT = (
    FT_ROOT / "balanced_joint_v4/candidates/REPLACE_WITH_LOCAL_RUN_ID/step_1250"
)
LOCAL_BASE_WEIGHT = Path("/content/low_rank_adapter_sources_v1/base_step1250.pt")
LOCAL_AUDIO_ROOT = Path("/content/delta_merge_eval_audio_v2")
EVAL_ROOT = REPORT_DIR / "lora_eval_v1"
PREVIOUS_EVAL_LATEST = REPORT_DIR / "delta_merge_eval_v2/LATEST.json"

LANGUAGES = ("kln", "mas")
STEPS = (500, 750, 1000, 1250, 1500)
BASE_PARAMETERS = 975_675_056
PARAMETERS_PER_ADAPTER = 9_898_496
TOTAL_NEURAL_PARAMETERS = 995_472_048
PART_SIZE = 32
BOOTSTRAP_REPETITIONS = 2_000
BOOTSTRAP_SEED = 19_700_004

for required in (
    DEV_SELECTION_PATH, TRAIN_SELECTION_PATH, PLAN_PATH, AUDIT_PATH,
    TRAINING_STATUS_PATH, HISTORICAL_CONFIG_PATH, LOCAL_BASE_WEIGHT,
):
    assert Path(required).exists(), required

audit = json.load(open(AUDIT_PATH, encoding="utf-8"))
training_status = json.load(open(TRAINING_STATUS_PATH, encoding="utf-8"))
assert audit["status"] == training_status["status"] == "PASS"
assert audit["parameters_per_adapter"] == PARAMETERS_PER_ADAPTER
assert audit["total_neural_parameters"] == TOTAL_NEURAL_PARAMETERS


def now_iso():
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path, block_size=16 << 20):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()


def canonical_sha256(value):
    raw = json.dumps(
        value, sort_keys=True, ensure_ascii=False, separators=(",", ":"), default=str
    ).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()


def atomic_json(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".tmp-{os.getpid()}")
    with open(temporary, "w", encoding="utf-8") as handle:
        json.dump(value, handle, ensure_ascii=False, indent=2, default=str)
    os.replace(temporary, path)
    print("json ->", path)


def atomic_parquet(records, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".tmp-{os.getpid()}")
    pq.write_table(
        pa.Table.from_pylist(records), temporary,
        compression="zstd", compression_level=3,
    )
    os.replace(temporary, path)


def frame_sha256(frame, sort_key="sample_key"):
    ordered = frame.copy()
    columns = sorted(ordered.columns)
    ordered = ordered[columns]
    if sort_key in ordered.columns:
        ordered = ordered.sort_values(sort_key, kind="mergesort")
    ordered = ordered.reset_index(drop=True)
    values = pd.util.hash_pandas_object(
        ordered.fillna("").astype(str), index=False
    ).values.tobytes()
    schema = json.dumps(columns, ensure_ascii=False).encode("utf-8")
    return hashlib.sha256(schema + values).hexdigest()


base_weight_sha256 = sha256_file(LOCAL_BASE_WEIGHT)
assert base_weight_sha256 == audit["base_weight_sha256"]
plan_sha256 = sha256_file(PLAN_PATH)
plan = pd.read_parquet(PLAN_PATH)
assert len(plan) == 289 and int(plan.parameters_per_adapter.sum()) == PARAMETERS_PER_ADAPTER


# -----------------------------------------------------------------------------
# 1. Inventaire des dix adaptateurs candidats
# -----------------------------------------------------------------------------

candidate_assets = {language: OrderedDict() for language in LANGUAGES}
for language in LANGUAGES:
    roots = [
        path for path in (FT_ROOT / f"lora_runs_v1/{language}/adapter_candidates").glob("*")
        if path.is_dir()
    ]
    valid_roots = []
    for root in roots:
        available = {
            int(path.name.split("_")[-1])
            for path in root.glob("step_*")
            if (path / "adapter.pt").exists() and (path / "adapter_manifest.json").exists()
        }
        if set(STEPS) <= available:
            valid_roots.append(root)
    assert len(valid_roots) == 1, (language, valid_roots)
    root = valid_roots[0]
    for step in STEPS:
        directory = root / f"step_{step}"
        adapter_path = directory / "adapter.pt"
        manifest_path = directory / "adapter_manifest.json"
        manifest = json.load(open(manifest_path, encoding="utf-8"))
        assert manifest["language"] == language and manifest["step"] == step
        assert manifest["base_weight_sha256"] == base_weight_sha256
        assert manifest["plan_sha256"] == plan_sha256
        assert manifest["adapter_parameters"] == PARAMETERS_PER_ADAPTER
        assert sha256_file(adapter_path) == manifest["adapter_sha256"]
        candidate_assets[language][f"step_{step}"] = {
            "step": step,
            "adapter_path": str(adapter_path),
            "adapter_sha256": manifest["adapter_sha256"],
            "manifest_path": str(manifest_path),
            "manifest_sha256": sha256_file(manifest_path),
        }
    print(language, "candidats :", list(candidate_assets[language]))


# -----------------------------------------------------------------------------
# 2. Dev V4 fige, normalisation et materialisation locale
# -----------------------------------------------------------------------------

_MOJIBAKE_MARKERS = set("ÃÂÅâðÐÑ¤¦¨©¬®¯°±²³´µ¶·¸¹º»¼½¾¿")
_PUNCT_RE = re.compile(r"[^\w\s'’\-]", re.UNICODE)
_WS_RE = re.compile(r"\s+")


def text_badness(value):
    value = str(value)
    score = 50 * value.count("�")
    score += 6 * sum(char in _MOJIBAKE_MARKERS for char in value)
    score += 10 * sum(ord(char) < 32 and char not in "\n\t\r" for char in value)
    return score


def repair_mojibake(value):
    raw = str(value or "")
    candidates = [(raw, "none")]
    if any(char in _MOJIBAKE_MARKERS for char in raw) or "�" in raw:
        for encoding in ("latin-1", "cp1252"):
            try:
                candidates.append((raw.encode(encoding).decode("utf-8"), encoding))
            except (UnicodeEncodeError, UnicodeDecodeError):
                continue
    best, _ = min(candidates, key=lambda item: text_badness(item[0]))
    if text_badness(best) >= text_badness(raw):
        return raw
    return best


def normalize_eval_text(value):
    value = unicodedata.normalize("NFC", repair_mojibake(value)).lower()
    value = _PUNCT_RE.sub(" ", value).replace("_", " ")
    return _WS_RE.sub(" ", value).strip()


dev_all = pd.read_parquet(DEV_SELECTION_PATH)
dev_selection_sha256 = frame_sha256(dev_all)
required_columns = {
    "sample_key", "language", "split", "speech_method", "adaptation_role",
    "source_kind", "audio_ref", "parquet_path", "row_index", "text", "seconds",
}
assert required_columns <= set(dev_all.columns), required_columns - set(dev_all.columns)
assert dev_all["split"].eq("dev").all() and dev_all["sample_key"].is_unique
train_keys = set(pd.read_parquet(
    TRAIN_SELECTION_PATH, columns=["sample_key"]
)["sample_key"].astype(str))
assert set(dev_all["sample_key"].astype(str)).isdisjoint(train_keys)

dev = dev_all[dev_all.language.isin(LANGUAGES)].copy()
dev["seconds"] = pd.to_numeric(dev["seconds"], errors="raise")
assert dev.seconds.between(0.5, 38.0).all()
dev["reference"] = dev.text.astype(str).map(normalize_eval_text)
assert dev.reference.str.len().gt(0).all()


def canonical_domain(row):
    method = str(row["speech_method"]).lower().replace("_", "-")
    if "unscript" in method or "spont" in method:
        return "unscripted"
    if "script" in method or "read" in method:
        return "scripted"
    raise ValueError((row["sample_key"], row["language"], row["speech_method"]))


dev["decode_domain"] = [canonical_domain(row) for row in dev.to_dict("records")]
hours = dev.groupby("language").seconds.sum().div(3600)
assert set(hours.index) == set(LANGUAGES) and hours.between(1.9, 2.1).all(), hours


def audio_destination(sample_key, language):
    digest = hashlib.sha256(str(sample_key).encode("utf-8")).hexdigest()[:24]
    return LOCAL_AUDIO_ROOT / language / f"{digest}.flac"


dev["eval_audio_path"] = [
    str(audio_destination(key, language))
    for key, language in zip(dev.sample_key, dev.language)
]
missing = dev[~dev.eval_audio_path.map(os.path.exists)]
if not missing.empty:
    print("Audio local incomplet : materialisation de", len(missing), "clips...")
    # Fichiers cures.
    for row in missing[missing.source_kind.eq("file")].to_dict("records"):
        source = Path(str(row["audio_ref"]))
        destination = audio_destination(row["sample_key"], row["language"])
        assert source.exists(), source
        destination.parent.mkdir(parents=True, exist_ok=True)
        temporary = destination.with_name(destination.name + f".tmp-{os.getpid()}")
        shutil.copyfile(source, temporary)
        os.replace(temporary, destination)
    # Caches spontanes : un parquet source est lu une fois.
    parquet_missing = missing[~missing.source_kind.eq("file")]
    for parquet_path, group in parquet_missing.groupby("parquet_path", sort=True):
        parquet_path = Path(str(parquet_path))
        assert parquet_path.exists(), parquet_path
        rows = pq.read_table(parquet_path, columns=["audio_bytes"]).to_pylist()
        for row in group.to_dict("records"):
            payload = bytes(rows[int(row["row_index"])]["audio_bytes"])
            destination = audio_destination(row["sample_key"], row["language"])
            destination.parent.mkdir(parents=True, exist_ok=True)
            temporary = destination.with_name(destination.name + f".tmp-{os.getpid()}")
            with open(temporary, "wb") as handle:
                handle.write(payload)
            os.replace(temporary, destination)

assert dev.eval_audio_path.map(os.path.exists).all()
print("\n=== DEV 19.7d ===")
display(dev.groupby(["language", "decode_domain"]).agg(
    clips=("sample_key", "size"),
    hours=("seconds", lambda values: float(values.sum()) / 3600),
    words=("reference", lambda values: sum(len(value.split()) for value in values)),
).reset_index())


# -----------------------------------------------------------------------------
# 3. Configuration KenLM V5 historique exacte
# -----------------------------------------------------------------------------

historical_config = json.load(open(HISTORICAL_CONFIG_PATH, encoding="utf-8"))
LM_PROFILE = {
    "kln": {"alpha": 0.50, "beta": 0.0, "beam": 250},
    "mas": {"alpha": 0.65, "beta": 2.0, "beam": 100},
}
lm_assets = {}
for language in LANGUAGES:
    expected = LM_PROFILE[language]
    binary = FT_ROOT / f"kenlm_models_v5/{language}.binary"
    unigrams = FT_ROOT / f"kenlm_models_v5/{language}_unigrams.txt"
    assert binary.exists() and unigrams.exists(), (binary, unigrams)
    for domain in ("scripted", "unscripted"):
        config = historical_config[f"{language}|{domain}"]
        assert os.path.realpath(config["lm_bin"]) == os.path.realpath(binary)
        assert float(config["alpha"]) == expected["alpha"]
        assert float(config["beta"]) == expected["beta"]
        assert int(config["beam"]) == expected["beam"]
    lm_assets[language] = {
        "binary": str(binary), "binary_sha256": sha256_file(binary),
        "unigrams": str(unigrams), "unigrams_sha256": sha256_file(unigrams),
        **expected,
    }


# -----------------------------------------------------------------------------
# 4. Modele base + wrappers LoRA interchangeables
# -----------------------------------------------------------------------------

from fairseq2.data.tokenizers.hub import load_tokenizer
from fairseq2.models.hub import load_model
from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline


def projection_dimensions(module):
    weight = getattr(module, "weight", None)
    if not isinstance(weight, torch.Tensor) or weight.ndim != 2:
        return None
    name = type(module).__name__.lower()
    namespace = type(module).__module__.lower()
    if any(token in name for token in ("embedding", "embed", "conv", "norm")):
        return None
    if any(True for _ in module.children()):
        return None
    input_dim = getattr(module, "in_features", getattr(module, "input_dim", None))
    output_dim = getattr(module, "out_features", getattr(module, "output_dim", None))
    input_dim = int(input_dim) if input_dim is not None else int(weight.shape[1])
    output_dim = int(output_dim) if output_dim is not None else int(weight.shape[0])
    if tuple(weight.shape) != (output_dim, input_dim):
        return None
    if not (isinstance(module, nn.Linear) or "linear" in name or "projection" in name
            or "projection" in namespace):
        return None
    return input_dim, output_dim


class EvalLoRAProjection(nn.Module):
    def __init__(self, base, rank):
        super().__init__()
        input_dim, output_dim = projection_dimensions(base)
        self.base = base
        self.rank = int(rank)
        self.scaling = 1.0
        self.enabled = False
        self.lora_A = nn.Parameter(torch.zeros(
            self.rank, input_dim, device=base.weight.device, dtype=base.weight.dtype
        ), requires_grad=False)
        self.lora_B = nn.Parameter(torch.zeros(
            output_dim, self.rank, device=base.weight.device, dtype=base.weight.dtype
        ), requires_grad=False)

    def forward(self, inputs, *args, **kwargs):
        output = self.base(inputs, *args, **kwargs)
        if not self.enabled:
            return output
        return output + F.linear(F.linear(inputs, self.lora_A), self.lora_B) * self.scaling


def parent_and_child(root, dotted_name):
    parts = dotted_name.split(".")
    parent = root
    for part in parts[:-1]:
        parent = getattr(parent, part)
    return parent, parts[-1]


def inject_eval_lora(root):
    wrappers = OrderedDict()
    for row in plan.sort_values("module").to_dict("records"):
        parent, child = parent_and_child(root, row["module"])
        base = getattr(parent, child)
        assert projection_dimensions(base) == (
            int(row["in_features"]), int(row["out_features"])
        )
        wrapper = EvalLoRAProjection(base, int(row["rank"]))
        setattr(parent, child, wrapper)
        wrappers[row["module"]] = wrapper
    return wrappers


def labels_from_tokenizer(tokenizer):
    tokenizer_model = getattr(tokenizer, "_model", None)
    for method in ("index_to_token", "id_to_piece", "IdToPiece"):
        if tokenizer_model is not None and hasattr(tokenizer_model, method):
            labels = [
                str(getattr(tokenizer_model, method)(index))
                for index in range(tokenizer.vocab_info.size)
            ]
            labels[0] = ""
            return labels
    raise RuntimeError("Vocabulaire tokenizer inaccessible")


for name in list(globals()):
    if name in ("pipe", "model", "_pipe", "_pipe_lm", "_pipe_joint"):
        globals().pop(name, None)
gc.collect(); torch.cuda.empty_cache()

model = load_model(FT_CONFIG["model_card"], device=torch.device("cuda"), dtype=torch.bfloat16)
raw = torch.load(str(LOCAL_BASE_WEIGHT), map_location="cpu", weights_only=False, mmap=True)
state = raw.get("model", raw) if isinstance(raw, dict) else raw
model.load_state_dict(state)
del raw, state
assert sum(parameter.numel() for parameter in model.parameters()) == BASE_PARAMETERS
model.eval()
wrappers = inject_eval_lora(model)
assert len(wrappers) == 289
tokenizer = load_tokenizer(FT_CONFIG["model_card"])
LABELS = labels_from_tokenizer(tokenizer)
labels_sha256 = hashlib.sha256("\0".join(LABELS).encode()).hexdigest()
pipe = ASRInferencePipeline(None, model=model, tokenizer=tokenizer)

decoders = {}
for language in LANGUAGES:
    asset = lm_assets[language]
    unigram_list = Path(asset["unigrams"]).read_text(encoding="utf-8").splitlines()
    decoders[language] = build_ctcdecoder(
        LABELS, asset["binary"], unigrams=unigram_list,
        alpha=asset["alpha"], beta=asset["beta"],
    )


def set_baseline():
    for wrapper in wrappers.values():
        wrapper.enabled = False


def load_adapter(asset):
    raw_adapter = torch.load(asset["adapter_path"], map_location="cpu", weights_only=False)
    adapter_state = raw_adapter["state_dict"]
    parameters = dict(model.named_parameters())
    expected_keys = {
        f"{module_name}.lora_A" for module_name in wrappers
    } | {
        f"{module_name}.lora_B" for module_name in wrappers
    }
    assert set(adapter_state) == expected_keys, (
        len(adapter_state), len(expected_keys), sorted(set(adapter_state) ^ expected_keys)[:5]
    )
    count = 0
    with torch.no_grad():
        for key, value in adapter_state.items():
            target = parameters[key]
            assert target.shape == value.shape
            target.copy_(value.to(device=target.device, dtype=target.dtype))
            count += target.numel()
        for wrapper in wrappers.values():
            wrapper.enabled = True
    assert count == PARAMETERS_PER_ADAPTER
    del raw_adapter, adapter_state


def capture_one(audio_path, omni_code):
    captured = []
    original_forward = pipe.model.forward
    def spy(source_seqs, source_seq_lens, *args, **kwargs):
        output = original_forward(source_seqs, source_seq_lens, *args, **kwargs)
        logits, layout = output[0], output[1]
        length = layout.seq_lens[0]
        length = int(length.item() if hasattr(length, "item") else length)
        captured.append(torch.log_softmax(
            logits[0, :length].detach().float(), dim=-1
        ).cpu().numpy())
        return output
    pipe.model.forward = spy
    try:
        with torch.inference_mode():
            text = pipe.transcribe([audio_path], lang=[omni_code], batch_size=1)[0]
    finally:
        pipe.model.forward = original_forward
    assert len(captured) == 1
    return normalize_eval_text(text), captured[0]


# -----------------------------------------------------------------------------
# 5. Contrat et evaluation reprenable
# -----------------------------------------------------------------------------

contract_payload = {
    "schema": 1,
    "purpose": "paired LoRA candidate evaluation on frozen V4 dev with historical V5 LM",
    "base_weight_sha256": base_weight_sha256,
    "plan_sha256": plan_sha256,
    "candidate_assets": candidate_assets,
    "dev_selection_sha256": dev_selection_sha256,
    "dev_clips": int(len(dev)),
    "languages": list(LANGUAGES),
    "lm_assets": lm_assets,
    "labels_sha256": labels_sha256,
    "batch_size": 1,
    "part_size": PART_SIZE,
    "bootstrap_repetitions": BOOTSTRAP_REPETITIONS,
}
contract_sha256 = canonical_sha256(contract_payload)
RUN_ROOT = EVAL_ROOT / contract_sha256[:16]
CACHE_ROOT = RUN_ROOT / "cache"
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
contract = {"contract": contract_payload, "contract_sha256": contract_sha256}
contract_path = RUN_ROOT / "contract.json"
if contract_path.exists():
    assert json.load(open(contract_path, encoding="utf-8")) == contract
else:
    atomic_json(contract, contract_path)


def edit_counts(reference, hypothesis):
    result = jiwer.process_words(reference, hypothesis)
    return {
        "hits": int(result.hits),
        "substitutions": int(result.substitutions),
        "deletions": int(result.deletions),
        "insertions": int(result.insertions),
    }


def existing_rows(directory, expected_keys, candidate_sha):
    frames = []
    for path in sorted(directory.glob("part-*.parquet")):
        frame = pd.read_parquet(path)
        assert frame.contract_sha256.eq(contract_sha256).all()
        assert frame.candidate_sha.eq(candidate_sha).all()
        frames.append(frame)
    if not frames:
        return set(), 0
    combined = pd.concat(frames, ignore_index=True)
    assert combined.sample_key.is_unique
    done = set(combined.sample_key.astype(str))
    assert done <= expected_keys
    next_part = 1 + max(int(path.stem.split("-")[-1]) for path in directory.glob("part-*.parquet"))
    return done, next_part


started_at = time.time()
for language in LANGUAGES:
    candidates = OrderedDict([
        ("baseline", {"step": 0, "adapter_sha256": base_weight_sha256}),
        *candidate_assets[language].items(),
    ])
    language_frame = dev[dev.language.eq(language)].sort_values(
        ["decode_domain", "sample_key"], kind="mergesort"
    )
    for candidate_name, asset in candidates.items():
        candidate_sha = asset["adapter_sha256"]
        directory = CACHE_ROOT / language / candidate_name
        directory.mkdir(parents=True, exist_ok=True)
        expected_keys = set(language_frame.sample_key.astype(str))
        done, part_number = existing_rows(directory, expected_keys, candidate_sha)
        missing_frame = language_frame[~language_frame.sample_key.astype(str).isin(done)]
        if missing_frame.empty:
            print("[cache complet]", language, candidate_name, len(done))
            continue
        if candidate_name == "baseline":
            set_baseline()
        else:
            load_adapter(asset)
        print("\n", language, candidate_name, f"{len(done)}/{len(language_frame)}")
        buffer = []
        candidate_started = time.time()
        for index, row in enumerate(missing_frame.to_dict("records"), 1):
            greedy, log_probs = capture_one(row["eval_audio_path"], omni_lang(language))
            lm_hypothesis = normalize_eval_text(decoders[language].decode(
                log_probs, beam_width=lm_assets[language]["beam"]
            ))
            greedy_counts = edit_counts(row["reference"], greedy)
            lm_counts = edit_counts(row["reference"], lm_hypothesis)
            buffer.append({
                "contract_sha256": contract_sha256,
                "candidate": candidate_name,
                "candidate_sha": candidate_sha,
                "language": language,
                "sample_key": str(row["sample_key"]),
                "decode_domain": row["decode_domain"],
                "seconds": float(row["seconds"]),
                "reference": row["reference"],
                "greedy_hypothesis": greedy,
                "lm_hypothesis": lm_hypothesis,
                **{f"greedy_{key}": value for key, value in greedy_counts.items()},
                **{f"lm_{key}": value for key, value in lm_counts.items()},
            })
            if len(buffer) >= PART_SIZE:
                destination = directory / f"part-{part_number:05d}.parquet"
                atomic_parquet(buffer, destination)
                part_number += 1
                buffer.clear()
            if index % 50 == 0 or index == len(missing_frame):
                print(
                    f"  {index}/{len(missing_frame)} nouveaux |",
                    round((time.time() - candidate_started) / 60, 1), "min",
                )
        if buffer:
            destination = directory / f"part-{part_number:05d}.parquet"
            atomic_parquet(buffer, destination)
        gc.collect(); torch.cuda.empty_cache()


# -----------------------------------------------------------------------------
# 6. Agregation, bootstrap et selection prudente
# -----------------------------------------------------------------------------

frames = [pd.read_parquet(path) for path in CACHE_ROOT.rglob("part-*.parquet")]
results = pd.concat(frames, ignore_index=True)
assert results.sample_key.groupby([results.language, results.candidate]).nunique().min() > 0


def aggregate(frame, prefix):
    substitutions = int(frame[f"{prefix}_substitutions"].sum())
    deletions = int(frame[f"{prefix}_deletions"].sum())
    insertions = int(frame[f"{prefix}_insertions"].sum())
    hits = int(frame[f"{prefix}_hits"].sum())
    words = hits + substitutions + deletions
    return {
        "wer": (substitutions + deletions + insertions) / words,
        "reference_words": words,
        "errors": substitutions + deletions + insertions,
    }


summary_rows, domain_rows = [], []
for (language, candidate), frame in results.groupby(["language", "candidate"], sort=True):
    assert len(frame) == len(dev[dev.language.eq(language)]), (language, candidate, len(frame))
    greedy = aggregate(frame, "greedy")
    lm = aggregate(frame, "lm")
    summary_rows.append({
        "language": language,
        "candidate": candidate,
        "step": 0 if candidate == "baseline" else int(candidate.split("_")[-1]),
        "clips": len(frame),
        "greedy_wer": greedy["wer"],
        "lm_wer": lm["wer"],
        "reference_words": lm["reference_words"],
    })
    for domain, domain_frame in frame.groupby("decode_domain"):
        domain_rows.append({
            "language": language, "candidate": candidate, "domain": domain,
            "clips": len(domain_frame), "lm_wer": aggregate(domain_frame, "lm")["wer"],
        })

summary = pd.DataFrame(summary_rows)
domains = pd.DataFrame(domain_rows)


def bootstrap_candidate(language, candidate, repetitions, seed):
    baseline = results[(results.language == language) & (results.candidate == "baseline")]
    adapted = results[(results.language == language) & (results.candidate == candidate)]
    baseline = baseline.set_index("sample_key").sort_index()
    adapted = adapted.set_index("sample_key").loc[baseline.index]
    words = (
        baseline.lm_hits.to_numpy(np.int64)
        + baseline.lm_substitutions.to_numpy(np.int64)
        + baseline.lm_deletions.to_numpy(np.int64)
    )
    assert np.array_equal(words, (
        adapted.lm_hits.to_numpy(np.int64)
        + adapted.lm_substitutions.to_numpy(np.int64)
        + adapted.lm_deletions.to_numpy(np.int64)
    ))
    base_errors = (
        baseline.lm_substitutions.to_numpy(np.int64)
        + baseline.lm_deletions.to_numpy(np.int64)
        + baseline.lm_insertions.to_numpy(np.int64)
    )
    adapted_errors = (
        adapted.lm_substitutions.to_numpy(np.int64)
        + adapted.lm_deletions.to_numpy(np.int64)
        + adapted.lm_insertions.to_numpy(np.int64)
    )
    rng = np.random.default_rng(seed)
    deltas = np.empty(repetitions)
    for repetition in range(repetitions):
        indices = rng.integers(0, len(words), size=len(words))
        denominator = words[indices].sum()
        deltas[repetition] = (
            adapted_errors[indices].sum() - base_errors[indices].sum()
        ) / denominator
    return {
        "delta_adapter_minus_baseline": float(
            summary[(summary.language == language) & (summary.candidate == candidate)].lm_wer.iloc[0]
            - summary[(summary.language == language) & (summary.candidate == "baseline")].lm_wer.iloc[0]
        ),
        "ci95_low": float(np.quantile(deltas, 0.025)),
        "ci95_high": float(np.quantile(deltas, 0.975)),
        "probability_adapter_better": float(np.mean(deltas < 0)),
    }


bootstrap_rows = []
selections = {}
for language in LANGUAGES:
    baseline_wer = float(summary[(summary.language == language) & (summary.candidate == "baseline")].lm_wer.iloc[0])
    baseline_domains = domains[(domains.language == language) & (domains.candidate == "baseline")].set_index("domain").lm_wer
    eligible = []
    for offset, candidate in enumerate(f"step_{step}" for step in STEPS):
        boot = bootstrap_candidate(
            language, candidate, BOOTSTRAP_REPETITIONS,
            BOOTSTRAP_SEED + 100 * list(LANGUAGES).index(language) + offset,
        )
        candidate_domains = domains[(domains.language == language) & (domains.candidate == candidate)].set_index("domain").lm_wer
        regressions = {
            domain: float(candidate_domains[domain] - baseline_domains[domain])
            for domain in baseline_domains.index
            if float(candidate_domains[domain] - baseline_domains[domain]) > 0.015
        }
        row = {"language": language, "candidate": candidate, **boot,
               "domain_regressions_over_0_015": regressions}
        bootstrap_rows.append(row)
        candidate_wer = float(summary[(summary.language == language) & (summary.candidate == candidate)].lm_wer.iloc[0])
        if (
            candidate_wer <= baseline_wer - 0.002
            and boot["probability_adapter_better"] >= 0.90
            and not regressions
        ):
            eligible.append((candidate_wer, candidate))
    if eligible:
        selected = min(eligible)[1]
        reason = "minimum LM-WER parmi les candidats franchissant les garde-fous"
    else:
        selected = "baseline"
        reason = "aucun adaptateur ne franchit les garde-fous preregistres"
    selections[language] = {"selected": selected, "reason": reason}

bootstrap_frame = pd.DataFrame(bootstrap_rows)

# Comparaison coherente avec le rapport complet 19.5 lorsqu'il est disponible.
proxy = None
if PREVIOUS_EVAL_LATEST.exists():
    latest = json.load(open(PREVIOUS_EVAL_LATEST, encoding="utf-8"))
    previous_report = json.load(open(latest["report"], encoding="utf-8"))
    baseline_all = previous_report["candidates"]["balanced_step1250"]["lm_language_wer"]
    for language in LANGUAGES:
        measured = float(summary[(summary.language == language) & (summary.candidate == "baseline")].lm_wer.iloc[0])
        assert abs(measured - float(baseline_all[language])) < 1e-12, (
            language, measured, baseline_all[language]
        )
    selected_all = dict(baseline_all)
    for language in LANGUAGES:
        chosen = selections[language]["selected"]
        selected_all[language] = float(summary[
            (summary.language == language) & (summary.candidate == chosen)
        ].lm_wer.iloc[0])
    proxy = {
        "baseline_language_wer": baseline_all,
        "selected_language_wer": selected_all,
        "baseline_macro_wer": float(np.mean(list(baseline_all.values()))),
        "selected_macro_wer": float(np.mean(list(selected_all.values()))),
    }
    proxy["macro_delta"] = proxy["selected_macro_wer"] - proxy["baseline_macro_wer"]

report = {
    "schema": 1,
    "created_at": now_iso(),
    "status": "PASS",
    "contract_sha256": contract_sha256,
    "summary": summary.to_dict("records"),
    "domains": domains.to_dict("records"),
    "bootstrap": bootstrap_rows,
    "selections": selections,
    "six_language_proxy": proxy,
    "parameter_accounting": {
        "base": BASE_PARAMETERS,
        "kln_adapter": PARAMETERS_PER_ADAPTER,
        "mas_adapter": PARAMETERS_PER_ADAPTER,
        "total": TOTAL_NEURAL_PARAMETERS,
        "limit": 1_000_000_000,
    },
    "final_pointer_mutated": False,
    "elapsed_minutes": round((time.time() - started_at) / 60, 2),
}
report_path = RUN_ROOT / "lora_candidate_eval_v1.json"
atomic_json(report, report_path)
atomic_json({
    "report": str(report_path), "contract_sha256": contract_sha256,
    "selections": selections, "created_at": report["created_at"],
}, EVAL_ROOT / "LATEST.json")

print("\n=== WER GLOBAL PAR LANGUE ===")
display(summary.sort_values(["language", "lm_wer"]))
print("\n=== WER PAR DOMAINE ===")
display(domains.pivot_table(
    index=["language", "domain"], columns="candidate", values="lm_wer"
))
print("\n=== BOOTSTRAP VS BASELINE ===")
display(bootstrap_frame)
print("\n=== SELECTION 19.7d ===")
print(json.dumps(selections, indent=2, ensure_ascii=False))
if proxy:
    print("Proxy macro step_1250 :", round(proxy["baseline_macro_wer"], 6))
    print("Proxy macro + LoRA    :", round(proxy["selected_macro_wer"], 6))
    print("Delta                 :", round(proxy["macro_delta"], 6))
print("Rapport ->", report_path)
print("Aucun adaptateur n'a encore ete deploye. Envoyer ces tableaux avant 19.7e.")

del pipe, model, tokenizer, wrappers, decoders
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# 19.7e — Valider les LoRA selectionnes avec la configuration LM du score 0.36880
#
# Prerequis : 19.7d vient de terminer dans ce runtime.
# Cette cellule ne modifie PAS la configuration active et ne lance PAS le test Kaggle.

import gc
import hashlib
import json
import os
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch
from IPython.display import display
from pyctcdecode import build_ctcdecoder


required_globals = (
    "dev", "plan", "candidate_assets", "inject_eval_lora", "labels_from_tokenizer",
    "load_adapter", "capture_one", "normalize_eval_text", "edit_counts",
)
missing_globals = [name for name in required_globals if name not in globals()]
assert not missing_globals, (
    "Relancer 19.7d avant 19.7e ; objets absents : " + ", ".join(missing_globals)
)

FT_ROOT = Path(
    "/content/afrivoices_project/"
    "finetune_runs/experiment_run"
)
REPORT_DIR = FT_ROOT / "reports"
LORA_EVAL_LATEST = REPORT_DIR / "lora_eval_v1/LATEST.json"
SCORE_CONFIG_PATH = REPORT_DIR / "kenlm_tuning_by_domain_v3_score_036880.json"
HISTORICAL_CONFIG_PATH = REPORT_DIR / "kenlm_tuning_by_domain_v3_pre_step1250.json"
OUTPUT_CONFIG_PATH = REPORT_DIR / "kenlm_tuning_by_domain_v3_lora_step1250_candidate.json"
REPORT_PATH = REPORT_DIR / "lora_lm_validation_score036880_v1.json"
CACHE_ROOT = REPORT_DIR / "lora_lm_validation_score036880_cache_v1"
LOCAL_BASE_WEIGHT = Path("/content/low_rank_adapter_sources_v1/base_step1250.pt")

LANGUAGES = ("kln", "mas")
SELECTED_STEP = 1250
PARAMETERS_PER_ADAPTER = 9_898_496
TOTAL_NEURAL_PARAMETERS = 995_472_048
PART_SIZE = 32
BOOTSTRAP_REPETITIONS = 2_000
BOOTSTRAP_SEED = 19_700_005

for path in (
    LORA_EVAL_LATEST, SCORE_CONFIG_PATH, HISTORICAL_CONFIG_PATH, LOCAL_BASE_WEIGHT
):
    assert path.exists(), path


def sha256_file(path, block_size=16 << 20):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()


def atomic_json(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".tmp-{os.getpid()}")
    with open(temporary, "w", encoding="utf-8") as handle:
        json.dump(value, handle, ensure_ascii=False, indent=2, default=str)
    os.replace(temporary, path)
    print("json ->", path)


def atomic_parquet(records, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".tmp-{os.getpid()}")
    pq.write_table(pa.Table.from_pylist(records), temporary, compression="zstd")
    os.replace(temporary, path)


latest = json.load(open(LORA_EVAL_LATEST, encoding="utf-8"))
lora_report_path = Path(latest["report"])
lora_report = json.load(open(lora_report_path, encoding="utf-8"))
assert lora_report["status"] == "PASS"
assert all(
    lora_report["selections"][language]["selected"] == f"step_{SELECTED_STEP}"
    for language in LANGUAGES
)

historical_cache_root = lora_report_path.parent / "cache"
score_config = json.load(open(SCORE_CONFIG_PATH, encoding="utf-8"))
historical_config = json.load(open(HISTORICAL_CONFIG_PATH, encoding="utf-8"))
assert set(score_config) == set(historical_config)


# -----------------------------------------------------------------------------
# 1. Recharger une seule fois base + wrappers, puis les LoRA step_1250
# -----------------------------------------------------------------------------

from fairseq2.data.tokenizers.hub import load_tokenizer
from fairseq2.models.hub import load_model
from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline

model = load_model(
    FT_CONFIG["model_card"], device=torch.device("cuda"), dtype=torch.bfloat16
)
raw = torch.load(str(LOCAL_BASE_WEIGHT), map_location="cpu", weights_only=False, mmap=True)
state = raw.get("model", raw) if isinstance(raw, dict) else raw
model.load_state_dict(state)
del raw, state
model.eval()
wrappers = inject_eval_lora(model)
tokenizer = load_tokenizer(FT_CONFIG["model_card"])
LABELS = labels_from_tokenizer(tokenizer)
pipe = ASRInferencePipeline(None, model=model, tokenizer=tokenizer)


def resolve_unigrams(language, config):
    for key in ("unigram_file", "unigrams_file", "uni_file", "unigrams"):
        value = config.get(key)
        if isinstance(value, str) and Path(value).exists():
            return Path(value)
    lm_bin = Path(config["lm_bin"])
    candidates = [
        lm_bin.with_name(f"{language}_unigrams.txt"),
        FT_ROOT / f"kenlm_models_v5/{language}_unigrams.txt",
    ]
    result = next((path for path in candidates if path.exists()), None)
    assert result is not None, (language, config, candidates)
    return result


decoders = {}
decoder_assets = {}
for language in LANGUAGES:
    key = f"{language}|unscripted"
    config = score_config[key]
    lm_bin = Path(config["lm_bin"])
    unigrams_path = resolve_unigrams(language, config)
    assert lm_bin.exists()
    unigrams = unigrams_path.read_text(encoding="utf-8").splitlines()
    decoders[language] = build_ctcdecoder(
        LABELS, str(lm_bin), unigrams=unigrams,
        alpha=float(config["alpha"]), beta=float(config["beta"]),
    )
    decoder_assets[language] = {
        "group": key,
        "lm_bin": str(lm_bin),
        "lm_sha256": sha256_file(lm_bin),
        "unigrams": str(unigrams_path),
        "unigrams_sha256": sha256_file(unigrams_path),
        "alpha": float(config["alpha"]),
        "beta": float(config["beta"]),
        "beam": int(config["beam"]),
    }


# -----------------------------------------------------------------------------
# 2. Decoder uniquement les groupes unscripted, seuls groupes modifies par 0.36880
# -----------------------------------------------------------------------------

CACHE_ROOT.mkdir(parents=True, exist_ok=True)
for language in LANGUAGES:
    historical_group = historical_config[f"{language}|unscripted"]
    score_group = score_config[f"{language}|unscripted"]
    assert (
        float(score_group["alpha"]), float(score_group["beta"]), int(score_group["beam"])
    ) != (
        float(historical_group["alpha"]), float(historical_group["beta"]),
        int(historical_group["beam"])
    ), (language, "Les deux configurations sont deja identiques")

    asset = candidate_assets[language][f"step_{SELECTED_STEP}"]
    load_adapter(asset)
    group = dev[(dev.language == language) & (dev.decode_domain == "unscripted")].copy()
    group = group.sort_values("sample_key", kind="mergesort")
    language_cache = CACHE_ROOT / language
    language_cache.mkdir(parents=True, exist_ok=True)

    existing_frames = [pd.read_parquet(path) for path in sorted(language_cache.glob("part-*.parquet"))]
    if existing_frames:
        existing = pd.concat(existing_frames, ignore_index=True)
        assert existing.sample_key.is_unique
        done = set(existing.sample_key.astype(str))
        next_part = 1 + max(
            int(path.stem.split("-")[-1]) for path in language_cache.glob("part-*.parquet")
        )
    else:
        done, next_part = set(), 0
    missing = group[~group.sample_key.astype(str).isin(done)]
    print(language, "score_036880 unscripted :", len(done), "/", len(group))
    buffer = []
    for index, row in enumerate(missing.to_dict("records"), 1):
        _, log_probs = capture_one(row["eval_audio_path"], omni_lang(language))
        hypothesis = normalize_eval_text(decoders[language].decode(
            log_probs, beam_width=decoder_assets[language]["beam"]
        ))
        counts = edit_counts(row["reference"], hypothesis)
        buffer.append({
            "language": language,
            "sample_key": str(row["sample_key"]),
            "reference": row["reference"],
            "hypothesis": hypothesis,
            **counts,
        })
        if len(buffer) >= PART_SIZE:
            atomic_parquet(buffer, language_cache / f"part-{next_part:05d}.parquet")
            next_part += 1
            buffer.clear()
        if index % 50 == 0 or index == len(missing):
            print(" ", index, "/", len(missing), "nouveaux")
    if buffer:
        atomic_parquet(buffer, language_cache / f"part-{next_part:05d}.parquet")


# -----------------------------------------------------------------------------
# 3. Comparaison appariee contre le decodage historique deja calcule en 19.7d
# -----------------------------------------------------------------------------

def counts_arrays(frame):
    errors = (
        frame.substitutions.to_numpy(np.int64)
        + frame.deletions.to_numpy(np.int64)
        + frame.insertions.to_numpy(np.int64)
    )
    words = (
        frame.hits.to_numpy(np.int64)
        + frame.substitutions.to_numpy(np.int64)
        + frame.deletions.to_numpy(np.int64)
    )
    return errors, words


rows = []
decisions = {}
for language_index, language in enumerate(LANGUAGES):
    tuned = pd.concat([
        pd.read_parquet(path) for path in sorted((CACHE_ROOT / language).glob("part-*.parquet"))
    ], ignore_index=True).set_index("sample_key").sort_index()
    historical = pd.concat([
        pd.read_parquet(path)
        for path in sorted(
            (historical_cache_root / language / f"step_{SELECTED_STEP}").glob("part-*.parquet")
        )
    ], ignore_index=True)
    historical = historical[historical.decode_domain == "unscripted"].copy()
    historical = historical.set_index("sample_key").sort_index()
    tuned = tuned.loc[historical.index]
    assert tuned.reference.equals(historical.reference)

    tuned_for_counts = tuned.rename(columns={
        "hits": "hits", "substitutions": "substitutions",
        "deletions": "deletions", "insertions": "insertions",
    })
    historical_for_counts = historical.rename(columns={
        "lm_hits": "hits", "lm_substitutions": "substitutions",
        "lm_deletions": "deletions", "lm_insertions": "insertions",
    })
    tuned_errors, words = counts_arrays(tuned_for_counts)
    historical_errors, historical_words = counts_arrays(historical_for_counts)
    assert np.array_equal(words, historical_words)
    tuned_wer = float(tuned_errors.sum() / words.sum())
    historical_wer = float(historical_errors.sum() / words.sum())

    rng = np.random.default_rng(BOOTSTRAP_SEED + language_index)
    deltas = np.empty(BOOTSTRAP_REPETITIONS)
    for repetition in range(BOOTSTRAP_REPETITIONS):
        indices = rng.integers(0, len(words), size=len(words))
        deltas[repetition] = (
            tuned_errors[indices].sum() - historical_errors[indices].sum()
        ) / words[indices].sum()
    delta = tuned_wer - historical_wer
    probability_tuned_better = float(np.mean(deltas < 0))

    # La configuration 0.36880 est deja prouvee par le leaderboard. On ne la
    # rejette que si elle regresse de plus de 0.001 sur ce dev complet.
    if delta <= 0.001:
        decision = "keep_score_036880"
        selected_config = score_config[f"{language}|unscripted"]
    else:
        decision = "restore_historical"
        selected_config = historical_config[f"{language}|unscripted"]
    decisions[f"{language}|unscripted"] = {
        "decision": decision,
        "historical_wer": historical_wer,
        "score_036880_wer": tuned_wer,
        "delta": delta,
        "ci95_low": float(np.quantile(deltas, 0.025)),
        "ci95_high": float(np.quantile(deltas, 0.975)),
        "probability_score_config_better": probability_tuned_better,
        "selected": selected_config,
    }
    rows.append({
        "language": language, "historical_wer": historical_wer,
        "score_036880_wer": tuned_wer, "delta": delta,
        "probability_score_config_better": probability_tuned_better,
        "decision": decision,
    })


# Scripted est identique entre les deux configurations ; conserver la version
# leaderboard figee. Construire un NOUVEAU fichier candidat, jamais le fichier actif.
candidate_config = json.loads(json.dumps(score_config))
for language in LANGUAGES:
    score_scripted = score_config[f"{language}|scripted"]
    historical_scripted = historical_config[f"{language}|scripted"]
    for field in ("lm_bin", "alpha", "beta", "beam"):
        left = score_scripted[field]
        right = historical_scripted[field]
        if field == "lm_bin":
            assert os.path.realpath(str(left)) == os.path.realpath(str(right))
        else:
            assert float(left) == float(right), (language, field, left, right)
    if decisions[f"{language}|unscripted"]["decision"] == "restore_historical":
        candidate_config[f"{language}|unscripted"] = historical_config[
            f"{language}|unscripted"
        ]
atomic_json(candidate_config, OUTPUT_CONFIG_PATH)

selected_adapters = {}
for language in LANGUAGES:
    asset = candidate_assets[language][f"step_{SELECTED_STEP}"]
    selected_adapters[language] = {
        "step": SELECTED_STEP,
        "adapter_path": asset["adapter_path"],
        "adapter_sha256": asset["adapter_sha256"],
        "parameters": PARAMETERS_PER_ADAPTER,
    }

report = {
    "schema": 1,
    "status": "PASS",
    "base_checkpoint": str(
        FT_ROOT / "balanced_joint_v4/candidates/REPLACE_WITH_LOCAL_RUN_ID/step_1250"
    ),
    "base_weight_sha256": sha256_file(LOCAL_BASE_WEIGHT),
    "selected_adapters": selected_adapters,
    "decisions": decisions,
    "candidate_lm_config": str(OUTPUT_CONFIG_PATH),
    "candidate_lm_config_sha256": sha256_file(OUTPUT_CONFIG_PATH),
    "total_neural_parameters": TOTAL_NEURAL_PARAMETERS,
    "parameter_limit": 1_000_000_000,
    "final_runtime_mutated": False,
}
atomic_json(report, REPORT_PATH)

print("\n=== VALIDATION LM 19.7e ===")
display(pd.DataFrame(rows))
print("Configuration candidate :", OUTPUT_CONFIG_PATH)
print("Rapport :", REPORT_PATH)
print("Aucun runtime final n'a encore ete modifie.")
print("Etape suivante : 19.7f, packaging et audit RAM avant inference test.")

del pipe, model, tokenizer, wrappers, decoders
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# 19.7f — Garde-fou final LoRA, packaging immuable et sonde RAM propre
#
# 1) compare base et LoRA avec LE MEME decodage du score leaderboard 0.36880 ;
# 2) ne conserve que les adaptateurs dont le gain est statistiquement credible ;
# 3) construit un package candidat sans modifier le runtime actif ;
# 4) mesure dans un sous-processus propre : base + un adaptateur charge paresseusement
#    + le plus gros KenLM actif. La limite officielle est 8 Gio de RSS.

import gc
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch
from IPython.display import display
from pyctcdecode import build_ctcdecoder


required_globals = (
    "dev", "plan", "candidate_assets", "inject_eval_lora", "labels_from_tokenizer",
    "set_baseline", "capture_one", "normalize_eval_text", "edit_counts",
)
missing = [name for name in required_globals if name not in globals()]
assert not missing, "Relancer 19.7d puis 19.7e avant 19.7f : " + ", ".join(missing)

FT_ROOT = Path(
    "/content/afrivoices_project/"
    "finetune_runs/experiment_run"
)
REPORT_DIR = FT_ROOT / "reports"
LM_VALIDATION_PATH = REPORT_DIR / "lora_lm_validation_score036880_v1.json"
LORA_EVAL_LATEST = REPORT_DIR / "lora_eval_v1/LATEST.json"
SCORE_CONFIG_PATH = REPORT_DIR / "kenlm_tuning_by_domain_v3_score_036880.json"
CANDIDATE_CONFIG_PATH = REPORT_DIR / "kenlm_tuning_by_domain_v3_lora_step1250_candidate.json"
LOCAL_BASE_WEIGHT = Path("/content/low_rank_adapter_sources_v1/base_step1250.pt")
PLAN_PATH = (
    REPORT_DIR / "lora_compatibility_audit_v1/REPLACE_WITH_LOCAL_RUN_ID/lora_plan.parquet"
)
TUNED_ADAPTER_CACHE = REPORT_DIR / "lora_lm_validation_score036880_cache_v1"
BASELINE_CACHE = REPORT_DIR / "lora_baseline_score036880_cache_v1"
FINAL_REPORT_PATH = REPORT_DIR / "lora_final_candidate_audit_v1.json"
PACKAGE_PARENT = FT_ROOT / "final_lora_candidate_v1"

LANGUAGES = ("kln", "mas")
SELECTED_STEP = 1250
BASE_PARAMETERS = 975_675_056
PARAMETERS_PER_ADAPTER = 9_898_496
OFFICIAL_PARAMETER_LIMIT = 1_000_000_000
RSS_LIMIT_BYTES = 8 * 2**30
PART_SIZE = 32
BOOTSTRAP_REPETITIONS = 2_000
BOOTSTRAP_SEED = 19_700_006

for path in (
    LM_VALIDATION_PATH, LORA_EVAL_LATEST, SCORE_CONFIG_PATH,
    CANDIDATE_CONFIG_PATH, LOCAL_BASE_WEIGHT, PLAN_PATH,
):
    assert path.exists(), path


def sha256_file(path, block_size=16 << 20):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()


def canonical_sha256(value):
    raw = json.dumps(
        value, sort_keys=True, ensure_ascii=False, separators=(",", ":"), default=str
    ).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()


def atomic_json(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".tmp-{os.getpid()}")
    with open(temporary, "w", encoding="utf-8") as handle:
        json.dump(value, handle, ensure_ascii=False, indent=2, default=str)
    os.replace(temporary, path)
    print("json ->", path)


def atomic_parquet(records, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".tmp-{os.getpid()}")
    pq.write_table(pa.Table.from_pylist(records), temporary, compression="zstd")
    os.replace(temporary, path)


lm_validation = json.load(open(LM_VALIDATION_PATH, encoding="utf-8"))
assert lm_validation["status"] == "PASS"
assert all(
    item["decision"] == "keep_score_036880"
    for item in lm_validation["decisions"].values()
)
score_config = json.load(open(CANDIDATE_CONFIG_PATH, encoding="utf-8"))


def resolve_unigrams(language, config):
    for key in ("unigram_file", "unigrams_file", "uni_file", "unigrams"):
        value = config.get(key)
        if isinstance(value, str) and Path(value).exists():
            return Path(value)
    lm_bin = Path(config["lm_bin"])
    candidates = [
        lm_bin.with_name(f"{language}_unigrams.txt"),
        FT_ROOT / f"kenlm_models_v5/{language}_unigrams.txt",
        FT_ROOT / f"kenlm_models_v4/{language}_unigrams.txt",
    ]
    result = next((path for path in candidates if path.exists()), None)
    assert result is not None, (language, candidates)
    return result


# -----------------------------------------------------------------------------
# 1. Calculer la baseline avec exactement les memes LM que les LoRA
# -----------------------------------------------------------------------------

from fairseq2.data.tokenizers.hub import load_tokenizer
from fairseq2.models.hub import load_model
from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline

model = load_model(
    FT_CONFIG["model_card"], device=torch.device("cuda"), dtype=torch.bfloat16
)
raw = torch.load(str(LOCAL_BASE_WEIGHT), map_location="cpu", weights_only=False, mmap=True)
state = raw.get("model", raw) if isinstance(raw, dict) else raw
model.load_state_dict(state)
del raw, state
model.eval()
wrappers = inject_eval_lora(model)
tokenizer = load_tokenizer(FT_CONFIG["model_card"])
LABELS = labels_from_tokenizer(tokenizer)
pipe = ASRInferencePipeline(None, model=model, tokenizer=tokenizer)
set_baseline()

decoders = {}
for language in LANGUAGES:
    config = score_config[f"{language}|unscripted"]
    unigrams_path = resolve_unigrams(language, config)
    decoders[language] = build_ctcdecoder(
        LABELS, config["lm_bin"],
        unigrams=unigrams_path.read_text(encoding="utf-8").splitlines(),
        alpha=float(config["alpha"]), beta=float(config["beta"]),
    )

BASELINE_CACHE.mkdir(parents=True, exist_ok=True)
for language in LANGUAGES:
    group = dev[(dev.language == language) & (dev.decode_domain == "unscripted")]
    group = group.sort_values("sample_key", kind="mergesort")
    directory = BASELINE_CACHE / language
    directory.mkdir(parents=True, exist_ok=True)
    frames = [pd.read_parquet(path) for path in sorted(directory.glob("part-*.parquet"))]
    if frames:
        existing = pd.concat(frames, ignore_index=True)
        assert existing.sample_key.is_unique
        done = set(existing.sample_key.astype(str))
        next_part = 1 + max(
            int(path.stem.split("-")[-1]) for path in directory.glob("part-*.parquet")
        )
    else:
        done, next_part = set(), 0
    missing_frame = group[~group.sample_key.astype(str).isin(done)]
    print(language, "baseline score_036880 :", len(done), "/", len(group))
    buffer = []
    config = score_config[f"{language}|unscripted"]
    for index, row in enumerate(missing_frame.to_dict("records"), 1):
        _, log_probs = capture_one(row["eval_audio_path"], omni_lang(language))
        hypothesis = normalize_eval_text(decoders[language].decode(
            log_probs, beam_width=int(config["beam"])
        ))
        counts = edit_counts(row["reference"], hypothesis)
        buffer.append({
            "language": language, "sample_key": str(row["sample_key"]),
            "reference": row["reference"], "hypothesis": hypothesis, **counts,
        })
        if len(buffer) >= PART_SIZE:
            atomic_parquet(buffer, directory / f"part-{next_part:05d}.parquet")
            next_part += 1
            buffer.clear()
        if index % 50 == 0 or index == len(missing_frame):
            print(" ", index, "/", len(missing_frame), "nouveaux")
    if buffer:
        atomic_parquet(buffer, directory / f"part-{next_part:05d}.parquet")

del pipe, model, tokenizer, wrappers, decoders
gc.collect(); torch.cuda.empty_cache()


# -----------------------------------------------------------------------------
# 2. Bootstrap LoRA vs baseline sous le decodage score_036880
# -----------------------------------------------------------------------------

def error_arrays(frame):
    errors = (
        frame.substitutions.to_numpy(np.int64)
        + frame.deletions.to_numpy(np.int64)
        + frame.insertions.to_numpy(np.int64)
    )
    words = (
        frame.hits.to_numpy(np.int64)
        + frame.substitutions.to_numpy(np.int64)
        + frame.deletions.to_numpy(np.int64)
    )
    return errors, words


comparison_rows = []
adapter_decisions = {}
for language_index, language in enumerate(LANGUAGES):
    baseline = pd.concat([
        pd.read_parquet(path)
        for path in sorted((BASELINE_CACHE / language).glob("part-*.parquet"))
    ], ignore_index=True).set_index("sample_key").sort_index()
    adapted = pd.concat([
        pd.read_parquet(path)
        for path in sorted((TUNED_ADAPTER_CACHE / language).glob("part-*.parquet"))
    ], ignore_index=True).set_index("sample_key").sort_index()
    adapted = adapted.loc[baseline.index]
    assert baseline.reference.equals(adapted.reference)
    baseline_errors, words = error_arrays(baseline)
    adapter_errors, adapter_words = error_arrays(adapted)
    assert np.array_equal(words, adapter_words)
    baseline_wer = float(baseline_errors.sum() / words.sum())
    adapter_wer = float(adapter_errors.sum() / words.sum())
    delta = adapter_wer - baseline_wer

    rng = np.random.default_rng(BOOTSTRAP_SEED + language_index)
    deltas = np.empty(BOOTSTRAP_REPETITIONS)
    for repetition in range(BOOTSTRAP_REPETITIONS):
        indices = rng.integers(0, len(words), size=len(words))
        deltas[repetition] = (
            adapter_errors[indices].sum() - baseline_errors[indices].sum()
        ) / words[indices].sum()
    probability_better = float(np.mean(deltas < 0))
    selected = bool(delta <= -0.002 and probability_better >= 0.90)
    adapter_decisions[language] = {
        "selected": selected,
        "step": SELECTED_STEP if selected else 0,
        "baseline_wer": baseline_wer,
        "adapter_wer": adapter_wer,
        "delta": delta,
        "ci95_low": float(np.quantile(deltas, 0.025)),
        "ci95_high": float(np.quantile(deltas, 0.975)),
        "probability_adapter_better": probability_better,
    }
    comparison_rows.append({"language": language, **adapter_decisions[language]})


# -----------------------------------------------------------------------------
# 3. Package immuable des adaptateurs selectionnes
# -----------------------------------------------------------------------------

selected_assets = {}
for language in LANGUAGES:
    if not adapter_decisions[language]["selected"]:
        continue
    asset = candidate_assets[language][f"step_{SELECTED_STEP}"]
    selected_assets[language] = {
        "step": SELECTED_STEP,
        "source": asset["adapter_path"],
        "sha256": asset["adapter_sha256"],
        "parameters": PARAMETERS_PER_ADAPTER,
    }

total_parameters = BASE_PARAMETERS + len(selected_assets) * PARAMETERS_PER_ADAPTER
assert total_parameters < OFFICIAL_PARAMETER_LIMIT
package_contract = {
    "schema": 1,
    "base_checkpoint": str(
        FT_ROOT / "balanced_joint_v4/candidates/REPLACE_WITH_LOCAL_RUN_ID/step_1250"
    ),
    "base_weight": str(LOCAL_BASE_WEIGHT),
    "base_weight_sha256": sha256_file(LOCAL_BASE_WEIGHT),
    "selected_adapters": selected_assets,
    "adapter_decisions": adapter_decisions,
    "lm_config_source": str(CANDIDATE_CONFIG_PATH),
    "lm_config_sha256": sha256_file(CANDIDATE_CONFIG_PATH),
    "base_parameters": BASE_PARAMETERS,
    "total_parameters": total_parameters,
    "parameter_limit": OFFICIAL_PARAMETER_LIMIT,
    "adapter_loading": "lazy; one language adapter resident at a time",
}
package_id = canonical_sha256(package_contract)
PACKAGE_ROOT = PACKAGE_PARENT / package_id[:16]
PACKAGE_ROOT.mkdir(parents=True, exist_ok=True)

packaged_assets = {}
for language, asset in selected_assets.items():
    source = Path(asset["source"])
    destination = PACKAGE_ROOT / f"adapter_{language}_step{SELECTED_STEP}.pt"
    if not destination.exists() or sha256_file(destination) != asset["sha256"]:
        temporary = destination.with_name(destination.name + f".tmp-{os.getpid()}")
        shutil.copy2(source, temporary)
        assert sha256_file(temporary) == asset["sha256"]
        os.replace(temporary, destination)
    packaged_assets[language] = {
        **asset, "packaged_path": str(destination),
        "bytes": int(destination.stat().st_size),
    }

lm_destination = PACKAGE_ROOT / "kenlm_tuning_by_domain.json"
shutil.copy2(CANDIDATE_CONFIG_PATH, lm_destination)
assert sha256_file(lm_destination) == package_contract["lm_config_sha256"]
package_manifest = {
    **package_contract,
    "package_id": package_id,
    "packaged_adapters": packaged_assets,
    "packaged_lm_config": str(lm_destination),
}
atomic_json(package_manifest, PACKAGE_ROOT / "package_manifest.json")


# -----------------------------------------------------------------------------
# 4. Sonde RSS dans un processus propre : base + 1 adaptateur + plus gros LM
# -----------------------------------------------------------------------------

score_config_original = json.load(open(SCORE_CONFIG_PATH, encoding="utf-8"))
largest_group, largest_config = max(
    score_config_original.items(),
    key=lambda item: Path(item[1]["lm_bin"]).stat().st_size,
)
largest_language = largest_group.split("|", 1)[0]
largest_unigrams = resolve_unigrams(largest_language, largest_config)
probe_adapter_language = next(iter(packaged_assets), None)

probe_result = None
if probe_adapter_language is not None:
    probe_script = Path("/content/lora_edge_rss_probe_v1.py")
    probe_output = Path("/content/lora_edge_rss_probe_v1.json")
    probe_source = r'''
import gc, json, math, os, resource, sys
from pathlib import Path
import pandas as pd
import psutil
import torch
import torch.nn as nn
import torch.nn.functional as F
from fairseq2.data.tokenizers.hub import load_tokenizer
from fairseq2.models.hub import load_model
from pyctcdecode import build_ctcdecoder

base_weight, plan_path, adapter_path, model_card, lm_bin, unigrams_path, out = sys.argv[1:]

def rss(): return psutil.Process(os.getpid()).memory_info().rss
def dims(module):
    weight = getattr(module, "weight", None)
    if not isinstance(weight, torch.Tensor) or weight.ndim != 2: return None
    name, namespace = type(module).__name__.lower(), type(module).__module__.lower()
    if any(x in name for x in ("embedding", "embed", "conv", "norm")): return None
    if any(True for _ in module.children()): return None
    i = getattr(module, "in_features", getattr(module, "input_dim", weight.shape[1]))
    o = getattr(module, "out_features", getattr(module, "output_dim", weight.shape[0]))
    i, o = int(i), int(o)
    if tuple(weight.shape) != (o, i): return None
    if not (isinstance(module, nn.Linear) or "linear" in name or "projection" in name or "projection" in namespace): return None
    return i, o
class Slot(nn.Module):
    def __init__(self, base, rank):
        super().__init__(); i, o = dims(base); self.base = base
        self.lora_A = nn.Parameter(torch.zeros(rank, i, dtype=base.weight.dtype), requires_grad=False)
        self.lora_B = nn.Parameter(torch.zeros(o, rank, dtype=base.weight.dtype), requires_grad=False)
    def forward(self, x, *a, **kw):
        return self.base(x, *a, **kw) + F.linear(F.linear(x, self.lora_A), self.lora_B)
def parent(root, name):
    parts=name.split("."); value=root
    for part in parts[:-1]: value=getattr(value, part)
    return value, parts[-1]

model = load_model(model_card, device=torch.device("cpu"), dtype=torch.bfloat16)
raw = torch.load(base_weight, map_location="cpu", weights_only=False, mmap=True)
state = raw.get("model", raw) if isinstance(raw, dict) else raw
model.load_state_dict(state); del raw, state; gc.collect()
base_params = sum(p.numel() for p in model.parameters())
plan = pd.read_parquet(plan_path)
for row in plan.sort_values("module").to_dict("records"):
    p, c = parent(model, row["module"]); base = getattr(p, c)
    setattr(p, c, Slot(base, int(row["rank"])))
payload = torch.load(adapter_path, map_location="cpu", weights_only=False)["state_dict"]
params = dict(model.named_parameters())
with torch.no_grad():
    for key, value in payload.items(): params[key].copy_(value.to(dtype=params[key].dtype))
del payload; gc.collect()
tokenizer = load_tokenizer(model_card); tm = tokenizer._model
method = next(x for x in ("index_to_token", "id_to_piece", "IdToPiece") if hasattr(tm, x))
labels = [str(getattr(tm, method)(i)) for i in range(tokenizer.vocab_info.size)]; labels[0] = ""
unigrams = Path(unigrams_path).read_text(encoding="utf-8").splitlines()
decoder = build_ctcdecoder(labels, lm_bin, unigrams=unigrams)
gc.collect()
current = rss(); peak = int(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss * 1024)
json.dump({
  "base_parameters": base_params,
  "resident_adapter_parameters": sum(p.numel() for n,p in model.named_parameters() if n.endswith("lora_A") or n.endswith("lora_B")),
  "rss_bytes": current, "peak_rss_bytes": peak,
  "lm_bin": lm_bin, "unigrams": unigrams_path,
}, open(out, "w"), indent=2)
'''
    probe_script.write_text(textwrap.dedent(probe_source).lstrip(), encoding="utf-8")
    probe = subprocess.run([
        sys.executable, str(probe_script), str(LOCAL_BASE_WEIGHT), str(PLAN_PATH),
        packaged_assets[probe_adapter_language]["packaged_path"], FT_CONFIG["model_card"],
        str(largest_config["lm_bin"]), str(largest_unigrams), str(probe_output),
    ], capture_output=True, text=True, timeout=1800)
    assert probe.returncode == 0, probe.stdout + "\n" + probe.stderr[-5000:]
    probe_result = json.load(open(probe_output, encoding="utf-8"))
    probe_result["rss_gib"] = probe_result["rss_bytes"] / 2**30
    probe_result["peak_rss_gib"] = probe_result["peak_rss_bytes"] / 2**30
    probe_result["under_8_gib"] = (
        probe_result["rss_bytes"] <= RSS_LIMIT_BYTES
        and probe_result["peak_rss_bytes"] <= RSS_LIMIT_BYTES
    )

status = (
    "PASS_READY_FOR_TEST_INFERENCE"
    if selected_assets and probe_result and probe_result["under_8_gib"]
    else "REVIEW"
)
final_report = {
    "schema": 1,
    "status": status,
    "adapter_comparison": comparison_rows,
    "adapter_decisions": adapter_decisions,
    "package": str(PACKAGE_ROOT),
    "package_id": package_id,
    "package_manifest": str(PACKAGE_ROOT / "package_manifest.json"),
    "total_parameters": total_parameters,
    "parameter_limit": OFFICIAL_PARAMETER_LIMIT,
    "rss_probe": probe_result,
    "final_runtime_mutated": False,
    "next_step": "19.7g test inference only if status PASS_READY_FOR_TEST_INFERENCE",
}
atomic_json(final_report, FINAL_REPORT_PATH)

print("\n=== COMPARAISON LORA VS BASE — MEME CONFIG 0.36880 ===")
display(pd.DataFrame(comparison_rows))
print("\n=== SONDE RAM ===")
print(json.dumps(probe_result, indent=2, ensure_ascii=False))
print("STATUT 19.7f :", status)
print("Package :", PACKAGE_ROOT)
print("Rapport :", FINAL_REPORT_PATH)
print("Aucun runtime actif et aucune soumission n'ont ete modifies.")

In [ ]:
# 19.7f-b — Repack BF16 et nouvel audit RAM/latence dans un processus propre
#
# Le pic de 35 Gio de 19.7f venait du chemin de chargement suivant :
#   1) chargement du modele de la carte d'origine ;
#   2) rechargement du checkpoint step_1250 par-dessus.
# Cette cellule publie un checkpoint BF16 autonome et le charge UNE SEULE FOIS via
# une carte dediee. Elle mesure ensuite le vrai pic RSS pendant une transcription
# CPU avec l'adaptateur Maasai et le plus gros KenLM actif.

import gc
import hashlib
import json
import os
import shutil
import subprocess
import sys
import textwrap
import time
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display


required_globals = ("FT_CONFIG", "dev", "omni_lang")
missing = [name for name in required_globals if name not in globals()]
assert not missing, "Relancer 19.7d, 19.7e puis 19.7f avant 19.7f-b : " + ", ".join(missing)

FT_ROOT = Path(
    "/content/afrivoices_project/"
    "finetune_runs/experiment_run"
)
REPORT_DIR = FT_ROOT / "reports"
SOURCE_REPORT = REPORT_DIR / "lora_final_candidate_audit_v1.json"
SOURCE_BASE = Path("/content/low_rank_adapter_sources_v1/base_step1250.pt")
PLAN_PATH = (
    REPORT_DIR / "lora_compatibility_audit_v1/REPLACE_WITH_LOCAL_RUN_ID/lora_plan.parquet"
)
EDGE_PARENT = FT_ROOT / "final_lora_edge_candidate_v1"
EDGE_REPORT = REPORT_DIR / "lora_edge_bf16_audit_v1.json"
LOCAL_BF16 = Path("/content/afrivoices_step1250_edge_bf16.pt")

BASE_PARAMETERS = 975_675_056
ADAPTER_PARAMETERS = 9_898_496
PARAMETER_LIMIT = 1_000_000_000
RSS_LIMIT = 8 * 2**30

for path in (SOURCE_REPORT, SOURCE_BASE, PLAN_PATH):
    assert path.exists(), path


def sha256_file(path, block_size=32 << 20):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()


def canonical_sha256(value):
    raw = json.dumps(
        value, sort_keys=True, ensure_ascii=False, separators=(",", ":"), default=str
    ).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()


def atomic_json(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".tmp-{os.getpid()}")
    with open(temporary, "w", encoding="utf-8") as handle:
        json.dump(value, handle, ensure_ascii=False, indent=2, default=str)
    os.replace(temporary, path)
    print("json ->", path)


source_report = json.load(open(SOURCE_REPORT, encoding="utf-8"))
decisions = source_report["adapter_decisions"]
assert decisions["mas"]["selected"] is True, decisions
assert decisions["kln"]["selected"] is False, decisions
SOURCE_PACKAGE = Path(source_report["package"])
SOURCE_MANIFEST = SOURCE_PACKAGE / "package_manifest.json"
assert SOURCE_MANIFEST.exists(), SOURCE_MANIFEST
source_manifest = json.load(open(SOURCE_MANIFEST, encoding="utf-8"))
MAS_ADAPTER = Path(source_manifest["packaged_adapters"]["mas"]["packaged_path"])
LM_CONFIG = Path(source_manifest["packaged_lm_config"])
assert MAS_ADAPTER.exists() and LM_CONFIG.exists()


# -----------------------------------------------------------------------------
# 1. Conversion BF16 deterministe du state_dict de base
# -----------------------------------------------------------------------------

source_hash = sha256_file(SOURCE_BASE)
conversion_meta_path = LOCAL_BF16.with_suffix(".json")
reuse_bf16 = False
if LOCAL_BF16.exists() and conversion_meta_path.exists():
    previous = json.load(open(conversion_meta_path, encoding="utf-8"))
    reuse_bf16 = (
        previous.get("source_sha256") == source_hash
        and previous.get("bf16_sha256") == sha256_file(LOCAL_BF16)
        and previous.get("parameters") == BASE_PARAMETERS
    )

if not reuse_bf16:
    assert shutil.disk_usage("/content").free >= 5 * 2**30, (
        "Au moins 5 Gio libres sont requis dans /content pour le repack BF16."
    )
    print("Conversion step_1250 FP32 -> BF16 (une seule fois)...")
    raw = torch.load(str(SOURCE_BASE), map_location="cpu", weights_only=False, mmap=True)
    state = raw.get("model", raw) if isinstance(raw, dict) else raw
    assert isinstance(state, dict) and len(state) == 807
    bf16_state = {}
    parameter_elements = 0
    for index, (key, value) in enumerate(state.items(), 1):
        assert isinstance(value, torch.Tensor), (key, type(value))
        parameter_elements += value.numel()
        if torch.is_floating_point(value):
            bf16_state[key] = value.to(dtype=torch.bfloat16).contiguous()
        else:
            bf16_state[key] = value.clone().contiguous()
        if index % 100 == 0 or index == len(state):
            print(f"  {index}/{len(state)} tenseurs convertis")
    assert parameter_elements == BASE_PARAMETERS, parameter_elements
    temporary = Path(str(LOCAL_BF16) + f".tmp-{os.getpid()}")
    torch.save({"model": bf16_state}, temporary)
    os.replace(temporary, LOCAL_BF16)
    del bf16_state, state, raw
    gc.collect()
    bf16_meta = {
        "schema": 1,
        "source": str(SOURCE_BASE),
        "source_sha256": source_hash,
        "bf16_path": str(LOCAL_BF16),
        "bf16_sha256": sha256_file(LOCAL_BF16),
        "bytes": LOCAL_BF16.stat().st_size,
        "parameters": parameter_elements,
        "dtype": "bfloat16",
    }
    atomic_json(bf16_meta, conversion_meta_path)
else:
    bf16_meta = json.load(open(conversion_meta_path, encoding="utf-8"))
    print("Checkpoint BF16 local reutilise :", LOCAL_BF16)

assert bf16_meta["bytes"] < SOURCE_BASE.stat().st_size * 0.60
print(
    "BF16 :", round(bf16_meta["bytes"] / 2**30, 2), "Gio |",
    bf16_meta["bf16_sha256"][:16],
)


# -----------------------------------------------------------------------------
# 2. Publication immuable du package edge
# -----------------------------------------------------------------------------

edge_contract = {
    "schema": 1,
    "source_package": str(SOURCE_PACKAGE),
    "source_package_manifest_sha256": sha256_file(SOURCE_MANIFEST),
    "base_source_sha256": source_hash,
    "base_bf16_sha256": bf16_meta["bf16_sha256"],
    "adapter_language": "mas",
    "adapter_step": 1250,
    "adapter_sha256": sha256_file(MAS_ADAPTER),
    "lm_config_sha256": sha256_file(LM_CONFIG),
    "base_parameters": BASE_PARAMETERS,
    "adapter_parameters": ADAPTER_PARAMETERS,
    "total_parameters": BASE_PARAMETERS + ADAPTER_PARAMETERS,
    "parameter_limit": PARAMETER_LIMIT,
    "loading_policy": "single BF16 checkpoint; one lazy Maasai LoRA slot",
}
assert edge_contract["total_parameters"] < PARAMETER_LIMIT
edge_id = canonical_sha256(edge_contract)
EDGE_ROOT = EDGE_PARENT / edge_id[:16]
EDGE_ROOT.mkdir(parents=True, exist_ok=True)

assets = {
    LOCAL_BF16: (EDGE_ROOT / "base_step1250_bf16.pt", bf16_meta["bf16_sha256"]),
    MAS_ADAPTER: (EDGE_ROOT / "adapter_mas_step1250.pt", edge_contract["adapter_sha256"]),
    LM_CONFIG: (EDGE_ROOT / "kenlm_tuning_by_domain.json", edge_contract["lm_config_sha256"]),
}
for source, (destination, expected_hash) in assets.items():
    if not destination.exists() or sha256_file(destination) != expected_hash:
        temporary = destination.with_name(destination.name + f".tmp-{os.getpid()}")
        shutil.copy2(source, temporary)
        assert sha256_file(temporary) == expected_hash
        os.replace(temporary, destination)
    assert sha256_file(destination) == expected_hash
    print("asset ->", destination, f"({destination.stat().st_size / 2**30:.2f} Gio)")

EDGE_BASE = EDGE_ROOT / "base_step1250_bf16.pt"
EDGE_ADAPTER = EDGE_ROOT / "adapter_mas_step1250.pt"
EDGE_LM_CONFIG = EDGE_ROOT / "kenlm_tuning_by_domain.json"

# La carte remplace le checkpoint de la carte parente : load_model ne charge donc
# plus le modele d'origine avant de recharger step_1250.
import omnilingual_asr as oa

EDGE_CARD_NAME = f"afrivoices_step1250_edge_bf16_{edge_id[:12]}"
EDGE_CARD_PATH = Path(oa.__file__).parent / "cards/models" / f"{EDGE_CARD_NAME}.yaml"
EDGE_CARD_PATH.parent.mkdir(parents=True, exist_ok=True)
EDGE_CARD_PATH.write_text(
    f"name: {EDGE_CARD_NAME}\n"
    f"base: {FT_CONFIG['model_card']}\n"
    f"checkpoint: \"file://{EDGE_BASE}\"\n",
    encoding="utf-8",
)
shutil.copy2(EDGE_CARD_PATH, EDGE_ROOT / "model_card.yaml")


# -----------------------------------------------------------------------------
# 3. Choix d'une courte sonde Maasai jamais issue du test Kaggle
# -----------------------------------------------------------------------------

probe_rows = dev[
    (dev.language == "mas")
    & (dev.decode_domain == "unscripted")
].copy()
assert len(probe_rows)
if "duration" in probe_rows.columns:
    probe_rows = probe_rows.sort_values("duration")
elif "seconds" in probe_rows.columns:
    probe_rows = probe_rows.sort_values("seconds")
PROBE_AUDIO = Path(str(probe_rows.iloc[0]["eval_audio_path"]))
assert PROBE_AUDIO.exists(), PROBE_AUDIO
PROBE_LANGUAGE = omni_lang("mas")

lm_config = json.load(open(EDGE_LM_CONFIG, encoding="utf-8"))
largest_group, largest_lm = max(
    lm_config.items(), key=lambda item: Path(item[1]["lm_bin"]).stat().st_size
)
largest_language = largest_group.split("|", 1)[0]


def resolve_unigrams(language, config):
    for key in ("unigram_file", "unigrams_file", "uni_file", "unigrams"):
        value = config.get(key)
        if isinstance(value, str) and Path(value).exists():
            return Path(value)
    lm_bin = Path(config["lm_bin"])
    candidates = [
        lm_bin.with_name(f"{language}_unigrams.txt"),
        FT_ROOT / f"kenlm_models_v5/{language}_unigrams.txt",
        FT_ROOT / f"kenlm_models_v4/{language}_unigrams.txt",
    ]
    found = next((path for path in candidates if path.exists()), None)
    assert found is not None, candidates
    return found


LARGEST_UNIGRAMS = resolve_unigrams(largest_language, largest_lm)


# -----------------------------------------------------------------------------
# 4. Processus neuf : chargement unique, LoRA, plus gros LM, vraie transcription
# -----------------------------------------------------------------------------

probe_script = Path("/content/lora_edge_bf16_runtime_probe.py")
probe_output = Path("/content/lora_edge_bf16_runtime_probe.json")
probe_source = r'''
import gc, json, os, resource, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import psutil
import torch
import torch.nn as nn
import torch.nn.functional as F
from fairseq2.data.tokenizers.hub import load_tokenizer
from fairseq2.models.hub import load_model
from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline
from pyctcdecode import build_ctcdecoder

card, plan_path, adapter_path, lm_bin, unigrams_path, audio_path, language_code, out = sys.argv[1:]
process = psutil.Process(os.getpid())
timeline = []
def rss(): return process.memory_info().rss
def mark(stage):
    gc.collect()
    timeline.append({"stage": stage, "rss_bytes": rss(),
                     "peak_rss_bytes": int(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss * 1024)})
def dims(module):
    weight = getattr(module, "weight", None)
    if not isinstance(weight, torch.Tensor) or weight.ndim != 2: return None
    name, namespace = type(module).__name__.lower(), type(module).__module__.lower()
    if any(x in name for x in ("embedding", "embed", "conv", "norm")): return None
    if any(True for _ in module.children()): return None
    i = int(getattr(module, "in_features", getattr(module, "input_dim", weight.shape[1])))
    o = int(getattr(module, "out_features", getattr(module, "output_dim", weight.shape[0])))
    if tuple(weight.shape) != (o, i): return None
    if not (isinstance(module, nn.Linear) or "linear" in name or "projection" in name or "projection" in namespace): return None
    return i, o
class Slot(nn.Module):
    def __init__(self, base, rank):
        super().__init__(); i, o = dims(base); self.base = base
        self.lora_A = nn.Parameter(torch.zeros(rank, i, dtype=base.weight.dtype), requires_grad=False)
        self.lora_B = nn.Parameter(torch.zeros(o, rank, dtype=base.weight.dtype), requires_grad=False)
    def forward(self, x, *args, **kwargs):
        return self.base(x, *args, **kwargs) + F.linear(F.linear(x, self.lora_A), self.lora_B)
def parent(root, name):
    parts = name.split("."); value = root
    for part in parts[:-1]: value = getattr(value, part)
    return value, parts[-1]

torch.set_grad_enabled(False)
mark("start")
t0 = time.perf_counter()
model = load_model(card, device=torch.device("cpu"), dtype=torch.bfloat16)
model.eval()
model_load_seconds = time.perf_counter() - t0
base_parameters = sum(p.numel() for p in model.parameters())
mark("base_loaded_once")

plan = pd.read_parquet(plan_path)
for row in plan.sort_values("module").to_dict("records"):
    p, child = parent(model, row["module"])
    setattr(p, child, Slot(getattr(p, child), int(row["rank"])))
payload = torch.load(adapter_path, map_location="cpu", weights_only=False, mmap=True)["state_dict"]
parameters = dict(model.named_parameters())
with torch.no_grad():
    for key, value in payload.items():
        parameters[key].copy_(value.to(dtype=parameters[key].dtype))
del payload
adapter_parameters = sum(
    p.numel() for name, p in model.named_parameters()
    if name.endswith("lora_A") or name.endswith("lora_B")
)
mark("adapter_loaded")

tokenizer = load_tokenizer(card)
tm = tokenizer._model
method = next(name for name in ("index_to_token", "id_to_piece", "IdToPiece") if hasattr(tm, name))
labels = [str(getattr(tm, method)(i)) for i in range(tokenizer.vocab_info.size)]
labels[0] = ""
unigrams = Path(unigrams_path).read_text(encoding="utf-8").splitlines()
decoder = build_ctcdecoder(labels, lm_bin, unigrams=unigrams)
decoder.decode(np.zeros((3, len(labels)), dtype=np.float32), beam_width=10)
mark("largest_kenlm_loaded")

pipe = ASRInferencePipeline(None, model=model, tokenizer=tokenizer)
t1 = time.perf_counter()
hypothesis = str(pipe.transcribe([audio_path], lang=[language_code], batch_size=1)[0])
inference_seconds = time.perf_counter() - t1
mark("one_real_inference_complete")

peak = max(item["peak_rss_bytes"] for item in timeline)
current = rss()
json.dump({
    "base_parameters": base_parameters,
    "resident_adapter_parameters": adapter_parameters,
    "total_neural_parameters": base_parameters + adapter_parameters,
    "model_load_seconds": model_load_seconds,
    "inference_seconds": inference_seconds,
    "hypothesis_nonempty": bool(hypothesis.strip()),
    "hypothesis_word_count": len(hypothesis.split()),
    "rss_bytes": current,
    "peak_rss_bytes": peak,
    "timeline": timeline,
    "lm_bin": lm_bin,
    "probe_audio": audio_path,
}, open(out, "w"), ensure_ascii=False, indent=2)
'''
probe_script.write_text(textwrap.dedent(probe_source).lstrip(), encoding="utf-8")

print("\nSonde edge propre : chargement BF16 unique + LoRA mas + plus gros KenLM + 1 audio...")
probe = subprocess.run(
    [
        sys.executable, str(probe_script), EDGE_CARD_NAME, str(PLAN_PATH),
        str(EDGE_ADAPTER), str(largest_lm["lm_bin"]), str(LARGEST_UNIGRAMS),
        str(PROBE_AUDIO), PROBE_LANGUAGE, str(probe_output),
    ],
    capture_output=True, text=True, timeout=2700,
)
if probe.returncode != 0:
    print(probe.stdout[-3000:])
    print(probe.stderr[-5000:])
assert probe.returncode == 0, "La sonde BF16 a echoue; envoyer les sorties ci-dessus."
runtime = json.load(open(probe_output, encoding="utf-8"))
runtime["rss_gib"] = runtime["rss_bytes"] / 2**30
runtime["peak_rss_gib"] = runtime["peak_rss_bytes"] / 2**30
runtime["under_8_gib"] = bool(
    runtime["rss_bytes"] <= RSS_LIMIT and runtime["peak_rss_bytes"] <= RSS_LIMIT
)
runtime["parameter_compliant"] = bool(
    runtime["total_neural_parameters"] < PARAMETER_LIMIT
)

status = (
    "PASS_READY_FOR_TEST_INFERENCE"
    if runtime["under_8_gib"]
    and runtime["parameter_compliant"]
    and runtime["hypothesis_nonempty"]
    else "REVIEW"
)
edge_manifest = {
    **edge_contract,
    "edge_id": edge_id,
    "edge_root": str(EDGE_ROOT),
    "base_bf16": str(EDGE_BASE),
    "base_bf16_bytes": EDGE_BASE.stat().st_size,
    "adapter": str(EDGE_ADAPTER),
    "lm_config": str(EDGE_LM_CONFIG),
    "model_card": str(EDGE_ROOT / "model_card.yaml"),
    "runtime_probe": runtime,
    "status": status,
}
atomic_json(edge_manifest, EDGE_ROOT / "edge_manifest.json")
atomic_json(edge_manifest, EDGE_REPORT)

print("\n=== AUDIT EDGE BF16 19.7f-b ===")
display(pd.DataFrame(runtime["timeline"]).assign(
    rss_gib=lambda frame: frame.rss_bytes / 2**30,
    peak_rss_gib=lambda frame: frame.peak_rss_bytes / 2**30,
))
print("Parametres :", f"{runtime['total_neural_parameters']:,}", "< 1,000,000,000")
print("RSS final  :", round(runtime["rss_gib"], 3), "Gio")
print("Pic RSS    :", round(runtime["peak_rss_gib"], 3), "Gio")
print("Latence CPU sonde :", round(runtime["inference_seconds"], 2), "s")
print("STATUT 19.7f-b :", status)
print("Package edge :", EDGE_ROOT)
print("Rapport      :", EDGE_REPORT)
print("Ne pas lancer 17.2. Etape suivante : 19.7g seulement si le statut est PASS.")

In [ ]:
# 19.7f-c — Correction de la mesure du pic RSS avec moniteur externe
#
# Dans 19.7f-b, resource.ru_maxrss valait deja 35.24 Gio a l'etape "start" :
# ce high-water mark etait herite du noyau Colab. Cette cellule surveille le RSS
# COURANT du worker depuis le processus parent toutes les 5 ms, puis ajoute une
# marge conservatrice de 15 %. Elle ne reconvertit ni ne modifie les poids.

import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd
import psutil
from IPython.display import display


assert "omni_lang" in globals(), "Relancer les definitions du notebook avant 19.7f-c."

FT_ROOT = Path(
    "/content/afrivoices_project/"
    "finetune_runs/experiment_run"
)
REPORT_DIR = FT_ROOT / "reports"
V1_REPORT = REPORT_DIR / "lora_edge_bf16_audit_v1.json"
V2_REPORT = REPORT_DIR / "lora_edge_bf16_audit_v2.json"
PLAN_PATH = (
    REPORT_DIR / "lora_compatibility_audit_v1/REPLACE_WITH_LOCAL_RUN_ID/lora_plan.parquet"
)
WORKER = Path("/content/lora_edge_bf16_runtime_probe.py")
WORKER_OUTPUT = Path("/content/lora_edge_bf16_runtime_probe_reaudit.json")
WORKER_LOG = Path("/content/lora_edge_bf16_runtime_probe_reaudit.log")

RSS_LIMIT = 8 * 2**30
SAFETY_FACTOR = 1.15
SAMPLE_INTERVAL_SECONDS = 0.005
BASE_PARAMETERS = 975_675_056
ADAPTER_PARAMETERS = 9_898_496
PARAMETER_LIMIT = 1_000_000_000

for path in (V1_REPORT, PLAN_PATH, WORKER):
    assert path.exists(), path


def sha256_file(path, block_size=32 << 20):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()


def atomic_json(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".tmp-{os.getpid()}")
    with open(temporary, "w", encoding="utf-8") as handle:
        json.dump(value, handle, ensure_ascii=False, indent=2, default=str)
    os.replace(temporary, path)
    print("json ->", path)


v1 = json.load(open(V1_REPORT, encoding="utf-8"))
EDGE_ROOT = Path(v1["edge_root"])
EDGE_BASE = Path(v1["base_bf16"])
EDGE_ADAPTER = Path(v1["adapter"])
EDGE_LM_CONFIG = Path(v1["lm_config"])
PACKAGE_CARD = Path(v1["model_card"])
for path in (EDGE_ROOT, EDGE_BASE, EDGE_ADAPTER, EDGE_LM_CONFIG, PACKAGE_CARD):
    assert path.exists(), path

# Reenregistrer la carte si necessaire, sans toucher au package.
card_text = PACKAGE_CARD.read_text(encoding="utf-8")
card_line = next(line for line in card_text.splitlines() if line.strip().startswith("name:"))
EDGE_CARD_NAME = card_line.split(":", 1)[1].strip().strip('"').strip("'")
import omnilingual_asr as oa

REGISTERED_CARD = Path(oa.__file__).parent / "cards/models" / f"{EDGE_CARD_NAME}.yaml"
REGISTERED_CARD.parent.mkdir(parents=True, exist_ok=True)
if not REGISTERED_CARD.exists() or REGISTERED_CARD.read_text(encoding="utf-8") != card_text:
    REGISTERED_CARD.write_text(card_text, encoding="utf-8")

lm_config = json.load(open(EDGE_LM_CONFIG, encoding="utf-8"))
largest_group, largest_lm = max(
    lm_config.items(), key=lambda item: Path(item[1]["lm_bin"]).stat().st_size
)
largest_language = largest_group.split("|", 1)[0]


def resolve_unigrams(language, config):
    for key in ("unigram_file", "unigrams_file", "uni_file", "unigrams"):
        value = config.get(key)
        if isinstance(value, str) and Path(value).exists():
            return Path(value)
    lm_bin = Path(config["lm_bin"])
    candidates = [
        lm_bin.with_name(f"{language}_unigrams.txt"),
        FT_ROOT / f"kenlm_models_v5/{language}_unigrams.txt",
        FT_ROOT / f"kenlm_models_v4/{language}_unigrams.txt",
    ]
    found = next((path for path in candidates if path.exists()), None)
    assert found is not None, candidates
    return found


largest_unigrams = resolve_unigrams(largest_language, largest_lm)
probe_audio = Path(v1["runtime_probe"]["probe_audio"])
assert probe_audio.exists(), probe_audio

command = [
    sys.executable, str(WORKER), EDGE_CARD_NAME, str(PLAN_PATH),
    str(EDGE_ADAPTER), str(largest_lm["lm_bin"]), str(largest_unigrams),
    str(probe_audio), omni_lang("mas"), str(WORKER_OUTPUT),
]

print("Sonde RSS externe : echantillonnage toutes les 5 ms...")
samples = []
peak_rss = 0
started = time.monotonic()
with open(WORKER_LOG, "w", encoding="utf-8") as log:
    worker = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, text=True)
    monitored = psutil.Process(worker.pid)
    deadline = started + 2700
    while worker.poll() is None:
        now = time.monotonic()
        if now > deadline:
            worker.kill()
            raise TimeoutError("Sonde edge > 45 minutes.")
        try:
            rss = monitored.memory_info().rss
            for child in monitored.children(recursive=True):
                try:
                    rss += child.memory_info().rss
                except (psutil.NoSuchProcess, psutil.AccessDenied):
                    pass
            peak_rss = max(peak_rss, rss)
            # Une trace a 10 Hz suffit pour le tableau; le maximum reste mesure a 200 Hz.
            if not samples or now - samples[-1]["monotonic"] >= 0.1:
                samples.append({
                    "monotonic": now,
                    "elapsed_seconds": now - started,
                    "rss_bytes": int(rss),
                })
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
        time.sleep(SAMPLE_INTERVAL_SECONDS)
    returncode = worker.wait()

if returncode != 0:
    print(WORKER_LOG.read_text(encoding="utf-8", errors="replace")[-6000:])
assert returncode == 0, f"Sonde worker en echec : code {returncode}"
assert WORKER_OUTPUT.exists(), WORKER_OUTPUT

runtime = json.load(open(WORKER_OUTPUT, encoding="utf-8"))
assert runtime["base_parameters"] == BASE_PARAMETERS
assert runtime["resident_adapter_parameters"] == ADAPTER_PARAMETERS
assert runtime["total_neural_parameters"] < PARAMETER_LIMIT
assert runtime["hypothesis_nonempty"]

conservative_peak = int(peak_rss * SAFETY_FACTOR)
runtime.update({
    # resource.ru_maxrss est conserve uniquement comme diagnostic invalide herite.
    "inherited_ru_maxrss_bytes": runtime.pop("peak_rss_bytes", None),
    "externally_monitored_peak_rss_bytes": int(peak_rss),
    "externally_monitored_peak_rss_gib": peak_rss / 2**30,
    "safety_factor": SAFETY_FACTOR,
    "conservative_peak_rss_bytes": conservative_peak,
    "conservative_peak_rss_gib": conservative_peak / 2**30,
    "rss_limit_bytes": RSS_LIMIT,
    "under_8_gib_with_15pct_margin": conservative_peak <= RSS_LIMIT,
    "monitor_interval_seconds": SAMPLE_INTERVAL_SECONDS,
    "monitor_samples": len(samples),
    "wall_seconds": time.monotonic() - started,
})

status = (
    "PASS_READY_FOR_TEST_INFERENCE"
    if runtime["under_8_gib_with_15pct_margin"]
    else "REVIEW"
)
report = {
    **v1,
    "schema": 2,
    "status": status,
    "measurement_correction": (
        "ru_maxrss etait deja eleve avant le chargement; le pic v2 est mesure "
        "depuis un moniteur externe sur le RSS courant du worker a 200 Hz"
    ),
    "runtime_probe_v2": runtime,
    "worker_log": str(WORKER_LOG),
    "source_report_v1": str(V1_REPORT),
    "source_report_v1_sha256": sha256_file(V1_REPORT),
    "next_step": "19.7g only if PASS_READY_FOR_TEST_INFERENCE",
}
atomic_json(report, EDGE_ROOT / "edge_manifest_v2.json")
atomic_json(report, V2_REPORT)

sample_frame = pd.DataFrame(samples)
if len(sample_frame):
    sample_frame["rss_gib"] = sample_frame.rss_bytes / 2**30
    # Afficher un resume temporel compact plutot que plusieurs milliers de lignes.
    display(pd.concat([
        sample_frame.head(3),
        sample_frame.nlargest(5, "rss_bytes"),
        sample_frame.tail(3),
    ]).drop_duplicates().sort_values("elapsed_seconds"))

print("\n=== AUDIT RSS CORRIGE 19.7f-c ===")
print("Pic mesure externe :", round(runtime["externally_monitored_peak_rss_gib"], 3), "Gio")
print("Pic + marge 15 %   :", round(runtime["conservative_peak_rss_gib"], 3), "Gio")
print("RSS limite         : 8.000 Gio")
print("Latence CPU        :", round(runtime["inference_seconds"], 2), "s")
print("Parametres         :", f"{runtime['total_neural_parameters']:,}")
print("STATUT 19.7f-c     :", status)
print("Rapport             :", V2_REPORT)
print("Ne pas lancer 17.2. Envoyer cette sortie avant 19.7g.")

In [ ]:
# 19.7g — Inference test ciblee : LoRA uniquement sur mas|unscripted
#
# Strategie prudente :
# - repartir exactement du CSV leaderboard 0.36880 ;
# - conserver toutes ses predictions, y compris mas|scripted ;
# - recalculer seulement les shards mas|unscripted avec le LoRA step_1250 ;
# - appliquer la configuration KenLM validee en 19.7e ;
# - sauvegarder chaque shard sur Drive pour une reprise automatique ;
# - produire un NOUVEAU CSV, sans ecraser l'ancienne soumission.

import base64
import gc
import hashlib
import io
import json
import os
import re
import shutil
import subprocess
import time
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import torch
from pyctcdecode import build_ctcdecoder


required_globals = ("inject_eval_lora", "labels_from_tokenizer", "omni_lang")
missing = [name for name in required_globals if name not in globals()]
assert not missing, "Relancer 19.7d avant 19.7g : " + ", ".join(missing)
assert torch.cuda.is_available(), (
    "19.7g requiert un GPU. Un L4 24 Gio suffit ; reconnecter un GPU puis "
    "relancer les definitions necessaires."
)

FT_ROOT = Path(
    "/content/afrivoices_project/"
    "finetune_runs/experiment_run"
)
REPORT_DIR = FT_ROOT / "reports"
EDGE_REPORT = REPORT_DIR / "lora_edge_bf16_audit_v2.json"
OLD_SUBMISSION = REPORT_DIR / "submission_v3_REPLACE_WITH_LOCAL_RUN_ID.csv"
OLD_SUBMISSION_SHA256 = "REPLACE_WITH_LOCAL_SHA256"
EXPECTED_TOTAL_ROWS = 41_733

TEST_ROOT_CANDIDATES = (
    Path("/content/kaggle_test_full_fast"),
    Path("/content/kaggle_test_full"),
    Path(
        "/content/afrivoices_project/"
        "kaggle_test_full"
    ),
)
CHUNKING = {
    "window_s": 38.0,
    "overlap_s": 6.0,
    "trim_s": 3.0,
    "short_threshold_s": 39.5,
}
_B64_CHARS = set(b"ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789+/=\n\r")


def sha256_file(path, block_size=16 << 20):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()


def canonical_sha256(value):
    raw = json.dumps(
        value, sort_keys=True, ensure_ascii=False, separators=(",", ":"), default=str
    ).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()


def sequence_sha256(values):
    return canonical_sha256([str(value) for value in values])


def atomic_json(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".tmp-{os.getpid()}")
    with open(temporary, "w", encoding="utf-8") as handle:
        json.dump(value, handle, ensure_ascii=False, indent=2, default=str)
    os.replace(temporary, path)
    print("json ->", path)


def atomic_csv(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".tmp-{os.getpid()}")
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def resolve_unigrams(language, config):
    for key in ("unigram_file", "unigrams_file", "uni_file", "unigrams"):
        value = config.get(key)
        if isinstance(value, str) and Path(value).exists():
            return Path(value)
    lm_bin = Path(config["lm_bin"])
    candidates = [
        lm_bin.with_name(f"{language}_unigrams.txt"),
        FT_ROOT / f"kenlm_models_v5/{language}_unigrams.txt",
        FT_ROOT / f"kenlm_models_v4/{language}_unigrams.txt",
    ]
    found = next((path for path in candidates if path.exists()), None)
    assert found is not None, candidates
    return found


for path in (EDGE_REPORT, OLD_SUBMISSION):
    assert path.exists(), path
assert sha256_file(OLD_SUBMISSION) == OLD_SUBMISSION_SHA256, (
    "Le CSV de base n'est plus le fichier ayant obtenu 0.36880."
)

edge = json.load(open(EDGE_REPORT, encoding="utf-8"))
assert edge["status"] == "PASS_READY_FOR_TEST_INFERENCE", edge["status"]
assert edge["runtime_probe_v2"]["under_8_gib_with_15pct_margin"]
EDGE_ROOT = Path(edge["edge_root"])
EDGE_BASE = Path(edge["base_bf16"])
EDGE_ADAPTER = Path(edge["adapter"])
EDGE_LM_CONFIG = Path(edge["lm_config"])
PACKAGE_CARD = Path(edge["model_card"])
for path in (EDGE_ROOT, EDGE_BASE, EDGE_ADAPTER, EDGE_LM_CONFIG, PACKAGE_CARD):
    assert path.exists(), path


# -----------------------------------------------------------------------------
# 1. Inventaire strict des shards mas|unscripted
# -----------------------------------------------------------------------------

TEST_ROOT = next((path for path in TEST_ROOT_CANDIDATES if path.exists()), None)
assert TEST_ROOT is not None, (
    "Test absent. Restaurer /content/kaggle_test_full_fast ou monter le Drive."
)
all_unscripted = sorted(TEST_ROOT.rglob("test_unscripted_*.parquet"))
mas_shards = [
    path for path in all_unscripted
    if "mas" in {part.lower() for part in path.parts}
    or "maasai" in {part.lower() for part in path.parts}
]
assert mas_shards, f"Aucun shard mas|unscripted sous {TEST_ROOT}"
assert len({path.resolve() for path in mas_shards}) == len(mas_shards)

old = pd.read_csv(OLD_SUBMISSION, dtype=str)
assert list(old.columns) == ["id", "language", "transcription"], old.columns.tolist()
assert len(old) == EXPECTED_TOTAL_ROWS and old.id.is_unique
assert set(old.language) == {"kik", "kln", "luo", "mas", "som", "swa"}
old_by_id = old.set_index("id", drop=False)

shard_inventory = []
all_target_ids = []
for shard in mas_shards:
    ids = pq.read_table(shard, columns=["id"]).column("id").to_pylist()
    ids = [str(value) for value in ids]
    assert len(ids) == len(set(ids))
    assert set(ids) <= set(old_by_id.index), (shard, "IDs absents du CSV 0.36880")
    all_target_ids.extend(ids)
    shard_inventory.append({
        "path": str(shard),
        "relative": str(shard.relative_to(TEST_ROOT)),
        "bytes": shard.stat().st_size,
        "rows": len(ids),
        "ids_sha256": sequence_sha256(ids),
    })
assert len(all_target_ids) == len(set(all_target_ids)), "IDs dupliques entre shards"
assert old_by_id.loc[all_target_ids, "language"].eq("mas").all()
print("Test root :", TEST_ROOT)
print("Shards mas|unscripted :", len(mas_shards), "| clips :", len(all_target_ids))


# -----------------------------------------------------------------------------
# 2. Carte, modele BF16, LoRA Maasai et decodeur valide
# -----------------------------------------------------------------------------

card_text = PACKAGE_CARD.read_text(encoding="utf-8")
card_line = next(line for line in card_text.splitlines() if line.strip().startswith("name:"))
EDGE_CARD_NAME = card_line.split(":", 1)[1].strip().strip('"').strip("'")
import omnilingual_asr as oa

registered_card = Path(oa.__file__).parent / "cards/models" / f"{EDGE_CARD_NAME}.yaml"
registered_card.parent.mkdir(parents=True, exist_ok=True)
registered_card.write_text(card_text, encoding="utf-8")

from fairseq2.data.tokenizers.hub import load_tokenizer
from fairseq2.models.hub import load_model
from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline

for name in ("model", "pipe", "wrappers", "decoder"):
    globals().pop(name, None)
gc.collect(); torch.cuda.empty_cache()

device = torch.device("cuda")
BASE_ARCH_CARD = globals().get("FT_CONFIG", {}).get(
    "model_card", "omniASR_CTC_1B_v2"
)
try:
    # Fonctionne dans un processus neuf, lorsque le registre a lu la carte edge.
    model = load_model(EDGE_CARD_NAME, device=device, dtype=torch.bfloat16)
except Exception as error:
    if type(error).__name__ != "ModelNotKnownError":
        raise
    # Dans un noyau deja initialise, Fairseq2 ne rescane pas les nouvelles cartes.
    # Charger alors l'architecture connue puis appliquer UNE fois les poids BF16 edge.
    print(
        "Carte edge non visible dans le registre courant ; "
        "chargement de l'architecture puis application du checkpoint BF16."
    )
    model = load_model(BASE_ARCH_CARD, device=device, dtype=torch.bfloat16)
    edge_raw = torch.load(
        str(EDGE_BASE), map_location="cpu", weights_only=False, mmap=True
    )
    edge_state = edge_raw.get("model", edge_raw) if isinstance(edge_raw, dict) else edge_raw
    incompatible = model.load_state_dict(edge_state)
    assert not incompatible.missing_keys and not incompatible.unexpected_keys, incompatible
    del edge_raw, edge_state
    gc.collect(); torch.cuda.empty_cache()
model.eval()
assert sum(parameter.numel() for parameter in model.parameters()) == 975_675_056
wrappers = inject_eval_lora(model)
assert len(wrappers) == 289

adapter_raw = torch.load(
    str(EDGE_ADAPTER), map_location="cpu", weights_only=False, mmap=True
)
adapter_state = adapter_raw["state_dict"]
parameters = dict(model.named_parameters())
expected_adapter_keys = {
    f"{module}.lora_A" for module in wrappers
} | {
    f"{module}.lora_B" for module in wrappers
}
assert set(adapter_state) == expected_adapter_keys
adapter_elements = 0
with torch.no_grad():
    for key, value in adapter_state.items():
        target = parameters[key]
        target.copy_(value.to(device=target.device, dtype=target.dtype))
        adapter_elements += target.numel()
    for wrapper in wrappers.values():
        wrapper.enabled = True
assert adapter_elements == 9_898_496
del adapter_raw, adapter_state
gc.collect(); torch.cuda.empty_cache()

# Le tokenizer est herite sans modification de la carte acoustique d'origine.
tokenizer = load_tokenizer(BASE_ARCH_CARD)
labels = labels_from_tokenizer(tokenizer)
pipe = ASRInferencePipeline(None, model=model, tokenizer=tokenizer)

lm_config = json.load(open(EDGE_LM_CONFIG, encoding="utf-8"))
config = lm_config["mas|unscripted"]
lm_bin = Path(config["lm_bin"])
unigrams_path = resolve_unigrams("mas", config)
assert lm_bin.exists() and unigrams_path.exists()
decoder = build_ctcdecoder(
    labels,
    str(lm_bin),
    unigrams=unigrams_path.read_text(encoding="utf-8").splitlines(),
    alpha=float(config["alpha"]),
    beta=float(config["beta"]),
)
decode_kwargs = {"beam_width": int(config["beam"])}
if config.get("tml") is not None:
    decode_kwargs["token_min_logp"] = float(config["tml"])
if config.get("bpl") is not None:
    decode_kwargs["beam_prune_logp"] = float(config["bpl"])


# -----------------------------------------------------------------------------
# 3. Audio, fenetrage et capture CTC batch=1 (chemin valide par 17.1e)
# -----------------------------------------------------------------------------

_PUNCT_RE = re.compile(r"[^\w\s'’-]", flags=re.UNICODE)
_WS_RE = re.compile(r"\s+")


def normalize_prediction(value):
    value = unicodedata.normalize("NFC", str(value or "")).lower()
    value = _PUNCT_RE.sub(" ", value).replace("_", " ")
    return _WS_RE.sub(" ", value).strip()


def ffmpeg_decode(raw, sample_rate=16000):
    try:
        process = subprocess.run(
            [
                "ffmpeg", "-v", "quiet", "-nostdin", "-i", "pipe:0",
                "-f", "f32le", "-ac", "1", "-ar", str(sample_rate), "pipe:1",
            ],
            input=raw, capture_output=True, check=True,
        )
        waveform = np.frombuffer(process.stdout, dtype=np.float32).copy()
        return (waveform, sample_rate) if len(waveform) else (None, None)
    except Exception:
        return None, None


def decode_audio(cell):
    raw = cell.get("bytes") if isinstance(cell, dict) else cell
    if raw is None:
        return None, None
    if hasattr(raw, "tobytes"):
        raw = raw.tobytes()
    elif isinstance(raw, (memoryview, bytearray)):
        raw = bytes(raw)
    if raw[:5] == b"UklGR" or (len(raw) > 32 and set(raw[:80]) <= _B64_CHARS):
        try:
            decoded = base64.b64decode(raw, validate=False)
            if decoded[:4] in (b"RIFF", b"fLaC", b"OggS") or decoded[:3] == b"ID3":
                raw = decoded
        except Exception:
            pass
    try:
        waveform, sample_rate = sf.read(
            io.BytesIO(raw), dtype="float32", always_2d=False
        )
    except Exception:
        return ffmpeg_decode(raw)
    if getattr(waveform, "ndim", 1) > 1:
        waveform = waveform.mean(axis=1)
    return waveform, int(sample_rate)


def windows(waveform, sample_rate):
    threshold = int(CHUNKING["short_threshold_s"] * sample_rate)
    if len(waveform) <= threshold:
        return [(waveform, False, False)]
    window = int(CHUNKING["window_s"] * sample_rate)
    hop = int((CHUNKING["window_s"] - CHUNKING["overlap_s"]) * sample_rate)
    output, offset = [], 0
    while offset < len(waveform):
        segment = waveform[offset:offset + window]
        last = offset + window >= len(waveform)
        if len(segment) >= int(0.2 * sample_rate):
            output.append((segment, offset > 0, not last))
        if last:
            break
        offset += hop
    return output


def capture_log_probs(item):
    captured = []
    original_forward = pipe.model.forward

    def spy(source_seqs, source_seq_lens, *args, **kwargs):
        output = original_forward(source_seqs, source_seq_lens, *args, **kwargs)
        logits, layout = output[0], output[1]
        length = layout.seq_lens[0]
        length = int(length.item() if hasattr(length, "item") else length)
        captured.append(
            torch.log_softmax(logits[0, :length].detach().float(), dim=-1)
            .cpu().numpy()
        )
        return output

    pipe.model.forward = spy
    try:
        with torch.inference_mode():
            pipe.transcribe([item], lang=[omni_lang("mas")], batch_size=1)
    finally:
        pipe.model.forward = original_forward
    assert len(captured) == 1
    return captured[0]


def transcribe_waveform(waveform, sample_rate):
    segments = windows(waveform, sample_rate)
    parts = []
    for segment, trim_left, trim_right in segments:
        logits = capture_log_probs({"waveform": segment, "sample_rate": sample_rate})
        frames_per_second = len(logits) / (len(segment) / float(sample_rate))
        trim = int(round(CHUNKING["trim_s"] * frames_per_second))
        left = trim if trim_left else 0
        right = len(logits) - (trim if trim_right else 0)
        assert right > left
        parts.append(logits[left:right])
    full = parts[0] if len(parts) == 1 else np.concatenate(parts, axis=0)
    return normalize_prediction(decoder.decode(full, **decode_kwargs))


# -----------------------------------------------------------------------------
# 4. Inference reprenable par shard
# -----------------------------------------------------------------------------

run_contract = {
    "schema": 1,
    "purpose": "replace only mas|unscripted in frozen leaderboard-0.36880 CSV",
    "edge_report_sha256": sha256_file(EDGE_REPORT),
    "edge_base_sha256": sha256_file(EDGE_BASE),
    "adapter_sha256": sha256_file(EDGE_ADAPTER),
    "lm_bin_sha256": sha256_file(lm_bin),
    "unigrams_sha256": sha256_file(unigrams_path),
    "lm_parameters": {
        "alpha": float(config["alpha"]), "beta": float(config["beta"]),
        "beam": int(config["beam"]),
    },
    "old_submission_sha256": OLD_SUBMISSION_SHA256,
    "chunking": CHUNKING,
    "batch_size": 1,
    "test_root": str(TEST_ROOT),
    "shards": shard_inventory,
}
RUN_ID = canonical_sha256(run_contract)[:16]
CACHE_ROOT = REPORT_DIR / "test_predictions_lora_mas_unscripted_v1" / RUN_ID
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
atomic_json(run_contract, CACHE_ROOT / "run_contract.json")

prediction_parts = []
bad_audio_total = 0
for shard_index, shard in enumerate(mas_shards, 1):
    inventory = next(item for item in shard_inventory if item["path"] == str(shard))
    output = CACHE_ROOT / f"{shard.stem}.csv"
    metadata_path = output.with_suffix(".meta.json")
    shard_spec = {"run_id": RUN_ID, **inventory}
    shard_spec_hash = canonical_sha256(shard_spec)
    valid = False
    if output.exists() and metadata_path.exists():
        try:
            metadata = json.load(open(metadata_path, encoding="utf-8"))
            cached = pd.read_csv(output, dtype=str)
            valid = (
                metadata.get("spec_sha256") == shard_spec_hash
                and metadata.get("csv_sha256") == sha256_file(output)
                and list(cached.columns) == ["id", "transcription"]
                and len(cached) == inventory["rows"]
                and sequence_sha256(cached.id.tolist()) == inventory["ids_sha256"]
            )
        except Exception:
            valid = False
    if valid:
        print(f"[cache valide {shard_index}/{len(mas_shards)}]", shard.name)
        prediction_parts.append(cached)
        continue

    started = time.time()
    table = pq.read_table(shard, columns=["id", "audio"]).to_pandas()
    ids = table.id.astype(str).tolist()
    assert sequence_sha256(ids) == inventory["ids_sha256"]
    predictions = []
    bad_audio = 0
    for row_index, row in table.iterrows():
        waveform, sample_rate = decode_audio(row["audio"])
        if waveform is None:
            prediction = ""
            bad_audio += 1
        else:
            if len(waveform) < int(0.2 * sample_rate):
                waveform = np.pad(
                    waveform, (0, int(0.2 * sample_rate) - len(waveform))
                )
            prediction = transcribe_waveform(waveform, sample_rate)
        predictions.append(prediction)
        done = row_index + 1
        if done % 50 == 0 or done == len(table):
            print(
                f"  shard {shard_index}/{len(mas_shards)} | "
                f"{done}/{len(table)} | {(time.time() - started) / 60:.1f} min"
            )
    frame = pd.DataFrame({"id": ids, "transcription": predictions})
    atomic_csv(frame, output)
    metadata = {
        "spec": shard_spec,
        "spec_sha256": shard_spec_hash,
        "csv_sha256": sha256_file(output),
        "bad_audio": bad_audio,
        "elapsed_minutes": (time.time() - started) / 60,
    }
    atomic_json(metadata, metadata_path)
    print(f"[calcule {shard_index}/{len(mas_shards)}]", shard.name)
    prediction_parts.append(frame)
    bad_audio_total += bad_audio


# -----------------------------------------------------------------------------
# 5. Remplacement cible, QA et nouveau CSV Drive
# -----------------------------------------------------------------------------

replacement = pd.concat(prediction_parts, ignore_index=True)
assert replacement.id.is_unique
assert set(replacement.id) == set(all_target_ids)
replacement = replacement.set_index("id")

candidate = old.copy()
candidate_by_id = candidate.set_index("id", drop=False)
before = candidate_by_id.loc[replacement.index, "transcription"].fillna("").astype(str)
after = replacement["transcription"].fillna("").astype(str)
changed = int((before != after).sum())
candidate_by_id.loc[replacement.index, "transcription"] = after
candidate = candidate_by_id.loc[old.id].reset_index(drop=True)
assert candidate.id.tolist() == old.id.tolist()
assert candidate.language.tolist() == old.language.tolist()
assert len(candidate) == EXPECTED_TOTAL_ROWS and candidate.id.is_unique

empty = candidate.transcription.fillna("").str.strip().eq("")
empty_target = empty & candidate.id.isin(replacement.index)
candidate.loc[empty_target, "transcription"] = "a"
# Les lignes non ciblees doivent rester bit-a-bit identiques a la soumission 0.36880.
not_target = ~candidate.id.isin(replacement.index)
assert candidate.loc[not_target].equals(old.loc[not_target])

OUTPUT = REPORT_DIR / f"submission_lora_mas_unscripted_{RUN_ID}.csv"
atomic_csv(candidate, OUTPUT)
output_hash = sha256_file(OUTPUT)

word_counts = candidate.assign(
    words=candidate.transcription.fillna("").str.split().str.len()
).groupby("language").words.agg(["count", "median", "max"])
report = {
    "schema": 1,
    "status": "PASS_READY_FOR_KAGGLE_MANUAL_SUBMISSION",
    "run_id": RUN_ID,
    "submission": str(OUTPUT),
    "submission_sha256": output_hash,
    "rows": len(candidate),
    "target_group": "mas|unscripted",
    "target_rows": len(replacement),
    "changed_target_predictions": changed,
    "unchanged_rows_from_score_036880": int(not_target.sum()),
    "bad_audio_replaced": int(empty_target.sum()),
    "base_submission": str(OLD_SUBMISSION),
    "base_submission_sha256": OLD_SUBMISSION_SHA256,
    "run_contract": run_contract,
    "cache_root": str(CACHE_ROOT),
    "automatic_kaggle_submission": False,
}
atomic_json(report, OUTPUT.with_suffix(".meta.json"))
atomic_json(report, REPORT_DIR / "lora_mas_unscripted_submission_latest.json")

print("\n=== QA SOUMISSION LORA MAS|UNSCRIPTED ===")
print(word_counts)
print("Lignes recalculees :", len(replacement))
print("Predictions changees:", changed)
print("Audios illisibles   :", int(empty_target.sum()))
print("Soumission          :", OUTPUT)
print("SHA256              :", output_hash)
print("✅ PASS_READY_FOR_KAGGLE_MANUAL_SUBMISSION")

In [ ]:
# 20.1 — Geler le score 0.36878 et auditer les donnees disponibles pour V5
#
# Cette cellule est volontairement CPU-only et idempotente.
# Elle :
#   1. fige le CSV exact de 19.7g et son score Kaggle 0.36878 ;
#   2. relit les metadonnees lm_text recoltees par 16.4 pour kik/kln/luo/som/mas ;
#   3. inventorie les durees <=38 s, 38-60 s, 60-90 s et >90 s ;
#   4. compare ces heures au melange V4 reellement entraine ;
#   5. scanne seulement les petits manifests Swahili si le token HF est disponible ;
#   6. sauvegarde tous les rapports sur Drive.
#
# Aucun audio n'est telecharge, copie, segmente ou entraine dans cette cellule.

from __future__ import annotations

import hashlib
import json
import math
import os
import re
import shutil
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


# -----------------------------------------------------------------------------
# 0. Configuration et utilitaires
# -----------------------------------------------------------------------------

DRIVE_ROOT = Path("/content/afrivoices_project")
FT_ROOT = DRIVE_ROOT / "finetune_runs/experiment_run"
REPORT_DIR = FT_ROOT / "reports"
DATASET_CACHE_ROOT = DRIVE_ROOT / "dataset_caches"
OUT_DIR = REPORT_DIR / "v5_preflight_20_1"
FROZEN_DIR = REPORT_DIR / "leaderboard_frozen"

LANGUAGES = ("sw", "kik", "kln", "luo", "som", "mas")
ANV_LANGUAGES = ("kik", "kln", "luo", "som", "mas")
SCORE_19_7G = 0.36878
SCORE_V4_BASELINE = 0.36880
EXPECTED_ROWS = 41_733

for directory in (OUT_DIR, FROZEN_DIR):
    directory.mkdir(parents=True, exist_ok=True)


def now_iso():
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path, block_size=16 << 20):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()


def canonical_sha256(value):
    raw = json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()


def atomic_json(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".tmp-{os.getpid()}")
    with open(temporary, "w", encoding="utf-8") as handle:
        json.dump(value, handle, ensure_ascii=False, indent=2, default=str)
    os.replace(temporary, path)
    print("json ->", path)


def atomic_csv(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".tmp-{os.getpid()}")
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)
    print("csv  ->", path)


def atomic_copy(source, destination):
    source, destination = Path(source), Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        assert sha256_file(destination) == sha256_file(source), (
            "Le fichier gele existe avec un contenu different", destination
        )
        return
    temporary = destination.with_name(destination.name + f".tmp-{os.getpid()}")
    shutil.copy2(source, temporary)
    assert sha256_file(temporary) == sha256_file(source)
    os.replace(temporary, destination)


def show(frame):
    try:
        display(frame)  # noqa: F821 - disponible dans Colab/Jupyter
    except NameError:
        print(frame.to_string(index=False))


def duration_bucket(seconds):
    try:
        seconds = float(seconds)
    except (TypeError, ValueError):
        return "invalid_or_missing"
    if not math.isfinite(seconds) or seconds < 0.5:
        return "invalid_or_missing"
    if seconds <= 38.0:
        return "00.5_38s"
    if seconds <= 60.0:
        return "38_60s"
    if seconds <= 90.0:
        return "60_90s"
    return "over_90s"


assert DRIVE_ROOT.exists(), "Monter Google Drive avant d'executer 20.1."


# -----------------------------------------------------------------------------
# 1. Gel reproductible du resultat leaderboard 19.7g
# -----------------------------------------------------------------------------

LATEST_19_7G = REPORT_DIR / "lora_mas_unscripted_submission_latest.json"
assert LATEST_19_7G.exists(), (
    "Rapport 19.7g absent. Executer/terminer 19.7g avant 20.1 : " + str(LATEST_19_7G)
)
submission_meta = json.load(open(LATEST_19_7G, encoding="utf-8"))
assert submission_meta.get("status") == "PASS_READY_FOR_KAGGLE_MANUAL_SUBMISSION"

source_submission = Path(submission_meta["submission"])
assert source_submission.exists(), source_submission
source_sha256 = sha256_file(source_submission)
assert source_sha256 == submission_meta["submission_sha256"], (
    "Le CSV 19.7g a change depuis sa creation.",
    source_sha256,
    submission_meta["submission_sha256"],
)

submission = pd.read_csv(source_submission, dtype=str)
assert list(submission.columns) == ["id", "language", "transcription"]
assert len(submission) == EXPECTED_ROWS and submission.id.is_unique
assert set(submission.language) == {"kik", "kln", "luo", "mas", "som", "swa"}

frozen_submission = FROZEN_DIR / (
    f"submission_score_036878_{source_sha256[:16]}.csv"
)
atomic_copy(source_submission, frozen_submission)
assert sha256_file(frozen_submission) == source_sha256

score_record = {
    "schema": 2,
    "recorded_at": now_iso(),
    "competition": "afri-voices-east-africa-asr-hackathon",
    "public_leaderboard_score": SCORE_19_7G,
    "previous_v4_score": SCORE_V4_BASELINE,
    "absolute_delta": round(SCORE_19_7G - SCORE_V4_BASELINE, 8),
    "lower_is_better": True,
    "interpretation": "gain numerique positif mais non materiel",
    "submission": str(frozen_submission),
    "submission_sha256": source_sha256,
    "rows": len(submission),
    "run_id": submission_meta.get("run_id"),
    "target_group": submission_meta.get("target_group"),
    "target_rows": submission_meta.get("target_rows"),
    "changed_target_predictions": submission_meta.get("changed_target_predictions"),
    "source_report": str(LATEST_19_7G),
    "source_report_sha256": sha256_file(LATEST_19_7G),
    "base_submission": submission_meta.get("base_submission"),
    "base_submission_sha256": submission_meta.get("base_submission_sha256"),
    "run_contract": submission_meta.get("run_contract"),
}
score_record["record_sha256"] = canonical_sha256(
    {key: value for key, value in score_record.items() if key not in {"recorded_at", "record_sha256"}}
)
atomic_json(score_record, REPORT_DIR / "leaderboard_result_v5_036878.json")
atomic_json(score_record, FROZEN_DIR / "score_036878.json")

print("\n=== SCORE 19.7g GELE ===")
print("Score             :", SCORE_19_7G)
print("Delta vs 0.36880  :", f"{SCORE_19_7G - SCORE_V4_BASELINE:+.5f}")
print("CSV gele          :", frozen_submission)
print("SHA256            :", source_sha256)


# -----------------------------------------------------------------------------
# 2. Statistiques exactes du dataset V4 materialise
# -----------------------------------------------------------------------------

V4_COMPLETE_CANDIDATES = (
    DATASET_CACHE_ROOT / "afrivoices_balanced_v4_dataset/_COMPLETE.json",
    Path("/content/afrivoices_balanced_v4/dataset/_COMPLETE.json"),
)
v4_complete_path = next(
    (path for path in V4_COMPLETE_CANDIDATES if path.exists()), None
)
assert v4_complete_path is not None, (
    "_COMPLETE.json du dataset V4 introuvable. La copie Drive de 18.4 est requise."
)
v4_complete = json.load(open(v4_complete_path, encoding="utf-8"))
v4_stats = pd.DataFrame(v4_complete.get("stats", []))
assert not v4_stats.empty, "Le marqueur V4 ne contient pas les statistiques de 18.4."
assert {"split", "language", "role", "clips", "hours"} <= set(v4_stats.columns)
v4_train = v4_stats[v4_stats.split.eq("train")].copy()

v4_summary_rows = []
for language in LANGUAGES:
    group = v4_train[v4_train.language.eq(language)]
    v4_summary_rows.append({
        "language": language,
        "v4_total_hours": float(group.hours.sum()),
        "v4_legacy_hours": float(
            group.loc[group.role.eq("legacy_replay"), "hours"].sum()
        ),
        "v4_new_hours": float(
            group.loc[group.role.eq("new_in_domain"), "hours"].sum()
        ),
        "v4_clips": int(group.clips.sum()),
    })
v4_summary = pd.DataFrame(v4_summary_rows)


# -----------------------------------------------------------------------------
# 3. Inventaire des metadonnees ANV train recoltees par 16.4
# -----------------------------------------------------------------------------

MOJIBAKE_MARKERS = ("Ã", "Â", "Å", "Ä", "â€", "ðŸ", "�")
bucket_rows = []
source_rows = []
audit_warnings = []
audit_blockers = []


def complete_stats(cache):
    path = cache / "_COMPLETE.json"
    if not path.exists():
        return {}
    try:
        return json.load(open(path, encoding="utf-8"))
    except Exception as error:
        audit_warnings.append({
            "code": "UNREADABLE_COMPLETE_JSON",
            "path": str(path),
            "detail": repr(error),
        })
        return {}


def cache_candidates(language):
    output = []
    for cache in DATASET_CACHE_ROOT.glob(f"topup_{language}_spont_*"):
        if not cache.is_dir() or cache.name.endswith((".partial", ".backup", ".staging")):
            continue
        parts = sorted((cache / "lm_text").glob("*.parquet"))
        if parts:
            output.append((len(parts), sum(path.stat().st_size for path in parts), cache))
    return sorted(output, reverse=True)


def audit_anv_cache(language):
    candidates = cache_candidates(language)
    if not candidates:
        audit_blockers.append({
            "code": "ANV_LM_TEXT_CACHE_MISSING",
            "language": language,
            "detail": "Relancer/restaurer le cache 16.4 versionne.",
        })
        return None

    _, _, cache = candidates[0]
    parts = sorted((cache / "lm_text").glob("*.parquet"))
    frames = []
    part_inventory = []
    for part in parts:
        parquet = pq.ParquetFile(part)
        names = set(parquet.schema_arrow.names)
        columns = [
            name for name in (
                "sample_id", "speaker_id", "text", "duration_s",
                "audio_class", "repaired"
            ) if name in names
        ]
        required = {"text", "duration_s"}
        if not required <= set(columns):
            audit_blockers.append({
                "code": "ANV_LM_TEXT_COLUMNS_MISSING",
                "language": language,
                "path": str(part),
                "missing": sorted(required - set(columns)),
            })
            continue
        frame = pq.read_table(part, columns=columns).to_pandas()
        frame["_part"] = str(part)
        frames.append(frame)
        part_inventory.append({
            "path": str(part),
            "rows": int(parquet.metadata.num_rows),
            "bytes": int(part.stat().st_size),
            "sha256": sha256_file(part),
        })
    if not frames:
        return None

    frame = pd.concat(frames, ignore_index=True)
    frame["text"] = frame.text.fillna("").astype(str)
    frame["duration_s"] = pd.to_numeric(frame.duration_s, errors="coerce")
    frame["duration_bucket"] = frame.duration_s.map(duration_bucket)
    frame["has_transcript"] = frame.text.str.strip().ne("")
    frame["words"] = frame.text.str.split().str.len()

    sample_ids = (
        frame.sample_id.fillna("").astype(str)
        if "sample_id" in frame.columns else pd.Series([""] * len(frame))
    )
    duplicate_ids = int(sample_ids[sample_ids.ne("")].duplicated().sum())
    if duplicate_ids:
        audit_warnings.append({
            "code": "ANV_SOURCE_ID_DUPLICATES",
            "language": language,
            "count": duplicate_ids,
        })

    valid = frame[frame.has_transcript & frame.duration_bucket.ne("invalid_or_missing")]
    for bucket in ("00.5_38s", "38_60s", "60_90s", "over_90s", "invalid_or_missing"):
        group = frame[frame.duration_bucket.eq(bucket)]
        bucket_rows.append({
            "language": language,
            "source": "ANV_train_lm_text_cache",
            "duration_bucket": bucket,
            "clips": int(len(group)),
            "hours": float(group.duration_s.fillna(0).clip(lower=0).sum() / 3600),
            "transcribed_clips": int(group.has_transcript.sum()),
            "speakers": int(group.speaker_id.nunique()) if "speaker_id" in group else None,
            "words": int(group.words.sum()),
        })

    metadata = complete_stats(cache)
    state = metadata.get("stats", {}) if isinstance(metadata, dict) else {}
    declared_text_rows = state.get("text_rows") if isinstance(state, dict) else None
    if declared_text_rows is not None and int(declared_text_rows) != len(frame):
        audit_warnings.append({
            "code": "ANV_DECLARED_ROWS_DIFFER",
            "language": language,
            "declared": int(declared_text_rows),
            "measured": int(len(frame)),
        })

    marker_counts = {
        "cs": int(frame.text.str.contains(r"\[/?cs\]", case=False, regex=True).sum()),
        "pause": int(frame.text.str.contains(r"\[pause\]", case=False, regex=True).sum()),
        "unclear": int(frame.text.str.contains(r"\[\?\]", case=False, regex=True).sum()),
        "mojibake": int(frame.text.map(
            lambda value: any(marker in value for marker in MOJIBAKE_MARKERS)
        ).sum()),
    }
    if marker_counts["mojibake"]:
        audit_warnings.append({
            "code": "MOJIBAKE_REMAINS_IN_LM_TEXT",
            "language": language,
            "count": marker_counts["mojibake"],
        })

    source_record = {
        "language": language,
        "cache": str(cache),
        "cache_build_id": metadata.get("build_id") if isinstance(metadata, dict) else None,
        "parts": len(parts),
        "rows": int(len(frame)),
        "valid_transcribed_rows": int(len(valid)),
        "hours": float(valid.duration_s.sum() / 3600),
        "hours_le_38s": float(
            valid.loc[valid.duration_bucket.eq("00.5_38s"), "duration_s"].sum() / 3600
        ),
        "hours_38_60s": float(
            valid.loc[valid.duration_bucket.eq("38_60s"), "duration_s"].sum() / 3600
        ),
        "hours_60_90s": float(
            valid.loc[valid.duration_bucket.eq("60_90s"), "duration_s"].sum() / 3600
        ),
        "hours_over_90s": float(
            valid.loc[valid.duration_bucket.eq("over_90s"), "duration_s"].sum() / 3600
        ),
        "hours_needing_segmentation_gt38s": float(
            valid.loc[valid.duration_s.gt(38.0), "duration_s"].sum() / 3600
        ),
        "speakers": int(valid.speaker_id.nunique()) if "speaker_id" in valid else None,
        "duplicate_source_ids": duplicate_ids,
        "marker_counts_after_existing_normalization": marker_counts,
        "part_inventory": part_inventory,
        "complete_json": str(cache / "_COMPLETE.json"),
        "complete_exists": (cache / "_COMPLETE.json").exists(),
    }
    source_rows.append(source_record)
    return source_record


for language in ANV_LANGUAGES:
    audit_anv_cache(language)


# -----------------------------------------------------------------------------
# 4. Audit leger du Swahili : manifests uniquement, jamais les archives audio
# -----------------------------------------------------------------------------

SW_REPO = "DigitalUmuganda/Afrivoice_Swahili"
SW_DOMAINS = ("agriculture", "health", "education", "financial", "government")


def find_hf_token():
    for path in (
        Path("/content/afrivoices_project"),
        DRIVE_ROOT / "hf_token.json",
    ):
        if path.exists():
            value = json.load(open(path, encoding="utf-8"))
            return value.get("key") or value.get("token") or value.get("hf_token")
    return None


def audit_swahili_manifests():
    try:
        from huggingface_hub import HfApi, hf_hub_download
    except Exception as error:
        audit_warnings.append({
            "code": "SWAHILI_MANIFEST_AUDIT_SKIPPED",
            "detail": f"huggingface_hub indisponible: {error!r}",
        })
        return None

    token = find_hf_token()
    if not token:
        audit_warnings.append({
            "code": "SWAHILI_MANIFEST_AUDIT_SKIPPED",
            "detail": "hf_token.json introuvable ; aucune archive audio n'a ete telechargee.",
        })
        return None

    try:
        files = HfApi(token=token).list_repo_files(
            SW_REPO, repo_type="dataset"
        )
    except Exception as error:
        audit_warnings.append({
            "code": "SWAHILI_MANIFEST_LIST_FAILED",
            "detail": repr(error),
        })
        return None

    pattern = re.compile(
        r"^(agriculture|health|education|financial|government)_swahili_train/"
        r"manifest_(\d+)\.jsonl$"
    )
    manifests = sorted(path for path in files if pattern.match(path))
    if not manifests:
        audit_warnings.append({
            "code": "SWAHILI_MANIFESTS_MISSING",
            "repo": SW_REPO,
        })
        return None

    rows = []
    manifest_inventory = []
    for index, remote_path in enumerate(manifests, 1):
        try:
            local = hf_hub_download(
                SW_REPO, remote_path, repo_type="dataset", token=token
            )
        except Exception as error:
            audit_warnings.append({
                "code": "SWAHILI_MANIFEST_DOWNLOAD_FAILED",
                "path": remote_path,
                "detail": repr(error),
            })
            continue
        count = 0
        with open(local, encoding="utf-8") as handle:
            for line_number, line in enumerate(handle, 1):
                try:
                    row = json.loads(line)
                except Exception:
                    audit_warnings.append({
                        "code": "SWAHILI_BAD_JSONL_ROW",
                        "path": remote_path,
                        "line": line_number,
                    })
                    continue
                text = str(
                    row.get("normalized_transcription")
                    or row.get("transcription")
                    or ""
                ).strip()
                try:
                    duration = float(row.get("duration") or 0)
                except (TypeError, ValueError):
                    duration = 0.0
                rows.append({
                    "sample_id": str(row.get("key") or row.get("id") or ""),
                    "speaker_id": str(
                        row.get("voice_creator_id") or row.get("uid") or ""
                    ),
                    "text": text,
                    "duration_s": duration,
                    "domain": pattern.match(remote_path).group(1),
                    "manifest": remote_path,
                })
                count += 1
        manifest_inventory.append({
            "path": remote_path,
            "rows": count,
            "sha256": sha256_file(local),
        })
        if index % 20 == 0 or index == len(manifests):
            print(f"Swahili manifests : {index}/{len(manifests)}")

    if not rows:
        return None
    frame = pd.DataFrame(rows)
    frame["duration_bucket"] = frame.duration_s.map(duration_bucket)
    frame["has_transcript"] = frame.text.str.strip().ne("")
    frame["words"] = frame.text.str.split().str.len()
    valid = frame[frame.has_transcript & frame.duration_bucket.ne("invalid_or_missing")]
    duplicate_ids = int(
        frame.loc[frame.sample_id.ne(""), "sample_id"].duplicated().sum()
    )
    if duplicate_ids:
        audit_warnings.append({
            "code": "SWAHILI_SOURCE_ID_DUPLICATES",
            "count": duplicate_ids,
        })

    for bucket in ("00.5_38s", "38_60s", "60_90s", "over_90s", "invalid_or_missing"):
        group = frame[frame.duration_bucket.eq(bucket)]
        bucket_rows.append({
            "language": "sw",
            "source": "Afrivoice_Swahili_train_manifests",
            "duration_bucket": bucket,
            "clips": int(len(group)),
            "hours": float(group.duration_s.fillna(0).clip(lower=0).sum() / 3600),
            "transcribed_clips": int(group.has_transcript.sum()),
            "speakers": int(group.speaker_id.replace("", np.nan).nunique()),
            "words": int(group.words.sum()),
        })

    source_record = {
        "language": "sw",
        "cache": None,
        "repository": SW_REPO,
        "source": "train manifests only",
        "parts": len(manifest_inventory),
        "rows": int(len(frame)),
        "valid_transcribed_rows": int(len(valid)),
        "hours": float(valid.duration_s.sum() / 3600),
        "hours_le_38s": float(
            valid.loc[valid.duration_bucket.eq("00.5_38s"), "duration_s"].sum() / 3600
        ),
        "hours_38_60s": float(
            valid.loc[valid.duration_bucket.eq("38_60s"), "duration_s"].sum() / 3600
        ),
        "hours_60_90s": float(
            valid.loc[valid.duration_bucket.eq("60_90s"), "duration_s"].sum() / 3600
        ),
        "hours_over_90s": float(
            valid.loc[valid.duration_bucket.eq("over_90s"), "duration_s"].sum() / 3600
        ),
        "hours_needing_segmentation_gt38s": float(
            valid.loc[valid.duration_s.gt(38.0), "duration_s"].sum() / 3600
        ),
        "speakers": int(valid.speaker_id.replace("", np.nan).nunique()),
        "duplicate_source_ids": duplicate_ids,
        "domains": valid.groupby("domain").duration_s.sum().div(3600).to_dict(),
        "manifest_inventory": manifest_inventory,
        "audio_downloaded": False,
    }
    source_rows.append(source_record)
    return source_record


audit_swahili_manifests()


# -----------------------------------------------------------------------------
# 5. Reference publique ANV et comparaison V4 -> potentiel V5
# -----------------------------------------------------------------------------

# Ces nombres sont une reference documentaire, pas des heures mesurees dans les caches.
# Source : carte MCAA1-MSU/anv_data_ke, revision a figer lors de 20.2.
ANV_PUBLIC_REFERENCE = {
    "luo": {"scripted_hours": 195.0, "unscripted_hours": 528.0,
            "total_hours": 723.0, "unscripted_transcribed_hours": 317.0},
    "kik": {"scripted_hours": 183.0, "unscripted_hours": 571.0,
            "total_hours": 754.0, "unscripted_transcribed_hours": 381.0},
    "som": {"scripted_hours": 118.0, "unscripted_hours": 384.0,
            "total_hours": 502.0, "unscripted_transcribed_hours": 384.0},
    "kln": {"scripted_hours": 122.0, "unscripted_hours": 399.0,
            "total_hours": 521.0, "unscripted_transcribed_hours": 383.0},
    "mas": {"scripted_hours": 51.0, "unscripted_hours": 454.0,
            "total_hours": 505.0, "unscripted_transcribed_hours": 454.0},
}

source_summary = pd.DataFrame([
    {key: value for key, value in row.items()
     if key not in {"part_inventory", "manifest_inventory", "marker_counts_after_existing_normalization"}}
    for row in source_rows
])

gap = v4_summary.copy()
if not source_summary.empty:
    gap = gap.merge(
        source_summary[[
            "language", "hours", "hours_le_38s", "hours_38_60s",
            "hours_60_90s", "hours_over_90s",
            "hours_needing_segmentation_gt38s", "rows", "speakers"
        ]].rename(columns={
            "hours": "metadata_train_hours",
            "rows": "metadata_train_clips",
            "speakers": "metadata_train_speakers",
        }),
        on="language", how="left",
    )
else:
    for column in (
        "metadata_train_hours", "hours_le_38s", "hours_38_60s",
        "hours_60_90s", "hours_over_90s",
        "hours_needing_segmentation_gt38s", "metadata_train_clips",
        "metadata_train_speakers"
    ):
        gap[column] = np.nan

gap["additional_metadata_hours_vs_v4_new"] = (
    gap.metadata_train_hours - gap.v4_new_hours
).clip(lower=0)
gap["metadata_multiple_of_v4_new"] = (
    gap.metadata_train_hours / gap.v4_new_hours.replace(0, np.nan)
)

bucket_table = pd.DataFrame(bucket_rows)
if not bucket_table.empty:
    bucket_table = bucket_table.sort_values(["language", "duration_bucket"])

atomic_csv(v4_summary, OUT_DIR / "v4_training_hours.csv")
atomic_csv(source_summary, OUT_DIR / "v5_source_metadata_summary.csv")
atomic_csv(bucket_table, OUT_DIR / "v5_duration_inventory.csv")
atomic_csv(gap, OUT_DIR / "v4_to_v5_gap.csv")


# -----------------------------------------------------------------------------
# 6. Gate et rapport signe
# -----------------------------------------------------------------------------

measured_languages = set(source_summary.language) if not source_summary.empty else set()
for language in ANV_LANGUAGES:
    if language not in measured_languages:
        audit_blockers.append({
            "code": "LANGUAGE_NOT_MEASURED",
            "language": language,
        })

if "sw" not in measured_languages:
    audit_warnings.append({
        "code": "SWAHILI_FULL_METADATA_NOT_MEASURED",
        "detail": "Non bloquant pour le pilote ANV 20.2 ; Swahili est deja la langue la plus forte.",
    })

for record in source_rows:
    if record["language"] in ANV_LANGUAGES:
        if record["hours_needing_segmentation_gt38s"] <= 0:
            audit_warnings.append({
                "code": "NO_LONG_AUDIO_MEASURED",
                "language": record["language"],
            })
        if record["hours_over_90s"] > 0:
            audit_warnings.append({
                "code": "AUDIO_OVER_ANV_90S_CAP",
                "language": record["language"],
                "hours": record["hours_over_90s"],
            })

status = (
    "PASS_READY_FOR_20_2_CTC_SEGMENTATION_PILOT"
    if not audit_blockers
    else "BLOCKED_RESTORE_OR_REBUILD_METADATA_CACHES"
)

contract = {
    "schema": 1,
    "score_submission_sha256": source_sha256,
    "v4_complete_path": str(v4_complete_path),
    "v4_complete_sha256": sha256_file(v4_complete_path),
    "selected_metadata_sources": [
        {
            "language": row.get("language"),
            "cache": row.get("cache"),
            "repository": row.get("repository"),
            "parts": row.get("parts"),
            "rows": row.get("rows"),
        }
        for row in source_rows
    ],
    "duration_buckets_seconds": {
        "short": [0.5, 38.0],
        "official_train_direct": [38.0, 60.0],
        "alignment_priority": [60.0, 90.0],
    },
    "anv_split_policy": {
        "train": "allowed for training",
        "dev": "validation only",
        "dev_test": "strictly excluded",
        "test": "strictly excluded; probable Kaggle hidden test",
    },
}
contract_sha256 = canonical_sha256(contract)

report = {
    "schema": 1,
    "created_at": now_iso(),
    "status": status,
    "contract": contract,
    "contract_sha256": contract_sha256,
    "frozen_score": score_record,
    "v4_complete": {
        "path": str(v4_complete_path),
        "build_sha256": v4_complete.get("build_sha256"),
        "audit_sha256": v4_complete.get("audit_sha256"),
        "plan_sha256": v4_complete.get("plan_sha256"),
    },
    "source_audits": source_rows,
    "anv_public_reference_not_measured": ANV_PUBLIC_REFERENCE,
    "blockers": audit_blockers,
    "warnings": audit_warnings,
    "outputs": {
        "v4_training_hours": str(OUT_DIR / "v4_training_hours.csv"),
        "source_metadata": str(OUT_DIR / "v5_source_metadata_summary.csv"),
        "duration_inventory": str(OUT_DIR / "v5_duration_inventory.csv"),
        "gap": str(OUT_DIR / "v4_to_v5_gap.csv"),
    },
    "next_action": (
        "20.2 — calibrer ctc-segmentation sur des pseudo-longs connus ; "
        "ne telecharger aucun audio ANV supplementaire avant PASS"
    ),
}
report["report_sha256"] = canonical_sha256(
    {key: value for key, value in report.items() if key not in {"created_at", "report_sha256"}}
)
report_path = OUT_DIR / f"v5_data_audit_{contract_sha256[:16]}.json"
atomic_json(report, report_path)
atomic_json(report, OUT_DIR / "LATEST.json")

print("\n=== DATASET V4 REELLEMENT ENTRAINE ===")
show(v4_summary.round(4))
print("\n=== METADONNEES TRAIN DISPONIBLES POUR V5 ===")
show(source_summary.drop(columns=["cache"], errors="ignore").round(4))
print("\n=== ECART V4 -> POTENTIEL V5 ===")
show(gap.round(4))
print("\nSTATUT 20.1 :", status)
print("Bloquants      :", audit_blockers or "aucun")
print("Warnings       :", audit_warnings or "aucun")
print("Rapport        :", report_path)
print("Etape suivante :", report["next_action"])

In [ ]:
# 20.2 — Calibration de ctc-segmentation sur des pseudo-longs a bornes connues
#
# Objectif : valider AVANT tout telechargement massif d'audio ANV long que le
# chemin suivant est suffisamment fiable :
#   step_1250 BF16 -> inference CTC fenetree -> couture -> ctc-segmentation.
#
# Ce pilote :
#   - utilise uniquement le dev V4 fige (jamais train/local_test/test Kaggle) ;
#   - separe calibration/audit par composantes locuteur + texte normalise ;
#   - fabrique 60 pseudo-longs de calibration et 30 d'audit par langue ;
#   - injecte des silences dont les vraies frontieres sont connues ;
#   - calibre quatre variantes ctc-segmentation sans refaire l'acoustique ;
#   - choisit un seuil de confiance par langue sur calibration seulement ;
#   - ouvre l'audit une seule fois avec les choix geles ;
#   - sauvegarde des blocs reprenables sur Drive, mais jamais les gros logits.
#
# Aucun audio ANV >38 s n'est telecharge par cette cellule.

from __future__ import annotations

import gc
import hashlib
import importlib.metadata
import io
import json
import math
import os
import re
import shutil
import subprocess
import sys
import time
import unicodedata
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import soundfile as sf
import torch


# -----------------------------------------------------------------------------
# 0. Configuration
# -----------------------------------------------------------------------------

DRIVE_ROOT = Path("/content/afrivoices_project")
FT_ROOT = DRIVE_ROOT / "finetune_runs/experiment_run"
REPORT_DIR = FT_ROOT / "reports"
V4_REPORT_DIR = REPORT_DIR / "balanced_v4"
PREFLIGHT_DIR = REPORT_DIR / "v5_preflight_20_1"
OUT_ROOT = REPORT_DIR / "v5_ctc_segmentation_20_2"

DEV_SELECTION_PATH = V4_REPORT_DIR / "selected_dev_v4.parquet"
TRAIN_SELECTION_PATH = V4_REPORT_DIR / "selected_records_v4.parquet"
EDGE_REPORT = REPORT_DIR / "lora_edge_bf16_audit_v2.json"

LOCAL_ROOT = Path("/content/v5_ctcseg_20_2")
LOCAL_AUDIO_ROOT = LOCAL_ROOT / "source_audio"
LOCAL_BASE_WEIGHT = LOCAL_ROOT / "base_step1250_bf16.pt"

LANGUAGES = ("kik", "kln", "luo", "som", "mas")
LANG_TO_OMNI = {
    "kik": "kik_Latn",
    "kln": "kln_Latn",
    "luo": "luo_Latn",
    "som": "som_Latn",
    "mas": "saq_Latn",
}

SEED = 20_200_001
PSEUDO_LONGS = {"tune": 60, "audit": 30}
NEGATIVES_PER_AUDIO = 6
SOURCE_POOL_RATIOS = {"tune": 0.50, "audit": 0.25, "reserve": 0.25}
SOURCE_DURATION_MIN_S = 0.8
SOURCE_DURATION_MAX_S = 28.0
TARGET_DURATION_BINS = ((45.0, 60.0), (60.0, 75.0), (75.0, 89.0))
TARGET_DURATION_WEIGHTS = (0.30, 0.40, 0.30)
MIN_SEGMENTS = 2
MAX_SEGMENTS = 12
PART_SIZE = 5

# Meme chemin que l'inference longue validee en 19.7g. 20.2 le mesure ; il ne
# suppose pas qu'il est correct.
STITCHING = {
    "window_s": 38.0,
    "overlap_s": 6.0,
    "trim_s": 3.0,
    "hop_s": 32.0,
}

# Ces quatre configurations reutilisent les MEMES posteriors acoustiques.
CTC_CONFIGS = (
    {"id": "blank0_L20", "blank_transition_cost_zero": False,
     "score_min_mean_over_L": 20},
    {"id": "blank0_L30", "blank_transition_cost_zero": False,
     "score_min_mean_over_L": 30},
    {"id": "blankfree_L20", "blank_transition_cost_zero": True,
     "score_min_mean_over_L": 20},
    {"id": "blankfree_L30", "blank_transition_cost_zero": True,
     "score_min_mean_over_L": 30},
)

# Garde-fous sur l'audit final, appliques langue par langue.
PASS_LIMITS = {
    "positive_acceptance_min": 0.95,
    "negative_false_accept_max": 0.01,
    "negative_false_accept_count_max": 0,
    "negative_false_accept_audio_max": 0.0,
    "structural_anomaly_max": 0.005,
    "cut_error_p95_s_max": 0.60,
    "long_gap_cut_inside_min": 0.95,
    "near_seam_cut_count_min": 10,
    "near_seam_cut_error_p95_s_max": 0.75,
    "near_seam_degradation_p95_s_max": 0.25,
    "speech_undercut_p95_s_max": 0.50,
    "endpoint_abs_error_p95_s_max": 1.00,
}

for directory in (OUT_ROOT, LOCAL_ROOT, LOCAL_AUDIO_ROOT):
    directory.mkdir(parents=True, exist_ok=True)


# -----------------------------------------------------------------------------
# 1. Utilitaires atomiques, hash et normalisation
# -----------------------------------------------------------------------------

def now_iso():
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path, block_size=32 << 20):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()


def canonical_sha256(value):
    raw = json.dumps(
        value, ensure_ascii=False, sort_keys=True,
        separators=(",", ":"), default=str,
    ).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()


def frame_sha256(frame, sort_key="sample_key"):
    ordered = frame.copy()
    columns = sorted(ordered.columns)
    ordered = ordered[columns]
    if sort_key in ordered.columns:
        ordered = ordered.sort_values(sort_key, kind="mergesort")
    ordered = ordered.reset_index(drop=True)
    values = pd.util.hash_pandas_object(
        ordered.fillna("").astype(str), index=False
    ).values.tobytes()
    return hashlib.sha256(
        json.dumps(columns, ensure_ascii=False).encode("utf-8") + values
    ).hexdigest()


def atomic_json(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".tmp-{os.getpid()}")
    with open(temporary, "w", encoding="utf-8") as handle:
        json.dump(value, handle, ensure_ascii=False, indent=2, default=str)
    os.replace(temporary, path)
    print("json ->", path)


def atomic_parquet(records, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".tmp-{os.getpid()}")
    table = pa.Table.from_pylist(records)
    pq.write_table(table, temporary, compression="zstd", compression_level=3)
    os.replace(temporary, path)


def atomic_frame_parquet(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".tmp-{os.getpid()}")
    frame.to_parquet(temporary, index=False)
    os.replace(temporary, path)


def show(frame):
    try:
        display(frame)  # noqa: F821 - Jupyter/Colab
    except NameError:
        print(frame.to_string(index=False))


_MOJIBAKE_MARKERS = set("ÃÂÅâðÐÑ¤¦¨©¬®¯°±²³´µ¶·¸¹º»¼½¾¿")
_PUNCT_RE = re.compile(r"[^\w\s'’\-]", re.UNICODE)
_WS_RE = re.compile(r"\s+")


def text_badness(value):
    value = str(value)
    return (
        50 * value.count("�")
        + 6 * sum(char in _MOJIBAKE_MARKERS for char in value)
        + 10 * sum(ord(char) < 32 and char not in "\n\t\r" for char in value)
    )


def repair_mojibake(value):
    raw = str(value or "")
    candidates = [raw]
    if any(char in _MOJIBAKE_MARKERS for char in raw) or "�" in raw:
        for encoding in ("latin-1", "cp1252"):
            try:
                candidates.append(raw.encode(encoding).decode("utf-8"))
            except (UnicodeEncodeError, UnicodeDecodeError):
                pass
    best = min(candidates, key=text_badness)
    return best if text_badness(best) < text_badness(raw) else raw


def normalize_text(value):
    value = unicodedata.normalize("NFC", repair_mojibake(value)).lower()
    value = _PUNCT_RE.sub(" ", value).replace("_", " ")
    return _WS_RE.sub(" ", value).strip()


def stable_unit_interval(value):
    digest = hashlib.sha256(str(value).encode("utf-8")).digest()
    return int.from_bytes(digest[:8], "big") / float(2**64)


def percentile(values, q):
    values = np.asarray(list(values), dtype=np.float64)
    values = values[np.isfinite(values)]
    return float(np.quantile(values, q)) if len(values) else None


def wilson_upper(successes, total, z=1.959963984540054):
    if total <= 0:
        return None
    p = successes / total
    denominator = 1.0 + z * z / total
    centre = p + z * z / (2.0 * total)
    radius = z * math.sqrt((p * (1.0 - p) + z * z / (4.0 * total)) / total)
    return float((centre + radius) / denominator)


assert DRIVE_ROOT.exists(), "Monter Google Drive avant 20.2."
assert torch.cuda.is_available(), (
    "20.2 requiert un GPU. Un L4 suffit ; un A100 est plus rapide."
)


# -----------------------------------------------------------------------------
# 2. Preflight signe : 20.1, dev V4 et base acoustique step_1250
# -----------------------------------------------------------------------------

preflight_latest = PREFLIGHT_DIR / "LATEST.json"
for required in (
    preflight_latest, DEV_SELECTION_PATH, TRAIN_SELECTION_PATH, EDGE_REPORT,
):
    assert Path(required).exists(), required

preflight = json.load(open(preflight_latest, encoding="utf-8"))
assert preflight["status"] == "PASS_READY_FOR_20_2_CTC_SEGMENTATION_PILOT", (
    "20.1 n'est pas PASS.", preflight.get("status")
)
edge = json.load(open(EDGE_REPORT, encoding="utf-8"))
assert edge["status"] == "PASS_READY_FOR_TEST_INFERENCE", edge["status"]
EDGE_BASE = Path(edge["base_bf16"])
assert EDGE_BASE.exists(), EDGE_BASE
EXPECTED_BASE_SHA256 = edge["base_bf16_sha256"]

try:
    installed_ctc_version = importlib.metadata.version("ctc-segmentation")
except importlib.metadata.PackageNotFoundError:
    installed_ctc_version = None
if installed_ctc_version != "1.7.4":
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "--upgrade", "ctc-segmentation==1.7.4"],
        check=True,
    )
    installed_ctc_version = importlib.metadata.version("ctc-segmentation")
assert installed_ctc_version == "1.7.4", installed_ctc_version

from ctc_segmentation import (  # noqa: E402
    CtcSegmentationParameters,
    ctc_segmentation,
    determine_utterance_segments,
    prepare_token_list,
)
from fairseq2.data.tokenizers.hub import load_tokenizer  # noqa: E402
from fairseq2.models.hub import load_model  # noqa: E402
from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline  # noqa: E402


def copy_verified(source, destination, expected_sha256, attempts=4):
    source, destination = Path(source), Path(destination)
    if destination.exists() and sha256_file(destination) == expected_sha256:
        return destination
    destination.parent.mkdir(parents=True, exist_ok=True)
    for attempt in range(1, attempts + 1):
        temporary = destination.with_name(destination.name + f".tmp-{os.getpid()}")
        try:
            if temporary.exists():
                temporary.unlink()
            shutil.copyfile(source, temporary)
            if sha256_file(temporary) != expected_sha256:
                raise IOError("SHA256 de la copie incorrect")
            os.replace(temporary, destination)
            return destination
        except OSError as error:
            print(f"Copie tentative {attempt}/{attempts} : {error}")
            if temporary.exists():
                temporary.unlink()
            if attempt == attempts:
                raise
            time.sleep(2 * attempt)
    raise RuntimeError("copie impossible")


print("Copie locale/verif du poids BF16 step_1250...")
copy_verified(EDGE_BASE, LOCAL_BASE_WEIGHT, EXPECTED_BASE_SHA256)
assert sha256_file(LOCAL_BASE_WEIGHT) == EXPECTED_BASE_SHA256


# -----------------------------------------------------------------------------
# 3. Dev V4 : disjonction train, composantes locuteur+texte, pools anti-fuite
# -----------------------------------------------------------------------------

dev_all = pd.read_parquet(DEV_SELECTION_PATH)
DEV_SELECTION_SHA256 = frame_sha256(dev_all)
# Le cache audio local est lie a l'empreinte exacte du dev. Une future selection
# ne reutilisera donc jamais silencieusement des octets d'un ancien contrat.
LOCAL_AUDIO_ROOT = LOCAL_ROOT / "source_audio" / DEV_SELECTION_SHA256[:16]
LOCAL_AUDIO_ROOT.mkdir(parents=True, exist_ok=True)
required_columns = {
    "sample_key", "language", "split", "source_kind", "audio_ref",
    "parquet_path", "row_index", "text", "seconds",
}
assert required_columns <= set(dev_all.columns), required_columns - set(dev_all.columns)
assert dev_all["split"].eq("dev").all() and dev_all.sample_key.is_unique

train_keys = set(pd.read_parquet(
    TRAIN_SELECTION_PATH, columns=["sample_key"]
).sample_key.astype(str))
assert set(dev_all.sample_key.astype(str)).isdisjoint(train_keys)

dev = dev_all[dev_all.language.isin(LANGUAGES)].copy()
dev["sample_key"] = dev.sample_key.astype(str)
dev["seconds"] = pd.to_numeric(dev.seconds, errors="coerce")
dev["reference"] = dev.text.astype(str).map(normalize_text)
dev["text_key"] = dev.reference.map(
    lambda value: hashlib.sha256(value.encode("utf-8")).hexdigest()
)
if "speaker_id" not in dev.columns:
    dev["speaker_id"] = ""
dev["speaker_key"] = dev.speaker_id.fillna("").astype(str).str.strip()
dev.loc[dev.speaker_key.eq(""), "speaker_key"] = (
    "missing:" + dev.loc[dev.speaker_key.eq(""), "sample_key"]
)
dev = dev[
    dev.seconds.between(SOURCE_DURATION_MIN_S, SOURCE_DURATION_MAX_S)
    & dev.reference.str.split().str.len().ge(2)
].copy()
assert set(dev.language) == set(LANGUAGES)


class UnionFind:
    def __init__(self, values):
        self.parent = {value: value for value in values}

    def find(self, value):
        parent = self.parent[value]
        if parent != value:
            self.parent[value] = self.find(parent)
        return self.parent[value]

    def union(self, left, right):
        left, right = self.find(left), self.find(right)
        if left != right:
            if str(left) > str(right):
                left, right = right, left
            self.parent[right] = left


pool_frames = []
pool_summary = []
for language, group in dev.groupby("language", sort=True):
    group = group.copy().reset_index(drop=True)
    uf = UnionFind(group.sample_key.tolist())
    for column in ("speaker_key", "text_key"):
        for _, linked in group.groupby(column, sort=False):
            keys = linked.sample_key.tolist()
            for key in keys[1:]:
                uf.union(keys[0], key)
    group["component"] = group.sample_key.map(uf.find)

    component_rows = []
    for component, linked in group.groupby("component", sort=False):
        signature = canonical_sha256(sorted(linked.sample_key.tolist()))
        component_rows.append({
            "component": component,
            "signature": signature,
            "seconds": float(linked.seconds.sum()),
        })
    components = pd.DataFrame(component_rows)
    # Allocation gloutonne par composante entiere. Elle garde locuteurs et textes
    # dupliques d'un seul cote, tout en evitant qu'un simple hash laisse un pool
    # presque vide lorsqu'un locuteur est tres long.
    components["tie"] = components.signature.map(
        lambda value: stable_unit_interval(f"{SEED}:{language}:{value}")
    )
    components = components.sort_values(
        ["seconds", "tie"], ascending=[False, True], kind="mergesort"
    ).reset_index(drop=True)
    total_seconds = float(components.seconds.sum())
    targets = {
        pool: total_seconds * ratio for pool, ratio in SOURCE_POOL_RATIOS.items()
    }
    assigned = {pool: 0.0 for pool in SOURCE_POOL_RATIOS}
    component_pools = []
    for component in components.to_dict("records"):
        pool = max(
            SOURCE_POOL_RATIOS,
            key=lambda name: (
                targets[name] - assigned[name],
                -assigned[name] / max(targets[name], 1e-9),
                name,
            ),
        )
        component_pools.append(pool)
        assigned[pool] += float(component["seconds"])
    components["pool"] = component_pools
    group = group.merge(components[["component", "pool"]], on="component")
    pool_frames.append(group)
    for pool, linked in group.groupby("pool"):
        pool_summary.append({
            "language": language,
            "pool": pool,
            "clips": int(len(linked)),
            "hours": float(linked.seconds.sum() / 3600),
            "speakers": int(linked.speaker_key.nunique()),
            "components": int(linked.component.nunique()),
        })

sources = pd.concat(pool_frames, ignore_index=True)
pool_table = pd.DataFrame(pool_summary).sort_values(["language", "pool"])
for language in LANGUAGES:
    for pool in ("tune", "audit"):
        count = int(((sources.language == language) & (sources.pool == pool)).sum())
        assert count >= 20, (language, pool, count, pool_table)

for language in LANGUAGES:
    tune = sources[(sources.language == language) & (sources.pool == "tune")]
    audit = sources[(sources.language == language) & (sources.pool == "audit")]
    assert set(tune.speaker_key).isdisjoint(set(audit.speaker_key))
    assert set(tune.text_key).isdisjoint(set(audit.text_key))
    assert set(tune.component).isdisjoint(set(audit.component))

print("\n=== POOLS SOURCES ANTI-FUITE ===")
show(pool_table)


# -----------------------------------------------------------------------------
# 4. Recettes deterministes de pseudo-longs
# -----------------------------------------------------------------------------

def choose_duration_bin(rng):
    index = int(rng.choice(
        len(TARGET_DURATION_BINS), p=np.asarray(TARGET_DURATION_WEIGHTS)
    ))
    return TARGET_DURATION_BINS[index]


def build_one_recipe(language, split, ordinal, pool):
    rng = np.random.default_rng(
        int.from_bytes(hashlib.sha256(
            f"{SEED}:{language}:{split}:{ordinal}".encode()
        ).digest()[:8], "little")
    )
    low, high = choose_duration_bin(rng)
    target = float(rng.uniform(low, high))
    records = pool.to_dict("records")

    for attempt in range(300):
        lead = float(rng.uniform(0.0, 1.2))
        trail = float(rng.uniform(0.0, 1.2))
        chosen, gaps = [], []
        used = set()
        current = lead + trail

        while len(chosen) < MAX_SEGMENTS and current < target:
            remaining = target - current
            candidates = [
                row for row in records
                if row["sample_key"] not in used
                and float(row["seconds"]) <= max(4.0, remaining + 1.5)
                and current + float(row["seconds"]) <= high + 2.0
            ]
            if not candidates:
                candidates = [
                    row for row in records
                    if row["sample_key"] not in used
                    and current + float(row["seconds"]) <= high + 2.0
                ]
            if not candidates:
                break
            # Tirer parmi les 24 durees les plus utiles conserve de la variete tout
            # en evitant de depasser 90 s.
            candidates = sorted(
                candidates,
                key=lambda row: abs(float(row["seconds"]) - max(2.0, remaining)),
            )[:24]
            row = candidates[int(rng.integers(0, len(candidates)))]
            gap = 0.0 if not chosen else float(rng.choice(
                [rng.uniform(0.20, 0.80), rng.uniform(0.80, 2.00), rng.uniform(2.00, 5.00)],
                p=[0.50, 0.35, 0.15],
            ))
            if current + gap + float(row["seconds"]) > high + 2.0:
                used.add(row["sample_key"])
                continue
            chosen.append(row)
            gaps.append(gap)
            used.add(row["sample_key"])
            current += gap + float(row["seconds"])

        forced_seam = None
        # Une recette sur deux force une vraie coupure pres d'une couture CTC
        # (35 s ou 67 s). Sans ce stress-test, une couture qui supprime des frames
        # pourrait passer simplement parce qu'aucune frontiere ne la traverse.
        if len(chosen) >= 2 and ordinal % 2 == 0:
            candidates = []
            cursor = lead
            for boundary in range(len(chosen) - 1):
                cursor += float(gaps[boundary]) + float(chosen[boundary]["seconds"])
                old_gap = float(gaps[boundary + 1])
                for seam in (35.0, 67.0):
                    needed_gap = 2.0 * (seam - cursor)
                    adjusted_total = current - old_gap + needed_gap
                    if (
                        0.20 <= needed_gap <= 5.00
                        and low <= adjusted_total <= high + 2.0
                    ):
                        candidates.append((
                            abs(needed_gap - old_gap), boundary + 1,
                            needed_gap, seam, adjusted_total,
                        ))
            if candidates:
                _, gap_index, new_gap, forced_seam, current = min(candidates)
                gaps[gap_index] = float(new_gap)

        seam_requirement_met = ordinal % 2 == 1 or forced_seam is not None
        if (
            seam_requirement_met
            and MIN_SEGMENTS <= len(chosen) <= MAX_SEGMENTS
            and low <= current <= high + 2.0
        ):
            recipe_id = hashlib.sha256(
                f"{language}:{split}:{ordinal}:{SEED}".encode()
            ).hexdigest()[:24]
            segments = []
            for index, (row, gap) in enumerate(zip(chosen, gaps)):
                segments.append({
                    "index": index,
                    "sample_key": row["sample_key"],
                    "reference": row["reference"],
                    "metadata_seconds": float(row["seconds"]),
                    "gap_before_s": float(gap),
                    "gain_db": float(rng.uniform(-3.0, 3.0)),
                })
            return {
                "recipe_id": recipe_id,
                "language": language,
                "split": split,
                "ordinal": int(ordinal),
                "duration_bin": f"{int(low)}_{int(high)}",
                "target_seconds": target,
                "planned_seconds": current,
                "forced_seam_s": forced_seam,
                "leading_silence_s": lead,
                "trailing_silence_s": trail,
                "segments_json": json.dumps(segments, ensure_ascii=False),
                "segment_count": len(segments),
            }
    raise RuntimeError((language, split, ordinal, "recette pseudo-long impossible"))


recipe_rows = []
for language in LANGUAGES:
    for split in ("tune", "audit"):
        pool = sources[(sources.language == language) & (sources.pool == split)]
        for ordinal in range(PSEUDO_LONGS[split]):
            recipe_rows.append(build_one_recipe(language, split, ordinal, pool))
recipes = pd.DataFrame(recipe_rows).sort_values(
    ["split", "language", "ordinal"], kind="mergesort"
).reset_index(drop=True)
assert recipes.recipe_id.is_unique

used_by_split = defaultdict(set)
for row in recipes.to_dict("records"):
    for segment in json.loads(row["segments_json"]):
        used_by_split[row["split"]].add(segment["sample_key"])
assert used_by_split["tune"].isdisjoint(used_by_split["audit"])


# -----------------------------------------------------------------------------
# 5. Contrat signe puis materialisation locale des seuls clips utilises
# -----------------------------------------------------------------------------

contract_payload = {
    "schema": 2,
    "purpose": "calibrate windowed omniASR CTC segmentation on synthetic long audio",
    "preflight_report": str(preflight_latest),
    "preflight_report_sha256": sha256_file(preflight_latest),
    "dev_selection_sha256": DEV_SELECTION_SHA256,
    "train_selection_sha256": frame_sha256(
        pd.read_parquet(TRAIN_SELECTION_PATH), sort_key="sample_key"
    ),
    "base_bf16_sha256": EXPECTED_BASE_SHA256,
    "ctc_segmentation_version": installed_ctc_version,
    "languages": list(LANGUAGES),
    "seed": SEED,
    "pseudo_longs": PSEUDO_LONGS,
    "negatives_per_audio": NEGATIVES_PER_AUDIO,
    "source_pool_ratios": SOURCE_POOL_RATIOS,
    "source_duration_s": [SOURCE_DURATION_MIN_S, SOURCE_DURATION_MAX_S],
    "target_duration_bins": TARGET_DURATION_BINS,
    "target_duration_weights": TARGET_DURATION_WEIGHTS,
    "stitching": STITCHING,
    "ctc_configs": CTC_CONFIGS,
    "pass_limits": PASS_LIMITS,
    "recipes_sha256": frame_sha256(recipes, sort_key="recipe_id"),
}
contract_sha256 = canonical_sha256(contract_payload)
RUN_ROOT = OUT_ROOT / contract_sha256[:16]
PARTS_ROOT = RUN_ROOT / "parts"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
atomic_json(
    {"created_at": now_iso(), "contract_sha256": contract_sha256,
     "contract": contract_payload},
    RUN_ROOT / "contract.json",
)
atomic_frame_parquet(recipes, RUN_ROOT / "pseudo_long_recipes.parquet")
atomic_frame_parquet(sources, RUN_ROOT / "source_pool_inventory.parquet")

source_by_key = sources.drop_duplicates("sample_key").set_index("sample_key", drop=False)
used_keys = sorted(used_by_split["tune"] | used_by_split["audit"])


def audio_destination(sample_key, language):
    digest = hashlib.sha256(str(sample_key).encode()).hexdigest()[:24]
    return LOCAL_AUDIO_ROOT / language / f"{digest}.audio"


for sample_key in used_keys:
    row = source_by_key.loc[sample_key]
    destination = audio_destination(sample_key, row["language"])
    destination.parent.mkdir(parents=True, exist_ok=True)


def cached_audio_valid(sample_key):
    row = source_by_key.loc[sample_key]
    path = audio_destination(sample_key, row["language"])
    if not path.exists() or path.stat().st_size < 128:
        return False
    try:
        info = sf.info(path)
        measured = float(info.frames) / float(info.samplerate)
        expected = float(row["seconds"])
        return (
            info.frames > 0 and info.samplerate > 0 and info.channels >= 1
            and abs(measured - expected) <= max(0.75, 0.03 * expected)
        )
    except Exception:
        return False


for sample_key in used_keys:
    path = audio_destination(sample_key, source_by_key.loc[sample_key, "language"])
    if path.exists() and not cached_audio_valid(sample_key):
        path.unlink()

missing_rows = [
    source_by_key.loc[key].to_dict() for key in used_keys
    if not cached_audio_valid(key)
]

# Sources fichiers : copie FUSE avec reprises courtes.
for number, row in enumerate(
    [item for item in missing_rows if item["source_kind"] == "file"], 1
):
    source = Path(str(row["audio_ref"]))
    destination = audio_destination(row["sample_key"], row["language"])
    assert source.exists(), source
    for attempt in range(1, 5):
        try:
            temporary = destination.with_name(destination.name + f".tmp-{os.getpid()}")
            shutil.copyfile(source, temporary)
            os.replace(temporary, destination)
            break
        except OSError:
            if temporary.exists():
                temporary.unlink()
            if attempt == 4:
                raise
            time.sleep(2 * attempt)
    if number % 100 == 0:
        print("  fichiers sources locaux :", number)

# Sources parquet : une lecture par partition.
parquet_missing = pd.DataFrame([
    item for item in missing_rows if item["source_kind"] != "file"
])
if not parquet_missing.empty:
    for number, (parquet_path, group) in enumerate(
        parquet_missing.groupby("parquet_path", sort=True), 1
    ):
        parquet_path = Path(str(parquet_path))
        assert parquet_path.exists(), parquet_path
        table = pq.read_table(parquet_path, columns=["audio_bytes"]).to_pylist()
        for row in group.to_dict("records"):
            payload = bytes(table[int(row["row_index"])] ["audio_bytes"])
            destination = audio_destination(row["sample_key"], row["language"])
            temporary = destination.with_name(destination.name + f".tmp-{os.getpid()}")
            with open(temporary, "wb") as handle:
                handle.write(payload)
            os.replace(temporary, destination)
        if number % 20 == 0:
            print("  partitions sources lues :", number)

assert all(
    cached_audio_valid(key)
    for key in used_keys
)
print("Clips sources locaux :", len(used_keys))


# -----------------------------------------------------------------------------
# 6. Audio, modele BF16 step_1250, tokenizer et capture CTC fenetree
# -----------------------------------------------------------------------------

def load_audio_file(path, target_rate=16000):
    waveform, sample_rate = sf.read(path, dtype="float32", always_2d=False)
    if waveform.ndim > 1:
        waveform = waveform.mean(axis=1)
    waveform = np.asarray(waveform, dtype=np.float32)
    if int(sample_rate) != target_rate:
        from scipy.signal import resample_poly
        divisor = math.gcd(int(sample_rate), target_rate)
        waveform = resample_poly(
            waveform, target_rate // divisor, int(sample_rate) // divisor
        ).astype(np.float32)
        sample_rate = target_rate
    assert waveform.ndim == 1 and len(waveform) >= int(0.2 * sample_rate)
    assert np.isfinite(waveform).all()
    return waveform, int(sample_rate)


def assemble_recipe(recipe):
    sample_rate = 16000
    arrays = [np.zeros(round(float(recipe["leading_silence_s"]) * sample_rate), np.float32)]
    truths, texts = [], []
    cursor = len(arrays[0])
    for segment in json.loads(recipe["segments_json"]):
        gap = np.zeros(round(float(segment["gap_before_s"]) * sample_rate), np.float32)
        arrays.append(gap)
        cursor += len(gap)
        source = source_by_key.loc[segment["sample_key"]]
        path = audio_destination(segment["sample_key"], source["language"])
        waveform, rate = load_audio_file(path, sample_rate)
        assert rate == sample_rate
        waveform = waveform * float(10.0 ** (float(segment["gain_db"]) / 20.0))
        start = cursor / sample_rate
        arrays.append(waveform)
        cursor += len(waveform)
        end = cursor / sample_rate
        truths.append((start, end))
        texts.append(segment["reference"])
    arrays.append(np.zeros(round(float(recipe["trailing_silence_s"]) * sample_rate), np.float32))
    waveform = np.concatenate(arrays).astype(np.float32, copy=False)
    peak = float(np.max(np.abs(waveform))) if len(waveform) else 0.0
    if peak > 0.98:
        waveform = waveform * (0.98 / peak)
    duration = len(waveform) / sample_rate
    assert 42.0 <= duration <= 92.0, (recipe["recipe_id"], duration)
    return waveform, sample_rate, texts, truths


def labels_from_tokenizer(tokenizer):
    tokenizer_model = getattr(tokenizer, "_model", None)
    for method in ("index_to_token", "id_to_piece", "IdToPiece"):
        if tokenizer_model is not None and hasattr(tokenizer_model, method):
            labels = [
                str(getattr(tokenizer_model, method)(index))
                for index in range(tokenizer.vocab_info.size)
            ]
            labels[0] = ""
            return labels
    raise RuntimeError("Vocabulaire tokenizer inaccessible")


for name in ("model", "pipe", "tokenizer"):
    globals().pop(name, None)
gc.collect()
torch.cuda.empty_cache()

BASE_MODEL_CARD = globals().get("FT_CONFIG", {}).get(
    "model_card", "omniASR_CTC_1B_v2"
)
device = torch.device("cuda")
model = load_model(BASE_MODEL_CARD, device=device, dtype=torch.bfloat16)
raw_state = torch.load(
    LOCAL_BASE_WEIGHT, map_location="cpu", weights_only=False, mmap=True
)
state = raw_state.get("model", raw_state) if isinstance(raw_state, dict) else raw_state
incompatible = model.load_state_dict(state)
assert not incompatible.missing_keys and not incompatible.unexpected_keys, incompatible
del raw_state, state
gc.collect()
torch.cuda.empty_cache()
model.eval()
assert sum(parameter.numel() for parameter in model.parameters()) == 975_675_056

tokenizer = load_tokenizer(BASE_MODEL_CARD)
labels = labels_from_tokenizer(tokenizer)
vocab_size = len(labels)
blank_id = 0
assert labels[blank_id] == ""
encoder = tokenizer.create_encoder()
vocab_info = tokenizer.vocab_info
special_ids = {blank_id}
for name in (
    "unk_idx", "bos_idx", "eos_idx", "pad_idx", "sep_idx",
):
    value = getattr(vocab_info, name, None)
    if value is not None:
        special_ids.add(int(value))
unk_id = getattr(vocab_info, "unk_idx", None)
pipe = ASRInferencePipeline(None, model=model, tokenizer=tokenizer)

token_cache = {}


def tokenize_reference(text):
    text = normalize_text(text)
    if text in token_cache:
        return token_cache[text]
    raw = encoder(text)
    ids = raw.tolist() if hasattr(raw, "tolist") else list(raw)
    ids = [int(value) for value in ids]
    if unk_id is not None and int(unk_id) in ids:
        raise ValueError(f"Token <unk> dans la reference : {text[:120]}")
    ids = [value for value in ids if value not in special_ids]
    assert ids and min(ids) >= 0 and max(ids) < vocab_size, (text, ids[:10])
    result = np.asarray(ids, dtype=np.int64)
    token_cache[text] = result
    return result


def language_code(language):
    existing = globals().get("omni_lang")
    if callable(existing):
        return existing(language)
    return LANG_TO_OMNI[language]


def capture_one_window(waveform, sample_rate, language):
    captured = []
    original_forward = pipe.model.forward

    def spy(source_seqs, source_seq_lens, *args, **kwargs):
        output = original_forward(source_seqs, source_seq_lens, *args, **kwargs)
        logits, layout = output[0], output[1]
        length = layout.seq_lens[0]
        length = int(length.item() if hasattr(length, "item") else length)
        log_probs = torch.log_softmax(
            logits[0, :length].detach().float(), dim=-1
        ).cpu().numpy().astype(np.float32, copy=False)
        captured.append(log_probs)
        return output

    pipe.model.forward = spy
    try:
        with torch.inference_mode():
            pipe.transcribe(
                [{"waveform": waveform, "sample_rate": sample_rate}],
                lang=[language_code(language)], batch_size=1,
            )
    finally:
        pipe.model.forward = original_forward
    assert len(captured) == 1
    return captured[0]


def capture_stitched(waveform, sample_rate, language):
    duration = len(waveform) / sample_rate
    window_samples = round(STITCHING["window_s"] * sample_rate)
    hop_samples = round(STITCHING["hop_s"] * sample_rate)
    trim_s = float(STITCHING["trim_s"])
    offsets = list(range(0, len(waveform), hop_samples))
    parts, kept_ranges, seam_times = [], [], []

    for index, offset in enumerate(offsets):
        segment = waveform[offset:offset + window_samples]
        if len(segment) < round(0.2 * sample_rate):
            continue
        is_first = index == 0
        is_last = offset + window_samples >= len(waveform)
        log_probs = capture_one_window(segment, sample_rate, language)
        fps = len(log_probs) / (len(segment) / sample_rate)
        leexperiment_run = 0.0 if is_first else trim_s
        right_s = len(segment) / sample_rate if is_last else len(segment) / sample_rate - trim_s
        left = int(round(leexperiment_run * fps))
        right = int(round(right_s * fps))
        assert 0 <= left < right <= len(log_probs)
        parts.append(log_probs[left:right])
        kept_ranges.append((offset / sample_rate + leexperiment_run, offset / sample_rate + right_s))
        if not is_last:
            seam_times.append(offset / sample_rate + right_s)
        if is_last:
            break

    assert parts
    for previous, current in zip(kept_ranges, kept_ranges[1:]):
        assert abs(previous[1] - current[0]) <= 0.08, (previous, current)
    stitched = parts[0] if len(parts) == 1 else np.concatenate(parts, axis=0)
    assert stitched.ndim == 2 and stitched.shape[1] == vocab_size
    assert np.isfinite(stitched).all()
    sample_frames = np.linspace(0, len(stitched) - 1, min(32, len(stitched))).astype(int)
    normalizers = np.logaddexp.reduce(stitched[sample_frames], axis=1)
    assert float(np.max(np.abs(normalizers))) < 2e-3, normalizers[:5]
    return stitched, seam_times, duration


# -----------------------------------------------------------------------------
# 7. Alignement, negatifs et metriques par pseudo-long
# -----------------------------------------------------------------------------

def structural_diagnostics(segments, audio_seconds, expected_count):
    if len(segments) != expected_count:
        return False, 1
    malformed = 0
    previous_start = -float("inf")
    for start, end, score in segments:
        if not (
            np.isfinite(start) and np.isfinite(end) and np.isfinite(score)
            and -0.25 <= start < end <= audio_seconds + 0.75
            and start >= previous_start - 0.05
        ):
            malformed += 1
        previous_start = start
    for left, right in zip(segments, segments[1:]):
        if left[1] > right[0] + 0.10:
            malformed += 1
    return malformed == 0, malformed


def align_texts(log_probs, audio_seconds, texts, config_spec):
    config = CtcSegmentationParameters(char_list=labels)
    config.blank = blank_id
    config.index_duration = float(audio_seconds / len(log_probs))
    config.score_min_mean_over_L = int(config_spec["score_min_mean_over_L"])
    config.blank_transition_cost_zero = bool(
        config_spec["blank_transition_cost_zero"]
    )
    config.replace_spaces_with_blanks = False
    token_lists = [tokenize_reference(text) for text in texts]
    ground_truth, utterance_indices = prepare_token_list(config, token_lists)
    assert len(log_probs) > len(ground_truth), (
        len(log_probs), len(ground_truth), [len(value) for value in token_lists]
    )
    timings, char_probs, _ = ctc_segmentation(
        config, log_probs, ground_truth
    )
    segments = determine_utterance_segments(
        config, utterance_indices, char_probs, timings, texts
    )
    segments = [(float(a), float(b), float(c)) for a, b, c in segments]
    structural_ok, malformed = structural_diagnostics(
        segments, audio_seconds, len(texts)
    )
    score = min((segment[2] for segment in segments), default=-float("inf"))
    return segments, structural_ok, malformed, float(score)


def negative_transcripts(recipe_id, language, texts, segment_keys, pool):
    rng = np.random.default_rng(
        int.from_bytes(hashlib.sha256(
            f"negative:{SEED}:{recipe_id}".encode()
        ).digest()[:8], "little")
    )
    outside = pool[~pool.sample_key.isin(segment_keys)].copy()
    assert not outside.empty
    outputs = []
    types = (
        "replace_middle", "replace_first", "replace_last",
        "delete_words", "repeat_words", "permute_words",
    )
    for negative_type in types[:NEGATIVES_PER_AUDIO]:
        corrupted = list(texts)
        if negative_type == "replace_middle":
            index = len(texts) // 2
            target_words = len(texts[index].split())
            candidates = outside[outside.reference.ne(texts[index])].assign(
                distance=outside.reference.str.split().str.len().sub(target_words).abs()
            ).nsmallest(min(20, len(outside)), "distance")
            assert not candidates.empty, (recipe_id, negative_type)
            corrupted[index] = candidates.iloc[int(rng.integers(len(candidates)))].reference
        elif negative_type in ("replace_first", "replace_last"):
            index = 0 if negative_type == "replace_first" else len(texts) - 1
            target_words = len(texts[index].split())
            candidates = outside[outside.reference.ne(texts[index])].assign(
                distance=outside.reference.str.split().str.len().sub(target_words).abs()
            ).nsmallest(min(20, len(outside)), "distance")
            assert not candidates.empty, (recipe_id, negative_type)
            corrupted[index] = candidates.iloc[int(rng.integers(len(candidates)))].reference
        else:
            eligible = [i for i, text in enumerate(texts) if len(text.split()) >= 4]
            index = eligible[int(rng.integers(len(eligible)))] if eligible else len(texts) // 2
            words = texts[index].split()
            if negative_type == "delete_words":
                count = max(1, math.ceil(0.25 * len(words)))
                remove = set(rng.choice(len(words), size=count, replace=False).tolist())
                corrupted[index] = " ".join(
                    word for position, word in enumerate(words) if position not in remove
                )
            elif negative_type == "repeat_words":
                start = int(rng.integers(0, max(1, len(words) - 1)))
                count = max(1, math.ceil(0.25 * len(words)))
                block = words[start:min(len(words), start + count)]
                corrupted[index] = " ".join(words[:start] + block + block + words[start:])
            elif negative_type == "permute_words":
                shuffled = list(words)
                rng.shuffle(shuffled)
                if shuffled == words:
                    shuffled = words[1:] + words[:1]
                corrupted[index] = " ".join(shuffled)
            else:
                raise AssertionError(negative_type)
        corrupted = [normalize_text(value) for value in corrupted]
        if corrupted == texts or not corrupted[index]:
            fallback = outside[outside.reference.ne(texts[index])]
            assert not fallback.empty, (recipe_id, negative_type, "fallback absent")
            corrupted[index] = fallback.iloc[int(rng.integers(len(fallback)))].reference
            corrupted = [normalize_text(value) for value in corrupted]
        assert corrupted != texts and corrupted[index], (recipe_id, negative_type)
        outputs.append({
            "type": negative_type,
            "corrupted_index": int(index),
            "texts": corrupted,
        })
    assert len(outputs) == NEGATIVES_PER_AUDIO, (recipe_id, len(outputs))
    return outputs


def positive_metrics(segments, truths, seam_times):
    start_errors, end_errors = [], []
    start_late, end_early = [], []
    for (pred_start, pred_end, _), (true_start, true_end) in zip(segments, truths):
        start_errors.append(abs(pred_start - true_start))
        end_errors.append(abs(pred_end - true_end))
        start_late.append(max(0.0, pred_start - true_start))
        end_early.append(max(0.0, true_end - pred_end))
    cut_errors, cut_in_gap, long_gap_cut_in_gap, near_seam_errors = [], [], [], []
    for index in range(len(segments) - 1):
        predicted_cut = 0.5 * (segments[index][1] + segments[index + 1][0])
        true_cut = 0.5 * (truths[index][1] + truths[index + 1][0])
        error = abs(predicted_cut - true_cut)
        cut_errors.append(error)
        cut_in_gap.append(bool(
            truths[index][1] <= predicted_cut <= truths[index + 1][0]
        ))
        if truths[index + 1][0] - truths[index][1] >= 1.0:
            long_gap_cut_in_gap.append(bool(
                truths[index][1] <= predicted_cut <= truths[index + 1][0]
            ))
        if any(abs(true_cut - seam) <= 1.0 for seam in seam_times):
            near_seam_errors.append(error)
    return {
        "start_errors_json": json.dumps(start_errors),
        "end_errors_json": json.dumps(end_errors),
        "start_late_json": json.dumps(start_late),
        "end_early_json": json.dumps(end_early),
        "cut_errors_json": json.dumps(cut_errors),
        "cut_in_gap_json": json.dumps(cut_in_gap),
        "long_gap_cut_in_gap_json": json.dumps(long_gap_cut_in_gap),
        "near_seam_cut_errors_json": json.dumps(near_seam_errors),
    }


def process_recipe(recipe, config_specs):
    started = time.perf_counter()
    waveform, sample_rate, texts, truths = assemble_recipe(recipe)
    log_probs, seam_times, audio_seconds = capture_stitched(
        waveform, sample_rate, recipe["language"]
    )
    segment_keys = [item["sample_key"] for item in json.loads(recipe["segments_json"])]
    pool = sources[
        (sources.language == recipe["language"])
        & (sources.pool == recipe["split"])
    ]
    negatives = negative_transcripts(
        recipe["recipe_id"], recipe["language"], texts, segment_keys, pool
    )
    output = []
    for spec in config_specs:
        base = {
            "contract_sha256": contract_sha256,
            "recipe_id": recipe["recipe_id"],
            "language": recipe["language"],
            "split": recipe["split"],
            "config_id": spec["id"],
            "audio_seconds": float(audio_seconds),
            "frames": int(len(log_probs)),
            "segment_count": int(len(texts)),
            "seam_count": int(len(seam_times)),
        }
        try:
            segments, structural_ok, malformed, score = align_texts(
                log_probs, audio_seconds, texts, spec
            )
            segment_scores = [float(segment[2]) for segment in segments]
            output.append({
                **base, "variant": "positive", "variant_index": -1,
                "corrupted_index": -1, "group_score": score,
                "target_score": score,
                "segment_scores_json": json.dumps(segment_scores),
                "structural_ok": bool(structural_ok),
                "malformed_count": int(malformed), "error": "",
                **positive_metrics(segments, truths, seam_times),
            })
        except Exception as error:
            output.append({
                **base, "variant": "positive", "variant_index": -1,
                "corrupted_index": -1, "group_score": -float("inf"),
                "target_score": -float("inf"),
                "segment_scores_json": "[]",
                "structural_ok": False, "malformed_count": 1,
                "error": f"{type(error).__name__}: {error}",
                "start_errors_json": "[]", "end_errors_json": "[]",
                "start_late_json": "[]", "end_early_json": "[]",
                "cut_errors_json": "[]", "cut_in_gap_json": "[]",
                "long_gap_cut_in_gap_json": "[]",
                "near_seam_cut_errors_json": "[]",
            })

        for index, negative in enumerate(negatives):
            try:
                negative_segments, structural_ok, malformed, score = align_texts(
                    log_probs, audio_seconds, negative["texts"], spec
                )
                segment_scores = [float(segment[2]) for segment in negative_segments]
                target_score = segment_scores[int(negative["corrupted_index"])]
                error_text = ""
            except Exception as error:
                structural_ok, malformed, score = False, 1, -float("inf")
                segment_scores, target_score = [], -float("inf")
                error_text = f"{type(error).__name__}: {error}"
            output.append({
                **base, "variant": negative["type"], "variant_index": index,
                "corrupted_index": negative["corrupted_index"],
                "group_score": float(score),
                "target_score": float(target_score),
                "segment_scores_json": json.dumps(segment_scores),
                "structural_ok": bool(structural_ok),
                "malformed_count": int(malformed), "error": error_text,
                "start_errors_json": "[]", "end_errors_json": "[]",
                "start_late_json": "[]", "end_early_json": "[]",
                "cut_errors_json": "[]", "cut_in_gap_json": "[]",
                "long_gap_cut_in_gap_json": "[]",
                "near_seam_cut_errors_json": "[]",
            })
    elapsed = time.perf_counter() - started
    for record in output:
        record["elapsed_seconds"] = float(elapsed)
    del log_probs, waveform
    gc.collect()
    torch.cuda.empty_cache()
    return output


def load_complete_records(split, language, expected_config_ids):
    directory = PARTS_ROOT / split / language
    records = []
    if directory.exists():
        for part_order, path in enumerate(sorted(directory.glob("part-*.parquet"))):
            part_records = pq.read_table(path).to_pylist()
            for record in part_records:
                record["_part_order"] = part_order
            records.extend(part_records)
    if not records:
        return [], set()
    frame = pd.DataFrame(records)
    key_columns = ["recipe_id", "config_id", "variant_index"]
    # Une interruption ne devrait jamais produire une part partielle (ecriture
    # atomique), mais on rend quand meme la reprise tolerante aux anciens restes.
    frame = frame.sort_values("_part_order", kind="mergesort").drop_duplicates(
        key_columns, keep="last"
    )
    expected_rows = len(expected_config_ids) * (1 + NEGATIVES_PER_AUDIO)
    counts = frame.groupby("recipe_id").size()
    configs = frame.groupby("recipe_id").config_id.agg(set)
    complete = set(
        recipe_id for recipe_id in counts.index
        if int(counts[recipe_id]) == expected_rows
        and configs[recipe_id] == set(expected_config_ids)
    )
    frame = frame[frame.recipe_id.isin(complete)].drop(columns=["_part_order"])
    return frame.to_dict("records"), complete


def run_split(split, config_by_language):
    all_records = []
    for language in LANGUAGES:
        specs = config_by_language[language]
        config_ids = [item["id"] for item in specs]
        existing, complete_ids = load_complete_records(split, language, config_ids)
        all_records.extend(existing)
        subset = recipes[(recipes.split == split) & (recipes.language == language)]
        pending = subset[~subset.recipe_id.isin(complete_ids)]
        print(f"\n{split} {language}: {len(complete_ids)}/{len(subset)} deja complets")
        buffer, part_index = [], 0
        directory = PARTS_ROOT / split / language
        directory.mkdir(parents=True, exist_ok=True)
        existing_parts = sorted(directory.glob("part-*.parquet"))
        if existing_parts:
            part_index = max(int(path.stem.split("-")[-1]) for path in existing_parts) + 1
        for ordinal, recipe in enumerate(pending.to_dict("records"), 1):
            buffer.extend(process_recipe(recipe, specs))
            if ordinal % PART_SIZE == 0 or ordinal == len(pending):
                path = directory / f"part-{part_index:05d}.parquet"
                atomic_parquet(buffer, path)
                all_records.extend(buffer)
                buffer = []
                part_index += 1
                print(f"  {ordinal}/{len(pending)} nouveaux | {path.name}")
    return pd.DataFrame(all_records)


# -----------------------------------------------------------------------------
# 8. Calibration : quatre configs, seuil par langue, sans lecture de l'audit
# -----------------------------------------------------------------------------

all_specs = {language: list(CTC_CONFIGS) for language in LANGUAGES}
tune_frame = run_split("tune", all_specs)
assert not tune_frame.empty


def extend_json_values(series):
    output = []
    for value in series:
        output.extend(json.loads(value or "[]"))
    return output


def choose_threshold(positive, negative, target_fpr=0.005):
    positive_scores = positive.group_score.to_numpy(float)
    # Chaque negatif ne corrompt qu'un segment. Son score cible est donc celui
    # du segment corrompu, et non le minimum global qui pourrait appartenir a un
    # autre segment legitime deja difficile.
    negative_scores = negative.target_score.to_numpy(float)
    candidates = sorted(set(
        value for value in np.concatenate([positive_scores, negative_scores])
        if np.isfinite(value)
    ))
    if candidates:
        candidates = [np.nextafter(candidates[0], -np.inf)] + candidates
    else:
        candidates = [0.0]
    best = None
    for threshold in candidates:
        pos_accept = positive.structural_ok.to_numpy(bool) & (positive_scores >= threshold)
        neg_accept = negative.structural_ok.to_numpy(bool) & (negative_scores >= threshold)
        fpr = float(neg_accept.mean()) if len(neg_accept) else 1.0
        retention = float(pos_accept.mean()) if len(pos_accept) else 0.0
        if fpr <= target_fpr + 1e-12:
            candidate = (retention, -float(threshold), float(threshold), fpr)
            if best is None or candidate[:2] > best[:2]:
                best = candidate
    if best is None:
        threshold = float(np.max(negative_scores[np.isfinite(negative_scores)])) + 1e-6
        pos_accept = positive.structural_ok.to_numpy(bool) & (positive_scores >= threshold)
        return threshold, float(pos_accept.mean()), 0.0
    return best[2], best[0], best[3]


calibration_rows = []
for language in LANGUAGES:
    for spec in CTC_CONFIGS:
        group = tune_frame[
            (tune_frame.language == language)
            & (tune_frame.config_id == spec["id"])
        ]
        positive = group[group.variant == "positive"].copy()
        negative = group[group.variant != "positive"].copy()
        assert len(positive) == PSEUDO_LONGS["tune"]
        assert len(negative) == PSEUDO_LONGS["tune"] * NEGATIVES_PER_AUDIO
        threshold, retention, fpr = choose_threshold(positive, negative)
        accepted = positive[
            positive.structural_ok & positive.group_score.ge(threshold)
        ]
        cut_errors = extend_json_values(accepted.cut_errors_json)
        cut_in_gap = extend_json_values(accepted.cut_in_gap_json)
        long_gap_cut_in_gap = extend_json_values(accepted.long_gap_cut_in_gap_json)
        undercuts = (
            extend_json_values(accepted.start_late_json)
            + extend_json_values(accepted.end_early_json)
        )
        endpoints = (
            extend_json_values(accepted.start_errors_json)
            + extend_json_values(accepted.end_errors_json)
        )
        calibration_rows.append({
            "language": language,
            "config_id": spec["id"],
            "threshold": float(threshold),
            "positive_acceptance": retention,
            "negative_false_accept": fpr,
            "structural_anomaly": float((~positive.structural_ok).mean()),
            "cut_error_p95_s": percentile(cut_errors, 0.95),
            "cut_inside_injected_gap": float(np.mean(cut_in_gap)) if cut_in_gap else 0.0,
            "long_gap_cut_inside": float(np.mean(long_gap_cut_in_gap)) if long_gap_cut_in_gap else 0.0,
            "speech_undercut_p95_s": percentile(undercuts, 0.95),
            "endpoint_abs_error_p95_s": percentile(endpoints, 0.95),
            "accepted_positive": int(len(accepted)),
            "positive_total": int(len(positive)),
            "negative_total": int(len(negative)),
        })

calibration_table = pd.DataFrame(calibration_rows)
selected = {}
for language in LANGUAGES:
    candidates = calibration_table[calibration_table.language == language].copy()
    candidates["guard"] = (
        candidates.positive_acceptance.ge(0.90)
        & candidates.negative_false_accept.le(0.01)
        & candidates.cut_error_p95_s.fillna(float("inf")).le(0.80)
    )
    eligible = candidates[candidates.guard]
    if eligible.empty:
        eligible = candidates
    chosen = eligible.sort_values(
        ["positive_acceptance", "negative_false_accept", "cut_error_p95_s"],
        ascending=[False, True, True], kind="mergesort",
    ).iloc[0]
    spec = next(item for item in CTC_CONFIGS if item["id"] == chosen.config_id)
    selected[language] = {
        "config": spec,
        "threshold": float(chosen.threshold),
        "tune_metrics": {
            key: (value.item() if hasattr(value, "item") else value)
            for key, value in chosen.drop(labels=["language", "guard"]).to_dict().items()
        },
    }

calibration = {
    "schema": 2,
    "created_at": now_iso(),
    "contract_sha256": contract_sha256,
    "selection_data": "tune only; audit unopened at selection time",
    "score_direction": "higher (closer to zero) is better",
    "selected": selected,
    "table": calibration_table.to_dict("records"),
}
calibration["calibration_sha256"] = canonical_sha256({
    key: value for key, value in calibration.items()
    if key not in {"created_at", "calibration_sha256"}
})
atomic_json(calibration, RUN_ROOT / "calibration.json")

print("\n=== CALIBRATION 20.2 ===")
show(calibration_table.sort_values(["language", "positive_acceptance"], ascending=[True, False]))
print("\nChoix geles avant audit :")
for language, choice in selected.items():
    print(language, "->", choice["config"]["id"], "seuil", round(choice["threshold"], 5))


# -----------------------------------------------------------------------------
# 9. Audit final : uniquement la config gelee de chaque langue
# -----------------------------------------------------------------------------

audit_specs = {language: [selected[language]["config"]] for language in LANGUAGES}
audit_frame = run_split("audit", audit_specs)
assert not audit_frame.empty

audit_rows = []
for language in LANGUAGES:
    choice = selected[language]
    group = audit_frame[
        (audit_frame.language == language)
        & (audit_frame.config_id == choice["config"]["id"])
    ]
    positive = group[group.variant == "positive"].copy()
    negative = group[group.variant != "positive"].copy()
    assert len(positive) == PSEUDO_LONGS["audit"]
    assert len(negative) == PSEUDO_LONGS["audit"] * NEGATIVES_PER_AUDIO
    threshold = float(choice["threshold"])
    positive_accept = positive.structural_ok & positive.group_score.ge(threshold)
    negative_accept = negative.structural_ok & negative.target_score.ge(threshold)
    accepted = positive[positive_accept]
    cut_errors = extend_json_values(accepted.cut_errors_json)
    cut_in_gap = extend_json_values(accepted.cut_in_gap_json)
    long_gap_cut_in_gap = extend_json_values(accepted.long_gap_cut_in_gap_json)
    near_seam = extend_json_values(accepted.near_seam_cut_errors_json)
    undercuts = (
        extend_json_values(accepted.start_late_json)
        + extend_json_values(accepted.end_early_json)
    )
    endpoints = (
        extend_json_values(accepted.start_errors_json)
        + extend_json_values(accepted.end_errors_json)
    )
    false_accepts = int(negative_accept.sum())
    false_accept_recipe_ids = set(negative.loc[negative_accept, "recipe_id"].astype(str))
    false_accept_audio_rate = len(false_accept_recipe_ids) / max(1, positive.recipe_id.nunique())
    cut_p95 = percentile(cut_errors, 0.95)
    near_seam_p95 = percentile(near_seam, 0.95)
    record = {
        "language": language,
        "config_id": choice["config"]["id"],
        "threshold": threshold,
        "positive_acceptance": float(positive_accept.mean()),
        "negative_false_accept": float(negative_accept.mean()),
        "negative_false_accept_count": false_accepts,
        "negative_total": int(len(negative)),
        "negative_false_accept_wilson95_high": wilson_upper(false_accepts, len(negative)),
        "negative_false_accept_audio": float(false_accept_audio_rate),
        "structural_anomaly": float((~positive.structural_ok).mean()),
        "cut_error_p50_s": percentile(cut_errors, 0.50),
        "cut_error_p95_s": cut_p95,
        "cut_error_p99_s": percentile(cut_errors, 0.99),
        "cut_inside_injected_gap": float(np.mean(cut_in_gap)) if cut_in_gap else 0.0,
        "long_gap_cut_inside": float(np.mean(long_gap_cut_in_gap)) if long_gap_cut_in_gap else 0.0,
        "near_seam_cut_error_p95_s": near_seam_p95,
        "near_seam_cut_count": int(len(near_seam)),
        "near_seam_degradation_p95_s": (
            float(near_seam_p95 - cut_p95)
            if near_seam_p95 is not None and cut_p95 is not None else None
        ),
        "speech_undercut_p95_s": percentile(undercuts, 0.95),
        "endpoint_abs_error_p95_s": percentile(endpoints, 0.95),
        "accepted_positive": int(positive_accept.sum()),
        "positive_total": int(len(positive)),
        "audit_audio_hours": float(positive.audio_seconds.sum() / 3600),
    }
    checks = {
        "positive_acceptance": record["positive_acceptance"] >= PASS_LIMITS["positive_acceptance_min"],
        "negative_false_accept": record["negative_false_accept"] <= PASS_LIMITS["negative_false_accept_max"],
        "negative_false_accept_count": record["negative_false_accept_count"] <= PASS_LIMITS["negative_false_accept_count_max"],
        "negative_false_accept_audio": record["negative_false_accept_audio"] <= PASS_LIMITS["negative_false_accept_audio_max"],
        "structural_anomaly": record["structural_anomaly"] <= PASS_LIMITS["structural_anomaly_max"],
        "cut_error_p95": (record["cut_error_p95_s"] is not None and record["cut_error_p95_s"] <= PASS_LIMITS["cut_error_p95_s_max"]),
        # Les petits gaps (0.2-0.8 s) sont un stress-test descriptif : le silence
        # natif en bord de fichier peut placer une bonne coupure juste en dehors.
        # Le garde-fou principal porte donc sur les gaps artificiels >=1 s.
        "long_gap_cut_inside": record["long_gap_cut_inside"] >= PASS_LIMITS["long_gap_cut_inside_min"],
        "near_seam_count": record["near_seam_cut_count"] >= PASS_LIMITS["near_seam_cut_count_min"],
        "near_seam_p95": (record["near_seam_cut_error_p95_s"] is not None and record["near_seam_cut_error_p95_s"] <= PASS_LIMITS["near_seam_cut_error_p95_s_max"]),
        "near_seam_degradation": (record["near_seam_degradation_p95_s"] is not None and record["near_seam_degradation_p95_s"] <= PASS_LIMITS["near_seam_degradation_p95_s_max"]),
        "speech_undercut_p95": (record["speech_undercut_p95_s"] is not None and record["speech_undercut_p95_s"] <= PASS_LIMITS["speech_undercut_p95_s_max"]),
        "endpoint_abs_error_p95": (record["endpoint_abs_error_p95_s"] is not None and record["endpoint_abs_error_p95_s"] <= PASS_LIMITS["endpoint_abs_error_p95_s_max"]),
    }
    record["checks"] = checks
    record["pass"] = bool(all(checks.values()))
    audit_rows.append(record)

audit_table = pd.DataFrame([{**row, "checks": json.dumps(row["checks"])} for row in audit_rows])
all_pass = all(row["pass"] for row in audit_rows)
catastrophic = any(
    row["positive_acceptance"] < 0.80
    or row["negative_false_accept"] > 0.05
    or row["structural_anomaly"] > 0.05
    for row in audit_rows
)
status = (
    "PASS_READY_FOR_20_3_LONG_AUDIO_MATERIALIZATION_PILOT"
    if all_pass else ("FAIL" if catastrophic else "REVIEW")
)

report = {
    "schema": 2,
    "created_at": now_iso(),
    "status": status,
    "contract_sha256": contract_sha256,
    "calibration_sha256": calibration["calibration_sha256"],
    "base_bf16_sha256": EXPECTED_BASE_SHA256,
    "ctc_segmentation_version": installed_ctc_version,
    "selected": selected,
    "audit": audit_rows,
    "pass_limits": PASS_LIMITS,
    "source_pool_summary": pool_table.to_dict("records"),
    "recipes": {
        "path": str(RUN_ROOT / "pseudo_long_recipes.parquet"),
        "sha256": frame_sha256(recipes, sort_key="recipe_id"),
        "tune": int((recipes.split == "tune").sum()),
        "audit": int((recipes.split == "audit").sum()),
    },
    "parts_root": str(PARTS_ROOT),
    "data_leakage_checks": {
        "train_vs_dev_sample_keys_disjoint": True,
        "tune_vs_audit_sample_keys_disjoint": True,
        "tune_vs_audit_speakers_disjoint": True,
        "tune_vs_audit_normalized_text_disjoint": True,
        "kaggle_test_used": False,
        "anv_long_audio_downloaded": False,
    },
    "next_step": (
        "20.3: materialiser un petit pilote reel 38-90 s puis auditer les segments"
        if all_pass else
        "Ne telecharger aucun lot ANV massif; examiner les checks en echec."
    ),
}
report["report_sha256"] = canonical_sha256({
    key: value for key, value in report.items()
    if key not in {"created_at", "report_sha256"}
})
atomic_json(report, RUN_ROOT / "ctc_segmentation_audit.json")
atomic_json(report, OUT_ROOT / "LATEST.json")

print("\n=== AUDIT FINAL 20.2 ===")
show(audit_table.drop(columns=["checks"]))
print("\nSTATUT 20.2 :", status)
print("Rapport       :", RUN_ROOT / "ctc_segmentation_audit.json")
if all_pass:
    print("Etape suivante : 20.3, petit pilote reel 38-90 s. Pas encore le lot complet.")
else:
    failed = {
        row["language"]: [name for name, passed in row["checks"].items() if not passed]
        for row in audit_rows if not row["pass"]
    }
    print("Checks a revoir :", failed)
    print("Ne telecharger aucun lot ANV massif avant correction.")

In [ ]:
import os
import subprocess
from pathlib import Path

SEARCH_ROOT = Path(
    "/content/afrivoices_project"
)

assert SEARCH_ROOT.exists(), (
    "Google Drive n'est pas monté ou le dossier projet est absent."
)

commands = {
    "TSV": [
        "find", str(SEARCH_ROOT), "-type", "f",
        "-name", "language_distribution_0.tsv"
    ],
    "KALENJIN": [
        "find", str(SEARCH_ROOT), "-type", "d",
        "-name", "language=kln_Latn"
    ],
    "MAASAI": [
        "find", str(SEARCH_ROOT), "-type", "d",
        "-name", "language=mas_Latn"
    ],
    "COMPLETE": [
        "find", str(SEARCH_ROOT), "-type", "f",
        "-name", "_COMPLETE.json"
    ],
}

print("=== RECHERCHE DU DATASET V4 SUR DRIVE ===")

for label, command in commands.items():
    print(f"\n--- {label} ---")
    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        timeout=900,
    )

    paths = [
        line.strip()
        for line in result.stdout.splitlines()
        if line.strip()
    ]

    if label == "COMPLETE":
        paths = [
            path for path in paths
            if "balanced" in path.lower()
            or "dataset" in path.lower()
        ]

    if paths:
        for path in paths[:50]:
            print(path)
    else:
        print("AUCUN RÉSULTAT")

    if result.stderr.strip():
        print("stderr :", result.stderr[-2000:])

In [ ]:
import gc
import torch

for name in ("pipe", "model", "tokenizer", "raw_state", "state", "wrappers"):
    globals().pop(name, None)

gc.collect()
torch.cuda.empty_cache()
print("VRAM nettoyée")

In [ ]:
# Audit Drive — configurations exactes des soumissions
import glob
import json
import os
from pathlib import Path

import pandas as pd

FT_ROOT = (
    "/content/afrivoices_project/"
    "finetune_runs/experiment_run"
)
REPORT_DIR = os.path.join(FT_ROOT, "reports")

CONFIGS = {
    "score_036529_pre_step1250": os.path.join(
        REPORT_DIR,
        "kenlm_tuning_by_domain_v3_pre_step1250.json",
    ),
    "score_036880_v4": os.path.join(
        REPORT_DIR,
        "kenlm_tuning_by_domain_v3_score_036880.json",
    ),
    "active": os.path.join(
        REPORT_DIR,
        "kenlm_tuning_by_domain_v3.json",
    ),
}

rows = []

for config_name, config_path in CONFIGS.items():
    print(config_name, "->", os.path.exists(config_path), config_path)

    if not os.path.exists(config_path):
        continue

    config = json.load(open(config_path, encoding="utf-8"))

    for group, values in sorted(config.items()):
        if "|" not in group or not isinstance(values, dict):
            continue

        lm_path = (
            values.get("lm_bin")
            or values.get("kenlm_model_path")
            or values.get("path")
            or ""
        )

        rows.append({
            "configuration": config_name,
            "group": group,
            "lm_name": (
                values.get("lm")
                or values.get("candidate")
                or values.get("tag")
            ),
            "lm_folder": (
                Path(lm_path).parent.name if lm_path else None
            ),
            "lm_file": (
                Path(lm_path).name if lm_path else None
            ),
            "lm_exists": (
                os.path.exists(lm_path) if lm_path else False
            ),
            "alpha": values.get("alpha"),
            "beta": values.get("beta"),
            "beam": values.get("beam"),
            "token_min_logp": values.get("tml"),
            "beam_prune_logp": values.get("bpl"),
        })

comparison = pd.DataFrame(rows)
display(comparison)

print("\n=== CONTENU kenlm_models_v5 ===")
V5_DIR = os.path.join(FT_ROOT, "kenlm_models_v5")

for path in sorted(glob.glob(
    os.path.join(V5_DIR, "**", "*"),
    recursive=True,
)):
    if os.path.isfile(path):
        print(
            os.path.relpath(path, V5_DIR),
            "—",
            round(os.path.getsize(path) / 2**20, 2),
            "Mio",
        )

print("\n=== CHECKPOINTS SPÉCIALISÉS HISTORIQUES ===")
for path in sorted(glob.glob(
    os.path.join(REPORT_DIR, "topup_choice_*.json")
)):
    try:
        value = json.load(open(path, encoding="utf-8"))
        print(os.path.basename(path), "->", value)
    except Exception as error:
        print(os.path.basename(path), "-> erreur", repr(error))

print("\n=== ANCIENNE SOUMISSION ===")
old_submission = os.path.join(REPORT_DIR, "submission_lm.csv")
print("submission_lm.csv existe :", os.path.exists(old_submission))

if os.path.exists(old_submission):
    frame = pd.read_csv(old_submission, dtype=str)
    print("Lignes :", len(frame))
    print("Colonnes :", frame.columns.tolist())
    print("IDs uniques :", frame["id"].is_unique)